# İhale Sonuç Fiyatı Tahmini (Tender Final Price Prediction)

Bu çalışmada, kamu ihale verileri kullanılarak bir ihalenin gerçekleşen sonuç fiyatının (tender_finalpriceUsd) tahmin edilmesi amaçlanmaktadır.

Problem, sürekli bir sayısal değerin tahmin edilmesini gerektirdiğinden **regresyon problemi** olarak ele alınmaktadır.

Modelin temel amacı:

* İhaleye ilişkin fiyat, rekabet, prosedür, tedarik, alıcı ve zaman bilgilerini kullanmak,
* İhalenin gerçekleşen sonuç fiyatını tahmin etmek,
* Tahmin edilen fiyat ile gerçek sonuç fiyatı arasındaki farkı mümkün olduğunca azaltmak,
* Fiyat tahmininde etkili olan değişkenlerin modele sağladığı katkıyı değerlendirmektir.


## 1. Gerekli Kütüphanelerin Yüklenmesi

Bu bölümde veri işleme, görselleştirme ve makine öğrenmesi modelleri için gerekli kütüphaneler içe aktarılmaktadır.

In [ ]:
!pip install unidecode

In [ ]:
import polars as pl
import numpy as np


import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go


from sklearn.model_selection import train_test_split



from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline



from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


## ⚙️ 2. Environment Setup / Ortam Hazırlığı

This section configures the execution environment and connects Google Drive.

Bu bölümde çalışma ortamı hazırlanmakta ve Google Drive bağlantısı kurulmaktadır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 💾 3. Dataset Loading / Veri Setinin Yüklenmesi

This section loads the procurement dataset and performs initial validation.

Bu bölümde ihale veri seti yüklenmekte ve ilk kontroller gerçekleştirilmektedir.

In [ ]:
import polars as pl
import zipfile
import os


zip_path = "/content/drive/MyDrive/Colab Notebooks/SAGE/GTI Global Public Procurement Dataset (GPPD) 12.zip"

csv_inside_zip = "GTI Global Public Procurement Dataset (GPPD) 12/DE_DIB_2023.csv.gz"

extract_path = "/content/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extract(
        csv_inside_zip,
        extract_path
    )


csv_path = "/content/GTI Global Public Procurement Dataset (GPPD) 12/DE_DIB_2023.csv.gz"

df = pl.read_csv(
    csv_path,
    infer_schema_length=10000,
    ignore_errors=True
)

## 🔍 4. Exploratory Data Analysis (EDA) / Keşifsel Veri Analizi

This section explores the dataset using descriptive statistics and visual analysis.

Bu bölümde veri setinin genel yapısı istatistiksel ve görsel yöntemlerle incelenmektedir.


In [ ]:
# Veri setinin genel yapısını inceleme


df.glimpse()

In [ ]:
# Veri setindeki her kolon için eksik (null) değer sayısını hesaplama

import matplotlib.pyplot as plt
import polars as pl

missing = (
    df
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count") / df.height * 100
        ).alias("missing_percentage")
    )
    .filter(
        pl.col("missing_percentage") > 0
    )
    .sort(
        "missing_percentage",
        descending=True
    )
)

missing_pd = missing.to_pandas()

plt.figure(figsize=(18, 7))

bars = plt.bar(
    missing_pd["column"],
    missing_pd["missing_percentage"]
)


for bar, value in zip(bars, missing_pd["missing_percentage"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=8
    )

plt.xticks(rotation=90)
plt.ylabel("Eksik Veri (%)")
plt.xlabel("Kolonlar")
plt.title("Kolon Bazında Eksik Veri Oranları")

plt.ylim(0, missing_pd["missing_percentage"].max() + 10)

plt.tight_layout()
plt.show()

In [ ]:
# Veri setindeki kolonların veri tiplerini inceleme

import polars as pl
import matplotlib.pyplot as plt

dtype_counts = (
    pl.DataFrame({
        "dtype": [str(dtype) for dtype in df.dtypes]
    })
    .group_by("dtype")
    .count()
    .sort(
        "count",
        descending=True
    )
)

dtype_names = dtype_counts["dtype"].to_list()
dtype_values = dtype_counts["count"].to_list()

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({count})"
    return inner

plt.figure(figsize=(8, 8))

plt.pie(
    dtype_values,
    labels=dtype_names,
    autopct=autopct_format(dtype_values),
    startangle=90,
    wedgeprops={"edgecolor": "white"}
)

plt.title("Veri Tiplerinin Dağılımı")

plt.tight_layout()
plt.show()

## 5.Hedef Değişkende Eksik Kayıtların Ayrılması

Bu aşamada tahmin edilecek değişken olan `tender_finalpriceUsd` kolonundaki eksik değerler incelenerek ayrılacaktır.

Model eğitimi için hedef değişkeni bilinen kayıtlar kullanılacaktır.

- `tender_finalpriceUsd` değeri bulunan kayıtlar modelleme veri setine alınacaktır.
- `tender_finalpriceUsd` değeri eksik olan kayıtlar ayrı bir veri setinde tutulacaktır.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np


# 1. TENDER FINAL PRICE İSTATİSTİKLERİ

# Bu bölümde modelin hedef değişkeni olan
# "tender_finalpriceUsd" kolonunun genel veri yapısı incelenir.
#
# Hesaplanan temel istatistikler:
# - Toplam gözlem sayısı
# - Dolu gözlem sayısı
# - Eksik gözlem sayısı
# - Ortalama fiyat
# - Medyan fiyat
# - Standart sapma
# - Maksimum fiyat
#
# Ortalama ve medyan birlikte incelenerek fiyat dağılımının
# çarpık olup olmadığı hakkında fikir edinilir.
# Özellikle ihale fiyatlarında yüksek değerli ihaleler
# ortalamayı ciddi şekilde yukarı çekebileceği için
# medyan önemli bir referans değeridir.

finalprice_analysis = (
    df
    .select([

        pl.len().alias("total_count"),

        pl.col("tender_finalpriceUsd")
        .is_not_null()
        .sum()
        .alias("filled_count"),

        pl.col("tender_finalpriceUsd")
        .null_count()
        .alias("missing_count"),

        pl.col("tender_finalpriceUsd")
        .mean()
        .alias("mean"),

        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median"),


        pl.col("tender_finalpriceUsd")
        .std()
        .alias("std"),

        pl.col("tender_finalpriceUsd")
        .max()
        .alias("max")
    ])

    .with_columns([
        (
            pl.col("filled_count") /
            pl.col("total_count") * 100
        ).alias("filled_percentage"),

        (
            pl.col("missing_count") /
            pl.col("total_count") * 100
        ).alias("missing_percentage")
    ])
)


# Hesaplanan istatistiklerin ilk satırını sözlük formatında alıyoruz.
# Böylece aşağıdaki dashboard içerisinde değerlere kolayca erişebiliriz.

stats = finalprice_analysis.row(
    0,
    named=True
)

# 2. FİYAT FORMATLAMA FONKSİYONU

# Büyük fiyat değerlerinin dashboard üzerinde daha okunabilir
# gösterilmesi amacıyla bir formatlama fonksiyonu oluşturulur.
#
# Örneğin:
# 1,500        -> 1.50 Bin $
# 2,500,000    -> 2.50 M $
# 1,500,000,000 -> 1.50 Milyar $

# Bu fonksiyon yalnızca görsel sunum amacıyla kullanılır.
# Verinin gerçek değerini değiştirmez.

def human_format(value):

    if value >= 1e9:
        return f"{value / 1e9:.2f} Milyar $"

    elif value >= 1e6:
        return f"{value / 1e6:.2f} M $"

    elif value >= 1e3:
        return f"{value / 1e3:.2f} Bin $"

    else:
        return f"{value:.0f} $"


# 3. FİYAT VERİSİNİN HAZIRLANMASI

# Histogram ve boxplot oluşturabilmek için
# tender_finalpriceUsd kolonundaki eksik değerler çıkarılır.
#
# Polars Series -> NumPy dönüşümü yapılır.
# Böylece matplotlib ile doğrudan görselleştirme yapılabilir.

values = (
    df
    .select("tender_finalpriceUsd")
    .drop_nulls()
    .to_series()
    .to_numpy()
)

# 4. TEK SAYFALIK DASHBOARD

# Hedef değişkenin temel özelliklerini tek bir görsel üzerinde
# toplamak amacıyla dashboard yapısı oluşturulur.
#
# Dashboard içerisinde:
#
# 1. KPI kartları
# 2. Veri doluluk donut grafiği
# 3. Fiyat dağılım histogramı
# 4. Boxplot
#
# birlikte gösterilir.
#
# Böylece hedef değişkenin hem sayısal özeti hem de
# dağılım yapısı tek bakışta incelenebilir.

fig = plt.figure(
    figsize=(18, 11)
)

fig.suptitle(
    "İhale Sonuç Fiyatı (tender_finalpriceUsd) Analizi",
    fontsize=22,
    fontweight="bold",
    y=0.98
)

# 5. KPI ALANI

# Hedef değişkenin en önemli özet bilgileri dashboard'un
# üst bölümünde KPI kartları olarak gösterilir.
#
# Kullanılan KPI'lar:
# - Toplam veri
# - Dolu kayıt
# - Doluluk oranı
# - Ortalama fiyat
# - Medyan fiyat
# - Maksimum fiyat
#
# Bu alan özellikle veri kalitesini ve fiyat ölçeğini
# hızlı şekilde değerlendirmek için kullanılır.

ax_kpi = fig.add_axes(
    [0.04, 0.68, 0.92, 0.23]
)

ax_kpi.axis("off")


# Dashboard üzerinde gösterilecek KPI bilgileri

kpis = [

    (
        "Toplam Veri",
        f"{stats['total_count']:,}"
    ),

    (
        "Dolu Kayıt",
        f"{stats['filled_count']:,}"
    ),

    (
        "Doluluk Oranı",
        f"{stats['filled_percentage']:.2f}%"
    ),

    (
        "Ortalama Fiyat",
        human_format(stats["mean"])
    ),

    (
        "Medyan Fiyat",
        human_format(stats["median"])
    ),

    (
        "Maksimum Fiyat",
        human_format(stats["max"])
    )
]


# KPI kartlarının dashboard üzerinde yatay olarak
# sıralanması sağlanır.

for i, (title, value) in enumerate(kpis):

    x = i / 6

    width = 0.145
    rect = plt.Rectangle(
        (x + 0.005, 0.10),
        width,
        0.75,
        fill=False,
        linewidth=1.5
    )

    ax_kpi.add_patch(rect)

    ax_kpi.text(
        x + width / 2 + 0.005,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )


    ax_kpi.text(
        x + width / 2 + 0.005,
        0.36,
        value,
        ha="center",
        va="center",
        fontsize=16,
        fontweight="bold"
    )


# 6. DONUT GRAFİĞİ - VERİ DOLULUK DURUMU

# Hedef değişkenin kaç gözleminde fiyat bulunduğunu ve
# kaç gözleminde eksik değer olduğunu gösterir.
#
# Modelleme açısından hedef değişkende eksik değer bulunması
# önemli bir veri kalitesi problemidir.
#
# Dolu oranının yüksek olması hedef değişkenin modelleme için
# kullanılabilirliğini destekler.

ax1 = fig.add_axes(
    [0.05, 0.36, 0.25, 0.27]
)

filled = stats["filled_count"]
missing = stats["missing_count"]


ax1.pie(
    [filled, missing],
    labels=["Dolu", "Eksik"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops=dict(
        width=0.42,
        edgecolor="white"
    ),
    textprops={
        "fontsize": 10
    }
)


ax1.text(
    0,
    0,
    f"{stats['total_count']:,}\nKayıt",
    ha="center",
    va="center",
    fontsize=13,
    fontweight="bold"
)


ax1.set_title(
    "Veri Doluluk Durumu",
    fontsize=13,
    fontweight="bold"
)

# 7. HİSTOGRAM - FİYAT DAĞILIMI

# Histogram, gerçekleşen ihale fiyatlarının hangi değer
# aralıklarında yoğunlaştığını gösterir.
#
# Bu grafik ile:
# - Fiyatların yoğunlaştığı bölgeler
# - Dağılımın simetrik olup olmadığı
# - Sağ/sol çarpıklık
# - Çok yüksek fiyatların varlığı
#
# incelenebilir.
#
# İhale fiyatlarında genellikle birkaç çok yüksek değer
# bulunabileceğinden sağa çarpık bir dağılım görülmesi
# beklenebilir.

ax2 = fig.add_axes(
    [0.35, 0.36, 0.60, 0.27]
)


ax2.hist(
    values,
    bins=60
)


ax2.set_title(
    "Gerçekleşen İhale Fiyatı Dağılımı",
    fontsize=13,
    fontweight="bold"
)

ax2.set_xlabel(
    "Final Fiyat ($)"
)

ax2.set_ylabel(
    "Frekans"
)


# X eksenindeki büyük sayıların bilimsel gösterim yerine
# normal sayı formatında gösterilmesini sağlar.

ax2.ticklabel_format(
    style="plain",
    axis="x"
)


# Y ekseninde yatay grid kullanılarak frekansların
# daha kolay karşılaştırılması sağlanır.

ax2.grid(
    axis="y",
    alpha=0.25
)

# 8. BOXPLOT - AYKIRI DEĞER ANALİZİ

# Boxplot ile fiyat değişkeninin:
# - Medyanı
# - Çeyrekler arası aralığı (IQR)
# - Alt/üst sınırları
# - Aykırı gözlemleri
# görsel olarak incelenebilir.

# Özellikle yüksek fiyatlı ihalelerin dağılım üzerindeki
# etkisini görmek açısından önemlidir.
#
# Ancak burada görülen her uç değer otomatik olarak
# veri hatası anlamına gelmez. Kamu ihalelerinde gerçekten
# çok yüksek tutarlı ihaleler bulunabileceği için
# aykırı değerlerin silinmesi ayrıca değerlendirilmelidir.



import numpy as np
import matplotlib.pyplot as plt


x = (
    df["tender_finalpriceUsd"]
    .drop_nulls()
    .to_numpy()
)


# IQR
q1 = np.percentile(x,25)
q3 = np.percentile(x,75)

iqr = q3-q1

low = q1 - 1.5*iqr
high = q3 + 1.5*iqr


normal = x[(x>=low)&(x<=high)]
outlier = x[(x<low)|(x>high)]


print("="*60)
print("IQR OUTLIER ANALİZİ")
print("="*60)

print(f"Q1              : ${q1:,.0f}")
print(f"Q3              : ${q3:,.0f}")
print(f"IQR             : ${iqr:,.0f}")
print(f"Alt sınır       : ${low:,.0f}")
print(f"Üst sınır       : ${high:,.0f}")
print(f"Normal kayıt    : {len(normal):,}")
print(f"Aykırı kayıt    : {len(outlier):,}")
print(f"Aykırı oranı    : %{len(outlier)/len(x)*100:.2f}")



# Grafik

plt.figure(figsize=(12,5))


# IQR alanı

plt.axvspan(
    q1,
    q3,
    alpha=0.25,
    label="IQR Alanı (Q1-Q3)"
)


# Normal

plt.scatter(
    normal,
    np.zeros(len(normal)),
    s=15,
    alpha=0.35,
    label="Normal"
)


# Outlier

plt.scatter(
    outlier,
    np.zeros(len(outlier)),
    s=30,
    alpha=0.8,
    label="Aykırı"
)


# Sınırlar

plt.axvline(
    low,
    linestyle="--",
    label="Alt Sınır"
)

plt.axvline(
    high,
    linestyle="--",
    label="Üst Sınır"
)


plt.title(
    "tender_finalpriceUsd - IQR Aykırı Değer Analizi",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    "Final Fiyat ($)"
)

plt.yticks([])

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.show()
# 9. GÖRSEL DÜZEN

# Hazırlanan tüm grafik ve KPI kartları tek dashboard
# üzerinde gösterilir.

plt.show()

In [ ]:
import polars as pl

col = "tender_finalpriceUsd"

negative_count = (
    df
    .filter(pl.col(col) < 0)
    .height
)

print(f"Negatif değer sayısı: {negative_count}")

In [ ]:
import polars as pl

# df_finalprice oluşturma
df_finalprice = (
    df
    .filter(
        pl.col("tender_finalpriceUsd").is_not_null()
    )
)

print("df_finalprice boyutu:", df_finalprice.shape)

In [ ]:
# tender_finalpriceUsd boş olan kayıtları ayırma
#tender_finalpriceUsd -> ihalenin gerçekleşen sonuç fiyatı


df_price_missing = (
    df
    .filter(
        pl.col("tender_finalpriceUsd").is_null()
    )
)


df_price_missing.shape

In [ ]:
# tender_finalpriceUsd dolu olan kayıtları alma

df_finalprice = (
    df
    .filter(
        pl.col("tender_finalpriceUsd").is_not_null()
    )
)

df_finalprice.shape

In [ ]:
df_finalprice.shape

## 🧹 6. Data Preprocessing / Veri Ön İşleme

This section cleans the dataset and prepares it for feature engineering.

Bu bölümde veri temizlenmekte ve özellik mühendisliği için hazır hale getirilmektedir.

## Gereksiz Değişkenlerin Çıkarılması

Gerçekleşen ihale fiyatının (`tender_finalpriceUsd`) tahmin edilmesi amacıyla veri setindeki değişkenler incelenmiştir. Modelin yalnızca ihale gerçekleşmeden önce erişilebilecek bilgiler ile tahmin yapabilmesi hedeflenmiştir.

Bu kapsamda;

- Benzersiz kimlik değişkenleri,
- Kurum ve teklif sahiplerine ait doğrudan tanımlayıcı bilgiler,
- İletişim ve adres bilgileri,
- Yüksek kardinaliteye sahip metin değişkenleri,
- Orijinal para birimindeki fiyat değişkenleri (USD karşılığı bulunan kolonlar tercih edildiği için),
- İhale sonucu açıklandıktan sonra oluşan ve veri sızıntısına neden olabilecek değişkenler

modelleme veri setinden çıkarılmıştır.

Fiyat değişkenlerinde farklı para birimlerinden kaynaklanabilecek tutarsızlıkları önlemek amacıyla USD bazlı değişkenler kullanılmıştır.

In [ ]:
# Veri setindeki toplam kolon sayısını görüntüleme

print("Toplam kolon sayısı:", df_finalprice.width)

print("\nKolon listesi:")
for i, col in enumerate(df_finalprice.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
# Tek değer içeren kolonları belirleme
# Identifying columns with a single value

constant_analysis = (
    df_finalprice
    .select([
        pl.col(col).n_unique().alias(col)
        for col in  df_finalprice.columns
    ])
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["unique_count"]
    )
    .sort("unique_count")
)

constant_analysis

In [ ]:
# Identify constant columns
# These columns have only one unique value across all observations
# and do not contribute to model learning, so they can be removed.

constant_cols = (
    constant_analysis
    .filter(
        pl.col("unique_count") == 1
    )
)

constant_cols

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

column = "filter_ok"


# ============================================================
# KPI HESAPLAMA
# ============================================================

total = df_finalprice.height
null_count = df_finalprice[column].null_count()
unique = df_finalprice[column].n_unique()


dist = (
    df_finalprice
    .group_by(column)
    .len()
    .rename({"len":"count"})
)


dist_dict = dict(zip(dist[column], dist["count"]))


true_count = dist_dict.get(True, 0)
false_count = dist_dict.get(False, 0)


true_pct = true_count / total * 100
false_pct = false_count / total * 100



# ============================================================
# GÖRSEL
# ============================================================

fig = plt.figure(figsize=(12,7))


fig.suptitle(
    "filter_ok True / False Dağılım Analizi",
    fontsize=18,
    fontweight="bold"
)



# ============================================================
# KPI KUTULARI
# ============================================================

kpis = [
    ("Toplam Kayıt", f"{total:,}"),
    ("True Kayıt", f"{true_count:,}"),
    ("False Kayıt", f"{false_count:,}"),
    ("Eksik Kayıt", f"{null_count:,}")
]


for i,(title,value) in enumerate(kpis):

    fig.text(
        0.12+i*0.23,
        0.82,
        f"{title}\n\n{value}",
        ha="center",
        fontsize=12,
        fontweight="bold",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            edgecolor="black"
        )
    )



# ============================================================
# PASTA GRAFİĞİ
# ============================================================

ax1 = fig.add_axes([0.15,0.15,0.35,0.5])


labels = ["True","False"]
values = [true_count,false_count]


ax1.pie(
    values,
    labels=labels,
    autopct="%1.2f%%",
    startangle=90,
    textprops={"fontsize":12}
)


ax1.set_title(
    "filter_ok Oranı",
    fontweight="bold"
)



# ============================================================
# BAR GRAFİK
# ============================================================

ax2 = fig.add_axes([0.6,0.2,0.3,0.4])


bars = ax2.bar(
    labels,
    values
)


for bar,value in zip(bars,values):

    ax2.text(
        bar.get_x()+bar.get_width()/2,
        value,
        f"{value:,}",
        ha="center",
        va="bottom",
        fontweight="bold"
    )


ax2.set_title(
    "Kayıt Sayısı",
    fontweight="bold"
)

ax2.set_ylabel("Adet")


plt.show()

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

df_false = df_finalprice.filter(
    pl.col("filter_ok")==False
)

# KPI
kpis = [
    ("False",f"{df_false.height:,}"),
    ("Oran",f"%{df_false.height/df_finalprice.height*100:.2f}"),
    ("Eksik Bid",f"{df_false['lot_bidscount'].null_count():,}"),
    ("Eksik Est.",f"{df_false['tender_estimatedprice'].null_count():,}")
]


missing = {
    "Bid":df_false["lot_bidscount"].null_count(),
    "Estimated":df_false["tender_estimatedprice"].null_count(),
    "Submission":df_false["submission_period"].null_count(),
    "Decision":df_false["decision_period"].null_count()
}


invalid = {
    "Fiyat":df_false.filter(pl.col("tender_finalpriceUsd")<=0).height,
    "Süre":df_false.filter(pl.col("submission_period")<0).height,
    "Lot":df_false.filter(pl.col("lot_bidscount")<=0).height,
    "Decision":df_false.filter(pl.col("decision_period")<0).height
}



fig=plt.figure(figsize=(12,6))
fig.suptitle("FILTER_OK=False Analizi",fontweight="bold")


# KPI
ax=fig.add_axes([.05,.75,.9,.15])
ax.axis("off")

for i,(t,v) in enumerate(kpis):
    x=i*.25
    ax.add_patch(plt.Rectangle((x+.02,.1),.21,.8,fill=False))
    ax.text(x+.125,.65,t,ha="center",fontweight="bold")
    ax.text(x+.125,.3,v,ha="center",fontsize=14,fontweight="bold")


# Eksik
ax1=fig.add_axes([.1,.25,.35,.35])
bars=ax1.bar(missing.keys(),missing.values())
ax1.set_title("Eksik Veri")
ax1.tick_params(axis="x",rotation=30)


for b,v in zip(bars,missing.values()):
    ax1.text(b.get_x()+b.get_width()/2,b.get_height(),f"{v:,}",ha="center")


# Geçersizlik
ax2=fig.add_axes([.55,.25,.35,.35])
bars=ax2.bar(invalid.keys(),invalid.values())
ax2.set_title("Geçersizlik")

for b,v in zip(bars,invalid.values()):
    ax2.text(
        b.get_x()+b.get_width()/2,
        b.get_height(),
        "Yok" if v==0 else str(v),
        ha="center"
    )

plt.show()

In [ ]:
# Sadece geçerli kayıtları al


print("Eski veri boyutu:", df_finalprice.shape)

df_finalprice= (
    df_finalprice
    .filter(
        pl.col("filter_ok") == True
    )
)

print("Yeni veri boyutu:", df_finalprice.shape)

In [ ]:
print( df_finalprice.shape)

 FILTER_OK kontrolü sonucunda yalnızca geçerli olarak işaretlenen
 (filter_ok = True) kayıtlar veri setinde tutulmuştur.

 filter_ok = False olan kayıtlar; eksik kritik alanlar veya
 veri kalite kontrollerinde başarısız olan gözlemler nedeniyle
 modelleme aşamasından çıkarılmıştır.

 Böylece model eğitimi, yalnızca veri kalitesi doğrulanmış
 kayıtlar üzerinden gerçekleştirilmiştir.

In [ ]:
# Modelleme amacıyla çıkarılacak kolonlar

drop_cols = [


    # Sabit veya model için anlamlı varyasyon içermeyen kolonlar
    "cancellation_reason",       # İhalenin iptal edilme nedeni.
    "lot_validbidscount",        # Geçerli teklif veren firma sayısı.
    "filter_framework",          # Veri filtreleme aşamasında kullanılan çerçeve anlaşması filtresi.
    "filter_buyer",              # Alıcı filtreleme işlemi için kullanılan yardımcı kolon.
    "filter_cancelled",          # İptal edilen ihaleleri filtrelemek için kullanılan yardımcı değişken.
    "filter_opentender",         # Açık ihale filtreleme bilgisi.
    "filter_losingbids",         # Kaybeden teklif bilgilerini filtrelemek için kullanılan yardımcı kolon.



    # Benzersiz kimlik kolonları
    "persistent_id",        # Veri kaynağındaki kalıcı benzersiz kayıt kimliği
    "tender_id",            # İhale benzersiz kimliği, tahmin için anlamlı özellik içermez
    "buyer_id",             # Alıcı kurum kimliği
    "buyer_masterid",       # Alıcı kurum ana kimliği
    "bidder_id",            # Teklif veren firma kimliği
    "bidder_masterid",      # Teklif veren firma ana kimliği
    "lot_id",               # İhale kalemi benzersiz kimliği
    "bid_id",               # Teklif benzersiz kimliği


    # Yüksek kardinaliteli metin kolonları
    "tender_title",         # İhale başlığı, çok fazla benzersiz değer içerir
    "lot_title",            # Lot başlığı


    # Kurum bilgileri ve iletişim alanları
    "buyer_name",           # Kurum adı, yüksek kardinaliteye sahip kategorik değişken
    "buyer_email",          # İletişim bilgisi, fiyat tahmini için anlamlı değil
    "buyer_phone",          # İletişim bilgisi
    "buyer_contactName",    # Kurum iletişim kişisi bilgisi
    "buyer_url",            # Kurum web adresi
    "buyer_street",         # Kurum açık adres bilgisi
    "buyer_postcode",       # Posta kodu bilgisi


    # Teklif veren firma bilgileri
    "bidder_name",          # Firma adı, yüksek kardinalite ve genelleme problemi oluşturabilir
    "bidder_email",         # Firma iletişim bilgisi
    "bidder_phone",         # Firma iletişim bilgisi
    "bidder_contactName",   # Firma iletişim kişisi
    "bidder_url",           # Firma web adresi
    "bidder_street",        # Firma açık adres bilgisi
    "bidder_postcode",      # Firma posta kodu bilgisi


    # Orijinal para birimindeki fiyat kolonları
    # USD karşılığı kullanıldığı için tekrar eden bilgiler çıkarılmıştır
    "tender_estimatedprice", # Tahmini fiyat (orijinal para birimi)
    "tender_finalprice",     # Gerçekleşen fiyat (orijinal para birimi), target olarak USD kullanılıyor
    "lot_estimatedprice",    # Lot tahmini fiyatı (orijinal para birimi)
    "bid_price",             # Teklif fiyatı (orijinal para birimi)


    # İhale sonucu oluştuktan sonra bilinen bilgiler
    # Veri sızıntısını (data leakage) önlemek için çıkarıldı

    "bid_issubcontracted",           # Alt yüklenici bilgisi
    "bid_subcontractedproportion",   # Alt yüklenici oranı
    "bid_isconsortium",              # Konsorsiyum bilgisi


    # İhale sonucu açıklandıktan sonra oluşan değişkenler
    # Modelin ihale öncesi tahmin yapabilmesi için çıkarılmıştır.

    "bid_priceUsd",                 # Teklif fiyatı, gerçekleşen fiyatla doğrudan ilişkili olduğu için veri sızıntısı oluşturabilir
    "bid_iswinning",                # Kazanan teklif bilgisi yalnızca ihale sonucu belirlendiği için model girdisi olarak kullanılamaz.
                                    # Gerçek hayatta ihale öncesinde bilinmediği için model girdisi olarak kullanılamaz.



    # Kaynak ve URL bilgileri
    "source",                              # Veri kaynağı bilgisi
    "notice_url",                          # İhale ilan bağlantısı
    "tender_publications_lastcontractawardurl", # Sözleşme ilan bağlantısı


    # Hazır filtreleme kolonları
    # Veri analizi için kullanılır, model özelliği olarak kullanılmaz

    "filter_bidder",          # Teklif veren filtre bilgisi
    "filter_year",            # Yıl filtre bilgisi
    "filter_ok",              # Veri kontrol filtre bilgisi, const kolon

    # Para birimi bilgisi
    "currency"               # USD dönüşümlü fiyatlar kullanıldığı ve Almanya verisi olduğu için çıkarıldı
]

# Listede bulunan ve dataframe içinde mevcut olan kolonları seçme
drop_cols_existing = [
    col for col in drop_cols
    if col in df_finalprice.columns
]

# Kolonları silme
df_finalprice =  df_finalprice.drop(drop_cols_existing)

# Silinen kolonları ve kalan kolon sayısını kontrol etme
print("Silinen kolon sayısı:", len(drop_cols_existing))
print("\nSilinen kolonlar:")
print(drop_cols_existing)

print("\nKalan kolon sayısı:", len(df_finalprice.columns))

In [ ]:
print(df_finalprice.shape)

In [ ]:
df_finalprice.columns

In [ ]:
# df_finalprice - Eksik Veri Analizi

import polars as pl
import matplotlib.pyplot as plt


pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)


missing_plot = (
    df_finalprice
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count")
            / df_finalprice.height
            * 100
        ).alias("missing_percentage")
    )
    .filter(
        pl.col("missing_count") > 0
    )
    .sort(
        "missing_percentage",
        descending=True
    )
)


print("=" * 80)
print("df_finalprice - Eksik Veri Analizi")
print("=" * 80)

print(missing_plot)


columns = (
    missing_plot["column"]
    .to_list()
)

missing_values = (
    missing_plot["missing_percentage"]
    .to_list()
)


plt.figure(
    figsize=(12, 7)
)



bars = plt.barh(
    columns,
    missing_values
)


plt.axvline(
    x=50,
    linestyle="--",
    linewidth=1.5,
    label="%50 Eksik Veri Eşiği"
)


plt.title(
    "df_finalprice - Eksik Veri Oranları",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel(
    "Eksik Veri Oranı (%)",
    fontsize=10
)

plt.ylabel(
    "Kolonlar",
    fontsize=10
)


plt.yticks(
    fontsize=7
)


plt.xticks(
    fontsize=8
)


plt.gca().invert_yaxis()


for bar, value in zip(
    bars,
    missing_values
):

    plt.text(
        value + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=7
    )


plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)


plt.legend(
    frameon=False,
    fontsize=8
)



plt.tight_layout(
    pad=1.0
)

plt.show()

In [ ]:
df_finalprice.group_by("tender_isawarded").agg(
    pl.len().alias("count")
)

In [ ]:
# =====================================================
# tender_finalpriceUsd tahmini için çıkarılacak kolonlar
# Tarih kolonları korunmuştur.
# =====================================================


drop_features_missing = [

    # Çok yüksek eksik oranına sahip kolonlar
    "lot_updateddurationdays",          # Lot güncelleme süresi, %99.97 eksik olduğu için çıkarılıyor.
    "tender_contractsignaturedate",     # Sözleşme imzalanma tarihi, çok yüksek eksiklik oranı nedeniyle çıkarılıyor. %99.86
    "tender_cancellationdate",          # İhale iptal tarihi, çoğu kayıtta bulunmadığı için çıkarılıyor. %98.46
    "corr_tax_haven",                   # Vergi cenneti risk göstergesi, %98 üzeri eksik olduğu için çıkarılıyor.

    "lot_estimatedpriceUsd",            # Lot seviyesinde tahmini fiyat, yüksek eksiklik oranına sahip ve benzer bilgi mevcut.
    "lot_smebidscount",                 # KOBİ teklif sayısı, %78 eksik olduğu için çıkarılıyor.
    "lot_electronicbidscount",          # Elektronik teklif sayısı, %70 üzeri eksik olduğu için çıkarılıyor.
    "lot_row_nr",                       # Veri sıra numarası, tahmin için bilgi taşımayan teknik kolon.

    "tender_isawarded",                 # İhalenin sonuçlandırılıp sonuçlandırılmadığı bilgisi.
                                        # Target sonrası oluşan sonuç bilgisi olduğu için
                                        # veri sızıntısı riski nedeniyle çıkarıldı.

    # Teknik / ID kolonları
    "bidder_sourceid_type",             # Teklif veren ID tipi, genelleme sağlamadığı için çıkarılıyor.
    "bidder_extra_source_id",           # Teklif veren benzersiz ID bilgisi, ezberleme riskini artırabilir.
    "buyer_sourceid_type",              # Alıcı ID tipi, fiyat tahminine doğrudan katkısı düşük.
    "buyer_extra_source_id",            # Alıcı ek ID bilgisi, teknik kolon.


]


df_finalprice = (
    df_finalprice
    .drop(drop_features_missing)
)


print("Önceki boyut:", df_finalprice.shape)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)



missing_percentage_model = (
    df_finalprice
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count") / df_finalprice.height * 100
        ).alias("missing_percentage")
    )
    .sort(
        "missing_percentage",
        descending=True
    )
)

display(missing_percentage_model)



columns = missing_percentage_model["column"].to_list()
percentages = missing_percentage_model["missing_percentage"].to_list()


plt.figure(figsize=(16, 8))

bars = plt.bar(
    columns,
    percentages
)

for bar, value in zip(bars, percentages):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=90
    )


plt.xticks(
    rotation=90,
    fontsize=8
)

plt.ylabel("Eksik Veri Oranı (%)")
plt.xlabel("Kolonlar")
plt.title("df_model Kalan Kolonlarda Eksik Veri Oranı")

plt.tight_layout()
plt.show()

## BIDDER_NUTS ANALİZ


In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# bidder_nuts  ANALİZ DASHBOARDU

column = "bidder_nuts"

# 1. Eksik değerleri Unknown olarak ele al


df_bidder_nuts = (
    df_finalprice
    .with_columns(
        pl.col(column)
        .fill_null("Unknown")
        .alias(column)
    )
)


# 2. Temel istatistikler

total_count = df_finalprice.height

missing_count = (
    df_finalprice
    .select(pl.col(column).null_count())
    .item()
)

unique_count = (
    df_bidder_nuts
    .select(pl.col(column).n_unique())
    .item()
)

missing_percentage = (
    missing_count / total_count * 100
)

# 3. Bölge bazında fiyat analizi


price_analysis = (
    df_bidder_nuts
    .group_by(column)
    .agg([
        pl.len().alias("lot_count"),

        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_finalprice"),

        pl.col("tender_finalpriceUsd")
        .mean()
        .alias("mean_finalprice")
    ])
    .sort(
        "lot_count",
        descending=True
    )
)

# 4. İlk 15 bölge


top_regions = (
    price_analysis
    .head(15)
)

# DASHBOARD

fig = plt.figure(
    figsize=(17, 11)
)


fig.suptitle(
    "bidder_nuts - Bölgesel Dağılım ve Final Fiyat Analizi",
    fontsize=21,
    fontweight="bold",
    y=0.98
)

# KPI ALANI

ax_kpi = fig.add_axes(
    [0.04, 0.78, 0.92, 0.14]
)

ax_kpi.axis("off")


kpis = [

    (
        "Toplam Kayıt",
        f"{total_count:,}"
    ),

    (
        "Eksik Kayıt",
        f"{missing_count:,}"
    ),

    (
        "Eksik Oranı",
        f"{missing_percentage:.2f}%"
    ),

    (
        "Unique Bölge",
        f"{unique_count:,}"
    ),

]

for i, (title, value) in enumerate(kpis):

    x = i / len(kpis)

    width = 0.21

    rect = plt.Rectangle(
        (x + 0.01, 0.05),
        width,
        0.85,
        fill=False,
        linewidth=1.5
    )

    ax_kpi.add_patch(rect)

    ax_kpi.text(
        x + width / 2 + 0.01,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax_kpi.text(
        x + width / 2 + 0.01,
        0.32,
        value,
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold"
    )

# GRAFİK 1
# EN ÇOK KULLANILAN 15 BÖLGE

ax1 = fig.add_axes(
    [0.06, 0.43, 0.40, 0.30]
)


regions = (
    top_regions
    .sort("lot_count")
)


categories = regions[column].to_list()
counts = regions["lot_count"].to_list()


colors_1 = [
    "#E45756" if x == "Unknown"
    else "#4C78A8"
    for x in categories
]


bars = ax1.barh(
    categories,
    counts,
    color=colors_1,
    edgecolor="black",
    linewidth=0.4
)


ax1.set_title(
    "En Çok Gözlem Bulunan 15 NUTS Bölgesi",
    fontsize=13,
    fontweight="bold"
)


ax1.set_xlabel(
    "Lot Sayısı"
)


ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    counts
):

    ax1.text(
        bar.get_width() + max(counts) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=8
    )


ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)


# GRAFİK 2
# MEDYAN FINAL PRICE


ax2 = fig.add_axes(
    [0.54, 0.43, 0.40, 0.30]
)


regions_price = (
    top_regions
    .sort("median_finalprice")
)


price_categories = (
    regions_price[column]
    .to_list()
)

median_prices = (
    regions_price["median_finalprice"]
    .to_list()
)


colors_2 = [
    "#E45756" if x == "Unknown"
    else "#59A14F"
    for x in price_categories
]


bars = ax2.barh(
    price_categories,
    median_prices,
    color=colors_2,
    edgecolor="black",
    linewidth=0.4
)


ax2.set_title(
    "Bölgelere Göre Medyan Final Fiyat",
    fontsize=13,
    fontweight="bold"
)


ax2.set_xlabel(
    "Medyan Final Fiyat ($)"
)


ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    median_prices
):

    ax2.text(
        bar.get_width() + max(median_prices) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"${value:,.0f}",
        va="center",
        fontsize=8
    )


ax2.ticklabel_format(
    style="plain",
    axis="x"
)


ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)



# GRAFİK 3
# GÖZLEM SAYISI - MEDYAN FİYAT İLİŞKİSİ


ax3 = fig.add_axes(
    [0.18, 0.08, 0.64, 0.27]
)


scatter_data = (
    price_analysis
    .filter(
        pl.col("lot_count") >= 100
    )
)


x = scatter_data["lot_count"].to_numpy()
y = scatter_data["median_finalprice"].to_numpy()
labels = scatter_data[column].to_list()


normal_mask = np.array(
    [label != "Unknown" for label in labels]
)

unknown_mask = ~normal_mask

ax3.scatter(
    x[normal_mask],
    y[normal_mask],
    s=45,
    alpha=0.55,
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.3,
    label="NUTS bölgeleri"
)

if unknown_mask.any():

    ax3.scatter(
        x[unknown_mask],
        y[unknown_mask],
        s=100,
        alpha=0.9,
        color="#E45756",
        edgecolor="black",
        linewidth=0.7,
        label="Unknown"
    )

top_labels = (
    scatter_data
    .sort(
        "lot_count",
        descending=True
    )
    .head(10)
)


for row in top_labels.iter_rows(
    named=True
):

    ax3.annotate(
        row[column],
        (
            row["lot_count"],
            row["median_finalprice"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=7
    )

ax3.set_title(
    "Gözlem Sayısı ve Medyan Final Fiyat İlişkisi",
    fontsize=13,
    fontweight="bold"
)

ax3.set_xlabel(
    "Lot Sayısı"
)

ax3.set_ylabel(
    "Medyan Final Fiyat ($)"
)

ax3.grid(
    linestyle="--",
    alpha=0.25
)

ax3.legend(
    frameon=False
)

ax3.ticklabel_format(
    style="plain",
    axis="y"
)

ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)


plt.show()

bidder_nuts kolonu, teklif verenin NUTS bölgesini göstermektedir.
 Yapılan analiz sonucunda 1.158 farklı kategori içerdiği görülmüştür.Bu yüksek kardinalite, kategorik değişken One-Hot Encoding ile işlendiğinde
modelde çok sayıda yeni özellik oluşmasına neden olabilir.

Ayrıca değişkenin %13.04 oranında eksik olması ve kategorilerin büyük
bölümünün düşük frekansta bulunması genelleme problemini artırmaktadır.

Her ne kadar bidder_nuts ile final fiyat arasında bölgesel farklılıklar
gözlemlense de, yüksek kardinalite ve model karmaşıklığı nedeniyle
modelleme aşamasında kullanılmamasına karar verilmiştir.


## BIDDER_COUNTRY ANALİZİ






In [ ]:
# bidder_country - GÖRSEL ANALİZ DASHBOARDU

import polars as pl
import matplotlib.pyplot as plt
import numpy as np



column = "bidder_country"


df_bidder_country = (
    df_finalprice
    .with_columns(
        pl.col(column)
        .fill_null("Unknown")
        .alias(column)
    )
)


total_count = df_finalprice.height

missing_count = (
    df_finalprice
    .select(
        pl.col(column).null_count()
    )
    .item()
)

unique_count = (
    df_bidder_country
    .select(
        pl.col(column).n_unique()
    )
    .item()
)

missing_percentage = (
    missing_count / total_count * 100
)

# 3. ÜLKE - FINAL PRICE ANALİZİ


price_analysis = (
    df_bidder_country
    .group_by(column)
    .agg([
        pl.len().alias("lot_count"),

        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_finalprice"),

        pl.col("tender_finalpriceUsd")
        .mean()
        .alias("mean_finalprice")
    ])
    .sort(
        "lot_count",
        descending=True
    )
)

# 4. EN ÇOK GÖZLEM BULUNAN 15 ÜLKE

top_countries = (
    price_analysis
    .head(15)
    .sort("lot_count")
)


categories = (
    top_countries[column]
    .to_list()
)

counts = (
    top_countries["lot_count"]
    .to_list()
)

# 5. TEK SAYFALIK DASHBOARD

fig = plt.figure(
    figsize=(15, 9)
)

fig.suptitle(
    "bidder_country - Ülke Dağılımı ve Final Fiyat Analizi",
    fontsize=20,
    fontweight="bold",
    y=0.98
)



# KPI ALANI

ax_kpi = fig.add_axes(
    [0.04, 0.78, 0.92, 0.14]
)

ax_kpi.axis("off")


kpis = [

    ("Toplam Kayıt", f"{total_count:,}"),

    ("Eksik Kayıt", f"{missing_count:,}"),

    ("Eksik Oranı", f"{missing_percentage:.2f}%"),

    ("Unique Ülke", f"{unique_count:,}")

]


for i, (title, value) in enumerate(kpis):

    x = i / len(kpis)

    width = 0.21

    rect = plt.Rectangle(
        (x + 0.01, 0.05),
        width,
        0.85,
        fill=False,
        linewidth=1.5
    )

    ax_kpi.add_patch(rect)

    ax_kpi.text(
        x + width / 2 + 0.01,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax_kpi.text(
        x + width / 2 + 0.01,
        0.32,
        value,
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold"
    )

# GRAFİK 1
# EN ÇOK KULLANILAN ÜLKELER

ax1 = fig.add_axes(
    [0.06, 0.43, 0.40, 0.30]
)


colors_1 = [
    "#E45756" if x == "Unknown"
    else "#4C78A8"
    for x in categories
]


bars = ax1.barh(
    categories,
    counts,
    color=colors_1,
    edgecolor="black",
    linewidth=0.4
)


ax1.set_title(
    "En Çok Gözlem Bulunan 15 Ülke",
    fontsize=13,
    fontweight="bold"
)

ax1.set_xlabel("Lot Sayısı")

ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    counts
):

    ax1.text(
        bar.get_width() + max(counts) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=8
    )


ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# GRAFİK 2
# ÜLKE BAZINDA MEDYAN FINAL PRICE

ax2 = fig.add_axes(
    [0.54, 0.43, 0.40, 0.30]
)


top_countries_price = (
    price_analysis
    .head(15)
    .sort("median_finalprice")
)


price_categories = (
    top_countries_price[column]
    .to_list()
)

median_prices = (
    top_countries_price["median_finalprice"]
    .to_list()
)


colors_2 = [
    "#E45756" if x == "Unknown"
    else "#59A14F"
    for x in price_categories
]


bars = ax2.barh(
    price_categories,
    median_prices,
    color=colors_2,
    edgecolor="black",
    linewidth=0.4
)


ax2.set_title(
    "Ülkelere Göre Medyan Final Fiyat",
    fontsize=13,
    fontweight="bold"
)

ax2.set_xlabel(
    "Medyan Final Fiyat ($)"
)

ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    median_prices
):

    ax2.text(
        bar.get_width() + max(median_prices) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"${value:,.0f}",
        va="center",
        fontsize=8
    )


ax2.ticklabel_format(
    style="plain",
    axis="x"
)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# GRAFİK 3
# GÖZLEM SAYISI - MEDYAN FİYAT İLİŞKİSİ


ax3 = fig.add_axes(
    [0.18, 0.08, 0.64, 0.27]
)


scatter_data = (
    price_analysis
    .filter(
        pl.col("lot_count") >= 100
    )
)


x = scatter_data["lot_count"].to_numpy()

y = scatter_data["median_finalprice"].to_numpy()

labels = scatter_data[column].to_list()


normal_mask = np.array(
    [
        label != "Unknown"
        for label in labels
    ]
)

unknown_mask = ~normal_mask


if normal_mask.any():

    ax3.scatter(
        x[normal_mask],
        y[normal_mask],
        s=70,
        alpha=0.65,
        color="#4C78A8",
        edgecolor="black",
        linewidth=0.4,
        label="Ülkeler"
    )


if unknown_mask.any():

    ax3.scatter(
        x[unknown_mask],
        y[unknown_mask],
        s=120,
        alpha=0.9,
        color="#E45756",
        edgecolor="black",
        linewidth=0.7,
        label="Unknown"
    )


top_labels = (
    scatter_data
    .sort(
        "lot_count",
        descending=True
    )
    .head(10)
)


for row in top_labels.iter_rows(
    named=True
):

    ax3.annotate(
        row[column],
        (
            row["lot_count"],
            row["median_finalprice"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )


ax3.set_title(
    "Gözlem Sayısı ve Medyan Final Fiyat İlişkisi",
    fontsize=13,
    fontweight="bold"
)

ax3.set_xlabel("Lot Sayısı")

ax3.set_ylabel(
    "Medyan Final Fiyat ($)"
)

ax3.grid(
    linestyle="--",
    alpha=0.25
)

ax3.legend(
    frameon=False
)

ax3.ticklabel_format(
    style="plain",
    axis="y"
)

ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)


plt.show()


bidder_country kolonu analiz sonucunda modelden çıkarılmıştır.

Kolonun eksik oranı %7.73 ile düşük olsa da gözlemlerin yaklaşık %90.53'ünün Almanya (DE) kategorisinde olması nedeniyle ciddi bir sınıf dengesizliği bulunmaktadır.

Ayrıca 105 farklı ülke kategorisi bulunmasına rağmen Almanya dışındaki ülkelerin büyük bölümü çok az sayıda gözlem içermektedir. Bu durum, One-Hot Encoding sonrasında bazı kategorilerin çok az temsil edilmesine ve modelin bu kategorilerden güvenilir genelleme yapamamasına neden olabilir.

bidder_country, ihale sonucunda teklif veren firmanın ülkesini ifade eder. Ancak çalışmanın amacı ihale sonuç fiyatını tahmin etmek olduğundan, bu değişkenin fiyat üzerindeki katkısının sınırlı olduğu değerlendirilmiştir.

İlgili değişken teorik olarak yurtiçi (Almanya) ve yurtdışı teklifleri şeklinde yeniden gruplandırılarak kullanılabilirdi. Ancak gözlemlerin yaklaşık %90'ının Almanya'ya ait olması nedeniyle bu dönüşüm de anlamlı bir ayrıştırma sağlamayacak, değişken büyük ölçüde tek bir kategori altında toplanacaktır. Bu nedenle böyle bir yeniden kodlamaya ihtiyaç duyulmamıştır.

Ayrıca buyer_country gibi ihale ve alıcı tarafını temsil eden daha doğrudan ve daha dengeli bir lokasyon bilgisi modelde bulunmaktadır.

Bu nedenlerle bidder_country, model karmaşıklığını azaltmak, gereksiz kategorik değişken kullanımını önlemek ve veri sızıntısı riski taşıyan yüklenici bilgilerini modele dahil etmemek amacıyla modelden çıkarılmıştır.


## BUYER_NUTS ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np


# buyer_nuts - GÖRSEL ANALİZ DASHBOARDU


column = "buyer_nuts"


# 1. ANALİZ DATAFRAME


if column in df_finalprice.columns:

    df_analysis = df_finalprice

elif column in df.columns:

    df_analysis = df

else:

    raise ValueError(
        f"'{column}' kolonu veri setinde bulunamadı."
    )

# 2. EKSİK DEĞERLERİ UNKNOWN OLARAK ELE AL


df_buyer_nuts = (
    df_analysis
    .with_columns(
        pl.col(column)
        .fill_null("Unknown")
        .alias(column)
    )
)

# 3. TEMEL İSTATİSTİKLER


total_count = df_analysis.height

missing_count = (
    df_analysis
    .select(
        pl.col(column).null_count()
    )
    .item()
)

unique_count = (
    df_buyer_nuts
    .select(
        pl.col(column).n_unique()
    )
    .item()
)

missing_percentage = (
    missing_count / total_count * 100
)


# 4. BUYER NUTS - FINAL PRICE ANALİZİ


price_analysis = (
    df_buyer_nuts
    .group_by(column)
    .agg([

        pl.len()
        .alias("lot_count"),

        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_finalprice"),

        pl.col("tender_finalpriceUsd")
        .mean()
        .alias("mean_finalprice")

    ])
    .sort(
        "lot_count",
        descending=True
    )
)


# 5. EN ÇOK GÖZLEM BULUNAN 15 BÖLGE


top_regions = (
    price_analysis
    .head(15)
    .sort("lot_count")
)


categories = (
    top_regions[column]
    .to_list()
)

counts = (
    top_regions["lot_count"]
    .to_list()
)


# 6. TEK SAYFALIK DASHBOARD

fig = plt.figure(
    figsize=(15, 9)
)


fig.suptitle(
    "buyer_nuts - Bölgesel Dağılım ve Final Fiyat Analizi",
    fontsize=20,
    fontweight="bold",
    y=0.98
)


# KPI ALANI


ax_kpi = fig.add_axes(
    [0.04, 0.78, 0.92, 0.14]
)

ax_kpi.axis("off")


kpis = [

    (
        "Toplam Kayıt",
        f"{total_count:,}"
    ),

    (
        "Eksik Kayıt",
        f"{missing_count:,}"
    ),

    (
        "Eksik Oranı",
        f"{missing_percentage:.2f}%"
    ),

    (
        "Unique NUTS",
        f"{unique_count:,}"
    )

]


for i, (title, value) in enumerate(kpis):

    x = i / len(kpis)

    width = 0.21


    rect = plt.Rectangle(
        (x + 0.01, 0.05),
        width,
        0.85,
        fill=False,
        linewidth=1.5
    )

    ax_kpi.add_patch(rect)


    ax_kpi.text(
        x + width / 2 + 0.01,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )


    ax_kpi.text(
        x + width / 2 + 0.01,
        0.32,
        value,
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold"
    )


# GRAFİK 1
# EN ÇOK KULLANILAN 15 BUYER NUTS


ax1 = fig.add_axes(
    [0.06, 0.43, 0.40, 0.30]
)


colors_1 = [

    "#E45756"
    if x == "Unknown"
    else "#4C78A8"

    for x in categories

]


bars = ax1.barh(
    categories,
    counts,
    color=colors_1,
    edgecolor="black",
    linewidth=0.4
)


ax1.set_title(
    "En Çok Gözlem Bulunan 15 Buyer NUTS Bölgesi",
    fontsize=13,
    fontweight="bold"
)


ax1.set_xlabel(
    "Lot Sayısı"
)


ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    counts
):

    ax1.text(
        bar.get_width()
        + max(counts) * 0.01,

        bar.get_y()
        + bar.get_height() / 2,

        f"{value:,}",

        va="center",
        fontsize=8
    )


ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)


# GRAFİK 2
# BUYER NUTS - MEDYAN FINAL PRICE


ax2 = fig.add_axes(
    [0.54, 0.43, 0.40, 0.30]
)


top_regions_price = (
    price_analysis
    .head(15)
    .sort("median_finalprice")
)


price_categories = (
    top_regions_price[column]
    .to_list()
)

median_prices = (
    top_regions_price["median_finalprice"]
    .to_list()
)


colors_2 = [

    "#E45756"
    if x == "Unknown"
    else "#59A14F"

    for x in price_categories

]


bars = ax2.barh(
    price_categories,
    median_prices,
    color=colors_2,
    edgecolor="black",
    linewidth=0.4
)


ax2.set_title(
    "Buyer NUTS Bölgelerine Göre Medyan Final Fiyat",
    fontsize=13,
    fontweight="bold"
)


ax2.set_xlabel(
    "Medyan Final Fiyat ($)"
)


ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    median_prices
):

    ax2.text(
        bar.get_width()
        + max(median_prices) * 0.01,

        bar.get_y()
        + bar.get_height() / 2,

        f"${value:,.0f}",

        va="center",
        fontsize=8
    )


ax2.ticklabel_format(
    style="plain",
    axis="x"
)


ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)


# GRAFİK 3
# GÖZLEM SAYISI - MEDYAN FİYAT İLİŞKİSİ


ax3 = fig.add_axes(
    [0.18, 0.08, 0.64, 0.27]
)


# En az 100 gözleme sahip bölgeler

scatter_data = (
    price_analysis
    .filter(
        pl.col("lot_count") >= 100
    )
)


x = (
    scatter_data["lot_count"]
    .to_numpy()
)

y = (
    scatter_data["median_finalprice"]
    .to_numpy()
)

labels = (
    scatter_data[column]
    .to_list()
)


normal_mask = np.array(
    [
        label != "Unknown"
        for label in labels
    ]
)


unknown_mask = ~normal_mask


# Normal bölgeler

if normal_mask.any():

    ax3.scatter(
        x[normal_mask],
        y[normal_mask],
        s=70,
        alpha=0.65,
        color="#4C78A8",
        edgecolor="black",
        linewidth=0.4,
        label="NUTS Bölgeleri"
    )


# Unknown

if unknown_mask.any():

    ax3.scatter(
        x[unknown_mask],
        y[unknown_mask],
        s=120,
        alpha=0.9,
        color="#E45756",
        edgecolor="black",
        linewidth=0.7,
        label="Unknown"
    )


# En fazla gözleme sahip ilk 10 bölgeyi etiketle

top_labels = (
    scatter_data
    .sort(
        "lot_count",
        descending=True
    )
    .head(10)
)


for row in top_labels.iter_rows(
    named=True
):

    ax3.annotate(
        row[column],

        (
            row["lot_count"],
            row["median_finalprice"]
        ),

        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )


ax3.set_title(
    "Gözlem Sayısı ve Medyan Final Fiyat İlişkisi",
    fontsize=13,
    fontweight="bold"
)


ax3.set_xlabel(
    "Lot Sayısı"
)


ax3.set_ylabel(
    "Medyan Final Fiyat ($)"
)


ax3.grid(
    linestyle="--",
    alpha=0.25
)


ax3.legend(
    frameon=False
)


ax3.ticklabel_format(
    style="plain",
    axis="y"
)


ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)


plt.show()

buyer_nuts kolonu analiz sonucunda modelden çıkarılmıştır.

Kolonun eksik oranı %3.07 ile düşük olmasına rağmen 552 farklı NUTS
bölgesi içermesi nedeniyle yüksek cardinality problemine sahiptir.

Bölgesel bazda final fiyatlarda farklılıklar gözlemlense de, birçok
bölgede gözlem sayısının düşük olması nedeniyle hesaplanan medyan ve
ortalama fiyat değerleri güvenilir olmayabilir. Özellikle düşük
gözlem sayısına sahip bölgelerde yüksek fiyat değerlerinin görülmesi,
bölgesel fiyat analizinde outlier etkisinin bulunduğunu göstermektedir.

Ayrıca buyer_nuts bilgisinin buyer_country ve diğer alıcı lokasyon
bilgileriyle örtüşen bölgesel bilgi taşıması nedeniyle modele ek bir
katkısının sınırlı olabileceği değerlendirilmiştir.

552 kategorinin One-Hot Encoding ile modele dahil edilmesi modelin
boyutunu ve karmaşıklığını artıracak, düşük frekanslı kategoriler ise
modelin genelleme kabiliyetini olumsuz etkileyebilecektir.

Bu nedenlerle modelin daha sade, daha az karmaşık ve daha
genellenebilir olması amacıyla buyer_nuts kolonu modelden çıkarılmıştır.

In [ ]:
# tender_finalpriceUsd tahmini için çıkarılacak kolonlar
# Tarih kolonları korunmuştur.


drop_features_missing = [

    # Çok yüksek missing oranı / düşük katkı / leakage riski

    "tender_estimatedpriceUsd",              # Tahmini ihale fiyatı; %74.98 eksik.
    "tender_isjointprocurement",             # Ortak ihale bilgisi; %41.25 eksik ve fiyat tahminine katkısı sınırlı.
    "tender_addressofimplementation_nuts",   # Uygulama adresi; %40.08 eksik, yüksek cardinality nedeniyle genelleme problemi oluşturabilir.


    # Teklif veren (bidder) lokasyon ve kimlik bilgileri

    "bidder_postcode_e",                     # Teklif veren posta kodu; %25.45 eksik, çok detaylı lokasyon bilgisi içerdiği için çıkarıldı.
    "bidder_nuts",                           # Teklif veren bölge bilgisi; %13.04 eksik, benzer bölge bilgileri mevcut olduğu için çıkarıldı.
    "bidder_nuts_e_clean",                   # Temizlenmiş teklif veren bölge bilgisi; bidder_nuts ile aynı bilgiyi taşıdığı için çıkarıldı.
    "bidder_city",                           # Teklif veren şehir bilgisi; yüksek cardinality ve düşük genelleme nedeniyle çıkarıldı.
    "bidder_city_e",                         # Teklif veren şehir kodu; tekrar eden lokasyon bilgisi olduğu için çıkarıldı.

    "bidder_country",                       # Teklif veren ülke bilgisi; buyer_country bilgisi yeterli görüldüğü için çıkarıldı.


    "bidder_country_e",                      # Teklif veren ülke kodu; tekrar eden bilgi içerdiği için çıkarıldı.
    "bidder_nuts_0",                         # NUTS seviye 0 (ülke seviyesi)
    "bidder_nuts_1",                         # NUTS seviye 1 (bölge seviyesi)
    "bidder_nuts_2",                         # NUTS seviye 2 (alt bölge seviyesi)
    "bidder_nuts_3",                         # NUTS seviye 3 (daha detaylı bölge seviyesi)



    # Teklif ve lot seviyesindeki kolonlar

    "bid_digiwhist_price",                   # Teklif fiyat kalite göstergesi; %21.12 eksik ve final fiyat ile leakage riski oluşturabilir.


    # Lokasyon tekrarları

    "buyer_nuts",                            # Alıcı bölge bilgisi; %3.07 eksik, buyer_country ve buyer_nuts_0 ile benzer bilgi içeriyor.
    "buyer_nuts_e_clean",                    # Temizlenmiş alıcı bölge bilgisi; buyer_nuts ile aynı bilgiyi tekrar ediyor.
    "buyer_postcode_e",                      # Alıcı posta kodu; düşük katkı ve fazla detay içerdiği için çıkarıldı.

]

# Gereksiz ve yüksek eksik oranına sahip kolonların çıkarılması
df_finalprice = df_finalprice.drop(drop_features_missing)

# Kalan kolonları kontrol etme
print("Kalan kolon sayısı:", len(df_finalprice.columns))

df_finalprice.head()

In [ ]:
import polars as pl


# df_finalprice - Tüm Kolonların Missing Veri Analizi

# Bu analiz ile:
# - Eksik değeri olan kolonlar
# - Tam dolu kolonlar
# birlikte incelenir.
#
# Çıktılar:
# column               -> Kolon adı
# missing_count        -> Eksik değer sayısı
# missing_percentage   -> Eksik değer yüzdesi



missing_all = (

    df_finalprice

    .null_count()


    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )


    .with_columns(

        (
            pl.col("missing_count") /
            df_finalprice.height * 100
        )
        .alias("missing_percentage")

    )


    .sort(
        "missing_percentage",
        descending=True
    )

)


# Tüm tabloyu görüntüleme ayarları

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)



print("="*100)
print("df_finalprice - Tüm Kolonların Eksik Veri Analizi")
print("="*100)


print(missing_all)



total_columns = df_finalprice.width


missing_columns = (
    missing_all
    .filter(
        pl.col("missing_count") > 0
    )
    .height
)


complete_columns = (
    missing_all
    .filter(
        pl.col("missing_count") == 0
    )
    .height
)



print("\n" + "="*100)

print("Toplam kolon sayısı       :", total_columns)

print("Eksik veri içeren kolon  :", missing_columns)

print("Eksiksiz kolon sayısı     :", complete_columns)

print("="*100)




# print(missing_all.to_pandas().to_string())

In [ ]:
# tender_finalpriceUsd tahmini için düşük katkılı kolonlar
# Tarih ve temel fiyat/rekabet kolonları korunmuştur.



drop_features_final = [

    "buyer_city",                 # Alıcı şehir bilgisi; yüksek detay içerdiği için modelin ezberleme riskini artırabilir.

    "buyer_city_e",               # Alıcı şehir kodu; buyer_city ile aynı bilgiyi tekrar ettiği için çıkarıldı.

    "buyer_country_e",            # Alıcı ülke kodu; buyer_country mevcut olduğu için tekrar eden bilgi olarak değerlendirildi.


    "buyer_nuts_0",               # Alıcı ülke seviyesinde NUTS bölge bilgisi; buyer_country ile benzer bilgi içerdiği için çıkarıldı.

    "buyer_nuts_1",               # Alıcı bölge seviyesi bilgisi; fiyat tahminine katkısı düşük ve lokasyon detayını artırıyor.

    "buyer_nuts_2",               # Alıcı alt bölge bilgisi; yüksek detay içerdiği için genelleme problemi oluşturabilir.

    "buyer_nuts_3",               # Alıcı en detaylı bölge bilgisi; yüksek cardinality nedeniyle çıkarıldı.


    "buyer_buyertype",            # Alıcı türü bilgisi; buyer_type ile benzer bilgi içerdiği için tekrar eden kolon olarak çıkarıldı.


]

# Düşük katkılı kolonların çıkarılması


df_finalprice = (
    df_finalprice
    .drop(drop_features_final)
)

# Boyut kontrolü

print("Yeni dataframe boyutu:")
print(df_finalprice.shape)

## LOT_STATUS ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np



# lot_status - GÖRSEL ANALİZ DASHBOARDU

column = "lot_status"


# 1. EKSİK DEĞERLERİ UNKNOWN OLARAK ELE AL

df_lot_status = (
    df_finalprice
    .with_columns(
        pl.col(column)
        .fill_null("Unknown")
        .alias(column)
    )
)

# 2. TEMEL İSTATİSTİKLER

total_count = df_finalprice.height

missing_count = (
    df_finalprice
    .select(
        pl.col(column).null_count()
    )
    .item()
)

unique_count = (
    df_lot_status
    .select(
        pl.col(column).n_unique()
    )
    .item()
)

missing_percentage = (
    missing_count / total_count * 100
)


# 3. LOT STATUS - FINAL PRICE ANALİZİ

price_analysis = (
    df_lot_status
    .group_by(column)
    .agg([

        pl.len()
        .alias("lot_count"),

        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_finalprice"),

        pl.col("tender_finalpriceUsd")
        .mean()
        .alias("mean_finalprice")

    ])
    .sort(
        "lot_count",
        descending=True
    )
)

# 4. KATEGORİLER

top_status = (
    price_analysis
    .head(15)
    .sort("lot_count")
)


categories = (
    top_status[column]
    .to_list()
)

counts = (
    top_status["lot_count"]
    .to_list()
)


# 5. TEK SAYFALIK DASHBOARD

fig = plt.figure(
    figsize=(15, 9)
)


fig.suptitle(
    "lot_status - Dağılım ve Final Fiyat Analizi",
    fontsize=20,
    fontweight="bold",
    y=0.98
)

# KPI ALANI

ax_kpi = fig.add_axes(
    [0.04, 0.78, 0.92, 0.14]
)

ax_kpi.axis("off")


kpis = [

    (
        "Toplam Kayıt",
        f"{total_count:,}"
    ),

    (
        "Eksik Kayıt",
        f"{missing_count:,}"
    ),

    (
        "Eksik Oranı",
        f"{missing_percentage:.2f}%"
    ),

    (
        "Unique Status",
        f"{unique_count:,}"
    )

]


for i, (title, value) in enumerate(kpis):

    x = i / len(kpis)

    width = 0.21


    rect = plt.Rectangle(
        (x + 0.01, 0.05),
        width,
        0.85,
        fill=False,
        linewidth=1.5
    )

    ax_kpi.add_patch(rect)


    ax_kpi.text(
        x + width / 2 + 0.01,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )


    ax_kpi.text(
        x + width / 2 + 0.01,
        0.32,
        value,
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold"
    )

# GRAFİK 1
# STATUS DAĞILIMI

ax1 = fig.add_axes(
    [0.06, 0.43, 0.40, 0.30]
)

colors_1 = [

    "#E45756"
    if x == "Unknown"
    else "#4C78A8"

    for x in categories

]

bars = ax1.barh(
    categories,
    counts,
    color=colors_1,
    edgecolor="black",
    linewidth=0.4
)


ax1.set_title(
    "Lot Status Dağılımı",
    fontsize=13,
    fontweight="bold"
)

ax1.set_xlabel(
    "Lot Sayısı"
)

ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

for bar, value in zip(
    bars,
    counts
):

    ax1.text(
        bar.get_width()
        + max(counts) * 0.01,

        bar.get_y()
        + bar.get_height() / 2,

        f"{value:,}",

        va="center",
        fontsize=8
    )


ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)


# GRAFİK 2
# STATUS - MEDYAN FINAL PRICE

ax2 = fig.add_axes(
    [0.54, 0.43, 0.40, 0.30]
)


status_price = (
    price_analysis
    .sort("median_finalprice")
)


price_categories = (
    status_price[column]
    .to_list()
)

median_prices = (
    status_price["median_finalprice"]
    .to_list()
)


colors_2 = [

    "#E45756"
    if x == "Unknown"
    else "#59A14F"

    for x in price_categories

]

bars = ax2.barh(
    price_categories,
    median_prices,
    color=colors_2,
    edgecolor="black",
    linewidth=0.4
)

ax2.set_title(
    "Lot Status'a Göre Medyan Final Fiyat",
    fontsize=13,
    fontweight="bold"
)

ax2.set_xlabel(
    "Medyan Final Fiyat ($)"
)

ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

for bar, value in zip(
    bars,
    median_prices
):

    ax2.text(
        bar.get_width()
        + max(median_prices) * 0.01,

        bar.get_y()
        + bar.get_height() / 2,

        f"${value:,.0f}",

        va="center",
        fontsize=8
    )


ax2.ticklabel_format(
    style="plain",
    axis="x"
)


ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# GRAFİK 3
# GÖZLEM SAYISI - MEDYAN FINAL PRICE

ax3 = fig.add_axes(
    [0.18, 0.08, 0.64, 0.27]
)


scatter_data = (
    price_analysis
    .filter(
        pl.col("lot_count") >= 10
    )
)


x = (
    scatter_data["lot_count"]
    .to_numpy()
)

y = (
    scatter_data["median_finalprice"]
    .to_numpy()
)

labels = (
    scatter_data[column]
    .to_list()
)


normal_mask = np.array(
    [
        label != "Unknown"
        for label in labels
    ]
)


unknown_mask = ~normal_mask


# Normal statuslar

if normal_mask.any():

    ax3.scatter(
        x[normal_mask],
        y[normal_mask],
        s=90,
        alpha=0.70,
        color="#4C78A8",
        edgecolor="black",
        linewidth=0.4,
        label="Lot Status"
    )


# Unknown

if unknown_mask.any():

    ax3.scatter(
        x[unknown_mask],
        y[unknown_mask],
        s=130,
        alpha=0.9,
        color="#E45756",
        edgecolor="black",
        linewidth=0.7,
        label="Unknown"
    )


# Tüm kategorileri etiketle

for row in scatter_data.iter_rows(
    named=True
):

    ax3.annotate(
        row[column],

        (
            row["lot_count"],
            row["median_finalprice"]
        ),

        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )


ax3.set_title(
    "Lot Status: Gözlem Sayısı ve Medyan Final Fiyat İlişkisi",
    fontsize=13,
    fontweight="bold"
)


ax3.set_xlabel(
    "Lot Sayısı"
)


ax3.set_ylabel(
    "Medyan Final Fiyat ($)"
)


ax3.grid(
    linestyle="--",
    alpha=0.25
)


ax3.legend(
    frameon=False
)


ax3.ticklabel_format(
    style="plain",
    axis="both"
)


ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

plt.show()

lot_status kolonu analiz sonucunda modelden çıkarılmamış ve modele dahil edilmiştir. Değişkenin veri kalitesi oldukça yüksektir; 319.036 kayıt içerisinde yalnızca 7 adet eksik değer bulunmakta olup eksik veri oranı ihmal edilebilir düzeydedir. Ayrıca değişken yalnızca 6 farklı kategori içerdiğinden One-Hot Encoding sonrasında model karmaşıklığında önemli bir artışa neden olmamaktadır. Yapılan analizlerde lot_status kategorileri arasında tender_finalpriceUsd medyan değerlerinin belirgin şekilde farklılaştığı görülmüştür. Örneğin, PREAWARDED durumundaki lotların medyan final fiyatı yaklaşık 1.009.864 USD iken AWARDED durumundaki lotlarda bu değer yaklaşık 521.713 USD, PREPARED durumundaki lotlarda ise yaklaşık 421.986 USD olarak hesaplanmıştır. Bu farklılıklar, değişkenin hedef değişken üzerinde anlamlı bir ayrıştırıcı bilgi taşıdığını ve model açısından açıklayıcı güce sahip olabileceğini göstermektedir.

Bu nedenle lot_status kolonu modelden çıkarılmamış, eksik olan çok az sayıdaki kayıt uygun şekilde ele alındıktan sonra kategorik değişken olarak One-Hot Encoding uygulanarak modelleme sürecine dahil edilmiştir. Ancak bu değişkenin ihale sürecinin ilerleyen aşamalarında oluşan bir bilgiyi temsil etmesi nedeniyle, model performansına katkısı ayrıca değerlendirilmiş ve olası veri sızıntısı (data leakage) riski de sonuçların yorumlanması sırasında dikkate alınmıştır.


In [ ]:
import polars as pl


# df_finalprice - Tüm Kolonların Missing Veri Analizi

# Bu analiz ile:
# - Eksik değeri olan kolonlar
# - Tam dolu kolonlar
# birlikte incelenir.
#
# Çıktılar:
# column               -> Kolon adı
# missing_count        -> Eksik değer sayısı
# missing_percentage   -> Eksik değer yüzdesi



missing_all = (

    df_finalprice

    .null_count()

    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )

    .with_columns(

        (
            pl.col("missing_count") /
            df_finalprice.height * 100
        )
        .alias("missing_percentage")

    )

    .sort(
        "missing_percentage",
        descending=True
    )
)

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)



print("="*100)
print("df_finalprice - Tüm Kolonların Eksik Veri Analizi")
print("="*100)


print(missing_all)

total_columns = df_finalprice.width

missing_columns = (
    missing_all
    .filter(
        pl.col("missing_count") > 0
    )
    .height
)

complete_columns = (
    missing_all
    .filter(
        pl.col("missing_count") == 0
    )
    .height
)

print("\n" + "="*100)

print("Toplam kolon sayısı       :", total_columns)

print("Eksik veri içeren kolon  :", missing_columns)

print("Eksiksiz kolon sayısı     :", complete_columns)

print("="*100)



In [ ]:
drop_features = [

    "tender_description_length",                    # İhale açıklamasının karakter/kelime uzunluğu; fiyat tahminine katkısı sınırlı
    "lot_description_length",                       # Lot açıklamasının uzunluğu; fiyatla doğrudan ilişkisi zayıf
    "tender_personalrequirements_length",           # Kişisel gereksinimlerin uzunluğu; fiyat açısından düşük katkılı
    "tender_technicalrequirements_length",          # Teknik gereksinimlerin uzunluğu; fiyatı doğrudan temsil etmez.Teknik şartların kapsamı işin karmaşıklığını yansıtabilir;
                                                    #ancak sadece uzunluk bilgisi zayıf bir temsilcidir.
    "tender_economicrequirements_length",           # Ekonomik gereksinimlerin uzunluğu; fiyat tahmininde düşük katkılı
    "tender_corrections_count",                     # İhalede yapılan düzeltme sayısı; fiyat tahminine doğrudan katkısı sınırlı
]


df_finalprice = df_finalprice.drop(drop_features)

print("Kalan kolon sayısı:", len(df_finalprice.columns))

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# Risk kolonları + CRI

risk_cols = [
    "corr_proc",
    "corr_nocft",
    "corr_decp",
    "corr_subm",
    "corr_singleb",
    "corr_buyer_concentration",
    "cri",
    "tender_finalpriceUsd"
]

# 1. Eksik veri analizi

missing_analysis = (
    df_finalprice
    .select(risk_cols)
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count")
            / df_finalprice.height
            * 100
        ).alias("missing_percentage")
    )
    .sort("missing_percentage", descending=True)
)

print("=" * 80)
print("RİSK KOLONLARI - EKSİK VERİ ANALİZİ")
print("=" * 80)

display(missing_analysis)

# 2. Korelasyon matrisi

corr_matrix = (
    df_finalprice
    .select(risk_cols)
    .corr()
)

corr_values = corr_matrix.to_numpy()

# 3. Korelasyon matrisi görselleştirme


fig, ax = plt.subplots(figsize=(12, 9))

im = ax.imshow(
    corr_values,
    aspect="auto"
)

ax.set_xticks(range(len(risk_cols)))
ax.set_yticks(range(len(risk_cols)))

ax.set_xticklabels(
    risk_cols,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(risk_cols)

for i in range(len(risk_cols)):
    for j in range(len(risk_cols)):
        ax.text(
            j,
            i,
            f"{corr_values[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Pearson Korelasyon Katsayısı")

ax.set_title(
    "Risk Göstergeleri ve Gerçekleşen İhale Fiyatı Korelasyon Matrisi",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# Risk kolonları + CRI

risk_cols = [
    "corr_proc",
    "corr_nocft",
    "corr_decp",
    "corr_subm",
    "corr_singleb",
    "corr_buyer_concentration",
    "cri",
    "tender_finalpriceUsd"
]

# 1. Eksik veri analizi

missing_analysis = (
    df_finalprice
    .select(risk_cols)
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count") / df_finalprice.height * 100
        ).alias("missing_percentage")
    )
    .sort("missing_percentage", descending=True)
)

print("=" * 80)
print("RİSK KOLONLARI - EKSİK VERİ ANALİZİ")
print("=" * 80)

display(missing_analysis)

# 2. Korelasyon için eksik değerleri -1 ile doldur
# (Orijinal df_finalprice değişmez)

risk_fill_cols = [
    "corr_proc",
    "corr_nocft",
    "corr_decp",
    "corr_subm",
    "corr_singleb",
    "corr_buyer_concentration",
    "cri"
]

df_corr = (
    df_finalprice
    .with_columns(
        [
            pl.col(col).fill_null(-1)
            for col in risk_fill_cols
        ]
    )
    .select(risk_cols)
)

# 3. Korelasyon matrisi

corr_matrix = df_corr.corr()
corr_values = corr_matrix.to_numpy()

print("\n")
print("=" * 80)
print("PEARSON KORELASYON MATRİSİ")
print("=" * 80)

display(corr_matrix)

# 4. Korelasyon matrisi görselleştirme

fig, ax = plt.subplots(figsize=(12, 9))

im = ax.imshow(
    corr_values,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(range(len(risk_cols)))
ax.set_yticks(range(len(risk_cols)))

ax.set_xticklabels(
    risk_cols,
    rotation=45,
    ha="right",
    fontsize=10
)

ax.set_yticklabels(
    risk_cols,
    fontsize=10
)

for i in range(len(risk_cols)):
    for j in range(len(risk_cols)):
        value = corr_values[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=9,
            color="white" if abs(value) >= 0.50 else "black"
        )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label(
    "Pearson Korelasyon Katsayısı",
    fontsize=11
)

ax.set_title(
    "Risk Göstergeleri ve Gerçekleşen İhale Fiyatı\nPearson Korelasyon Matrisi",
    fontsize=15,
    fontweight="bold",
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np


# RİSK KOLONLARI

risk_cols = [
    "corr_proc",
    "corr_nocft",
    "corr_decp",
    "corr_subm",
    "corr_singleb",
    "corr_buyer_concentration",
    "cri",
]


target = "tender_finalpriceUsd"


# RİSK KOLONLARININ KISA İSİMLERİ

risk_labels = {
    "corr_proc": "Procedure Risk",
    "corr_nocft": "No Call Risk",
    "corr_decp": "Decision Risk",
    "corr_subm": "Submission Risk",
    "corr_singleb": "Single Bid Risk",
    "corr_buyer_concentration": "Buyer Concentration",
    "cri": "CRI",
}


# DASHBOARD 1
# VERİ KALİTESİ + RİSK DAĞILIMI

# Eksik veri oranları

missing_analysis = (
    df_finalprice
    .select(risk_cols)
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count")
            / df_finalprice.height
            * 100
        ).alias("missing_percentage")
    )
    .sort(
        "missing_percentage",
        descending=True
    )
)

# Risk kolonlarının ortalama değerleri

risk_means = (
    df_finalprice
    .select([
        pl.col(col).mean().alias(col)
        for col in risk_cols
    ])
)


mean_values = [
    risk_means[col][0]
    for col in risk_cols
]

# Risk kolonlarının 0 / 1 dağılımı

risk_counts = []

for col in risk_cols:

    count_1 = (
        df_finalprice
        .select(
            pl.col(col)
            .eq(1)
            .sum()
        )
        .item()
    )

    count_0 = (
        df_finalprice
        .select(
            pl.col(col)
            .eq(0)
            .sum()
        )
        .item()
    )

    risk_counts.append(
        (count_0, count_1)
    )

# DASHBOARD 1

fig = plt.figure(
    figsize=(16, 10)
)

fig.suptitle(
    "Risk Göstergeleri - Veri Kalitesi ve Dağılım Dashboardu",
    fontsize=20,
    fontweight="bold",
    y=0.98
)

# KPI ALANI

ax_kpi = fig.add_axes(
    [0.04, 0.82, 0.92, 0.12]
)

ax_kpi.axis("off")


total_count = df_finalprice.height

total_missing = (
    missing_analysis["missing_count"]
    .sum()
)

max_missing = (
    missing_analysis["missing_percentage"]
    .max()
)


kpis = [
    ("Toplam Kayıt", f"{total_count:,}"),
    ("Risk Kolonu", f"{len(risk_cols)}"),
    ("Toplam Eksik Hücre", f"{total_missing:,}"),
    ("En Yüksek Missing", f"{max_missing:.2f}%"),
]


for i, (title, value) in enumerate(kpis):

    x = i / len(kpis)

    width = 0.21

    rect = plt.Rectangle(
        (x + 0.01, 0.05),
        width,
        0.85,
        fill=False,
        linewidth=1.5
    )

    ax_kpi.add_patch(rect)

    ax_kpi.text(
        x + width / 2 + 0.01,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax_kpi.text(
        x + width / 2 + 0.01,
        0.30,
        value,
        ha="center",
        va="center",
        fontsize=17,
        fontweight="bold"
    )

# GRAFİK 1
# EKSİK VERİ ORANI

ax1 = fig.add_axes(
    [0.06, 0.45, 0.40, 0.28]
)


missing_cols = (
    missing_analysis["column"]
    .to_list()
)

missing_values = (
    missing_analysis["missing_percentage"]
    .to_list()
)


missing_display = [
    risk_labels[x]
    for x in missing_cols
]


bars = ax1.barh(
    missing_display,
    missing_values,
    color="#E45756",
    edgecolor="black",
    linewidth=0.4
)

ax1.set_title(
    "Risk Kolonları - Eksik Veri Oranı",
    fontsize=13,
    fontweight="bold"
)

ax1.set_xlabel(
    "Eksik Veri (%)"
)

ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

for bar, value in zip(
    bars,
    missing_values
):

    ax1.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center",
        fontsize=8
    )


ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# GRAFİK 2
# RİSK KOLONLARININ ORTALAMA DEĞERLERİ

ax2 = fig.add_axes(
    [0.54, 0.45, 0.40, 0.28]
)


display_labels = [
    risk_labels[x]
    for x in risk_cols
]


bars = ax2.barh(
    display_labels,
    mean_values,
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.4
)


ax2.set_title(
    "Risk Göstergelerinin Ortalama Değeri",
    fontsize=13,
    fontweight="bold"
)

ax2.set_xlabel(
    "Ortalama Risk Oranı / Değeri"
)

ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)


for bar, value in zip(
    bars,
    mean_values
):

    ax2.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        fontsize=8
    )


ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)


# GRAFİK 3
# 0 / 1 RİSK DAĞILIMI

ax3 = fig.add_axes(
    [0.18, 0.08, 0.64, 0.27]
)


x_positions = np.arange(len(risk_cols))

zero_values = [
    x[0]
    for x in risk_counts
]

one_values = [
    x[1]
    for x in risk_counts
]


ax3.bar(
    x_positions,
    zero_values,
    label="Risk = 0",
    color="#59A14F",
    edgecolor="black",
    linewidth=0.3
)

ax3.bar(
    x_positions,
    one_values,
    bottom=zero_values,
    label="Risk = 1",
    color="#E45756",
    edgecolor="black",
    linewidth=0.3
)

ax3.set_xticks(
    x_positions
)

ax3.set_xticklabels(
    display_labels,
    rotation=30,
    ha="right"
)


ax3.set_ylabel(
    "Gözlem Sayısı"
)

ax3.set_title(
    "Risk Göstergelerinin 0 / 1 Dağılımı",
    fontsize=13,
    fontweight="bold"
)

ax3.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax3.legend(
    frameon=False
)

ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)


plt.show()

## CORR_PROC ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


risk_col="corr_proc"
target="tender_finalpriceUsd"


# Veri
df = df_finalprice.with_columns(
    pl.col(risk_col).fill_null(0)
)


# KPI
kpis=[
    ("Toplam Kayıt",f"{df.height:,}"),
    ("Risk Değeri",f"{df[risk_col].n_unique():,}"),
    ("Eksik",f"{df_finalprice[risk_col].null_count():,}"),
    ("Riskli Kayıt",f"{df.filter(pl.col(risk_col)==1).height:,}")
]


# Analiz
dist=(
    df.group_by(risk_col)
    .len()
    .rename({"len":"count"})
)


price=(
    df.group_by(risk_col)
    .agg([
        pl.col(target).median().alias("median"),
        pl.col(target).mean().alias("mean")
    ])
)



# Figure
fig=plt.figure(figsize=(15,8))

fig.suptitle(
    f"{risk_col} Analizi",
    fontsize=18,
    fontweight="bold"
)


# KPI ALANI
ax=fig.add_axes([.04,.72,.92,.15])
ax.axis("off")


for i,(t,v) in enumerate(kpis):

    x=i*.25

    ax.add_patch(
        plt.Rectangle(
            (x+.02,.1),
            .21,.8,
            fill=False,
            linewidth=1.5
        )
    )

    ax.text(
        x+.125,.65,
        t,
        ha="center",
        fontweight="bold"
    )

    ax.text(
        x+.125,.3,
        v,
        ha="center",
        fontsize=16,
        fontweight="bold"
    )



# Risk pasta

ax1=fig.add_axes([.05,.15,.3,.4])

ax1.pie(
    dist["count"],
    labels=dist[risk_col].cast(pl.String),
    autopct="%1.1f%%"
)

ax1.set_title("Risk Dağılımı")



# Final fiyat

ax2=fig.add_axes([.45,.15,.35,.4])

x=range(len(price))
w=.35

ax2.bar(
    [i-w/2 for i in x],
    price["median"],
    w,
    label="Medyan"
)

ax2.bar(
    [i+w/2 for i in x],
    price["mean"],
    w,
    alpha=.6,
    label="Ortalama"
)

ax2.set_xticks(x)
ax2.set_xticklabels(
    price[risk_col].cast(pl.String)
)

ax2.set_title("Risk - Final Fiyat")
ax2.set_ylabel("$")
ax2.legend()
ax2.grid(axis="y",alpha=.3)


plt.tight_layout()
plt.show()

corr_proc kolonu eksik veri içermeyen ve düşük cardinality yapısına sahip bir risk göstergesidir. Kolon değerleri arasında final fiyat seviyelerinde farklılık gözlenmektedir. Özellikle orta risk seviyesinde medyan fiyat artışı görülmesi, prosedürel risk göstergesinin ihale maliyeti üzerinde bilgi taşıyabileceğini göstermektedir. Bu nedenle tender_finalpriceUsd tahmin modelinde korunmuştur.

## CORR_NOCFT ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


risk_col="corr_nocft"
target="tender_finalpriceUsd"


df=df_finalprice.with_columns(
    pl.col(risk_col).fill_null(0)
)


# KPI
kpis=[
    ("Toplam",f"{df.height:,}"),
    ("Unique",f"{df[risk_col].n_unique():,}"),
    ("Eksik",f"{df_finalprice[risk_col].null_count():,}"),
    ("Riskli",f"{df.filter(pl.col(risk_col)>0).height:,}")
]


# Analiz
dist=df.group_by(risk_col).len().rename({"len":"count"})

price=df.group_by(risk_col).agg([
    pl.col(target).median().alias("median"),
    pl.col(target).mean().alias("mean")
])


# Figure
fig=plt.figure(figsize=(16,8))
fig.suptitle(f"{risk_col} Analizi",fontsize=18,fontweight="bold")


# KPI
ax=fig.add_axes([.04,.72,.92,.15])
ax.axis("off")

for i,(t,v) in enumerate(kpis):
    x=i*.25
    ax.add_patch(
        plt.Rectangle((x+.02,.1),.21,.8,fill=False)
    )
    ax.text(x+.125,.65,t,ha="center",fontweight="bold")
    ax.text(x+.125,.3,v,ha="center",fontsize=16,fontweight="bold")



# Pasta
ax1=fig.add_axes([.05,.15,.25,.4])
ax1.pie(
    dist["count"],
    labels=dist[risk_col].cast(pl.String),
    autopct="%1.1f%%"
)
ax1.set_title("Risk Dağılımı")



# Final fiyat karşılaştırma

ax2 = fig.add_axes([.40,.15,.45,.4])

x = range(len(price))
w = 0.35

ax2.bar(
    [i-w/2 for i in x],
    price["median"],
    w,
    label="Medyan"
)

ax2.bar(
    [i+w/2 for i in x],
    price["mean"],
    w,
    label="Ortalama",
    alpha=.6
)

ax2.set_xticks(x)
ax2.set_xticklabels(
    price[risk_col].cast(pl.String)
)

ax2.set_title(
    "Risk - Final Fiyat",
    fontweight="bold"
)

ax2.set_ylabel("USD")
ax2.legend()
ax2.grid(axis="y",alpha=.3)


plt.show()

corr_nocft kolonu analiz edilmiştir.

Kolonda herhangi bir eksik veri bulunmamaktadır.
Gözlemlerin %73.6'sında normal ihale süreci görülürken,
%26.4'lük bölümde teklif çağrısı olmadan gerçekleşen süreçler bulunmaktadır.

corr_nocft değerlerine göre gerçekleşen fiyatlar incelendiğinde,
teklif çağrısı olmayan ihalelerde medyan final fiyatın daha düşük olduğu
gözlemlenmiştir.

Bu değişken, ihale prosedürünü temsil eden anlamlı bir risk göstergesi
olduğu ve fiyat oluşum sürecine dolaylı katkı sağlayabileceği için
modelden çıkarılmamıştır.

## CORR_DECP ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


column="corr_decp"


# Veri
df = df_finalprice.with_columns(
    pl.col(column).cast(pl.String).fill_null("Unknown")
)


# KPI
kpis=[
    ("Toplam",f"{df.height:,}"),
    ("Unique",f"{df[column].n_unique():,}"),
    ("Eksik",f"{df_finalprice[column].null_count():,}"),
    ("Unknown",f"{df.filter(pl.col(column)=="Unknown").height:,}")
]


# Analiz
dist=(
    df.group_by(column)
    .len()
    .rename({"len":"count"})
)

price=(
    df.group_by(column)
    .agg(
        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median")
    )
)



# Figure
fig=plt.figure(figsize=(15,8))

fig.suptitle(
    f"{column} Analizi",
    fontsize=18,
    fontweight="bold"
)


# KPI
ax=fig.add_axes([.04,.72,.92,.15])
ax.axis("off")

for i,(t,v) in enumerate(kpis):
    x=i*.25
    ax.add_patch(
        plt.Rectangle(
            (x+.02,.1),.21,.8,
            fill=False
        )
    )
    ax.text(x+.125,.65,t,ha="center",fontweight="bold")
    ax.text(x+.125,.3,v,ha="center",fontsize=16,fontweight="bold")



# Pasta
ax1=fig.add_axes([.05,.15,.35,.4])

ax1.pie(
    dist["count"],
    labels=dist[column],
    autopct="%1.1f%%"
)

ax1.set_title("Risk Dağılımı")



# Fiyat
ax2=fig.add_axes([.55,.15,.35,.4])

ax2.bar(
    price[column],
    price["median"]
)

ax2.set_title("Medyan Final Fiyat")
ax2.set_ylabel("$")


plt.show()

corr_decp kolonu analiz sonucunda modelde tutulmuştur.

Kolonda herhangi bir eksik veri bulunmamaktadır (%0.00).
Değişken iki farklı kategoriye sahiptir ve veri dağılımı yeterli seviyededir.

corr_decp = 1 olan ihalelerde medyan gerçekleşen fiyatın
corr_decp = 0 durumuna göre daha düşük olduğu görülmüştür.

Bu fark, karar süreci ile ilişkili risk göstergesinin ihale sonuç fiyatı
üzerinde bilgi taşıyabileceğini göstermektedir.

Her ne kadar kolon doğrudan fiyat bilgisini temsil etmese de,
ihale süreçlerindeki farklılıkları yansıtabileceği için
tender_finalpriceUsd tahmin modelinde tutulmasına karar verilmiştir.

## CORR_SUBM ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


column = "corr_subm"
target = "tender_finalpriceUsd"


# Veri
df = df_finalprice.with_columns(
    pl.col(column).cast(pl.String).fill_null("Unknown")
)


# KPI
kpis = [
    ("Toplam", f"{df.height:,}"),
    ("Unique", f"{df[column].n_unique():,}"),
    ("Eksik", f"{df_finalprice[column].null_count():,}"),
    ("Ort. Fiyat", f"${df[target].mean():,.0f}")
]


# Risk dağılımı
dist = (
    df.group_by(column)
    .len()
    .rename({"len":"count"})
)


# Target analizi
price = (
    df.group_by(column)
    .agg([
        pl.col(target).median().alias("median"),
        pl.col(target).mean().alias("mean")
    ])
    .sort("median")
)



# Figure
fig = plt.figure(figsize=(15,8))

fig.suptitle(
    f"{column} - {target} Analizi",
    fontsize=18,
    fontweight="bold"
)



# KPI
ax = fig.add_axes([.04,.72,.92,.15])
ax.axis("off")


for i,(t,v) in enumerate(kpis):

    x=i*.25

    ax.add_patch(
        plt.Rectangle(
            (x+.02,.1),.21,.8,
            fill=False,
            linewidth=1.5
        )
    )

    ax.text(
        x+.125,.65,t,
        ha="center",
        fontweight="bold"
    )

    ax.text(
        x+.125,.3,v,
        ha="center",
        fontsize=15,
        fontweight="bold"
    )



# Risk dağılımı

ax1 = fig.add_axes([.05,.15,.30,.4])

ax1.pie(
    dist["count"],
    labels=dist[column],
    autopct="%1.1f%%"
)

ax1.set_title("Risk Dağılımı")



# Target etkisi

ax2 = fig.add_axes([.45,.15,.35,.4])


x = range(len(price))
w=.35


ax2.bar(
    [i-w/2 for i in x],
    price["median"],
    w,
    label="Medyan"
)


ax2.bar(
    [i+w/2 for i in x],
    price["mean"],
    w,
    alpha=.6,
    label="Ortalama"
)


ax2.set_xticks(x)

ax2.set_xticklabels(
    price[column]
)


ax2.set_title(
    "Risk Değerine Göre Final Fiyat"
)

ax2.set_ylabel("$")

ax2.legend()

ax2.grid(
    axis="y",
    alpha=.3
)


plt.tight_layout()
plt.show()

corr_subm Analizi
Toplam kayıt: 319.036
Eksik veri: 104.472 (%32.75)
Eksik değerler modelleme öncesinde Unknown olarak değerlendirilecektir.
Dağılım

corr_subm = 1.0 → 105.030 kayıt

Unknown → 104.472 kayıt

corr_subm = 0.5 → 64.605 kayıt

corr_subm = 0.0 → 44.929 kayıt

Final Fiyat İlişkisi
En yüksek medyan fiyat: corr_subm = 0.5 ($635K)
corr_subm = 1.0 ve 0.0 gruplarında fiyat farkı sınırlıdır.
Unknown grubunun farklı fiyat davranışı göstermesi, eksik değerlerin bilgi taşıyabileceğini göstermektedir.
Modelleme Kararı
 corr_subm kolonu modelde tutulacaktır.

Tek başına güçlü fiyat belirleyici olmasa da ihale risk/süreç bilgisini temsil ettiği için modele katkı sağlayabilir.

## CORR_SINGLEB ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


column = "corr_singleb"
target = "tender_finalpriceUsd"


# Veri
df = df_finalprice.with_columns(
    pl.col(column)
    .cast(pl.String)
    .fill_null("Unknown")
)


# KPI
kpis=[
    ("Toplam",f"{df.height:,}"),
    ("Unique",f"{df[column].n_unique():,}"),
    ("Eksik",f"{df_finalprice[column].null_count():,}"),
    ("Ort. Fiyat",f"${df[target].mean():,.0f}")
]


# Risk dağılımı
dist=(
    df.group_by(column)
    .len()
    .rename({"len":"count"})
)


# Target analizi
price=(
    df.group_by(column)
    .agg([
        pl.len().alias("count"),
        pl.col(target).median().alias("median"),
        pl.col(target).mean().alias("mean")
    ])
    .sort("median")
)



# Figure
fig=plt.figure(figsize=(15,8))

fig.suptitle(
    f"{column} - {target} Analizi",
    fontsize=18,
    fontweight="bold"
)



# KPI
ax=fig.add_axes([.04,.72,.92,.15])
ax.axis("off")


for i,(t,v) in enumerate(kpis):

    x=i*.25

    ax.add_patch(
        plt.Rectangle(
            (x+.02,.1),
            .21,.8,
            fill=False,
            linewidth=1.5
        )
    )

    ax.text(
        x+.125,.65,
        t,
        ha="center",
        fontweight="bold"
    )

    ax.text(
        x+.125,.3,
        v,
        ha="center",
        fontsize=16,
        fontweight="bold"
    )



# Risk dağılımı

ax1=fig.add_axes([.05,.15,.3,.4])

ax1.pie(
    dist["count"],
    labels=dist[column],
    autopct="%1.1f%%"
)

ax1.set_title("Risk Dağılımı")



# Target etkisi

ax2=fig.add_axes([.45,.15,.35,.4])

x=range(len(price))
w=.35


ax2.bar(
    [i-w/2 for i in x],
    price["median"],
    w,
    label="Medyan"
)


ax2.bar(
    [i+w/2 for i in x],
    price["mean"],
    w,
    alpha=.6,
    label="Ortalama"
)


ax2.set_xticks(x)
ax2.set_xticklabels(
    price[column]
)


ax2.set_title(
    "Risk Seviyesine Göre Final Fiyat"
)

ax2.set_ylabel("$")

ax2.legend()
ax2.grid(axis="y",alpha=.3)



plt.tight_layout()
plt.show()

corr_singleb Analizi

- Toplam kayıt: 319.036
- Eksik veri: 70.952 (%22.24)
- Eksik değerler modelleme öncesinde Unknown olarak ele alınacaktır.
Dağılım
- corr_singleb = 0 → 206.986 kayıt
- Unknown → 70.952 kayıt
- corr_singleb = 1 → 41.098 kayıt
Final Fiyat İlişkisi
- Unknown grubunun medyan final fiyatı $834K ile diğer gruplardan belirgin şekilde yüksektir.
- corr_singleb = 0 ve corr_singleb = 1 gruplarında medyan fiyatlar birbirine yakındır:
  - 0 → 508K
  - 1 → 519K

- Unknown grubundaki yüksek fiyat farkı, eksik değerlerin rastgele olmadığını ve veri kaybının bilgi taşıyabileceğini göstermektedir.

Modelleme Kararı
corr_singleb kolonu modelde tutulacaktır.

- Teklif yoğunluğu ve rekabet yapısı ile ilişkili olduğu için gerçekleşen fiyat tahmininde katkı sağlayabilir.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# Risk Kolonları

risk_cols = [
    "corr_proc",
    "corr_nocft",
    "corr_decp",
    "corr_subm",
    "corr_singleb",
    "corr_buyer_concentration",
    "cri"
]

# Unique değer sayıları

summary = []

for col in risk_cols:

    unique_count = (
        df_finalprice
        .select(
            pl.col(col)
            .cast(pl.String)
            .fill_null("Unknown")
            .n_unique()
        )
        .item()
    )

    values = (
        df_finalprice
        .select(
            pl.col(col)
            .cast(pl.String)
            .fill_null("Unknown")
            .unique()
            .sort()
        )
        .to_series()
        .to_list()
    )

    summary.append(
        {
            "column": col,
            "unique_count": unique_count,
            "values": ", ".join(map(str, values))
        }
    )

summary_df = pl.DataFrame(summary)

display(summary_df)

# Görselleştirme

plt.figure(figsize=(10,5))

bars = plt.bar(
    summary_df["column"].to_list(),
    summary_df["unique_count"].to_list(),
    edgecolor="black"
)

plt.title(
    "Risk Kolonlarının Benzersiz Değer Sayıları",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Risk Kolonu")
plt.ylabel("Unique Değer Sayısı")

plt.xticks(rotation=30, ha="right")

for bar in bars:

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        str(int(bar.get_height())),
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


Risk Göstergeleri Analizi
Bu bölümde ihale risk göstergeleri incelenmiş ve gerçekleşen ihale fiyatı tahmin modelinde kullanılacak risk değişkenleri belirlenmiştir.

İncelenen risk kolonları:

corr_proc,
corr_nocft,
corr_decp,
corr_subm,
corr_singleb,
corr_buyer_concentration,
cri

Risk Değişkenlerinin Açıklaması
corr_proc
İhale prosedüründen kaynaklanan riskleri temsil eder.
İhale sürecinin rekabet ve şeffaflık açısından risk seviyesini gösterir.

corr_nocft
İhale çağrısı yapılmaması ile ilişkili riski ifade eder.
Açık rekabet ortamının olup olmadığını gösterir.

corr_decp
İhale karar sürecindeki riskleri temsil eder.

corr_subm
Teklif verme sürecindeki riskleri gösterir.
Teklif süresi ve katılım koşulları ile ilişkilidir.

corr_singleb
Tek teklifli ihale durumunu ifade eder.
Tek teklif bulunması, düşük rekabet ve fiyat oluşumu açısından risk göstergesidir.

corr_buyer_concentration
Alıcı kurumun belirli tedarikçilere yoğunlaşmasını gösteren sürekli risk skorudur.

cri
Diğer risk göstergelerinin birleşiminden oluşturulan toplam risk skorudur.
Diğer risk değişkenleriyle örtüşebileceği için model aşamasında kullanılmamıştır.

Modelde Kullanılacak Risk Özellikleri
Gerçekleşen ihale fiyatı (tender_finalpriceUsd) tahmin modelinde aşağıdaki risk değişkenleri kullanılacaktır:

corr_proc,
corr_nocft,
corr_decp,
corr_subm,
corr_singleb,
corr_buyer_concentration

Bu değişkenler, ihale sürecindeki rekabet, şeffaflık ve karar mekanizmalarını temsil ettiği için fiyat tahmininde bilgi sağlayabilecek özellikler olarak değerlendirilmiştir.

In [ ]:
# Risk kolonlarını kaldırma
drop_risk_cols = [
    "cri",

]

df_finalprice = df_finalprice.drop(drop_risk_cols)

# Kontrol
print("Silinen kolonlar:", drop_risk_cols)
print("Kalan kolon sayısı:", len(df_finalprice.columns))

 Risk göstergeleri analiz edilmiştir.

 Yapılan incelemeler sonucunda risk kolonları arasında
 modelleme açısından uygun olmayan veya ek bilgi sağlamayan
 CRI kolonu veri setinden çıkarılmıştır.

 Diğer risk göstergeleri, ihale özellikleri ve final fiyat
 üzerindeki etkileri değerlendirilerek modelleme sürecinde
 kullanılmak üzere korunmuştur.

## Gereksiz ve Modelleme Amacına Uygun Olmayan Değişkenlerin Çıkarılması

Bu çalışmada amaç, ihale süreci sonunda oluşan gerçekleşen fiyat değerini (`tender_finalpriceUsd`) tahmin etmektir. Bu nedenle modelleme aşamasında yalnızca gerçekleşen fiyat üzerinde etkili olabilecek, ihale süreci, rekabet düzeyi, teklif bilgileri, kurum özellikleri ve zaman bilgilerini temsil eden değişkenlerin kullanılması hedeflenmiştir.

Veri setinde bulunan benzersiz kimlik değişkenleri, yüksek kardinaliteye sahip açıklama/metin alanları, hedef değişkenle doğrudan ilişkili olmayan teknik kolonlar ve model tarafından öğrenme sürecine katkı sağlamayacak değişkenler veri setinden çıkarılmıştır.

Ayrıca hedef değişken olan `tender_finalpriceUsd` ile doğrudan bilgi sızıntısına (data leakage) neden olabilecek kolonlar kaldırılmıştır. Böylece modelin yalnızca ihale gerçekleşmeden önce elde edilebilecek bilgiler kullanılarak fiyat tahmini yapması sağlanmıştır.


## ⚙️ 7. Feature Engineering / Özellik Mühendisliği

This section creates new informative features to improve model performance.

Bu bölümde model performansını artırmak amacıyla yeni özellikler oluşturulmaktadır.

In [ ]:
for col, dtype in zip(df_finalprice.columns, df_finalprice.dtypes):
    if any(x in col.lower() for x in ["date", "time", "year"]):
        print(f"{col}: {dtype}")

In [ ]:
df_finalprice = df_finalprice.with_columns([
    pl.col("tender_awarddecisiondate")
      .str.to_datetime(strict=False)
      .alias("tender_awarddecisiondate"),

    pl.col("tender_publications_firstdcontractawarddate")
      .str.to_datetime(strict=False)
      .alias("tender_publications_firstdcontractawarddate"),

    pl.col("tender_publications_firstcallfortenderdate")
      .str.to_datetime(strict=False)
      .alias("tender_publications_firstcallfortenderdate"),

    pl.col("tender_year")
      .cast(pl.Int32, strict=False)
      .alias("tender_year")
])

In [ ]:
# İlan tarihinden karar tarihine kadar geçen gün sayısını hesaplama
# Bu değer ihale değerlendirme süresini temsil eder

df_finalprice = df_finalprice.with_columns(
    (
        pl.col("tender_awarddecisiondate")
        - pl.col("tender_publications_firstcallfortenderdate")
    )
    .dt.total_days()
    .alias("decision_duration_days")
)


In [ ]:
drop_date_cols = [
    "tender_awarddecisiondate",                        # İhale kararının verildiği tarih; tahmin anında bilinmeyen bilgi olabileceği için leakage riski taşır.
    "tender_publications_firstdcontractawarddate",     # Sözleşme ödülünün ilk yayınlandığı tarih; ihale sonucu hakkında bilgi içerdiği için leakage riski taşır.
    "tender_publications_firstcallfortenderdate"       # İhale çağrısının ilk yayınlandığı tarih; ham tarih yerine bundan türetilen zaman özellikleri kullanılabilir.
]

# Ham tarih kolonlarının veri setinden çıkarılması
df_finalprice = df_finalprice.drop(drop_date_cols)

print("Kalan kolon sayısı:", len(df_finalprice.columns))

In [ ]:
# İhale sürecinin toplam süresini hesaplama
# Teklif verme süresi (submission_period) ile
# karar verme süresinin (decision_period) toplamı alınarak
# ihale ilanından sonuçlanmasına kadar geçen toplam süre oluşturulur.

#df_finalprice = df_finalprice.with_columns(
   # (
    #    pl.col("submission_period") +
     #   pl.col("decision_period")
    #)
   # .alias("total_process_duration")
#)

In [ ]:
# İhalenin ilgili lotuna verilen toplam teklif sayısını ifade eder

print("="*70)
print("lot_bidscount İSTATİSTİKLERİ")
print("="*70)


# Temel istatistikler
display(
    df_finalprice
    .select(
        pl.col("lot_bidscount")
    )
    .describe()
)


# Unique değer sayısı
unique_count = (
    df_finalprice
    .select(
        pl.col("lot_bidscount")
        .n_unique()
    )
    .item()
)


# Eksik veri sayısı
missing_count = (
    df_finalprice
    .select(
        pl.col("lot_bidscount")
        .null_count()
    )
    .item()
)


# Toplam kayıt
total_count = df_finalprice.height


# Eksik oranı
missing_percentage = (
    missing_count / total_count * 100
)


print("\n")
print("="*70)
print("lot_bidscount EK ANALİZ")
print("="*70)

print(
    f"Unique değer sayısı : {unique_count:,}"
)

print(
    f"Eksik kayıt sayısı  : {missing_count:,}"
)

print(
    f"Eksik oranı         : %{missing_percentage:.2f}"
)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# lot_bidscount - FINAL PRICE ANALİZ DASHBOARDU

column = "lot_bidscount"


# 1. VERİ HAZIRLAMA

df_bid = df_finalprice.with_columns(
    pl.col(column)
    .fill_null(0)
    .cast(pl.Int32)
)

# 2. KPI BİLGİLERİ

kpi = (
    df_bid
    .select(
        pl.len().alias("total"),
        pl.col(column).null_count().alias("missing"),
        pl.col(column).mean().alias("mean"),
        pl.col(column).median().alias("median"),
        pl.col(column).max().alias("max")
    )
    .row(0)
)


total_count, missing_count, mean_bid, median_bid, max_bid = kpi

# 3. TEKLİF SAYISI DAĞILIMI

bid_distribution = (
    df_bid
    .group_by(column)
    .agg(
        pl.len().alias("count")
    )
    .sort(
        "count",
        descending=True
    )
    .head(15)
)

# 4. FINAL PRICE ANALİZİ

price_analysis = (
    df_bid
    .group_by(column)
    .agg(
        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_price"),

        pl.len()
        .alias("lot_count")
    )
    .filter(
        pl.col("lot_count") >= 50
    )
    .sort(
        "median_price",
        descending=True
    )
    .head(15)
)

# DASHBOARD

fig = plt.figure(figsize=(16,10))


fig.suptitle(
    "lot_bidscount - Teklif Sayısı ve Final Fiyat Analizi",
    fontsize=20,
    fontweight="bold"
)

# KPI ALANI

ax_kpi = fig.add_axes(
    [0.05,0.80,0.9,0.12]
)

ax_kpi.axis("off")


kpis = [
    ("Toplam Kayıt", f"{total_count:,}"),
    ("Eksik Değer", f"{missing_count:,}"),
    ("Ortalama Teklif", f"{mean_bid:.2f}"),
    ("Medyan Teklif", f"{median_bid:.0f}"),
    ("Maksimum Teklif", f"{max_bid:.0f}")
]


for i, (title,value) in enumerate(kpis):

    x = i * 0.19

    ax_kpi.text(
        x+0.09, 0.65,
        title,
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

    ax_kpi.text(
        x+0.09, 0.25,
        value,
        ha="center",
        fontsize=18,
        fontweight="bold"
    )

# GRAFİK 1 - TEKLİF SAYISI DAĞILIMI

ax1 = fig.add_axes(
    [0.06,0.43,0.38,0.27]
)


categories = bid_distribution[column].cast(pl.String).to_list()
counts = bid_distribution["count"].to_list()


bars = ax1.bar(
    categories,
    counts,
    color="#4C78A8",
    edgecolor="black"
)


ax1.set(
    title="En Yaygın Teklif Sayıları",
    xlabel="Teklif Sayısı",
    ylabel="Lot Sayısı"
)

ax1.tick_params(
    axis="x",
    rotation=45
)

ax1.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

# GRAFİK 2 - TEKLİF SAYISI / FİYAT

ax2 = fig.add_axes(
    [0.55,0.43,0.38,0.27]
)


price_categories = price_analysis[column].cast(pl.String).to_list()
median_prices = price_analysis["median_price"].to_list()


bars = ax2.barh(
    price_categories[::-1],
    median_prices[::-1],
    color="#59A14F",
    edgecolor="black"
)


ax2.set(
    title="Teklif Sayısına Göre Medyan Final Fiyat",
    xlabel="Medyan Final Fiyat ($)"
)


ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

# GRAFİK 3 - FİYAT ETKİSİ

ax3 = fig.add_axes(
    [0.20,0.08,0.60,0.25]
)


bars = ax3.barh(
    price_categories[::-1],
    median_prices[::-1],
    color="#E45756",
    edgecolor="black"
)


ax3.set(
    title="Teklif Sayısına Göre Medyan Final Fiyat Etkisi",
    xlabel="Medyan Final Fiyat ($)",
    ylabel="Teklif Sayısı"
)


ax3.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)


ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)


plt.tight_layout()
plt.show()

lot_bidscount değişkeni, ihaleye verilen toplam teklif sayısını ifade etmekte olup ihale rekabet seviyesini temsil eden önemli bir değişken olarak değerlendirilmiştir. Yapılan analizlerde değişkenin yüksek sağ çarpıklığa sahip olduğu ve bazı yüksek uç değerler içerdiği görülmüştür. Bu nedenle değişkenin ham haliyle modele verilmesi yerine, daha dengeli ve yorumlanabilir bir temsil oluşturulması planlanmıştır.

Teklif sayısından türetilen farklı değişkenler değerlendirilmiş, ancak model karmaşıklığını artırmamak ve gereksiz özellik kullanımını önlemek amacıyla yalnızca rekabet seviyesini temsil eden competition_level değişkeninin oluşturulmasına karar verilmiştir.

Oluşturulacak competition_level değişkeninde; 0 teklif verilen ihaleler "No Bid", 1-3 teklif aralığındaki ihaleler "Low Competition", 4-7 teklif aralığındaki ihaleler "Medium Competition", 8 ve üzeri teklif alan ihaleler ise "High Competition" olarak sınıflandırılacaktır.

Bu dönüşüm ile teklif sayısındaki aşırı değerlerin model üzerindeki etkisinin azaltılması, aynı zamanda ihale rekabet seviyesine ilişkin bilginin korunması amaçlanmaktadır. Böylece lot_bidscount değişkeni doğrudan modele verilmek yerine, rekabet yapısını daha iyi temsil eden kategorik bir değişken olarak modelleme sürecinde kullanılacaktır.


In [ ]:
# ============================================================
# Rekabet seviyesinin oluşturulması
#
# 0-3 teklif  : Low Competition
# 4-7 teklif  : Medium Competition
# 8+ teklif   : High Competition
# ============================================================

df_finalprice = df_finalprice.with_columns(
    pl.when(pl.col("lot_bidscount") <= 3)
    .then(pl.lit("Low Competition"))       # 0-3 teklif

    .when(pl.col("lot_bidscount") <= 7)
    .then(pl.lit("Medium Competition"))    # 4-7 teklif

    .otherwise(pl.lit("High Competition")) # 8+ teklif

    .alias("competition_level")
)


# ============================================================
# Oluşturulan kolonun kontrolü
# ============================================================

print("competition_level kolonu oluşturuldu.")

display(
    df_finalprice
    .select("competition_level")
    .group_by("competition_level")
    .len()
    .sort("len")
)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


col="competition_level"
target="tender_finalpriceUsd"


# Analiz

dist=(
    df_finalprice
    .group_by(col)
    .len()
    .rename({"len":"count"})
)


price=(
    df_finalprice
    .group_by(col)
    .agg([
        pl.len().alias("count"),
        pl.col(target).median().alias("median"),
        pl.col(target).mean().alias("mean")
    ])
)



# KPI

kpis=[
    ("Toplam Lot",f"{df_finalprice.height:,}"),
    ("Seviye",f"{df_finalprice[col].n_unique()}"),
    ("En Yoğun",dist.sort("count",descending=True)[col][0]),
    ("En Yüksek Fiyat",
     f"${price.sort('median',descending=True)['median'][0]:,.0f}")
]



# Figure

fig=plt.figure(figsize=(15,8))

fig.suptitle(
    "Competition Level Analizi",
    fontsize=18,
    fontweight="bold"
)



# KPI

ax=fig.add_axes([.04,.72,.92,.15])
ax.axis("off")


for i,(t,v) in enumerate(kpis):

    x=i*.25

    ax.add_patch(
        plt.Rectangle(
            (x+.02,.1),
            .21,.8,
            fill=False,
            linewidth=1.5
        )
    )

    ax.text(x+.125,.65,t,ha="center",fontweight="bold")
    ax.text(x+.125,.3,v,ha="center",fontsize=14,fontweight="bold")



# Dağılım

ax1=fig.add_axes([.07,.18,.35,.4])

ax1.pie(
    dist["count"],
    labels=dist[col],
    autopct="%1.1f%%"
)

ax1.set_title("Rekabet Dağılımı")



# Fiyat

ax2=fig.add_axes([.55,.18,.35,.4])

ax2.barh(
    price[col],
    price["median"]
)

ax2.set_title("Rekabet Seviyesine Göre Medyan Fiyat")
ax2.set_xlabel("$")


plt.tight_layout()
plt.show()

competition_level değişkeni, lot_bidscount değişkeninden türetilerek ihale rekabet seviyesini kategorik olarak temsil etmek amacıyla oluşturulmuştur. lot_bidscount değişkenindeki yüksek çarpıklık ve uç değerlerin etkisini azaltmak için teklif sayıları dört seviyeye ayrılmıştır: No Bid (0 teklif), Low Competition (1-3 teklif), Medium Competition (4-7 teklif) ve High Competition (8+ teklif).

Bu dönüşüm ile teklif sayısındaki değişkenlik azaltılarak rekabet bilgisinin korunması ve model tarafından daha anlamlı şekilde öğrenilmesi amaçlanmıştır. Yapılan analizlerde kategoriler arasında final fiyat medyanlarında farklılıklar gözlenmiş, bu nedenle competition_level modelde kullanılacak önemli bir özellik olarak değerlendirilmiştir.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Pearson Korelasyon Matrisi

numeric_cols = [
    "lot_bidscount",
    "tender_recordedbidscount",
    "submission_period",
    "decision_period",
    "decision_duration_days",
    "tender_finalpriceUsd"
]

missing = [c for c in numeric_cols if c not in df_finalprice.columns]

if missing:
    print("Eksik kolonlar:", missing)

else:

    corr = (
        df_finalprice
        .select(numeric_cols)
        .drop_nulls()
        .to_pandas()
        .corr(method="pearson")
    )

    display(corr)

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="RdYlBu_r",
        vmin=-1,
        vmax=1,
        center=0,
        square=True,
        linewidths=0.8,
        linecolor="white",
        annot_kws={"size":11, "weight":"bold"},
        cbar_kws={"label":"Pearson Correlation"}
    )

    plt.title(
        "Pearson Correlation Matrix",
        fontsize=16,
        fontweight="bold"
    )

    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.show()

Pearson korelasyon matrisi incelendiğinde, değişkenlerin büyük çoğunluğu arasında oldukça zayıf veya ihmal edilebilir düzeyde doğrusal bir ilişki olduğu görülmektedir. Öne çıkan tek anlamlı ilişki, submission_period (teklif sunma süresi) ile decision_duration_days (karar süresi) arasındaki 0.66 düzeyindeki pozitif ve orta-güçlü korelasyondur; bu durum teklif süreci uzayan ihalelerde karar süresinin de uzama eğiliminde olduğunu göstermektedir. Ayrıca decision_period ile decision_duration_days arasında da 0.31'lik pozitif bir ilişki mevcuttur. Öte yandan, ihale bedelini temsil eden tender_finalpriceUsd ile teklif sayılarını gösteren lot_bidscount ve tender_recordedbidscount değişkenlerinin diğer parametrelerle doğrusal bir ilişkisi bulunmamaktadır (-0.00 ile 0.08 arası). Sonuç olarak,0.66'lık ilişki modellerde olası bir çoklu doğrusal bağlantı (multicollinearity) riskine karşı dikkate alınmalı, doğrusal ilişkisi düşük çıkan değişkenler için ise doğrusal olmayan ilişki yapıları veya öznitelik mühendisliği (feature engineering) adımları değerlendirilmelidir.

## 8. Data Preparation and Feature Selection for Machine Learning / Makine Öğrenmesi İçin Veri Hazırlama ve Özellik Seçimi

This section focuses on preparing the dataset for machine learning models by analyzing available features, removing irrelevant or redundant columns, and selecting the most informative variables for prediction. Feature selection is performed by evaluating missing values, data quality, variable relevance, and their potential contribution to the prediction task. After the feature selection process, categorical variables are encoded, numerical variables are scaled, and the dataset is split into training and test sets.

Bu bölümde veri seti, makine öğrenmesi modellerinde kullanılmak üzere hazırlanmaktadır. İlk olarak mevcut kolonlar analiz edilerek gereksiz, yüksek eksik oranına sahip veya model performansına katkı sağlamayacağı değerlendirilen değişkenler belirlenerek veri setinden çıkarılmaktadır. Özellik seçimi sürecinde eksik veri oranları, değişken türleri, veri kalitesi ve hedef değişken ile ilişkileri dikkate alınmaktadır. Seçilen özellikler üzerinden kategorik değişkenlere kodlama işlemi uygulanmakta, sayısal değişkenler ölçeklendirilmekte ve veri seti eğitim ile test kümelerine ayrılarak modelleme aşamasına hazır hale getirilmektedir.

In [ ]:
print("Toplam kolon sayısı:", len(df_finalprice.columns))

df_finalprice.columns

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# decision_duration_days ANALİZİ


column = "decision_duration_days"

# 1. Eksik Veri Analizi

total_count = df_finalprice.height

missing_count = (
    df_finalprice
    .select(pl.col(column).null_count())
    .item()
)

missing_percentage = missing_count / total_count * 100

print("=" * 70)
print(f"{column} ANALİZİ")
print("=" * 70)
print(f"Toplam kayıt : {total_count:,}")
print(f"Eksik kayıt  : {missing_count:,}")
print(f"Eksik oranı  : %{missing_percentage:.2f}")

# 2. İstatistikler

display(
    df_finalprice
    .select(column)
    .describe()
)

# 3. Analiz DataFrame

df_analysis = (
    df_finalprice
    .select([column, "tender_finalpriceUsd"])
    .drop_nulls()
)

# 4. Süre Grupları

df_group = (
    df_analysis
    .with_columns(
        pl.when(pl.col(column) <= 30)
        .then(pl.lit("0-30"))

        .when(pl.col(column) <= 60)
        .then(pl.lit("31-60"))

        .when(pl.col(column) <= 90)
        .then(pl.lit("61-90"))

        .otherwise(pl.lit("90+"))
        .alias("duration_group")
    )
    .group_by("duration_group")
    .agg(
        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_price")
    )
)

# Scatter için örnekleme

sample = (
    df_analysis
    .sample(
        n=min(5000, df_analysis.height),
        seed=42
    )
)

# DASHBOARD

fig, axes = plt.subplots(
    2,
    2,
    figsize=(16,10)
)

fig.suptitle(
    "Decision Duration Days Analysis",
    fontsize=18,
    fontweight="bold"
)

# Grafik 1 - Histogram (Logaritmik Ölçekli)
duration_values = (
    df_analysis
    .filter(
        pl.col(column).is_not_null()
    )[column]
    .to_list()
)

# Tüm değerleri kullan (-6486 ile 3398 arası)
axes[0, 0].hist(
    duration_values,
    bins=80,
    color="#4E79A7",
    edgecolor="white"
)

axes[0, 0].set_yscale('log')

axes[0, 0].set_title(
    "Decision Duration Distribution (Log Scale)",
    fontsize=13,
    fontweight="bold"
)

axes[0, 0].set_xlabel(
    "Decision Duration (Days)"
)

axes[0, 0].set_ylabel(
    "Number of Tenders (Log Scale)"
)

axes[0, 0].grid(
    axis="both",
    alpha=0.25,
    which="both"
)

# Grafik 2 - Violin Plot

sns.violinplot(
    x=df_analysis[column].to_list(),
    ax=axes[0,1],
    color="#76B7B2",
    inner="quartile",
    linewidth=1.2
)

axes[0,1].set_title(
    "Decision Duration Distribution",
    fontsize=13,
    fontweight="bold"
)

axes[0,1].set_xlabel("Days")

# Grafik 3 - Medyan Fiyat
# String süre gruplarını doğru sırala

order = {
    "0-30":1,
    "31-60":2,
    "61-120":3,
    "121-180":4,
    "180+":5
}

df_group = (
    df_group
    .with_columns(
        pl.col("duration_group")
        .replace(order)
        .alias("order")
    )
    .sort("order")
)


# Grafik

axes[1,0].bar(
    df_group["duration_group"].to_list(),
    df_group["median_price"].to_list(),
    color="#F28E2B",
    edgecolor="black"
)

axes[1,0].set_title(
    "Median Final Price by Decision Duration"
)

axes[1,0].set_xlabel("Decision Duration Group")
axes[1,0].set_ylabel("Median Final Price (USD)")

# Grafik 4 - Scatter

axes[1,1].scatter(
    sample[column].to_list(),
    sample["tender_finalpriceUsd"].to_list(),
    color="#E15759",
    alpha=0.35,
    s=18
)

axes[1,1].set_title(
    "Decision Duration vs Final Price",
    fontsize=13,
    fontweight="bold"
)

axes[1,1].set_xlabel("Decision Duration (Days)")
axes[1,1].set_ylabel("Final Price (USD)")


# Genel

for ax in axes.flatten():

    ax.grid(
        linestyle="--",
        alpha=0.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

Decision Duration Days Analysis

Grafikler incelendiğinde; ihale karar sürelerinin çoğunlukla 0-500 gün arasında yoğunlaştığı, logaritmik histogram ve violin plot panellerinde -6486 güne varan eksi değerler ile 3000 günü aşan uç değerlerin yer aldığı görülmektedir. Karar süresi uzadıkça medyan ihale bedeli 425.000 USD'den 550.000 USD'ye doğru belirgin bir artış gösterirken, yüksek bütçeli ihaleler ağırlıklı olarak $0-500$ gün aralığında kümelenmektedir. Veri ön işleme aşamasında; karar süresi negatif olamayacağı için eksi değerler NaN olarak işaretlenecek, 3000 gün üzerindeki değerler ise gerçek ihale süreçlerinde karşılaşılabilecek olası durumları temsil ettiği için veri setinde olduğu gibi korunacaktır.

In [ ]:
drop_features = [
    "tender_biddeadline",                    # İhalenin teklif verme son tarihini ifade eder; ham tarih bilgisi yerine submission_period kullanılmaktadır.
    "lot_bidscount",                         # İlgili lota verilen toplam teklif sayısıdır; competition_level bu değişkenden türetildiği için ham değişken çıkarılır.
]

# Belirlenen kolonların veri setinden çıkarılması
df_finalprice = df_finalprice.drop(drop_features)

print("Silinen kolon sayısı:", len(drop_features))
print("Kalan kolon sayısı:", len(df_finalprice.columns))
print("\nKalan kolonlar:")
print(df_finalprice.columns)

In [ ]:
model_features = [

    "tender_proceduretype",          # Genel ihale prosedür türü (OPEN, RESTRICTED vb.)
    "tender_nationalproceduretype",  # Ulusal prosedür türü, daha detaylı süreç bilgisi
    "tender_supplytype",             # Tedarik türü (mal, hizmet, yapım)

    "tender_lotscount",              # İhaledeki toplam lot sayısı
    "tender_recordedbidscount",      # Verilen toplam teklif sayısı
    "competition_level",             # Teklif sayısından türetilen rekabet seviyesi
    "lot_status",                    # Bir ihale lotunun mevcut durumunu ifade eder.
    "tender_digiwhist_price",        # İhale için Digiwhist veri kaynağı tarafından sağlanan fiyat bilgisini ifade eder.

    "tender_isframeworkagreement",   # Çerçeve anlaşma bilgisi
    "tender_isdps",                  # Dinamik satın alma sistemi bilgisi
    "tender_selectionmethod",        # Kazanan teklif seçim yöntemi
    "tender_awardcriteria_count",    # Değerlendirme kriteri sayısı

    "tender_cpvs",                   # İhale konusu sektör/ürün CPV kodları
    "tender_maincpv",                # Ana CPV kategorisi

    "tender_iseufunded",             # AB fonu kullanımı bilgisi

    "buyer_country",                 # Alıcı kurumun ülkesi
    "buyer_mainactivities",          # Alıcının faaliyet alanı
    "buyer_type",                    # Alıcı kurum tipi
    "bidder_type",                   # Teklif veren firma tipi

    "tender_year",                   # İhalenin gerçekleştiği yıl
    "submission_period",             # Teklif verme süresi
    "decision_period",               # Karar süresi
    "decision_duration_days",        # İhale ilanının yayınlanmasından ihale kararının verilmesine kadar geçen süreyi gün cinsinden ifade eder.

    "corr_singleb",                  # Tek teklif riski göstergesi
    "corr_proc",                     # Prosedür riski göstergesi
    "corr_subm",                     # Teklif süreci risk göstergesi
    "corr_nocft",                    # İlan yapılmaması risk göstergesi
    "corr_decp",                     # Karar süreci risk göstergesi
    "corr_buyer_concentration",      # İhaleyi gerçekleştiren alıcı kurumun satın alma yoğunlaşması riskini ifade eden bir risk göstergesidir.

    "competition_level"              # İhaledeki rekabet seviyesini temsil eden türetilmiş kategorik değişkendir.
]


target = "tender_finalpriceUsd"       # Tahmin edilecek gerçekleşen ihale final fiyatı

## TENDER_NATIONALPROCEDURETYPE ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


col = "tender_nationalproceduretype"


# Veri hazırlama
df_analysis = (
    df_finalprice
    .with_columns(
        pl.col(col).cast(pl.String).fill_null("Unknown"),
        pl.col("tender_finalpriceUsd")
        .log1p()
        .alias("log_finalprice")
    )
)


# KPI
total_count = df_analysis.height
missing_count = df_finalprice[col].null_count()
missing_rate = missing_count / total_count * 100
unique_count = df_analysis[col].n_unique()


# Özet tablo
summary = (
    df_analysis
    .group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_price"),
        pl.col("log_finalprice")
        .median()
        .alias("median_log_price")
    )
    .with_columns(
        (pl.col("count") / total_count * 100)
        .round(2)
        .alias("percentage")
    )
    .sort("count", descending=True)
    .head(10)
)


# Grafik verileri
dist = summary.sort("percentage")
price = summary.sort("median_price")
log_price = summary.sort("median_log_price")


# Dashboard
fig, axes = plt.subplots(
    2,
    2,
    figsize=(16,10)
)

fig.suptitle(
    "Tender National Procedure Type Analysis",
    fontsize=18,
    fontweight="bold"
)



# 1 - Dağılım

bars = axes[0,0].barh(
    dist[col].to_list(),
    dist["percentage"].to_list(),
    color="#4E79A7"
)

axes[0,0].set_title(
    "Category Distribution (%)",
    fontweight="bold"
)

axes[0,0].set_xlabel("%")


for b,v in zip(
    bars,
    dist["percentage"].to_list()
):
    axes[0,0].text(
        b.get_width(),
        b.get_y()+b.get_height()/2,
        f"{v:.1f}%",
        va="center",
        fontsize=9
    )



# 2 - Median Final Price

bars = axes[0,1].barh(
    price[col].to_list(),
    price["median_price"].to_list(),
    color="#F28E2B"
)

axes[0,1].set_title(
    "Median Final Price",
    fontweight="bold"
)

axes[0,1].set_xlabel(
    "USD"
)


for b,v in zip(
    bars,
    price["median_price"].to_list()
):
    axes[0,1].text(
        b.get_width(),
        b.get_y()+b.get_height()/2,
        f"${v:,.0f}",
        va="center",
        fontsize=9
    )



# 3 - Log Price

axes[1,0].barh(
    log_price[col].to_list(),
    log_price["median_log_price"].to_list(),
    color="#59A14F"
)

axes[1,0].set_title(
    "Median Log Final Price",
    fontweight="bold"
)

axes[1,0].set_xlabel(
    "log(USD)"
)



# 4 - Feature Quality

axes[1,1].axis("off")

axes[1,1].text(
    0.05,
    0.5,
    (
        "FEATURE QUALITY\n\n"
        f"Records : {total_count:,}\n\n"
        f"Missing : {missing_count:,} "
        f"({missing_rate:.2f}%)\n\n"
        f"Unique Categories : {unique_count}\n\n"
        f"Most Frequent : {summary[col][0]}"
    ),
    fontsize=13,
    bbox=dict(
        boxstyle="round",
        facecolor="#F5F5F5"
    )
)


# Stil

for ax in axes.flat:
    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)



plt.tight_layout()
plt.show()

 tender_nationalproceduretype:
 İhalenin ulusal prosedür türünü ifade eder. Farklı prosedür türleri
 arasında hem gerçekleşen final fiyatların hem de log dönüşümlü final
 fiyatların medyanlarında belirgin farklılıklar gözlenmiştir. Bu farklılıklar
 yalnızca düşük gözlem sayısına sahip kategorilerde değil, yüksek gözlem
 sayısına sahip kategorilerde de görülmektedir. Bu nedenle değişkenin
 final fiyat tahmininde bilgi taşıdığı değerlendirilerek modelde tutulmuştur.

In [ ]:
# İki prosedür kolonunun birbirleriyle ilişkisini kontrol et

procedure_relation = (
    df_finalprice
    .group_by([
        "tender_proceduretype",
        "tender_nationalproceduretype"
    ])
    .len()
    .rename({"len": "lot_count"})
    .sort("lot_count", descending=True)
)

print(procedure_relation)

tender_nationalproceduretype:
İhalenin ulusal prosedür türünü ifade eder. Daha detaylı ve ülkeye özgü prosedür bilgisi sağladığı için farklı prosedür kategorileri arasında gerçekleşen tender_finalpriceUsd değerlerinde farklılıklar gözlenmiştir. Bu nedenle final fiyat tahmininde bilgi taşıdığı değerlendirilerek modelde tutulmuştur. tender_proceduretype değişkenine göre daha spesifik bilgi sunduğu için modelde tercih edilmiştir.

tender_proceduretype:
İhalenin genel prosedür türünü ifade eder. OPEN, RESTRICTED, NEGOTIATED gibi üst seviye prosedür sınıflarını içerir. İhale süreci ve rekabet yapısı üzerinde etkili olabileceği için gerçekleşen final fiyat ile ilişkisi incelenmiştir. Ancak tender_nationalproceduretype daha detaylı ulusal prosedür bilgisi sağladığından, modelleme aşamasında bilgi tekrarı oluşturmaması amacıyla değerlendirilmiş ve tender_nationalproceduretype değişkeni tercih edilmiştir.

In [ ]:
df_finalprice = df_finalprice.drop("tender_proceduretype")

## TENDER_ISFRAMEWORKAGREEMENT ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# tender_isframeworkagreement FEATURE ANALYSIS


col = "tender_isframeworkagreement"

# Veri hazırlama
df_analysis = (
    df_finalprice
    .with_columns(
        pl.col(col).cast(pl.String).fill_null("Unknown"),
        pl.col("tender_finalpriceUsd").log1p().alias("log_finalprice")
    )
)

# Genel bilgiler
total = df_analysis.height
missing = df_finalprice.select(pl.col(col).null_count()).item()
missing_rate = missing / total * 100
unique = df_analysis.select(pl.col(col).n_unique()).item()

# Özet tablo
summary = (
    df_analysis
    .group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col("tender_finalpriceUsd").median().alias("median_price"),
        pl.col("log_finalprice").median().alias("median_log_price")
    )
    .with_columns(
        (pl.col("count") / total * 100).round(2).alias("percentage")
    )
    .sort("count", descending=True)
)

categories = summary[col].to_list()

# Dashboard

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(
    "Framework Agreement Analysis",
    fontsize=18,
    fontweight="bold"
)

# 1. Donut Chart

colors = ["#4E79A7", "#F28E2B", "#BAB0AC"]

axes[0,0].pie(
    summary["count"],
    labels=categories,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors[:len(categories)],
    wedgeprops=dict(width=0.4, edgecolor="white")
)

axes[0,0].set_title("Category Distribution", fontweight="bold")


# 2. Count

axes[0,1].bar(
    categories,
    summary["count"],
    color="#4E79A7"
)

axes[0,1].set_title("Number of Tenders", fontweight="bold")
axes[0,1].set_ylabel("Count")

# 3. Median Price

axes[1,0].bar(
    categories,
    summary["median_price"],
    color="#F28E2B"
)

axes[1,0].set_title("Median Final Price", fontweight="bold")
axes[1,0].set_ylabel("USD")

# 4. Feature Quality

axes[1,1].axis("off")

axes[1,1].text(
    0.05,
    0.5,
    (
        "FEATURE QUALITY\n\n"
        f"Records : {total:,}\n\n"
        f"Missing : {missing:,} ({missing_rate:.2f}%)\n\n"
        f"Unique Values : {unique}\n\n"
        f"Categories : {', '.join(categories)}"
    ),
    fontsize=13,
    bbox=dict(
        boxstyle="round",
        facecolor="#F5F5F5",
        edgecolor="#BDBDBD"
    )
)


for ax in axes.flat:
    if ax != axes[1,1]:
        ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

 tender_isframeworkagreement:
 İhalenin çerçeve anlaşması kapsamında olup olmadığını ifade eder.
 Çerçeve anlaşması olan ve olmayan ihaleler arasında gerçekleşen final
 fiyatların medyanlarında belirgin farklılık gözlenmiştir. Bu fark log
 dönüşümlü fiyatlarda da korunmaktadır. Değişkende eksik değer bulunmaması
 ve her iki kategoride de yeterli gözlem olması nedeniyle final fiyat
 tahmininde modelde tutulmuştur.

## TENDER_ISDPS ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# tender_isdps FEATURE ANALYSIS

col = "tender_isdps"

# Veri hazırlama
df_analysis = (
    df_finalprice
    .with_columns(
        pl.col(col).cast(pl.String).fill_null("Unknown")
    )
)

# Özet tablo
summary = (
    df_analysis
    .group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col("tender_finalpriceUsd").median().alias("median_price")
    )
    .with_columns(
        (pl.col("count") / df_analysis.height * 100)
        .round(2)
        .alias("percentage")
    )
    .sort("count", descending=True)
)

# Dashboard
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("tender_isdps Analysis", fontsize=18, fontweight="bold")

colors = ["#4E79A7", "#F28E2B", "#BAB0AC"]

# Donut Chart
axes[0].pie(
    summary["count"],
    labels=summary[col],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors[:summary.height],
    wedgeprops=dict(width=0.4, edgecolor="white")
)
axes[0].set_title("Distribution")

# Median Price
axes[1].bar(
    summary[col],
    summary["median_price"],
    color="#F28E2B"
)
axes[1].set_title("Median Final Price")
axes[1].set_ylabel("USD")
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

# Feature Info
axes[2].axis("off")
axes[2].text(
    0.05, 0.5,
    (
        f"Records : {df_analysis.height:,}\n\n"
        f"Missing : {df_finalprice.select(pl.col(col).null_count()).item():,}\n\n"
        f"Unique Values : {summary.height}\n\n"
        f"Categories : {', '.join(summary[col].to_list())}"
    ),
    fontsize=13,
    bbox=dict(boxstyle="round", facecolor="#F5F5F5")
)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

 tender_isdps:
 İhalenin Dinamik Satın Alma Sistemi (DPS) kapsamında olup olmadığını
 ifade eder. DPS kapsamında olan ve olmayan ihaleler arasında final fiyat
 medyanlarında bir miktar farklılık gözlenmesine rağmen, log dönüşümlü
 fiyatlardaki farkın da sınırlı olması ve "t" kategorisinin yalnızca 3.305
 gözlem içermesi nedeniyle değişkenin final fiyat tahminine katkısının
 düşük olduğu değerlendirilmiştir. Bu nedenle modelden çıkarılmıştır.

## TENDER_ISEUFUNDED ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# tender_iseufunded FEATURE ANALYSIS

col = "tender_iseufunded"

# Veri hazırlama
df_analysis = (
    df_finalprice
    .with_columns(
        pl.col(col).cast(pl.String).fill_null("Unknown")
    )
)

# Özet tablo
summary = (
    df_analysis
    .group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col("tender_finalpriceUsd").median().alias("median_price")
    )
    .with_columns(
        (pl.col("count") / df_analysis.height * 100)
        .round(2)
        .alias("percentage")
    )
    .sort("count", descending=True)
)

# Genel bilgiler
total = df_analysis.height
missing = df_finalprice.select(pl.col(col).null_count()).item()


# Dashboard


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("EU Funded Tender Analysis", fontsize=18, fontweight="bold")

colors = ["#4E79A7", "#F28E2B", "#BAB0AC"]


# 1. Donut Chart

axes[0].pie(
    summary["count"],
    labels=summary[col],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors[:summary.height],
    wedgeprops=dict(width=0.4, edgecolor="white")
)

axes[0].set_title("EU Funding Distribution")

# 2. Median Final Price

axes[1].bar(
    summary[col],
    summary["median_price"],
    color="#F28E2B",
    edgecolor="black"
)

axes[1].set_title("Median Final Price")
axes[1].set_ylabel("USD")
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

for i, v in enumerate(summary["median_price"]):
    axes[1].text(
        i, v,
        f"${v:,.0f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# 3. Feature Quality

axes[2].axis("off")

axes[2].text(
    0.05,
    0.5,
    (
        "FEATURE QUALITY\n\n"
        f"Records : {total:,}\n\n"
        f"Missing : {missing:,}\n\n"
        f"Missing Rate : {missing / total * 100:.2f}%\n\n"
        f"Categories : {', '.join(summary[col].to_list())}"
    ),
    fontsize=13,
    bbox=dict(
        boxstyle="round",
        facecolor="#F5F5F5",
        edgecolor="#BDBDBD"
    )
)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

 tender_iseufunded:
 İhalenin Avrupa Birliği fonlarıyla finanse edilip edilmediğini gösteren
 değişkendir. AB fonlu ve fonlanmayan ihalelerin medyan gerçekleşen fiyatları
 arasında farklılık gözlenmesine rağmen, bu değişkenin final fiyat tahminine
 anlamlı bir katkı sağladığına dair yeterli kanıt elde edilememiştir.
 Ayrıca değişkende 31.158 adet eksik gözlem bulunmaktadır. Bu nedenlerle
değişken modelleme aşamasında kapsam dışı bırakılmıştır.

## TENDER_AWARDCRITERIA ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


# tender_awardcriteria_count ANALYSIS

col = "tender_awardcriteria_count"

# Veri hazırlama
df_analysis = (
    df_finalprice
    .with_columns(
        pl.col(col).fill_null(0).cast(pl.Int32)
    )
)

# Genel bilgiler
total = df_analysis.height
missing = df_finalprice.select(pl.col(col).null_count()).item()

# Dağılım
distribution = (
    df_analysis
    .group_by(col)
    .agg(pl.len().alias("count"))
    .sort(col)
)

# Medyan fiyat analizi
price = (
    df_analysis
    .group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col("tender_finalpriceUsd").median().alias("median_price")
    )
    .filter(pl.col("count") >= 30)
    .sort(col)
)

# Dashboard


fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Tender Award Criteria Count Analysis",
             fontsize=18,
             fontweight="bold")

# 1. Dağılım

axes[0].bar(
    distribution[col],
    distribution["count"],
    color="#4E79A7"
)

axes[0].set_title("Distribution")
axes[0].set_xlabel("Award Criteria Count")
axes[0].set_ylabel("Tender Count")
axes[0].grid(axis="y", alpha=0.3)


# 2. Medyan Final Fiyat

axes[1].bar(
    price[col].cast(pl.String),
    price["median_price"],
    color="#F28E2B"
)

axes[1].set_title("Median Final Price")
axes[1].set_xlabel("Award Criteria Count")
axes[1].set_ylabel("USD")
axes[1].grid(axis="y", alpha=0.3)

# 3. Feature Quality

axes[2].axis("off")

axes[2].text(
    0.05,
    0.5,
    (
        "FEATURE QUALITY\n\n"
        f"Records : {total:,}\n\n"
        f"Missing : {missing:,}\n\n"
        f"Missing Rate : {missing/total*100:.2f}%\n\n"
        f"Mean : {df_analysis[col].mean():.2f}\n\n"
        f"Median : {df_analysis[col].median():.0f}\n\n"
        f"Max : {df_analysis[col].max():.0f}"
    ),
    fontsize=13,
    bbox=dict(
        boxstyle="round",
        facecolor="#F5F5F5",
        edgecolor="#BDBDBD"
    )
)

for ax in axes[:2]:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0,0,1,0.92])
plt.show()

tender_awardcriteria_count:
İhalede kullanılan değerlendirme kriterlerinin sayısını ifade etmektedir. Değişkenin tek başına gerçekleşen final fiyat üzerinde doğrusal ve güçlü bir etkisi gözlenmemiştir. Ancak ihale değerlendirme yapısını temsil eden bu değişkenin, diğer ihale özellikleriyle (prosedür türü, rekabet seviyesi, teklif sayısı ve seçim yöntemi gibi) birlikte değerlendirildiğinde modele ek bilgi sağlayabileceği düşünülmüştür.

Kriter sayısındaki artış doğrudan fiyat artışı veya azalışı şeklinde yorumlanamasa da, farklı değerlendirme mekanizmalarının ihale sonuçları üzerinde dolaylı etkileri olabilir. Bu nedenle değişkenin yalnızca tek değişkenli analiz sonuçlarına göre çıkarılmasının bilgi kaybına yol açabileceği değerlendirilmiş ve modelde tutulmuştur.

## BUYER_MAINACTIVITIES ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


col = "buyer_mainactivities"


# Veri hazırlama
df_act = (
    df_finalprice
    .with_columns(pl.col(col).fill_null("MISSING"))
)


# Analiz
df_an = (
    df_act
    .group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col("tender_finalpriceUsd").median().alias("median_price"),
        pl.col("tender_finalpriceUsd").sum().alias("total_value")
    )
    .filter(pl.col("count") >= 100)
)


# Dashboard
fig, ax = plt.subplots(2, 2, figsize=(16,12))


# 1) İhale sayısı
tmp = df_an.sort("count")

ax[0,0].barh(tmp[col], tmp["count"])
ax[0,0].set_title("İhale Sayısı")


# 2) Medyan fiyat
tmp = df_an.sort("median_price")

ax[0,1].barh(tmp[col], tmp["median_price"])
ax[0,1].set_title("Medyan Final Fiyat")
ax[0,1].xaxis.set_major_formatter(
    FuncFormatter(lambda x,_: f"{x/1e6:.1f}M")
)


# 3) Toplam değer
tmp = df_an.sort("total_value")

ax[1,0].barh(tmp[col], tmp["total_value"])
ax[1,0].set_title("Toplam İhale Değeri")
ax[1,0].xaxis.set_major_formatter(
    FuncFormatter(lambda x,_: f"{x/1e9:.1f}B")
)


# 4) Gözlem sayısı - fiyat ilişkisi
ax[1,1].scatter(
    df_an["count"],
    df_an["median_price"],
    s=80
)

ax[1,1].set_title("Gözlem Sayısı - Medyan Fiyat")
ax[1,1].set_xlabel("İhale Sayısı")
ax[1,1].set_ylabel("Medyan Fiyat")


plt.suptitle(
    "buyer_mainactivities - tender_finalprice Analizi",
    fontsize=15
)

plt.tight_layout()
plt.show()

 buyer_mainactivities:
İhaleyi gerçekleştiren alıcı kurumun ana faaliyet alanını ifade eder.
Farklı faaliyet alanları arasında gerçekleşen final fiyat medyanlarında belirgin farklılıklar gözlenmiştir. Değişkende yalnızca 3.464 eksik gözlem
bulunması ve ana kategorilerin yeterli gözlem sayısına sahip olması nedeniyle
final fiyat tahmininde model özelliği olarak tutulmuştur.

## BIDDER_TYPE ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


col = "bidder_type"

# Veri hazırlama

df_bidder = (
    df_finalprice
    .with_columns(
        pl.col(col)
        .fill_null("MISSING")
    )
    .group_by(col)
    .agg(
        pl.len().alias("count"),

        pl.col("tender_finalpriceUsd")
        .median()
        .alias("median_price"),

        pl.col("tender_finalpriceUsd")
        .sum()
        .alias("total_value"),

        pl.col("tender_finalpriceUsd")
        .std()
        .alias("std_price")
    )
    .filter(
        pl.col("count") >= 100
    )
)

colors = [
    "#4E79A7",
    "#F28E2B",
    "#59A14F",
    "#E15759",
    "#B07AA1"
]

# Dashboard

fig, ax = plt.subplots(
    2,
    2,
    figsize=(14,10)
)


fig.suptitle(
    "bidder_type - tender_finalprice Analizi",
    fontsize=16,
    fontweight="bold"
)

# 1. Kategori Dağılımı

ax[0,0].pie(
    df_bidder["count"].to_list(),
    labels=df_bidder[col].to_list(),
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"width":0.4},
    colors=colors[:df_bidder.height]
)


ax[0,0].set_title(
    "Bidder Type Dağılımı",
    fontweight="bold"
)

# 2. Medyan Final Fiyat

tmp = df_bidder.sort(
    "median_price"
)

bars = ax[0,1].barh(
    tmp[col].to_list(),
    tmp["median_price"].to_list(),
    color="#4E79A7"
)

ax[0,1].set_title(
    "Medyan Final Fiyat",
    fontweight="bold"
)

ax[0,1].xaxis.set_major_formatter(
    FuncFormatter(
        lambda x,_: f"{x/1e6:.1f}M"
    )
)

for b,v in zip(
    bars,
    tmp["median_price"].to_list()
):
    ax[0,1].text(
        b.get_width(),
        b.get_y()+b.get_height()/2,
        f"${v/1e6:.2f}M",
        va="center",
        fontsize=9
    )

# 3. Toplam Final Değeri

tmp = df_bidder.sort(
    "total_value"
)

bars = ax[1,0].barh(
    tmp[col].to_list(),
    tmp["total_value"].to_list(),
    color="#F28E2B"
)

ax[1,0].set_title(
    "Toplam Final Değeri",
    fontweight="bold"
)

ax[1,0].xaxis.set_major_formatter(
    FuncFormatter(
        lambda x,_: f"{x/1e9:.1f}B"
    )
)

for b,v in zip(
    bars,
    tmp["total_value"].to_list()
):
    ax[1,0].text(
        b.get_width(),
        b.get_y()+b.get_height()/2,
        f"{v/1e9:.2f}B",
        va="center",
        fontsize=9
    )

# 4. Fiyat Değişkenliği

tmp = df_bidder.sort(
    "std_price"
)

bars = ax[1,1].barh(
    tmp[col].to_list(),
    tmp["std_price"].to_list(),
    color="#59A14F"
)

ax[1,1].set_title(
    "Final Fiyat Değişkenliği",
    fontweight="bold"
)

ax[1,1].xaxis.set_major_formatter(
    FuncFormatter(
        lambda x,_: f"{x/1e6:.1f}M"
    )
)

for b,v in zip(
    bars,
    tmp["std_price"].to_list()
):
    ax[1,1].text(
        b.get_width(),
        b.get_y()+b.get_height()/2,
        f"{v/1e6:.2f}M",
        va="center",
        fontsize=9
    )

for a in ax.flat:
    a.grid(
        axis="x",
        linestyle="--",
        alpha=0.3
    )

    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)



plt.tight_layout()
plt.show()

 bidder_type:
 Teklif veren kuruluşun türünü ifade eder. Ancak değişkenin tüm dolu
gözlemleri "supplier" kategorisinde toplanmıştır ve ikinci bir anlamlı
 kategori bulunmamaktadır. Bu nedenle değişken yeterli ayırt edici bilgi
 taşımamakta ve model açısından varyasyon sağlamamaktadır. Final fiyat
 tahmininde faydalı bir özellik olmadığı değerlendirilerek modelden
 çıkarılmıştır.

In [ ]:
# Model açısından düşük bilgi taşıdığı değerlendirilen kolonlar
drop_features = [
    "tender_isdps",       # DPS kapsamında olup olmadığını gösterir; final fiyatla ilişkisi sınırlı bulundu.
    "tender_iseufunded",  # AB fonlu olup olmadığını gösterir; final fiyat tahminine katkısı düşük bulundu.
    "bidder_type",        # Dolu değerlerin tamamı "supplier" olduğu için ayırt edici bilgi taşımıyor.

]

# Kolonları veri setinden sil
df_finalprice = df_finalprice.drop(drop_features)

print("Silinen kolonlar:")
for col in drop_features:
    print("-", col)

print("\nKalan kolon sayısı:", len(df_finalprice.columns))

## CVP KOLONLARI ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


cpv_columns = [
    "tender_cpvs",
    "tender_maincpv"
]


for col in cpv_columns:

    # Veri
    total = df_finalprice.height

    null_count = df_finalprice[col].null_count()

    unique_count = df_finalprice[col].n_unique()


    freq = (
        df_finalprice
        .group_by(col)
        .len()
        .rename({"len":"count"})
        .sort("count", descending=True)
        .head(10)
    )


    # Figure
    fig = plt.figure(figsize=(15,7))

    fig.suptitle(
        f"{col} Cardinality Analizi",
        fontsize=18,
        fontweight="bold"
    )


    # KPI

    ax_kpi = fig.add_axes([.04,.75,.92,.13])
    ax_kpi.axis("off")


    kpis=[
        ("Toplam Kayıt",f"{total:,}"),
        ("Unique Değer",f"{unique_count:,}"),
        ("Eksik Kayıt",f"{null_count:,}"),
        ("Eksik Oranı",f"%{null_count/total*100:.2f}")
    ]


    for i,(t,v) in enumerate(kpis):

        x=i*.25

        ax_kpi.add_patch(
            plt.Rectangle(
                (x+.02,.1),
                .21,.8,
                fill=False,
                linewidth=1.5
            )
        )

        ax_kpi.text(
            x+.125,.65,
            t,
            ha="center",
            fontsize=11,
            fontweight="bold"
        )

        ax_kpi.text(
            x+.125,.3,
            v,
            ha="center",
            fontsize=15,
            fontweight="bold"
        )


    # En sık değerler

    ax = fig.add_axes([.15,.15,.7,.45])


    bars = ax.barh(
        freq[col].cast(pl.String).to_list()[::-1],
        freq["count"].to_list()[::-1],
        color=[
            "#2563EB",
            "#16A34A",
            "#F59E0B",
            "#DC2626",
            "#9333EA",
            "#06B6D4",
            "#84CC16",
            "#F97316",
            "#8B5CF6",
            "#14B8A6"
        ]
    )


    ax.set_title(
        "En Sık Kullanılan 10 CPV",
        fontweight="bold"
    )

    ax.set_xlabel("Lot Sayısı")


    for bar in bars:

        ax.text(
            bar.get_width(),
            bar.get_y()+bar.get_height()/2,
            f"{int(bar.get_width()):,}",
            va="center",
            fontsize=9
        )


    ax.grid(
        axis="x",
        alpha=.3
    )


    plt.tight_layout()
    plt.show()

 tender_maincpv:
 İhalenin ana CPV bilgisini temsil etmesi beklenen değişkendir.
 Ancak veri setinde yalnızca 2 benzersiz değer bulunmuş ve değerlerin
 317.370'i "t", 1.666'sı ise "f" olarak gözlenmiştir. Bu nedenle kolon
 gerçek anlamda CPV kodlarını temsil etmemekte ve ihale türleri arasında
 yeterli ayırt edici bilgi sağlamamaktadır. Ayrıca tender_cpvs kolonu
 CPV bilgilerini çok daha detaylı ve çeşitli şekilde içerdiğinden,
 tender_maincpv'nin modele eklenmesinin gereksiz olduğu değerlendirilerek
 modelden çıkarılmıştır.

In [ ]:
drop_features = [
    "tender_maincpv",      # Yalnızca t/f değerlerinden oluştuğu için anlamlı CPV çeşitliliği sağlamıyor.
]

df_finalprice = df_finalprice.drop(drop_features)

print("Silinen kolonlar:")
for col in drop_features:
    print("-", col)

print("\nKalan kolon sayısı:", len(df_finalprice.columns))

## TENDER_CPVS ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


col, target = "tender_cpvs", "tender_finalpriceUsd"


df = df_finalprice.with_columns(
    pl.col(col).fill_null("Unknown")
)


cpv = (
    df.group_by(col)
    .agg(
        pl.len().alias("count"),
        pl.col(target).median().alias("median_price")
    )
)


top = cpv.sort("count", descending=True).head(10)

price = (
    cpv.filter(pl.col("count")>=100)
    .sort("median_price", descending=True)
    .head(10)
)


fig, ax = plt.subplots(2,2, figsize=(14,9))


# Kullanım sıklığı
ax[0,0].bar(
    top[col].cast(str),
    top["count"],
    color="mediumslateblue"
)
ax[0,0].set_title("Top CPV Kullanımı")
ax[0,0].tick_params(axis="x",rotation=70)


# Fiyat etkisi
ax[0,1].barh(
    price[col].cast(str),
    price["median_price"],
    color="gold"
)
ax[0,1].set_title("CPV - Medyan Final Fiyat")


# Cardinality
ax[1,0].pie(
    [df[col].n_unique(), df.height],
    labels=["Unique","Total"],
    autopct="%1.1f%%",
    colors=["hotpink","lightblue"]
)
ax[1,0].set_title("CPV Cardinality")


# Sıklık - fiyat
ax[1,1].scatter(
    cpv["count"],
    cpv["median_price"],
    color="seagreen"
)
ax[1,1].set_title("CPV Sıklığı - Fiyat")


plt.suptitle(
    "tender_cpvs Analizi",
    fontsize=14
)

plt.tight_layout()
plt.show()

tender_supplytype ve tender_cpvs Değerlendirmesi

tender_supplytype:

İhalenin genel tedarik türünü (mal, hizmet, yapım vb.) ifade eder. Düşük cardinality'ye sahip olması ve genel satın alma yapısını temsil etmesi nedeniyle modelde tutulmuştur.

tender_cpvs:

İhale konusu ürün veya hizmeti detaylı CPV kodları ile sınıflandıran değişkendir. 52.159 farklı kategori içermesi ve birçok kategorinin çok az gözlemde bulunması nedeniyle yüksek cardinality problemine sahiptir.

Bu nedenle:

tender_supplytype → Modelde tutulmuştur.

tender_cpvs → Yüksek cardinality ve overfitting riski nedeniyle çıkarılmıştır.

tender_cpvs değişkeninin çıkarılma nedeni tender_supplytype ile aynı bilgiyi taşıması değil, ham kategorik yapısının modelleme açısından uygun olmamasıdır.

## BUYER_TYPE ANALİZ

In [ ]:
import polars as pl


col = "buyer_type"


summary = (
    df_finalprice
    .filter(
        pl.col(col).is_not_null()
    )
    .group_by(col)
    .agg(
        pl.len().alias("Lot Sayısı"),
        (
            pl.len() / df_finalprice.height * 100
        ).round(2).alias("Yüzde (%)")
    )
    .sort(
        "Lot Sayısı",
        descending=True
    )
)


display(summary)

buyer_type Değişkeninin Modelden Çıkarılması

buyer_type değişkeni, ihaleyi gerçekleştiren alıcının kurum türünü belirtmektedir. Ancak yapılan veri analizi sonucunda bu değişkenin veri setinde yalnızca 1 farklı değer içerdiği görülmüştür.

Bir değişkenin model açısından bilgi taşıyabilmesi için gözlemler arasında yeterli değişkenlik bulunması gerekir. buyer_type değişkeninde ise tüm gözlemler aynı kategoriye ait olduğundan değişken herhangi bir ayırt edici bilgi sağlamamaktadır.

Ayrıca sabit bir değişkenin One-Hot Encoding ile modele dahil edilmesi modelin tahmin gücüne anlamlı bir katkı sağlamaz ve gereksiz bir özellik oluşturur.

Bu nedenle:

* buyer_type → Modelden çıkarılmıştır.
* Çıkarma nedeni → Tek unique değere sahip olması ve değişkenlik içermemesidir.

Bu karar, değişkenin missing oranından bağımsız olarak alınmıştır. Temel gerekçe, buyer_type değişkeninin veri setinde constant (sabit) bir değişken haline gelmiş olmasıdır.


In [ ]:
# Modelleme açısından kullanılmayacak kolonların çıkarılması
# buyer_type  -> Tek unique değere sahip olduğu için
# tender_cpvs  -> Çok yüksek cardinality nedeniyle

drop_features = [
    "buyer_type",
    "tender_cpvs"
]

df_finalprice = df_finalprice.drop(drop_features)

print("Çıkarılan kolonlar:", drop_features)
print("Kalan kolon sayısı:", len(df_finalprice.columns))

## TENDER_DIGIWHIST_PRICE ANALİZ

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


col, target = "tender_digiwhist_price", "tender_finalpriceUsd"


df = (
    df_finalprice
    .filter((pl.col(col)>0) & (pl.col(target)>0))
    .with_columns(
        pl.col(col).log1p().alias("log_digi"),
        (pl.col(target)/pl.col(col)).alias("ratio")
    )
)


relation = (
    df.with_columns(
        pl.col("log_digi").qcut(10).alias("group")
    )
    .group_by("group")
    .agg(
        pl.col(target).median().alias("median_final")
    )
)


fig, ax = plt.subplots(2,2, figsize=(14,10))


# Digiwhist dağılımı
ax[0,0].hist(
    df["log_digi"],
    bins=40,
    color="purple"
)
ax[0,0].set_title("Digiwhist Dağılımı")


# Medyan fiyat karşılaştırması
ax[0,1].bar(
    ["Digiwhist","Final"],
    [df[col].median(), df[target].median()],
    color=["royalblue","gold"]
)
ax[0,1].set_title("Medyan Fiyat")


# İlişki
ax[1,0].plot(
    range(len(relation)),
    relation["median_final"],
    marker="o",
    color="mediumblue"
)
ax[1,0].set_title("Digiwhist - Final İlişkisi")
ax[1,0].set_xlabel("Digiwhist Seviyesi")


# Oran
ax[1,1].hist(
    df["ratio"],
    bins=50,
    color="gold"
)
ax[1,1].set_title("Final / Digiwhist Oranı")


plt.suptitle(
    "tender_digiwhist_price Analizi",
    fontsize=14
)

plt.tight_layout()
plt.show()

 tender_digiwhist_price:
 tender_finalprice mevcut olduğunda final fiyat bilgisini doğrudan
 kullandığı için hedef değişkenle bilgi sızıntısına (data leakage)
 neden olmaktadır. Yapılan analizde tender_digiwhist_price ile
 tender_finalpriceUsd arasındaki Pearson korelasyonu 0.99975,
 log dönüşümlü korelasyon ise 0.999709 olarak bulunmuştur.
 Bu nedenle modelin final fiyatı bağımsız olarak öğrenmesini engelleyerek
yapay şekilde yüksek performans oluşturabileceğinden modelden çıkarılmıştır.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

col = "buyer_country"
target = "tender_finalpriceUsd"

# Veri
df = df_finalprice.with_columns(
    pl.col(col).cast(pl.String).fill_null("Unknown")
)

# Renkler
colors = plt.cm.tab10.colors

# KPI
kpis = [
    ("Toplam Kayıt", f"{df.height:,}"),
    ("Ülke Sayısı", f"{df[col].n_unique():,}"),
    ("Eksik", f"{df_finalprice[col].null_count():,}"),
    ("Ort. Fiyat", f"${df[target].mean():,.0f}")
]

# En çok ihale bulunan ülkeler
country = (
    df.group_by(col)
    .len()
    .rename({"len": "count"})
    .sort("count", descending=True)
    .head(10)
)

top_countries = country[col].to_list()

# Medyan fiyat
price = (
    df.group_by(col)
    .agg(
        pl.col(target).median().alias("median_price")
    )
    .filter(
        pl.col(col).is_in(top_countries)
    )
    .sort("median_price")
)

# Figure

fig = plt.figure(figsize=(15, 8))

fig.suptitle(
    "buyer_country Analizi",
    fontsize=18,
    fontweight="bold"
)

# KPI
ax = fig.add_axes([.04, .72, .92, .15])
ax.axis("off")

for i, (t, v) in enumerate(kpis):
    x = i * .25

    ax.add_patch(
        plt.Rectangle(
            (x + .02, .1),
            .21,
            .8,
            fill=False,
            linewidth=1.5
        )
    )

    ax.text(x + .125, .65, t,
            ha="center",
            fontweight="bold")

    ax.text(x + .125, .3, v,
            ha="center",
            fontsize=15,
            fontweight="bold")

# Ülke dağılımı
ax1 = fig.add_axes([.06, .15, .4, .4])

bars = ax1.bar(
    country[col].to_list(),
    country["count"].to_list(),
    color=colors[:len(country)],
    edgecolor="black"
)

ax1.set_title("En Çok İhale Bulunan 10 Ülke", fontweight="bold")
ax1.set_ylabel("İhale Sayısı")
ax1.tick_params(axis="x", rotation=45)

for bar, value in zip(bars, country["count"]):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

# Medyan fiyat
ax2 = fig.add_axes([.55, .15, .35, .4])

bars = ax2.barh(
    price[col].to_list(),
    price["median_price"].to_list(),
    color="#16A34A",
    edgecolor="black"
)

ax2.set_title("Ülkelere Göre Medyan Final Fiyat", fontweight="bold")
ax2.set_xlabel("Medyan Final Fiyat (USD)")

for b, v in zip(bars, price["median_price"]):
    ax2.text(
        b.get_width(),
        b.get_y() + b.get_height()/2,
        f"${v:,.0f}",
        va="center",
        fontsize=8
    )

plt.tight_layout()
plt.show()

 buyer_country Değerlendirmesi

buyer_country, ihaleyi gerçekleştiren alıcı kurumun ülkesini ifade eden kategorik değişkendir.

Analiz sonucunda 13 farklı ülke değeri bulunmasına rağmen kayıtların yaklaşık %99.95'inin Almanya (DE) olduğu görülmüştür. Diğer ülkelerdeki gözlem sayılarının çok düşük olması nedeniyle ülke bazlı fiyat farklılıklarının model tarafından güvenilir şekilde öğrenilemeyeceği değerlendirilmiştir.

Bu nedenle değişkenin gerçekleşen ihale fiyatı tahminine sağlayacağı katkının sınırlı olacağı düşünülmüş ve modelden çıkarılmıştır.

In [ ]:
# buyer_country kolonunu çıkarma

df_finalprice = df_finalprice.drop(
    "buyer_country"
)

print("buyer_country kolonu silindi.")
print("Kalan kolon sayısı:", len(df_finalprice.columns))

## KALAN KOLONLAR

In [ ]:
# Data leakage oluşturduğu için çıkarılıyor
df_finalprice = df_finalprice.drop("tender_digiwhist_price")

print("Kalan kolon sayısı:", len(df_finalprice.columns))

In [ ]:
# Kalan kolonları görüntüleme

print("Toplam kolon sayısı:", len(df_finalprice.columns))

for col in df_finalprice.columns:
    print(col)

In [ ]:
print("Kalan kolon sayısı:", len(df_finalprice.columns))

print("\nKalan kolonlar:")
for col in df_finalprice.columns:
    print("-", col)

In [ ]:
# ============================================================
# MODEL FEATURES
# Hedef değişken: tender_finalpriceUsd
# ============================================================

model_features = [

    # İhale bilgileri
    "tender_nationalproceduretype",     # Ulusal prosedür türü
    "tender_supplytype",                # Tedarik türü (mal/hizmet/yapım)

    # İhale büyüklüğü ve rekabet
    "tender_lotscount",                 # İhaledeki toplam lot sayısı
    "tender_recordedbidscount",         # Toplam kayıtlı teklif sayısı
    "competition_level",                # Teklif sayısından türetilen rekabet seviyesi

    # İhale değerlendirme bilgileri
    "tender_isframeworkagreement",      # Çerçeve anlaşma durumu
    "tender_selectionmethod",           # Kazanan seçim yöntemi
    "tender_awardcriteria_count",       # Kullanılan değerlendirme kriteri sayısı

    # Alıcı bilgileri
    "buyer_mainactivities",             # Alıcı kurumun faaliyet alanı

    # Zaman özellikleri
    "tender_year",                      # İhale yılı
    "submission_period",                # Teklif verme süresi
    "decision_period",                  # Karar süresi
    "decision_duration_days",           # İlan ve karar tarihi arasındaki toplam süre

    # Risk göstergeleri
    "corr_singleb",                     # Tek teklif riski
    "corr_proc",                        # Prosedür kaynaklı risk
    "corr_subm",                        # Teklif süreci riski
    "corr_nocft",                       # İlan yapılmaması riski
    "corr_decp",                        # Karar süreci riski
    "corr_buyer_concentration",          # Alıcı yoğunlaşması riski

    # Lot bilgileri
    "lot_status",                       # Lot durumu (aktif/sonuçlanan vb.)

    "tender_finalpriceUsd"              # target
]




print("Model feature sayısı:", len(model_features))
print("Target:", target)

## ⚙️ 9. Missing Value Handling / Eksik Değer Doldurma

This section handles missing values in the dataset using appropriate imputation techniques to improve data quality and prepare the dataset for further analysis and modeling.

Bu bölümde veri setindeki eksik değerler uygun doldurma yöntemleri kullanılarak ele alınmakta, veri kalitesi artırılmakta ve sonraki analiz/modelleme aşamalarına hazırlık yapılmaktadır.

In [ ]:
# ============================================================
# MODEL FEATURE'LARI İÇİN MISSING VALUE ANALİZİ
# ============================================================

missing_analysis = (
    df_finalprice
    .select(
        [
            pl.col(col).null_count().alias(f"{col}_missing")
            for col in model_features
        ]
    )
)

# Tek satırlık sonucu uzun formata çevir
missing_analysis = (
    missing_analysis
    .transpose(
        include_header=True,
        header_name="feature",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count") /
            len(df_finalprice) * 100
        ).alias("missing_percentage")
    )
    .sort("missing_percentage", descending=True)
)

print("📊 MODEL FEATURE'LARI MISSING ANALİZİ")
print("=" * 70)
print(missing_analysis)

In [ ]:
# ============================================================
# NUMERIC VE CATEGORICAL FEATURE'LARIN AYRILMASI
# ============================================================

numeric_features = [

    "tender_lotscount",                  # İhaledeki toplam lot sayısı
    "tender_recordedbidscount",           # Toplam teklif sayısı
    "tender_awardcriteria_count",         # Değerlendirme kriteri sayısı

    "tender_year",                        # İhale yılı

    "submission_period",                  # Teklif verme süresi
    "decision_period",                   # Karar süresi
    "decision_duration_days",             # İlan ve karar tarihi arasındaki süre

    # Risk göstergeleri
    "corr_singleb",                       # Tek teklif riski
    "corr_proc",                          # Prosedür riski
    "corr_subm",                          # Teklif süreci riski
    "corr_nocft",                         # İlan yapılmaması riski
    "corr_decp",                          # Karar süreci riski
    "corr_buyer_concentration",           # Alıcı yoğunlaşması riski

]


categorical_features = [

    "tender_nationalproceduretype",       # Ulusal prosedür türü
    "tender_supplytype",                  # Tedarik türü (mal/hizmet/yapım)

    "lot_status",                         # Lot durumu

    "tender_isframeworkagreement",        # Çerçeve anlaşma durumu

    "tender_selectionmethod",              # Kazanan seçim yöntemi

    "competition_level",                  # Teklif sayısına göre rekabet seviyesi

    "buyer_mainactivities"                # Alıcı kurumun faaliyet alanı

]


print("Numeric feature sayısı:", len(numeric_features))
print("Categorical feature sayısı:", len(categorical_features))

In [ ]:
print("Numeric feature sayısı:", len(numeric_features))
print("Categorical feature sayısı:", len(categorical_features))

print("\nNumeric Features:")
for col in numeric_features:
    print("-", col)

print("\nCategorical Features:")
for col in categorical_features:
    print("-", col)

In [ ]:
import matplotlib.pyplot as plt
import math


# NUMERIC FEATURE HISTOGRAMLARI

n_cols = 2
n_rows = math.ceil(len(numeric_features) / n_cols)


fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, n_rows * 5)
)


axes = axes.flatten()


for i, col in enumerate(numeric_features):

    ax = axes[i]


    missing_count = (
        df_finalprice[col]
        .null_count()
    )

    missing_percentage = (
        missing_count
        / len(df_finalprice)
        * 100
    )


    values = (
        df_finalprice
        .select(pl.col(col))
        .to_series()
        .drop_nulls()
        .to_list()
    )


    ax.hist(
        values,
        bins=40,
        edgecolor="black",
        alpha=0.8
    )


    ax.set_title(
        f"{col}\nMissing: {missing_percentage:.2f}%",
        fontsize=13,
        fontweight="bold"
    )


    ax.set_xlabel(
        col,
        fontsize=11
    )

    ax.set_ylabel(
        "Frekans",
        fontsize=11
    )


    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )


# boş grafik alanlarını kaldır

for j in range(
    len(numeric_features),
    len(axes)
):
    fig.delaxes(
        axes[j]
    )


fig.suptitle(
    "Numerik Model Değişkenlerinin Dağılımları",
    fontsize=20,
    fontweight="bold",
    y=1.02
)


plt.tight_layout()

plt.show()

Grafikler incelendiğinde; ihale karakteristiklerini gösteren değişkenlerin (tender_lotscount, tender_recordedbidscount vb.) ve yolsuzluk/risk skorlarının çoğunlukla belirgin bir şekilde sağa çarpık veya kesikli (0-1 ağırlıklı) bir dağılım sergilediği görülmektedir. Sürekli zaman değişkenleri olan submission_period (%26.56) ve decision_period (%34.12) eksiklikleri, aşırı değerlerden etkilenmemesi adına Medyan (Median) yöntemiyle doldurulacaktır. Buna karşın, oran veya indeks yapısındaki corr_singleb (%16.69), corr_subm (%32.80) ve corr_buyer_concentration (%22.19) değişkenlerindeki eksiklikler, verinin yapısını bozmamak ve eksiklik durumunun modeller tarafından ayrı bir örüntü olarak öğrenilmesini sağlamak amacıyla -1 (Eksik Veri Göstergesi) değeri ile doldurulacaktır. Son olarak, kronolojik hatalar içeren decision_duration_days (%34.28) değişkenindeki negatif değerler öncelikle NaN yapılarak temizlenecek, ardından oluşan tüm eksiklikler Medyan yöntemiyle tamamlanacaktır.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

target_values = (
    df_finalprice
    .select("tender_finalpriceUsd")
    .to_series()
    .drop_nulls()
    .to_numpy()
)

upper = np.percentile(target_values, 99)

plt.figure(figsize=(9,5))

plt.hist(
    target_values,
    bins=60,
    range=(0, upper),
    edgecolor="black"
)

plt.title("tender_finalpriceUsd Dağılımı (%99)")
plt.xlabel("tender_finalpriceUsd")
plt.ylabel("Frekans")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# submission_period ve decision_period eksik değerlerini
# kendi medyanları ile doldurma

df_finalprice = df_finalprice.with_columns([
    pl.col("submission_period")
    .fill_null(pl.col("submission_period").median())
    .alias("submission_period"),

    pl.col("decision_period")
    .fill_null(pl.col("decision_period").median())
    .alias("decision_period")
])

In [ ]:
df_finalprice.select([
    pl.col("submission_period").null_count().alias("submission_period_missing"),
    pl.col("decision_period").null_count().alias("decision_period_missing")
])

In [ ]:
import polars as pl

cols = [
    "submission_period",
    "decision_period",
    "decision_duration_days"
]

# İstatistik + negatif kontrol
display(
    df_finalprice
    .select(
        cols +
        [
            (pl.col(c)<0).sum().alias(f"{c}_negative")
            for c in cols
        ]
    )
    .describe()
)

In [ ]:
df_finalprice = df_finalprice.with_columns(
    pl.when(pl.col("decision_duration_days") < 0)
    .then(None)
    .otherwise(pl.col("decision_duration_days"))
    .alias("decision_duration_days")
)

decision_duration_days değişkeninde negatif süre değerleri tespit edilmiştir. Negatif süreler tarihsel olarak mümkün olmadığından veri hatası olarak değerlendirilmiş ve eksik değer (null) olarak değiştirilmiştir. Bu yaklaşım, hatalı gözlemleri veri setinden tamamen çıkarmak yerine modelleme aşamasında uygun yöntemlerle ele alınmasına olanak sağlamaktadır.

In [ ]:
import polars as pl


# Süre değişkeni - median ile doldurma
df_finalprice = df_finalprice.with_columns(

    pl.col("decision_duration_days")
    .fill_null(
        pl.col("decision_duration_days").median()
    )

)


# Risk göstergeleri -1 ile doldurma

risk_missing_cols = [
    "corr_singleb",
    "corr_subm",
    "corr_buyer_concentration"
]


df_finalprice = df_finalprice.with_columns(

    [
        pl.col(col)
        .fill_null(-1)
        .alias(col)

        for col in risk_missing_cols
    ]

)


# Kontrol

df_finalprice.select(
    [
        "decision_duration_days",
        "corr_singleb",
        "corr_subm",
        "corr_buyer_concentration"
    ]
).null_count()

## çıkar

In [ ]:
import polars as pl

risk_missing_cols = [
    "corr_singleb",
    "corr_subm",
    "corr_buyer_concentration"
]

# ============================================================
# RİSK KOLONLARINDA MISSING VALUE DÜZELTMESİ
# ============================================================

for col in risk_missing_cols:

    # Missing flag oluştur
    df_finalprice = df_finalprice.with_columns(
        (pl.col(col) == -1)
        .cast(pl.Int8)
        .alias(f"{col}_is_missing")
    )

    # Yapay -1 değerlerini tekrar NULL yap
    df_finalprice = df_finalprice.with_columns(
        pl.when(pl.col(col) == -1)
        .then(None)
        .otherwise(pl.col(col))
        .alias(col)
    )

# ============================================================
# KONTROL
# ============================================================

print("=" * 90)
print("RİSK KOLONLARI — MISSING DÜZELTMESİ SONRASI")
print("=" * 90)

for col in risk_missing_cols:

    flag_col = f"{col}_is_missing"

    total = df_finalprice.height

    null_count = (
        df_finalprice
        .select(
            pl.col(col)
            .is_null()
            .sum()
            .alias("count")
        )
        .item()
    )

    flag_count = (
        df_finalprice
        .select(
            pl.col(flag_col)
            .sum()
            .alias("count")
        )
        .item()
    )

    remaining_minus_one = (
        df_finalprice
        .select(
            (pl.col(col) == -1)
            .sum()
            .alias("count")
        )
        .item()
    )

    print(f"\n{col}")
    print("-" * 60)
    print(f"NULL sayısı          : {null_count:,} ({null_count / total * 100:.2f}%)")
    print(f"Missing flag = 1     : {flag_count:,} ({flag_count / total * 100:.2f}%)")
    print(f"Kalan -1 sayısı      : {remaining_minus_one:,}")

print("\n" + "=" * 90)
print("MISSING DÜZELTMESİ TAMAMLANDI")
print("=" * 90)

In [ ]:
import polars as pl

for col in [
    "corr_singleb",
    "corr_subm",
    "corr_buyer_concentration"
]:
    df_finalprice = df_finalprice.with_columns(
        pl.when(pl.col(col) == -1)
        .then(float("nan"))
        .otherwise(pl.col(col))
        .alias(col)
    )

In [ ]:
# ============================================================
# YENİ MISSING FLAG KOLONLARINI KONTROL ET
# ============================================================

new_risk_features = [
    "corr_singleb_is_missing",
    "corr_subm_is_missing",
    "corr_buyer_concentration_is_missing"
]

print("=" * 90)
print("DF_FINALPRICE — YENİ RİSK MISSING FLAG KOLONLARI")
print("=" * 90)

print("\nYeni kolonlar:")

for i, col in enumerate(new_risk_features, 1):
    print(f"{i}. {col}")

print("\nToplam yeni kolon:", len(new_risk_features))

print("\nDF_FINALPRICE SHAPE:")
print(df_finalprice.shape)

print("\nYeni kolonların dağılımı:")

for col in new_risk_features:

    print(f"\n{col}")
    print("-" * 60)

    print(
        df_finalprice
        .group_by(col)
        .len()
        .sort(col)
    )

# ============================================================
# KOLONLAR GERÇEKTEN VAR MI?
# ============================================================

missing_columns = [
    col
    for col in new_risk_features
    if col not in df_finalprice.columns
]

print("\n" + "=" * 90)

if len(missing_columns) == 0:
    print("✓ TÜM YENİ KOLONLAR DF_FINALPRICE İÇİNDE MEVCUT")
else:
    print("⚠️ EKSİK KOLONLAR:")
    for col in missing_columns:
        print(col)

print("=" * 90)

## kategorik dağılım

In [ ]:
import matplotlib.pyplot as plt
import math

# KATEGORİK FEATURE'LARIN DAĞILIMLARI + MISSING BİLGİSİ

n_cols = 2
n_rows = math.ceil(len(categorical_features) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 5 * n_rows)
)

axes = axes.flatten()

for i, col in enumerate(categorical_features):

    ax = axes[i]


    # Missing bilgisi

    missing_count = df_finalprice[col].null_count()

    missing_percentage = (
        missing_count / len(df_finalprice) * 100
    )

    # Kategori dağılımı

    distribution = (
        df_finalprice
        .with_columns(
            pl.col(col)
            .cast(pl.String)
            .fill_null("Missing")
            .alias("_category")
        )
        .group_by("_category")
        .len()
        .sort("len", descending=True)
        .head(10)
    )

    categories = distribution["_category"].to_list()
    counts = distribution["len"].to_list()

    # Grafik

    bars = ax.bar(
        categories,
        counts,
        edgecolor="black",
        alpha=0.8
    )

    ax.set_title(
        f"{col}\n"
        f"Missing: {missing_count:,} "
        f"(%{missing_percentage:.2f})",
        fontsize=12,
        fontweight="bold"
    )

    ax.set_xlabel(
        col,
        fontsize=10
    )

    ax.set_ylabel(
        "Lot Sayısı",
        fontsize=10
    )

    ax.tick_params(
        axis="x",
        rotation=45
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    # Bar değerleri

    for bar in bars:

        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{int(height):,}",
            ha="center",
            va="bottom",
            fontsize=9
        )

# Boş subplotları kaldır

for j in range(len(categorical_features), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle(
    "Kategorik Model Değişkenlerinin Dağılımları ve Missing Değerleri",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

plt.tight_layout(
    rect=[0, 0, 1, 0.96]
)

Kategorik değişkenlerin dağılımları incelendiğinde; tender_proceduretype (%0.13), tender_nationalproceduretype (%17.71), tender_selectionmethod (%2.38) ve buyer_mainactivities (%1.15) gibi değişkenlerde en yüksek frekansa sahip basbayağı öne çıkan baskın kategoriler (örneğin OPEN, pt_open, LOWEST_PRICE vb.) gözlemlendiğinden, bu alanlardaki eksik veriler Mod (En Sık Tekrar Eden Değer) yöntemiyle doldurulacaktır. Üç ana dengeli kategoriye (WORKS, SERVICES, SUPPLIES) ayrılan tender_supplytype (%0.42) değişkeninde ise eksiklikler, diğer ihale öznitelikleriyle olan ilişkiler dikkate alınarak KNN (K-Nearest Neighbors Imputation) algoritması ile tahmin edilerek doldurulacaktır. lot_status, tender_isframeworkagreement ve competition_level değişkenlerinde ise eksik veri bulunmamakta veya ihmal edilebilir düzeyde (%0.00) yer almaktadır.

In [ ]:
# KATEGORİK DEĞİŞKENLERİN MISSING DEĞERLERİNİN DOLDURULMASI

df_finalprice = df_finalprice.with_columns([


    # Yüksek missing oranı → MOD
    pl.col("tender_nationalproceduretype")
    .fill_null("pt_open")
    .alias("tender_nationalproceduretype"),


    # Kategoriler dengeli → MOD
  pl.col("tender_selectionmethod")
  .fill_null("LOWEST_PRICE")
  .alias("tender_selectionmethod"),

    # Düşük missing → MOD
    # En sık kategori: GENERAL_PUBLIC_SERVICES
    pl.col("buyer_mainactivities")
    .fill_null("GENERAL_PUBLIC_SERVICES")
    .alias("buyer_mainactivities"),

    # Düşük missing → MOD
    # En sık kategori: AWARDED
    pl.col("lot_status")
    .fill_null("AWARDED")
    .alias("lot_status")

])

## KNN İLE DOLDURMA

In [ ]:
# Eksik olmayan kayıtları ayır

df_train = df_finalprice.filter(
    pl.col("tender_supplytype").is_not_null()
)

In [ ]:
knn_features = [

    # İhale tipi
    "tender_nationalproceduretype",  # Ulusal ihale prosedür sınıflandırması
    "tender_selectionmethod",        # Kazananın belirlenme yöntemi

    # Kurum bilgisi
    "buyer_mainactivities",          # İhaleyi gerçekleştiren kurumun ana faaliyet alanı

    # Rekabet bilgisi
    "competition_level",             # Teklif sayısına göre oluşturulan rekabet seviyesi

    # Çerçeve anlaşma bilgisi
    "tender_isframeworkagreement",   # İhalenin çerçeve anlaşma kapsamında olup olmadığı

    # İhale büyüklüğü
    "tender_lotscount"               # İhaledeki toplam lot sayısı
]

KNN modelinde tender_supplytype eksik değerlerini doldurmak amacıyla ihale yapısı, kurum özellikleri ve rekabet durumunu temsil eden değişkenler kullanılmıştır. Seçilen özellikler; ihale prosedürü, seçim yöntemi, kurum faaliyet alanı, rekabet seviyesi, çerçeve anlaşma durumu ve lot sayısı gibi supplytype kategorisini etkileyebilecek temel ihale özelliklerini içermektedir. Bu değişkenler, benzer ihale kayıtlarının bulunması ve eksik değerlerin daha doğru tahmin edilmesi amacıyla modele dahil edilmiştir.

In [ ]:
import polars as pl

target = "tender_supplytype"

df_known = df_finalprice.filter(
    pl.col(target).is_not_null()
)

df_missing = df_finalprice.filter(
    pl.col(target).is_null()
)

print("Dolu kayıt:", df_known.height)
print("Eksik kayıt:", df_missing.height)

In [ ]:
knn_features = [

    # İhale tipi
    "tender_nationalproceduretype",  # Ulusal ihale prosedür sınıflandırması
    "tender_selectionmethod",        # Kazananın belirlenme yöntemi

    # Kurum bilgisi
    "buyer_mainactivities",          # İhaleyi gerçekleştiren kurumun ana faaliyet alanı

    # Rekabet bilgisi
    "competition_level",             # Teklif sayısına göre oluşturulan rekabet seviyesi

    # Çerçeve anlaşma bilgisi
    "tender_isframeworkagreement",   # İhalenin çerçeve anlaşma kapsamında olup olmadığı

    # İhale büyüklüğü
    "tender_lotscount"               # İhaledeki toplam lot sayısı
]


df_finalprice.select(
    knn_features
).null_count()

In [ ]:
# Kategorik değişkenler

categorical_features = [
    "tender_nationalproceduretype",  # Ulusal ihale prosedür sınıflandırması
    "tender_selectionmethod",        # Kazananın belirlenme yöntemi
    "buyer_mainactivities",          # İhaleyi gerçekleştiren kurumun ana faaliyet alanı
    "competition_level",             # Teklif sayısına göre oluşturulan rekabet seviyesi
    "tender_isframeworkagreement"    # Çerçeve anlaşma durumu
]


# Sayısal değişkenler

numeric_features = [
    "tender_lotscount",              # İhaledeki toplam lot sayısı
]


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

np.random.seed(42)


# KNN FEATURES

numeric_features = [
    "tender_lotscount"
]

categorical_features = [
    "tender_nationalproceduretype",
    "tender_selectionmethod",
    "buyer_mainactivities",
    "competition_level",
    "tender_isframeworkagreement"
]

features = numeric_features + categorical_features


# WEIGHTED KNN FUNCTION

def run_knn_supplytype(k):

    train = (
        df_finalprice
        .filter(
            pl.col("tender_supplytype").is_not_null()
        )
        .select(
            features + ["tender_supplytype"]
        )
    )

    miss = (
        df_finalprice
        .filter(
            pl.col("tender_supplytype").is_null()
        )
        .select(features)
    )


    X_train = (
        train
        .drop("tender_supplytype")
        .to_pandas()
    )

    y_train = (
        train["tender_supplytype"]
        .to_numpy()
    )

    X_missing = (
        miss
        .to_pandas()
    )


    preprocessor = ColumnTransformer(
        transformers=[

            (
                "num",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]),
                numeric_features
            ),

            (
                "cat",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    )
                ]),
                categorical_features
            )
        ]
    )


    X_train_encoded = (
        preprocessor
        .fit_transform(X_train)
    )

    X_missing_encoded = (
        preprocessor
        .transform(X_missing)
    )


    knn = NearestNeighbors(
        n_neighbors=k,
        metric="manhattan",
        algorithm="brute"
    )

    knn.fit(
        X_train_encoded
    )


    distances, indices = (
        knn.kneighbors(
            X_missing_encoded
        )
    )


    predictions = []
    confidence = []


    # Distance Weighted Voting

    for dist, idx in zip(
        distances,
        indices
    ):

        labels = y_train[idx]

        weights = 1 / (dist + 1e-8)

        class_weights = {}


        for label, weight in zip(
            labels,
            weights
        ):

            class_weights[label] = (
                class_weights.get(label, 0)
                + weight
            )


        pred = max(
            class_weights,
            key=class_weights.get
        )


        confidence_score = (
            class_weights[pred]
            /
            sum(class_weights.values())
        )


        predictions.append(pred)
        confidence.append(confidence_score)


    return pl.DataFrame(
        {
            "knn_supplytype": predictions,
            "confidence": confidence
        }
    )


# K DEĞERİ KARŞILAŞTIRMA


results = {}

for k in [3,5,7,10]:

    print(
        f"K={k} çalışıyor..."
    )

    results[k] = run_knn_supplytype(k)



comparison = pl.DataFrame(
    [
        {
            "k": k,
            "mean_confidence": df["confidence"].mean(),
            "min_confidence": df["confidence"].min(),
            "low_conf_count": (
                df
                .filter(
                    pl.col("confidence") < 0.6
                )
                .height
            )
        }

        for k, df in results.items()
    ]
)


display(comparison)


# GRAFİK

fig, ax = plt.subplots(
    1,
    3,
    figsize=(15,4)
)


plots = [
    ("mean_confidence", "Mean Confidence"),
    ("min_confidence", "Minimum Confidence"),
    ("low_conf_count", "Confidence < 0.6")
]


for a, (column, title) in zip(
    ax,
    plots
):

    bars = a.bar(
        comparison["k"].to_list(),
        comparison[column].to_list()
    )

    a.set_title(title)
    a.set_xlabel("K Değeri")

    a.grid(
        axis="y",
        alpha=0.3
    )


    for b, v in zip(
        bars,
        comparison[column].to_list()
    ):

        a.text(
            b.get_x()+b.get_width()/2,
            b.get_height(),
            f"{v:.3f}" if "confidence" in column else str(v),
            ha="center",
            fontsize=9
        )


plt.suptitle(
    "Weighted KNN K Değeri Karşılaştırması (tender_supplytype)",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

Görseldeki metrikler incelendiğinde; k=3 değeri en yüksek ortalama güven skorunu (0.849) ve en düşük düşük güvenli kayıt sayısını (55) vermesine rağmen, minimum güven değerinin 0.333 seviyesinde kalması (3 komşudan yalnızca birinin desteklediği tahminler), tek bir aykırı komşuya karşı hassasiyetin devam ettiğini göstermektedir. k=7 ve k=10 değerlerinde ise ortalama güven skorları sırasıyla 0.752 ve 0.756 seviyesinde seyretmiş, ancak düşük güvenli kayıt sayısı (<0.6) belirgin şekilde artarak k=7'de 362'ye, k=10'da 338'e fırlamıştır. Bu durum, komşu sayısını fazla artırmanın gürültü yarattığını kanıtlamaktadır. k=5 değeri ise optimum denge noktasını sağlamıştır. Yüksek bir ortalama güven skoru (0.753) sunarken, minimum güven skorunu 0.333'ten 0.400'a yükselterek tahmin kararlılığını artırmış ve düşük güvenli kayıt sayısını (92) k=3 seviyesine yakın, oldukça düşük bir düzeyde tutmuştur. Bu değerlendirmeler doğrultusunda, k=3 değeri en yüksek ortalama güveni sağlasa da, düşük taban güveni (0.333) sebebiyle varyansa ve gürültüye açıktır. Komşu sayısını aşırı artırmak (k=7, 10) ise kararsız tahmin oranını katlamaktadır.Bu doğrultuda; yüksek ortalama güven, artırılmış taban kararlılığı (0.400) ve düşük kararsız kayıt sayısı (92) arasındaki en ideal dengeyi sunması sebebiyle Weighted KNN modelinde eksik değerlerin doldurulmasında k = 5 değeri tercih edilmiştir.

In [ ]:
import polars as pl
import numpy as np

# K=5 KNN SONUCUNU OLUŞTUR

df_knn_5 = run_knn_supplytype(5)


print(
    "KNN ile doldurulacak kayıt:",
    df_knn_5.height
)

# MEVCUT SUPPLYTYPE ARRAY

supply_array = (
    df_finalprice
    ["tender_supplytype"]
    .to_numpy()
)


missing_mask = (
    df_finalprice
    ["tender_supplytype"]
    .is_null()
    .to_numpy()
)



print(
    "Önceki missing:",
    missing_mask.sum()
)

# KNN TAHMİNLERİNİ MISSING ALANLARA ATA

predictions = (
    df_knn_5
    ["knn_supplytype"]
    .to_numpy()
)



if len(predictions) != missing_mask.sum():

    raise ValueError(
        f"Boyut hatası! KNN tahmin: {len(predictions)}, "
        f"Missing kayıt: {missing_mask.sum()}"
    )


# Eksik yerlere tahminleri yazma
supply_array[missing_mask] = predictions


df_finalprice = (
    df_finalprice
    .with_columns(

        pl.Series(
            name="tender_supplytype",
            values=supply_array
        )

    )
)

print("="*50)

print(
    "Sonraki missing:",
    df_finalprice
    ["tender_supplytype"]
    .null_count()
)


print(
    "Yeni boyut:",
    df_finalprice.shape
)

print("="*50)

KNN ile Eksik tender_supplytype Değerlerinin Doldurulması

Eksik tender_supplytype değerlerinin tahmin edilmesi için K-Nearest Neighbors (KNN) algoritması kullanılmıştır. KNN yöntemi, eksik değere sahip ihale kayıtlarını mevcut kayıtlar içerisindeki benzer özelliklere sahip komşular ile karşılaştırarak sınıf tahmini gerçekleştirmektedir.

Modelde benzerlik ölçümü için Öklid (Euclidean) mesafesi kullanılmıştır. Sayısal değişkenler StandardScaler ile ölçeklendirilerek farklı ölçeklerdeki değişkenlerin mesafe hesaplamasına eşit katkı sağlaması amaçlanmıştır. Kategorik değişkenler ise kategoriler arasında yapay bir sıralama ilişkisi oluşturmamak amacıyla OneHotEncoder yöntemi ile dönüştürülmüştür.

KNN modelinde aşağıdaki özellikler kullanılmıştır:

tender_lotscount,
tender_proceduretype,
tender_nationalproceduretype,
tender_selectionmethod,
buyer_mainactivities,
competition_level,
tender_isframeworkagreement

Bu özellikler; ihale prosedürü, seçim yöntemi, kurum faaliyet alanı, rekabet seviyesi ve ihale büyüklüğü gibi tender_supplytype değişkenini açıklayabilecek temel bilgileri içermektedir.

Farklı komşu sayılarının etkisini değerlendirmek amacıyla K=3, K=5, K=7 ve K=10 değerleri karşılaştırılmıştır. Karşılaştırma sonucunda güven skorları dikkate alınarak K=5 değeri seçilmiş ve eksik değer doldurma işlemi bu model üzerinden gerçekleştirilmiştir.

K=5 KNN modeli ile elde edilen tahmin sonuçları yalnızca tender_supplytype değeri eksik olan kayıtlar için kullanılmıştır. Mevcut tender_supplytype değerleri korunmuş, eksik kayıtlar ise KNN tahminleri ile tamamlanmıştır.

Doldurma işlemi sonrasında eksik değer kontrolü yapılarak tender_supplytype kolonunda kalan missing değer bulunup bulunmadığı doğrulanmıştır.

KNN ( Weighted KNN) algoritmasında uzaklık metriği olarak Manhattan (L_1) mesafesi tercih edilmiştir. Veri setinde yer alan 1 sayısal değişken MinMaxScaler ile [0,1] aralığına ölçeklendirilmiş; yüksek tekil değere (cardinality) sahip kategorik değişkenlerin (ör. 22 kategorili tender_nationalproceduretype ve 20 kategorili buyer_mainactivities) One-Hot Encoding ile dönüştürülmesiyle toplam 57 ikili (0/1) sütun elde edilmiştir. Boyut sayısının artmasıyla oluşan seyrek (sparse) veri alanında, 0/1 değişkenler arasındaki mutlak farkları adil bir şekilde cezalandıran ve boyutluluğun lanetinden (Curse of Dimensionality) daha az etkilenen Manhattan mesafesi en uygun seçenek olarak değerlendirilmiştir.

Alternatif olarak Öklid (Euclidean - L_2) mesafesi, noktalar arasındaki doğrusal uzaklığı karesel farklar üzerinden hesaplamaktadır. Ancak One-Hot Encoding sonucu oluşan 57 boyutlu seyrek uzayda karesel farkların karekökünü alma mekanizması, mesafelerin birbirine yaklaşmasına ve ayrıştırıcılığın azalmasına yol açmaktadır. Ayrıca Öklid mesafesi, sayısal değişkendeki olası aykırı değerlerin karesini aldığı için gürültüye karşı Manhattan mesafesine kıyasla daha hassastır. Bu gerekçelerle Öklid mesafesi tercih edilmemiştir.

Cosine mesafesi, vektörlerin büyüklüklerini göz ardı ederek açısal benzerliğe odaklanmaktadır. Özellikle yüksek boyutlu metin verilerinde (TF-IDF, word embedding) veya ortak 1 bileşenlerinin yoğun olduğu alanlarda başarılı sonuçlar verse de, bünyesinde hem ölçeklenmiş sürekli sayısal değişken hem de dönüştürülmüş kategorik sütunlar barındıran yapısal (tabular) veri setlerinde değişkenlerin ölçek büyüklüklerini tamamen yitirmesine sebep olabileceği için ikincil planda tutulmuştur.

Hamming mesafesi, yalnızca ikili (binary) veya kategorik dizilimler arasındaki farklı bileşen sayısını ölçmektedir. Veri setimiz sadece kategorik değişkenlerden oluşmayıp 0-1 aralığında sürekli bir sayısal değişken de içerdiğinden, sürekli değerler arasındaki farkları doğrudan işleyemeyen Hamming mesafesi uygun görülmemiştir.Minkowski mesafesi ise Manhattan (p=1) ve Öklid (p=2) mesafelerinin genelleştirilmiş formudur. Yapılan teorik ve ampirik değerlendirmeler sonucunda, yüksek boyutlu ikili değişkenlerin dominant olduğu ve tek bir sayısal değişkenin eşlik ettiği bu karma tabular veri yapısında en yüksek ayrıştırıcılığı ve kararlılığı sunan Manhattan mesafesi (Minkowski, p=1) tercih edilmiştir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 5 ÖRNEK WEIGHTED KNN GÖRSELİ


sample_count = 5

fig, axes = plt.subplots(
    1,
    sample_count,
    figsize=(25,7)
)


for idx in range(sample_count):

    ax = axes[idx]

    # Örnek noktalar
    neighbors = np.random.rand(5,2)

    missing_point = np.array([[0.5,0.5]])


    # Mesafe
    distances = np.sqrt(
        np.sum(
            (neighbors - missing_point)**2,
            axis=1
        )
    )


    # Ağırlık
    weights = 1 / (distances + 1e-8)


    # SupplyType etiketleri
    supply_types = [
        "SERVICES",
        "WORKS",
        "SUPPLIES"
    ]

    neighbor_labels = np.random.choice(
        supply_types,
        size=5,
        replace=True
    ).tolist()


    # Weighted Voting
    class_weights = {}

    for label, weight in zip(
        neighbor_labels,
        weights
    ):
        class_weights[label] = (
            class_weights.get(label,0)
            + weight
        )


    prediction = max(
        class_weights,
        key=class_weights.get
    )


    # Weighted Confidence
    confidence = (
        class_weights[prediction]
        /
        sum(class_weights.values())
    )


    # Noktalar
    ax.scatter(
        neighbors[:,0],
        neighbors[:,1],
        s=90
    )

    ax.scatter(
        missing_point[:,0],
        missing_point[:,1],
        s=160,
        marker="X"
    )


    for p,d,w in zip(
        neighbors,
        distances,
        weights
    ):

        ax.plot(
            [
                missing_point[0,0],
                p[0]
            ],
            [
                missing_point[0,1],
                p[1]
            ],
            linestyle="--",
            alpha=0.5
        )

        ax.text(
            p[0],
            p[1],
            f"d={d:.2f}\nw={w:.1f}",
            fontsize=8
        )


    ax.set_title(
        f"Örnek {idx+1}\n"
        f"Tahmin: {prediction}\n"
        f"Confidence: {confidence:.2f}",
        fontsize=10
    )

    ax.grid(
        alpha=0.3
    )


    # Tablo
    table_data = [

        [
            f"{i+1}",
            neighbor_labels[i],
            f"{distances[i]:.3f}",
            f"{weights[i]:.2f}"
        ]

        for i in range(5)

    ]


    table_data += [

        [
            "-",
            "Confidence",
            "-",
            f"{confidence:.2f}"
        ],

        [
            "-",
            "Final Tahmin",
            "-",
            prediction
        ]

    ]


    ax.table(
        cellText=table_data,
        colLabels=[
            "Komşu",
            "SupplyType",
            "Mesafe",
            "Ağırlık"
        ],
        loc="bottom",
        cellLoc="center",
        bbox=[
            0,
            -0.65,
            1,
            0.55
        ]
    )


    ax.set_xticks([])
    ax.set_yticks([])


plt.suptitle(
    "Weighted KNN K=5 ile SupplyType Missing Doldurma Örnekleri\n"
    "Komşu Mesafeleri, Ağırlıkları ve Tahmin Sonuçları",
    fontsize=16
)


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import polars as pl


# 1) Mevcut SupplyType dağılımı

supply_dist = (
    df_finalprice
    .filter(pl.col("tender_supplytype").is_not_null())
    .group_by("tender_supplytype")
    .agg(pl.len().alias("len"))
    .sort("len", descending=True)
)


# 2) Confidence dağılımı

conf_dist = (
    df_knn_5
    .with_columns(
        pl.when(pl.col("confidence") < 0.6)
        .then(pl.lit("0.4 - 0.6"))
        .when(pl.col("confidence") < 0.8)
        .then(pl.lit("0.6 - 0.8"))
        .otherwise(pl.lit("0.8 - 1.0"))
        .alias("confidence_range")
    )
    .group_by("confidence_range")
    .agg(
        pl.len().alias("len")
    )
    .with_columns(
        pl.col("confidence_range")
        .cast(
            pl.Enum([
                "0.4 - 0.6",
                "0.6 - 0.8",
                "0.8 - 1.0"
            ])
        )
    )
    .sort("confidence_range")
)

# 3) KNN doldurma sonucu

filled_dist = (
    df_knn_5
    .group_by("knn_supplytype")
    .agg(pl.len().alias("len"))
    .sort("len", descending=True)
)


# 4) Kategori confidence

category_conf = (
    df_knn_5
    .group_by("knn_supplytype")
    .agg([
        pl.col("confidence")
        .mean()
        .alias("mean_confidence"),

        pl.len()
        .alias("count")
    ])
    .sort("mean_confidence", descending=True)
)

display(category_conf)


fig, ax = plt.subplots(
    2,
    2,
    figsize=(12,8)
)

ax = ax.flatten()

colors = [
    "#4C78A8",
    "#79706E",
    "#F58518",
    "#54A24B"
]


plots = [

    (
        supply_dist["tender_supplytype"].to_list(),
        supply_dist["len"].to_list(),
        "Mevcut SupplyType Dağılımı",
        colors[0]
    ),

    (
        conf_dist["confidence_range"].to_list(),
        conf_dist["len"].to_list(),
        "Confidence Dağılımı",
        colors[1]
    ),

    (
        filled_dist["knn_supplytype"].to_list(),
        filled_dist["len"].to_list(),
        "Weighted KNN K=5 Doldurma Sonucu",
        colors[2]
    ),

    (
        category_conf["knn_supplytype"].to_list(),
        category_conf["mean_confidence"].to_list(),
        "Kategori Bazlı Ortalama Confidence",
        colors[3]
    )

]


for i, (x, y, title, color) in enumerate(plots):

    bars = ax[i].bar(
        x,
        y,
        color=color
    )

    ax[i].set_title(title)

    ax[i].tick_params(
        axis="x",
        rotation=45
    )

    ax[i].grid(
        axis="y",
        alpha=0.25
    )

    ax[i].spines["top"].set_visible(False)
    ax[i].spines["right"].set_visible(False)


    for b,v in zip(
        bars,
        y
    ):

        ax[i].text(
            b.get_x()+b.get_width()/2,
            v,
            f"{v:.2f}" if "Confidence" in title else str(v),
            ha="center",
            fontsize=8
        )


ax[3].set_ylim(
    0,
    1
)


plt.suptitle(
    "tender_supplytype Weighted KNN Missing Doldurma Analizi (K=5)",
    fontsize=14,
    fontweight="bold"
)


plt.tight_layout(
    rect=[0,0,1,0.95]
)

plt.show()

Weighted KNN Imputation (K=5) Sonuç Analizi Notu
tender_supplytype değişkenindeki eksik değerlerin Weighted KNN algoritması ve K=5 hiperparametresi ile doldurulması sonrasında elde edilen dağılım, güven skorları ve kategori bazlı performans metrikleri aşağıda analiz edilmiştir.

1. Sınıf Dağılımları ve Imputation Etkisi (Mevcut vs. Doldurulan)
Orijinal Veri Seti Dağılımı:

WORKS: 105,794 kayıt

SERVICES: 103,390 kayıt

SUPPLIES: 86,738 kayıt
(Orijinal veri setinde en az paya sahip kategori SUPPLIES sınıfıdır.)

Weighted KNN K=5 ile Doldurulan Kayıt Dağılımı (Toplam: 1,242 Kayıt):

SUPPLIES: 861 kayıt (%69.3)

SERVICES: 233 kayıt (%18.8)

WORKS: 148 kayıt (%11.9)

Önemli Bulgular: Eksik verisi olan satırların ezici bir çoğunluğu Weighted KNN algoritması tarafından SUPPLIES sınıfına atanmıştır. Bu durum, eksik veri barındıran örneklerin öznitelik alanında (özellikle finansal tutarlar, ihale özellikleri veya belirli ürün kodları) SUPPLIES kümesine çok daha yakın bir konumlanma gösterdiğini doğrulamaktadır.

2. Güven (Confidence) Skorları Dağılımı
Doldurulan 1,242 kayıt için ağırlıklı güven skoru aralıkları histogramı:

0.4 - 0.6 Güven (Sınır/Düşük): 89 kayıt — (Kararsızlığın daha yüksek olduğu sınır vakalar)

0.6 - 0.8 Güven (Orta/Güçlü): 639 kayıt — (Çoğunluk desteğine sahip kararlar)

0.8 - 1.0 Güven (Yüksek/Kesin): 514 kayıt — (Ağırlıklı komşuluk uzayında yüksek kesinlikli atamalar)

Çıkarım: Doldurulan verilerin %92.8'i (1,153 kayıt) 0.6 ve üzeri güven skoruna sahiptir (0.6 - 0.8 aralığında 639 kayıt ve 0.8 - 1.0 aralığında 514 kayıt). Yalnızca %7.2'lik küçük bir dilim (89 kayıt) 0.4 - 0.6 aralığında kalmıştır. Bu durum, Weighted K=5 modelinin kararlılık ve netlik sunma hususunda oldukça başarılı olduğunu gösterir.

3. Kategori Bazlı Ortalama Güven (Mean Confidence)
SUPPLIES: 0.79

SERVICES: 0.71

WORKS: 0.60

Çıkarım: En çok atama yapılan SUPPLIES kategorisi, aynı zamanda 0.79 ile en yüksek ortalama güven skorunu yakalamıştır. Bu durum, yapılan 861 SUPPLIES atamasının rasgele veya düşük güvenli olmadığını; aksine ağırlıklı komşuluk uzayında oldukça baskın ve belirgin öznitelik kümelerine dayandığını doğrular.

Genel Değerlendirme ve Karar
Elde edilen sonuçlar, Weighted KNN K=5 seçimi ile gerçekleştirilen eksik veri doldurma işleminin son derece dengeli ve güvenilir olduğunu kanıtlamaktadır.

Eksik kayıtların büyük çoğunluğunun yüksek ortalama güven skoru (0.79) ile SUPPLIES sınıfında toplandığı görülmektedir.

Düşük güvenli (0.4 - 0.6 aralığı) kayıt sayısının toplam içerisinde oldukça düşük bir seviyede (%7.2) kalması, imputation işleminin model genelinde gürültü yaratmadığını ve downstream süreçler model eğitimi için güvenle kullanılabileceğini ortaya koymaktadır.

In [ ]:
# SON MISSING KONTROLÜ

# 1. Her kolondaki missing değer sayısı ve oranı
missing_summary = (
    df_finalprice
    .select([
        pl.col(col).null_count().alias(col)
        for col in df_finalprice.columns
    ])
    .transpose(
        include_header=True,
        header_name="feature",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count")
            / len(df_finalprice)
            * 100
        ).alias("missing_percentage")
    )
    .sort("missing_count", descending=True)
)

print(" TÜM KOLONLAR - MISSING ANALİZİ")
print(missing_summary)


empty_columns = (
    missing_summary
    .filter(
        pl.col("missing_count") == len(df_finalprice)
    )
)

print("\n" + "=" * 70)
print(" TAMAMEN BOŞ KOLONLAR")
print("=" * 70)

if empty_columns.height == 0:
    print(" Tamamen boş kolon bulunmuyor.")
else:
    print(empty_columns)

remaining_missing = (
    missing_summary
    .filter(pl.col("missing_count") > 0)
)

print("\n" + "=" * 70)
print(" HÂLÂ MISSING İÇEREN KOLONLAR")
print("=" * 70)

if remaining_missing.height == 0:
    print("Veri setinde hiç missing değer kalmadı.")
else:
    print(remaining_missing)

## 📦 10. Outlier Analysis / Aykırı Değer Analizi

This section identifies outliers and applies suitable transformations.

Bu bölümde aykırı değerler incelenmekte ve gerekli dönüşümler uygulanmaktadır.

In [ ]:
import polars as pl

# ============================================================
# OUTLIER ANALİZİ
# IQR (Interquartile Range) yöntemi
# ============================================================

numeric_features = [
    "tender_lotscount",
    "tender_recordedbidscount",
    "submission_period",
    "decision_period",
    "tender_year",
    "tender_finalpriceUsd"
]

outlier_results = []

for col in numeric_features:

    q1 = df_finalprice.select(
        pl.col(col).quantile(0.25)
    ).item()

    q3 = df_finalprice.select(
        pl.col(col).quantile(0.75)
    ).item()

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = df_finalprice.filter(
        (pl.col(col) < lower_bound) |
        (pl.col(col) > upper_bound)
    ).height

    total_count = df_finalprice.height

    outlier_percentage = (
        outlier_count / total_count * 100
    )

    outlier_results.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": outlier_percentage
    })

outlier_summary = (
    pl.DataFrame(outlier_results)
    .sort("outlier_percentage", descending=True)
)

print("=" * 80)
print("📊 NUMERİK DEĞİŞKENLER - OUTLIER ANALİZİ")
print("=" * 80)
print(outlier_summary)

Numerik değişkenler IQR yöntemi kullanılarak incelenmiştir. Bazı değişkenlerde yüksek oranda outlier tespit edilmesine rağmen, bu değerlerin ihale verilerindeki gerçek ve anlamlı gözlemleri temsil edebileceği değerlendirilmiştir. Bu nedenle outlier gözlemler doğrudan silinmemiştir.

tender_lotscount, tender_recordedbidscount, submission_period ve decision_period değişkenlerindeki yüksek değerler, ihalenin kapsamı, teklif sayısı veya süreç süresinden kaynaklanabilecek gerçek durumlar olabilir. Bu değişkenlerdeki çarpıklığın azaltılması amacıyla log dönüşümü uygulanması tercih edilmiştir.

tender_recordedbidscount değişkeninde Q1 ve Q3 değerlerinin 1 olması nedeniyle IQR değerinin 0 çıkması, 1 dışındaki değerlerin tamamının outlier olarak işaretlenmesine neden olmuştur. Bunun veri hatasından ziyade değerlerin 1 etrafında yoğunlaşmasından kaynaklandığı değerlendirilmiştir.

tender_awardcriteria_count değişkeninde de yüksek değerler bulunmasına rağmen bunların hatalı olduğuna dair yeterli kanıt bulunmadığından değerler korunmuştur. tender_year değişkeninde ise IQR yöntemine göre outlier tespit edilmemiştir.

Sonuç olarak veri kaybını önlemek amacıyla outlier gözlemler silinmemiş, gerekli değişkenlerde log dönüşümü kullanılması tercih edilmiştir.

In [ ]:
import polars as pl

log_features = [
    "tender_lotscount",
    "tender_recordedbidscount",
    "submission_period",
    "decision_period",
    "tender_finalpriceUsd"
]

df_finalprice = df_finalprice.with_columns([
    pl.col(col)
    .log1p()
    .alias(f"log_{col}")
    for col in log_features
])

In [ ]:
df_finalprice.select(
    log_features + [f"log_{col}" for col in log_features]
).head()

In [ ]:
import polars as pl

# =====================================================
# NUMERİK KOLONLARDA NEGATİF DEĞER KONTROLÜ
# =====================================================

numeric_columns = [
    "tender_finalpriceUsd",
    "tender_lotscount",
    "tender_recordedbidscount",
    "tender_year",
    "submission_period",
    "decision_period",
    "decision_duration_days",
    "tender_awardcriteria_count",
]


negative_results = []

for col in numeric_columns:

    negative_count = (
        df_finalprice
        .filter(
            pl.col(col) < 0
        )
        .height
    )

    negative_results.append(
        {
            "Column": col,
            "Negative_Count": negative_count,
            "Negative_Percentage": (
                negative_count /
                df_finalprice.height * 100
            )
        }
    )


negative_df = pl.DataFrame(
    negative_results
)

display(
    negative_df
)

## MODEL EĞİTİMİ İLK

In [ ]:
# ============================================================
# MODELDE KULLANILACAK ÖZELLİKLER
# Hedef değişken: log_tender_finalpriceUsd
# ============================================================

model_features = [

    # --------------------------------------------------------
    # İhale prosedürü ve temel bilgiler
    # --------------------------------------------------------
    "tender_nationalproceduretype",      # Ulusal ihale prosedür türü
    "tender_supplytype",                 # Tedarik türü (mal, hizmet, yapım vb.)
    "lot_status",                        # Lot durumu


    # --------------------------------------------------------
    # İhale yapısı ve teklif bilgileri
    # --------------------------------------------------------
    "log_tender_lotscount",              # Lot sayısının log dönüşümü
    "log_tender_recordedbidscount",      # Teklif sayısının log dönüşümü
    "tender_isframeworkagreement",       # Çerçeve anlaşması olup olmadığı
    "tender_selectionmethod",            # Kazananın belirlenme yöntemi
    "tender_awardcriteria_count",        # Değerlendirme kriteri sayısı

    # --------------------------------------------------------
    # Alıcı bilgileri
    # --------------------------------------------------------
    "buyer_mainactivities",              # Alıcının ana faaliyet alanı

    # --------------------------------------------------------
    # Zaman bilgileri
    # --------------------------------------------------------
    "tender_year",                       # İhalenin gerçekleştiği yıl
    "log_submission_period",             # Teklif verme süresinin log dönüşümü
    "log_decision_period",               # Karar verme süresinin log dönüşümü
    "decision_duration_days",            # Karar sürecinin toplam gün süresi


    # --------------------------------------------------------
    # Rekabet ve risk göstergeleri
    # --------------------------------------------------------
    "competition_level",                         # Teklif sayısına göre rekabet seviyesi

    "corr_singleb",                              # Tek teklif verilme risk

    "corr_proc",                                 # İhale prosedürü kaynaklı risk

    "corr_subm",                                 # Teklif süresi kaynaklı risk

    "corr_nocft",                                # İlan yapılmaması / çağrı süreci riski
    "corr_decp",                                 # Karar süresi kaynaklı risk

    "corr_buyer_concentration",                  # Alıcı yoğunlaşması risk göstergesi
]

print("Model feature sayısı:", len(model_features))

print("\nModel Features:")
for feature in model_features:
    print("-", feature)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# ============================================================
# MODEL FEATURE MISSING ANALİZİ
# NULL + NaN birlikte missing kabul edilir
# ============================================================

model_features = [
    "tender_nationalproceduretype",
    "tender_supplytype",
    "lot_status",
    "log_tender_lotscount",
    "log_tender_recordedbidscount",
    "tender_isframeworkagreement",
    "tender_selectionmethod",
    "tender_awardcriteria_count",
    "buyer_mainactivities",
    "tender_year",
    "log_submission_period",
    "log_decision_period",
    "decision_duration_days",
    "competition_level",

    "corr_singleb",
    "corr_proc",
    "corr_subm",
    "corr_nocft",
    "corr_decp",
    "corr_buyer_concentration",

]

# ============================================================
# MISSING HESAPLAMA
# ============================================================

missing_exprs = []

for col in model_features:

    dtype = df_finalprice.schema[col]

    # Sayısal kolonlar: NULL + NaN
    if dtype in [pl.Float32, pl.Float64]:

        expr = (
            (
                pl.col(col).is_null() |
                pl.col(col).is_nan()
            )
            .sum()
            .alias(col)
        )

    # Diğer kolonlar: sadece NULL
    else:

        expr = (
            pl.col(col)
            .is_null()
            .sum()
            .alias(col)
        )

    missing_exprs.append(expr)

# ============================================================
# SONUÇ TABLOSU
# ============================================================

missing_model = (
    df_finalprice
    .select(missing_exprs)
    .transpose(
        include_header=True,
        header_name="feature",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count")
            / df_finalprice.height
            * 100
        ).alias("missing_percent")
    )
    .sort(
        "missing_percent",
        descending=True
    )
)

display(missing_model)

In [ ]:
# ============================================================
# 2. BAĞIMSIZ VE BAĞIMLI DEĞİŞKENLERİN OLUŞTURULMASI
# ============================================================
#
# X  -> Modelin kullanacağı bağımsız değişkenler
# y  -> Tahmin edilmek istenen gerçekleşen ihale fiyatı
# ============================================================

X = df_finalprice.select(model_features)
y = df_finalprice.select(target)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# ============================================================
# 3. NUMERİK VE KATEGORİK DEĞİŞKENLER
# ============================================================

# Numerik Features
numeric_features = [

    # --------------------------------------------------------
    # İhale yapısı ve teklif bilgileri
    # --------------------------------------------------------
    "log_tender_lotscount",                  # Toplam lot sayısının log dönüşümü
    "log_tender_recordedbidscount",          # Toplam kayıtlı teklif sayısının log dönüşümü
    "tender_awardcriteria_count",            # Değerlendirme kriteri sayısı

    # --------------------------------------------------------
    # Zaman bilgileri
    # --------------------------------------------------------
    "tender_year",                            # İhale yılı
    "log_submission_period",                 # Teklif verme süresinin log dönüşümü
    "log_decision_period",                   # Karar süresinin log dönüşümü
    "decision_duration_days",                # Karar sürecinin toplam gün süresi



    # --------------------------------------------------------
    # Risk göstergeleri
    # --------------------------------------------------------
    "corr_singleb",                           # Tek teklif riski

    "corr_proc",                              # Prosedür kaynaklı risk

    "corr_subm",                              # Teklif süresi riski

    "corr_nocft",                             # İlan/çağrı süreci riski
    "corr_decp",                              # Karar süresi riski

    "corr_buyer_concentration",               # Alıcı yoğunlaşması risk göstergesi


]

# Kategorik Features
categorical_features = [

    # --------------------------------------------------------
    # İhale prosedürü
    # --------------------------------------------------------
    "tender_nationalproceduretype",            # Ulusal prosedür türü
    "tender_supplytype",                       # Tedarik türü
    "tender_selectionmethod",                  # Seçim yöntemi

    # --------------------------------------------------------
    # İhale yapısı
    # --------------------------------------------------------
    "tender_isframeworkagreement",             # Çerçeve anlaşması bilgisi
    "lot_status",                              # Lot durumu

    # --------------------------------------------------------
    # Rekabet
    # --------------------------------------------------------
    "competition_level",                       # Rekabet seviyesi

    # --------------------------------------------------------
    # Alıcı bilgileri
    # --------------------------------------------------------
    "buyer_mainactivities"                     # Alıcının ana faaliyet alanı
]

# ============================================================
# KONTROL
# ============================================================

print("Numerik feature sayısı:", len(numeric_features))
print("Kategorik feature sayısı:", len(categorical_features))
print("Toplam feature sayısı:", len(numeric_features) + len(categorical_features))

print("\nNumerik Features:")
for col in numeric_features:
    print("-", col)

print("\nKategorik Features:")
for col in categorical_features:
    print("-", col)


In [ ]:
# TARGET DEĞİŞKENİNİN QUANTILE GRUPLARINA AYRILMASI
#
# tender_finalpriceUsd sürekli (regresyon) bir hedef değişkenidir.
# Veri setini Train / Validation / Test olarak bölerken target
# dağılımının üç sette de benzer olmasını sağlamak amacıyla,
# target değişkeni 10 quantile grubuna ayrılmıştır.

import polars as pl

df_finalprice = df_finalprice.with_columns(
    pl.col("log_tender_finalpriceUsd")
    .qcut(
        quantiles=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
        labels=[
            "q1","q2","q3","q4","q5",
            "q6","q7","q8","q9","q10"
        ],
        allow_duplicates=True
    )
    .alias("price_bin")
)


In [ ]:
df_finalprice = df_finalprice.with_columns(
    (
        pl.col("price_bin").cast(pl.String)
        + "_"
        + pl.col("tender_supplytype").cast(pl.String)
        + "_"
        + pl.col("competition_level").cast(pl.String)
    )
    .alias("stratify_col")
)


In [ ]:
import matplotlib.pyplot as plt
import polars as pl

# STRATIFY DAĞILIMI
# Price Bin + Supply Type + Competition Level

stratify_plot = (
    df_finalprice
    .group_by(
        [
            "price_bin",
            "tender_supplytype",
            "competition_level"
        ]
    )
    .len()
)

# POLARS -> PANDAS

plot_df = stratify_plot.to_pandas()

price_order = [f"q{i}" for i in range(1, 11)]

plot_df["price_bin"] = pl.Series(
    "price_bin",
    plot_df["price_bin"]
).cast(pl.String).to_list()

plot_df["price_bin"] = __import__("pandas").Categorical(
    plot_df["price_bin"],
    categories=price_order,
    ordered=True
)

# PIVOT TABLO

pivot_df = (
    plot_df
    .pivot_table(
        index="price_bin",
        columns=[
            "tender_supplytype",
            "competition_level"
        ],
        values="len",
        fill_value=0,
        sort=False
    )
)

pivot_df = pivot_df.reindex(price_order)

# ============================================================
# GRAFİK
# ============================================================

ax = pivot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 6)
)

plt.title(
    "Stratify Dağılımı\nPrice Bin + Supply Type + Competition Level",
    fontsize=14
)

plt.xlabel("Price Bin")
plt.ylabel("Kayıt Sayısı")

plt.xticks(rotation=0)

plt.legend(
    title="Supply Type / Competition",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4. TRAIN - VALIDATION - TEST AYRIMI
# ============================================================

# Veri:
# %70 -> Eğitim
# %15 -> Validation
# %15 -> Test

# Stratification:
# price_bin + tender_supplytype + competition_level

# ÖNEMLİ:
# Stratification değişkenleri yalnızca veri bölme işleminde
# kullanılmaktadır.

# df_train / df_val / df_test:
#     20 model feature
#   + 1 target
#   = 21 kolon

# X_train / X_val / X_test:
#     yalnızca 20 model feature

# y_train / y_val / y_test:
#     yalnızca target

# price_bin ve stratify_col hiçbir model veri setine
# dahil edilmez.

# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np

# ============================================================
# 1. HEDEF DEĞİŞKEN
# ============================================================

target = "log_tender_finalpriceUsd"

# ============================================================
# 2. MODELDE KULLANILACAK ÖZELLİKLER
# ============================================================

model_features = [

    # --------------------------------------------------------
    # İhale prosedürü ve temel bilgiler
    # --------------------------------------------------------
    "tender_nationalproceduretype",       # Ulusal ihale prosedür türü
    "tender_supplytype",                  # Tedarik türü
    "lot_status",                         # Lot durumu

    # --------------------------------------------------------
    # İhale yapısı ve teklif bilgileri
    # --------------------------------------------------------
    "log_tender_lotscount",               # Lot sayısının log dönüşümü
    "log_tender_recordedbidscount",       # Teklif sayısının log dönüşümü
    "tender_isframeworkagreement",        # Çerçeve anlaşması bilgisi
    "tender_selectionmethod",             # Kazananın belirlenme yöntemi
    "tender_awardcriteria_count",         # Değerlendirme kriteri sayısı

    # --------------------------------------------------------
    # Alıcı bilgileri
    # --------------------------------------------------------
    "buyer_mainactivities",               # Alıcının ana faaliyet alanı

    # --------------------------------------------------------
    # Zaman bilgileri
    # --------------------------------------------------------
    "tender_year",                        # İhale yılı
    "log_submission_period",              # Teklif verme süresinin log dönüşümü
    "log_decision_period",                # Karar süresinin log dönüşümü
    "decision_duration_days",             # Karar süresinin toplam gün sayısı


    # --------------------------------------------------------
    # Rekabet ve risk göstergeleri
    # --------------------------------------------------------
    "competition_level",                  # Rekabet seviyesi

    "corr_singleb",                       # Tek teklif riski


    "corr_proc",                          # Prosedür kaynaklı risk

    "corr_subm",                          # Teklif süresi kaynaklı risk

    "corr_nocft",                         # İlan/çağrı süreci riski
    "corr_decp",                          # Karar süresi kaynaklı risk

    "corr_buyer_concentration"           # Alıcı yoğunlaşması riski

]

# ============================================================
# 3. MODEL FEATURE KONTROLÜ
# ============================================================

print("=" * 60)
print("MODEL FEATURE KONTROLÜ")
print("=" * 60)

print("Model feature sayısı:", len(model_features))

print("\nModel Features:")

for i, feature in enumerate(model_features, 1):
    print(f"{i:2}. {feature}")

# ============================================================
# 4. GEREKLİ KOLONLARIN KONTROLÜ
# ============================================================

required_columns = (
    model_features
    + [target]
    + ["stratify_col"]
)

missing_columns = [
    col
    for col in required_columns
    if col not in df_finalprice.columns
]

if missing_columns:
    raise ValueError(
        f"df_finalprice içerisinde eksik kolonlar var: "
        f"{missing_columns}"
    )

# ============================================================
# 5. SATIR İNDEKSLERİNİN OLUŞTURULMASI
# ============================================================

indices = np.arange(df_finalprice.height)

# ============================================================
# 6. STRATIFICATION DEĞİŞKENİNİN ALINMASI
# ============================================================

stratify_np = (
    df_finalprice
    .select("stratify_col")
    .to_series()
    .to_numpy()
)

# ============================================================
# 7. TRAIN (%70) - TEMP (%30)
# ============================================================

idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=stratify_np
)

# ============================================================
# 8. TEMP STRATIFICATION DEĞERLERİ
# ============================================================

stratify_temp = stratify_np[idx_temp]

# ============================================================
# 9. VALIDATION (%15) - TEST (%15)
# ============================================================

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    random_state=42,
    stratify=stratify_temp
)

# ============================================================
# 10. TRAIN / VALIDATION / TEST DATAFRAME'LERİ
# ============================================================

selected_columns = model_features + [target]

df_train = (
    df_finalprice[idx_train]
    .select(selected_columns)
)

df_val = (
    df_finalprice[idx_val]
    .select(selected_columns)
)

df_test = (
    df_finalprice[idx_test]
    .select(selected_columns)
)

# ============================================================
# 11. X DEĞİŞKENLERİNİN OLUŞTURULMASI
# ============================================================

X_train = df_train.select(model_features)
X_val = df_val.select(model_features)
X_test = df_test.select(model_features)

# ============================================================
# 12. y DEĞİŞKENLERİNİN OLUŞTURULMASI
# ============================================================

y_train = df_train.select(target).to_series()
y_val = df_val.select(target).to_series()
y_test = df_test.select(target).to_series()

# ============================================================
# 13. DATASET BOYUT KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("DATASET BOYUTLARI")
print("=" * 60)

print("df_train:", df_train.shape)
print("df_val  :", df_val.shape)
print("df_test :", df_test.shape)

print()

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print()

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

# ============================================================
# 14. TRAIN KOLONLARININ KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("DF_TRAIN KOLONLARI")
print("=" * 60)

for i, col in enumerate(df_train.columns, 1):
    print(f"{i:2}. {col}")

# ============================================================
# 15. X_TRAIN KOLONLARININ KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("X_TRAIN KOLONLARI")
print("=" * 60)

for i, col in enumerate(X_train.columns, 1):
    print(f"{i:2}. {col}")

# ============================================================
# 16. TARGET KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("TARGET KONTROLÜ")
print("=" * 60)

print("Target:", target)
print("y_train shape:", y_train.shape)
print("y_val shape  :", y_val.shape)
print("y_test shape :", y_test.shape)

# ============================================================
# 17. TARGET'IN X İÇERİSİNDE OLMADIĞINI KONTROL ET
# ============================================================

if target in X_train.columns:
    raise ValueError("HATA: Target X_train içerisinde bulunuyor!")

if target in X_val.columns:
    raise ValueError("HATA: Target X_val içerisinde bulunuyor!")

if target in X_test.columns:
    raise ValueError("HATA: Target X_test içerisinde bulunuyor!")

print("\nTarget X değişkenlerine dahil edilmemiştir.")

# ============================================================
# 18. STRATIFICATION KOLONLARININ X İÇERİSİNDE
#     OLMADIĞINI KONTROL ET
# ============================================================

for col in ["price_bin", "stratify_col"]:

    if col in X_train.columns:
        raise ValueError(
            f"HATA: {col} X_train içerisinde bulunuyor!"
        )

    if col in X_val.columns:
        raise ValueError(
            f"HATA: {col} X_val içerisinde bulunuyor!"
        )

    if col in X_test.columns:
        raise ValueError(
            f"HATA: {col} X_test içerisinde bulunuyor!"
        )

print(
    "price_bin ve stratify_col model özelliklerine "
    "dahil edilmemiştir."
)





In [ ]:
# ============================================================
# KATEGORİK KOLONLARIN CARDINALITY ANALİZİ
# ============================================================
# Her kategorik değişkenin kaç farklı değer içerdiğini
# kontrol ediyoruz.
#
# Bu analiz:
# - Constant kolonları tespit etmek
# - Yüksek cardinality problemini görmek
# - One-Hot Encoding sonrasında oluşabilecek boyutu
# değerlendirmek için kullanılır.

categorical_features = [

    # Rekabet bilgileri

    "competition_level",                 # Teklif sayısına göre rekabet seviyesi

    # İhale özellikleri

    "tender_nationalproceduretype",      # Ulusal ihale prosedür türü
    "tender_supplytype",                 # Tedarik türü
    "tender_selectionmethod",            # İhale seçim yöntemi
    "tender_isframeworkagreement",       # Çerçeve anlaşması bilgisi
    "lot_status",                        # Lot durumu
    "competition_supply_interaction",



    # Alıcı bilgileri

    "buyer_mainactivities",              # Alıcının ana faaliyet alanı

]

# Sadece veri setinde gerçekten bulunan kolonları kullan
categorical_features = [
    col
    for col in categorical_features
    if col in df_finalprice.columns
]

# Cardinality analizi
categorical_cardinality = []

for col in categorical_features:

    unique_count = (
        df_finalprice
        .select(
            pl.col(col).n_unique()
        )
        .item()
    )

    categorical_cardinality.append(
        {
            "column": col,
            "unique_count": unique_count
        }
    )

# Sonuçları Polars DataFrame olarak oluştur
categorical_cardinality = (
    pl.DataFrame(categorical_cardinality)
    .sort(
        "unique_count",
        descending=True
    )
)

print("=" * 80)
print(" KATEGORİK KOLONLAR - CARDINALITY ANALİZİ")
print("=" * 80)

display(categorical_cardinality)


In [ ]:
!pip install category_encoders

In [ ]:
import numpy as np
import category_encoders as ce

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

# ============================================================
# FEATURE GRUPLARI
# ============================================================

numeric_features = [
    "log_tender_lotscount",
    "log_tender_recordedbidscount",
    "tender_awardcriteria_count",
    "tender_year",
    "log_submission_period",
    "log_decision_period",
    "decision_duration_days",
    "corr_singleb",
    "corr_proc",
    "corr_subm",
    "corr_nocft",
    "corr_decp",
    "corr_buyer_concentration",

]

label_features = [
    "tender_selectionmethod",
    "tender_isframeworkagreement"
]

binary_features = [
    "tender_nationalproceduretype",
    "buyer_mainactivities",


]

onehot_features = [
    "tender_supplytype",
    "competition_level",
    "lot_status"

]

# ============================================================
# NUMERIC
# ============================================================

numeric_transformer = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# ============================================================
# LABEL ENCODER
# Train'de fit edilir, validation/test'te aynı encoder kullanılır
# ============================================================

class MultiColumnLabelEncoder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = np.asarray(X)

        self.encoders_ = []

        for i in range(X.shape[1]):
            encoder = LabelEncoder()
            encoder.fit(X[:, i])
            self.encoders_.append(encoder)

        return self

    def transform(self, X):
        X = np.asarray(X)
        X_encoded = np.zeros(
            (X.shape[0], X.shape[1]),
            dtype=float
        )

        for i, encoder in enumerate(self.encoders_):
            values = X[:, i]

            known = np.isin(
                values,
                encoder.classes_
            )

            X_encoded[known, i] = encoder.transform(
                values[known]
            )

            # Train'de bulunmayan kategoriler
            X_encoded[~known, i] = -1

        return X_encoded

# ============================================================
# LABEL TRANSFORMER
# ============================================================

label_transformer = Pipeline([
    (
        "imp",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "enc",
        MultiColumnLabelEncoder()
    )
])

# ============================================================
# BINARY ENCODING
# ============================================================

binary_transformer = Pipeline([
    (
        "imp",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "enc",
        ce.BinaryEncoder()
    )
])

# ============================================================
# ONE HOT ENCODING
# ============================================================

onehot_transformer = Pipeline([
    (
        "imp",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "enc",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# ============================================================
# COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            "num",
            numeric_transformer,
            numeric_features
        ),

        (
            "label",
            label_transformer,
            label_features
        ),

        (
            "binary",
            binary_transformer,
            binary_features
        ),

        (
            "onehot",
            onehot_transformer,
            onehot_features
        )
    ]
)
# ============================================================
# SADECE TRAIN ÜZERİNDE FIT
# ============================================================

X_train_encoded = preprocessor.fit_transform(
    X_train
)

# ============================================================
# VALIDATION / TEST
# Train'de öğrenilen encoder kullanılır
# ============================================================

X_val_encoded = preprocessor.transform(
    X_val
)

X_test_encoded = preprocessor.transform(
    X_test
)

# ============================================================
# BOYUT KONTROLÜ
# ============================================================

print("Train      :", X_train_encoded.shape)
print("Validation :", X_val_encoded.shape)
print("Test       :", X_test_encoded.shape)




In [ ]:
# ============================================================
# 8. PREPROCESSING'İN UYGULANMASI
# ============================================================
#
# preprocessor SADECE X_train üzerinde fit edilir.
#
# X_train:
#     fit_transform()
#
# X_validation:
#     transform()
#
# X_test:
#     transform()
#
# Böylece validation ve test verisi preprocessing
# aşamasında kullanılmaz ve veri sızıntısı önlenir.
# ============================================================

# TRAIN:
# Transformer parametreleri burada öğrenilir
X_train_encoded = preprocessor.fit_transform(
    X_train
)

# VALIDATION:
# Öğrenilen dönüşümler uygulanır
X_val_encoded = preprocessor.transform(
    X_val
)

# TEST:
# Öğrenilen dönüşümler uygulanır
X_test_encoded = preprocessor.transform(
    X_test
)

# ============================================================
# BOYUT KONTROLÜ
# ============================================================

print("Encoder sonrası X_train :", X_train_encoded.shape)
print("Encoder sonrası X_val   :", X_val_encoded.shape)
print("Encoder sonrası X_test  :", X_test_encoded.shape)



In [ ]:
# ============================================================
# 9. MODEL VERİSİ SON KONTROL
# ============================================================

print("=" * 70)
print("MODEL VERİSİ SON DURUM")
print("=" * 70)

print("Orijinal X:", X.shape)

print("\nTrain:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)

print("\nTest:")
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nEncoder sonrası:")

print("X_train_encoded:",
      X_train_encoded.shape)

print("X_val_encoded  :",
      X_val_encoded.shape)

print("X_test_encoded :",
      X_test_encoded.shape)

In [ ]:
import numpy as np
import pandas as pd

splits = {
    "Train": y_train,
    "Validation": y_val,
    "Test": y_test
}

thresholds = [16, 17, 18, 19, 20]

rows = []

for split_name, y in splits.items():
    y = np.asarray(y)

    for threshold in thresholds:
        count = np.sum(y > threshold)

        rows.append({
            "Split": split_name,
            "Threshold": f"> {threshold}",
            "Count": int(count),
            "Percentage": round(count / len(y) * 100, 3)
        })

df_distribution = pd.DataFrame(rows)

print("\n" + "=" * 80)
print("HIGH-PRICE DISTRIBUTION — TRAIN / VALIDATION / TEST")
print("=" * 80)

print(df_distribution.to_string(index=False))

print("\nSPLIT SIZES")
for name, y in splits.items():
    print(f"{name:<12}: {len(y):,}")

## 🤖 11. Data Preparation for Machine Learning / Makine Öğrenmesi İçin Veri Hazırlama

This section prepares the selected features for machine learning algorithms by applying preprocessing steps such as encoding categorical variables, scaling numerical features, and splitting the dataset into training and test sets.

Bu bölümde seçilen özellikler, makine öğrenmesi algoritmalarında kullanılmak üzere hazırlanmaktadır. Kategorik değişkenler kodlanmakta, sayısal değişkenler ölçeklendirilmekte ve veri seti eğitim ile test kümelerine ayrılarak modelleme aşamasına hazır hale getirilmektedir.

## competition_supply_interaction

In [ ]:
import polars as pl

df_finalprice = df_finalprice.with_columns(
    (
        pl.col("competition_level")
        .cast(pl.Utf8)
        .fill_null("MISSING")
        + "__" +
        pl.col("tender_supplytype")
        .cast(pl.Utf8)
        .fill_null("MISSING")
    ).alias("competition_supply_interaction")
)

print("Yeni kolon eklendi:")
print("competition_supply_interaction")

print(
    df_finalprice
    .select("competition_supply_interaction")
    .to_series()
    .value_counts()
    .sort("count", descending=True)
)

Bu kod, competition_level ve tender_supplytype değişkenlerini birleştirerek her gözlem için competition_supply_interaction adlı yeni bir etkileşim kategorisi oluşturur.

In [ ]:
# ============================================================
# MODELDE KULLANILACAK ÖZELLİKLER
# Hedef değişken: log_tender_finalpriceUsd
# ============================================================

model_features = [

    # --------------------------------------------------------
    # İhale prosedürü ve temel bilgiler
    # --------------------------------------------------------
    "tender_nationalproceduretype",      # Ulusal ihale prosedür türü
    "tender_supplytype",                 # Tedarik türü (mal, hizmet, yapım vb.)
    "lot_status",                        # Lot durumu
    "competition_supply_interaction",    # Rekabet seviyesi × tedarik türü etkileşimi

    # --------------------------------------------------------
    # İhale yapısı ve teklif bilgileri
    # --------------------------------------------------------
    "log_tender_lotscount",              # Lot sayısının log dönüşümü
    "log_tender_recordedbidscount",      # Teklif sayısının log dönüşümü
    "tender_isframeworkagreement",       # Çerçeve anlaşması olup olmadığı
    "tender_selectionmethod",            # Kazananın belirlenme yöntemi
    "tender_awardcriteria_count",        # Değerlendirme kriteri sayısı

    # --------------------------------------------------------
    # Alıcı bilgileri
    # --------------------------------------------------------
    "buyer_mainactivities",              # Alıcının ana faaliyet alanı

    # --------------------------------------------------------
    # Zaman bilgileri
    # --------------------------------------------------------
    "tender_year",                       # İhalenin gerçekleştiği yıl
    "tender_year_since_start",            # Başlangıç yılından itibaren geçen süre
    "log_submission_period",             # Teklif verme süresinin log dönüşümü
    "log_decision_period",               # Karar verme süresinin log dönüşümü
    "decision_duration_days",            # Karar sürecinin toplam gün süresi

    # --------------------------------------------------------
    # Türetilmiş teklif / zaman feature'ları
    # --------------------------------------------------------
    "log_bids_per_lot",                  # Lot başına teklif sayısının log dönüşümü
    "decision_to_submission_ratio",      # Karar süresi / teklif süresi oranı
    "is_long_submission",                # Teklif süresinin uzun olup olmadığını gösteren bayrak
    "is_long_decision",                  # Karar süresinin uzun olup olmadığını gösteren bayrak

    # --------------------------------------------------------
    # Rekabet ve risk göstergeleri
    # --------------------------------------------------------
    "competition_level",                         # Teklif sayısına göre rekabet seviyesi

    "corr_singleb",                              # Tek teklif verilme riski
    "corr_singleb_is_missing",                   # Tek teklif risk göstergesinin eksikliği (1=eksik, 0=mevcut)

    "corr_proc",                                 # İhale prosedürü kaynaklı risk

    "corr_subm",                                 # Teklif süresi kaynaklı risk
    "corr_subm_is_missing",                      # Teklif süresi risk göstergesinin eksikliği (1=eksik, 0=mevcut)

    "corr_nocft",                                # İlan yapılmaması / çağrı süreci riski
    "corr_decp",                                 # Karar süresi kaynaklı risk

    "corr_buyer_concentration",                  # Alıcı yoğunlaşması risk göstergesi
    "corr_buyer_concentration_is_missing"        # Alıcı yoğunlaşması bilgisinin eksikliği (1=eksik, 0=mevcut)
]

print("Model feature sayısı:", len(model_features))

print("\nModel Features:")
for feature in model_features:
    print("-", feature)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# ============================================================
# MODEL FEATURE MISSING ANALİZİ
# NULL + NaN birlikte missing kabul edilir
# ============================================================

model_features = [
    "tender_nationalproceduretype",
    "tender_supplytype",
    "lot_status",
    "log_tender_lotscount",
    "log_tender_recordedbidscount",
    "tender_isframeworkagreement",
    "tender_selectionmethod",
    "tender_awardcriteria_count",
    "buyer_mainactivities",
    "tender_year",
    "tender_year_since_start",
    "log_submission_period",
    "log_decision_period",
    "decision_duration_days",
    "competition_level",

    "corr_singleb",
    "corr_singleb_is_missing",
    "corr_proc",
    "corr_subm",
    "corr_subm_is_missing",
    "corr_nocft",
    "corr_decp",
    "corr_buyer_concentration",
    "corr_buyer_concentration_is_missing",

    "competition_supply_interaction",
    "log_bids_per_lot",
    "decision_to_submission_ratio",
    "is_long_submission",
    "is_long_decision"
]

# ============================================================
# MISSING HESAPLAMA
# ============================================================

missing_exprs = []

for col in model_features:

    dtype = df_finalprice.schema[col]

    # Sayısal kolonlar: NULL + NaN
    if dtype in [pl.Float32, pl.Float64]:

        expr = (
            (
                pl.col(col).is_null() |
                pl.col(col).is_nan()
            )
            .sum()
            .alias(col)
        )

    # Diğer kolonlar: sadece NULL
    else:

        expr = (
            pl.col(col)
            .is_null()
            .sum()
            .alias(col)
        )

    missing_exprs.append(expr)

# ============================================================
# SONUÇ TABLOSU
# ============================================================

missing_model = (
    df_finalprice
    .select(missing_exprs)
    .transpose(
        include_header=True,
        header_name="feature",
        column_names=["missing_count"]
    )
    .with_columns(
        (
            pl.col("missing_count")
            / df_finalprice.height
            * 100
        ).alias("missing_percent")
    )
    .sort(
        "missing_percent",
        descending=True
    )
)

display(missing_model)

In [ ]:
# ============================================================
# 2. BAĞIMSIZ VE BAĞIMLI DEĞİŞKENLERİN OLUŞTURULMASI
# ============================================================
#
# X  -> Modelin kullanacağı bağımsız değişkenler
# y  -> Tahmin edilmek istenen gerçekleşen ihale fiyatı
# ============================================================

X = df_finalprice.select(model_features)
y = df_finalprice.select(target)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# ============================================================
# 3. NUMERİK VE KATEGORİK DEĞİŞKENLER
# ============================================================

# Numerik Features
numeric_features = [

    # --------------------------------------------------------
    # İhale yapısı ve teklif bilgileri
    # --------------------------------------------------------
    "log_tender_lotscount",                  # Toplam lot sayısının log dönüşümü
    "log_tender_recordedbidscount",          # Toplam kayıtlı teklif sayısının log dönüşümü
    "tender_awardcriteria_count",            # Değerlendirme kriteri sayısı

    # --------------------------------------------------------
    # Zaman bilgileri
    # --------------------------------------------------------
    "tender_year",                            # İhale yılı
    "tender_year_since_start",                # Başlangıç yılından itibaren geçen süre
    "log_submission_period",                 # Teklif verme süresinin log dönüşümü
    "log_decision_period",                   # Karar süresinin log dönüşümü
    "decision_duration_days",                # Karar sürecinin toplam gün süresi

    # --------------------------------------------------------
    # Türetilmiş numerik feature'lar
    # --------------------------------------------------------
    "log_bids_per_lot",                       # Lot başına teklif sayısının log dönüşümü
    "decision_to_submission_ratio",           # Karar süresi / teklif süresi oranı

    # --------------------------------------------------------
    # Risk göstergeleri
    # --------------------------------------------------------
    "corr_singleb",                           # Tek teklif riski
    "corr_singleb_is_missing",                # Tek teklif risk göstergesinin eksikliği (1=eksik, 0=mevcut)

    "corr_proc",                              # Prosedür kaynaklı risk

    "corr_subm",                              # Teklif süresi riski
    "corr_subm_is_missing",                   # Teklif süresi risk göstergesinin eksikliği (1=eksik, 0=mevcut)

    "corr_nocft",                             # İlan/çağrı süreci riski
    "corr_decp",                              # Karar süresi riski

    "corr_buyer_concentration",               # Alıcı yoğunlaşması risk göstergesi
    "corr_buyer_concentration_is_missing",    # Alıcı yoğunlaşması bilgisinin eksikliği (1=eksik, 0=mevcut)

    # --------------------------------------------------------
    # Binary numerik feature'lar
    # --------------------------------------------------------
    "is_long_submission",                     # Teklif süresinin uzunluğu göstergesi
    "is_long_decision"                        # Karar süresinin uzunluğu göstergesi
]

# Kategorik Features
categorical_features = [

    # --------------------------------------------------------
    # İhale prosedürü
    # --------------------------------------------------------
    "tender_nationalproceduretype",            # Ulusal prosedür türü
    "tender_supplytype",                       # Tedarik türü
    "tender_selectionmethod",                  # Seçim yöntemi
    "competition_supply_interaction",          # Rekabet × tedarik türü etkileşimi

    # --------------------------------------------------------
    # İhale yapısı
    # --------------------------------------------------------
    "tender_isframeworkagreement",             # Çerçeve anlaşması bilgisi
    "lot_status",                              # Lot durumu

    # --------------------------------------------------------
    # Rekabet
    # --------------------------------------------------------
    "competition_level",                       # Rekabet seviyesi

    # --------------------------------------------------------
    # Alıcı bilgileri
    # --------------------------------------------------------
    "buyer_mainactivities"                     # Alıcının ana faaliyet alanı
]

# ============================================================
# KONTROL
# ============================================================

print("Numerik feature sayısı:", len(numeric_features))
print("Kategorik feature sayısı:", len(categorical_features))
print("Toplam feature sayısı:", len(numeric_features) + len(categorical_features))

print("\nNumerik Features:")
for col in numeric_features:
    print("-", col)

print("\nKategorik Features:")
for col in categorical_features:
    print("-", col)

In [ ]:
# TARGET DEĞİŞKENİNİN QUANTILE GRUPLARINA AYRILMASI
#
# tender_finalpriceUsd sürekli (regresyon) bir hedef değişkenidir.
# Veri setini Train / Validation / Test olarak bölerken target
# dağılımının üç sette de benzer olmasını sağlamak amacıyla,
# target değişkeni 10 quantile grubuna ayrılmıştır.

import polars as pl

df_finalprice = df_finalprice.with_columns(
    pl.col("log_tender_finalpriceUsd")
    .qcut(
        quantiles=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
        labels=[
            "q1","q2","q3","q4","q5",
            "q6","q7","q8","q9","q10"
        ],
        allow_duplicates=True
    )
    .alias("price_bin")
)

In [ ]:
# Veriyi fiyat aralığı (price_bin), tedarik türü (tender_supplytype) ve
# rekabet seviyesine (competition_level) göre gruplayarak her grubun frekansını
# hesaplar ve küçükten büyüğe sıralar.

df_finalprice.group_by(
    [
        "price_bin",
        "tender_supplytype",
        "competition_level"

    ]
).len().sort("len")

Stratification Değişkenlerinin Belirlenmesi

Veri setinin Train (%70), Validation (%15) ve Test (%15) olarak bölünmesi sırasında veri dağılımının dengeli korunması amacıyla price_bin, tender_supplytype ve competition_level değişkenleri stratification için seçilmiştir.

* price_bin: tender_finalpriceUsd hedef değişkeninin 10 quantile grubuna ayrılmasıyla oluşturulmuştur. Regresyon probleminde target doğrudan stratification için kullanılamadığı için, düşük ve yüksek fiyat seviyelerinin üç veri setinde benzer oranlarda bulunmasını sağlamak amacıyla kullanılmıştır.
* tender_supplytype: İhalelerin tedarik türlerini temsil eder. Farklı tedarik türlerinin Train, Validation ve Test setlerinde dengesiz dağılmasını önlemek amacıyla stratification'a dahil edilmiştir.
* competition_level: İhalelerin rekabet düzeyini temsil eder. Rekabet seviyelerinin veri setleri arasında benzer oranlarda dağılması, modelin farklı rekabet koşullarındaki performansının daha sağlıklı değerlendirilmesini sağlar.

Bu üç değişkenin birlikte kullanılmasıyla hem hedef değişkenin fiyat dağılımı hem de ihale yapısındaki temel kategorik dağılımlar korunmuştur. Diğer model değişkenleri ise stratification'a dahil edilmemiştir; çünkü çok sayıda değişkenin birleştirilmesi çok fazla kombinasyon oluşturabilir ve bazı grupların örnek sayısının aşırı azalmasına neden olabilir.

Yapılan kombinasyon kontrolünde 90 farklı stratification grubu oluşmuş ve en küçük grubun 886 gözlem içerdiği görülmüştür. Bu nedenle oluşturulan stratification yapısının %70-%15-%15 bölme işlemi için yeterli örnek sayısına sahip olduğu değerlendirilmiştir.


In [ ]:
df_finalprice = df_finalprice.with_columns(
    (
        pl.col("price_bin").cast(pl.String)
        + "_"
        + pl.col("tender_supplytype").cast(pl.String)
        + "_"
        + pl.col("competition_level").cast(pl.String)
    )
    .alias("stratify_col")
)

 Veri setinin Train (%70), Validation (%15) ve Test (%15)
 olarak dengeli şekilde bölünmesi amacıyla stratify_col
 değişkeni oluşturulmuştur.

 Stratification için üç değişken birlikte kullanılmıştır:

 1. price_bin:
    tender_finalpriceUsd hedef değişkeninin %10'luk dilimlere
    ayrılmasıyla oluşturulan fiyat gruplarını temsil eder.

 2. tender_supplytype:
    İhalenin tedarik türünü (SUPPLIES, WORKS, SERVICES vb.)
    temsil eder.

 3. competition_level:
    İhaledeki rekabet seviyesini temsil eder.

 Bu üç değişken birleştirilerek her kayıt için benzersiz bir
 stratification grubu oluşturulmuştur.

 Böylece veri Train, Validation ve Test setlerine ayrılırken
 fiyat dağılımı, tedarik türü ve rekabet seviyesinin
 setler arasında mümkün olduğunca benzer kalması sağlanmıştır.

In [ ]:
df_finalprice.select(
    pl.col("price_bin")
).group_by(
    "price_bin"
).len()

In [ ]:
import matplotlib.pyplot as plt
import polars as pl


# STRATIFY DAĞILIMI
# Price Bin + Supply Type + Competition Level


stratify_plot = (
    df_finalprice
    .group_by(
        [
            "price_bin",
            "tender_supplytype",
            "competition_level"
        ]
    )
    .len()
)

# POLARS -> PANDAS


plot_df = stratify_plot.to_pandas()


price_order = [f"q{i}" for i in range(1, 11)]

plot_df["price_bin"] = pl.Series(
    "price_bin",
    plot_df["price_bin"]
).cast(pl.String).to_list()

plot_df["price_bin"] = __import__("pandas").Categorical(
    plot_df["price_bin"],
    categories=price_order,
    ordered=True
)

# PIVOT TABLO


pivot_df = (
    plot_df
    .pivot_table(
        index="price_bin",
        columns=[
            "tender_supplytype",
            "competition_level"
        ],
        values="len",
        fill_value=0,
        sort=False
    )
)

pivot_df = pivot_df.reindex(price_order)

# ============================================================
# GRAFİK
# ============================================================

ax = pivot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 6)
)

plt.title(
    "Stratify Dağılımı\nPrice Bin + Supply Type + Competition Level",
    fontsize=14
)

plt.xlabel("Price Bin")
plt.ylabel("Kayıt Sayısı")

plt.xticks(rotation=0)

plt.legend(
    title="Supply Type / Competition",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. TRAIN - VALIDATION - TEST AYRIMI
# ============================================================

# Veri:
# %70 -> Eğitim
# %15 -> Validation
# %15 -> Test

# Stratification:
# price_bin + tender_supplytype + competition_level

# ÖNEMLİ:
# Stratification değişkenleri yalnızca veri bölme işleminde
# kullanılmaktadır.

# df_train / df_val / df_test:
#     29 model feature
#   + 1 target
#   = 30 kolon

# X_train / X_val / X_test:
#     yalnızca 29 model feature

# y_train / y_val / y_test:
#     yalnızca target

# price_bin ve stratify_col hiçbir model veri setine
# dahil edilmez.

# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np

# ============================================================
# 1. HEDEF DEĞİŞKEN
# ============================================================

target = "log_tender_finalpriceUsd"

# ============================================================
# 2. MODELDE KULLANILACAK ÖZELLİKLER
# ============================================================

model_features = [

    # --------------------------------------------------------
    # İhale prosedürü ve temel bilgiler
    # --------------------------------------------------------
    "tender_nationalproceduretype",       # Ulusal ihale prosedür türü
    "tender_supplytype",                  # Tedarik türü
    "lot_status",                         # Lot durumu
    "competition_supply_interaction",     # Rekabet × tedarik türü etkileşimi

    # --------------------------------------------------------
    # İhale yapısı ve teklif bilgileri
    # --------------------------------------------------------
    "log_tender_lotscount",               # Lot sayısının log dönüşümü
    "log_tender_recordedbidscount",       # Teklif sayısının log dönüşümü
    "tender_isframeworkagreement",        # Çerçeve anlaşması bilgisi
    "tender_selectionmethod",             # Kazananın belirlenme yöntemi
    "tender_awardcriteria_count",         # Değerlendirme kriteri sayısı

    # --------------------------------------------------------
    # Alıcı bilgileri
    # --------------------------------------------------------
    "buyer_mainactivities",               # Alıcının ana faaliyet alanı

    # --------------------------------------------------------
    # Zaman bilgileri
    # --------------------------------------------------------
    "tender_year",                        # İhale yılı
    "tender_year_since_start",            # Başlangıç yılından itibaren geçen süre
    "log_submission_period",              # Teklif verme süresinin log dönüşümü
    "log_decision_period",                # Karar süresinin log dönüşümü
    "decision_duration_days",             # Karar süresinin toplam gün sayısı

    # --------------------------------------------------------
    # Türetilmiş feature'lar
    # --------------------------------------------------------
    "log_bids_per_lot",                   # Lot başına teklif sayısının log dönüşümü
    "decision_to_submission_ratio",       # Karar süresi / teklif süresi oranı
    "is_long_submission",                # Uzun teklif süresi göstergesi
    "is_long_decision",                  # Uzun karar süresi göstergesi

    # --------------------------------------------------------
    # Rekabet ve risk göstergeleri
    # --------------------------------------------------------
    "competition_level",                  # Rekabet seviyesi

    "corr_singleb",                       # Tek teklif riski
    "corr_singleb_is_missing",            # Teklif riski bilgisinin eksiklik göstergesi

    "corr_proc",                          # Prosedür kaynaklı risk

    "corr_subm",                          # Teklif süresi kaynaklı risk
    "corr_subm_is_missing",               # Teklif süresi riski bilgisinin eksiklik göstergesi

    "corr_nocft",                         # İlan/çağrı süreci riski
    "corr_decp",                          # Karar süresi kaynaklı risk

    "corr_buyer_concentration",           # Alıcı yoğunlaşması riski
    "corr_buyer_concentration_is_missing" # Alıcı yoğunlaşması eksiklik göstergesi
]

# ============================================================
# 3. MODEL FEATURE KONTROLÜ
# ============================================================

print("=" * 60)
print("MODEL FEATURE KONTROLÜ")
print("=" * 60)

print("Model feature sayısı:", len(model_features))

print("\nModel Features:")

for i, feature in enumerate(model_features, 1):
    print(f"{i:2}. {feature}")

# ============================================================
# 4. GEREKLİ KOLONLARIN KONTROLÜ
# ============================================================

required_columns = (
    model_features
    + [target]
    + ["stratify_col"]
)

missing_columns = [
    col
    for col in required_columns
    if col not in df_finalprice.columns
]

if missing_columns:
    raise ValueError(
        f"df_finalprice içerisinde eksik kolonlar var: "
        f"{missing_columns}"
    )

# ============================================================
# 5. SATIR İNDEKSLERİNİN OLUŞTURULMASI
# ============================================================

indices = np.arange(df_finalprice.height)

# ============================================================
# 6. STRATIFICATION DEĞİŞKENİNİN ALINMASI
# ============================================================

stratify_np = (
    df_finalprice
    .select("stratify_col")
    .to_series()
    .to_numpy()
)

# ============================================================
# 7. TRAIN (%70) - TEMP (%30)
# ============================================================

idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=stratify_np
)

# ============================================================
# 8. TEMP STRATIFICATION DEĞERLERİ
# ============================================================

stratify_temp = stratify_np[idx_temp]

# ============================================================
# 9. VALIDATION (%15) - TEST (%15)
# ============================================================

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    random_state=42,
    stratify=stratify_temp
)

# ============================================================
# 10. TRAIN / VALIDATION / TEST DATAFRAME'LERİ
# ============================================================

selected_columns = model_features + [target]

df_train = (
    df_finalprice[idx_train]
    .select(selected_columns)
)

df_val = (
    df_finalprice[idx_val]
    .select(selected_columns)
)

df_test = (
    df_finalprice[idx_test]
    .select(selected_columns)
)

# ============================================================
# 11. X DEĞİŞKENLERİNİN OLUŞTURULMASI
# ============================================================

X_train = df_train.select(model_features)
X_val = df_val.select(model_features)
X_test = df_test.select(model_features)

# ============================================================
# 12. y DEĞİŞKENLERİNİN OLUŞTURULMASI
# ============================================================

y_train = df_train.select(target).to_series()
y_val = df_val.select(target).to_series()
y_test = df_test.select(target).to_series()

# ============================================================
# 13. DATASET BOYUT KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("DATASET BOYUTLARI")
print("=" * 60)

print("df_train:", df_train.shape)
print("df_val  :", df_val.shape)
print("df_test :", df_test.shape)

print()

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print()

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

# ============================================================
# 14. TRAIN KOLONLARININ KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("DF_TRAIN KOLONLARI")
print("=" * 60)

for i, col in enumerate(df_train.columns, 1):
    print(f"{i:2}. {col}")

# ============================================================
# 15. X_TRAIN KOLONLARININ KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("X_TRAIN KOLONLARI")
print("=" * 60)

for i, col in enumerate(X_train.columns, 1):
    print(f"{i:2}. {col}")

# ============================================================
# 16. TARGET KONTROLÜ
# ============================================================

print("\n" + "=" * 60)
print("TARGET KONTROLÜ")
print("=" * 60)

print("Target:", target)
print("y_train shape:", y_train.shape)
print("y_val shape  :", y_val.shape)
print("y_test shape :", y_test.shape)

# ============================================================
# 17. TARGET'IN X İÇERİSİNDE OLMADIĞINI KONTROL ET
# ============================================================

if target in X_train.columns:
    raise ValueError("HATA: Target X_train içerisinde bulunuyor!")

if target in X_val.columns:
    raise ValueError("HATA: Target X_val içerisinde bulunuyor!")

if target in X_test.columns:
    raise ValueError("HATA: Target X_test içerisinde bulunuyor!")

print("\nTarget X değişkenlerine dahil edilmemiştir.")

# ============================================================
# 18. STRATIFICATION KOLONLARININ X İÇERİSİNDE
#     OLMADIĞINI KONTROL ET
# ============================================================

for col in ["price_bin", "stratify_col"]:

    if col in X_train.columns:
        raise ValueError(
            f"HATA: {col} X_train içerisinde bulunuyor!"
        )

    if col in X_val.columns:
        raise ValueError(
            f"HATA: {col} X_val içerisinde bulunuyor!"
        )

    if col in X_test.columns:
        raise ValueError(
            f"HATA: {col} X_test içerisinde bulunuyor!"
        )

print(
    "price_bin ve stratify_col model özelliklerine "
    "dahil edilmemiştir."
)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

# ============================================================
# TRAIN - VALIDATION - TEST DAĞILIMLARI
# PASTA GRAFİĞİ + YÜZDE + KAYIT SAYISI
# ============================================================

# Stratification:
# price_bin + tender_supplytype + competition_level
#
# Bu değişkenler sadece stratified split'in kontrolü
# amacıyla kullanılmaktadır.
# ============================================================


# ============================================================
# 1. SPLIT VERİLERİNİN OLUŞTURULMASI
# ============================================================

df_train_dist = (
    df_finalprice[idx_train]
    .select([
        "price_bin",
        "tender_supplytype",
        "competition_level"
    ])
)

df_val_dist = (
    df_finalprice[idx_val]
    .select([
        "price_bin",
        "tender_supplytype",
        "competition_level"
    ])
)

df_test_dist = (
    df_finalprice[idx_test]
    .select([
        "price_bin",
        "tender_supplytype",
        "competition_level"
    ])
)


# ============================================================
# 2. PASTA GRAFİĞİ FONKSİYONU
# ============================================================

def plot_pie_distribution(
    train_df,
    val_df,
    test_df,
    column,
    title
):

    # --------------------------------------------------------
    # Dağılım hesaplama
    # --------------------------------------------------------

    def get_distribution(df):

        distribution = (
            df
            .group_by(column)
            .len()
            .sort(column)
        )

        labels = distribution[column].to_list()
        values = distribution["len"].to_list()

        return labels, values

    train_labels, train_values = get_distribution(train_df)
    val_labels, val_values = get_distribution(val_df)
    test_labels, test_values = get_distribution(test_df)


    # ========================================================
    # PASTA GRAFİKLERİ
    # ========================================================

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(20, 7)
    )


    # ========================================================
    # TRAIN
    # ========================================================

    train_total = sum(train_values)

    axes[0].pie(
        train_values,
        labels=train_labels,
        autopct=lambda pct: (
            f"{pct:.1f}%\n"
            f"({int(round(pct * train_total / 100)):,})"
        ),
        startangle=90
    )

    axes[0].set_title(
        f"Train\n(n={train_total:,})",
        fontsize=14,
        fontweight="bold",
        pad=15
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_total = sum(val_values)

    axes[1].pie(
        val_values,
        labels=val_labels,
        autopct=lambda pct: (
            f"{pct:.1f}%\n"
            f"({int(round(pct * val_total / 100)):,})"
        ),
        startangle=90
    )

    axes[1].set_title(
        f"Validation\n(n={val_total:,})",
        fontsize=14,
        fontweight="bold",
        pad=15
    )


    # ========================================================
    # TEST
    # ========================================================

    test_total = sum(test_values)

    axes[2].pie(
        test_values,
        labels=test_labels,
        autopct=lambda pct: (
            f"{pct:.1f}%\n"
            f"({int(round(pct * test_total / 100)):,})"
        ),
        startangle=90
    )

    axes[2].set_title(
        f"Test\n(n={test_total:,})",
        fontsize=14,
        fontweight="bold",
        pad=15
    )


    # ========================================================
    # ANA BAŞLIK
    # ========================================================

    fig.suptitle(
        title,
        fontsize=17,
        fontweight="bold",
        y=0.98
    )


    # ========================================================
    # GRAFİK BOŞLUKLARI
    # ========================================================

    # Üstte ana başlık için boşluk bırakılır.
    plt.tight_layout(
        rect=[0, 0, 1, 0.90]
    )

    plt.show()


# ============================================================
# 3. PRICE BIN DAĞILIMI
# ============================================================

plot_pie_distribution(
    df_train_dist,
    df_val_dist,
    df_test_dist,
    "price_bin",
    "Train - Validation - Test Price Bin Dağılımı"
)


# ============================================================
# 4. TENDER SUPPLY TYPE DAĞILIMI
# ============================================================

plot_pie_distribution(
    df_train_dist,
    df_val_dist,
    df_test_dist,
    "tender_supplytype",
    "Train - Validation - Test Tender Supply Type Dağılımı"
)


# ============================================================
# 5. COMPETITION LEVEL DAĞILIMI
# ============================================================

plot_pie_distribution(
    df_train_dist,
    df_val_dist,
    df_test_dist,
    "competition_level",
    "Train - Validation - Test Competition Level Dağılımı"
)

Train - Validation - Test Veri Bölme (Data Split) Analiz Raporu

 1. Genel Veri Hacmi ve Bölünme Oranları
Toplam veri seti 297,164 gözlemden oluşmakta olup belirlenen indekslerle şu oranlarda bölünmüştür:
* Train Seti: n = 208,014 (%70.0)
* Validation Seti: n = 44,575 (%15.0)
* Test Seti: n = 44,575 (%15.0)

Bu ayrım, yaklaşık 70 / 15 / 15 standardına uygun şekilde dengeli bir hacim dağılımı sağlandığını göstermektedir.

 2. Kategorik Değişken Dağılım Analizleri

 A. Price Bin (Fiyat Dilimleri) Dağılımı
* Toplam 10 kuantil dilimi q1 – q10) temsil edilmektedir.
* Train: Her bir dilim 20,787 ile 20,816 arasında satır içerir (%10.0 oranında homojen).
* Validation & Test: Her iki sette de her bir dilim 4,453 ile 4,461 arasında değer alarak kusursuz bir şekilde %10.0 paya sahiptir.
* Yorum: Fiyat segmentasyonu açısından model, eğitildiği fiyat dağılımının birebir aynı oranlarıyla doğrulanacak ve test edilecektir.

B. Tender Supply Type (İhale Tedarik Türü) Dağılımı
* WORKS: Train (74,159 - %35.7), Validation (15,892 - %35.7), Test (15,891 - %35.7)
* SERVICES: Train (72,531 - %34.9), Validation (15,544 - %34.9), Test (15,541 - %34.9)
* SUPPLIES: Train (61,324 - %29.5), Validation (13,139 - %29.5), Test (13,143 - %29.5)
* Yorum: Sınıf oranları 3 veri setinde de küsuratına kadar homojendir. İhale türüne bağlı olarak yaşanabilecek olası bir veri kayması (data drift) engellenmiştir.

C. Competition Level (Rekabet Seviyesi) Dağılımı
* Low Competition: Train (82,023 - %39.4), Validation (17,575 - %39.4), Test (17,575 - %39.4)
* High Competition: Train (70,717 - %34.0), Validation (15,155 - %34.0), Test (15,154 - %34.0)
* Medium Competition: Train (55,274 - %26.6), Validation (11,845 - %26.6), Test (11,846 - %26.6)
* Yorum: Rekabet yoğunluğunun dağılımı eğitim, doğrulama ve test aşamalarında tam uyum göstermektedir.

 3. Sonuç ve Modelleme Öncesi Onay
Çok kolonlu katmanlı bölme (multi-column stratified split) stratejisi başarıyla uygulanmıştır:
1. Representativeness (Temsiliyet): Validation ve Test setleri, Train setinin tam ve tarafsız bir kopyası niteliğindedir.
2. Data Drift Riski Yok: Sınıf dağılımlarındaki dengesizlikten kaynaklanabilecek test skoru yanılsamaları önlenmiştir.
3. Model Eğitimi: Veri alt kümeleri modelleme ve hiperparametre optimizasyonu adımlarına geçmeye hazırdır.

## BIDS_PER_LOT

In [ ]:
import polars as pl

df_finalprice = df_finalprice.with_columns(
    (
        pl.col("tender_recordedbidscount").cast(pl.Float64)
        / pl.col("tender_lotscount").cast(pl.Float64)
    )
    .fill_nan(0)
    .fill_null(0)
    .alias("bids_per_lot")
)

print("Yeni feature: bids_per_lot")

print(
    df_finalprice
    .select("bids_per_lot")
    .describe()
)

Bu kod, ihaledeki kayıtlı teklif sayısını lot sayısına bölerek her lot başına düşen ortalama teklif sayısını bids_per_lot adlı yeni bir feature olarak oluşturur.

In [ ]:
import numpy as np
import polars as pl

df_finalprice = df_finalprice.with_columns(
    pl.col("bids_per_lot")
    .log1p()
    .alias("log_bids_per_lot")
)

print("Yeni feature: log_bids_per_lot")

print(
    df_finalprice
    .select("log_bids_per_lot")
    .describe()
)

## RARE

In [ ]:
import polars as pl

RARE_MIN_COUNT = 200
COL = "tender_nationalproceduretype"

# Sadece TRAIN frekanslarını kullan
train_counts = (
    X_train
    .with_columns(
        pl.col(COL)
        .cast(pl.Utf8)
        .fill_null("MISSING")
        .alias("_procedure")
    )
    .group_by("_procedure")
    .len()
    .rename({"len": "count"})
)

# TRAIN'de 100'den az görülen kategoriler
rare_categories = (
    train_counts
    .filter(pl.col("count") < RARE_MIN_COUNT)
    .get_column("_procedure")
    .to_list()
)

print(f"Rare threshold : < {RARE_MIN_COUNT}")
print(f"Rare category sayısı : {len(rare_categories)}")
print(f"Rare kategoriler : {rare_categories}")

# Aynı rare listesini Train / Validation / Test'e uygula
def group_rare(df):
    return df.with_columns(
        pl.when(
            pl.col(COL)
            .cast(pl.Utf8)
            .fill_null("MISSING")
            .is_in(rare_categories)
        )
        .then(pl.lit("RARE"))
        .otherwise(
            pl.col(COL)
            .cast(pl.Utf8)
            .fill_null("MISSING")
        )
        .alias(COL)
    )

X_train = group_rare(X_train)
X_val = group_rare(X_val)
X_test = group_rare(X_test)

# Kontrol
print("\nTRAIN dağılımı:")
print(
    X_train
    .get_column(COL)
    .value_counts()
    .sort("count", descending=True)
)

print("\nVAL dağılımı:")
print(
    X_val
    .get_column(COL)
    .value_counts()
    .sort("count", descending=True)
)

print("\nTEST dağılımı:")
print(
    X_test
    .get_column(COL)
    .value_counts()
    .sort("count", descending=True)
)

Rare Category Grouping için minimum gözlem sayısı 200 olarak belirlenmiştir. Bu eşik, tender_nationalproceduretype kolonundaki düşük frekanslı kategorilerin model tarafından ayrı ayrı öğrenilmesini engellemek amacıyla seçilmiştir. Özellikle residual analizinde yüksek hata üreten **f18_pt_negotiated_without_publication_contract_notice** kategorisinin eğitim setinde yalnızca 175 gözleme sahip olması, bu kategorinin güvenilir bir örüntü öğrenmek için oldukça sınırlı olduğunu göstermektedir. 100 gözlem eşiği kullanıldığında bu kategori RARE grubuna alınmadığı için temel problem devam etmekteydi. Eşik 200 olarak yükseltildiğinde ise bu kategori ve benzer şekilde az gözlenen toplam 5 kategori **RARE** altında birleştirilmiştir. Böylece modelin çok az örnek içeren kategorilere özgü gürültüyü öğrenme riski azaltılmış ve nadir prosedür tiplerinin daha istikrarlı bir şekilde modellenmesi hedeflenmiştir. Eşik yalnızca eğitim setindeki frekanslara göre belirlenmiş, ardından aynı kategori listesi validation ve test setlerine uygulanarak veri sızıntısının önüne geçilmiştir.


In [ ]:
# ============================================================
# LOT_STATUS — RARE CATEGORY GROUPING
# ============================================================

RARE_STATUS_MIN_COUNT = 200
STATUS_COL = "lot_status"

# Sadece TRAIN frekanslarını kullan
status_counts = (
    X_train
    .with_columns(
        pl.col(STATUS_COL)
        .cast(pl.Utf8)
        .fill_null("MISSING")
        .alias("_status")
    )
    .group_by("_status")
    .len()
    .rename({"len": "count"})
)

# TRAIN'de 100'den az görülen kategoriler
rare_status_categories = (
    status_counts
    .filter(pl.col("count") < RARE_STATUS_MIN_COUNT)
    .get_column("_status")
    .to_list()
)

print(f"Rare status threshold : < {RARE_STATUS_MIN_COUNT}")
print(f"Rare status category sayısı : {len(rare_status_categories)}")
print(f"Rare status kategorileri : {rare_status_categories}")


def group_rare_status(df):
    return df.with_columns(
        pl.when(
            pl.col(STATUS_COL)
            .cast(pl.Utf8)
            .fill_null("MISSING")
            .is_in(rare_status_categories)
        )
        .then(pl.lit("RARE_STATUS"))
        .otherwise(
            pl.col(STATUS_COL)
            .cast(pl.Utf8)
            .fill_null("MISSING")
        )
        .alias(STATUS_COL)
    )


# Aynı TRAIN kuralını VAL ve TEST'e uygula
X_train = group_rare_status(X_train)
X_val = group_rare_status(X_val)
X_test = group_rare_status(X_test)


# Kontrol
print("\nTRAIN lot_status dağılımı:")
print(
    X_train
    .get_column(STATUS_COL)
    .value_counts()
    .sort("count", descending=True)
)

print("\nVAL lot_status dağılımı:")
print(
    X_val
    .get_column(STATUS_COL)
    .value_counts()
    .sort("count", descending=True)
)

print("\nTEST lot_status dağılımı:")
print(
    X_test
    .get_column(STATUS_COL)
    .value_counts()
    .sort("count", descending=True)
)

## TENDER_YEAR_SINCE_START

In [ ]:
import polars as pl

# Veri setindeki ilk yılı referans al
START_YEAR = df_finalprice["tender_year"].min()

df_finalprice = df_finalprice.with_columns(
    (
        pl.col("tender_year").cast(pl.Float64) - START_YEAR
    ).alias("tender_year_since_start")
)

print("Başlangıç yılı:", START_YEAR)

print(
    df_finalprice
    .select([
        "tender_year",
        "tender_year_since_start"
    ])
    .describe()
)

Bu kod, ilk ihale yılını referans alarak her ihalenin başlangıç yılından kaç yıl sonra gerçekleştiğini tender_year_since_start adlı yeni feature olarak oluşturur.

## ZAMAN FEATURES

In [ ]:
import polars as pl

submission_threshold = df_finalprice["submission_period"].quantile(0.90)
decision_threshold = df_finalprice["decision_period"].quantile(0.90)

df_finalprice = df_finalprice.with_columns([
    pl.lit(submission_threshold).alias("submission_threshold"),
    pl.lit(decision_threshold).alias("decision_threshold")
])

print(f"Submission duration threshold (90%): {submission_threshold:.2f}")
print(f"Decision duration threshold (90%): {decision_threshold:.2f}")

In [ ]:
import polars as pl

# Eşik değerlerini df_finalprice üzerinden belirle
submission_threshold = (
    df_finalprice["submission_period"]
    .quantile(0.90)
)

decision_threshold = (
    df_finalprice["decision_period"]
    .quantile(0.90)
)

print(
    f"Submission duration threshold (90%): "
    f"{submission_threshold:.2f}"
)

print(
    f"Decision duration threshold (90%): "
    f"{decision_threshold:.2f}"
)

# Eski feature varsa kaldır
columns_to_drop = [
    col for col in [
        "decision_to_submission_ratio",
        "is_long_submission",
        "is_long_decision"
    ]
    if col in df_finalprice.columns
]

if columns_to_drop:
    df_finalprice = df_finalprice.drop(columns_to_drop)

# Yeni feature'ları oluştur
df_finalprice = df_finalprice.with_columns([

    # Karar süresi / başvuru süresi
    pl.when(
        pl.col("submission_period") > 0
    )
    .then(
        pl.col("decision_period").cast(pl.Float64)
        / pl.col("submission_period").cast(pl.Float64)
    )
    .otherwise(None)
    .alias("decision_to_submission_ratio"),

    # Uzun başvuru süresi
    (
        pl.col("submission_period")
        > submission_threshold
    )
    .cast(pl.Int8)
    .alias("is_long_submission"),

    # Uzun karar süresi
    (
        pl.col("decision_period")
        > decision_threshold
    )
    .cast(pl.Int8)
    .alias("is_long_decision")
])

# Ratio medianını df_finalprice üzerinden hesapla
ratio_median = (
    df_finalprice
    .select(
        pl.col("decision_to_submission_ratio")
        .median()
    )
    .item()
)

print(
    f"Ratio median: {ratio_median:.6f}"
)

# Eksik ratio değerlerini median ile doldur
df_finalprice = df_finalprice.with_columns(
    pl.col("decision_to_submission_ratio")
    .fill_null(ratio_median)
    .alias("decision_to_submission_ratio")
)

# Son kontrol
print("\nYeni feature'lar oluşturuldu.")

print(
    df_finalprice
    .select([
        "decision_to_submission_ratio",
        "is_long_submission",
        "is_long_decision"
    ])
    .describe()
)

# Uzun süreli gözlem sayıları
print("\nUzun süreli gözlem sayıları:")

print(
    df_finalprice
    .select([
        pl.col("is_long_submission")
        .sum()
        .alias("long_submission_count"),

        pl.col("is_long_decision")
        .sum()
        .alias("long_decision_count")
    ])
)

Bu kod, df_finalprice üzerinden başvuru ve karar sürelerinin %90 eşiklerini belirleyip decision_to_submission_ratio, is_long_submission ve is_long_decision olmak üzere üç yeni feature oluşturur ve eksik oran değerlerini median ile doldurur.

In [ ]:
print(
    "submission_period = 0:",
    df_finalprice
    .filter(pl.col("submission_period") == 0)
    .height
)

print(
    "decision_period = 0:",
    df_finalprice
    .filter(pl.col("decision_period") == 0)
    .height
)

In [ ]:
print("X_train kolonları:")
for i, col in enumerate(X_train.columns, 1):
    print(f"{i:2d}. {col}")

print("\nToplam kolon:", len(X_train.columns))

In [ ]:
# ============================================================
# KATEGORİK KOLONLARIN CARDINALITY ANALİZİ
# ============================================================
# Her kategorik değişkenin kaç farklı değer içerdiğini
# kontrol ediyoruz.
#
# Bu analiz:
# - Constant kolonları tespit etmek
# - Yüksek cardinality problemini görmek
# - One-Hot Encoding sonrasında oluşabilecek boyutu
# değerlendirmek için kullanılır.

categorical_features = [

    # Rekabet bilgileri

    "competition_level",                 # Teklif sayısına göre rekabet seviyesi

    # İhale özellikleri

    "tender_nationalproceduretype",      # Ulusal ihale prosedür türü
    "tender_supplytype",                 # Tedarik türü
    "tender_selectionmethod",            # İhale seçim yöntemi
    "tender_isframeworkagreement",       # Çerçeve anlaşması bilgisi
    "lot_status",                        # Lot durumu
    "competition_supply_interaction",





    # Alıcı bilgileri

    "buyer_mainactivities",              # Alıcının ana faaliyet alanı

]

# Sadece veri setinde gerçekten bulunan kolonları kullan
categorical_features = [
    col
    for col in categorical_features
    if col in df_finalprice.columns
]


# Cardinality analizi
categorical_cardinality = []

for col in categorical_features:

    unique_count = (
        df_finalprice
        .select(
            pl.col(col).n_unique()
        )
        .item()
    )

    categorical_cardinality.append(
        {
            "column": col,
            "unique_count": unique_count
        }
    )


# Sonuçları Polars DataFrame olarak oluştur
categorical_cardinality = (
    pl.DataFrame(categorical_cardinality)
    .sort(
        "unique_count",
        descending=True
    )
)


print("=" * 80)
print(" KATEGORİK KOLONLAR - CARDINALITY ANALİZİ")
print("=" * 80)

display(categorical_cardinality)

Kategorik Değişkenlerde Encoding Seçimi

Modelde kullanılan kategorik değişkenlerin cardinality (benzersiz kategori sayısı) değerleri incelenerek her değişken için uygun encoding yöntemi belirlenmiştir.

İki kategoriye sahip ikili değişkenlerde (tender_selectionmethod, tender_isframeworkagreement) Label Encoding kullanılmıştır. Bu değişkenler yalnızca iki farklı değer içerdiğinden kategorilerin 0 ve 1 olarak kodlanması herhangi bir sıralama yanlılığı oluşturmamaktadır. Ayrıca tek sütun kullanılarak gereksiz özellik artışı engellenmiştir.

Düşük cardinality değerine sahip değişkenlerde (tender_proceduretype, competition_level, lot_status, tender_supplytype) One-Hot Encoding tercih edilmiştir. Bu değişkenlerde kategori sayısı düşük olduğundan oluşturulan yeni özellik sayısı sınırlıdır. One-Hot Encoding kategoriler arasında yapay bir sıralama oluşturmadığı ve her kategoriyi bağımsız temsil ettiği için modelin kategorik bilgiyi daha doğru öğrenmesini sağlamaktadır.

Orta seviyede cardinality değerine sahip değişkenlerde (tender_nationalproceduretype, buyer_mainactivities) ise Binary Encoding kullanılmıştır. One-Hot Encoding uygulanması durumunda bu iki değişken için toplam 42 yeni özellik oluşturulacak ve model boyutu gereksiz şekilde artacaktır. Binary Encoding, kategori bilgisini korurken çok daha az sayıda özellik üreterek bellek kullanımını ve hesaplama maliyetini azaltmaktadır.

Alternatif olarak değerlendirilen Target Encoding, kategorileri hedef değişken istatistiklerine göre dönüştürdüğü için özellikle yanlış uygulandığında veri sızıntısı (data leakage) riski oluşturabilmektedir. Frequency Encoding ise tek sütun üretmesine rağmen kategoriler arasındaki ayrıştırıcı bilgiyi azaltabilmektedir.İki farklı kategori aynı frekansa sahipse ayırt edilemeyebilir. Ordinal (Label) Encoding ise ikiden fazla kategoriye sahip sırasız değişkenlerde yapay bir büyüklük ilişkisi oluşturduğundan uygun görülmemiştir. Bu nedenlerle orta cardinality değerine sahip değişkenlerde Binary Encoding, düşük cardinality değerine sahip değişkenlerde ise One-Hot Encoding tercih edilmiştir.

Değişken	Kategori Sayısı	Encoding	Gerekçe

**tender_selectionmethod**	2	Label Encoding	İkili yapı nedeniyle tek sütun yeterlidir ve yapay sıralama oluşturmaz.

**tender_isframeworkagreement**	2	Label Encoding	İki kategori bulunduğundan basit ve verimli bir dönüşüm sağlar.

**competition_level**	4	One-Hot Encoding	Az sayıda kategori içerdiğinden One-Hot Encoding hem yorumlanabilir hem de sıralama yanlılığı oluşturmaz.

**lot_status**	4	One-Hot Encoding	Düşük cardinality sayesinde gereksiz özellik artışı oluşturmaz ve kategorileri ayrı temsil eder.

**tender_supplytype**	3	One-Hot Encoding	Az sayıda kategori bulunduğundan bilgi kaybı olmadan doğrudan temsil edilir.

**tender_nationalproceduretype**	22	Binary Encoding	Orta cardinality nedeniyle One-Hot Encoding'in oluşturacağı çok sayıda sütunu azaltır.

**buyer_mainactivities**	20	Binary Encoding	Kategori bilgisini korurken model boyutunu ve hesaplama maliyetini düşürür.

Binary encoding ağaç tabanlı modellerle uyumludur.

In [ ]:
!pip install category_encoders

In [ ]:
import numpy as np
import category_encoders as ce

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

# ============================================================
# FEATURE GRUPLARI
# ============================================================

numeric_features = [
    "log_tender_lotscount",
    "log_tender_recordedbidscount",
    "tender_awardcriteria_count",
    "tender_year",
    "log_submission_period",
    "log_decision_period",
    "decision_duration_days",
    "corr_singleb",
    "corr_proc",
    "corr_subm",
    "corr_nocft",
    "corr_decp",
    "corr_buyer_concentration",
    "log_bids_per_lot",
    "tender_year_since_start",
]

label_features = [
    "tender_selectionmethod",
    "tender_isframeworkagreement"
]

binary_features = [
    "tender_nationalproceduretype",
    "buyer_mainactivities",



]

onehot_features = [
    "tender_supplytype",
    "competition_level",
    "lot_status",
    "competition_supply_interaction"
]

# ============================================================
# NUMERIC
# ============================================================

numeric_transformer = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# ============================================================
# LABEL ENCODER
# Train'de fit edilir, validation/test'te aynı encoder kullanılır
# ============================================================

class MultiColumnLabelEncoder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = np.asarray(X)

        self.encoders_ = []

        for i in range(X.shape[1]):
            encoder = LabelEncoder()
            encoder.fit(X[:, i])
            self.encoders_.append(encoder)

        return self

    def transform(self, X):
        X = np.asarray(X)
        X_encoded = np.zeros(
            (X.shape[0], X.shape[1]),
            dtype=float
        )

        for i, encoder in enumerate(self.encoders_):
            values = X[:, i]

            known = np.isin(
                values,
                encoder.classes_
            )

            X_encoded[known, i] = encoder.transform(
                values[known]
            )

            # Train'de bulunmayan kategoriler
            X_encoded[~known, i] = -1

        return X_encoded

# ============================================================
# LABEL TRANSFORMER
# ============================================================

label_transformer = Pipeline([
    (
        "imp",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "enc",
        MultiColumnLabelEncoder()
    )
])

# ============================================================
# BINARY ENCODING
# ============================================================

binary_transformer = Pipeline([
    (
        "imp",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "enc",
        ce.BinaryEncoder()
    )
])

# ============================================================
# ONE HOT ENCODING
# ============================================================

onehot_transformer = Pipeline([
    (
        "imp",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "enc",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# ============================================================
# COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            "num",
            numeric_transformer,
            numeric_features
        ),

        (
            "label",
            label_transformer,
            label_features
        ),

        (
            "binary",
            binary_transformer,
            binary_features
        ),

        (
            "onehot",
            onehot_transformer,
            onehot_features
        )
    ]
)

# ============================================================
# SADECE TRAIN ÜZERİNDE FIT
# ============================================================

X_train_encoded = preprocessor.fit_transform(
    X_train
)

# ============================================================
# VALIDATION / TEST
# Train'de öğrenilen encoder kullanılır
# ============================================================

X_val_encoded = preprocessor.transform(
    X_val
)

X_test_encoded = preprocessor.transform(
    X_test
)

# ============================================================
# BOYUT KONTROLÜ
# ============================================================

print("Train      :", X_train_encoded.shape)
print("Validation :", X_val_encoded.shape)
print("Test       :", X_test_encoded.shape)

In [ ]:
# ============================================================
# 8. PREPROCESSING'İN UYGULANMASI
# ============================================================
#
# preprocessor SADECE X_train üzerinde fit edilir.
#
# X_train:
#     fit_transform()
#
# X_validation:
#     transform()
#
# X_test:
#     transform()
#
# Böylece validation ve test verisi preprocessing
# aşamasında kullanılmaz ve veri sızıntısı önlenir.
# ============================================================


# TRAIN:
# Transformer parametreleri burada öğrenilir
X_train_encoded = preprocessor.fit_transform(
    X_train
)


# VALIDATION:
# Öğrenilen dönüşümler uygulanır
X_val_encoded = preprocessor.transform(
    X_val
)


# TEST:
# Öğrenilen dönüşümler uygulanır
X_test_encoded = preprocessor.transform(
    X_test
)


# ============================================================
# BOYUT KONTROLÜ
# ============================================================

print("Encoder sonrası X_train :", X_train_encoded.shape)
print("Encoder sonrası X_val   :", X_val_encoded.shape)
print("Encoder sonrası X_test  :", X_test_encoded.shape)

Preprocessing adımı yalnızca eğitim verisi üzerinde fit edilmiştir. Test verisi üzerinde herhangi bir istatistiksel bilgi öğrenilmemiş, yalnızca eğitim verisinden elde edilen dönüşümler uygulanmıştır. Böylece veri sızıntısı önlenerek model performansının gerçekçi şekilde değerlendirilmesi sağlanmıştır.

In [ ]:
# ============================================================
# 9. MODEL VERİSİ SON KONTROL
# ============================================================

print("=" * 70)
print("MODEL VERİSİ SON DURUM")
print("=" * 70)


print("Orijinal X:", X.shape)


print("\nTrain:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)


print("\nValidation:")
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)


print("\nTest:")
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


print("\nEncoder sonrası:")

print("X_train_encoded:",
      X_train_encoded.shape)

print("X_val_encoded  :",
      X_val_encoded.shape)

print("X_test_encoded :",
      X_test_encoded.shape)

In [ ]:
import numpy as np

SELECTION_COL = "tender_selectionmethod"

# TRAIN target üzerinden %75 fiyat eşiği
price_threshold = np.quantile(y_train, 0.75)

print(f"High-price threshold (75%): {price_threshold:.4f}")

# Başlangıçta bütün gözlemler eşit ağırlıklı
sample_weights = np.ones(len(y_train), dtype=np.float32)

# LOWEST_PRICE + yüksek fiyat
mask = (
    (X_train[SELECTION_COL].to_numpy() == "LOWEST_PRICE") &
    (np.asarray(y_train) >= price_threshold)
)

# Hedef grubun ağırlığını 2 katına çıkar
sample_weights[mask] = 2.0

print(f"\nTRAIN observations              : {len(sample_weights):,}")
print(f"LOWEST_PRICE + high-price      : {mask.sum():,}")
print(f"Normal weight (1.0)             : {(sample_weights == 1.0).sum():,}")
print(f"High weight (2.0)               : {(sample_weights == 2.0).sum():,}")

Bu kod, TRAIN verisinde tender_selectionmethod = LOWEST_PRICE olan ve hedef fiyatı %75 quantile'ın üzerinde bulunan gözlemlerin sample weight değerini 1.0'dan 2.0'a çıkararak modelin bu gruba daha fazla önem vermesini sağlar.

Sample Weighting yöntemi, residual analizinde tender_selectionmethod değişkeninin özellikle LOWEST_PRICE kategorisinde ve yüksek bütçeli ihalelerde sistematik under-prediction oluşturması nedeniyle uygulanmıştır. Bu yöntemde modelin verisi değiştirilmemekte veya yeni bir özellik oluşturulmamakta; yalnızca eğitim sırasında belirli gözlemlerin hatalarına verilen önem artırılmaktadır. Bunun için önce yalnızca X_train üzerinden hedef değişkenin 75. yüzdelik değeri hesaplanmış ve 14.2502 değeri yüksek fiyatlı ihalelerin başlangıç eşiği olarak belirlenmiştir. Daha sonra tender_selectionmethod = LOWEST_PRICE olan ve hedef değeri bu eşikten büyük veya eşit olan gözlemler belirlenmiştir. Eğitim setindeki 208.014 gözlemin 28.206'sının bu koşulu sağladığı görülmüş ve bu gözlemlere 2.0, diğer 179.808 gözleme ise 1.0 ağırlık verilmiştir. Böylece model, özellikle yüksek fiyatlı LOWEST_PRICE ihalelerinde yaptığı hataları eğitim sırasında iki kat daha önemli kabul ederek bu bölgedeki under-prediction eğilimini azaltmaya yönlendirilmiştir. Validation ve test verilerine herhangi bir ağırlık uygulanmamıştır; böylece modelin gerçek performansı değiştirilmemiş veri dağılımı üzerinde ölçülmeye devam edecektir.

## 🤖 12. AutoGluon ile Makine Öğrenmesi / Machine Learning with AutoGluon

This section prepares the training, validation, and test datasets for machine learning using AutoGluon. The target variable `tender_finalpriceUsd` is separated from the input features, while the training, validation, and test sets are used for model development and evaluation.

In this stage, the prepared datasets are AutoGluon modelleme sürecine aktarılmaktadır. Hedef değişken olarak `tender_finalpriceUsd` kullanılmakta, model özellikleri ise hedef değişkenden ayrı tutulmaktadır. AutoGluon, farklı makine öğrenmesi algoritmalarını otomatik olarak değerlendirerek uygun modelleri seçmekte ve hiperparametre optimizasyonu gerçekleştirerek tahmin performansını artırmayı amaçlamaktadır.

Bu aşamada veri sızıntısını önlemek amacıyla test verisi model eğitiminde kullanılmamakta ve yalnızca nihai model performansının değerlendirilmesi için ayrılmaktadır.

**Veri Biliminde Otomasyonu Sağlayan Temel AI Teknolojileri**

Bu kavramlar ve yaklaşımlar, modern yapay zeka ve veri bilimi süreçlerinin otomasyonunu, canlı sistemlere entegrasyonunu ve geliştirmesini kolaylaştıran temel disiplinleri oluşturmaktadır.

**AutoML,** verilen bir veri seti üzerinde en iyi makine öğrenimi algoritmasını seçen, eksik verileri tamamlayan, ölçeklendirme işlemlerini gerçekleştiren ve en yüksek performans veren modeli uçtan uca otomatik olarak inşa eden bir yaklaşımdır; bu süreçte PyCaret, Auto-sklearn ve AutoGluon gibi kütüphaneler yaygın şekilde kullanılır.

**MLOps,** eğitilen yapay zeka modellerinin sunucu veya bulut gibi canlı sistemlere hatasız bir şekilde entegre edilmesini, zamanla performans düşüşü yaşanıp yaşanmadığının izlenmesini ve yeni veriler geldikçe modelin otomatik olarak güncellenmesini kapsar; MLflow, Kubeflow ve DVC bu döngüyü yöneten başlıca araçlar arasındadır.

**NAS (Neural Architecture Search),** derin öğrenme ağlarında katman bağlantıları, aktivasyon fonksiyonları ve filtre seçimi gibi karmaşık mimari kararları manuel denemeler yerine algoritmik olarak binlerce olasılığı test ederek en optimum sinir ağı yapısını tasarlar ve bu alanda NASLib ile NNablaNAS gibi kütüphaneler öne çıkar.

**Low-Code / No-Code AI,** karmaşık kod yapıları veya Python kütüphaneleriyle vakit kaybetmek istemeyen kullanıcılara sürükle-bırak yöntemleri ya da minimal komutlarla web ve masaüstü tabanlı görsel yapay zeka arayüzleri sunar; Gradio, Streamlit ve çeşitli bulut AI platformları bu yaklaşımın popüler örneklerindendir.

**Otomatik Öznitelik Mühendisliği,** dağınık ve çoklu tablolarda (özellikle müşteri hareketleri, zaman serileri veya tarih-saat verilerinde) modelin öğrenme kapasitesini artıracak anlamlı yeni sütunlar ve matematiksel türevler (feature) oluşturur; Featuretools, TSFresh ve Featurewiz gibi araçlar bu süreci otomatikleştirmek için tercih edilir.

In [ ]:
import numpy as np
import polars as pl

# ============================================================
# AUTOML İÇİN 50.000 KAYITLIK STRATIFIED ÖRNEKLEM
# ============================================================
#
# Örneklem SADECE X_train içinden alınır.
#
# X_train:
#     208.014 kayıt
#
# AutoML:
#     50.000 kayıt
#
# Stratification:
#     price_bin + tender_supplytype + competition_level
#     → stratify_col
#
# Validation ve Test KULLANILMAZ.
#
# AutoML'ye preprocessing yapılmamış X_train verisi verilir.
# AutoGluon kendi preprocessing işlemlerini yapacaktır.
# ============================================================

# ============================================================
# 1. AYARLAR
# ============================================================

sample_size = 50_000
random_state = 42

# ============================================================
# 2. TRAIN KONTROLÜ
# ============================================================

print("=" * 70)
print("AUTOML TRAIN ÖRNEKLEMİ")
print("=" * 70)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

if X_train.height != len(y_train):
    raise ValueError(
        "X_train ve y_train satır sayıları eşleşmiyor."
    )

# ============================================================
# 3. TRAIN VERİSİNİ HAZIRLA
# ============================================================
#
# ÖNEMLİ:
# Örnekleme kaynağı X_train'dir.
#
# stratify_col X_train içinde varsa doğrudan kullanılır.
# Yoksa aynı TRAIN kayıtlarına ait stratify_col,
# df_finalprice[idx_train] üzerinden alınır.
# ============================================================

if "stratify_col" in X_train.columns:

    df_train_sampling = (
        X_train
        .select(model_features + ["stratify_col"])
        .with_columns(
            pl.Series(
                target,
                y_train.to_list()
            )
        )
    )

else:

    train_stratify = (
        df_finalprice[idx_train]
        .select("stratify_col")
    )

    if train_stratify.height != X_train.height:
        raise ValueError(
            "X_train ile TRAIN stratify verisinin "
            "satır sayıları eşleşmiyor."
        )

    df_train_sampling = (
        X_train
        .select(model_features)
        .with_columns(
            pl.Series(
                target,
                y_train.to_list()
            )
        )
        .hstack(train_stratify)
    )

print("\nTrain örneklem havuzu:", df_train_sampling.shape)

# ============================================================
# 4. TRAIN STRATIFY DAĞILIMI
# ============================================================

stratify_distribution = (
    df_train_sampling
    .group_by("stratify_col")
    .len()
    .sort("len", descending=True)
)

print(
    "Stratify kombinasyon sayısı:",
    stratify_distribution.height
)

# ============================================================
# 5. GRUP SAYILARI
# ============================================================

group_counts = (
    df_train_sampling
    .group_by("stratify_col")
    .len()
    .rename({"len": "group_count"})
)

total_train = df_train_sampling.height

# ============================================================
# 6. İDEAL ÖRNEKLEM SAYISI
# ============================================================

group_counts = group_counts.with_columns(
    (
        pl.col("group_count")
        / total_train
        * sample_size
    ).alias("ideal_sample_count")
)

# ============================================================
# 7. TABAN ÖRNEKLEM SAYISI
# ============================================================

group_counts = group_counts.with_columns(
    pl.col("ideal_sample_count")
    .floor()
    .cast(pl.Int64)
    .alias("sample_count")
)

current_total = (
    group_counts
    .select(
        pl.col("sample_count").sum()
    )
    .item()
)

remaining = sample_size - current_total

# ============================================================
# 8. KALAN KAYITLARI EN BÜYÜK KALANLARA DAĞIT
# ============================================================

group_counts = group_counts.with_columns(
    (
        pl.col("ideal_sample_count")
        - pl.col("sample_count")
    ).alias("remainder")
)

if remaining > 0:

    group_counts = (
        group_counts
        .sort(
            "remainder",
            descending=True
        )
        .with_columns(
            pl.when(
                pl.int_range(pl.len()) < remaining
            )
            .then(
                pl.col("sample_count") + 1
            )
            .otherwise(
                pl.col("sample_count")
            )
            .alias("sample_count")
        )
    )

# ============================================================
# 9. ÖRNEKLEM SAYISI KONTROLÜ
# ============================================================

calculated_total = (
    group_counts
    .select(
        pl.col("sample_count").sum()
    )
    .item()
)

if calculated_total != sample_size:
    raise ValueError(
        f"Örneklem boyutu hatalı: "
        f"{calculated_total} != {sample_size}"
    )

# ============================================================
# 10. RANDOM STATE
# ============================================================

rng = np.random.default_rng(random_state)

# ============================================================
# 11. HER STRATIFY GRUBUNDAN ÖRNEK AL
# ============================================================

sampled_parts = []

for row in group_counts.iter_rows(named=True):

    group_name = row["stratify_col"]
    n_samples = row["sample_count"]

    if n_samples <= 0:
        continue

    group_df = (
        df_train_sampling
        .filter(
            pl.col("stratify_col") == group_name
        )
    )

    # Güvenlik kontrolü
    if n_samples > group_df.height:
        n_samples = group_df.height

    selected_indices = rng.choice(
        group_df.height,
        size=n_samples,
        replace=False
    )

    sampled_parts.append(
        group_df[selected_indices]
    )

# ============================================================
# 12. ÖRNEKLEMLERİ BİRLEŞTİR
# ============================================================

df_automl_sample = pl.concat(
    sampled_parts
)

# ============================================================
# 13. KARIŞTIR
# ============================================================

shuffle_indices = rng.permutation(
    df_automl_sample.height
)

df_automl_sample = (
    df_automl_sample[shuffle_indices]
)

# ============================================================
# 14. ÖRNEKLEM SONUÇ KONTROLÜ
# ============================================================

print("\n" + "=" * 70)
print("AUTOML ÖRNEKLEM SONUCU")
print("=" * 70)

print(
    "Train toplam kayıt       :",
    total_train
)

print(
    "AutoML örneklem boyutu   :",
    df_automl_sample.shape
)

print(
    "Beklenen kayıt sayısı    :",
    sample_size
)

if df_automl_sample.height != sample_size:
    raise ValueError(
        "AutoML örneklem boyutu 50.000 değil."
    )

# ============================================================
# 15. AUTOML STRATIFY DAĞILIMI
# ============================================================

print("\n" + "=" * 70)
print("AUTOML ÖRNEKLEM STRATIFY DAĞILIMI")
print("=" * 70)

display(
    df_automl_sample
    .group_by("stratify_col")
    .len()
    .sort("len", descending=True)
)

# ============================================================
# 16. TRAIN vs AUTOML DAĞILIM KONTROLÜ
# ============================================================

train_dist = (
    df_train_sampling
    .group_by("stratify_col")
    .len()
    .rename({"len": "train_count"})
)

automl_dist = (
    df_automl_sample
    .group_by("stratify_col")
    .len()
    .rename({"len": "automl_count"})
)

distribution = (
    train_dist
    .join(
        automl_dist,
        on="stratify_col",
        how="full",
        coalesce=True
    )
    .with_columns([
        pl.col("train_count").fill_null(0),
        pl.col("automl_count").fill_null(0)
    ])
    .with_columns([
        (
            pl.col("train_count")
            / total_train
            * 100
        ).alias("train_pct"),

        (
            pl.col("automl_count")
            / sample_size
            * 100
        ).alias("automl_pct")
    ])
    .with_columns(
        (
            pl.col("automl_pct")
            - pl.col("train_pct")
        ).alias("percentage_difference")
    )
    .with_columns(
        pl.col("percentage_difference")
        .abs()
        .alias("absolute_difference")
    )
)

mean_abs_diff = (
    distribution
    .select(
        pl.col("absolute_difference").mean()
    )
    .item()
)

max_abs_diff = (
    distribution
    .select(
        pl.col("absolute_difference").max()
    )
    .item()
)

print("\nOrtalama mutlak yüzde farkı:",
      f"{mean_abs_diff:.4f}%")

print(
    "Maksimum mutlak yüzde farkı :",
    f"{max_abs_diff:.4f}%"
)

# ============================================================
# 17. MODEL VERİSİ
# ============================================================
#
# stratify_col AutoML'ye verilmez.
# Target ayrı olarak tutulur.
#
# X_automl:
#     50.000 x model_features
#
# y_automl:
#     50.000
# ============================================================

X_automl = (
    df_automl_sample
    .select(model_features)
)

y_automl = (
    df_automl_sample
    .select(target)
    .to_series()
)

# ============================================================
# 18. SON KONTROL
# ============================================================

print("\n" + "=" * 70)
print("AUTOML MODEL VERİSİ")
print("=" * 70)

print("X_automl:", X_automl.shape)
print("y_automl:", y_automl.shape)

print(
    "\nX_automl kolon sayısı:",
    len(X_automl.columns)
)

print(
    "Target:",
    target
)

print("\nX_automl kolonları:")

for i, col in enumerate(
    X_automl.columns,
    1
):
    print(f"{i:2}. {col}")

# ============================================================
# 19. SON DOĞRULAMALAR
# ============================================================

assert X_automl.height == sample_size
assert y_automl.len() == sample_size
assert len(X_automl.columns) == len(model_features)

print("\n" + "=" * 70)
print("✓ AUTOML VERİSİ HAZIR")
print("=" * 70)

print(
    f"✓ {sample_size:,} kayıt"
)

print(
    f"✓ {len(model_features)} feature"
)

print(
    "✓ Sadece TRAIN üzerinden örnekleme"
)

print(
    "✓ Stratified sampling"
)

print(
    "✓ Validation kullanılmadı"
)

print(
    "✓ Test kullanılmadı"
)

print(
    "✓ Preprocessing uygulanmadı"
)

Bu işlem, AutoML model eğitim süreçlerini hızlandırmak ve kaynak kullanımını optimize etmek amacıyla, yalnızca Eğitim (TRAIN) veri setinden katmanlı (stratified) bir yöntemle 50.000 kayıtlık temsilci bir alt veri seti oluşturmaktadır. İşlem sırasında doğrulama (validation) ve test verilerine hiçbir şekilde temas edilmeyerek veri sızıntısı (data leakage) kesin olarak engellenmiş; fiyat aralığı, ihale tedarik türü ve rekabet seviyesini temsil eden birleşik değişken (stratify_col) temel alınarak ana veri setindeki sınıf dağılım oranları birebir korunmuştur.208.014 kayıtlık ana eğitim verisinin yaklaşık %24'ünü temsil eden bu alt kümede, toplam 90 farklı katman kombinasyonunun tamamı ana veri setindeki ağırlıklarına paralel biçimde yer almıştır. Nitekim katman bazlı maksimum sapmanın %0,0010 seviyesinde kalması (ortalama mutlak yüzde farkı %0,0005), ana veri setindeki dağılımın kusursuz bir şekilde aktarıldığını doğrulamaktadır. En yüksek payı 1.350 kayıt ile orta fiyat seviyesindeki düşük rekabetli tedarik ihaleleri (q5_SUPPLIES_Low Competition) alırken, 149 kayıtlık en küçük alt grup dahi (q1_SUPPLIES_High Competition) veri kaybı yaşanmadan temsil edilmiştir.Küsuratlı örneklem payları Hamilton algoritması ile düzeltilip SEED = 42 parametresiyle rastgele karıştırılarak sürecin tekrarlanabilirliği garantilenmiştir. Ardından stratification amaçlı geçici değişkenler elenerek AutoML eğitiminde kullanılmak üzere 20 öz nitelikten oluşan X_automl ($50.000 \times 20$) ile hedef değişken y_automl (log_tender_finalpriceUsd, $50.000$) matrisleri eksiksiz bir şekilde hazırlanmıştır.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# TRAIN vs AUTOML STRATIFY DAĞILIM GÖRSELLEŞTİRME
# ============================================================

# TRAIN:
# 208.014 kayıt
#
# AUTOML:
# 50.000 kayıt
#
# Amaç:
# AutoML örnekleminin TRAIN dağılımını ne kadar
# koruduğunu görsel olarak incelemek.
#
# Stratification:
# price_bin + tender_supplytype + competition_level
# ↓
# stratify_col

# ============================================================
# 1. TRAIN DAĞILIMI
# ============================================================

train_dist = (
    df_finalprice[idx_train]
    .select("stratify_col")
    .group_by("stratify_col")
    .len()
    .rename({"len": "train_count"})
)

# ============================================================
# 2. AUTOML DAĞILIMI
# ============================================================

automl_dist = (
    df_automl_sample
    .select("stratify_col")
    .group_by("stratify_col")
    .len()
    .rename({"len": "automl_count"})
)

# ============================================================
# 3. DAĞILIMLARI BİRLEŞTİR
# ============================================================

distribution = (
    train_dist
    .join(
        automl_dist,
        on="stratify_col",
        how="full",
        coalesce=True
    )
    .fill_null(0)
)

# ============================================================
# 4. TOPLAM KAYIT SAYILARI
# ============================================================

train_total = (
    distribution
    .select(pl.col("train_count").sum())
    .item()
)

automl_total = (
    distribution
    .select(pl.col("automl_count").sum())
    .item()
)

# ============================================================
# 5. YÜZDELERİ HESAPLA
# ============================================================

distribution = distribution.with_columns([

    (
        pl.col("train_count")
        / train_total
        * 100
    ).alias("train_pct"),

    (
        pl.col("automl_count")
        / automl_total
        * 100
    ).alias("automl_pct")
])

# ============================================================
# 6. YÜZDE FARKI
# ============================================================

distribution = distribution.with_columns(

    (
        pl.col("automl_pct")
        - pl.col("train_pct")
    ).alias("percentage_difference")
)

# ============================================================
# 7. MUTLAK YÜZDE FARKI
# ============================================================

distribution = distribution.with_columns(

    pl.col("percentage_difference")
    .abs()
    .alias("absolute_difference")
)

# ============================================================
# 8. EN ÇOK FARK GÖSTEREN 10 GRUP
# ============================================================

top10 = (
    distribution
    .sort("absolute_difference", descending=True)
    .head(10)
    .sort("absolute_difference")
)

# ============================================================
# 9. GRAFİK VERİLERİ
# ============================================================

labels = top10["stratify_col"].to_list()

train_pct = top10["train_pct"].to_list()
automl_pct = top10["automl_pct"].to_list()

# ============================================================
# 10. YATAY BAR GRAFİĞİ
# ============================================================

y = np.arange(len(labels))

height = 0.35

fig, ax = plt.subplots(figsize=(15, 9))

ax.barh(
    y - height / 2,
    train_pct,
    height,
    label="Train"
)

ax.barh(
    y + height / 2,
    automl_pct,
    height,
    label="AutoML"
)

# ============================================================
# 11. DEĞERLERİ BARLARIN ÜZERİNE YAZ
# ============================================================

for i, value in enumerate(train_pct):

    ax.text(
        value + 0.01,
        i - height / 2,
        f"{value:.3f}%",
        va="center",
        fontsize=9
    )

for i, value in enumerate(automl_pct):

    ax.text(
        value + 0.01,
        i + height / 2,
        f"{value:.3f}%",
        va="center",
        fontsize=9
    )

# ============================================================
# 12. GRAFİK AYARLARI
# ============================================================

ax.set_yticks(y)

ax.set_yticklabels(
    labels,
    fontsize=9
)

ax.set_xlabel(
    "Dağılım (%)",
    fontsize=12
)

ax.set_ylabel(
    "Stratify Grubu",
    fontsize=12
)

ax.set_title(
    "Train vs AutoML – En Çok Fark Gösteren 10 Stratify Grubu",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.legend()

ax.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.show()

Grafik, Train veri seti ile AutoML için ayrılan küme arasındaki en çok fark gösteren 10 bileşik stratify grubunun yüzdesel dağılımlarını karşılaştırmaktadır.

Yapılan incelemede, iki küme arasındaki farkların %0.001 (binde bir) düzeyinde kaldığı görülmektedir. Bu durum, veri seti bölünürken uygulanan katmanlama (stratification) işleminin son derece hassas ve başarılı bir şekilde gerçekleştiğini kanıtlamaktadır.

In [ ]:
# ============================================================
# AUTOML MODEL VERİSİ SON KONTROL
# ============================================================

print("=" * 70)
print("AUTOML MODEL VERİSİ")
print("=" * 70)

print(f"X_automl shape : {X_automl.shape}")
print(f"y_automl shape : {y_automl.shape}")

print("\nTarget:")
print(target)

print("\nX_automl kolonları:")
for i, col in enumerate(X_automl.columns, 1):
    print(f"{i:2}. {col}")

print("\nTarget istatistikleri:")
print(
    y_automl.describe()
)

In [ ]:
# AUTOGLUON KURULUMU

!pip install -q "autogluon.tabular"

In [ ]:
# AUTOGLUON IMPORT

from autogluon.tabular import TabularPredictor

print("AutoGluon başarıyla yüklendi.")

In [ ]:
# AUTOML İÇİN MODEL VERİSİNİN HAZIRLANMASI

from autogluon.tabular import TabularPredictor

# X + target birleştirilir
df_automl = X_automl.with_columns(
    y_automl.alias(target)
)

print("AUTOML MODEL VERİSİ")

print("Shape:", df_automl.shape)
print("Feature sayısı:", X_automl.width)
print("Target:", target)

print("\nKolonlar:")
for i, col in enumerate(X_automl.columns, 1):
    print(f"{i:2}. {col}")

AutoGluon'da Scaling Kullanımı

AutoGluon eğitiminde feature değişkenlerine manuel olarak StandardScaler, MinMaxScaler veya benzeri bir scaling işlemi uygulanmasına gerek yoktur. AutoGluon, farklı model türlerinin ihtiyaçlarına uygun preprocessing işlemlerini kendi pipeline yapısı içerisinde otomatik olarak yönetir. Özellikle Neural Network modellerinde modelin ihtiyaç duyduğu preprocessing AutoGluon tarafından ele alınırken, Random Forest, Extra Trees, LightGBM ve CatBoost gibi ağaç tabanlı modeller için scaling zaten gerekli değildir. Bu nedenle AutoGluon'a veri aktarılırken feature'ların önceden manuel olarak ölçeklendirilmemesi ve preprocessing işlemlerinin AutoGluon'a bırakılması tercih edilmiştir. Target değişkeni (log_tender_finalpriceUsd) için de manuel standardizasyon uygulanmamıştır. Böylece farklı model aileleri aynı veri seti üzerinde AutoGluon'un kendi preprocessing ve modelleme mekanizmaları kullanılarak adil şekilde karşılaştırılabilmektedir

In [ ]:
# ============================================================
# AUTOGLUON - MODEL EĞİTİMİ
# ============================================================

import os
import polars as pl

from autogluon.tabular import TabularPredictor


# ============================================================
# 1. AYARLAR
# ============================================================

TARGET = "log_tender_finalpriceUsd"

AUTOGLUON_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/sage/AutoGluon_1Hour"
)

TIME_LIMIT = 3600  # 1 saat


# ============================================================
# 2. VERİ KONTROLLERİ
# ============================================================

if not isinstance(X_automl, pl.DataFrame):
    raise TypeError(
        "X_automl Polars DataFrame olmalıdır."
    )

if not isinstance(y_automl, pl.Series):
    raise TypeError(
        "y_automl Polars Series olmalıdır."
    )

if y_automl.null_count() > 0:
    raise ValueError(
        "Target içerisinde null değerler bulunuyor."
    )


print("=" * 80)
print("AUTOGLUON MODEL EĞİTİMİ")
print("=" * 80)

print(f"X boyutu      : {X_automl.shape}")
print(f"y boyutu      : {y_automl.shape}")
print(f"Target        : {TARGET}")
print(f"Eğitim süresi : {TIME_LIMIT / 60:.0f} dakika")
print(f"Kayıt yolu    : {AUTOGLUON_PATH}")


# ============================================================
# 3. X + y BİRLEŞTİR
# ============================================================

df_automl = X_automl.with_columns(
    y_automl.alias(TARGET)
)

print(f"\nAutoGluon veri boyutu: {df_automl.shape}")


# ============================================================
# 4. POLARS → PANDAS
# AutoGluon TabularPredictor Pandas DataFrame kullanır
# ============================================================

automl_data = df_automl.to_pandas()


# ============================================================
# 5. AUTOGLUON PREDICTOR
# ============================================================

predictor = TabularPredictor(
    label=TARGET,
    problem_type="regression",
    eval_metric="root_mean_squared_error",
    path=AUTOGLUON_PATH
)


# ============================================================
# 6. MODEL EĞİTİMİ
# ============================================================

print("\n" + "=" * 80)
print("AUTOGLUON EĞİTİMİ BAŞLIYOR")
print("=" * 80)

predictor.fit(
    train_data=automl_data,
    time_limit=TIME_LIMIT,
    presets="best_quality",
    verbosity=2
)


# ============================================================
# 7. EĞİTİM SONU
# ============================================================

print("\n" + "=" * 80)
print("AUTOGLUON EĞİTİMİ TAMAMLANDI")
print("=" * 80)

print(f"Model kayıt yolu : {AUTOGLUON_PATH}")
print(f"Best Model       : {predictor.model_best}")

Bu kod bloğu, hazırlanan 50.000 kayıtlık örneklem verisi üzerinde otomatik makine öğrenmesi süreçlerini yürütmek amacıyla AutoGluon kütüphanesini kullanarak bir regresyon modeli eğitimi başlatır. İlk aşamada hedef değişken (log_tender_finalpriceUsd) ve Google Drive üzerindeki model kayıt yolu tanımlanıp Polars formatındaki öz nitelikler (X_automl) ile hedef veri (y_automl) tipleri ve eksik değer kontrollerinden geçirilerek birleştirilir. AutoGluon yapısına uyum sağlamak adına veri seti Pandas DataFrame formatına dönüştürüldükten sonra değerlendirme metriği olarak RMSE (Root Mean Squared Error) belirlenerek bir TabularPredictor nesnesi oluşturulur. Son aşamada ise yüksek performanslı ensemble ve istifleme (stacking) tekniklerini devreye sokan presets="best_quality" konfigürasyonuyla 1 saatlik (3600 saniye) zaman kısıtı dahlinde eğitim süreci çalıştırılır ve eğitilen en başarılı model ile çıktılar belirtilen dizine otomatik olarak kaydedilir.

In [ ]:
import os

MODELS_PATH = os.path.join(
    AUTOGLUON_PATH,
    "models"
)

print("Models klasörü:", MODELS_PATH)
print()

for root, dirs, files in os.walk(MODELS_PATH):
    level = root.replace(MODELS_PATH, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)

        print(
            f"{indent}    {file} "
            f"({size_mb:.2f} MB)"
        )

Bu kod, AUTOGLUON_PATH içindeki "models" klasörünü özyinelemeli (os.walk) olarak tarar. Dosya dizin yapısını klasör derinliğine göre girintilendirerek ağaç formunda ekrana yazdırır ve her dosyanın boyutunu bayttan megabayta (MB) çevirerek yanında gösterir. Temel amacı, eğitilen modellerin dosya hiyerarşisini ve kapladığı disk alanını özetlemektir.

In [ ]:
import os
from autogluon.tabular import TabularPredictor

AUTOGLUON_PATH = "/content/drive/MyDrive/Colab Notebooks/sage/AutoGluon_1Hour"

print("=" * 80)
print("AUTOGLUON KLASÖR KONTROLÜ")
print("=" * 80)

print("Klasör mevcut mu:", os.path.exists(AUTOGLUON_PATH))
print("Klasör:", AUTOGLUON_PATH)

if os.path.exists(AUTOGLUON_PATH):
    print("\nİçerik:")
    for item in os.listdir(AUTOGLUON_PATH):
        print(" -", item)

# Mevcut AutoGluon eğitimini yükle
predictor = TabularPredictor.load(AUTOGLUON_PATH)

print("\n" + "=" * 80)
print("AUTOGLUON EĞİTİMİ YÜKLENDİ")
print("=" * 80)

print("Problem type:", predictor.problem_type)
print("Target:", predictor.label)

print("\nEğitilmiş modeller:")
print(predictor.model_names())

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ============================================================
# AUTOGLUON LEADERBOARD
# ============================================================

leaderboard = predictor.leaderboard(
    extra_info=True,
    silent=True
)

# ============================================================
# TARGET: LOG
# ============================================================

y_log = np.asarray(
    y_automl.to_numpy() if hasattr(y_automl, "to_numpy") else y_automl,
    dtype=float
)

# ============================================================
# MODEL SONUÇLARI
# ============================================================

results = []

for _, row in leaderboard.iterrows():
    model_name = str(row["model"])

    try:
        pred_log = np.asarray(
            predictor.predict_oof(model=model_name),
            dtype=float
        )

        rmse_log = float(-row["score_val"])

        results.append({
            "Model": model_name,
            "RMSE_Log": rmse_log,
            "Fit_Time": float(row["fit_time"]),
            "Pred_Time": float(row["pred_time_val"])
        })

    except Exception as e:
        print(f"{model_name} hesaplanamadı: {e}")

# ============================================================
# SONUÇ DATAFRAME
# ============================================================

df = (
    pd.DataFrame(results)
    .sort_values("RMSE_Log", ascending=True)
    .reset_index(drop=True)
)

df.insert(0, "Rank", np.arange(1, len(df) + 1))

# ============================================================
# SONUÇ TABLOSU
# ============================================================

print("=" * 80)
print("AUTOGLUON MODEL SONUÇLARI")
print("=" * 80)

print(
    df[
        [
            "Rank",
            "Model",
            "RMSE_Log"
        ]
    ].to_string(index=False)
)

# ============================================================
# LOG RMSE GRAFİĞİ
# ============================================================

plot_df = df.sort_values(
    "RMSE_Log",
    ascending=True
)

fig = px.bar(
    plot_df,
    x="Model",
    y="RMSE_Log",
    text="RMSE_Log",
    title="AutoGluon Models — Log RMSE",
    labels={
        "RMSE_Log": "Validation RMSE (Log)",
        "Model": "Model"
    },
    hover_data=[
        "Rank",
        "RMSE_Log",
        "Fit_Time",
        "Pred_Time"
    ]
)

fig.update_traces(
    texttemplate="%{text:.4f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    height=650,
    width=max(1100, len(plot_df) * 140),
    title_x=0.5,
    xaxis_tickangle=-45,
    yaxis_title="Validation RMSE (Log)",
    xaxis_title="Model",
    margin=dict(
        l=100,
        r=50,
        t=100,
        b=180
    ),
    showlegend=False
)

fig.show()

AutoGluon ve XGBoost model sonuçları, değerlendirmenin yapıldığı ölçeğe bağlı olarak farklı öne çıkan modellere işaret etmektedir. Modellerin eğitildiği logaritmik hedef değişken üzerindeki hata metriği (Validation RMSE Log) esas alındığında, AutoGluon topluluk modeli WeightedEnsemble_L3 en düşük logaritmik hatayı vererek 1.1164 değeri ile ilk sırada yer almıştır. XGBoost altyapısını kullanan XGBoost_BAG_L2 ve ExtraTreesMSE_BAG_L2 modelleri ise 1.1220'lik aynı hata değerini elde ederek ilk sıradaki WeightedEnsemble_L3'ün hemen ardından gelmiştir.

XGBoost_BAG_L1 ile XGBoost_BAG_L2 arasındaki temel fark, modellerin mimaride bulundukları katman ve buna bağlı olarak kullandıkları girdi verisidir. Seviye 1'de yer alan XGBoost_BAG_L1, yalnızca veri setindeki ham öznitelikleri (Orijinal Feature'lar) kullanarak eğitilen bağımsız bir temel modelken; Seviye 2'de yer alan XGBoost_BAG_L2, hem ham öznitelikleri hem de birinci katmandaki (L1) tüm modellerin ürettiği katman dışı tahminleri (L1 OOF Tahminleri) bir arada girdi olarak kullanan birleştirici bir meta-modeldir. L2 katmanı, L1 modellerinin öğrendiği kalıpları ve kolektif tahmin gücünü harmanladığı için performansı belirgin şekilde artırmış ve 1.1791 olan Log RMSE hata değerini 1.1220 seviyesine düşürerek daha yüksek bir doğruluk elde etmiştir.

In [ ]:
model_info = predictor.info()["model_info"]

print(model_info["XGBoost_BAG_L2"])

In [ ]:
# AUTOGLUON MODEL ANALİZİ - XGBoost_BAG_L2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

MODEL_NAME = "XGBoost_BAG_L2"

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150


# YARDIMCI FONKSİYONLAR

def show_visual_table(df, title, figsize=(12, 5), font_size=10, row_height=0.08):
    """DataFrame'i görsel tablo olarak gösterir."""

    if df is None or df.empty:
        print(f"{title}: gösterilecek veri bulunamadı.")
        return

    rows = len(df)
    height = max(figsize[1], rows * row_height + 1.5)

    fig, ax = plt.subplots(figsize=(figsize[0], height))
    ax.axis("off")

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.5)

    ax.set_title(title, fontsize=16, fontweight="bold", pad=20)

    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(0.8)

        if row == 0:
            cell.set_text_props(fontweight="bold")
            cell.set_height(0.10)
        else:
            cell.set_height(0.07)

    plt.tight_layout()
    plt.show()


def format_value(value):
    """Değerleri görsel tablolar için okunabilir hale getirir."""

    if value is None:
        return "None"

    if isinstance(value, float):
        if np.isnan(value):
            return "NaN"
        if abs(value) >= 1000:
            return f"{value:,.4f}"
        return f"{value:.6f}"

    return str(value)

# 1. MODELİ YÜKLE

model = predictor._trainer.load_model(MODEL_NAME)

# 2. GENEL MODEL BİLGİLERİ

general_info = {
    "Model": MODEL_NAME,
    "Model Sınıfı": type(model).__name__,
    "Problem Type": getattr(model, "problem_type", None),
    "Evaluation Metric": getattr(model, "eval_metric", None),
    "Stopping Metric": getattr(model, "stopping_metric", None),
    "Path": getattr(model, "path", None)
}

general_df = pd.DataFrame(
    [[key, format_value(value)] for key, value in general_info.items()],
    columns=["Özellik", "Değer"]
)

show_visual_table(
    general_df,
    "XGBoost_BAG_L2 — MODEL GENEL BİLGİLERİ",
    figsize=(14, 5),
    font_size=11
)

# 3. MODEL PARAMETRELERİ

params = getattr(model, "params", {})

params_df = pd.DataFrame(
    [[key, format_value(value)] for key, value in params.items()],
    columns=["Parametre", "Değer"]
)

show_visual_table(
    params_df,
    "XGBoost_BAG_L2 — MODEL PARAMETRELERİ",
    figsize=(14, max(5, len(params_df) * 0.35)),
    font_size=10
)


# 4. STACKING / BAGGING

stacking_params = {
    "use_orig_features": getattr(model, "use_orig_features", None),
    "valid_stacker": getattr(model, "valid_stacker", None),
    "max_base_models": getattr(model, "max_base_models", None),
    "max_base_models_per_type": getattr(model, "max_base_models_per_type", None),
    "save_bag_folds": getattr(model, "save_bag_folds", None),
    "stratify": getattr(model, "stratify", None),
    "bin": getattr(model, "bin", None),
    "n_bins": getattr(model, "n_bins", None),
    "vary_seed_across_folds": getattr(model, "vary_seed_across_folds", None),
    "custom_splits": getattr(model, "custom_splits", None),
    "use_child_oof": getattr(model, "use_child_oof", None),
    "model_random_seed": getattr(model, "model_random_seed", None)
}

stacking_df = pd.DataFrame(
    [[key, format_value(value)] for key, value in stacking_params.items()],
    columns=["Stacking / Bagging Parametresi", "Değer"]
)

show_visual_table(
    stacking_df,
    "XGBoost_BAG_L2 — STACKING / BAGGING YAPISI",
    figsize=(15, 7),
    font_size=10
)


# 5. L1 MODELLERİ

base_model_names = getattr(model, "base_model_names", [])

if base_model_names is None:
    base_model_names = []

child_df = pd.DataFrame({
    "L1 Model": base_model_names
})

show_visual_table(
    child_df,
    "XGBoost_BAG_L2 — L1 CHILD MODELLERİ",
    figsize=(14, max(4, len(child_df) * 0.35)),
    font_size=10
)

# 6. MODEL MİMARİ ÖZETİ

l1_count = len(base_model_names)
child_count = getattr(model, "n_children", None)

layer_info = pd.DataFrame({
    "Bilgi": [
        "Model Adı",
        "Model Tipi",
        "Stacking Layer",
        "Bagging",
        "Orijinal Feature Kullanımı",
        "OOF Tahmin Kullanımı",
        "L1 Model Sayısı",
        "Child Model Sayısı"
    ],
    "Değer": [
        MODEL_NAME,
        "XGBoost",
        "L2",
        "BAG",
        format_value(getattr(model, "use_orig_features", None)),
        format_value(getattr(model, "use_child_oof", None)),
        l1_count,
        child_count
    ]
})

show_visual_table(
    layer_info,
    "XGBoost_BAG_L2 — MODEL MİMARİ ÖZETİ",
    figsize=(14, 6),
    font_size=11
)

# 7. XGBOOST HİPERPARAMETRELERİ

xgb_param_names = [
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda"
]

xgb_params = {
    key: params[key]
    for key in xgb_param_names
    if key in params
}

xgb_df = pd.DataFrame({
    "XGBoost Parametresi": list(xgb_params.keys()),
    "Değer": [format_value(value) for value in xgb_params.values()]
})

show_visual_table(
    xgb_df,
    "XGBoost_BAG_L2 — XGBOOST HİPERPARAMETRELERİ",
    figsize=(14, 6),
    font_size=11
)

# 8. XGBOOST HİPERPARAMETRE GRAFİĞİ

if xgb_params:

    numeric_params = pd.Series(xgb_params, dtype=float)

    min_value = numeric_params.min()
    max_value = numeric_params.max()

    if max_value != min_value:
        normalized = (numeric_params - min_value) / (max_value - min_value)
    else:
        normalized = numeric_params * 0

    fig, ax = plt.subplots(figsize=(13, 7))

    ax.barh(
        normalized.index,
        normalized.values
    )

    ax.set_xlabel("Normalize Edilmiş Değer", fontsize=11)
    ax.set_ylabel("Parametre", fontsize=11)
    ax.set_title(
        "XGBoost_BAG_L2 — Hiperparametre Profili",
        fontsize=16,
        fontweight="bold",
        pad=15
    )

    ax.set_xlim(0, 1.15)
    ax.grid(axis="x", alpha=0.3)

    for i, (name, value) in enumerate(numeric_params.items()):
        ax.text(
            normalized.iloc[i] + 0.02,
            i,
            f"{value:.5g}",
            va="center",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

# 9. L1 MODEL ORANLARI

if base_model_names:

    l1_ratio = 100 / len(base_model_names)

    l1_ratio_df = pd.DataFrame({
        "L1 Model": base_model_names,
        "Model Oranı (%)": [l1_ratio] * len(base_model_names)
    })

    show_visual_table(
        l1_ratio_df,
        "XGBoost_BAG_L2 — L1 MODEL ORANLARI",
        figsize=(14, max(4, len(l1_ratio_df) * 0.35)),
        font_size=10
    )

    fig, ax = plt.subplots(figsize=(14, 7))

    bars = ax.barh(
        l1_ratio_df["L1 Model"],
        l1_ratio_df["Model Oranı (%)"]
    )

    ax.set_xlabel("Model İçerisindeki Oran (%)", fontsize=11)
    ax.set_ylabel("L1 Model", fontsize=11)
    ax.set_title(
        "XGBoost_BAG_L2 — L1 Modellerinin Model İçerisindeki Oranı",
        fontsize=16,
        fontweight="bold",
        pad=15
    )

    ax.set_xlim(0, max(l1_ratio * 1.35, 10))
    ax.grid(axis="x", alpha=0.3)

    for bar, value in zip(bars, l1_ratio_df["Model Oranı (%)"]):
        ax.text(
            value + 0.05,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}%",
            va="center",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()


# 10. MODEL KATMANLARI — DONUT GRAFİĞİ

if architecture_counts:

    labels = list(architecture_counts.keys())
    values = list(architecture_counts.values())

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.pie(
        values,
        labels=labels,
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops={"width": 0.42}
    )

    ax.set_title(
        "XGBoost_BAG_L2 — Model Katman Dağılımı",
        fontsize=16,
        fontweight="bold",
        pad=20
    )

    plt.tight_layout()
    plt.show()

# 11. MODEL MİMARİSİ

fig, ax = plt.subplots(figsize=(16, 9))
ax.axis("off")

boxes = [
    (0.03, 0.55, 0.20, 0.25, "ORİJİNAL\nFEATURE'LAR"),
    (0.36, 0.68, 0.24, 0.18, f"L1\nMODELLERİ\n({l1_count} MODEL)"),
    (0.36, 0.32, 0.24, 0.18, "L1 OOF\nTAHMİNLERİ"),
    (0.70, 0.55, 0.24, 0.25, "XGBOOST\nMSE — L2"),
    (0.70, 0.15, 0.24, 0.20, f"L2 BAGGING\n{child_count} CHILD")
]

for x, y, w, h, text in boxes:

    rectangle = plt.Rectangle(
        (x, y),
        w,
        h,
        fill=False,
        linewidth=2.5
    )

    ax.add_patch(rectangle)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold"
    )

arrows = [
    ((0.23, 0.675), (0.36, 0.77)),
    ((0.23, 0.675), (0.36, 0.41)),
    ((0.60, 0.77), (0.70, 0.675)),
    ((0.60, 0.41), (0.70, 0.675)),
    ((0.82, 0.55), (0.82, 0.35))
]

for start, end in arrows:

    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 2.5
        }
    )

ax.set_title(
    "XGBoost_BAG_L2 — AutoGluon Model Mimarisi",
    fontsize=18,
    fontweight="bold",
    pad=20
)

plt.tight_layout()
plt.show()


# 12. ÖZET RAPOR

summary_df = pd.DataFrame({
    "Ölçüm": [
        "Model",
        "Model Tipi",
        "Stacking Layer",
        "L1 Model Sayısı",
        "L2 Child Model Sayısı",
        "Orijinal Feature Kullanımı",
        "OOF Tahmin Kullanımı",
        "N_estimators",
        "Max Depth",
        "Learning Rate"
    ],
    "Değer": [
        MODEL_NAME,
        "XGBoost",
        "L2",
        l1_count,
        child_count,
        format_value(getattr(model, "use_orig_features", None)),
        format_value(getattr(model, "use_child_oof", None)),
        format_value(params.get("n_estimators", None)),
        format_value(params.get("max_depth", None)),
        format_value(params.get("learning_rate", None))
    ]
})

show_visual_table(
    summary_df,
    "XGBoost_BAG_L2 — ÖZET MODEL RAPORU",
    figsize=(14, 7),
    font_size=11
)


print("\n" + "=" * 80)
print("XGBOOST_BAG_L2 GÖRSEL ANALİZİ TAMAMLANDI")
print("=" * 80)

print(f"Model              : {MODEL_NAME}")
print(f"L1 Model Sayısı    : {l1_count}")
print(f"L2 Child Model     : {child_count}")

if "n_estimators" in params:
    print(f"N Estimators       : {params['n_estimators']}")

if "max_depth" in params:
    print(f"Max Depth          : {params['max_depth']}")

if "learning_rate" in params:
    print(f"Learning Rate      : {params['learning_rate']}")

print("=" * 80)

Şemadaki model mimarisi, XGBoost_BAG_L2 modelinin Çok Katmanlı Yığınlama (Multi-layer Stacking) ve Torbalama (Bagging) tekniklerini bir arada nasıl kullandığını göstermektedir. Süreç, ham özniteliklerin (Orijinal Feature'lar) alınmasıyla başlar; bu veriler ilk katmanda (L1) yer alan temel modellere beslenir. L1 katmanındaki modeller, veri sızıntısını ve aşırı öğrenmeyi önlemek amacıyla k-fold çapraz doğrulama ile eğitilerek katman dışı tahminlerini (L1 OOF Tahminleri) üretir. İkinci katmanda (L2) yer alan birleştirici XGBoost meta-modeli ise hem orijinal öznitelikleri hem de L1 modellerinden gelen OOF tahminlerini girdi olarak alır. Son aşamada, modelin kararlılığını artırmak ve varyansı minimize etmek amacıyla L2 seviyesindeki XGBoost modeli de torbalama (L2 Bagging Fold'ları) yaklaşımıyla eğitilerek nihai regresyon tahminleri elde edilir.

## W & B

In [ ]:
import json
import numpy as np
import pandas as pd
import wandb

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

wandb.login()

PROJECT_NAME = "GPPD_XGBoost_Optuna"
ENTITY_NAME = "iremerkurt34-personal"

TARGET_BEST_MODEL = "XGBoost_BAG_L2"
MAX_CHART_POINTS = 10000

leaderboard = (
    predictor.leaderboard(extra_info=True, silent=True)
    .sort_values("score_val", ascending=False)
    .reset_index(drop=True)
)

all_models = leaderboard["model"].astype(str).tolist()

if TARGET_BEST_MODEL not in all_models:
    print(f"Aranan best model bulunamadı: {TARGET_BEST_MODEL}")
    print("Mevcut modeller:")
    for model in all_models:
        print(f" - {model}")
    raise ValueError(
        f"{TARGET_BEST_MODEL} AutoGluon leaderboard içinde bulunamadı."
    )

best_model = TARGET_BEST_MODEL

print("=" * 70)
print("AUTOGLUON → W&B")
print(f"Best Model   : {best_model}")
print(f"Toplam Model : {len(all_models)}")
print(f"Project      : {PROJECT_NAME}")
print("=" * 70)

y_log = np.asarray(
    y_automl.to_numpy() if hasattr(y_automl, "to_numpy") else y_automl,
    dtype=np.float64
)

if not np.all(np.isfinite(y_log)):
    raise ValueError("y_log içinde NaN veya Inf var.")

y_usd = np.maximum(np.expm1(y_log), 0)

if not np.all(np.isfinite(y_usd)):
    raise ValueError("y_usd içinde NaN veya Inf var.")


def safe_value(value):
    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, (list, tuple, dict)):
        try:
            return json.dumps(value, default=str)
        except Exception:
            return str(value)

    if isinstance(value, float) and not np.isfinite(value):
        return None

    try:
        json.dumps(value)
        return value
    except Exception:
        return str(value)


def flatten_dict(d, prefix=""):
    result = {}

    if not isinstance(d, dict):
        return result

    for key, value in d.items():
        name = f"{prefix}_{key}" if prefix else str(key)

        if isinstance(value, dict):
            result.update(flatten_dict(value, name))
        else:
            value = safe_value(value)
            if value is not None:
                result[name] = value

    return result


def load_model_object(model_name):
    try:
        return predictor._trainer.load_model(model_name)
    except Exception as e:
        print(f"Model object alınamadı: {model_name} → {e}")
        return None


def get_all_model_parameters(model_name):
    params = {}
    model_obj = load_model_object(model_name)

    if model_obj is None:
        return params

    attributes = [
        "params",
        "params_aux",
        "hyperparameters",
        "hyperparameter",
        "_params",
        "_params_aux",
        "_hyperparameters",
        "ag_params",
        "model_params",
        "model_hyperparameters"
    ]

    for attr in attributes:
        try:
            value = getattr(model_obj, attr, None)

            if isinstance(value, dict):
                flat = flatten_dict(value, attr)

                for key, val in flat.items():
                    val = safe_value(val)

                    if val is not None:
                        params[key] = val

        except Exception:
            pass

    try:
        for key, value in vars(model_obj).items():
            if isinstance(value, dict):
                flat = flatten_dict(value, str(key))

                for p_key, p_value in flat.items():
                    p_value = safe_value(p_value)

                    if p_value is not None:
                        params[p_key] = p_value

    except Exception:
        pass

    return params


def get_candidate_objects(model_obj):
    candidates = [model_obj]

    attributes = [
        "model",
        "_model",
        "model_internal",
        "_model_internal",
        "model_base",
        "_model_base",
        "model_obj",
        "_model_obj",
        "booster",
        "_booster"
    ]

    for attr in attributes:
        try:
            obj = getattr(model_obj, attr, None)

            if obj is not None:
                candidates.append(obj)

        except Exception:
            pass

    return candidates


def get_xgb_params(model_name):
    params = {}
    model_obj = load_model_object(model_name)

    if model_obj is None:
        return params

    candidates = get_candidate_objects(model_obj)

    for obj in candidates:
        for attr in [
            "params",
            "params_aux",
            "hyperparameters",
            "_params"
        ]:
            try:
                value = getattr(obj, attr, None)

                if isinstance(value, dict):
                    flat = flatten_dict(value)

                    for key, val in flat.items():
                        val = safe_value(val)

                        if val is not None:
                            params[f"xgb_{key}"] = val

            except Exception:
                pass

    for obj in candidates:
        try:
            if hasattr(obj, "get_params"):
                xgb_params = obj.get_params()

                if isinstance(xgb_params, dict):
                    for key, value in xgb_params.items():
                        value = safe_value(value)

                        if value is not None:
                            params[f"xgb_{key}"] = value

        except Exception:
            pass

    for obj in candidates:
        try:
            if hasattr(obj, "get_booster"):
                booster = obj.get_booster()

                if booster is not None:
                    config_json = booster.save_config()
                    config_dict = json.loads(config_json)

                    flat_config = flatten_dict(
                        config_dict,
                        prefix="booster"
                    )

                    for key, value in flat_config.items():
                        value = safe_value(value)

                        if value is not None:
                            params[f"xgb_{key}"] = value

        except Exception:
            pass

    important_params = [
        "max_depth",
        "learning_rate",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "colsample_bylevel",
        "colsample_bynode",
        "gamma",
        "reg_alpha",
        "reg_lambda",
        "n_estimators",
        "early_stopping_rounds",
        "objective",
        "eval_metric",
        "tree_method",
        "booster",
        "random_state",
        "n_jobs",
        "max_bin",
        "grow_policy",
        "max_leaves"
    ]

    for parameter in important_params:
        for key, value in params.items():
            key_lower = str(key).lower()
            parameter_lower = parameter.lower()

            if (
                key_lower.endswith(parameter_lower)
                or key_lower.endswith("_" + parameter_lower)
            ):
                params[f"hp_{parameter}"] = value
                break

    return params


def get_chart_indices(n):
    if n <= MAX_CHART_POINTS:
        return np.arange(n)

    return np.linspace(
        0,
        n - 1,
        MAX_CHART_POINTS,
        dtype=int
    )


successful_models = []
failed_models = []

for idx, row in leaderboard.iterrows():

    model_name = str(row["model"])
    is_best = model_name == best_model

    print(f"\n[{idx + 1}/{len(leaderboard)}] {model_name}")

    if is_best:
        print("⭐ BEST MODEL")

    try:
        pred_log = np.asarray(
            predictor.predict_oof(model=model_name),
            dtype=np.float64
        )

    except Exception as e:
        print(f"❌ OOF alınamadı: {e}")
        failed_models.append((model_name, str(e)))
        continue

    if len(pred_log) != len(y_log):
        error = "OOF-target length mismatch"
        print(f"❌ {error}")
        failed_models.append((model_name, error))
        continue

    if not np.all(np.isfinite(pred_log)):
        error = "OOF predictions içinde NaN veya Inf bulundu."
        print(f"❌ {error}")
        failed_models.append((model_name, error))
        continue

    pred_usd = np.maximum(np.expm1(pred_log), 0)

    rmse_log = float(
        np.sqrt(
            mean_squared_error(
                y_log,
                pred_log
            )
        )
    )

    rmse_usd = float(
        np.sqrt(
            mean_squared_error(
                y_usd,
                pred_usd
            )
        )
    )

    mae_usd = float(
        mean_absolute_error(
            y_usd,
            pred_usd
        )
    )

    r2_usd = float(
        r2_score(
            y_usd,
            pred_usd
        )
    )

    residual_usd = y_usd - pred_usd
    absolute_error_usd = np.abs(residual_usd)

    try:
        hyperparameters = (
            predictor.get_model_hyperparameters(model_name)
            or {}
        )
    except Exception:
        hyperparameters = {}

    hyperparameters = flatten_dict(hyperparameters)

    all_model_params = get_all_model_parameters(model_name)

    xgb_params = {}

    if "XGBoost" in model_name:
        xgb_params = get_xgb_params(model_name)

    config = {
        "framework": "AutoGluon",
        "problem_type": "regression",
        "target": str(predictor.label),
        "eval_metric": str(predictor.eval_metric.name),
        "model": model_name,
        "model_family": model_name.split("_")[0],
        "rank": int(idx + 1),
        "total_models": int(len(all_models)),
        "is_best_model": bool(is_best),
        "best_model_name": best_model,
        "RMSE_Log": rmse_log,
        "RMSE_USD": rmse_usd,
        "MAE_USD": mae_usd,
        "R2_USD": r2_usd,
        "Fit_Time": (
            float(row["fit_time"])
            if pd.notna(row["fit_time"])
            else None
        ),
        "Pred_Time_Val": (
            float(row["pred_time_val"])
            if pd.notna(row["pred_time_val"])
            else None
        ),
        "OOF_Sample_Size": int(len(y_log)),
        "Stack_Level": (
            int(row["stack_level"])
            if (
                "stack_level" in row
                and pd.notna(row["stack_level"])
            )
            else None
        )
    }

    for key, value in hyperparameters.items():
        value = safe_value(value)

        if value is not None:
            config[f"hp_{key}"] = value

    for key, value in all_model_params.items():
        value = safe_value(value)

        if value is not None:
            config[f"model_{key}"] = value

    for key, value in xgb_params.items():
        value = safe_value(value)

        if value is not None:
            config[key] = value

    run = wandb.init(
        entity=ENTITY_NAME,
        project=PROJECT_NAME,
        name=f"AutoGluon_{model_name}",
        group="AutoGluon_Models",
        job_type="model_comparison",
        config=config
    )

    run.log({
        "RMSE_Log": rmse_log,
        "RMSE_USD": rmse_usd,
        "MAE_USD": mae_usd,
        "R2_USD": r2_usd,
        "Rank": int(idx + 1),
        "Best_Model_Flag": int(is_best)
    })

    oof_table = wandb.Table(
        columns=[
            "Index",
            "Actual_Log",
            "Predicted_Log",
            "Actual_USD",
            "Predicted_USD",
            "Residual_USD",
            "Absolute_Error_USD"
        ]
    )

    for i in range(len(y_log)):
        oof_table.add_data(
            int(i),
            float(y_log[i]),
            float(pred_log[i]),
            float(y_usd[i]),
            float(pred_usd[i]),
            float(residual_usd[i]),
            float(absolute_error_usd[i])
        )

    run.log({
        "OOF_Predictions": oof_table
    })

    chart_idx = get_chart_indices(len(y_usd))

    line_table = wandb.Table(
        columns=[
            "Index",
            "Actual_USD",
            "Predicted_USD"
        ]
    )

    for i in chart_idx:
        line_table.add_data(
            int(i),
            float(y_usd[i]),
            float(pred_usd[i])
        )

    line_chart = wandb.plot.line(
        line_table,
        "Index",
        ["Actual_USD", "Predicted_USD"],
        title=f"{model_name} - Actual vs Predicted"
    )

    run.log({
        "Charts/Line_Actual_vs_Predicted": line_chart
    })

    scatter_table = wandb.Table(
        columns=[
            "Actual_USD",
            "Predicted_USD"
        ]
    )

    for i in chart_idx:
        scatter_table.add_data(
            float(y_usd[i]),
            float(pred_usd[i])
        )

    scatter_chart = wandb.plot.scatter(
        scatter_table,
        "Actual_USD",
        "Predicted_USD",
        title=f"{model_name} - Actual vs Predicted"
    )

    run.log({
        "Charts/Scatter_Actual_vs_Predicted": scatter_chart
    })

    residual_table = wandb.Table(
        columns=[
            "Index",
            "Residual_USD"
        ]
    )

    for i in chart_idx:
        residual_table.add_data(
            int(i),
            float(residual_usd[i])
        )

    residual_chart = wandb.plot.line(
        residual_table,
        "Index",
        "Residual_USD",
        title=f"{model_name} - Residual Analysis"
    )

    run.log({
        "Charts/Residuals": residual_chart
    })

    rmse_bar_table = wandb.Table(
        columns=["Metric", "Value"]
    )

    rmse_bar_table.add_data(
        "RMSE_USD",
        rmse_usd
    )

    rmse_bar_chart = wandb.plot.bar(
        rmse_bar_table,
        "Metric",
        "Value",
        title=f"{model_name} - RMSE"
    )

    run.log({
        "Charts/Bar_RMSE": rmse_bar_chart
    })

    mae_bar_table = wandb.Table(
        columns=["Metric", "Value"]
    )

    mae_bar_table.add_data(
        "MAE_USD",
        mae_usd
    )

    mae_bar_chart = wandb.plot.bar(
        mae_bar_table,
        "Metric",
        "Value",
        title=f"{model_name} - MAE"
    )

    run.log({
        "Charts/Bar_MAE": mae_bar_chart
    })

    r2_bar_table = wandb.Table(
        columns=["Metric", "Value"]
    )

    r2_bar_table.add_data(
        "R2_USD",
        r2_usd
    )

    r2_bar_chart = wandb.plot.bar(
        r2_bar_table,
        "Metric",
        "Value",
        title=f"{model_name} - R²"
    )

    run.log({
        "Charts/Bar_R2": r2_bar_chart
    })

    run.summary.update({
        "Model": model_name,
        "Model_Family": model_name.split("_")[0],
        "Best_Model": best_model,
        "Is_Best_Model": bool(is_best),
        "RMSE_Log": rmse_log,
        "RMSE_USD": rmse_usd,
        "MAE_USD": mae_usd,
        "R2_USD": r2_usd,
        "Rank": int(idx + 1),
        "OOF_Sample_Size": int(len(y_log)),
        "Mean_Absolute_Error_USD": float(
            np.mean(absolute_error_usd)
        ),
        "Median_Absolute_Error_USD": float(
            np.median(absolute_error_usd)
        ),
        "Max_Absolute_Error_USD": float(
            np.max(absolute_error_usd)
        ),
        "Mean_Residual_USD": float(
            np.mean(residual_usd)
        ),
        "Std_Residual_USD": float(
            np.std(residual_usd)
        ),
        "Parameter_Count": int(len(config))
    })

    if is_best:
        run.summary.update({
            "BEST_MODEL_SELECTED": True,
            "BEST_MODEL_REASON": "Manually selected XGBoost_BAG_L2"
        })

    run.finish()

    successful_models.append(model_name)

    print(f"✓ RMSE Log : {rmse_log:.6f}")
    print(f"✓ RMSE USD : ${rmse_usd:,.2f}")
    print(f"✓ MAE USD  : ${mae_usd:,.2f}")
    print(f"✓ R²       : {r2_usd:.6f}")
    print(f"✓ Params   : {len(config)}")
    print("✓ OOF table")
    print("✓ Line chart")
    print("✓ Scatter chart")
    print("✓ Residual chart")
    print("✓ RMSE bar")
    print("✓ MAE bar")
    print("✓ R² bar")

print("\n" + "=" * 70)
print("AUTOGLUON → W&B TAMAMLANDI")
print("=" * 70)
print(f"Toplam Model : {len(all_models)}")
print(f"Başarılı     : {len(successful_models)}")
print(f"Başarısız    : {len(failed_models)}")
print(f"Best Model   : {best_model}")
print(f"Project      : {PROJECT_NAME}")
print("=" * 70)

if failed_models:
    print("\nBaşarısız modeller:")
    for model, error in failed_models:
        print(f"- {model}: {error}")

print("\nBaşarılı modeller:")

for model in successful_models:
    prefix = "⭐" if model == best_model else "✓"
    print(f"{prefix} {model}")

print("=" * 70)

## 13. Model Eğitimi

## TRIAL

In [ ]:
!pip install -q optuna

In [ ]:
# ============================================================
# XGBOOST + OPTUNA + STRATIFIED K-FOLD
# EARLY STOPPING + OVERFITTING + BALANCE ANALYSIS
# ============================================================

import os
import json
import pickle
import numpy as np
import pandas as pd
import polars as pl
import optuna
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

# ============================================================
# 1. AYARLAR
# ============================================================

RANDOM_STATE = 42
N_JOBS = -1
N_TRIALS = 20

MIN_K = 3
MAX_K = 10

MAX_ESTIMATORS = 5000

MIN_EARLY_STOPPING = 5
MAX_EARLY_STOPPING = 50

MAX_RELATIVE_GAP = 0.20

OVERFIT_MIN_DELTA = 1e-4
OVERFIT_PATIENCE = 10
OVERFIT_FOLD_RATIO = 0.20

TARGET_IS_LOG = True

STUDY_NAME = "XGBoost_StratifiedKFold_Optuna"
OPTUNA_DIR = "xgb_optuna_stratified_results"

os.makedirs(OPTUNA_DIR, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

ALL_HISTORY = []
ALL_FOLD_RESULTS = []
ALL_BALANCE_RESULTS = []

# ============================================================
# 2. VERİ
# ============================================================

X_cv = np.asarray(X_train_encoded)
y_cv = np.asarray(y_train, dtype=np.float64)

print("=" * 80)
print("XGBOOST + OPTUNA + STRATIFIED K-FOLD")
print("EARLY STOPPING + OVERFITTING CONTROL")
print("=" * 80)
print(f"X shape            : {X_cv.shape}")
print(f"y shape            : {y_cv.shape}")
print(f"K range            : {MIN_K}-{MAX_K}")
print(f"Trials             : {N_TRIALS}")
print(f"Max estimators     : {MAX_ESTIMATORS}")
print(f"Max relative gap   : {MAX_RELATIVE_GAP:.2%}")
print(f"Overfit patience   : {OVERFIT_PATIENCE}")
print(f"Overfit fold ratio : {OVERFIT_FOLD_RATIO:.2%}")

# ============================================================
# 3. STRATIFICATION
# ============================================================

def get_column(data, column):
    if isinstance(data, pl.DataFrame):
        return data[column].to_list()
    if isinstance(data, pd.DataFrame):
        return data[column].tolist()
    return np.asarray(data).tolist()

price_bin = pd.qcut(
    pd.Series(y_cv),
    q=10,
    labels=[f"q{i}" for i in range(1, 11)],
    duplicates="drop"
).astype(str)

supply_type = np.asarray(
    get_column(X_train, "tender_supplytype"),
    dtype=str
)

supply_type = np.where(
    pd.isna(supply_type) | (supply_type == "None"),
    "Unknown",
    supply_type
)

competition_level = np.asarray(
    get_column(X_train, "competition_level"),
    dtype=str
)

competition_level = np.where(
    pd.isna(competition_level) | (competition_level == "None"),
    "Unknown",
    competition_level
)

# 3 KOLON BİRLİKTE STRATIFICATION
stratify_label = np.array([
    f"{p}_{s}_{c}"
    for p, s, c in zip(
        price_bin,
        supply_type,
        competition_level
    )
])

label_counts = pd.Series(stratify_label).value_counts()

print(f"\nStratification combinations: {len(label_counts)}")
print(f"Minimum class count        : {label_counts.min()}")

if label_counts.min() < MAX_K:
    raise ValueError(
        f"3 kolonlu stratification için bazı sınıflarda "
        f"{label_counts.min()} örnek var. "
        f"MAX_K={MAX_K} kullanılamaz. "
        f"K değerini düşür veya çok küçük kombinasyonları birleştir."
    )

PRICE_BINS = sorted(pd.Series(price_bin).unique())
SUPPLY_TYPES = sorted(pd.Series(supply_type).unique())
COMPETITION_LEVELS = sorted(pd.Series(competition_level).unique())

print("\nPrice bins:", PRICE_BINS)
print("Supply types:", SUPPLY_TYPES)
print("Competition levels:", COMPETITION_LEVELS)

# ============================================================
# 4. OBJECTIVE
# ============================================================

def objective(trial):

    k = trial.suggest_int("k", MIN_K, MAX_K)

    early_stopping_rounds = trial.suggest_int(
        "early_stopping_rounds",
        MIN_EARLY_STOPPING,
        MAX_EARLY_STOPPING
    )

    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.15, log=True
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 2, 15
        ),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 5.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.1, 10.0, log=True
        ),
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS,
        "eval_metric": "rmse"
    }

    skf = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    fold_results = []

    # ========================================================
    # FOLD LOOP
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_cv, stratify_label), 1
    ):

        print(f"\nTrial {trial.number:02d} | Fold {fold}/{k}")

        train_price = np.asarray(price_bin.iloc[train_idx])
        val_price = np.asarray(price_bin.iloc[val_idx])

        train_supply = supply_type[train_idx]
        val_supply = supply_type[val_idx]

        train_competition = competition_level[train_idx]
        val_competition = competition_level[val_idx]

        balance_result = {
            "Trial": trial.number,
            "K": k,
            "Early_Stopping": early_stopping_rounds,
            "Fold": fold,
            "Train_Size": len(train_idx),
            "Validation_Size": len(val_idx),
            "Train_Validation_Size_Ratio":
                len(train_idx) / len(val_idx)
        }

        # ====================================================
        # PRICE DISTRIBUTION
        # ====================================================

        for category in PRICE_BINS:

            train_ratio = np.mean(train_price == category)
            val_ratio = np.mean(val_price == category)

            balance_result[f"Train_Price_{category}_Ratio"] = train_ratio
            balance_result[f"Validation_Price_{category}_Ratio"] = val_ratio
            balance_result[f"Price_{category}_Abs_Diff"] = abs(
                train_ratio - val_ratio
            )

        # ====================================================
        # SUPPLY DISTRIBUTION
        # ====================================================

        for category in SUPPLY_TYPES:

            train_ratio = np.mean(train_supply == category)
            val_ratio = np.mean(val_supply == category)

            balance_result[f"Train_Supply_{category}_Ratio"] = train_ratio
            balance_result[f"Validation_Supply_{category}_Ratio"] = val_ratio
            balance_result[f"Supply_{category}_Abs_Diff"] = abs(
                train_ratio - val_ratio
            )

        # ====================================================
        # COMPETITION DISTRIBUTION
        # ====================================================

        for category in COMPETITION_LEVELS:

            train_ratio = np.mean(train_competition == category)
            val_ratio = np.mean(val_competition == category)

            balance_result[
                f"Train_Competition_{category}_Ratio"
            ] = train_ratio

            balance_result[
                f"Validation_Competition_{category}_Ratio"
            ] = val_ratio

            balance_result[
                f"Competition_{category}_Abs_Diff"
            ] = abs(train_ratio - val_ratio)

        # ====================================================
        # MAX DISTRIBUTION DIFFERENCES
        # ====================================================

        price_diffs = [
            balance_result[f"Price_{c}_Abs_Diff"]
            for c in PRICE_BINS
        ]

        supply_diffs = [
            balance_result[f"Supply_{c}_Abs_Diff"]
            for c in SUPPLY_TYPES
        ]

        competition_diffs = [
            balance_result[f"Competition_{c}_Abs_Diff"]
            for c in COMPETITION_LEVELS
        ]

        balance_result["Max_Price_Distribution_Diff"] = max(
            price_diffs, default=0
        )

        balance_result["Max_Supply_Distribution_Diff"] = max(
            supply_diffs, default=0
        )

        balance_result["Max_Competition_Distribution_Diff"] = max(
            competition_diffs, default=0
        )

        balance_result["Max_Overall_Distribution_Diff"] = max(
            balance_result["Max_Price_Distribution_Diff"],
            balance_result["Max_Supply_Distribution_Diff"],
            balance_result["Max_Competition_Distribution_Diff"]
        )

        balance_result.update({
            "max_depth": params["max_depth"],
            "learning_rate": params["learning_rate"],
            "min_child_weight": params["min_child_weight"],
            "subsample": params["subsample"],
            "colsample_bytree": params["colsample_bytree"],
            "gamma": params["gamma"],
            "reg_alpha": params["reg_alpha"],
            "reg_lambda": params["reg_lambda"]
        })

        ALL_BALANCE_RESULTS.append(balance_result)

        # ====================================================
        # MODEL
        # ====================================================

        model = xgb.XGBRegressor(
            n_estimators=MAX_ESTIMATORS,
            early_stopping_rounds=early_stopping_rounds,
            **params
        )

        model.fit(
            X_cv[train_idx],
            y_cv[train_idx],
            eval_set=[
                (X_cv[train_idx], y_cv[train_idx]),
                (X_cv[val_idx], y_cv[val_idx])
            ],
            verbose=False
        )

        # ====================================================
        # TRAINING HISTORY
        # ====================================================

        history = model.evals_result()

        train_history = np.asarray(
            history["validation_0"]["rmse"]
        )

        val_history = np.asarray(
            history["validation_1"]["rmse"]
        )

        best_idx = int(np.argmin(val_history))

        for iteration in range(len(train_history)):

            train_rmse = float(train_history[iteration])
            val_rmse = float(val_history[iteration])

            absolute_gap = val_rmse - train_rmse

            relative_gap = absolute_gap / max(
                abs(val_rmse), 1e-12
            )

            ALL_HISTORY.append({
                "Trial": trial.number,
                "K": k,
                "Early_Stopping": early_stopping_rounds,
                "max_depth": params["max_depth"],
                "learning_rate": params["learning_rate"],
                "min_child_weight": params["min_child_weight"],
                "subsample": params["subsample"],
                "colsample_bytree": params["colsample_bytree"],
                "gamma": params["gamma"],
                "reg_alpha": params["reg_alpha"],
                "reg_lambda": params["reg_lambda"],
                "Fold": fold,
                "Iteration": iteration + 1,
                "Train_RMSE": train_rmse,
                "Validation_RMSE": val_rmse,
                "Absolute_Train_Val_Gap": absolute_gap,
                "Relative_Train_Val_Gap": relative_gap,
                "Is_Best_Iteration": iteration == best_idx
            })

        # ====================================================
        # BEST ITERATION
        # ====================================================

        best_tree = best_idx + 1
        best_val_rmse = float(val_history[best_idx])
        best_train_rmse = float(train_history[best_idx])

        absolute_gap = best_val_rmse - best_train_rmse
        relative_gap = absolute_gap / max(
            abs(best_val_rmse), 1e-12
        )

        gap_overfit = relative_gap > MAX_RELATIVE_GAP

        # ====================================================
        # VALIDATION TREND OVERFITTING
        # ====================================================

        consecutive_bad_rounds = 0
        max_consecutive_bad_rounds = 0
        validation_trend_overfit = False

        for i in range(best_idx + 1, len(val_history)):

            val_change = val_history[i] - val_history[i - 1]
            train_change = train_history[i] - train_history[i - 1]

            validation_worse = val_change > OVERFIT_MIN_DELTA
            train_improving = train_change < -OVERFIT_MIN_DELTA

            if validation_worse and train_improving:
                consecutive_bad_rounds += 1
            else:
                consecutive_bad_rounds = 0

            max_consecutive_bad_rounds = max(
                max_consecutive_bad_rounds,
                consecutive_bad_rounds
            )

            if consecutive_bad_rounds >= OVERFIT_PATIENCE:
                validation_trend_overfit = True
                break

        actual_best_iteration = int(
            model.best_iteration + 1
            if model.best_iteration is not None
            else best_tree
        )

        early_stopped = actual_best_iteration < MAX_ESTIMATORS

        fold_overfit = (
            gap_overfit or validation_trend_overfit
        )

        # ====================================================
        # VALIDATION PREDICTION
        # ====================================================

        pred = model.predict(
            X_cv[val_idx],
            iteration_range=(0, actual_best_iteration)
        )

        if TARGET_IS_LOG:

            y_true_usd = np.expm1(y_cv[val_idx])
            y_pred_usd = np.maximum(np.expm1(pred), 0)

        else:

            y_true_usd = y_cv[val_idx]
            y_pred_usd = pred

        usd_rmse = np.sqrt(
            mean_squared_error(
                y_true_usd,
                y_pred_usd
            )
        )

        # ====================================================
        # FOLD RESULT
        # ====================================================

        fold_result = {
            "Trial": trial.number,
            "K": k,
            "Early_Stopping": early_stopping_rounds,
            "max_depth": params["max_depth"],
            "learning_rate": params["learning_rate"],
            "min_child_weight": params["min_child_weight"],
            "subsample": params["subsample"],
            "colsample_bytree": params["colsample_bytree"],
            "gamma": params["gamma"],
            "reg_alpha": params["reg_alpha"],
            "reg_lambda": params["reg_lambda"],
            "Fold": fold,
            "Train_Size": len(train_idx),
            "Validation_Size": len(val_idx),
            "Best_Tree": best_tree,
            "Actual_Best_Iteration": actual_best_iteration,
            "Train_RMSE": best_train_rmse,
            "Validation_RMSE": best_val_rmse,
            "Log_RMSE": best_val_rmse,
            "USD_RMSE": usd_rmse,
            "Absolute_Train_Val_Gap": absolute_gap,
            "Relative_Train_Val_Gap": relative_gap,
            "Gap_Overfit": gap_overfit,
            "Validation_Trend_Overfit": validation_trend_overfit,
            "Max_Consecutive_Bad_Rounds":
                max_consecutive_bad_rounds,
            "Early_Stopped": early_stopped,
            "Overfit": fold_overfit
        }

        fold_results.append(fold_result)
        ALL_FOLD_RESULTS.append(fold_result.copy())

        print(
            f"  Tree={actual_best_iteration} | "
            f"Train={best_train_rmse:.6f} | "
            f"Val={best_val_rmse:.6f} | "
            f"Gap={relative_gap:.2%} | "
            f"USD=${usd_rmse:,.2f} | "
            f"Balance="
            f"{balance_result['Max_Overall_Distribution_Diff']:.2%} | "
            f"Overfit={fold_overfit}"
        )

    # ========================================================
    # TRIAL METRİKLERİ
    # ========================================================

    fold_df = pd.DataFrame(fold_results)

    mean_log_rmse = fold_df["Log_RMSE"].mean()
    std_log_rmse = fold_df["Log_RMSE"].std()

    mean_usd_rmse = fold_df["USD_RMSE"].mean()
    std_usd_rmse = fold_df["USD_RMSE"].std()

    mean_train_rmse = fold_df["Train_RMSE"].mean()
    mean_val_rmse = fold_df["Validation_RMSE"].mean()

    mean_absolute_gap = fold_df[
        "Absolute_Train_Val_Gap"
    ].mean()

    mean_relative_gap = fold_df[
        "Relative_Train_Val_Gap"
    ].mean()

    max_relative_gap = fold_df[
        "Relative_Train_Val_Gap"
    ].max()

    mean_best_tree = fold_df["Best_Tree"].mean()
    median_best_tree = fold_df["Best_Tree"].median()
    min_best_tree = fold_df["Best_Tree"].min()
    max_best_tree = fold_df["Best_Tree"].max()

    # ========================================================
    # OVERFITTING
    # ========================================================

    overfit_fold_count = int(fold_df["Overfit"].sum())
    overfit_ratio = overfit_fold_count / k

    gap_overfit_count = int(
        fold_df["Gap_Overfit"].sum()
    )

    trend_overfit_count = int(
        fold_df["Validation_Trend_Overfit"].sum()
    )

    trial_is_overfit = overfit_ratio >= OVERFIT_FOLD_RATIO

    # ========================================================
    # BALANCE METRICS
    # ========================================================

    trial_balance_df = pd.DataFrame([
        x for x in ALL_BALANCE_RESULTS
        if x["Trial"] == trial.number
    ])

    mean_balance_diff = trial_balance_df[
        "Max_Overall_Distribution_Diff"
    ].mean()

    max_balance_diff = trial_balance_df[
        "Max_Overall_Distribution_Diff"
    ].max()

    mean_price_balance = trial_balance_df[
        "Max_Price_Distribution_Diff"
    ].mean()

    mean_supply_balance = trial_balance_df[
        "Max_Supply_Distribution_Diff"
    ].mean()

    mean_competition_balance = trial_balance_df[
        "Max_Competition_Distribution_Diff"
    ].mean()

    # ========================================================
    # OBJECTIVE
    # ========================================================

    objective_value = mean_usd_rmse

    if trial_is_overfit:
        objective_value *= 10.0

    # ========================================================
    # OPTUNA ATTRIBUTES
    # ========================================================

    metrics = {
        "mean_log_rmse": mean_log_rmse,
        "std_log_rmse": std_log_rmse,
        "mean_usd_rmse": mean_usd_rmse,
        "std_usd_rmse": std_usd_rmse,
        "mean_train_rmse": mean_train_rmse,
        "mean_val_rmse": mean_val_rmse,
        "mean_absolute_train_val_gap": mean_absolute_gap,
        "mean_relative_train_val_gap": mean_relative_gap,
        "max_relative_train_val_gap": max_relative_gap,
        "mean_best_tree": mean_best_tree,
        "median_best_tree": median_best_tree,
        "min_best_tree": min_best_tree,
        "max_best_tree": max_best_tree,
        "overfit_fold_count": overfit_fold_count,
        "overfit_fold_ratio": overfit_ratio,
        "gap_overfit_fold_count": gap_overfit_count,
        "validation_trend_overfit_fold_count":
            trend_overfit_count,
        "mean_balance_distribution_diff": mean_balance_diff,
        "max_balance_distribution_diff": max_balance_diff,
        "mean_price_balance_diff": mean_price_balance,
        "mean_supply_balance_diff": mean_supply_balance,
        "mean_competition_balance_diff":
            mean_competition_balance,
        "trial_is_overfit": trial_is_overfit,
        "objective_value": objective_value,
        "fold_best_trees": fold_df["Best_Tree"].tolist(),
        "fold_train_rmse": fold_df["Train_RMSE"].tolist(),
        "fold_val_rmse": fold_df["Validation_RMSE"].tolist(),
        "fold_usd_rmse": fold_df["USD_RMSE"].tolist(),
        "fold_absolute_gap":
            fold_df["Absolute_Train_Val_Gap"].tolist(),
        "fold_relative_gap":
            fold_df["Relative_Train_Val_Gap"].tolist(),
        "fold_gap_overfit":
            fold_df["Gap_Overfit"].tolist(),
        "fold_validation_trend_overfit":
            fold_df["Validation_Trend_Overfit"].tolist(),
        "fold_overfit":
            fold_df["Overfit"].tolist()
    }

    for key, value in metrics.items():
        if isinstance(value, np.generic):
            value = value.item()
        trial.set_user_attr(key, value)

    print(
        f"\nTrial {trial.number:02d} | "
        f"K={k} | "
        f"ES={early_stopping_rounds} | "
        f"Trees={mean_best_tree:.1f} | "
        f"Train={mean_train_rmse:.6f} | "
        f"Val={mean_val_rmse:.6f} | "
        f"Gap={mean_relative_gap:.2%} | "
        f"Balance={max_balance_diff:.2%} | "
        f"USD=${mean_usd_rmse:,.2f} | "
        f"Overfit={trial_is_overfit}"
    )

    return objective_value

# ============================================================
# 5. OPTUNA
# ============================================================

study = optuna.create_study(
    direction="minimize",
    study_name=STUDY_NAME
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

# ============================================================
# 6. TRIAL SONUÇLARI
# ============================================================

results_df = study.trials_dataframe(
    attrs=("number", "value", "params", "user_attrs", "state")
)

results_df = (
    results_df
    .sort_values("value")
    .reset_index(drop=True)
)

# ============================================================
# 7. NON-OVERFIT TRIAL'LAR
# ============================================================

non_overfit_df = results_df[
    results_df["user_attrs_trial_is_overfit"] == False
].copy()

non_overfit_df = (
    non_overfit_df
    .sort_values("user_attrs_mean_usd_rmse")
    .reset_index(drop=True)
)

if len(non_overfit_df) == 0:
    raise RuntimeError("Non-overfit trial bulunamadı.")

# ============================================================
# 8. BEST TRIAL
# ============================================================

BEST_TRIAL = int(non_overfit_df.iloc[0]["number"])
best_trial = study.trials[BEST_TRIAL]

BEST_PARAMS = best_trial.params.copy()
BEST_K = BEST_PARAMS["k"]
BEST_EARLY_STOPPING = BEST_PARAMS["early_stopping_rounds"]

BEST_MEAN_TREE = best_trial.user_attrs["mean_best_tree"]
BEST_MEDIAN_TREE = best_trial.user_attrs["median_best_tree"]

BEST_LOG_RMSE = best_trial.user_attrs["mean_log_rmse"]
BEST_LOG_STD = best_trial.user_attrs["std_log_rmse"]

BEST_USD_RMSE = best_trial.user_attrs["mean_usd_rmse"]
BEST_USD_STD = best_trial.user_attrs["std_usd_rmse"]

BEST_TRAIN_RMSE = best_trial.user_attrs["mean_train_rmse"]
BEST_VAL_RMSE = best_trial.user_attrs["mean_val_rmse"]

BEST_ABSOLUTE_GAP = best_trial.user_attrs[
    "mean_absolute_train_val_gap"
]

BEST_RELATIVE_GAP = best_trial.user_attrs[
    "mean_relative_train_val_gap"
]

BEST_MAX_RELATIVE_GAP = best_trial.user_attrs[
    "max_relative_train_val_gap"
]

BEST_OVERFIT_FOLDS = best_trial.user_attrs[
    "overfit_fold_count"
]

BEST_OVERFIT_RATIO = best_trial.user_attrs[
    "overfit_fold_ratio"
]

BEST_GAP_OVERFIT_FOLDS = best_trial.user_attrs[
    "gap_overfit_fold_count"
]

BEST_TREND_OVERFIT_FOLDS = best_trial.user_attrs[
    "validation_trend_overfit_fold_count"
]

BEST_MEAN_BALANCE_DIFF = best_trial.user_attrs[
    "mean_balance_distribution_diff"
]

BEST_MAX_BALANCE_DIFF = best_trial.user_attrs[
    "max_balance_distribution_diff"
]

BEST_IS_OVERFIT = best_trial.user_attrs[
    "trial_is_overfit"
]

# ============================================================
# 9. DATAFRAME'LER
# ============================================================

trial_fold_df = pd.DataFrame(ALL_FOLD_RESULTS)
history_df = pd.DataFrame(ALL_HISTORY)
balance_df = pd.DataFrame(ALL_BALANCE_RESULTS)

# ============================================================
# 10. K KARŞILAŞTIRMASI
# ============================================================

k_results = (
    results_df
    .groupby("params_k", as_index=False)
    .agg(
        Mean_USD_RMSE=(
            "user_attrs_mean_usd_rmse", "mean"
        ),
        Mean_Log_RMSE=(
            "user_attrs_mean_log_rmse", "mean"
        ),
        Mean_Relative_Gap=(
            "user_attrs_mean_relative_train_val_gap",
            "mean"
        ),
        Mean_Overfit_Ratio=(
            "user_attrs_overfit_fold_ratio", "mean"
        ),
        Mean_Balance_Diff=(
            "user_attrs_mean_balance_distribution_diff",
            "mean"
        ),
        Max_Balance_Diff=(
            "user_attrs_max_balance_distribution_diff",
            "max"
        ),
        Trial_Count=("number", "count")
    )
    .sort_values("Mean_USD_RMSE")
)

# ============================================================
# 11. EARLY STOPPING KARŞILAŞTIRMASI
# ============================================================

early_results = (
    results_df
    .groupby(
        "params_early_stopping_rounds",
        as_index=False
    )
    .agg(
        Mean_USD_RMSE=(
            "user_attrs_mean_usd_rmse", "mean"
        ),
        Mean_Log_RMSE=(
            "user_attrs_mean_log_rmse", "mean"
        ),
        Mean_Relative_Gap=(
            "user_attrs_mean_relative_train_val_gap",
            "mean"
        ),
        Mean_Overfit_Ratio=(
            "user_attrs_overfit_fold_ratio", "mean"
        ),
        Mean_Balance_Diff=(
            "user_attrs_mean_balance_distribution_diff",
            "mean"
        ),
        Trial_Count=("number", "count")
    )
    .sort_values("Mean_USD_RMSE")
)

# ============================================================
# 12. PARAMETRE + METRİK TABLOSU
# ============================================================

parameter_results = results_df[
    [
        "number",
        "params_k",
        "params_early_stopping_rounds",
        "params_max_depth",
        "params_learning_rate",
        "params_min_child_weight",
        "params_subsample",
        "params_colsample_bytree",
        "params_gamma",
        "params_reg_alpha",
        "params_reg_lambda",
        "user_attrs_mean_train_rmse",
        "user_attrs_mean_val_rmse",
        "user_attrs_mean_log_rmse",
        "user_attrs_std_log_rmse",
        "user_attrs_mean_usd_rmse",
        "user_attrs_std_usd_rmse",
        "user_attrs_mean_best_tree",
        "user_attrs_mean_relative_train_val_gap",
        "user_attrs_max_relative_train_val_gap",
        "user_attrs_overfit_fold_ratio",
        "user_attrs_mean_balance_distribution_diff",
        "user_attrs_max_balance_distribution_diff",
        "user_attrs_mean_price_balance_diff",
        "user_attrs_mean_supply_balance_diff",
        "user_attrs_mean_competition_balance_diff",
        "user_attrs_trial_is_overfit",
        "value"
    ]
].copy()

parameter_results = parameter_results.rename(
    columns={
        "number": "Trial",
        "params_k": "K",
        "params_early_stopping_rounds":
            "Early_Stopping",
        "params_max_depth": "max_depth",
        "params_learning_rate": "learning_rate",
        "params_min_child_weight":
            "min_child_weight",
        "params_subsample": "subsample",
        "params_colsample_bytree":
            "colsample_bytree",
        "params_gamma": "gamma",
        "params_reg_alpha": "reg_alpha",
        "params_reg_lambda": "reg_lambda",
        "user_attrs_mean_train_rmse":
            "Mean_Train_RMSE",
        "user_attrs_mean_val_rmse":
            "Mean_Validation_RMSE",
        "user_attrs_mean_log_rmse":
            "Mean_Log_RMSE",
        "user_attrs_std_log_rmse":
            "Std_Log_RMSE",
        "user_attrs_mean_usd_rmse":
            "Mean_USD_RMSE",
        "user_attrs_std_usd_rmse":
            "Std_USD_RMSE",
        "user_attrs_mean_best_tree":
            "Mean_Best_Tree",
        "user_attrs_mean_relative_train_val_gap":
            "Mean_Relative_Gap",
        "user_attrs_max_relative_train_val_gap":
            "Max_Relative_Gap",
        "user_attrs_overfit_fold_ratio":
            "Overfit_Fold_Ratio",
        "user_attrs_mean_balance_distribution_diff":
            "Mean_Balance_Diff",
        "user_attrs_max_balance_distribution_diff":
            "Max_Balance_Diff",
        "user_attrs_mean_price_balance_diff":
            "Mean_Price_Balance_Diff",
        "user_attrs_mean_supply_balance_diff":
            "Mean_Supply_Balance_Diff",
        "user_attrs_mean_competition_balance_diff":
            "Mean_Competition_Balance_Diff",
        "user_attrs_trial_is_overfit":
            "Trial_Overfit",
        "value": "Objective_Value"
    }
)

# ============================================================
# 13. DOSYA YOLLARI
# ============================================================

results_path = os.path.join(
    OPTUNA_DIR, "optuna_trials.csv"
)

parameter_path = os.path.join(
    OPTUNA_DIR, "optuna_parameter_results.csv"
)

fold_path = os.path.join(
    OPTUNA_DIR, "optuna_trial_fold_results.csv"
)

history_path = os.path.join(
    OPTUNA_DIR, "optuna_boosting_history.csv"
)

balance_path = os.path.join(
    OPTUNA_DIR, "optuna_fold_balance_results.csv"
)

k_path = os.path.join(
    OPTUNA_DIR, "k_comparison.csv"
)

early_path = os.path.join(
    OPTUNA_DIR, "early_stopping_comparison.csv"
)

non_overfit_path = os.path.join(
    OPTUNA_DIR, "non_overfit_trials.csv"
)

best_path = os.path.join(
    OPTUNA_DIR, "best_params.json"
)

study_path = os.path.join(
    OPTUNA_DIR, "xgboost_optuna_study.pkl"
)

# ============================================================
# 14. CSV EXPORT
# ============================================================

results_df.to_csv(results_path, index=False)
parameter_results.to_csv(parameter_path, index=False)
trial_fold_df.to_csv(fold_path, index=False)
history_df.to_csv(history_path, index=False)
balance_df.to_csv(balance_path, index=False)
k_results.to_csv(k_path, index=False)
early_results.to_csv(early_path, index=False)
non_overfit_df.to_csv(non_overfit_path, index=False)

# ============================================================
# 15. BEST PARAMS JSON
# ============================================================

best_export = {
    "best_trial": BEST_TRIAL,
    "best_k": BEST_K,
    "best_early_stopping": BEST_EARLY_STOPPING,
    "mean_best_tree": BEST_MEAN_TREE,
    "median_best_tree": BEST_MEDIAN_TREE,
    "mean_train_rmse": BEST_TRAIN_RMSE,
    "mean_validation_rmse": BEST_VAL_RMSE,
    "mean_absolute_train_val_gap": BEST_ABSOLUTE_GAP,
    "mean_relative_train_val_gap": BEST_RELATIVE_GAP,
    "max_relative_train_val_gap": BEST_MAX_RELATIVE_GAP,
    "mean_log_rmse": BEST_LOG_RMSE,
    "std_log_rmse": BEST_LOG_STD,
    "mean_usd_rmse": BEST_USD_RMSE,
    "std_usd_rmse": BEST_USD_STD,
    "mean_balance_distribution_diff":
        BEST_MEAN_BALANCE_DIFF,
    "max_balance_distribution_diff":
        BEST_MAX_BALANCE_DIFF,
    "gap_overfit_fold_count":
        BEST_GAP_OVERFIT_FOLDS,
    "validation_trend_overfit_fold_count":
        BEST_TREND_OVERFIT_FOLDS,
    "overfit_fold_count": BEST_OVERFIT_FOLDS,
    "overfit_fold_ratio": BEST_OVERFIT_RATIO,
    "trial_is_overfit": BEST_IS_OVERFIT,
    "max_relative_gap": MAX_RELATIVE_GAP,
    "overfit_min_delta": OVERFIT_MIN_DELTA,
    "overfit_patience": OVERFIT_PATIENCE,
    "overfit_fold_ratio_threshold":
        OVERFIT_FOLD_RATIO,
    "best_params": BEST_PARAMS
}

with open(best_path, "w", encoding="utf-8") as f:
    json.dump(
        best_export,
        f,
        indent=4,
        ensure_ascii=False
    )

# ============================================================
# 16. OPTUNA STUDY
# ============================================================

with open(study_path, "wb") as f:
    pickle.dump(study, f)

# ============================================================
# 17. GLOBAL DEĞİŞKENLER
# ============================================================

N_ESTIMATORS = MAX_ESTIMATORS
EARLY_STOPPING_ROUNDS = BEST_EARLY_STOPPING

# ============================================================
# 18. ÖZET
# ============================================================

print("\n" + "=" * 80)
print("EN İYİ NON-OVERFIT XGBOOST")
print("=" * 80)

print(f"Best Trial              : {BEST_TRIAL}")
print(f"Best K                  : {BEST_K}")
print(f"Best Early Stopping     : {BEST_EARLY_STOPPING}")
print(f"Mean Best Tree          : {BEST_MEAN_TREE:.1f}")
print(f"Mean Log RMSE           : {BEST_LOG_RMSE:.6f}")
print(f"Mean USD RMSE           : ${BEST_USD_RMSE:,.2f}")
print(f"Mean Relative Gap       : {BEST_RELATIVE_GAP:.2%}")
print(f"Max Relative Gap        : {BEST_MAX_RELATIVE_GAP:.2%}")
print(
    f"Mean Balance Difference : "
    f"{BEST_MEAN_BALANCE_DIFF:.4%}"
)
print(
    f"Max Balance Difference  : "
    f"{BEST_MAX_BALANCE_DIFF:.4%}"
)
print(
    f"Overfit Folds           : "
    f"{BEST_OVERFIT_FOLDS}/{BEST_K}"
)
print(f"Overfit Ratio           : {BEST_OVERFIT_RATIO:.2%}")
print(f"Trial Overfit           : {BEST_IS_OVERFIT}")

print("\nBest Parameters:")
for key, value in BEST_PARAMS.items():
    print(f"{key:25}: {value}")

# ============================================================
# 19. KAYDEDİLEN DOSYALAR
# ============================================================

print("\n" + "=" * 80)
print("KAYDEDİLEN DOSYALAR")
print("=" * 80)

print(f"✓ Trial sonuçları       : {results_path}")
print(f"✓ Parameter sonuçları   : {parameter_path}")
print(f"✓ Fold sonuçları        : {fold_path}")
print(f"✓ Boosting history      : {history_path}")
print(f"✓ Fold balance          : {balance_path}")
print(f"✓ K karşılaştırması     : {k_path}")
print(f"✓ Early stopping        : {early_path}")
print(f"✓ Non-overfit trials    : {non_overfit_path}")
print(f"✓ Best parameters       : {best_path}")
print(f"✓ Optuna study          : {study_path}")

# ============================================================
# 20. GÖRÜNTÜLE
# ============================================================

print("\n" + "=" * 80)
print("TRIAL PARAMETER + METRİKLER")
print("=" * 80)
display(parameter_results)

print("\n" + "=" * 80)
print("TRIAL × FOLD")
print("=" * 80)
display(trial_fold_df)

print("\n" + "=" * 80)
print("FOLD BALANCE")
print("=" * 80)
display(balance_df.head(20))

print("\n" + "=" * 80)
print("BOOSTING HISTORY")
print("=" * 80)
display(history_df.head(20))

print("\n" + "=" * 80)
print("K KARŞILAŞTIRMASI")
print("=" * 80)
display(k_results)

print("\n" + "=" * 80)
print("EARLY STOPPING KARŞILAŞTIRMASI")
print("=" * 80)
display(early_results)

Kodda overfitting, train ve validation RMSE arasındaki fark ve eğitim sürecindeki RMSE trendi üzerinden tespit ediliyor. Her fold için validation RMSE'nin en düşük olduğu best iteration bulunuyor ve bu noktadaki train-validation farkı hesaplanıyor. Bu farkın validation RMSE'ye oranı %20 (MAX_RELATIVE_GAP) değerini aşarsa fold overfit kabul ediliyor. Ayrıca best iteration sonrasında train RMSE düşmeye devam ederken validation RMSE artıyorsa, bu durumun 10 (OVERFIT_PATIENCE) ardışık iteration sürmesi halinde ikinci bir overfitting sinyali oluşturuluyor.

Her fold ayrı değerlendirildikten sonra, fold'ların en az %20'si (OVERFIT_FOLD_RATIO) overfit ise tüm trial overfit kabul ediliyor. Overfit olmayan trial'lar tercih edilsin diye bu trial'ın mean_usd_rmse değeri 10 kat cezalandırılıyor. Böylece Optuna yalnızca düşük RMSE'ye değil, aynı zamanda daha az overfitting gösteren ve daha dengeli modelleri seçmeye yönlendiriliyor. Early Stopping ise validation performansı belirlenen süre boyunca iyileşmediğinde eğitimi durdurarak gereksiz boosting iteration'larının önüne geçiyor.

**XGBoost Optuna Sonuçları ve Overfitting Analizi**

Bu deneyde 20 Optuna trialı, K=3–10 Stratified K-Fold ve overfitting kontrolü ile karşılaştırılmıştır. En iyi non-overfit model Trial 03 olmuştur. Model K=9, early stopping=9 ve ortalama 1008 ağaç kullanmıştır.

Modelin Mean Log RMSE değeri 1.0835, USD RMSE değeri yaklaşık 17.22 milyon $'dır. Train–validation arasındaki ortalama fark %16.24, maksimum fark ise %18.38'dir ve belirlenen %20 sınırının altında kalmıştır. Ayrıca 9 fold'un hiçbirinde overfitting tespit edilmemiştir (0/9).

Bazı trial'lar daha düşük validation RMSE üretse de %20'nin üzerinde gap oluşturarak ciddi overfitting göstermiştir. Bu nedenle yalnızca en düşük validation hatasına değil, genelleme ve overfitting kontrolüne göre seçim yapılmıştır.

Sonuç: Trial 03, performans ile overfitting kontrolü arasında en iyi dengeyi sağlayan modeldir.

## W & B

In [ ]:
# OPTUNA 20 TRIAL → W&B
# HER TRIAL = AYRI W&B RUN
#
# AKTARILANLAR:
# - TÜM TRIAL PARAMETRELERİ
# - TÜM TRIAL METRİKLERİ
# - OBJECTIVE VALUE
# - TRAIN / VALIDATION RMSE
# - LOG RMSE
# - USD RMSE
# - STD METRİKLER
# - BEST TREE
# - TRAIN/VALIDATION GAP
# - OVERFITTING
# - BALANCE ANALİZİ
# - FOLD SONUÇLARI
# - BOOSTING HISTORY


import os
import json
import numpy as np
import pandas as pd
import wandb

# 1. W&B AYARLARI

PROJECT_NAME = "GPPD_XGBoost_Optuna"
ENTITY_NAME = "iremerkurt34-personal"

OPTUNA_DIR = "xgb_optuna_stratified_results"

wandb.login()

# 2. DOSYALAR

TRIALS_PATH = os.path.join(
    OPTUNA_DIR,
    "optuna_trials.csv"
)

PARAMETER_PATH = os.path.join(
    OPTUNA_DIR,
    "optuna_parameter_results.csv"
)

FOLD_PATH = os.path.join(
    OPTUNA_DIR,
    "optuna_trial_fold_results.csv"
)

HISTORY_PATH = os.path.join(
    OPTUNA_DIR,
    "optuna_boosting_history.csv"
)

BALANCE_PATH = os.path.join(
    OPTUNA_DIR,
    "optuna_fold_balance_results.csv"
)

BEST_PATH = os.path.join(
    OPTUNA_DIR,
    "best_params.json"
)

# 3. DOSYALARI OKU

trials_df = pd.read_csv(TRIALS_PATH)
parameter_df = pd.read_csv(PARAMETER_PATH)
fold_df = pd.read_csv(FOLD_PATH)
history_df = pd.read_csv(HISTORY_PATH)
balance_df = pd.read_csv(BALANCE_PATH)

with open(BEST_PATH, "r", encoding="utf-8") as f:
    best_info = json.load(f)

BEST_TRIAL = int(best_info["best_trial"])

print("=" * 80)
print("OPTUNA → W&B")
print("=" * 80)

print(f"Trials        : {len(parameter_df)}")
print(f"Fold records  : {len(fold_df)}")
print(f"History rows  : {len(history_df)}")
print(f"Balance rows  : {len(balance_df)}")
print(f"Best Trial    : {BEST_TRIAL}")

# 4. KONTROL

if len(parameter_df) != 20:
    print(
        f"UYARI: Beklenen 20 trial yerine "
        f"{len(parameter_df)} trial bulundu."
    )

# 5. TRIAL'LARI AKTAR

for _, trial_row in parameter_df.iterrows():

    trial_number = int(trial_row["Trial"])

    print("\n" + "=" * 80)
    print(f"TRIAL {trial_number}")
    print("=" * 80)

    # --------------------------------------------------------
    # FOLD VERİLERİ
    # --------------------------------------------------------

    trial_folds = fold_df[
        fold_df["Trial"] == trial_number
    ].copy()

    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    trial_history = history_df[
        history_df["Trial"] == trial_number
    ].copy()

    # --------------------------------------------------------
    # BALANCE
    # --------------------------------------------------------

    trial_balance = balance_df[
        balance_df["Trial"] == trial_number
    ].copy()

    # --------------------------------------------------------
    # BEST TRIAL KONTROLÜ
    # --------------------------------------------------------

    is_best_trial = trial_number == BEST_TRIAL

    # 6. W&B CONFIG

    config = {

        # ----------------------------------------------------
        # EXPERIMENT
        # ----------------------------------------------------

        "experiment": "XGBoost + Optuna + Stratified K-Fold",

        "trial": trial_number,

        "is_best_trial": is_best_trial,

        # ----------------------------------------------------
        # CROSS VALIDATION
        # ----------------------------------------------------

        "k": int(trial_row["K"]),

        "early_stopping_rounds": int(
            trial_row["Early_Stopping"]
        ),

        # ----------------------------------------------------
        # XGBOOST PARAMETERS
        # ----------------------------------------------------

        "max_depth": int(
            trial_row["max_depth"]
        ),

        "learning_rate": float(
            trial_row["learning_rate"]
        ),

        "min_child_weight": int(
            trial_row["min_child_weight"]
        ),

        "subsample": float(
            trial_row["subsample"]
        ),

        "colsample_bytree": float(
            trial_row["colsample_bytree"]
        ),

        "gamma": float(
            trial_row["gamma"]
        ),

        "reg_alpha": float(
            trial_row["reg_alpha"]
        ),

        "reg_lambda": float(
            trial_row["reg_lambda"]
        ),

        # ----------------------------------------------------
        # MODEL SETTINGS
        # ----------------------------------------------------

        "objective": "reg:squarederror",

        "tree_method": "hist",

        "random_state": 42,

        "n_jobs": -1,

        "max_estimators": 5000,

        "target_is_log": True,

        # ----------------------------------------------------
        # OVERFITTING SETTINGS
        # ----------------------------------------------------

        "max_relative_gap": 0.20,

        "overfit_min_delta": 1e-4,

        "overfit_patience": 10,

        "overfit_fold_ratio_threshold": 0.20,

        # ----------------------------------------------------
        # TRIAL METRICS
        # ----------------------------------------------------

        "mean_train_rmse": float(
            trial_row["Mean_Train_RMSE"]
        ),

        "mean_validation_rmse": float(
            trial_row["Mean_Validation_RMSE"]
        ),

        "mean_log_rmse": float(
            trial_row["Mean_Log_RMSE"]
        ),

        "std_log_rmse": float(
            trial_row["Std_Log_RMSE"]
        ),

        "mean_usd_rmse": float(
            trial_row["Mean_USD_RMSE"]
        ),

        "std_usd_rmse": float(
            trial_row["Std_USD_RMSE"]
        ),

        # ----------------------------------------------------
        # TREE
        # ----------------------------------------------------

        "mean_best_tree": float(
            trial_row["Mean_Best_Tree"]
        ),

        # ----------------------------------------------------
        # GAP
        # ----------------------------------------------------

        "mean_relative_gap": float(
            trial_row["Mean_Relative_Gap"]
        ),

        "max_relative_gap": float(
            trial_row["Max_Relative_Gap"]
        ),

        # ----------------------------------------------------
        # OVERFITTING
        # ----------------------------------------------------

        "overfit_fold_ratio": float(
            trial_row["Overfit_Fold_Ratio"]
        ),

        "trial_overfit": bool(
            trial_row["Trial_Overfit"]
        ),

        # ----------------------------------------------------
        # BALANCE
        # ----------------------------------------------------

        "mean_balance_diff": float(
            trial_row["Mean_Balance_Diff"]
        ),

        "max_balance_diff": float(
            trial_row["Max_Balance_Diff"]
        ),

        "mean_price_balance_diff": float(
            trial_row["Mean_Price_Balance_Diff"]
        ),

        "mean_supply_balance_diff": float(
            trial_row["Mean_Supply_Balance_Diff"]
        ),

        "mean_competition_balance_diff": float(
            trial_row["Mean_Competition_Balance_Diff"]
        ),

        # ----------------------------------------------------
        # OBJECTIVE
        # ----------------------------------------------------

        "objective_value": float(
            trial_row["Objective_Value"]
        )
    }


    # 7. W&B RUN

    run_name = f"Optuna_Trial_{trial_number:02d}"

    if is_best_trial:
        run_name += "_BEST"

    run = wandb.init(
        project=PROJECT_NAME,
        entity=ENTITY_NAME,
        name=run_name,
        group="Optuna_20_Trials",
        job_type="optuna_trial",
        config=config,
        reinit="finish_previous"
    )

    # 8. ANA TRIAL METRİKLERİ

    run.log({

        # Objective
        "Objective_Value":
            float(trial_row["Objective_Value"]),

        # RMSE
        "Mean_Train_RMSE":
            float(trial_row["Mean_Train_RMSE"]),

        "Mean_Validation_RMSE":
            float(trial_row["Mean_Validation_RMSE"]),

        "Mean_Log_RMSE":
            float(trial_row["Mean_Log_RMSE"]),

        "Std_Log_RMSE":
            float(trial_row["Std_Log_RMSE"]),

        "Mean_USD_RMSE":
            float(trial_row["Mean_USD_RMSE"]),

        "Std_USD_RMSE":
            float(trial_row["Std_USD_RMSE"]),

        # Trees
        "Mean_Best_Tree":
            float(trial_row["Mean_Best_Tree"]),

        # Gap
        "Mean_Relative_Gap":
            float(trial_row["Mean_Relative_Gap"]),

        "Max_Relative_Gap":
            float(trial_row["Max_Relative_Gap"]),

        # Overfitting
        "Overfit_Fold_Ratio":
            float(trial_row["Overfit_Fold_Ratio"]),

        # Balance
        "Mean_Balance_Diff":
            float(trial_row["Mean_Balance_Diff"]),

        "Max_Balance_Diff":
            float(trial_row["Max_Balance_Diff"]),

        "Mean_Price_Balance_Diff":
            float(trial_row["Mean_Price_Balance_Diff"]),

        "Mean_Supply_Balance_Diff":
            float(trial_row["Mean_Supply_Balance_Diff"]),

        "Mean_Competition_Balance_Diff":
            float(
                trial_row[
                    "Mean_Competition_Balance_Diff"
                ]
            )
    })

    # 9. FOLD TABLE

    if len(trial_folds) > 0:

        fold_table = wandb.Table(
            dataframe=trial_folds
        )

        run.log({
            "Fold_Results": fold_table
        })

        # ----------------------------------------------
        # FOLD METRİKLERİNİ AYRI AYRI LOG
        # ----------------------------------------------

        for _, fold_row in trial_folds.iterrows():

            fold_number = int(
                fold_row["Fold"]
            )

            run.log({

                f"Fold_{fold_number}/Train_RMSE":
                    float(
                        fold_row["Train_RMSE"]
                    ),

                f"Fold_{fold_number}/Validation_RMSE":
                    float(
                        fold_row["Validation_RMSE"]
                    ),

                f"Fold_{fold_number}/Log_RMSE":
                    float(
                        fold_row["Log_RMSE"]
                    ),

                f"Fold_{fold_number}/USD_RMSE":
                    float(
                        fold_row["USD_RMSE"]
                    ),

                f"Fold_{fold_number}/Best_Tree":
                    float(
                        fold_row["Best_Tree"]
                    ),

                f"Fold_{fold_number}/Absolute_Gap":
                    float(
                        fold_row[
                            "Absolute_Train_Val_Gap"
                        ]
                    ),

                f"Fold_{fold_number}/Relative_Gap":
                    float(
                        fold_row[
                            "Relative_Train_Val_Gap"
                        ]
                    ),

                f"Fold_{fold_number}/Balance":
                    float(
                        trial_balance[
                            trial_balance["Fold"]
                            == fold_number
                        ][
                            "Max_Overall_Distribution_Diff"
                        ].iloc[0]
                    ),

                f"Fold_{fold_number}/Overfit":
                    int(
                        bool(
                            fold_row["Overfit"]
                        )
                    )
            })

    # 10. BOOSTING HISTORY TABLE


    if len(trial_history) > 0:

        history_table = wandb.Table(
            dataframe=trial_history
        )

        run.log({
            "Boosting_History": history_table
        })


    # 11. BALANCE TABLE


    if len(trial_balance) > 0:

        balance_table = wandb.Table(
            dataframe=trial_balance
        )

        run.log({
            "Fold_Balance": balance_table
        })


    # 12. BEST TRIAL


    if is_best_trial:

        run.summary["BEST_TRIAL"] = True

        run.summary["Best_Trial_Number"] = (
            BEST_TRIAL
        )

        # JSON'daki final best bilgiler
        run.summary["Best_Mean_USD_RMSE"] = (
            best_info["mean_usd_rmse"]
        )

        run.summary["Best_Mean_Log_RMSE"] = (
            best_info["mean_log_rmse"]
        )

        run.summary["Best_Mean_Best_Tree"] = (
            best_info["mean_best_tree"]
        )

    else:

        run.summary["BEST_TRIAL"] = False

    # 13. SUMMARY

    run.summary["Trial_Number"] = trial_number

    run.summary["K"] = int(
        trial_row["K"]
    )

    run.summary["Early_Stopping"] = int(
        trial_row["Early_Stopping"]
    )

    run.summary["Mean_USD_RMSE"] = float(
        trial_row["Mean_USD_RMSE"]
    )

    run.summary["Mean_Log_RMSE"] = float(
        trial_row["Mean_Log_RMSE"]
    )

    run.summary["Mean_Relative_Gap"] = float(
        trial_row["Mean_Relative_Gap"]
    )

    run.summary["Overfit_Fold_Ratio"] = float(
        trial_row["Overfit_Fold_Ratio"]
    )

    run.summary["Mean_Balance_Diff"] = float(
        trial_row["Mean_Balance_Diff"]
    )

    run.summary["Objective_Value"] = float(
        trial_row["Objective_Value"]
    )

    # 14. FINISH

    run.finish()

    print(
        f"✓ Trial {trial_number:02d} aktarıldı | "
        f"K={int(trial_row['K'])} | "
        f"USD RMSE="
        f"${float(trial_row['Mean_USD_RMSE']):,.2f} | "
        f"Best={is_best_trial}"
    )

# 15. TAMAMLANDI

print("\n" + "=" * 80)
print("W&B AKTARIMI TAMAMLANDI")
print("=" * 80)

print(f"Project : {PROJECT_NAME}")
print(f"Runs    : {len(parameter_df)}")
print(f"Best    : Trial {BEST_TRIAL}")

print("\nHer trial ayrı W&B Run olarak oluşturuldu.")

## TRAIN & VALIDATION

In [ ]:
# XGBOOST NON-OVERFIT FINAL
# STRATIFIED 9-FOLD + TEK FINAL MODEL
#
# KAYDEDİLENLER:
# 1. 9 adet fold modeli
# 2. En iyi fold modeli
# 3. TEK FINAL MODEL
# 4. BEST_PARAMS
# 5. MODEL_PARAMS
# 6. fold_df
# 7. history_df
# 8. oof_df
# 9. fold_models
# 10. summary
# 11. train metrics
# 12. validation metrics
# 13. train/validation percentage metrics
# 14. overfitting metrics
# 15. training loss history
# 16. OOF validation predictions
#
# FINAL MODEL:
# xgboost_final_model.json
#
# Final model:
# Train + Validation verisinin TAMAMI üzerinde
# median best tree sayısı ile eğitilir.

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# AYARLAR
SEED = 42
K = 9
N_ESTIMATORS = 5000
EARLY_STOPPING_ROUNDS = 9
CURRENT_GAP_THRESHOLD = 0.20
POST_BEST_THRESHOLD = 0.05
TARGET_IS_LOG = True

MODEL_NAME = "XGBoost_NonOverfit_Best"
OUTPUT_DIR = "xgboost_non_over_fit_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# BEST PARAMETERS
BEST_PARAMS = {
    "max_depth": 9,
    "learning_rate": 0.06780801631911669,
    "min_child_weight": 15,
    "subsample": 0.7072478977786331,
    "colsample_bytree": 0.8784958115626409,
    "gamma": 1.0206765935941309,
    "reg_alpha": 4.940127643286809,
    "reg_lambda": 0.31923634586063737
}

# XGBOOST MODEL PARAMETERS
MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    **BEST_PARAMS,
    "seed": SEED,
    "nthread": -1
}

# EĞİTİM KONFİGÜRASYONUNU KAYDET
training_configuration = {
    "model_name": MODEL_NAME,
    "framework": "XGBoost",
    "seed": SEED,
    "K": K,
    "n_estimators": N_ESTIMATORS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "current_gap_threshold": CURRENT_GAP_THRESHOLD,
    "post_best_threshold": POST_BEST_THRESHOLD,
    "target_is_log": TARGET_IS_LOG,
    "best_params": BEST_PARAMS,
    "model_params": MODEL_PARAMS
}

with open(
    os.path.join(OUTPUT_DIR, "training_parameters.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        training_configuration,
        f,
        indent=4,
        ensure_ascii=False
    )

# ENCODE EDİLMİŞ VERİLER
if not isinstance(X_train_encoded, pd.DataFrame):
    X_train_encoded = pd.DataFrame(X_train_encoded)

if not isinstance(X_val_encoded, pd.DataFrame):
    X_val_encoded = pd.DataFrame(X_val_encoded)

# TRAIN + VALIDATION BİRLEŞTİR
X_cv = pd.concat(
    [X_train_encoded, X_val_encoded],
    ignore_index=True
)

y_cv = pd.concat(
    [pd.Series(y_train), pd.Series(y_val)],
    ignore_index=True
)

y_cv_np = np.asarray(y_cv, dtype=np.float64)

# TARGET DÖNÜŞÜMÜ
if TARGET_IS_LOG:
    y_log = y_cv_np
    y_usd = np.maximum(np.expm1(y_log), 0)
else:
    y_usd = np.maximum(y_cv_np, 0)
    y_log = np.log1p(y_usd)

# STRATIFICATION KOLONLARI
supply_cv = np.concatenate([
    np.asarray(
        X_train["tender_supplytype"].to_list(),
        dtype=object
    ),
    np.asarray(
        X_val["tender_supplytype"].to_list(),
        dtype=object
    )
])

competition_cv = np.concatenate([
    np.asarray(
        X_train["competition_level"].to_list(),
        dtype=object
    ),
    np.asarray(
        X_val["competition_level"].to_list(),
        dtype=object
    )
])

# PRICE BIN
price_bin = pd.qcut(
    y_cv_np,
    q=10,
    labels=False,
    duplicates="drop"
)

stratify_df = pd.DataFrame({
    "price_bin": price_bin,
    "tender_supplytype": supply_cv,
    "competition_level": competition_cv
})

stratify_col = (
    stratify_df.astype(str)
    .agg("_".join, axis=1)
)

counts = stratify_col.value_counts()

# VERİ KONTROLLERİ
if len(X_cv) != len(y_cv):
    raise ValueError("X ve y uzunlukları eşleşmiyor.")

if len(X_cv) != len(stratify_col):
    raise ValueError(
        "X ve stratification uzunlukları eşleşmiyor."
    )

if counts.min() < K:
    raise ValueError(
        f"En küçük stratification kombinasyonu "
        f"{counts.min()} örnek. K={K} için yeterli değil."
    )

# OVERFITTING CALLBACK
class OverfittingCallback(xgb.callback.TrainingCallback):

    def __init__(self):
        self.history = []
        self.best_iteration = -1
        self.best_val = np.inf
        self.best_train = np.inf
        self.no_improvement = 0
        self.stopping_reason = "MAX_ESTIMATORS"

    def after_iteration(self, model, epoch, evals_log):

        train = float(
            evals_log["train"]["rmse"][-1]
        )

        val = float(
            evals_log["validation"]["rmse"][-1]
        )

        # CURRENT RELATIVE GAP
        gap = (
            (val - train) /
            max(train, 1e-12)
        )

        # BEST VALIDATION
        if val < self.best_val:
            self.best_val = val
            self.best_train = train
            self.best_iteration = epoch
            self.no_improvement = 0
        else:
            self.no_improvement += 1

        # POST-BEST DEGRADATION
        degradation = (
            (
                (val - self.best_val) /
                max(self.best_val, 1e-12)
            )
            if epoch > self.best_iteration
            else 0
        )

        # OVERFITTING FLAGS
        gap_overfit = gap > CURRENT_GAP_THRESHOLD
        degradation_overfit = (
            degradation > POST_BEST_THRESHOLD
        )

        overfit = (
            gap_overfit or
            degradation_overfit
        )

        # HISTORY
        self.history.append({
            "iteration": epoch + 1,
            "train_rmse": train,
            "validation_rmse": val,
            "relative_gap": gap,
            "best_iteration": self.best_iteration,
            "best_validation_rmse": self.best_val,
            "best_train_rmse": self.best_train,
            "post_best_degradation_ratio": degradation,
            "no_improvement_rounds": self.no_improvement,
            "current_gap_overfit": gap_overfit,
            "degradation_overfit": degradation_overfit,
            "overfit": overfit
        })

        # OVERFIT STOP
        if overfit:
            self.stopping_reason = "OVERFITTING"

            print(
                f"  OVERFIT STOP {epoch + 1} | "
                f"Best={self.best_iteration + 1} | "
                f"Gap={gap * 100:.2f}% | "
                f"Degradation={degradation * 100:.2f}%"
            )

            return True

        # EARLY STOPPING
        if self.no_improvement >= EARLY_STOPPING_ROUNDS:
            self.stopping_reason = "EARLY_STOPPING"

            print(
                f"  EARLY STOP {epoch + 1} | "
                f"Best={self.best_iteration + 1}"
            )

            return True

        return False

# METRİK FONKSİYONU
def calculate_metrics(y_true_log, pred_log):

    y_true_usd = np.maximum(
        np.expm1(y_true_log),
        0
    )

    pred_usd = np.maximum(
        np.expm1(pred_log),
        0
    )

    return {
        # LOG METRICS
        "rmse_log": np.sqrt(
            mean_squared_error(
                y_true_log,
                pred_log
            )
        ),
        "mae_log": mean_absolute_error(
            y_true_log,
            pred_log
        ),
        "r2_log": r2_score(
            y_true_log,
            pred_log
        ),

        # USD METRICS
        "rmse_usd": np.sqrt(
            mean_squared_error(
                y_true_usd,
                pred_usd
            )
        ),
        "mae_usd": mean_absolute_error(
            y_true_usd,
            pred_usd
        ),
        "r2_usd": r2_score(
            y_true_usd,
            pred_usd
        )
    }

# STRATIFIED K-FOLD
skf = StratifiedKFold(
    n_splits=K,
    shuffle=True,
    random_state=SEED
)

fold_models = []
fold_results = []
history_all = []

oof = np.empty(
    len(X_cv),
    dtype=np.float64
)

# FOLD EĞİTİMLERİ
print("\n" + "=" * 80)
print(f"XGBOOST NON-OVERFIT — STRATIFIED {K}-FOLD")
print("=" * 80)

print(f"Toplam CV veri  : {len(X_cv):,}")
print(f"Encoded feature : {X_cv.shape[1]}")
print(f"K               : {K}")

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_cv, stratify_col),
    1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{K}")
    print("=" * 70)

    print(f"Fold Train      : {len(train_idx):,}")
    print(f"Fold Validation : {len(val_idx):,}")

    # FOLD DATA
    X_train_fold = X_cv.iloc[train_idx]
    X_val_fold = X_cv.iloc[val_idx]

    y_train_fold = y_log[train_idx]
    y_val_fold = y_log[val_idx]

    # D-MATRIX
    dtrain = xgb.DMatrix(
        X_train_fold,
        label=y_train_fold
    )

    dval = xgb.DMatrix(
        X_val_fold,
        label=y_val_fold
    )

    # CALLBACK
    callback = OverfittingCallback()

    # MODEL TRAINING
    model = xgb.train(
        params=MODEL_PARAMS,
        dtrain=dtrain,
        num_boost_round=N_ESTIMATORS,
        evals=[
            (dtrain, "train"),
            (dval, "validation")
        ],
        callbacks=[callback],
        verbose_eval=False
    )

    # TRAINING HISTORY
    history = pd.DataFrame(callback.history)
    history["fold"] = fold

    history["is_best_iteration"] = (
        history["iteration"] ==
        callback.best_iteration + 1
    )

    history_all.append(history)

    # BEST TREE
    best_tree = callback.best_iteration + 1

    # TRAIN PREDICTION
    pred_train = model.predict(
        dtrain,
        iteration_range=(0, best_tree)
    )

    # VALIDATION PREDICTION
    pred_val = model.predict(
        dval,
        iteration_range=(0, best_tree)
    )

    # OOF
    oof[val_idx] = pred_val

    # METRICS
    train_metrics = calculate_metrics(
        y_train_fold,
        pred_train
    )

    val_metrics = calculate_metrics(
        y_val_fold,
        pred_val
    )

    # TRAIN / VALIDATION YÜZDELİKLERİ
    validation_rmse_vs_train_pct = (
        (
            val_metrics["rmse_usd"] -
            train_metrics["rmse_usd"]
        ) /
        max(
            train_metrics["rmse_usd"],
            1e-12
        )
    ) * 100

    validation_mae_vs_train_pct = (
        (
            val_metrics["mae_usd"] -
            train_metrics["mae_usd"]
        ) /
        max(
            train_metrics["mae_usd"],
            1e-12
        )
    ) * 100

    # TRAIN / VALIDATION RMSE PAYLARI
    total_rmse = (
        train_metrics["rmse_usd"] +
        val_metrics["rmse_usd"]
    )

    train_rmse_pct = (
        train_metrics["rmse_usd"] /
        max(total_rmse, 1e-12)
    ) * 100

    validation_rmse_pct = (
        val_metrics["rmse_usd"] /
        max(total_rmse, 1e-12)
    ) * 100

    # TRAIN / VALIDATION MAE PAYLARI
    total_mae = (
        train_metrics["mae_usd"] +
        val_metrics["mae_usd"]
    )

    train_mae_pct = (
        train_metrics["mae_usd"] /
        max(total_mae, 1e-12)
    ) * 100

    validation_mae_pct = (
        val_metrics["mae_usd"] /
        max(total_mae, 1e-12)
    ) * 100

    # R² YÜZDELERİ
    train_r2_pct = train_metrics["r2_usd"] * 100
    validation_r2_pct = val_metrics["r2_usd"] * 100

    # BEST LOSS
    best_row = history.iloc[callback.best_iteration]

    best_train = float(
        best_row["train_rmse"]
    )

    best_val = float(
        best_row["validation_rmse"]
    )

    best_gap = (
        (best_val - best_train) /
        max(best_train, 1e-12)
    )

    # FOLD RESULTS
    fold_results.append({

        # BASIC
        "fold": fold,
        "train_size": len(train_idx),
        "validation_size": len(val_idx),
        "actual_training_rounds": len(history),
        "stopping_reason": callback.stopping_reason,
        "best_iteration": callback.best_iteration,
        "best_tree": best_tree,

        # TRAIN LOG
        "train_rmse_log": train_metrics["rmse_log"],
        "train_mae_log": train_metrics["mae_log"],
        "train_r2_log": train_metrics["r2_log"],

        # VALIDATION LOG
        "validation_rmse_log": val_metrics["rmse_log"],
        "validation_mae_log": val_metrics["mae_log"],
        "validation_r2_log": val_metrics["r2_log"],

        # TRAIN USD
        "train_rmse_usd": train_metrics["rmse_usd"],
        "train_mae_usd": train_metrics["mae_usd"],
        "train_r2_usd": train_metrics["r2_usd"],

        # VALIDATION USD
        "validation_rmse_usd": val_metrics["rmse_usd"],
        "validation_mae_usd": val_metrics["mae_usd"],
        "validation_r2_usd": val_metrics["r2_usd"],

        # TRAIN / VALIDATION PERCENTAGES
        "validation_rmse_vs_train_pct":
            validation_rmse_vs_train_pct,
        "validation_mae_vs_train_pct":
            validation_mae_vs_train_pct,
        "train_r2_pct": train_r2_pct,
        "validation_r2_pct": validation_r2_pct,
        "train_rmse_pct": train_rmse_pct,
        "validation_rmse_pct": validation_rmse_pct,
        "train_mae_pct": train_mae_pct,
        "validation_mae_pct": validation_mae_pct,

        # BEST LOSS / GAP
        "best_train_rmse": best_train,
        "best_validation_rmse": best_val,
        "best_relative_gap": best_gap,
        "max_relative_gap":
            history["relative_gap"].max(),
        "max_post_best_degradation":
            history[
                "post_best_degradation_ratio"
            ].max(),

        # OVERFIT FLAGS
        "current_gap_overfit":
            bool(
                history[
                    "current_gap_overfit"
                ].any()
            ),
        "degradation_overfit":
            bool(
                history[
                    "degradation_overfit"
                ].any()
            ),
        "overfit":
            bool(
                history["overfit"].any()
            )
    })

    # MODEL BELLEKTE
    fold_models.append(model)

    # FOLD PRINT
    print("\n" + "-" * 60)
    print("TRAIN PERFORMANCE")
    print("-" * 60)

    print(
        f"Train RMSE Log : "
        f"{train_metrics['rmse_log']:.6f}"
    )

    print(
        f"Train RMSE USD : "
        f"${train_metrics['rmse_usd']:,.2f}"
    )

    print(
        f"Train MAE USD  : "
        f"${train_metrics['mae_usd']:,.2f}"
    )

    print(
        f"Train R² USD   : "
        f"{train_metrics['r2_usd']:.6f}"
    )

    print(
        f"Train R²       : "
        f"{train_r2_pct:.2f}%"
    )

    print("\n" + "-" * 60)
    print("VALIDATION PERFORMANCE")
    print("-" * 60)

    print(
        f"Val RMSE Log   : "
        f"{val_metrics['rmse_log']:.6f}"
    )

    print(
        f"Val RMSE USD   : "
        f"${val_metrics['rmse_usd']:,.2f}"
    )

    print(
        f"Val MAE USD    : "
        f"${val_metrics['mae_usd']:,.2f}"
    )

    print(
        f"Val R² USD     : "
        f"{val_metrics['r2_usd']:.6f}"
    )

    print(
        f"Val R²         : "
        f"{validation_r2_pct:.2f}%"
    )

    print("\n" + "-" * 60)
    print("TRAIN / VALIDATION PERCENTAGE")
    print("-" * 60)

    print(
        f"Train RMSE %             : "
        f"{train_rmse_pct:.2f}%"
    )

    print(
        f"Validation RMSE %        : "
        f"{validation_rmse_pct:.2f}%"
    )

    print(
        f"Train MAE %              : "
        f"{train_mae_pct:.2f}%"
    )

    print(
        f"Validation MAE %         : "
        f"{validation_mae_pct:.2f}%"
    )

    print(
        f"Validation RMSE vs Train : "
        f"{validation_rmse_vs_train_pct:.2f}%"
    )

    print(
        f"Validation MAE vs Train  : "
        f"{validation_mae_vs_train_pct:.2f}%"
    )

    print("\n" + "-" * 60)

    print(f"Best Tree      : {best_tree}")
    print(f"Best Gap       : {best_gap * 100:.2f}%")
    print(
        f"Stop Reason    : "
        f"{callback.stopping_reason}"
    )

# DATAFRAME'LER
fold_df = pd.DataFrame(fold_results)

history_df = pd.concat(
    history_all,
    ignore_index=True
)

# OOF METRICS
oof_metrics = calculate_metrics(
    y_log,
    oof
)

# EN İYİ FOLD
best_fold_idx = fold_df[
    "validation_rmse_log"
].idxmin()

best_fold = int(
    fold_df.loc[
        best_fold_idx,
        "fold"
    ]
)

best_fold_tree = int(
    fold_df.loc[
        best_fold_idx,
        "best_tree"
    ]
)

# MEDIAN BEST TREE
FINAL_BEST_TREE = int(
    round(
        fold_df["best_tree"].median()
    )
)

MEAN_BEST_TREE = float(
    fold_df["best_tree"].mean()
)

MEDIAN_BEST_TREE = float(
    fold_df["best_tree"].median()
)

print("\n" + "=" * 80)
print("FOLD TRAINING TAMAMLANDI")
print("=" * 80)

print(f"Mean Best Tree   : {MEAN_BEST_TREE:.2f}")
print(f"Median Best Tree : {MEDIAN_BEST_TREE:.2f}")
print(f"Final Best Tree  : {FINAL_BEST_TREE}")

# TEK FINAL MODEL EĞİTİMİ
print("\n" + "=" * 80)
print("TEK FINAL MODEL EĞİTİMİ")
print("=" * 80)

print("Final training data : Train + Validation")
print(f"Final samples       : {len(X_cv):,}")
print(f"Final features      : {X_cv.shape[1]}")
print(f"Final tree count    : {FINAL_BEST_TREE}")

# FINAL DTRAIN
dtrain_final = xgb.DMatrix(
    X_cv,
    label=y_log
)

# FINAL MODEL
final_model = xgb.train(
    params=MODEL_PARAMS,
    dtrain=dtrain_final,
    num_boost_round=FINAL_BEST_TREE,
    evals=[
        (dtrain_final, "train")
    ],
    verbose_eval=False
)

# FINAL MODEL TRAIN PREDICTION
final_train_pred_log = final_model.predict(
    dtrain_final
)

final_train_metrics = calculate_metrics(
    y_log,
    final_train_pred_log
)

# FINAL MODELİ KAYDET
FINAL_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_model.json"
)

final_model.save_model(FINAL_MODEL_PATH)

print("\nFinal model kaydedildi:")
print(FINAL_MODEL_PATH)

# EN İYİ FOLD MODELİ KAYDET
fold_models[best_fold - 1].save_model(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_best_fold_model.json"
    )
)

# SUMMARY
summary = {
    "model_name": MODEL_NAME,
    "framework": "XGBoost",
    "fold_strategy": "StratifiedKFold",
    "n_splits": K,
    "total_cv_samples": len(X_cv),
    "encoded_features": X_cv.shape[1],

    # TREE
    "mean_best_tree": MEAN_BEST_TREE,
    "median_best_tree": MEDIAN_BEST_TREE,
    "final_best_tree": FINAL_BEST_TREE,

    # VALIDATION
    "mean_validation_rmse_log":
        fold_df["validation_rmse_log"].mean(),
    "mean_validation_mae_log":
        fold_df["validation_mae_log"].mean(),
    "mean_validation_r2_log":
        fold_df["validation_r2_log"].mean(),
    "mean_validation_rmse_usd":
        fold_df["validation_rmse_usd"].mean(),
    "mean_validation_mae_usd":
        fold_df["validation_mae_usd"].mean(),
    "mean_validation_r2_usd":
        fold_df["validation_r2_usd"].mean(),

    # TRAIN / VALIDATION PERCENTAGES
    "mean_validation_rmse_vs_train_pct":
        fold_df[
            "validation_rmse_vs_train_pct"
        ].mean(),
    "mean_validation_mae_vs_train_pct":
        fold_df[
            "validation_mae_vs_train_pct"
        ].mean(),
    "mean_train_r2_pct":
        fold_df["train_r2_pct"].mean(),
    "mean_validation_r2_pct":
        fold_df["validation_r2_pct"].mean(),
    "mean_train_rmse_pct":
        fold_df["train_rmse_pct"].mean(),
    "mean_validation_rmse_pct":
        fold_df["validation_rmse_pct"].mean(),
    "mean_train_mae_pct":
        fold_df["train_mae_pct"].mean(),
    "mean_validation_mae_pct":
        fold_df["validation_mae_pct"].mean(),

    # OOF
    "oof_metrics": oof_metrics,

    # OVERFITTING
    "mean_best_gap":
        fold_df["best_relative_gap"].mean(),
    "mean_max_gap":
        fold_df["max_relative_gap"].mean(),
    "max_gap":
        fold_df["max_relative_gap"].max(),
    "overfit_folds":
        int(fold_df["overfit"].sum()),
    "overfit_ratio":
        fold_df["overfit"].mean(),

    # FINAL TRAIN
    "final_train_metrics":
        final_train_metrics,

    # BEST FOLD
    "best_fold": best_fold,
    "best_fold_tree": best_fold_tree,

    # FINAL MODEL
    "final_model": {
        "file": "xgboost_final_model.json",
        "training_data": "Train + Validation",
        "samples": len(X_cv),
        "features": X_cv.shape[1],
        "num_boost_round": FINAL_BEST_TREE,
        "tree_selection": "median_best_tree"
    },

    # STOPPING
    "stopping_configuration": {
        "current_gap_threshold":
            CURRENT_GAP_THRESHOLD,
        "post_best_threshold":
            POST_BEST_THRESHOLD,
        "early_stopping_rounds":
            EARLY_STOPPING_ROUNDS,
        "max_estimators":
            N_ESTIMATORS
    },

    # PARAMETERS
    "best_params": BEST_PARAMS,
    "model_params": MODEL_PARAMS
}

# OOF DATAFRAME
oof_df = pd.DataFrame({
    "index": np.arange(len(X_cv)),
    "actual_log": y_log,
    "predicted_log": oof,
    "actual_usd": y_usd,
    "predicted_usd": np.maximum(
        np.expm1(oof),
        0
    )
})

oof_df["residual_usd"] = (
    oof_df["actual_usd"] -
    oof_df["predicted_usd"]
)

oof_df["absolute_error_usd"] = (
    oof_df["residual_usd"].abs()
)

# FOLD CSV
fold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_metrics.csv"
    ),
    index=False
)

# HISTORY CSV
history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_training_history.csv"
    ),
    index=False
)

# OOF CSV
oof_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_oof_predictions.csv"
    ),
    index=False
)

# SUMMARY JSON
with open(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_complete_results.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=4,
        ensure_ascii=False
    )

# TÜM FOLD MODELLERİNİ KAYDET
for i, model in enumerate(fold_models, 1):
    model.save_model(
        os.path.join(
            OUTPUT_DIR,
            f"xgboost_fold_{i}.json"
        )
    )

# FOLD RESULTS JSON
fold_records = (
    fold_df
    .replace({np.nan: None})
    .to_dict(orient="records")
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "fold_results_for_wandb.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        fold_records,
        f,
        indent=4,
        ensure_ascii=False
    )

# TRAINING HISTORY JSON
history_records = (
    history_df
    .replace({np.nan: None})
    .to_dict(orient="records")
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "training_history_for_wandb.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        history_records,
        f,
        indent=4,
        ensure_ascii=False
    )

# W&B PARAMETERS
wandb_parameters = {
    "model_name": MODEL_NAME,
    "framework": "XGBoost",
    "seed": SEED,
    "k": K,
    "n_estimators": N_ESTIMATORS,
    "early_stopping_rounds":
        EARLY_STOPPING_ROUNDS,
    "current_gap_threshold":
        CURRENT_GAP_THRESHOLD,
    "post_best_threshold":
        POST_BEST_THRESHOLD,
    "target_is_log":
        TARGET_IS_LOG,
    "best_params": BEST_PARAMS,
    "model_params": MODEL_PARAMS,

    "final_model": {
        "filename":
            "xgboost_final_model.json",
        "train_data":
            "Train + Validation",
        "n_samples":
            len(X_cv),
        "n_features":
            X_cv.shape[1],
        "final_best_tree":
            FINAL_BEST_TREE,
        "tree_selection":
            "median_best_tree"
    }
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "wandb_parameters.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        wandb_parameters,
        f,
        indent=4,
        ensure_ascii=False
    )

# TRAIN LOSS VS VALIDATION LOSS
plt.figure(figsize=(14, 7))

for fold in range(1, K + 1):

    h = history_df[
        history_df["fold"] == fold
    ]

    plt.plot(
        h["iteration"],
        h["train_rmse"],
        alpha=0.30
    )

    plt.plot(
        h["iteration"],
        h["validation_rmse"],
        alpha=0.65,
        linestyle="--",
        label=f"Fold {fold}"
    )

plt.xlabel("Boosting Round")
plt.ylabel("RMSE Loss (Log)")
plt.title(
    f"{K}-Fold Train Loss vs Validation Loss"
)
plt.grid(alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "train_validation_loss.png"
    ),
    dpi=300
)

plt.show()

# 9-FOLD ORTALAMA TRAIN / VALIDATION LOSS
mean_history = (
    history_df
    .groupby("iteration")[
        ["train_rmse", "validation_rmse"]
    ]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 7))

plt.plot(
    mean_history["iteration"],
    mean_history["train_rmse"],
    linewidth=3,
    label="Mean Train Loss"
)

plt.plot(
    mean_history["iteration"],
    mean_history["validation_rmse"],
    linewidth=3,
    linestyle="--",
    label="Mean Validation Loss"
)

plt.xlabel("Boosting Round")
plt.ylabel("RMSE Loss (Log)")
plt.title(
    f"{K}-Fold Mean Train Loss vs Validation Loss"
)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "mean_train_validation_loss.png"
    ),
    dpi=300
)

plt.show()

# RELATIVE GAP
plt.figure(figsize=(14, 7))

for fold in range(1, K + 1):

    h = history_df[
        history_df["fold"] == fold
    ]

    plt.plot(
        h["iteration"],
        h["relative_gap"] * 100,
        alpha=0.5
    )

plt.axhline(
    CURRENT_GAP_THRESHOLD * 100,
    linestyle="--",
    label="20% Threshold"
)

plt.xlabel("Boosting Round")
plt.ylabel("Train-Validation Gap (%)")
plt.title("Current Relative Gap")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "relative_gap.png"
    ),
    dpi=300
)

plt.show()

# FINAL SUMMARY
print("\n" + "=" * 80)
print("XGBOOST NON-OVERFIT FINAL SUMMARY")
print("=" * 80)

# BASIC
print(f"K                : {K}")
print(f"Total CV Samples : {len(X_cv):,}")
print(f"Encoded Features : {X_cv.shape[1]}")
print(f"Max Estimators   : {N_ESTIMATORS}")
print(f"Early Stopping   : {EARLY_STOPPING_ROUNDS}")
print(
    f"Gap Threshold    : "
    f"{CURRENT_GAP_THRESHOLD * 100:.1f}%"
)

# TREE
print("\n" + "-" * 80)
print("TREE INFORMATION")
print("-" * 80)

print(
    f"Mean Best Tree   : "
    f"{summary['mean_best_tree']:.2f}"
)

print(
    f"Median Best Tree : "
    f"{summary['median_best_tree']:.2f}"
)

print(
    f"FINAL Best Tree  : "
    f"{summary['final_best_tree']}"
)

# VALIDATION PERFORMANCE
print("\n" + "-" * 80)
print("VALIDATION PERFORMANCE")
print("-" * 80)

print(
    f"Mean Validation RMSE Log : "
    f"{summary['mean_validation_rmse_log']:.6f}"
)

print(
    f"Mean Validation MAE Log  : "
    f"{summary['mean_validation_mae_log']:.6f}"
)

print(
    f"Mean Validation R² Log   : "
    f"{summary['mean_validation_r2_log']:.6f}"
)

print(
    f"Mean Validation RMSE USD : "
    f"${summary['mean_validation_rmse_usd']:,.2f}"
)

print(
    f"Mean Validation MAE USD  : "
    f"${summary['mean_validation_mae_usd']:,.2f}"
)

print(
    f"Mean Validation R² USD   : "
    f"{summary['mean_validation_r2_usd']:.6f}"
)

# TRAIN / VALIDATION PERCENTAGE
print("\n" + "-" * 80)
print("TRAIN / VALIDATION PERCENTAGE ANALYSIS")
print("-" * 80)

print(
    f"Mean Train RMSE %        : "
    f"{summary['mean_train_rmse_pct']:.2f}%"
)

print(
    f"Mean Validation RMSE %   : "
    f"{summary['mean_validation_rmse_pct']:.2f}%"
)

print(
    f"Mean Train MAE %         : "
    f"{summary['mean_train_mae_pct']:.2f}%"
)

print(
    f"Mean Validation MAE %    : "
    f"{summary['mean_validation_mae_pct']:.2f}%"
)

print(
    f"Mean Train R² %          : "
    f"{summary['mean_train_r2_pct']:.2f}%"
)

print(
    f"Mean Validation R² %     : "
    f"{summary['mean_validation_r2_pct']:.2f}%"
)

# GENERALIZATION GAP
print("\n" + "-" * 80)
print("TRAIN → VALIDATION GENERALIZATION GAP")
print("-" * 80)

print(
    f"Validation RMSE vs Train : "
    f"{summary['mean_validation_rmse_vs_train_pct']:.2f}%"
)

print(
    f"Validation MAE vs Train  : "
    f"{summary['mean_validation_mae_vs_train_pct']:.2f}%"
)

# OOF
print("\n" + "-" * 80)
print("OOF PERFORMANCE")
print("-" * 80)

print(
    f"OOF RMSE Log     : "
    f"{oof_metrics['rmse_log']:.6f}"
)

print(
    f"OOF MAE Log      : "
    f"{oof_metrics['mae_log']:.6f}"
)

print(
    f"OOF R² Log       : "
    f"{oof_metrics['r2_log']:.6f}"
)

print(
    f"OOF RMSE USD     : "
    f"${oof_metrics['rmse_usd']:,.2f}"
)

print(
    f"OOF MAE USD      : "
    f"${oof_metrics['mae_usd']:,.2f}"
)

print(
    f"OOF R² USD       : "
    f"{oof_metrics['r2_usd']:.6f}"
)

# FINAL TRAIN
print("\n" + "-" * 80)
print("FINAL MODEL TRAIN PERFORMANCE")
print("-" * 80)

print(
    f"Final Train RMSE : "
    f"${final_train_metrics['rmse_usd']:,.2f}"
)

print(
    f"Final Train MAE  : "
    f"${final_train_metrics['mae_usd']:,.2f}"
)

print(
    f"Final Train R²   : "
    f"{final_train_metrics['r2_usd']:.6f}"
)

# OVERFITTING
print("\n" + "-" * 80)
print("OVERFITTING ANALYSIS")
print("-" * 80)

print(
    f"Mean Best Gap    : "
    f"{summary['mean_best_gap'] * 100:.2f}%"
)

print(
    f"Mean Max Gap     : "
    f"{summary['mean_max_gap'] * 100:.2f}%"
)

print(
    f"Global Max Gap   : "
    f"{summary['max_gap'] * 100:.2f}%"
)

print(
    f"Overfit Folds    : "
    f"{summary['overfit_folds']}/{K}"
)

print(
    f"Overfit Ratio    : "
    f"{summary['overfit_ratio'] * 100:.2f}%"
)

# BEST FOLD
print("\n" + "-" * 80)
print("BEST FOLD")
print("-" * 80)

print(f"Best Fold        : {best_fold}")
print(f"Best Fold Tree   : {best_fold_tree}")

# KAYDEDİLEN MODEL DOSYALARI
print("\n" + "=" * 80)
print("KAYDEDİLEN MODEL DOSYALARI")
print("=" * 80)

for i in range(1, K + 1):
    print(
        os.path.join(
            OUTPUT_DIR,
            f"xgboost_fold_{i}.json"
        )
    )

print(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_best_fold_model.json"
    )
)

print(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_final_model.json"
    )
)

# W&B DEĞİŞKENLERİ
print("\n" + "=" * 80)
print("W&B AKTARIM DEĞİŞKENLERİ HAZIR")
print("=" * 80)

print(
    "fold_df       -> "
    "Fold train/validation sonuçları"
)

print(
    "history_df    -> "
    "Her boosting round train/validation loss"
)

print(
    "oof_df        -> "
    "OOF validation tahminleri"
)

print(
    "fold_models   -> "
    "Eğitilmiş 9 fold modeli"
)

print(
    "summary       -> "
    "Genel sonuçlar"
)

print(
    "BEST_PARAMS   -> "
    "Best hyperparameters"
)

print(
    "MODEL_PARAMS  -> "
    "XGBoost model parameters"
)

print(
    "final_model   -> "
    "Train + Validation üzerinde "
    "eğitilmiş tek final model"
)

print(
    "FINAL_BEST_TREE -> "
    "Median best tree"
)

print("\n" + "=" * 80)

print(
    f"Output Directory: "
    f"{OUTPUT_DIR}"
)

print("=" * 80)

**XGBoost 9-Fold Çapraz Doğrulama ve Performans Analiz Raporu**

Model eğitiminize ait 9-katlı (9-Fold) çapraz doğrulama (CV) metrikleri ve grafik değerlendirmesi aşağıdadır:Model genelde oldukça kararlı ve istikrarlı bir öğrenme süreci sergilemiştir. Toplam 252.589 gözlem ve 35 kodlanmış öznitelik ile yapılan bu eğitimde, $R^2$ değerinin genelde 0.80 civarında seyretmesi modelin açıklayıcılık gücünün yüksek olduğunu göstermektedir. Eğitim ve doğrulama kayıpları (Log RMSE) düzenli bir düşüş eğilimi izlemiş, ortalama en iyi ağaç sayısı 1207 seviyelerinde şekillenmiştir. Grafiklerde de görüldüğü üzere, boosting turları ilerledikçe Train ve Validation arasındaki fark (gap) kademeli olarak artarak eşik değere yaklaşmıştır.Eğitim sürecinde aşırı öğrenmeyi (overfitting) dizginlemek için konulan %20'lik fark eşiği (Gap Threshold) oldukça etkili çalışmıştır. 9 kattan 3 tanesi (%33.33) doğrudan %20'lik aşırı öğrenme sınırına takılarak "OVERFITTING" sebebiyle durdurulmuş, geri kalan 6 kat ise standart erken durdurma (early stopping) ile en iyi ağaç sayılarına ulaşmıştır. Ortalama ve maksimum farkın yaklaşık %19.2 seviyelerinde kalması, kontrol mekanizmasının modeli ezberleme aşamasına geçmeden tam zamanında frenlediğini kanıtlamaktadır.Buna karşın, veri setinde katlar arasında bazı aykırı değerler veya veri dağılımı uyumsuzlukları göze çarpmaktadır. Özellikle Fold 2 (R^2: 0.670) ve Fold 8 (R^2: 0.646) katlarında doğrulama RMSE değerleri $20M+ USD seviyelerine çıkarak diğer katlara oranla belirgin bir performans düşüşü yaşamıştır. Fold 4, 5 ve 6 gibi katlarda doğrulama R^2 değerleri 0.84 - 0.87 seviyelerine kadar çıkarken, Fold 2 ve 8'deki bu düşüş ilgili katların doğrulama kümelerinde yüksek tutarlı/uç değerlerin bulunduğuna işaret etmektedir.Genel bir değerlendirme yapmak gerekirse, model ezberleme riskine karşı %20 gap sınırı ile başarıyla korunmuş ve 2.06M MAE ortalaması ile tutarlı bir genelleme kapasitesi yakalamıştır.

## 0.35

Bu çalışmada yüksek fiyatlı ihalelerin model tarafından daha iyi öğrenilebilmesi amacıyla iki temel yaklaşım uygulanmıştır. İlk olarak, eğitim sırasında yüksek fiyatlı örneklere ağırlık verilmiştir. İhale fiyatlarının yüzdelik dilimlerine göre ağırlıklar belirlenmiş; normal gözlemlere 1.0, P75–P90 aralığına 1.2, P90–P95 aralığına 1.5, P95–P99 aralığına 2.0, P99–P99.5 aralığına 10 ve P99.5 üzerindeki en yüksek fiyatlı ihalelere 25 ağırlık verilmiştir. Böylece model, yüksek fiyatlı ihalelerde yaptığı hataları eğitim sırasında daha önemli kabul ederek bu gözlemlere daha fazla odaklanmıştır. Özellikle veri setinde sayıca az bulunan ekstrem yüksek fiyatlı ihalelerin öğrenme sürecindeki etkisinin artırılması hedeflenmiştir.

İkinci olarak, hedef değişkenin dönüşümünde kullanılan POWER değeri 0.25'ten 0.35'e yükseltilmiştir. POWER dönüşümü, hedef değişkenin dağılımını sıkıştırarak modelin çok büyük fiyat değerleriyle daha dengeli çalışmasını sağlayan bir kuvvet dönüşümüdür. POWER 0.25 kullanıldığında yüksek fiyat değerleri daha fazla sıkıştırılırken, POWER 0.35 kullanılmasıyla bu sıkıştırma azaltılmış ve yüksek fiyatlar arasındaki farklılıkların dönüşüm sonrasında daha fazla korunması sağlanmıştır. Bu sayede modelin özellikle yüksek fiyatlı ihaleler arasındaki farklılıkları öğrenebilmesi amaçlanmıştır. Bu değişiklik, ağırlıklandırmadan farklı olarak örneklerin önemini değil, modelin öğrenmeye çalıştığı hedef değerlerin ölçeğini ve dağılımını değiştirmektedir. Sonuç olarak, yüksek fiyatlı örneklere ağırlık verilmesi bu örneklerin eğitimdeki etkisini artırırken, POWER=0.35 kullanılması yüksek fiyatların hedef dönüşümünde daha az sıkıştırılmasını sağlamıştır. Bu iki yaklaşım hem XGBoost hem de Keras modellerinde uygulanarak yüksek fiyatlı ihalelerde gözlenen underfitting probleminin azaltılması ve modelin özellikle yüksek fiyat aralığındaki tahmin performansının iyileştirilmesi hedeflenmiştir.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
K = 9
N_ESTIMATORS = 5000
EARLY_STOPPING_ROUNDS = 15

CURRENT_GAP_THRESHOLD = 0.20
POST_BEST_THRESHOLD = 0.05
OVERFIT_PATIENCE = 5

TARGET_IS_LOG = True
POWER = 0.35

MODEL_NAME = "XGBoost_Weighted_HighPrice_9Fold_FinalBest"
OUTPUT_DIR = "xgboost_weighted_high_price"

os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_PARAMS = {
    "max_depth": 11,
    "learning_rate": 0.06780801631911669,
    "min_child_weight": 5,
    "subsample": 0.7072478977786331,
    "colsample_bytree": 0.90,
    "gamma": 0.5,
    "reg_alpha": 1.0,
    "reg_lambda": 0.31923634586063737
}

MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    **BEST_PARAMS,
    "seed": SEED,
    "nthread": -1
}

# TRAIN + VALIDATION
if isinstance(X_train_encoded, np.ndarray):
    X_train_encoded = pd.DataFrame(X_train_encoded)

if isinstance(X_val_encoded, np.ndarray):
    X_val_encoded = pd.DataFrame(X_val_encoded)

if X_train_encoded.shape[1] != X_val_encoded.shape[1]:
    raise ValueError(
        f"Train ve validation feature sayıları farklı: "
        f"{X_train_encoded.shape[1]} vs {X_val_encoded.shape[1]}"
    )

feature_names = [
    str(i) for i in range(X_train_encoded.shape[1])
]

X_train_encoded.columns = feature_names
X_val_encoded.columns = feature_names

X_cv = pd.concat(
    [X_train_encoded, X_val_encoded],
    ignore_index=True
)

# TEST
if isinstance(X_test_encoded, np.ndarray):
    X_test_encoded = pd.DataFrame(
        X_test_encoded,
        columns=feature_names
    )
else:
    if X_test_encoded.shape[1] != len(feature_names):
        raise ValueError(
            f"Test feature sayısı farklı: "
            f"{X_test_encoded.shape[1]} vs {len(feature_names)}"
        )

    X_test_encoded = X_test_encoded.copy()
    X_test_encoded.columns = feature_names

y_cv = np.concatenate(
    [
        np.asarray(y_train),
        np.asarray(y_val)
    ]
).astype(np.float64)

# LOG1P -> USD
if TARGET_IS_LOG:
    y_usd = np.maximum(
        np.expm1(y_cv),
        0
    )
else:
    y_usd = np.maximum(
        y_cv,
        0
    )

# USD -> USD^0.25
y_target = np.power(
    y_usd,
    POWER
)

# HIGH PRICE WEIGHTING
p50, p75, p90, p95, p99 = np.percentile(
    y_usd,
    [50, 75, 90, 95, 99]
)

p995 = np.percentile(
    y_usd,
    99.5
)

sample_weights = np.ones(
    len(y_usd),
    dtype=np.float32
)

sample_weights[y_usd >= p75] = 1.2
sample_weights[y_usd >= p90] = 1.5
sample_weights[y_usd >= p95] = 2.0
sample_weights[y_usd >= p99] = 10.0
sample_weights[y_usd >= p995] = 25.0

print("\n" + "=" * 80)
print("HIGH PRICE WEIGHTING")
print("=" * 80)

print(f"Target Transform : POWER {POWER:.2f}")
print(f"P50   : ${p50:,.2f} -> 1.0")
print(f"P75   : ${p75:,.2f} -> 1.2")
print(f"P90   : ${p90:,.2f} -> 1.5")
print(f"P95   : ${p95:,.2f} -> 2.0")
print(f"P99   : ${p99:,.2f} -> 10.0")
print(f"P99.5 : ${p995:,.2f} -> 25.0")

print("\nWeight Distribution")

for w in [1.0, 1.2, 1.5, 2.0, 10.0, 25.0]:
    print(
        f"Weight {w:<4}: "
        f"{(sample_weights == w).sum():,}"
    )

# STRATIFICATION
supply_cv = np.concatenate([
    np.asarray(
        X_train["tender_supplytype"],
        dtype=object
    ),
    np.asarray(
        X_val["tender_supplytype"],
        dtype=object
    )
])

competition_cv = np.concatenate([
    np.asarray(
        X_train["competition_level"],
        dtype=object
    ),
    np.asarray(
        X_val["competition_level"],
        dtype=object
    )
])

price_bin = pd.qcut(
    y_usd,
    q=10,
    labels=False,
    duplicates="drop"
)

stratify_df = pd.DataFrame({
    "price_bin": price_bin,
    "tender_supplytype": supply_cv,
    "competition_level": competition_cv
})

stratify_col = stratify_df.astype(str).agg(
    "_".join,
    axis=1
)

counts = stratify_col.value_counts()

if len(X_cv) != len(y_target):
    raise ValueError(
        "X ve y uzunlukları eşleşmiyor."
    )

if len(X_cv) != len(stratify_col):
    raise ValueError(
        "X ve stratification uzunlukları eşleşmiyor."
    )

if counts.min() < K:
    raise ValueError(
        f"En küçük grup {counts.min()}. "
        f"K={K} için yetersiz."
    )

# OVERFITTING CALLBACK
class OverfittingCallback(xgb.callback.TrainingCallback):

    def __init__(self):
        self.history = []
        self.best_iteration = -1
        self.best_val = np.inf
        self.best_train = np.inf
        self.no_improvement = 0
        self.overfit_rounds = 0
        self.stopping_reason = "MAX_ESTIMATORS"

    def after_iteration(
        self,
        model,
        epoch,
        evals_log
    ):

        train = float(
            evals_log["train"]["rmse"][-1]
        )

        val = float(
            evals_log["validation"]["rmse"][-1]
        )

        gap = (
            (val - train) /
            max(train, 1e-12)
        )

        if val < self.best_val:
            self.best_val = val
            self.best_train = train
            self.best_iteration = epoch
            self.no_improvement = 0
        else:
            self.no_improvement += 1

        degradation = (
            (val - self.best_val) /
            max(self.best_val, 1e-12)
            if epoch > self.best_iteration
            else 0
        )

        gap_overfit = (
            gap > CURRENT_GAP_THRESHOLD
        )

        degradation_overfit = (
            degradation > POST_BEST_THRESHOLD
        )

        if gap_overfit or degradation_overfit:
            self.overfit_rounds += 1
        else:
            self.overfit_rounds = 0

        overfit = (
            self.overfit_rounds >= OVERFIT_PATIENCE
        )

        self.history.append({
            "iteration": epoch + 1,
            "train_rmse": train,
            "validation_rmse": val,
            "relative_gap": gap,
            "best_iteration": self.best_iteration,
            "best_validation_rmse": self.best_val,
            "best_train_rmse": self.best_train,
            "post_best_degradation_ratio": degradation,
            "no_improvement_rounds": self.no_improvement,
            "overfit_rounds": self.overfit_rounds,
            "current_gap_overfit": gap_overfit,
            "degradation_overfit": degradation_overfit,
            "overfit": overfit
        })

        if overfit:

            self.stopping_reason = "OVERFITTING"

            print(
                f"  OVERFIT STOP {epoch + 1} | "
                f"Best={self.best_iteration + 1} | "
                f"Gap={gap * 100:.2f}% | "
                f"Degradation={degradation * 100:.2f}%"
            )

            return True

        if self.no_improvement >= EARLY_STOPPING_ROUNDS:

            self.stopping_reason = "EARLY_STOPPING"

            print(
                f"  EARLY STOP {epoch + 1} | "
                f"Best={self.best_iteration + 1}"
            )

            return True

        return False


# METRICS
def metrics(y_true_target, pred_target):

    # POWER -> USD
    y_true_usd = np.power(
        np.maximum(y_true_target, 0),
        1 / POWER
    )

    pred_usd = np.power(
        np.maximum(pred_target, 0),
        1 / POWER
    )

    return {
        "rmse_target": np.sqrt(
            mean_squared_error(
                y_true_target,
                pred_target
            )
        ),
        "mae_target": mean_absolute_error(
            y_true_target,
            pred_target
        ),
        "r2_target": r2_score(
            y_true_target,
            pred_target
        ),
        "rmse_usd": np.sqrt(
            mean_squared_error(
                y_true_usd,
                pred_usd
            )
        ),
        "mae_usd": mean_absolute_error(
            y_true_usd,
            pred_usd
        ),
        "r2_usd": r2_score(
            y_true_usd,
            pred_usd
        )
    }


# 9-FOLD
skf = StratifiedKFold(
    n_splits=K,
    shuffle=True,
    random_state=SEED
)

fold_models = []
fold_results = []
history_all = []

oof = np.empty(
    len(X_cv),
    dtype=np.float64
)

print("\n" + "=" * 80)
print(
    f"XGBOOST WEIGHTED HIGH PRICE — {K}-FOLD"
)
print("=" * 80)

print(f"CV Samples : {len(X_cv):,}")
print(f"Features   : {X_cv.shape[1]}")
print(f"K          : {K}")
print(f"Power      : {POWER}")
print("P99 Weight : 10.0")
print("P99.5 Weight: 25.0")

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_cv, stratify_col),
    1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{K}")
    print("=" * 70)

    Xtr = X_cv.iloc[train_idx]
    Xva = X_cv.iloc[val_idx]

    ytr = y_target[train_idx]
    yva = y_target[val_idx]

    wtr = sample_weights[train_idx]
    wva = sample_weights[val_idx]

    print(f"Train : {len(train_idx):,}")
    print(f"Val   : {len(val_idx):,}")

    print(
        f"Train Weight Mean : "
        f"{wtr.mean():.3f}"
    )

    print(
        f"Val Weight Mean   : "
        f"{wva.mean():.3f}"
    )

    dtr = xgb.DMatrix(
        Xtr,
        label=ytr,
        weight=wtr,
        feature_names=feature_names
    )

    dva = xgb.DMatrix(
        Xva,
        label=yva,
        weight=wva,
        feature_names=feature_names
    )

    callback = OverfittingCallback()

    model = xgb.train(
        params=MODEL_PARAMS,
        dtrain=dtr,
        num_boost_round=N_ESTIMATORS,
        evals=[
            (dtr, "train"),
            (dva, "validation")
        ],
        callbacks=[callback],
        verbose_eval=False
    )

    history = pd.DataFrame(
        callback.history
    )

    history["fold"] = fold

    history["is_best_iteration"] = (
        history["iteration"] ==
        callback.best_iteration + 1
    )

    history_all.append(history)

    best_tree = callback.best_iteration + 1

    pred_tr = model.predict(
        dtr,
        iteration_range=(0, best_tree)
    )

    pred_va = model.predict(
        dva,
        iteration_range=(0, best_tree)
    )

    oof[val_idx] = pred_va

    mt = metrics(
        ytr,
        pred_tr
    )

    mv = metrics(
        yva,
        pred_va
    )

    best_row = history.iloc[
        callback.best_iteration
    ]

    best_train = float(
        best_row["train_rmse"]
    )

    best_val = float(
        best_row["validation_rmse"]
    )

    best_gap = (
        (best_val - best_train) /
        max(best_train, 1e-12)
    )

    rmse_gap = (
        (mv["rmse_usd"] - mt["rmse_usd"]) /
        max(mt["rmse_usd"], 1e-12)
    ) * 100

    mae_gap = (
        (mv["mae_usd"] - mt["mae_usd"]) /
        max(mt["mae_usd"], 1e-12)
    ) * 100

    fold_results.append({

        "fold": fold,
        "train_size": len(train_idx),
        "validation_size": len(val_idx),

        "actual_training_rounds":
            len(history),

        "stopping_reason":
            callback.stopping_reason,

        "best_iteration":
            callback.best_iteration,

        "best_tree":
            best_tree,

        "train_rmse_target":
            mt["rmse_target"],

        "train_mae_target":
            mt["mae_target"],

        "train_r2_target":
            mt["r2_target"],

        "validation_rmse_target":
            mv["rmse_target"],

        "validation_mae_target":
            mv["mae_target"],

        "validation_r2_target":
            mv["r2_target"],

        "train_rmse_usd":
            mt["rmse_usd"],

        "train_mae_usd":
            mt["mae_usd"],

        "train_r2_usd":
            mt["r2_usd"],

        "validation_rmse_usd":
            mv["rmse_usd"],

        "validation_mae_usd":
            mv["mae_usd"],

        "validation_r2_usd":
            mv["r2_usd"],

        "validation_rmse_vs_train_pct":
            rmse_gap,

        "validation_mae_vs_train_pct":
            mae_gap,

        "best_train_rmse":
            best_train,

        "best_validation_rmse":
            best_val,

        "best_relative_gap":
            best_gap,

        "max_relative_gap":
            history["relative_gap"].max(),

        "max_post_best_degradation":
            history[
                "post_best_degradation_ratio"
            ].max(),

        "current_gap_overfit":
            bool(
                history[
                    "current_gap_overfit"
                ].any()
            ),

        "degradation_overfit":
            bool(
                history[
                    "degradation_overfit"
                ].any()
            ),

        "overfit":
            bool(
                history["overfit"].any()
            ),

        "mean_train_weight":
            float(wtr.mean()),

        "mean_validation_weight":
            float(wva.mean())
    })

    fold_models.append(model)

    model.save_model(
        os.path.join(
            OUTPUT_DIR,
            f"xgboost_fold_{fold}.json"
        )
    )

    print(
        f"Train RMSE : "
        f"${mt['rmse_usd']:,.2f}"
    )

    print(
        f"Val RMSE   : "
        f"${mv['rmse_usd']:,.2f}"
    )

    print(
        f"Train MAE  : "
        f"${mt['mae_usd']:,.2f}"
    )

    print(
        f"Val MAE    : "
        f"${mv['mae_usd']:,.2f}"
    )

    print(
        f"Train R²   : "
        f"{mt['r2_usd']:.6f}"
    )

    print(
        f"Val R²     : "
        f"{mv['r2_usd']:.6f}"
    )

    print(
        f"Best Tree  : {best_tree}"
    )

    print(
        f"Best Gap   : "
        f"{best_gap * 100:.2f}%"
    )

    print(
        f"Stop       : "
        f"{callback.stopping_reason}"
    )


fold_df = pd.DataFrame(
    fold_results
)

history_df = pd.concat(
    history_all,
    ignore_index=True
)

# OOF
oof_metrics = metrics(
    y_target,
    oof
)

# 9 FOLD ORTALAMALARI
mean_val_rmse_usd = (
    fold_df["validation_rmse_usd"].mean()
)

mean_val_mae_usd = (
    fold_df["validation_mae_usd"].mean()
)

mean_val_r2_usd = (
    fold_df["validation_r2_usd"].mean()
)

mean_val_rmse_target = (
    fold_df["validation_rmse_target"].mean()
)

mean_val_mae_target = (
    fold_df["validation_mae_target"].mean()
)

mean_val_r2_target = (
    fold_df["validation_r2_target"].mean()
)

mean_best_tree = (
    fold_df["best_tree"].mean()
)

FINAL_BEST_TREE = int(
    round(mean_best_tree)
)

print("\n" + "=" * 80)
print("9 FOLD ORTALAMA SONUCU")
print("=" * 80)

print(
    f"Mean Best Tree : "
    f"{mean_best_tree:.2f}"
)

print(
    f"Final Best Tree: "
    f"{FINAL_BEST_TREE}"
)

print(
    f"Mean Power RMSE : "
    f"{mean_val_rmse_target:.6f}"
)

print(
    f"Mean Val RMSE  : "
    f"${mean_val_rmse_usd:,.2f}"
)

print(
    f"Mean Val MAE   : "
    f"${mean_val_mae_usd:,.2f}"
)

print(
    f"Mean Val R²    : "
    f"{mean_val_r2_usd:.6f}"
)

# FINAL MODEL
print("\n" + "=" * 80)
print("FINAL BEST MODEL — TRAIN + VALIDATION")
print("=" * 80)

print(
    f"Training Samples : "
    f"{len(X_cv):,}"
)

print(
    f"Features         : "
    f"{X_cv.shape[1]}"
)

print(
    f"Boosting Rounds  : "
    f"{FINAL_BEST_TREE}"
)

dtrain_final = xgb.DMatrix(
    X_cv,
    label=y_target,
    weight=sample_weights,
    feature_names=feature_names
)

final_model = xgb.train(
    params=MODEL_PARAMS,
    dtrain=dtrain_final,
    num_boost_round=FINAL_BEST_TREE,
    evals=[
        (dtrain_final, "train")
    ],
    verbose_eval=False
)

final_train_pred_target = final_model.predict(
    dtrain_final
)

final_train_metrics = metrics(
    y_target,
    final_train_pred_target
)

BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_best_model.json"
)

FINAL_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_model.json"
)

final_model.save_model(
    BEST_MODEL_PATH
)

final_model.save_model(
    FINAL_MODEL_PATH
)

print("\nBEST MODEL KAYDEDİLDİ:")
print(BEST_MODEL_PATH)

print("\nFINAL MODEL KAYDEDİLDİ:")
print(FINAL_MODEL_PATH)

# TEST
print("\n" + "=" * 80)
print("FINAL TEST — BEST MODEL")
print("=" * 80)

print(
    f"Test Features : "
    f"{X_test_encoded.shape[1]}"
)

dtest = xgb.DMatrix(
    X_test_encoded,
    feature_names=feature_names
)

test_pred_target = final_model.predict(
    dtest
)

# POWER -> USD
test_pred_usd = np.power(
    np.maximum(test_pred_target, 0),
    1 / POWER
)

y_test_np = np.asarray(
    y_test,
    dtype=np.float64
)

# LOG1P -> USD
if TARGET_IS_LOG:
    test_true_usd = np.maximum(
        np.expm1(y_test_np),
        0
    )
else:
    test_true_usd = np.maximum(
        y_test_np,
        0
    )

# USD -> POWER
test_true_target = np.power(
    test_true_usd,
    POWER
)

test_metrics = metrics(
    test_true_target,
    test_pred_target
)

print(
    f"Test Samples : "
    f"{len(test_true_usd):,}"
)

print(
    f"Test RMSE    : "
    f"${test_metrics['rmse_usd']:,.2f}"
)

print(
    f"Test MAE     : "
    f"${test_metrics['mae_usd']:,.2f}"
)

print(
    f"Test R²      : "
    f"{test_metrics['r2_usd']:.6f}"
)

# TEST PREDICTIONS
test_df = pd.DataFrame({

    "index":
        np.arange(len(test_true_usd)),

    "actual_target":
        test_true_target,

    "predicted_target":
        test_pred_target,

    "actual_usd":
        test_true_usd,

    "predicted_usd":
        test_pred_usd
})

test_df["residual_usd"] = (
    test_df["actual_usd"] -
    test_df["predicted_usd"]
)

test_df["absolute_error_usd"] = (
    test_df["residual_usd"].abs()
)

test_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_test_predictions.csv"
    ),
    index=False
)

# OOF DATAFRAME
oof_pred_usd = np.power(
    np.maximum(oof, 0),
    1 / POWER
)

oof_df = pd.DataFrame({

    "index":
        np.arange(len(X_cv)),

    "actual_target":
        y_target,

    "predicted_target":
        oof,

    "actual_usd":
        y_usd,

    "predicted_usd":
        oof_pred_usd,

    "sample_weight":
        sample_weights
})

oof_df["residual_usd"] = (
    oof_df["actual_usd"] -
    oof_df["predicted_usd"]
)

oof_df["absolute_error_usd"] = (
    oof_df["residual_usd"].abs()
)

# SAVE TABLES
fold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_metrics.csv"
    ),
    index=False
)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_training_history.csv"
    ),
    index=False
)

oof_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_oof_predictions.csv"
    ),
    index=False
)

# SUMMARY
summary = {

    "model_name":
        MODEL_NAME,

    "framework":
        "XGBoost",

    "target_transform":
        f"power_{POWER:.2f}",

    "power":
        POWER,

    "original_target":
        "tender_finalpriceUsd",

    "source_target_transform":
        "log1p",

    "effective_training_target":
        f"tender_finalpriceUsd^{POWER:.2f}",

    "inverse_transform":
        f"power_{1 / POWER:.10f}",

    "fold_strategy":
        "StratifiedKFold",

    "n_splits":
        K,

    "total_cv_samples":
        len(X_cv),

    "test_samples":
        len(test_true_usd),

    "encoded_features":
        X_cv.shape[1],

    "weighting": {

        "p50":
            float(p50),

        "p75":
            float(p75),

        "p90":
            float(p90),

        "p95":
            float(p95),

        "p99":
            float(p99),

        "p995":
            float(p995),

        "weights": {
            "P50-P75": 1.0,
            "P75-P90": 1.2,
            "P90-P95": 1.5,
            "P95-P99": 2.0,
            "P99-P99.5": 10.0,
            "P99.5+": 25.0
        }
    },

    "tree_selection": {

        "fold_best_trees":
            fold_df["best_tree"].tolist(),

        "mean_best_tree":
            float(mean_best_tree),

        "final_best_tree":
            FINAL_BEST_TREE,

        "method":
            "mean_of_9_fold_best_trees",

        "final_model_training":
            "Train + Validation"
    },

    "mean_validation": {

        "rmse_target":
            float(mean_val_rmse_target),

        "mae_target":
            float(mean_val_mae_target),

        "r2_target":
            float(mean_val_r2_target),

        "rmse_usd":
            float(mean_val_rmse_usd),

        "mae_usd":
            float(mean_val_mae_usd),

        "r2_usd":
            float(mean_val_r2_usd)
    },

    "oof_metrics":
        oof_metrics,

    "test_metrics":
        test_metrics,

    "overfitting": {

        "mean_best_gap":
            float(
                fold_df[
                    "best_relative_gap"
                ].mean()
            ),

        "mean_max_gap":
            float(
                fold_df[
                    "max_relative_gap"
                ].mean()
            ),

        "global_max_gap":
            float(
                fold_df[
                    "max_relative_gap"
                ].max()
            ),

        "overfit_folds":
            int(
                fold_df["overfit"].sum()
            ),

        "overfit_ratio":
            float(
                fold_df["overfit"].mean()
            )
    },

    "final_model": {

        "type":
            "single_final_best_model",

        "number_of_fold_models":
            K,

        "training_data":
            "Train + Validation",

        "samples":
            len(X_cv),

        "features":
            X_cv.shape[1],

        "num_boost_round":
            FINAL_BEST_TREE,

        "tree_selection":
            "mean_of_9_fold_best_trees",

        "weighted_training":
            True,

        "file":
            "xgboost_best_model.json",

        "test_prediction":
            "single_final_best_model"
    },

    "final_train_metrics":
        final_train_metrics,

    "best_model_path":
        BEST_MODEL_PATH,

    "final_model_path":
        FINAL_MODEL_PATH,

    "best_params":
        BEST_PARAMS,

    "model_params":
        MODEL_PARAMS
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_complete_results.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4,
        ensure_ascii=False
    )

# LOSS
plt.figure(figsize=(14, 7))

for fold in range(1, K + 1):

    h = history_df[
        history_df["fold"] == fold
    ]

    plt.plot(
        h["iteration"],
        h["train_rmse"],
        alpha=0.20
    )

    plt.plot(
        h["iteration"],
        h["validation_rmse"],
        alpha=0.55,
        linestyle="--"
    )

plt.xlabel("Boosting Round")

plt.ylabel(
    f"Weighted RMSE Loss (Power {POWER:.2f} Target)"
)

plt.title(
    f"{K}-Fold Weighted Train vs Validation Loss"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "train_validation_loss.png"
    ),
    dpi=300
)

plt.show()

# MEAN LOSS
mean_history = (
    history_df
    .groupby("iteration")[
        ["train_rmse", "validation_rmse"]
    ]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 7))

plt.plot(
    mean_history["iteration"],
    mean_history["train_rmse"],
    linewidth=3,
    label="Mean Train Loss"
)

plt.plot(
    mean_history["iteration"],
    mean_history["validation_rmse"],
    linewidth=3,
    linestyle="--",
    label="Mean Validation Loss"
)

plt.xlabel("Boosting Round")

plt.ylabel(
    f"Weighted RMSE Loss (Power {POWER:.2f} Target)"
)

plt.title(
    f"{K}-Fold Mean Weighted Loss"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "mean_train_validation_loss.png"
    ),
    dpi=300
)

plt.show()

# GAP
plt.figure(figsize=(14, 7))

for fold in range(1, K + 1):

    h = history_df[
        history_df["fold"] == fold
    ]

    plt.plot(
        h["iteration"],
        h["relative_gap"] * 100,
        alpha=0.45
    )

plt.axhline(
    CURRENT_GAP_THRESHOLD * 100,
    linestyle="--",
    label="20% Threshold"
)

plt.xlabel("Boosting Round")
plt.ylabel("Train-Validation Gap (%)")

plt.title(
    "Weighted Relative Gap"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "relative_gap.png"
    ),
    dpi=300
)

plt.show()

# FINAL SUMMARY
print("\n" + "=" * 80)
print("FINAL RESULT — 9 FOLD MEAN -> BEST MODEL -> TEST")
print("=" * 80)

print(
    f"CV Samples       : "
    f"{len(X_cv):,}"
)

print(
    f"Test Samples     : "
    f"{len(test_true_usd):,}"
)

print(
    f"Features         : "
    f"{X_cv.shape[1]}"
)

print(
    f"K                : {K}"
)

print(
    f"Target Transform : POWER {POWER:.2f}"
)

print("\nWEIGHTS")

print(
    f"P75              : "
    f"{p75:,.2f} -> 1.2"
)

print(
    f"P90              : "
    f"{p90:,.2f} -> 1.5"
)

print(
    f"P95              : "
    f"{p95:,.2f} -> 2.0"
)

print(
    f"P99              : "
    f"{p99:,.2f} -> 10.0"
)

print(
    f"P99.5            : "
    f"{p995:,.2f} -> 25.0"
)

print("\n9 FOLD ORTALAMA")

print(
    f"Mean Best Tree   : "
    f"{mean_best_tree:.2f}"
)

print(
    f"Final Best Tree  : "
    f"{FINAL_BEST_TREE}"
)

print(
    f"Mean Val RMSE    : "
    f"${mean_val_rmse_usd:,.2f}"
)

print(
    f"Mean Val MAE     : "
    f"${mean_val_mae_usd:,.2f}"
)

print(
    f"Mean Val R²      : "
    f"{mean_val_r2_usd:.6f}"
)

print("\nOOF")

print(
    f"OOF RMSE         : "
    f"${oof_metrics['rmse_usd']:,.2f}"
)

print(
    f"OOF MAE          : "
    f"${oof_metrics['mae_usd']:,.2f}"
)

print(
    f"OOF R²           : "
    f"{oof_metrics['r2_usd']:.6f}"
)

print("\nFINAL BEST MODEL — TRAIN")

print(
    f"Train RMSE       : "
    f"${final_train_metrics['rmse_usd']:,.2f}"
)

print(
    f"Train MAE        : "
    f"${final_train_metrics['mae_usd']:,.2f}"
)

print(
    f"Train R²         : "
    f"{final_train_metrics['r2_usd']:.6f}"
)

print("\nFINAL TEST")

print(
    f"Test RMSE        : "
    f"${test_metrics['rmse_usd']:,.2f}"
)

print(
    f"Test MAE         : "
    f"${test_metrics['mae_usd']:,.2f}"
)

print(
    f"Test R²          : "
    f"{test_metrics['r2_usd']:.6f}"
)

print("\nOVERFITTING")

print(
    f"Mean Best Gap    : "
    f"{fold_df['best_relative_gap'].mean() * 100:.2f}%"
)

print(
    f"Global Max Gap   : "
    f"{fold_df['max_relative_gap'].max() * 100:.2f}%"
)

print(
    f"Overfit Folds    : "
    f"{fold_df['overfit'].sum()}/{K}"
)

print("\nSAVED BEST MODEL")

print(
    f"Best Model       : "
    f"{BEST_MODEL_PATH}"
)

print("\nFINAL MODEL")

print(
    "9 fold best tree ortalaması "
    "-> tek final model "
    "-> test"
)

print("\n" + "=" * 80)
print("TAMAMLANDI")
print("=" * 80)

Bu kodda, XGBoost ile POWER=0.35 dönüşümü ve yüksek fiyatlara duyarlı ağırlıklandırma kullanılarak 9-fold Stratified Cross-Validation gerçekleştiriliyor; stratification işleminde fiyat dilimi, tender_supplytype ve competition_level birlikte korunuyor. Her fold için early stopping ve özel overfitting kontrolü ile en iyi boosting ağacı belirleniyor, ardından 9 fold'un en iyi tree sayısının ortalaması alınarak Train+Validation verisiyle tek bir final model eğitiliyor. Modelin RMSE, MAE ve R² değerleri train, validation, OOF ve bağımsız test setinde hesaplanıyor; ayrıca residual tahminleri, fold eğitim geçmişleri, overfitting bilgileri, model parametreleri ve sonuçlar CSV/JSON olarak kaydediliyor. Son olarak final model test setinde değerlendirilip xgboost_best_model.json ve xgboost_final_model.json olarak saklanıyor.

## TEST

0.25

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

SEED = 42
POWER = 0.35
TARGET_IS_LOG = True

MODEL_NAME = "XGBoost_Weighted_HighPrice_0.30_FinalBest"
OUTPUT_DIR = "xgboost_weighted_high_price"

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_best_model.json"
)

RESULT_PATH = os.path.join(
    OUTPUT_DIR,
    "test_results.json"
)

PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "test_predictions.csv"
)

# MODEL
print("\n" + "=" * 80)
print("XGBOOST TEST — SAVED BEST MODEL")
print("=" * 80)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model bulunamadı: {MODEL_PATH}"
    )

model = xgb.Booster()
model.load_model(MODEL_PATH)

print(f"Model : {MODEL_PATH}")
print(f"Power : {POWER}")

# FEATURE NAMES
model_feature_names = model.feature_names

if model_feature_names is None:
    model_feature_names = [
        str(i)
        for i in range(X_test_encoded.shape[1])
    ]

print(
    f"Model Features : "
    f"{len(model_feature_names)}"
)

# TEST DATA
if isinstance(X_test_encoded, np.ndarray):

    X_test = pd.DataFrame(
        X_test_encoded,
        columns=model_feature_names
    )

else:

    if X_test_encoded.shape[1] != len(
        model_feature_names
    ):
        raise ValueError(
            f"Test feature sayısı {X_test_encoded.shape[1]}, "
            f"model feature sayısı {len(model_feature_names)}."
        )

    X_test = X_test_encoded.copy()
    X_test.columns = model_feature_names

# DTEST
dtest = xgb.DMatrix(
    X_test,
    feature_names=model_feature_names
)

# PREDICTION
pred_target = model.predict(
    dtest
)

# POWER -> USD
pred_usd = np.power(
    np.maximum(pred_target, 0),
    1 / POWER
)

# TRUE TARGET
y_test_np = np.asarray(
    y_test,
    dtype=np.float64
)

# LOG1P -> USD
if TARGET_IS_LOG:

    actual_usd = np.maximum(
        np.expm1(y_test_np),
        0
    )

else:

    actual_usd = np.maximum(
        y_test_np,
        0
    )

# USD -> POWER
actual_target = np.power(
    actual_usd,
    POWER
)

# METRICS — USD
mse_usd = mean_squared_error(
    actual_usd,
    pred_usd
)

rmse_usd = np.sqrt(
    mse_usd
)

mae_usd = mean_absolute_error(
    actual_usd,
    pred_usd
)

r2_usd = r2_score(
    actual_usd,
    pred_usd
)

# METRICS — POWER TARGET
mse_target = mean_squared_error(
    actual_target,
    pred_target
)

rmse_target = np.sqrt(
    mse_target
)

mae_target = mean_absolute_error(
    actual_target,
    pred_target
)

r2_target = r2_score(
    actual_target,
    pred_target
)

# RESIDUALS
residual_usd = (
    actual_usd -
    pred_usd
)

absolute_error_usd = (
    np.abs(residual_usd)
)

# SUMMARY
print("\n" + "=" * 80)
print("TEST SONUÇLARI")
print("=" * 80)

print(
    f"Test Samples : "
    f"{len(actual_usd):,}"
)

print(
    f"Features     : "
    f"{X_test.shape[1]}"
)

print(
    f"Power        : "
    f"{POWER}"
)

print("\nUSD METRICS")

print(
    f"MSE          : "
    f"${mse_usd:,.2f}"
)

print(
    f"RMSE         : "
    f"${rmse_usd:,.2f}"
)

print(
    f"MAE          : "
    f"${mae_usd:,.2f}"
)

print(
    f"R²           : "
    f"{r2_usd:.6f}"
)

print("\nPOWER TARGET METRICS")

print(
    f"MSE          : "
    f"{mse_target:.6f}"
)

print(
    f"RMSE         : "
    f"{rmse_target:.6f}"
)

print(
    f"MAE          : "
    f"{mae_target:.6f}"
)

print(
    f"R²           : "
    f"{r2_target:.6f}"
)

# PREDICTION TABLE
test_predictions = pd.DataFrame({

    "index":
        np.arange(
            len(actual_usd)
        ),

    "actual_target":
        actual_target,

    "predicted_target":
        pred_target,

    "actual_usd":
        actual_usd,

    "predicted_usd":
        pred_usd,

    "residual_usd":
        residual_usd,

    "absolute_error_usd":
        absolute_error_usd
})

test_predictions.to_csv(
    PREDICTION_PATH,
    index=False
)

# RESULT JSON
results = {

    "model_name":
        MODEL_NAME,

    "model_path":
        MODEL_PATH,

    "target":
        "tender_finalpriceUsd",

    "target_is_log":
        TARGET_IS_LOG,

    "power":
        POWER,

    "training_target":
        f"tender_finalpriceUsd^{POWER}",

    "inverse_transform":
        f"prediction^(1/{POWER})",

    "test_samples":
        int(len(actual_usd)),

    "features":
        int(X_test.shape[1]),

    "metrics_usd": {

        "mse":
            float(mse_usd),

        "rmse":
            float(rmse_usd),

        "mae":
            float(mae_usd),

        "r2":
            float(r2_usd)
    },

    "metrics_power_target": {

        "mse":
            float(mse_target),

        "rmse":
            float(rmse_target),

        "mae":
            float(mae_target),

        "r2":
            float(r2_target)
    },

    "prediction_file":
        PREDICTION_PATH
}

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\n" + "=" * 80)
print("KAYDEDİLEN DOSYALAR")
print("=" * 80)

print(
    f"Model       : {MODEL_PATH}"
)

print(
    f"Predictions : {PREDICTION_PATH}"
)

print(
    f"Results     : {RESULT_PATH}"
)

print("\n" + "=" * 80)
print("TEST TAMAMLANDI")
print("=" * 80)

Bu test sonucuna göre XGBoost modeli bağımsız test setinde başarılı bir performans göstermiştir. USD ölçeğinde RMSE 15.73M, MAE 2.78M ve R²=0.8496 elde edilmesi, modelin fiyatlardaki varyansın yaklaşık %85’ini açıkladığını gösteriyor. Özellikle MAE’nin 2.78M olması, tahminlerin ortalama mutlak hatasının yaklaşık 2.8 milyon dolar seviyesinde olduğunu ifade ediyor. POWER=0.25 dönüşümlü hedefte ise RMSE=10.97 ve R²=0.5528 elde edilmiş; ancak asıl iş hedefi USD olduğu için modelin değerlendirilmesinde USD metrikleri daha anlamlıdır. Genel olarak R²=0.85 test sonucu güçlü, fakat yüksek fiyatlı örneklerdeki hata dağılımı ayrıca dikkate alınmalıdır.

## W & B

In [ ]:
# ============================================================
# XGBOOST NON-OVERFIT → W&B
#
# TAM OLARAK 4 RUN:
# 1. XGBoost_TRAIN
# 2. XGBoost_VALIDATION
# 3. XGBoost_TEST
# 4. XGBoost_OVERFITTING
#
# + FINAL TEST PARAMETRELERİ
# + FINAL MODEL PARAMETRELERİ
# + CV
# + OVERFITTING
# + TEST METRİKLERİ
# + ACTUAL / PREDICTED TEST VERİLERİ
# + MANUEL W&B SCATTER İÇİN TABLE
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import wandb

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# W&B
# ============================================================

wandb.login()

PROJECT_NAME = "GPPD_XGBoost_Optuna"
ENTITY_NAME = "iremerkurt34-personal"
GROUP_NAME = "XGBoost_NonOverfit_Final"

# ============================================================
# GEREKLİ DEĞİŞKEN KONTROLÜ
# ============================================================

required_variables = [
    "fold_df",
    "history_df",
    "summary",
    "fold_models",
    "X_test_encoded",
    "y_test"
]

missing_variables = [
    v for v in required_variables
    if v not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Aşağıdaki değişkenler notebook belleğinde bulunamadı:\n"
        + "\n".join(missing_variables)
        + "\n\n"
        "Bu W&B kodunu eğitim kodunun hemen ardından çalıştır."
    )

# ============================================================
# PARAMETRELER
# ============================================================

BEST_PARAMS = summary.get("best_params", {})
MODEL_PARAMS = summary.get("model_params", {})
STOPPING_CONFIG = summary.get("stopping_configuration", {})

K = int(
    summary.get(
        "n_splits",
        fold_df["fold"].nunique()
    )
)

SEED = int(
    MODEL_PARAMS.get(
        "seed",
        42
    )
)

N_ESTIMATORS = int(
    STOPPING_CONFIG.get(
        "max_estimators",
        5000
    )
)

EARLY_STOPPING_ROUNDS = int(
    STOPPING_CONFIG.get(
        "early_stopping_rounds",
        9
    )
)

CURRENT_GAP_THRESHOLD = float(
    STOPPING_CONFIG.get(
        "current_gap_threshold",
        0.20
    )
)

POST_BEST_THRESHOLD = float(
    STOPPING_CONFIG.get(
        "post_best_threshold",
        0.05
    )
)

# ============================================================
# FINAL MODEL TREE SAYISI
# ============================================================

if "FINAL_BEST_TREE" in globals():
    FINAL_BEST_TREE = int(FINAL_BEST_TREE)
else:
    FINAL_BEST_TREE = int(
        round(
            fold_df["best_tree"].median()
        )
    )

MEAN_BEST_TREE = float(
    fold_df["best_tree"].mean()
)

MEDIAN_BEST_TREE = float(
    fold_df["best_tree"].median()
)

FINAL_MODEL_NAME = globals().get(
    "FINAL_MODEL_NAME",
    "XGBoost_NonOverfit_Final"
)

TARGET_IS_LOG_VALUE = bool(
    globals().get(
        "TARGET_IS_LOG",
        True
    )
)

# ============================================================
# BEST FOLD
# ============================================================

best_fold_idx = fold_df[
    "validation_rmse_log"
].idxmin()

BEST_FOLD = int(
    fold_df.loc[
        best_fold_idx,
        "fold"
    ]
)

BEST_FOLD_TREE = int(
    fold_df.loc[
        best_fold_idx,
        "best_tree"
    ]
)

# ============================================================
# ORTAK W&B CONFIG
# ============================================================

COMMON_CONFIG = {

    "model_name": FINAL_MODEL_NAME,
    "framework": "XGBoost",

    "target": "tender_finalpriceUsd",
    "target_transformation": "log1p",
    "target_is_log": TARGET_IS_LOG_VALUE,

    # CV
    "fold_strategy": "StratifiedKFold",
    "n_splits": K,
    "random_state": SEED,

    "total_cv_samples": int(
        summary.get(
            "total_cv_samples",
            len(fold_df["train_size"])
        )
    ),

    "encoded_features": int(
        summary.get(
            "encoded_features",
            0
        )
    ),

    # Optuna
    "max_depth": BEST_PARAMS.get("max_depth"),
    "learning_rate": BEST_PARAMS.get("learning_rate"),
    "min_child_weight": BEST_PARAMS.get("min_child_weight"),
    "subsample": BEST_PARAMS.get("subsample"),
    "colsample_bytree": BEST_PARAMS.get("colsample_bytree"),
    "gamma": BEST_PARAMS.get("gamma"),
    "reg_alpha": BEST_PARAMS.get("reg_alpha"),
    "reg_lambda": BEST_PARAMS.get("reg_lambda"),

    # Training
    "max_estimators": N_ESTIMATORS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,

    "current_gap_threshold": CURRENT_GAP_THRESHOLD,
    "post_best_threshold": POST_BEST_THRESHOLD,

    # Tree
    "mean_best_tree": MEAN_BEST_TREE,
    "median_best_tree": MEDIAN_BEST_TREE,
    "final_best_tree": FINAL_BEST_TREE,

    "best_fold": BEST_FOLD,
    "best_fold_tree": BEST_FOLD_TREE,

    "seed": SEED
}

# ============================================================
# RUN 1 — TRAIN
# ============================================================

print("\nRUN 1/4 → TRAIN")

train_run = wandb.init(
    project=PROJECT_NAME,
    entity=ENTITY_NAME,
    name="XGBoost_TRAIN",
    group=GROUP_NAME,
    job_type="train",
    config={
        **COMMON_CONFIG,
        "run_type": "TRAIN"
    }
)

train_summary = {

    "Mean_Train_RMSE_Log":
        float(
            fold_df[
                "train_rmse_log"
            ].mean()
        ),

    "Mean_Train_MAE_Log":
        float(
            fold_df[
                "train_mae_log"
            ].mean()
        ),

    "Mean_Train_R2_Log":
        float(
            fold_df[
                "train_r2_log"
            ].mean()
        ),

    "Mean_Train_RMSE_USD":
        float(
            fold_df[
                "train_rmse_usd"
            ].mean()
        ),

    "Mean_Train_MAE_USD":
        float(
            fold_df[
                "train_mae_usd"
            ].mean()
        ),

    "Mean_Train_R2_USD":
        float(
            fold_df[
                "train_r2_usd"
            ].mean()
        )
}

wandb.log(train_summary)

# ------------------------------------------------------------
# TRAIN FOLD TABLE
# ------------------------------------------------------------

train_columns = [
    "fold",
    "train_size",
    "best_tree",
    "train_rmse_log",
    "train_mae_log",
    "train_r2_log",
    "train_rmse_usd",
    "train_mae_usd",
    "train_r2_usd"
]

train_table_df = fold_df[
    train_columns
].copy()

wandb.log({
    "TRAIN_Fold_Results":
        wandb.Table(
            dataframe=train_table_df
        )
})

# ------------------------------------------------------------
# TRAIN BAR
# ------------------------------------------------------------

train_bar = fold_df[
    [
        "fold",
        "train_rmse_usd",
        "train_mae_usd",
        "train_r2_usd"
    ]
].copy()

train_bar["fold_label"] = (
    "Fold "
    + train_bar["fold"].astype(str)
)

wandb.log({

    "TRAIN_RMSE_BAR":
        wandb.plot.bar(
            wandb.Table(
                dataframe=train_bar[
                    [
                        "fold_label",
                        "train_rmse_usd"
                    ]
                ]
            ),
            "fold_label",
            "train_rmse_usd",
            title="Train RMSE by Fold"
        ),

    "TRAIN_MAE_BAR":
        wandb.plot.bar(
            wandb.Table(
                dataframe=train_bar[
                    [
                        "fold_label",
                        "train_mae_usd"
                    ]
                ]
            ),
            "fold_label",
            "train_mae_usd",
            title="Train MAE by Fold"
        ),

    "TRAIN_R2_BAR":
        wandb.plot.bar(
            wandb.Table(
                dataframe=train_bar[
                    [
                        "fold_label",
                        "train_r2_usd"
                    ]
                ]
            ),
            "fold_label",
            "train_r2_usd",
            title="Train R² by Fold"
        )
})

train_run.summary.update(train_summary)

wandb.finish()

# ============================================================
# RUN 2 — VALIDATION
# ============================================================

print("\nRUN 2/4 → VALIDATION")

validation_run = wandb.init(
    project=PROJECT_NAME,
    entity=ENTITY_NAME,
    name="XGBoost_VALIDATION",
    group=GROUP_NAME,
    job_type="validation",
    config={
        **COMMON_CONFIG,
        "run_type": "VALIDATION"
    }
)

validation_summary = {

    "Mean_Validation_RMSE_Log":
        float(
            fold_df[
                "validation_rmse_log"
            ].mean()
        ),

    "Mean_Validation_MAE_Log":
        float(
            fold_df[
                "validation_mae_log"
            ].mean()
        ),

    "Mean_Validation_R2_Log":
        float(
            fold_df[
                "validation_r2_log"
            ].mean()
        ),

    "Mean_Validation_RMSE_USD":
        float(
            fold_df[
                "validation_rmse_usd"
            ].mean()
        ),

    "Mean_Validation_MAE_USD":
        float(
            fold_df[
                "validation_mae_usd"
            ].mean()
        ),

    "Mean_Validation_R2_USD":
        float(
            fold_df[
                "validation_r2_usd"
            ].mean()
        )
}

wandb.log(validation_summary)

# ------------------------------------------------------------
# VALIDATION FOLD TABLE
# ------------------------------------------------------------

validation_columns = [
    "fold",
    "validation_size",
    "best_tree",
    "validation_rmse_log",
    "validation_mae_log",
    "validation_r2_log",
    "validation_rmse_usd",
    "validation_mae_usd",
    "validation_r2_usd"
]

validation_table_df = fold_df[
    validation_columns
].copy()

wandb.log({
    "VALIDATION_Fold_Results":
        wandb.Table(
            dataframe=validation_table_df
        )
})

# ------------------------------------------------------------
# VALIDATION BAR
# ------------------------------------------------------------

validation_bar = fold_df[
    [
        "fold",
        "validation_rmse_usd",
        "validation_mae_usd",
        "validation_r2_usd"
    ]
].copy()

validation_bar["fold_label"] = (
    "Fold "
    + validation_bar["fold"].astype(str)
)

wandb.log({

    "VALIDATION_RMSE_BAR":
        wandb.plot.bar(
            wandb.Table(
                dataframe=validation_bar[
                    [
                        "fold_label",
                        "validation_rmse_usd"
                    ]
                ]
            ),
            "fold_label",
            "validation_rmse_usd",
            title="Validation RMSE by Fold"
        ),

    "VALIDATION_MAE_BAR":
        wandb.plot.bar(
            wandb.Table(
                dataframe=validation_bar[
                    [
                        "fold_label",
                        "validation_mae_usd"
                    ]
                ]
            ),
            "fold_label",
            "validation_mae_usd",
            title="Validation MAE by Fold"
        ),

    "VALIDATION_R2_BAR":
        wandb.plot.bar(
            wandb.Table(
                dataframe=validation_bar[
                    [
                        "fold_label",
                        "validation_r2_usd"
                    ]
                ]
            ),
            "fold_label",
            "validation_r2_usd",
            title="Validation R² by Fold"
        )
})

validation_run.summary.update(
    validation_summary
)

wandb.finish()

# ============================================================
# RUN 3 — TEST
# ============================================================

print("\nRUN 3/4 → TEST")

# ============================================================
# TEST DATA
# ============================================================

if not isinstance(
    X_test_encoded,
    pd.DataFrame
):
    X_test_encoded_df = pd.DataFrame(
        X_test_encoded
    )
else:
    X_test_encoded_df = X_test_encoded.copy()

y_test_np = np.asarray(
    y_test,
    dtype=np.float64
)

# ============================================================
# TARGET
# ============================================================

if TARGET_IS_LOG_VALUE:

    y_test_log = y_test_np

else:

    y_test_usd_temp = np.maximum(
        y_test_np,
        0
    )

    y_test_log = np.log1p(
        y_test_usd_temp
    )

# ============================================================
# FINAL MODEL
# ============================================================

if "final_model" in globals():

    test_model = final_model

    TEST_MODEL_SOURCE = (
        "Final model trained on Train+Validation"
    )

    test_iteration_end = FINAL_BEST_TREE

else:

    test_model = fold_models[
        BEST_FOLD - 1
    ]

    TEST_MODEL_SOURCE = (
        "Best CV fold model"
    )

    test_iteration_end = BEST_FOLD_TREE

# ============================================================
# TEST AYARLARI
# ============================================================

TEST_UPLOAD_LIMIT = 10000

TEST_CONFIG = {

    **COMMON_CONFIG,

    "run_type": "TEST",

    "test_num_boost_round":
        int(test_iteration_end),

    "test_best_tree_selection":
        "median_best_tree",

    "train_validation_samples":
        int(
            len(
                globals().get(
                    "X_cv",
                    []
                )
            )
        ),

    "test_samples":
        int(
            len(
                X_test_encoded_df
            )
        ),

    "test_encoded_features":
        int(
            X_test_encoded_df.shape[1]
        ),

    "final_model_name":
        FINAL_MODEL_NAME,

    "test_model_source":
        TEST_MODEL_SOURCE,

    "test_upload_limit":
        TEST_UPLOAD_LIMIT
}

test_run = wandb.init(
    project=PROJECT_NAME,
    entity=ENTITY_NAME,
    name="XGBoost_TEST",
    group=GROUP_NAME,
    job_type="test",
    config=TEST_CONFIG
)

# ============================================================
# TEST PREDICTION
# ============================================================

dtest = xgb.DMatrix(
    X_test_encoded_df
)

test_pred_log = test_model.predict(
    dtest,
    iteration_range=(
        0,
        test_iteration_end
    )
)

# ============================================================
# USD DÖNÜŞÜMÜ
# ============================================================

test_actual_usd = np.maximum(
    np.expm1(y_test_log),
    0
)

test_pred_usd = np.maximum(
    np.expm1(test_pred_log),
    0
)

# ============================================================
# RESIDUAL
# ============================================================

test_residual = (
    test_actual_usd
    - test_pred_usd
)

test_absolute_error = np.abs(
    test_residual
)

# ============================================================
# TEST METRİKLERİ
# ============================================================

test_rmse_log = np.sqrt(
    mean_squared_error(
        y_test_log,
        test_pred_log
    )
)

test_mae_log = mean_absolute_error(
    y_test_log,
    test_pred_log
)

test_r2_log = r2_score(
    y_test_log,
    test_pred_log
)

test_rmse_usd = np.sqrt(
    mean_squared_error(
        test_actual_usd,
        test_pred_usd
    )
)

test_mae_usd = mean_absolute_error(
    test_actual_usd,
    test_pred_usd
)

test_r2_usd = r2_score(
    test_actual_usd,
    test_pred_usd
)

# ============================================================
# TRAIN-TEST GAP
# ============================================================

train_test_gap = None

if "train_final_metrics" in globals():

    train_rmse_log = (
        train_final_metrics[
            "rmse_log"
        ]
    )

    train_test_gap = (
        test_rmse_log
        - train_rmse_log
    ) / max(
        train_rmse_log,
        1e-12
    )

# ============================================================
# ACTUAL / PREDICTED TEST DATA
#
# ÖNEMLİ:
# Bu tablo W&B'de manuel scatter oluşturmak
# için kullanılacak.
#
# actual_usd      → X ekseni
# predicted_usd   → Y ekseni
# ============================================================

test_prediction_df = pd.DataFrame({

    "index":
        np.arange(
            len(y_test_log)
        ),

    # LOG
    "actual_log":
        y_test_log,

    "predicted_log":
        test_pred_log,

    # USD
    "actual_usd":
        test_actual_usd,

    "predicted_usd":
        test_pred_usd,

    # ERROR
    "residual_usd":
        test_residual,

    "absolute_error_usd":
        test_absolute_error
})

# ============================================================
# TEST METRİKLERİ
# ============================================================

test_metrics = {

    "Test_RMSE_Log":
        float(test_rmse_log),

    "Test_MAE_Log":
        float(test_mae_log),

    "Test_R2_Log":
        float(test_r2_log),

    "Test_RMSE_USD":
        float(test_rmse_usd),

    "Test_MAE_USD":
        float(test_mae_usd),

    "Test_R2_USD":
        float(test_r2_usd),

    "Test_Samples":
        int(len(y_test_log)),

    "Test_Num_Boost_Round":
        int(test_iteration_end),

    "Test_Model_Source":
        TEST_MODEL_SOURCE
}

if train_test_gap is not None:

    test_metrics[
        "Train_Test_RMSE_Gap_Percent"
    ] = float(
        train_test_gap * 100
    )

wandb.log(
    test_metrics
)

# ============================================================
# ACTUAL / PREDICTED ÖZETLERİ
# ============================================================

test_run.summary.update({

    "Actual_Test_Samples":
        int(
            len(
                test_actual_usd
            )
        ),

    "Actual_USD_Mean":
        float(
            np.mean(
                test_actual_usd
            )
        ),

    "Actual_USD_Median":
        float(
            np.median(
                test_actual_usd
            )
        ),

    "Actual_USD_Min":
        float(
            np.min(
                test_actual_usd
            )
        ),

    "Actual_USD_Max":
        float(
            np.max(
                test_actual_usd
            )
        ),

    "Predicted_USD_Mean":
        float(
            np.mean(
                test_pred_usd
            )
        ),

    "Predicted_USD_Median":
        float(
            np.median(
                test_pred_usd
            )
        ),

    "Predicted_USD_Min":
        float(
            np.min(
                test_pred_usd
            )
        ),

    "Predicted_USD_Max":
        float(
            np.max(
                test_pred_usd
            )
        ),

    "Residual_USD_Mean":
        float(
            np.mean(
                test_residual
            )
        ),

    "Absolute_Error_USD_Mean":
        float(
            np.mean(
                test_absolute_error
            )
        )
})

# ============================================================
# W&B'YE TEST TABLOSU
#
# 10.000 satır yüklenir.
#
# Bu tablo içerisinde:
#
# actual_usd
# predicted_usd
#
# kolonları ayrı ayrı bulunur.
#
# Böylece W&B arayüzünden manuel olarak
# scatter plot oluşturabilirsin.
# ============================================================

test_upload_df = test_prediction_df.head(
    TEST_UPLOAD_LIMIT
).copy()

wandb.log({

    "TEST_Actual_Predicted_Table":
        wandb.Table(
            dataframe=test_upload_df
        )
})

# ============================================================
# AYRI ACTUAL TABLOSU
# ============================================================

wandb.log({

    "TEST_Actual_USD":
        wandb.Table(
            dataframe=test_upload_df[
                [
                    "index",
                    "actual_usd"
                ]
            ]
        )
})

# ============================================================
# AYRI PREDICTED TABLOSU
# ============================================================

wandb.log({

    "TEST_Predicted_USD":
        wandb.Table(
            dataframe=test_upload_df[
                [
                    "index",
                    "predicted_usd"
                ]
            ]
        )
})

# ============================================================
# ACTUAL VS PREDICTED SCATTER
#
# Otomatik grafik.
# Manuel grafik için yukarıdaki
# TEST_Actual_Predicted_Table kullanılabilir.
# ============================================================

scatter_df = test_upload_df[
    [
        "actual_usd",
        "predicted_usd"
    ]
].copy()

actual_predicted_table = wandb.Table(
    dataframe=scatter_df
)

wandb.log({

    "TEST_Actual_vs_Predicted":
        wandb.plot.scatter(
            actual_predicted_table,
            "actual_usd",
            "predicted_usd",
            title=(
                "XGBoost Test - "
                "Actual vs Predicted"
            )
        )
})

# ============================================================
# ACTUAL VS PREDICTED LINE
# ============================================================

sorted_scatter_df = (
    test_upload_df
    .sort_values(
        "actual_usd"
    )
    .reset_index(
        drop=True
    )
)

wandb.log({

    "TEST_Actual_Predicted_Line":
        wandb.plot.line(
            wandb.Table(
                dataframe=sorted_scatter_df[
                    [
                        "actual_usd",
                        "predicted_usd"
                    ]
                ]
            ),
            "actual_usd",
            "predicted_usd",
            title=(
                "Test Actual vs Predicted"
            )
        )
})

# ============================================================
# RESIDUAL VS ACTUAL
# ============================================================

wandb.log({

    "TEST_Residual_vs_Actual":
        wandb.plot.scatter(
            wandb.Table(
                dataframe=test_upload_df[
                    [
                        "actual_usd",
                        "residual_usd"
                    ]
                ]
            ),
            "actual_usd",
            "residual_usd",
            title=(
                "Test Residual vs Actual"
            )
        )
})

# ============================================================
# ABSOLUTE ERROR VS ACTUAL
# ============================================================

wandb.log({

    "TEST_Absolute_Error_vs_Actual":
        wandb.plot.scatter(
            wandb.Table(
                dataframe=test_upload_df[
                    [
                        "actual_usd",
                        "absolute_error_usd"
                    ]
                ]
            ),
            "actual_usd",
            "absolute_error_usd",
            title=(
                "Test Absolute Error vs Actual"
            )
        )
})

test_run.summary.update(
    test_metrics
)

wandb.finish()

# ============================================================
# RUN 4 — OVERFITTING
# ============================================================

print("\nRUN 4/4 → OVERFITTING")

overfit_run = wandb.init(
    project=PROJECT_NAME,
    entity=ENTITY_NAME,
    name="XGBoost_OVERFITTING",
    group=GROUP_NAME,
    job_type="overfitting",
    config={
        **COMMON_CONFIG,
        "run_type": "OVERFITTING"
    }
)

# ============================================================
# HISTORY
# ============================================================

history_columns = [
    "fold",
    "iteration",
    "train_rmse",
    "validation_rmse",
    "relative_gap",
    "post_best_degradation_ratio",
    "best_validation_rmse",
    "best_train_rmse",
    "no_improvement_rounds",
    "current_gap_overfit",
    "degradation_overfit",
    "overfit"
]

history_log_df = history_df[
    history_columns
].copy()

history_log_df[
    "relative_gap_percent"
] = (
    history_log_df[
        "relative_gap"
    ] * 100
)

history_log_df[
    "post_best_degradation_percent"
] = (
    history_log_df[
        "post_best_degradation_ratio"
    ] * 100
)

wandb.log({

    "OVERFITTING_History":
        wandb.Table(
            dataframe=history_log_df
        )
})

# ============================================================
# FOLD HISTORY
# ============================================================

for fold in sorted(
    history_df["fold"].unique()
):

    h = history_df[
        history_df["fold"] == fold
    ].sort_values(
        "iteration"
    )

    for _, row in h.iterrows():

        wandb.log({

            f"Fold_{int(fold)}/Train_Loss":
                float(
                    row["train_rmse"]
                ),

            f"Fold_{int(fold)}/Validation_Loss":
                float(
                    row["validation_rmse"]
                ),

            f"Fold_{int(fold)}/Relative_Gap":
                float(
                    row["relative_gap"]
                ) * 100,

            f"Fold_{int(fold)}/Post_Best_Degradation":
                float(
                    row[
                        "post_best_degradation_ratio"
                    ]
                ) * 100,

            f"Fold_{int(fold)}/Overfit":
                int(
                    bool(
                        row["overfit"]
                    )
                ),

            "iteration":
                int(
                    row["iteration"]
                )
        })

# ============================================================
# MEAN HISTORY
# ============================================================

mean_history = (
    history_df
    .groupby("iteration")[
        [
            "train_rmse",
            "validation_rmse",
            "relative_gap",
            "post_best_degradation_ratio"
        ]
    ]
    .mean()
    .reset_index()
)

for _, row in mean_history.iterrows():

    wandb.log({

        "Mean_Train_Loss":
            float(
                row["train_rmse"]
            ),

        "Mean_Validation_Loss":
            float(
                row["validation_rmse"]
            ),

        "Mean_Relative_Gap":
            float(
                row["relative_gap"]
            ) * 100,

        "Mean_Post_Best_Degradation":
            float(
                row[
                    "post_best_degradation_ratio"
                ]
            ) * 100,

        "iteration":
            int(
                row["iteration"]
            )
    })

# ============================================================
# MEAN LOSS LINE
# ============================================================

mean_loss_table_df = mean_history[
    [
        "iteration",
        "train_rmse",
        "validation_rmse"
    ]
].copy()

wandb.log({

    "LINE_MEAN_LOSS":
        wandb.plot.line_series(

            xs=mean_loss_table_df[
                "iteration"
            ].tolist(),

            ys=[
                mean_loss_table_df[
                    "train_rmse"
                ].tolist(),

                mean_loss_table_df[
                    "validation_rmse"
                ].tolist()
            ],

            keys=[
                "Train RMSE",
                "Validation RMSE"
            ],

            title=(
                "Mean Train vs "
                "Validation Loss"
            )
        )
})

# ============================================================
# RELATIVE GAP
# ============================================================

wandb.log({

    "LINE_RELATIVE_GAP":
        wandb.plot.line_series(

            xs=mean_history[
                "iteration"
            ].tolist(),

            ys=[
                (
                    mean_history[
                        "relative_gap"
                    ] * 100
                ).tolist()
            ],

            keys=[
                "Train-Validation Gap (%)"
            ],

            title=(
                "Mean Relative "
                "Train-Validation Gap"
            )
        )
})

# ============================================================
# POST BEST DEGRADATION
# ============================================================

wandb.log({

    "LINE_POST_BEST":
        wandb.plot.line_series(

            xs=mean_history[
                "iteration"
            ].tolist(),

            ys=[
                (
                    mean_history[
                        "post_best_degradation_ratio"
                    ] * 100
                ).tolist()
            ],

            keys=[
                "Post-Best Degradation (%)"
            ],

            title=(
                "Mean Post-Best "
                "Validation Degradation"
            )
        )
})

# ============================================================
# GAP BAR
# ============================================================

gap_bar_df = fold_df[
    [
        "fold",
        "best_relative_gap",
        "max_relative_gap"
    ]
].copy()

gap_bar_df[
    "best_gap_percent"
] = (
    gap_bar_df[
        "best_relative_gap"
    ] * 100
)

gap_bar_df[
    "max_gap_percent"
] = (
    gap_bar_df[
        "max_relative_gap"
    ] * 100
)

gap_bar_df["fold_label"] = (
    "Fold "
    + gap_bar_df["fold"].astype(str)
)

gap_table = wandb.Table(
    dataframe=gap_bar_df[
        [
            "fold_label",
            "best_gap_percent",
            "max_gap_percent"
        ]
    ]
)

wandb.log({

    "BAR_GAP":
        wandb.plot.bar(
            gap_table,
            "fold_label",
            "max_gap_percent",
            title=(
                "Maximum Train-Validation "
                "Gap by Fold"
            )
        )
})

# ============================================================
# OVERFIT BAR
# ============================================================

overfit_df = fold_df[
    [
        "fold",
        "current_gap_overfit",
        "degradation_overfit",
        "overfit"
    ]
].copy()

overfit_df["fold_label"] = (
    "Fold "
    + overfit_df["fold"].astype(str)
)

overfit_table = wandb.Table(
    dataframe=overfit_df[
        [
            "fold_label",
            "current_gap_overfit",
            "degradation_overfit",
            "overfit"
        ]
    ]
)

wandb.log({

    "BAR_OVERFIT_FOLDS":
        wandb.plot.bar(
            overfit_table,
            "fold_label",
            "overfit",
            title=(
                "Overfitting Detection "
                "by Fold"
            )
        )
})

# ============================================================
# OVERFIT SUMMARY
# ============================================================

overfit_summary = {

    "Mean_Best_Gap_Percent":
        float(
            summary["mean_best_gap"]
        ) * 100,

    "Mean_Max_Gap_Percent":
        float(
            summary["mean_max_gap"]
        ) * 100,

    "Global_Max_Gap_Percent":
        float(
            summary["max_gap"]
        ) * 100,

    "Overfit_Folds":
        int(
            summary["overfit_folds"]
        ),

    "Overfit_Ratio_Percent":
        float(
            summary["overfit_ratio"]
        ) * 100,

    "Gap_Threshold_Percent":
        CURRENT_GAP_THRESHOLD * 100,

    "Post_Best_Threshold_Percent":
        POST_BEST_THRESHOLD * 100
}

wandb.log(
    overfit_summary
)

overfit_run.summary.update(
    overfit_summary
)

wandb.finish()

# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 80)
print("W&B AKTARIM TAMAMLANDI")
print("=" * 80)

print("\nOluşturulan Run sayısı : 4")
print("1 → XGBoost_TRAIN")
print("2 → XGBoost_VALIDATION")
print("3 → XGBoost_TEST")
print("4 → XGBoost_OVERFITTING")

print(
    f"\nBest Fold       : {BEST_FOLD}"
)

print(
    f"Best Fold Tree  : {BEST_FOLD_TREE}"
)

print(
    f"Median Tree     : "
    f"{MEDIAN_BEST_TREE:.0f}"
)

print(
    f"Final Best Tree : "
    f"{FINAL_BEST_TREE}"
)

print(
    f"\nMean Validation RMSE : "
    f"${summary['mean_validation_rmse_usd']:,.2f}"
)

print(
    f"OOF RMSE             : "
    f"${summary['oof_metrics']['rmse_usd']:,.2f}"
)

print(
    f"OOF MAE              : "
    f"${summary['oof_metrics']['mae_usd']:,.2f}"
)

print(
    f"OOF R²               : "
    f"{summary['oof_metrics']['r2_usd']:.6f}"
)

print(
    f"\nTest RMSE USD        : "
    f"${test_rmse_usd:,.2f}"
)

print(
    f"Test MAE USD         : "
    f"${test_mae_usd:,.2f}"
)

print(
    f"Test R² USD          : "
    f"{test_r2_usd:.6f}"
)

print(
    f"\nActual Mean USD      : "
    f"${np.mean(test_actual_usd):,.2f}"
)

print(
    f"Predicted Mean USD   : "
    f"${np.mean(test_pred_usd):,.2f}"
)

print(
    f"\nW&B Actual/Predicted "
    f"Table : {len(test_upload_df):,} satır"
)

print(
    "\nManuel scatter için W&B'de:"
)

print(
    "X → actual_usd"
)

print(
    "Y → predicted_usd"
)

print(
    "\nOverfit Folds        : "
    f"{summary['overfit_folds']}/{K}"
)

print(
    f"Overfit Ratio        : "
    f"{summary['overfit_ratio'] * 100:.2f}%"
)

print("=" * 80)

Regresyon modellerinin tahmin başarısını ölçmek ve hataların yapısını anlamak için kullanılan bu dört temel metrik, modelin gerçek değerlerden ne kadar saptığını farklı matematiksel yaklaşımlarla ortaya koyar.

**MAE (Mean Absolute Error - Ortalama Mutlak Hata),** modelin tahmin ettiği değerler ile gerçek değerler arasındaki farkların mutlak değerlerinin ortalamasını alır. Hataların yönüne (+ veya -) bakmaksızın tüm sapmaları eşit ağırlıkta topladığı için modelin genel olarak ne kadarlık bir fiyatlama hatası yaptığını en sade ve doğrusal şekilde gösteren, yorumlaması en kolay metriktir.

**MSE (Mean Squared Error - Ortalama Kare Hata)** ise hataların tek tek karesini alıp bunların ortalamasını hesaplar. Farkların karesi alındığı için büyük hatalar çarpan etkisiyle devasa boyutlara ulaşır ve bu durum MSE'yi verideki uç değerlere (outliers) karşı aşırı hassas hale getirir; nitekim analizinizdeki hizmet ihalelerinde görülen trilyon seviyesindeki MSE değeri, modelin az sayıdaki büyük ihalede yaptığı devasa hatalardan kaynaklanmaktadır.

**RMSE (Root Mean Squared Error - Kök Ortalama Kare Hata),** MSE değerinin karekökünün alınmasıyla elde edilir ve MSE'nin büyüterek cezalandırdığı hataları yeniden hedef değişkenin orijinal birimine (sizin modelinizde Milyon USD seviyesine) geri döndürür. MAE ile RMSE arasındaki farkın açılması veride büyük sapmalar yapan uç değerlerin varlığını kesinleştirirken, RMSE'nin düşük olması modelin büyük ve riskli hatalardan ne kadar kaçınabildiğini gösterir.

**R^2 (Belirlilik Katsayısı - R-Squared),** bağımlı değişkendeki (ihale fiyatlarındaki) toplam değişimin veya varyansın yüzde kaçının modelde kullanılan bağımsız değişkenler tarafından açıklanabildiğini gösteren bir uyum iyiliğidir. Genellikle 0 ile 1 arasında değişen bu değer, doğrudan bir fiyat hatası göstermek yerine modelin verideki genel trendi ve matematiği ne kadar başarıyla kavrayıp temsil edebildiğinin oranını verir.

## 13. XGBoost Model Değerlendirme ve Hata Analizi / XGBoost Model Evaluation and Error Analysis

Bu bölümde, eğitimi ve optimizasyonu tamamlanan XGBoost modelinin train, validation ve test performansları değerlendirilmekte ve modelin tahmin hataları ayrıntılı olarak analiz edilmektedir. Model performansının yalnızca genel metriklerle değerlendirilmesi yerine, modelin hangi gözlemlerde ve hangi özellik değerlerinde hata yaptığı incelenmektedir.

Bu aşamada tender_finalpriceUsd hedef değişkenine ait gerçek değerler ile XGBoost modelinin tahminleri karşılaştırılarak RMSE, MAE ve R² gibi performans metrikleri hesaplanmaktadır. Bunun yanında residual (artık) değerleri ve mutlak tahmin hataları analiz edilerek modelin underprediction (düşük tahmin) ve overprediction (yüksek tahmin) eğilimleri belirlenmektedir.

Modelin hangi özellikleri kullanarak tahmin gerçekleştirdiğini ve hangi özelliklerde daha fazla hata yaptığını belirlemek amacıyla feature importance, SHAP analizi, feature-error ilişkileri ve hata dağılımları görselleştirilmektedir. Böylece yalnızca modelin ne kadar başarılı olduğu değil, aynı zamanda hangi veri koşullarında başarısız olduğu ve tahmin hatalarının hangi özelliklerle ilişkili olduğu ortaya konulmaktadır.

Bu değerlendirme sürecinde test verisi model eğitiminden tamamen ayrı tutulmakta ve yalnızca nihai performans ile hata analizinde kullanılmaktadır. Böylece modelin görülmemiş veriler üzerindeki genelleme başarısı objektif biçimde ölçülmekte ve olası sistematik hata bölgeleri görsel ve istatistiksel analizlerle belirlenmektedir.

In [ ]:
# ============================================================
# XGBOOST FINAL MODEL → RESIDUAL ANALYSIS
# TEST SETİ ÜZERİNDE MODEL HATA ANALİZİ
# ============================================================

import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import wandb

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. AYARLAR
# ============================================================

SEED = 42

PROJECT_NAME = "GPPD_XGBoost_Optuna"
ENTITY_NAME = "iremerkurt34-personal"
RUN_NAME = "XGBoost_ERROR_ANALYSIS"

TARGET_NAME = "tender_finalpriceUsd"
LOG_TARGET_NAME = "log_tender_finalpriceUsd"

TARGET_IS_LOG = True

MODEL_FILENAME = "xgboost_final_model.json"
MODEL_DIR_NAME = "xgboost_non_over_fit_final"

# ============================================================
# 2. MODEL DOSYASINI OTOMATİK BUL
# ============================================================

print("=" * 70)
print("FINAL XGBOOST MODEL KONTROLÜ")
print("=" * 70)

cwd = os.getcwd()

print(f"Current Working Directory:\n{cwd}")

# Olası yollar
possible_paths = [
    os.path.join(
        cwd,
        MODEL_DIR_NAME,
        MODEL_FILENAME
    ),

    os.path.join(
        cwd,
        "..",
        MODEL_DIR_NAME,
        MODEL_FILENAME
    ),

    os.path.join(
        "/content",
        MODEL_DIR_NAME,
        MODEL_FILENAME
    ),

    os.path.join(
        "/mnt/data",
        MODEL_DIR_NAME,
        MODEL_FILENAME
    )
]

FINAL_MODEL_PATH = None

for path in possible_paths:

    path = os.path.abspath(path)

    if os.path.exists(path):

        FINAL_MODEL_PATH = path
        break

# ============================================================
# 3. CURRENT DIRECTORY ALTINDA MODEL ARA
# ============================================================

if FINAL_MODEL_PATH is None:

    search_pattern = os.path.join(
        cwd,
        "**",
        MODEL_FILENAME
    )

    found_models = glob.glob(
        search_pattern,
        recursive=True
    )

    if found_models:

        FINAL_MODEL_PATH = os.path.abspath(
            found_models[0]
        )

# ============================================================
# 4. MODEL BULUNAMAZSA DETAYLI HATA
# ============================================================

if FINAL_MODEL_PATH is None:

    print("\nModel bulunamadı.")

    print("\nAranan yollar:")

    for path in possible_paths:
        print(path)

    print("\nCurrent directory:")
    print(cwd)

    raise FileNotFoundError(
        "\nFinal XGBoost modeli bulunamadı.\n"
        f"Aranan dosya: {MODEL_FILENAME}\n"
        f"Beklenen klasör: {MODEL_DIR_NAME}"
    )

print("\nModel bulundu:")
print(FINAL_MODEL_PATH)

# ============================================================
# 5. MODELİ YÜKLE
# ============================================================

print("\nFinal model yükleniyor...")

model = xgb.XGBRegressor()

model.load_model(
    FINAL_MODEL_PATH
)

print("Final model başarıyla yüklendi.")

# ============================================================
# 6. TEST VERİSİ KONTROLÜ
# ============================================================

required_variables = [
    "X_test",
    "y_test"
]

missing_variables = [
    v
    for v in required_variables
    if v not in globals()
]

if missing_variables:

    raise NameError(
        "\nEksik değişkenler:\n"
        + "\n".join(missing_variables)
        + "\n\nNotebook'ta X_test ve y_test "
        "değişkenlerinin hazır olduğundan emin olun."
    )

print("\nTest verisi bulundu.")

print(
    f"X_test shape : "
    f"{getattr(X_test, 'shape', 'Unknown')}"
)

print(
    f"y_test shape : "
    f"{getattr(y_test, 'shape', 'Unknown')}"
)

# ============================================================
# 7. MODEL FEATURE KONTROLÜ
# ============================================================

try:

    model_feature_count = (
        model.get_booster().num_features()
    )

except Exception:

    model_feature_count = None

if model_feature_count is not None:

    if hasattr(X_test, "shape"):

        test_feature_count = X_test.shape[1]

        print(
            f"\nModel feature sayısı : "
            f"{model_feature_count}"
        )

        print(
            f"X_test feature sayısı: "
            f"{test_feature_count}"
        )

        if model_feature_count != test_feature_count:

            raise ValueError(
                "\nMODEL FEATURE SAYISI İLE X_TEST "
                "FEATURE SAYISI UYUŞMUYOR!\n\n"
                f"Model : {model_feature_count}\n"
                f"X_test: {test_feature_count}\n\n"
                "X_test'in final model ile aynı preprocessing "
                "işleminden geçtiğinden emin olun."
            )

# ============================================================
# 8. TEST TAHMİNİ
# ============================================================

print("\nTest tahminleri oluşturuluyor...")

y_test_array = np.asarray(
    y_test
).reshape(-1)

pred_log = model.predict(
    X_test
)

pred_log = np.asarray(
    pred_log
).reshape(-1)

# ============================================================
# 9. BOYUT KONTROLÜ
# ============================================================

if len(y_test_array) != len(pred_log):

    raise ValueError(
        "\ny_test ve prediction uzunlukları farklı!\n"
        f"y_test : {len(y_test_array)}\n"
        f"pred   : {len(pred_log)}"
    )

# ============================================================
# 10. LOG → USD
# ============================================================

if TARGET_IS_LOG:

    y_test_usd = np.expm1(
        y_test_array
    )

    pred_usd = np.expm1(
        pred_log
    )

else:

    y_test_usd = y_test_array
    pred_usd = pred_log

# Negatif tahminleri engelle
pred_usd = np.maximum(
    pred_usd,
    0
)

# ============================================================
# 11. RESIDUAL HESAPLAMA
# ============================================================

# Residual = Actual - Predicted

residual_log = (
    y_test_array
    - pred_log
)

residual_usd = (
    y_test_usd
    - pred_usd
)

absolute_error_usd = np.abs(
    residual_usd
)

# ============================================================
# 12. YÜZDESEL HATA
# ============================================================

epsilon = 1e-9

absolute_percentage_error = (
    np.abs(
        (y_test_usd - pred_usd)
        / np.maximum(
            np.abs(y_test_usd),
            epsilon
        )
    ) * 100
)

# ============================================================
# 13. ERROR DIRECTION
# ============================================================

error_direction = np.where(
    residual_usd > 0,
    "Underprediction",
    "Overprediction"
)

error_direction = np.where(
    np.isclose(
        residual_usd,
        0,
        atol=1e-8
    ),
    "Accurate",
    error_direction
)

# ============================================================
# 14. ERROR DATAFRAME
# ============================================================

error_df = pd.DataFrame({

    "actual_usd":
        y_test_usd,

    "predicted_usd":
        pred_usd,

    "residual_usd":
        residual_usd,

    "absolute_error_usd":
        absolute_error_usd,

    "absolute_percentage_error":
        absolute_percentage_error,

    "actual_log":
        y_test_array,

    "predicted_log":
        pred_log,

    "residual_log":
        residual_log,

    "error_direction":
        error_direction
})

# ============================================================
# 15. TEST METRİKLERİ
# ============================================================

rmse_usd = np.sqrt(
    mean_squared_error(
        y_test_usd,
        pred_usd
    )
)

mae_usd = mean_absolute_error(
    y_test_usd,
    pred_usd
)

r2_usd = r2_score(
    y_test_usd,
    pred_usd
)

rmse_log = np.sqrt(
    mean_squared_error(
        y_test_array,
        pred_log
    )
)

mae_log = mean_absolute_error(
    y_test_array,
    pred_log
)

median_absolute_error = np.median(
    absolute_error_usd
)

mean_absolute_percentage_error = np.mean(
    absolute_percentage_error
)

# ============================================================
# 16. UNDER / OVERPREDICTION
# ============================================================

under_count = np.sum(
    residual_usd > 0
)

over_count = np.sum(
    residual_usd < 0
)

accurate_count = np.sum(
    np.isclose(
        residual_usd,
        0,
        atol=1e-8
    )
)

total_count = len(
    residual_usd
)

under_ratio = (
    under_count
    / total_count
    * 100
)

over_ratio = (
    over_count
    / total_count
    * 100
)

accurate_ratio = (
    accurate_count
    / total_count
    * 100
)

# ============================================================
# 17. SONUÇLAR
# ============================================================

print("\n" + "=" * 70)
print("XGBOOST RESIDUAL ANALYSIS")
print("=" * 70)

print(
    f"Test Samples              : "
    f"{total_count:,}"
)

print("\n--- USD SCALE ---")

print(
    f"RMSE                      : "
    f"${rmse_usd:,.2f}"
)

print(
    f"MAE                       : "
    f"${mae_usd:,.2f}"
)

print(
    f"R²                        : "
    f"{r2_usd:.6f}"
)

print(
    f"Median Absolute Error     : "
    f"${median_absolute_error:,.2f}"
)

print(
    f"Mean Absolute % Error     : "
    f"{mean_absolute_percentage_error:.2f}%"
)

print("\n--- LOG SCALE ---")

print(
    f"Log RMSE                  : "
    f"{rmse_log:.6f}"
)

print(
    f"Log MAE                   : "
    f"{mae_log:.6f}"
)

print("\n--- ERROR DIRECTION ---")

print(
    f"Underprediction           : "
    f"{under_count:,} "
    f"({under_ratio:.2f}%)"
)

print(
    f"Overprediction            : "
    f"{over_count:,} "
    f"({over_ratio:.2f}%)"
)

print(
    f"Accurate                  : "
    f"{accurate_count:,} "
    f"({accurate_ratio:.2f}%)"
)

print("=" * 70)

# ============================================================
# 18. WORST PREDICTIONS
# ============================================================

worst_predictions = (
    error_df
    .sort_values(
        "absolute_error_usd",
        ascending=False
    )
    .head(100)
    .copy()
)

worst_underpredictions = (
    error_df[
        error_df["residual_usd"] > 0
    ]
    .sort_values(
        "residual_usd",
        ascending=False
    )
    .head(100)
    .copy()
)

worst_overpredictions = (
    error_df[
        error_df["residual_usd"] < 0
    ]
    .sort_values(
        "residual_usd",
        ascending=True
    )
    .head(100)
    .copy()
)

# ============================================================
# 19. FEATURE VERİSİNİ ERROR DATAFRAME'E EKLE
# ============================================================

available_numeric_features = []
available_categorical_features = []

# X_test DataFrame ise kolon isimlerini kullan
if isinstance(X_test, pd.DataFrame):

    available_numeric_features = [
        f
        for f in RESIDUAL_NUMERIC_FEATURES
        if f in X_test.columns
    ]

    available_categorical_features = [
        f
        for f in RESIDUAL_CATEGORICAL_FEATURES
        if f in X_test.columns
    ]

    for feature in (
        available_numeric_features
        + available_categorical_features
    ):

        error_df[feature] = (
            X_test[feature]
            .values
        )

else:

    print(
        "\nUYARI:"
        "\nX_test pandas DataFrame değil."
        "\nBu nedenle orijinal feature isimleri "
        "residual dataframe'e eklenemedi."
    )

# ============================================================
# 20. W&B
# ============================================================

wandb.login()

run = wandb.init(
    project=PROJECT_NAME,
    entity=ENTITY_NAME,
    name=RUN_NAME,
    group="XGBoost_Error_Analysis",
    job_type="evaluation",
    config={
        "model": "XGBoost",
        "evaluation_stage":
            "TEST_ERROR_ANALYSIS",
        "target":
            TARGET_NAME,
        "log_target":
            LOG_TARGET_NAME,
        "target_is_log":
            TARGET_IS_LOG,
        "seed":
            SEED,
        "final_model_path":
            FINAL_MODEL_PATH,
        "numeric_residual_features":
            RESIDUAL_NUMERIC_FEATURES,
        "categorical_residual_features":
            RESIDUAL_CATEGORICAL_FEATURES
    }
)

# ============================================================
# 21. W&B METRİKLERİ
# ============================================================

wandb.log({

    "test/rmse_usd":
        rmse_usd,

    "test/mae_usd":
        mae_usd,

    "test/r2_usd":
        r2_usd,

    "test/median_absolute_error_usd":
        median_absolute_error,

    "test/mape":
        mean_absolute_percentage_error,

    "test/rmse_log":
        rmse_log,

    "test/mae_log":
        mae_log,

    "error/underprediction_count":
        int(under_count),

    "error/overprediction_count":
        int(over_count),

    "error/underprediction_ratio":
        under_ratio,

    "error/overprediction_ratio":
        over_ratio,

    "error/accurate_ratio":
        accurate_ratio
})

# ============================================================
# 22. RESIDUAL VS FITTED
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.scatter(
    pred_usd,
    residual_usd,
    alpha=0.35,
    s=12
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=2
)

plt.xlabel(
    "Fitted Values / Predicted Price (USD)"
)

plt.ylabel(
    "Residual (Actual - Predicted)"
)

plt.title(
    "XGBoost Residuals vs Fitted Values"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

wandb.log({
    "residual_analysis/residual_vs_fitted":
        wandb.Image(plt)
})

plt.show()
plt.close()

# ============================================================
# 23. RESIDUAL HISTOGRAM
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.hist(
    residual_usd,
    bins=60,
    edgecolor="black",
    alpha=0.8
)

plt.axvline(
    x=0,
    linestyle="--",
    linewidth=2
)

plt.xlabel(
    "Residual (USD)"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of XGBoost Residuals"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

wandb.log({
    "residual_analysis/residual_distribution":
        wandb.Image(plt)
})

plt.show()
plt.close()

# ============================================================
# 24. RESIDUAL VS OBSERVATION ORDER
# ============================================================

plt.figure(
    figsize=(12, 6)
)

plt.scatter(
    np.arange(
        len(residual_usd)
    ),
    residual_usd,
    alpha=0.35,
    s=12
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=2
)

plt.xlabel(
    "Observation Order"
)

plt.ylabel(
    "Residual (USD)"
)

plt.title(
    "XGBoost Residuals vs Observation Order"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

wandb.log({
    "residual_analysis/residual_vs_observation_order":
        wandb.Image(plt)
})

plt.show()
plt.close()

# ============================================================
# 25. SAYISAL FEATURE → RESIDUAL
# ============================================================

feature_residual_stats = []

for feature in available_numeric_features:

    feature_values = pd.to_numeric(
        error_df[feature],
        errors="coerce"
    )

    mask = (
        feature_values.notna()
        & np.isfinite(
            feature_values
        )
        & np.isfinite(
            residual_usd
        )
    )

    x = feature_values[mask]

    y = pd.Series(
        residual_usd,
        index=error_df.index
    )[mask]

    if len(x) < 10:
        continue

    correlation = x.corr(y)

    feature_residual_stats.append({

        "feature":
            feature,

        "residual_correlation":
            correlation,

        "mean_residual":
            y.mean(),

        "median_residual":
            y.median(),

        "mean_absolute_error":
            np.abs(y).mean(),

        "rmse":
            np.sqrt(
                np.mean(
                    y ** 2
                )
            )
    })

    # --------------------------------------------------------
    # Residual Plot
    # --------------------------------------------------------

    plt.figure(
        figsize=(9, 6)
    )

    plt.scatter(
        x,
        y,
        alpha=0.30,
        s=12
    )

    plt.axhline(
        y=0,
        linestyle="--",
        linewidth=2
    )

    plt.xlabel(
        feature
    )

    plt.ylabel(
        "Residual (USD)"
    )

    plt.title(
        f"Residual vs {feature}"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    wandb.log({
        f"residual_analysis/numeric/{feature}":
            wandb.Image(plt)
    })

    plt.show()
    plt.close()

# ============================================================
# 26. FEATURE RESIDUAL İSTATİSTİKLERİ
# ============================================================

feature_residual_stats_df = pd.DataFrame(
    feature_residual_stats
)

if not feature_residual_stats_df.empty:

    feature_residual_stats_df = (
        feature_residual_stats_df
        .sort_values(
            "mean_absolute_error",
            ascending=False
        )
    )

    print(
        "\n" + "=" * 70
    )

    print(
        "FEATURE → RESIDUAL ANALYSIS"
    )

    print(
        "=" * 70
    )

    print(
        feature_residual_stats_df.to_string(
            index=False
        )
    )

# ============================================================
# 27. KATEGORİK FEATURE → RESIDUAL
# ============================================================

categorical_stats = []

for feature in available_categorical_features:

    temp = error_df[
        [
            feature,
            "residual_usd"
        ]
    ].copy()

    temp[feature] = (
        temp[feature]
        .astype(str)
    )

    top_categories = (
        temp[feature]
        .value_counts()
        .head(15)
        .index
    )

    temp = temp[
        temp[feature]
        .isin(top_categories)
    ]

    if temp.empty:
        continue

    categories = list(
        temp[feature]
        .value_counts()
        .index
    )

    data = [

        temp.loc[
            temp[feature] == category,
            "residual_usd"
        ].values

        for category in categories
    ]

    # --------------------------------------------------------
    # BOXPLOT
    # --------------------------------------------------------

    plt.figure(
        figsize=(
            max(
                10,
                len(categories) * 0.8
            ),
            6
        )
    )

    plt.boxplot(
        data,
        labels=categories,
        showfliers=False
    )

    plt.axhline(
        y=0,
        linestyle="--",
        linewidth=2
    )

    plt.xlabel(
        feature
    )

    plt.ylabel(
        "Residual (USD)"
    )

    plt.title(
        f"Residual Distribution by {feature}"
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()

    wandb.log({
        f"residual_analysis/categorical/{feature}":
            wandb.Image(plt)
    })

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # GRUP İSTATİSTİKLERİ
    # --------------------------------------------------------

    grouped = (
        temp
        .groupby(feature)["residual_usd"]
        .agg(
            count="count",
            mean_residual="mean",
            median_residual="median",
            mean_absolute_error=
                lambda x: np.abs(x).mean(),
            rmse=
                lambda x:
                np.sqrt(
                    np.mean(
                        x ** 2
                    )
                )
        )
        .reset_index()
    )

    grouped["feature"] = feature

    categorical_stats.append(
        grouped
    )

# ============================================================
# 28. KATEGORİK RESIDUAL İSTATİSTİKLERİ
# ============================================================

if categorical_stats:

    categorical_residual_stats_df = pd.concat(
        categorical_stats,
        ignore_index=True
    )

    print(
        "\n" + "=" * 70
    )

    print(
        "CATEGORICAL FEATURE → RESIDUAL ANALYSIS"
    )

    print(
        "=" * 70
    )

    print(
        categorical_residual_stats_df.to_string(
            index=False
        )
    )

# ============================================================
# 29. W&B TABLES
# ============================================================

wandb.log({

    "error_analysis/top_100_worst_predictions":
        wandb.Table(
            dataframe=worst_predictions
        ),

    "error_analysis/top_100_underpredictions":
        wandb.Table(
            dataframe=worst_underpredictions
        ),

    "error_analysis/top_100_overpredictions":
        wandb.Table(
            dataframe=worst_overpredictions
        )
})

# ============================================================
# 30. FULL ERROR TABLE
# ============================================================

wandb.log({

    "error_analysis/test_error_table":
        wandb.Table(
            dataframe=error_df.head(10000)
        )
})

# ============================================================
# 31. RESIDUAL SUMMARY TABLE
# ============================================================

residual_summary = pd.DataFrame({

    "metric": [

        "Test Samples",
        "RMSE USD",
        "MAE USD",
        "R2 USD",
        "Median Absolute Error",
        "MAPE",
        "Log RMSE",
        "Log MAE",
        "Underprediction %",
        "Overprediction %",
        "Accurate %"
    ],

    "value": [

        total_count,
        rmse_usd,
        mae_usd,
        r2_usd,
        median_absolute_error,
        mean_absolute_percentage_error,
        rmse_log,
        mae_log,
        under_ratio,
        over_ratio,
        accurate_ratio
    ]
})

wandb.log({

    "error_analysis/residual_summary":
        wandb.Table(
            dataframe=residual_summary
        )
})

# ============================================================
# 32. CSV DOSYALARINI KAYDET
# ============================================================

OUTPUT_DIR = (
    "xgboost_error_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

error_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "test_error_analysis.csv"
    ),
    index=False
)

worst_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "top_100_worst_predictions.csv"
    ),
    index=False
)

worst_underpredictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "top_100_underpredictions.csv"
    ),
    index=False
)

worst_overpredictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "top_100_overpredictions.csv"
    ),
    index=False
)

if not feature_residual_stats_df.empty:

    feature_residual_stats_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "feature_residual_statistics.csv"
        ),
        index=False
    )

if categorical_stats:

    categorical_residual_stats_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "categorical_residual_statistics.csv"
        ),
        index=False
    )

# ============================================================
# 33. W&B DOSYALARI
# ============================================================

for filename in os.listdir(
    OUTPUT_DIR
):

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    wandb.save(
        filepath
    )

# ============================================================
# 34. FINAL SUMMARY
# ============================================================

wandb.summary.update({

    "test_rmse_usd":
        rmse_usd,

    "test_mae_usd":
        mae_usd,

    "test_r2_usd":
        r2_usd,

    "test_mape":
        mean_absolute_percentage_error,

    "underprediction_ratio":
        under_ratio,

    "overprediction_ratio":
        over_ratio,

    "worst_absolute_error_usd":
        float(
            worst_predictions[
                "absolute_error_usd"
            ].iloc[0]
        )
})

run.finish()

print(
    "\n" + "=" * 70
)

print(
    "RESIDUAL ANALYSIS TAMAMLANDI"
)

print(
    "=" * 70
)

print(
    f"Model        : {FINAL_MODEL_PATH}"
)

print(
    f"W&B Run      : {RUN_NAME}"
)

print(
    f"Output Folder: {OUTPUT_DIR}"
)

print(
    "=" * 70
)

## TENDER_SUPPLYTYPE (İhalenin tedarik/temin türünü ifade eden kategorik bir değişkendir. İhalenin hangi tür mal veya hizmet tedarikine yönelik olduğunu belirtir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "tender_supplytype"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
A4_YLIM = (-3, 3)

if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

possible_supply_columns = [
    "tender_supplytype",
    "supply_type",
    "supplytype",
    "tender_supply_type"
]

supply_col = next(
    (c for c in possible_supply_columns if c in df_test.columns),
    None
)

if supply_col is None:
    raise KeyError(
        f"Supply Type kolonu bulunamadı.\n"
        f"Aranan kolonlar: {possible_supply_columns}\n"
        f"Mevcut kolonlar:\n{list(df_test.columns)}"
    )

print(f"Supply Type kolonu: {supply_col}")
print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {
    c.lower(): c for c in test_predictions.columns
}

actual_candidates = [
    "actual_usd",
    "actual",
    "y_true",
    "true",
    "target",
    "actual_price",
    "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd",
    "predicted",
    "prediction",
    "pred",
    "y_pred",
    "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd",
    "residual",
    "error",
    "prediction_error"
]

actual_col = next(
    (
        columns_lower[c]
        for c in actual_candidates
        if c in columns_lower
    ),
    None
)

predicted_col = next(
    (
        columns_lower[c]
        for c in predicted_candidates
        if c in columns_lower
    ),
    None
)

residual_col = next(
    (
        columns_lower[c]
        for c in residual_candidates
        if c in columns_lower
    ),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon     : {actual_col}")
print(f"Prediction kolon : {predicted_col}")
print(f"Residual kolon   : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test: {len(df_test):,}\n"
        f"test_predictions: {len(test_predictions):,}"
    )

supply_df = pd.DataFrame({
    "supply_type": df_test[supply_col].to_numpy(),
    "actual": pd.to_numeric(
        test_predictions[actual_col],
        errors="coerce"
    ).to_numpy(),
    "prediction": pd.to_numeric(
        test_predictions[predicted_col],
        errors="coerce"
    ).to_numpy(),
    "residual": pd.to_numeric(
        test_predictions[residual_col],
        errors="coerce"
    ).to_numpy()
})

def normalize_supply_type(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value).strip().upper()

    if value in [
        "GOODS",
        "GOOD",
        "SUPPLIES",
        "SUPPLY",
        "MATERIAL",
        "MATERIALS"
    ]:
        return "SUPPLIES"

    if value in [
        "SERVICE",
        "SERVICES"
    ]:
        return "SERVICES"

    if value in [
        "WORK",
        "WORKS",
        "CONSTRUCTION",
        "CONSTRUCTIONS"
    ]:
        return "WORKS"

    if value in [
        "NONE",
        "NAN",
        "NULL",
        "",
        "MISSING"
    ]:
        return "MISSING"

    return value

supply_df["supply_type"] = (
    supply_df["supply_type"]
    .apply(normalize_supply_type)
)

supply_df["absolute_error"] = (
    supply_df["residual"].abs()
)

valid_mask = (
    np.isfinite(supply_df["actual"].to_numpy())
    & np.isfinite(supply_df["prediction"].to_numpy())
    & np.isfinite(supply_df["residual"].to_numpy())
)

supply_df = (
    supply_df
    .loc[valid_mask]
    .reset_index(drop=True)
)

print(f"\nGeçerli gözlem sayısı: {len(supply_df):,}")

category_stats = (
    supply_df
    .groupby("supply_type")
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        std_residual=("residual", "std"),
        MAE=("absolute_error", "mean"),
        MSE=("residual", lambda x: np.mean(x ** 2)),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

effect_stats = (
    category_stats[
        category_stats["count"] >= MIN_CATEGORY_COUNT
    ]
    .copy()
    .reset_index(drop=True)
)

global_std = supply_df["residual"].std()

if global_std == 0 or not np.isfinite(global_std):
    global_std = 1.0

effect_stats["standardized_effect"] = (
    effect_stats["mean_residual"] / global_std
)

effect_stats["absolute_standardized_effect"] = (
    effect_stats["standardized_effect"].abs()
)

preferred_order = [
    "SUPPLIES",
    "SERVICES",
    "WORKS",
    "MISSING"
]

existing_categories = (
    effect_stats["supply_type"]
    .astype(str)
    .unique()
    .tolist()
)

final_order = [
    c for c in preferred_order
    if c in existing_categories
]

final_order += [
    c for c in existing_categories
    if c not in final_order
]

effect_stats["supply_type"] = pd.Categorical(
    effect_stats["supply_type"].astype(str),
    categories=final_order,
    ordered=True
)

effect_stats = (
    effect_stats
    .sort_values("supply_type")
    .reset_index(drop=True)
)

visual_table = effect_stats[
    [
        "supply_type",
        "count",
        "mean_residual",
        "MAE",
        "MSE",
        "RMSE",
        "standardized_effect"
    ]
].copy()

visual_table.columns = [
    "Supply Type",
    "Observations",
    "Mean Residual",
    "MAE",
    "MSE",
    "RMSE",
    "Standardized Effect"
]

def highlight_max(series):
    mask = series == series.max()

    return [
        "background-color: #ffd6d6; color: #8b0000; font-weight: bold;"
        if value else ""
        for value in mask
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "${:,.0f}",
        "MAE": "${:,.0f}",
        "MSE": "${:,.0f}",
        "RMSE": "${:,.0f}",
        "Standardized Effect": "{:+.3f}"
    })
    .apply(
        highlight_max,
        subset=[
            "MAE",
            "MSE",
            "RMSE",
            "Mean Residual",
            "Standardized Effect"
        ]
    )
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "12pt",
            "padding": "10px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#404040"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("padding", "10px")
            ]
        }
    ])
    .set_caption(
        "TENDER SUPPLY TYPE — RESIDUAL ANALYSIS"
    )
)

display(styled_table)

residual_values = (
    supply_df["residual"].to_numpy()
)

prediction_values = (
    supply_df["prediction"].to_numpy()
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(20, 15)
)

ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — Normal Probability Plot
if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(42)

    residual_plot = rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
else:
    residual_plot = residual_values

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - 0.5) / n
)

residual_mean = np.mean(sorted_residuals)

residual_std = np.std(
    sorted_residuals,
    ddof=1
)

residual_std = (
    1
    if residual_std == 0
    else residual_std
)

standardized_residuals = (
    sorted_residuals - residual_mean
) / residual_std

lower_percentile, upper_percentile = np.percentile(
    standardized_residuals,
    [0.5, 99.5]
)

a1_mask = (
    (standardized_residuals >= lower_percentile)
    & (standardized_residuals <= upper_percentile)
)

x_a1 = theoretical_quantiles[a1_mask]
y_a1 = standardized_residuals[a1_mask]

ax1.scatter(
    x_a1,
    y_a1,
    s=14,
    alpha=0.22,
    rasterized=True
)

x_line = np.array([
    x_a1.min(),
    x_a1.max()
])

ax1.plot(
    x_line,
    x_line,
    linestyle="--",
    linewidth=2.5,
    label="Normal reference"
)

# A1 Y ekseni tamamen lineer
ax1.set_yscale("linear")

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5,
    color="black",
    alpha=0.8
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual",
    fontsize=13
)

ax1.set_title(
    "A1 — Normal Probability Plot\n"
    "Central 99% of Standardized Residuals",
    fontsize=16,
    fontweight="bold"
)

ax1.tick_params(
    axis="both",
    labelsize=11
)

ax1.grid(
    alpha=0.25,
    linestyle="--"
)

ax1.text(
    0.03,
    0.03,
    "Central 99% shown\n"
    "Extreme residuals excluded",
    transform=ax1.transAxes,
    fontsize=12,
    fontweight="bold",
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="lightgray",
        alpha=0.90
    )
)

ax1.legend(
    loc="upper left",
    fontsize=10
)

# A2 — Residuals vs Fits
prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(
    prediction_m,
    [0.5, 99.5]
)

y_low, y_high = np.percentile(
    residual_m,
    [0.5, 99.5]
)

plot_mask = (
    (prediction_m >= x_low)
    & (prediction_m <= x_high)
    & (residual_m >= y_low)
    & (residual_m <= y_high)
)

x_plot = prediction_m[plot_mask]
y_plot = residual_m[plot_mask]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=45,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=0.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual = np.median(y_plot)

ax2.axhline(
    median_residual,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=0.02
)

cbar.set_label(
    "Observation Density\n(log scale)",
    fontsize=13
)

cbar.ax.tick_params(
    labelsize=12
)

ax2.set_xlabel(
    "Predicted Price (Million USD)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Residuals Versus Fits\n"
    "Density-Based Visualization",
    fontsize=16,
    fontweight="bold"
)

ax2.tick_params(
    axis="both",
    labelsize=11
)

ax2.grid(
    alpha=0.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)

# A3 — Residual Histogram
residual_low, residual_high = np.percentile(
    residual_values,
    [0.5, 99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

central_m = central_residuals / 1e6

ax3.hist(
    central_m,
    bins=35,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.5
)

mean_residual = np.mean(
    residual_values
) / 1e6

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax3.axvline(
    mean_residual,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Mean = {mean_residual:.2f}M"
)

hist_low, hist_high = np.percentile(
    central_m,
    [0.5, 99.5]
)

padding = max(
    0.05 * (hist_high - hist_low),
    0.5
)

ax3.set_xlim(
    hist_low - padding,
    hist_high + padding
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Residual Distribution\n"
    "Central 99% of Observations",
    fontsize=16,
    fontweight="bold"
)

ax3.tick_params(
    axis="both",
    labelsize=11
)

ax3.legend(
    fontsize=10
)

ax3.grid(
    axis="y",
    alpha=0.20
)

# A4 — Supply Type Boxplot
box_categories = [
    c for c in final_order
    if c in effect_stats["supply_type"].astype(str).tolist()
]

box_data = []
valid_box_categories = []

for category in box_categories:
    values = supply_df.loc[
        supply_df["supply_type"].astype(str) == category,
        "residual"
    ].to_numpy() / 1e6

    if len(values) >= MIN_CATEGORY_COUNT:
        box_data.append(values)
        valid_box_categories.append(category)

bp = ax4.boxplot(
    box_data,
    labels=valid_box_categories,
    patch_artist=True,
    showmeans=True,
    showfliers=False,
    meanprops={
        "marker": "D",
        "markerfacecolor": "black",
        "markeredgecolor": "black",
        "markersize": 7
    },
    medianprops={
        "color": "darkred",
        "linewidth": 2.8
    },
    boxprops={
        "linewidth": 1.5
    },
    whiskerprops={
        "linewidth": 1.5
    },
    capprops={
        "linewidth": 1.5
    }
)

category_colors = {
    "SUPPLIES": "#5B8FF9",
    "SERVICES": "#61DDAA",
    "WORKS": "#F6BD16",
    "MISSING": "#999999"
}

for patch, category in zip(
    bp["boxes"],
    valid_box_categories
):
    patch.set_facecolor(
        category_colors.get(
            category,
            "#7A7A7A"
        )
    )

    patch.set_alpha(0.80)
    patch.set_edgecolor("black")
    patch.set_linewidth(1.4)

for median in bp["medians"]:
    median.set_color("#8B0000")
    median.set_linewidth(3)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2.2,
    color="red",
    label="Residual = 0",
    zorder=0
)

ax4.plot(
    [],
    [],
    marker="D",
    linestyle="None",
    markerfacecolor="black",
    markeredgecolor="black",
    markersize=7,
    label="◆ Mean"
)

ax4.set_ylim(A4_YLIM)

ax4.set_ylabel(
    "Residual (Million USD)",
    fontsize=14,
    labelpad=10
)

ax4.set_xlabel(
    "Supply Type",
    fontsize=14,
    labelpad=10
)

ax4.tick_params(
    axis="x",
    labelsize=13
)

ax4.tick_params(
    axis="y",
    labelsize=12
)

for label in ax4.get_xticklabels():
    label.set_rotation(0)
    label.set_fontweight("bold")

ax4.set_title(
    "A4 — Residuals by Supply Type\n"
    "Central Residual Range (−3M to +3M USD)",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(
    axis="y",
    alpha=0.22,
    linestyle="--"
)

if len(effect_stats) > 0:
    most_biased = effect_stats.loc[
        effect_stats["absolute_standardized_effect"].idxmax()
    ]

    biased_type = str(
        most_biased["supply_type"]
    )

    biased_effect = (
        most_biased["standardized_effect"]
    )

    biased_mean = (
        most_biased["mean_residual"] / 1e6
    )

    biased_mae = (
        most_biased["MAE"] / 1e6
    )

    direction = (
        "UNDER-PREDICTION"
        if biased_effect > 0
        else "OVER-PREDICTION"
    )

    info_text = (
        f"Most biased type: {biased_type}\n"
        f"Mean residual: ${biased_mean:+.2f}M\n"
        f"MAE: ${biased_mae:.2f}M\n"
        f"Std. effect: {biased_effect:+.3f}\n"
        f"Direction: {direction}"
    )

    ax4.text(
        0.03,
        0.97,
        info_text,
        transform=ax4.transAxes,
        fontsize=11,
        fontweight="bold",
        linespacing=1.45,
        verticalalignment="top",
        bbox=dict(
            boxstyle="round,pad=0.55",
            facecolor="white",
            edgecolor="lightgray",
            alpha=0.94
        )
    )

ax4.legend(
    loc="upper right",
    fontsize=10,
    frameon=True,
    framealpha=0.92
)

fig.suptitle(
    "TENDER SUPPLY TYPE — RESIDUAL DIAGNOSTICS",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

plt.show()
plt.close()

# B — Standardized Supply Type Effects
effects = effect_stats[
    "standardized_effect"
].to_numpy()

effects_sorted = np.sort(effects)
m = len(effects_sorted)

if m >= 2:
    theoretical_effect_quantiles = stats.norm.ppf(
        (np.arange(1, m + 1) - 0.5) / m
    )

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=130,
        alpha=0.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope * x_ref,
        linestyle="--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():
        idx = np.argmin(
            np.abs(
                effects_sorted
                - row["standardized_effect"]
            )
        )

        ax.annotate(
            str(row["supply_type"]),
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(10, 8),
            textcoords="offset points",
            fontsize=13,
            fontweight="bold"
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=1.8
    )

    ax.set_xlabel(
        "Theoretical Normal Quantiles",
        fontsize=13
    )

    ax.set_ylabel(
        "Standardized Supply-Type Effect",
        fontsize=13
    )

    ax.set_title(
        "B — Normal Plot of Standardized\n"
        "Supply-Type Effects",
        fontsize=17,
        fontweight="bold"
    )

    ax.tick_params(
        axis="both",
        labelsize=11
    )

    ax.grid(
        alpha=0.20
    )

    plt.tight_layout()
    plt.show()
    plt.close()

# Component + Residual Plots
categories = (
    effect_stats["supply_type"]
    .astype(str)
    .tolist()
)

if len(categories) > 0:
    fig, axes = plt.subplots(
        1,
        len(categories),
        figsize=(22, 8),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, category in zip(
        axes,
        categories
    ):
        category_data = supply_df[
            supply_df["supply_type"].astype(str)
            == category
        ].copy()

        x = (
            category_data["prediction"].to_numpy()
            / 1e6
        )

        y = (
            category_data["residual"].to_numpy()
            / 1e6
        )

        if len(x) < MIN_CATEGORY_COUNT:
            continue

        x_low, x_high = np.percentile(
            x,
            [1, 99]
        )

        y_low, y_high = np.percentile(
            y,
            [1, 99]
        )

        mask = (
            (x >= x_low)
            & (x <= x_high)
            & (y >= y_low)
            & (y <= y_high)
        )

        x_plot = x[mask]
        y_plot = y[mask]

        if len(x_plot) > 12000:
            rng = np.random.default_rng(42)

            sample_idx = rng.choice(
                len(x_plot),
                size=12000,
                replace=False
            )

            x_scatter = x_plot[sample_idx]
            y_scatter = y_plot[sample_idx]
        else:
            x_scatter = x_plot
            y_scatter = y_plot

        ax.scatter(
            x_scatter,
            y_scatter,
            s=12,
            alpha=0.15,
            rasterized=True
        )

        if len(x_plot) >= 3:
            slope, intercept = np.polyfit(
                x_plot,
                y_plot,
                1
            )

            x_line = np.linspace(
                x_plot.min(),
                x_plot.max(),
                300
            )

            ax.plot(
                x_line,
                intercept + slope * x_line,
                linestyle="--",
                linewidth=2.5,
                label="Linear fit"
            )

            sort_idx = np.argsort(x_plot)

            x_sorted = x_plot[sort_idx]
            y_sorted = y_plot[sort_idx]

            component_df = pd.DataFrame({
                "x": x_sorted,
                "y": y_sorted
            })

            smooth_window = max(
                50,
                min(
                    500,
                    len(component_df) // 20
                )
            )

            rolling_component = (
                component_df["y"]
                .rolling(
                    smooth_window,
                    center=True,
                    min_periods=max(
                        20,
                        smooth_window // 4
                    )
                )
                .median()
            )

            ax.plot(
                x_sorted,
                rolling_component,
                linewidth=3,
                label="Smoothed trend"
            )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=2.2,
            color="red",
            label="Residual = 0"
        )

        ax.set_xlabel(
            "Predicted Price (Million USD)",
            fontsize=14
        )

        ax.set_ylabel(
            "Residual (Million USD)",
            fontsize=14
        )

        ax.tick_params(
            axis="both",
            labelsize=12
        )

        ax.set_title(
            str(category),
            fontsize=19,
            fontweight="bold",
            pad=15
        )

        ax.grid(
            alpha=0.22,
            linestyle="--"
        )

        row = effect_stats[
            effect_stats["supply_type"].astype(str)
            == category
        ].iloc[0]

        mean_res = (
            row["mean_residual"] / 1e6
        )

        mae = row["MAE"] / 1e6
        rmse = row["RMSE"] / 1e6
        effect = row["standardized_effect"]

        direction = (
            "UNDER-PREDICTION"
            if effect > 0
            else "OVER-PREDICTION"
        )

        metrics_text = (
            f"Observations : {int(row['count']):,}\n"
            f"Mean Residual: ${mean_res:+.2f}M\n"
            f"MAE          : ${mae:.2f}M\n"
            f"RMSE         : ${rmse:.2f}M\n"
            f"Std. Effect  : {effect:+.3f}\n"
            f"Direction    : {direction}"
        )

        ax.text(
            0.03,
            0.97,
            metrics_text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=11,
            fontweight="bold",
            linespacing=1.5,
            bbox=dict(
                boxstyle="round,pad=0.6",
                facecolor="white",
                edgecolor="lightgray",
                alpha=0.92
            )
        )

        ax.legend(
            loc="lower right",
            fontsize=10,
            frameon=True,
            framealpha=0.90
        )

    fig.suptitle(
        "COMPONENT + RESIDUAL PLOTS\n"
        "Supply Type",
        fontsize=22,
        fontweight="bold",
        y=1.02
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.95]
    )

    plt.show()
    plt.close()

# MAE karşılaştırması
mae_plot = effect_stats.sort_values(
    "MAE",
    ascending=False
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

bars = ax.bar(
    mae_plot["supply_type"].astype(str),
    mae_plot["MAE"] / 1e6
)

ax.set_xlabel(
    "Supply Type",
    fontsize=13
)

ax.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax.set_title(
    "Supply Type — MAE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    labelsize=12
)

for bar, value in zip(
    bars,
    mae_plot["MAE"] / 1e6
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# MSE karşılaştırması
mse_plot = effect_stats.sort_values(
    "MSE",
    ascending=False
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

bars = ax.bar(
    mse_plot["supply_type"].astype(str),
    mse_plot["MSE"] / 1e12
)

ax.set_xlabel(
    "Supply Type",
    fontsize=13
)

ax.set_ylabel(
    "MSE (Trillion USD²)",
    fontsize=13
)

ax.set_title(
    "Supply Type — MSE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    labelsize=12
)

for bar, value in zip(
    bars,
    mse_plot["MSE"] / 1e12
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height()
        + max(
            mse_plot["MSE"].max() / 1e12 * 0.01,
            0.01
        ),
        f"{value:.2f}T",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# En problemli kategoriler
if len(effect_stats) > 0:
    worst_mae = effect_stats.loc[
        effect_stats["MAE"].idxmax()
    ]

    worst_mse = effect_stats.loc[
        effect_stats["MSE"].idxmax()
    ]

    worst_rmse = effect_stats.loc[
        effect_stats["RMSE"].idxmax()
    ]

    worst_effect = effect_stats.loc[
        effect_stats[
            "absolute_standardized_effect"
        ].idxmax()
    ]

    print(
        "\nSUPPLY TYPE — HATA ANALİZİ"
    )

    print("=" * 70)

    print(
        f"\nEn yüksek MAE   : "
        f"{worst_mae['supply_type']} → "
        f"${worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek MSE   : "
        f"{worst_mse['supply_type']} → "
        f"${worst_mse['MSE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : "
        f"{worst_rmse['supply_type']} → "
        f"${worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En güçlü etki   : "
        f"{worst_effect['supply_type']} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    if worst_effect["standardized_effect"] > 0:
        print(
            "\nYorum: Pozitif residual → "
            "Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )
    else:
        print(
            "\nYorum: Negatif residual → "
            "Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )

# Kategori dağılımı
print("\nSUPPLY TYPE DAĞILIMI")
print("=" * 70)

distribution = (
    supply_df["supply_type"]
    .value_counts()
    .rename_axis("Supply Type")
    .reset_index(name="Observations")
)

distribution["Percentage"] = (
    distribution["Observations"]
    / len(supply_df)
    * 100
)

print(
    distribution.to_string(
        index=False,
        formatters={
            "Observations": "{:,.0f}".format,
            "Percentage": "{:.2f}%".format
        }
    )
)

**Component + Residual Plots (Supplies / Services / Works)**

Kategorilere göre tahmin edilen fiyat (Predicted Price) arttıkça artıkların (Residuals) varyansının dramatik şekilde genişlediği (Heteroscedasticity) görülmektedir. Turuncu trend çizgisinin sıfır referans hattının üzerinde seyretmesi, modelin tüm ihale tiplerinde sistematik olarak piyasa fiyatının altında tahmin ürettiğini (Under-prediction) doğrulamaktadır. SUPPLIES grubundaki uç değer yoğunluğu düşük kalırken, WORKS ve SERVICES kategorilerinde yüksek bütçeli ihalelere doğru hata sapması belirginleşmektedir.

**Residual Diagnostics Paneli (A1 - A4)A1 (Normal QQ Plot):** Standartlaştırılmış artıkların Q-Q eğrisinde sağa doğru sert bir yukarı sapma (Right-skewed / Heavy-tail) yapması, hataların normal dağılmadığını ve aşırı yüksek bütçeli ihalelerden kaynaklanan aşırı aykırı değerlerin mevcudiyetini göstermektedir.

**A2 (Residuals vs Fits - Yoğunluk):** Gözlemlerin ezici bir çoğunluğu (Hexbin yoğunluk haritasında sarı/yeşil bölge) 0–5 Milyon USD tahmin aralığında ve 0 çizgisi etrafında kümelenmiştir; ancak bütçe büyüdükçe veri noktaları 50M+ USD seviyelerine kadar seyrelerek saçılmaktadır.

**A3 (Residual Distribution):** Dağılımın medyanının 0'a çok yakın (-0.01M), fakat ortalamasının 1.60M seviyesinde olması, sağa çarpık yapının model ortalamasını sürekli yukarı çektiğini netleştirmektedir.

**A4 (Residuals by Supply Type):** WORKS en yüksek ortalama hata sapmasına (Mean Residual = +1.85M, Standardized Effect = +0.118) sahip olup modelin en çok zorlandığı kategoridir.

**Sütun Grafikleri (MAE, MSE ve Standardized Effects)**

**MAE Comparison:** Mutlak ortalama hatada WORKS (2.34M) en yüksek sapmayı gösterirken SUPPLIES 1.61M en kararlı gruptur.

**MSE Comparison:** SERVICES kategorisi (319.14T) en yüksek karesel hataya sahiptir; bu durum SERVICES ihale grubunda nadir görülen fakat son derece devasa bütçeli aşırı uç değerlerin (outliers) bulunduğunu kanıtlar.

**Normal Plot of Standardized Effects:** SUPPLIES (+0.080), SERVICES (+0.103) ve WORKS (+0.118) etki katsayıları teorik normallik ekseninde doğrusal bir hat üzerinde dizilerek hatanın ihale karmaşıklığıyla (Supplies Services  Works) orantılı olarak arttığını doğrular.

XGBoost modeli genel trendi başarıyla yakalamış olsa da, özellikle yüksek bütçeli ihale kal kalemlerinde belirgin bir altında tahmin (under-prediction) eğilimi ve heteroskedastisite sergilemektedir. Model başarısını artırmak için hedef değişkene logaritmik dönüşüm (log(y)) uygulanması ve karesel hataları temsil eden SERVICES/WORKS uç değerlerinin yalıtılması önerilir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "tender_supplytype"
N_BINS = 5

if "X_test" not in globals():
    raise NameError("X_test bulunamadı.")

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")

# X_test -> pandas
if hasattr(X_test, "select"):  # Polars
    Xf = X_test.select(FEATURE).to_pandas().reset_index(drop=True)
else:  # Pandas
    Xf = X_test[[FEATURE]].copy().reset_index(drop=True)

# ------------------------------------------------------------
# TEST PREDICTIONS
# ------------------------------------------------------------

if hasattr(test_predictions, "to_pandas"):  # Polars
    pred_df = test_predictions.to_pandas().reset_index(drop=True)
elif isinstance(test_predictions, pd.DataFrame):
    pred_df = test_predictions.reset_index(drop=True).copy()
else:
    pred_df = None

if pred_df is not None:
    pred_candidates = [
        "prediction",
        "predicted",
        "y_pred",
        "pred",
        "prediction_usd",
        "predicted_usd",
        "predicted_price"
    ]

    actual_candidates = [
        "actual",
        "y_true",
        "target",
        "actual_usd",
        "true",
        "true_price",
        "tender_finalpriceUsd"
    ]

    pred_col = next(
        (c for c in pred_candidates if c in pred_df.columns),
        None
    )

    actual_col = next(
        (c for c in actual_candidates if c in pred_df.columns),
        None
    )

    if pred_col is None or actual_col is None:
        raise ValueError(
            "test_predictions içindeki prediction/actual kolonları "
            f"bulunamadı.\nMevcut kolonlar: {list(pred_df.columns)}"
        )

    y_pred = pd.to_numeric(
        pred_df[pred_col],
        errors="coerce"
    ).to_numpy()

    y_true = pd.to_numeric(
        pred_df[actual_col],
        errors="coerce"
    ).to_numpy()

else:
    if "y_test" not in globals():
        raise NameError(
            "test_predictions array ise gerçek değerler için "
            "y_test gerekli."
        )

    y_pred = np.asarray(test_predictions).reshape(-1)
    y_true = np.asarray(y_test).reshape(-1)

# ------------------------------------------------------------
# UZUNLUK KONTROLÜ
# ------------------------------------------------------------

if not (len(Xf) == len(y_true) == len(y_pred)):
    raise ValueError(
        f"Uzunluklar eşleşmiyor: "
        f"X_test={len(Xf)}, "
        f"y_true={len(y_true)}, "
        f"y_pred={len(y_pred)}"
    )

# ------------------------------------------------------------
# ANALİZ DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame({
    FEATURE: Xf[FEATURE].astype(str),
    "actual": y_true,
    "predicted": y_pred
})

df["actual"] = pd.to_numeric(df["actual"], errors="coerce")
df["predicted"] = pd.to_numeric(df["predicted"], errors="coerce")

df = df.dropna(
    subset=["actual", "predicted"]
).copy()

# Residual = Actual - Predicted
# Pozitif = UNDERPREDICTION
# Negatif = OVERPREDICTION
df["residual"] = df["actual"] - df["predicted"]
df["abs_error"] = df["residual"].abs()

# ------------------------------------------------------------
# PREDICTED PRICE QUANTILE
# ------------------------------------------------------------

edges = df["predicted"].quantile(
    np.linspace(0, 1, N_BINS + 1)
).to_numpy()

edges = np.unique(edges)

if len(edges) < 3:
    raise ValueError(
        "Predicted price yeterli çeşitlilik içermiyor."
    )

df["price_bin"] = pd.cut(
    df["predicted"],
    bins=edges,
    labels=False,
    include_lowest=True,
    duplicates="drop"
)

# ------------------------------------------------------------
# METRİKLER
# ------------------------------------------------------------

rows = []

for (supply, price_bin), g in df.groupby(
    [FEATURE, "price_bin"],
    observed=True
):
    r = g["residual"].to_numpy()

    rows.append({
        "Supply Type": supply,
        "Price Bin": int(price_bin) + 1,
        "N": len(g),
        "Predicted Mean ($M)": g["predicted"].mean() / 1e6,
        "Mean Residual ($M)": r.mean() / 1e6,
        "Median Residual ($M)": np.median(r) / 1e6,
        "MAE ($M)": np.mean(np.abs(r)) / 1e6,
        "RMSE ($M)": np.sqrt(np.mean(r ** 2)) / 1e6,
        "Residual P90 ($M)": np.percentile(r, 90) / 1e6,
        "Residual P99 ($M)": np.percentile(r, 99) / 1e6,
        "Underprediction %": np.mean(r > 0) * 100,
        "Overprediction %": np.mean(r < 0) * 100
    })

summary = pd.DataFrame(rows).sort_values(
    ["Supply Type", "Price Bin"]
).reset_index(drop=True)

# ------------------------------------------------------------
# SONUÇ TABLOSU
# ------------------------------------------------------------

print("\n" + "=" * 105)
print("TENDER SUPPLY TYPE × PREDICTED PRICE — RESIDUAL ANALYSIS")
print("=" * 105)

print(
    summary.to_string(
        index=False,
        formatters={
            "Predicted Mean ($M)": "{:.2f}".format,
            "Mean Residual ($M)": "{:+.2f}".format,
            "Median Residual ($M)": "{:+.2f}".format,
            "MAE ($M)": "{:.2f}".format,
            "RMSE ($M)": "{:.2f}".format,
            "Residual P90 ($M)": "{:+.2f}".format,
            "Residual P99 ($M)": "{:+.2f}".format,
            "Underprediction %": "{:.1f}".format,
            "Overprediction %": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# PIVOTLAR
# ------------------------------------------------------------

pivot_mean = summary.pivot(
    index="Supply Type",
    columns="Price Bin",
    values="Mean Residual ($M)"
)

pivot_mae = summary.pivot(
    index="Supply Type",
    columns="Price Bin",
    values="MAE ($M)"
)

pivot_under = summary.pivot(
    index="Supply Type",
    columns="Price Bin",
    values="Underprediction %"
)

print("\n" + "=" * 80)
print("MEAN RESIDUAL ($M)")
print("=" * 80)
print(pivot_mean.round(2))

print("\n" + "=" * 80)
print("MAE ($M)")
print("=" * 80)
print(pivot_mae.round(2))

print("\n" + "=" * 80)
print("UNDERPREDICTION (%)")
print("=" * 80)
print(pivot_under.round(1))

# ------------------------------------------------------------
# GRAFİK 1 — MEAN RESIDUAL
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for supply in pivot_mean.index:
    ax.plot(
        pivot_mean.columns,
        pivot_mean.loc[supply],
        marker="o",
        linewidth=2,
        label=str(supply)
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.set_title(
    "Supply Type × Predicted Price — Mean Residual",
    fontsize=15,
    fontweight="bold"
)
ax.set_xlabel("Predicted Price Quantile")
ax.set_ylabel("Mean Residual (Million USD)")
ax.set_xticks(range(1, len(pivot_mean.columns) + 1))
ax.legend(title="Supply Type")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# GRAFİK 2 — MAE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for supply in pivot_mae.index:
    ax.plot(
        pivot_mae.columns,
        pivot_mae.loc[supply],
        marker="o",
        linewidth=2,
        label=str(supply)
    )

ax.set_title(
    "Supply Type × Predicted Price — MAE",
    fontsize=15,
    fontweight="bold"
)
ax.set_xlabel("Predicted Price Quantile")
ax.set_ylabel("MAE (Million USD)")
ax.set_xticks(range(1, len(pivot_mae.columns) + 1))
ax.legend(title="Supply Type")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# GRAFİK 3 — UNDERPREDICTION %
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for supply in pivot_under.index:
    ax.plot(
        pivot_under.columns,
        pivot_under.loc[supply],
        marker="o",
        linewidth=2,
        label=str(supply)
    )

ax.axhline(
    50,
    linestyle="--",
    linewidth=1.5
)

ax.set_title(
    "Supply Type × Predicted Price — Underprediction Rate",
    fontsize=15,
    fontweight="bold"
)
ax.set_xlabel("Predicted Price Quantile")
ax.set_ylabel("Underprediction (%)")
ax.set_ylim(0, 100)
ax.set_xticks(range(1, len(pivot_under.columns) + 1))
ax.legend(title="Supply Type")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# EN PROBLEMLİ SEGMENTLER
# ------------------------------------------------------------

worst_bias = summary.loc[
    summary["Mean Residual ($M)"].abs()
    .sort_values(ascending=False)
    .index
].head(10)

print("\n" + "=" * 100)
print("EN YÜKSEK BIAS GÖSTEREN SEGMENTLER")
print("=" * 100)

print(
    worst_bias[
        [
            "Supply Type",
            "Price Bin",
            "N",
            "Mean Residual ($M)",
            "MAE ($M)",
            "RMSE ($M)",
            "Underprediction %"
        ]
    ].to_string(
        index=False,
        formatters={
            "Mean Residual ($M)": "{:+.2f}".format,
            "MAE ($M)": "{:.2f}".format,
            "RMSE ($M)": "{:.2f}".format,
            "Underprediction %": "{:.1f}".format
        }
    )
)

**Kodun Yaptığı İşlem**

Kod, modelin tahmin performansını ihale türüne (tender_supplytype) ve tahmin edilen fiyat dilimlerine (Predicted Price Quantile) bölerek duyarlılık ve artık (residual) analizi yapmaktadır. Tahmin edilen değerleri 5 eşit dilime (1. Dilim: En düşük bütçeli, 5. Dilim: En yüksek bütçeli) bölerek her segment için Mean Residual, MAE ve Underprediction % metriklerini hesaplar, özet tablolar çıkarır ve bunları çizdirir.

**Grafiklerin Analizi**

Supply Type × Predicted Price — Mean Residual: Düşük bütçeli ilk 4 dilimde ortalama artık 0'a yakın bir seyir izlerken (0.1M - 2.2M USD aralığı), 5. dilim olan en yüksek bütçeli ihalelerde sıçrama yapmaktadır. WORKS kategorisinde ortalama artı hatası ~5.4M USD'ye, SUPPLIES'da ~5.2M USD'ye ve SERVICES'ta ~4.4M USD'ye ulaşmaktadır.

Supply Type × Predicted Price — MAE: Mutlak hata miktarı (MAE) bütçe dilimi büyüdükçe sürekli bir artış trendindedir. WORKS grubu 1. dilimdeki ~0.6M USD hata ile başlarken, 5. dilimde ~7.2M USD ile en yüksek mutlak hataya sahip kategoridir. SUPPLIES ve SERVICES grupları da 5. dilimde sırasıyla ~6.1M USD ve ~5.0M USD seviyelerine çıkmaktadır.

Supply Type × Predicted Price — Underprediction Rate: İlk 4 dilimde modelin altında tahmin etme oranı %40-%50 bandında dengede dururken, 5. dilimde SUPPLIES %63, SERVICES %62 ve WORKS %56 seviyesine yükselmektedir. Bu da modelin yüksek bütçeli ihalelerde belirgin biçimde piyasa değerinin altında tahmin ürettiğini doğrular.

**SONUÇ**

İhale türü ve bütçe dilimleri kırılımında gerçekleştirilen artık analizi, XGBoost modelinin düşük ve orta ölçekli ihalelerde (1-4. kantiller) oldukça kararlı ve düşük sapmayla çalıştığını, ancak en yüksek bütçeli %20'lik dilimde (5. kantil) belirgin bir performans kaybı yaşadığını ortaya koymaktadır. Bütçe büyüdükçe mutlak hatanın (MAE) tüm ihale tiplerinde üssel bir artış gösterdiği ve özellikle WORKS (İnşaat/Yapım) kategorisinde 5. dilimde $7.2M USD seviyesine ulaştığı görülmektedir. Ortalama artık (Mean Residual) ve underprediction oranlarındaki eşzamanlı yükseliş, modelin yüksek tutarlı ihaleleri sistematik olarak gerçek değerinin altında tahmin ettiğini (under-prediction) doğrulamaktadır; nitekim 5. dilimdeki SUPPLIES ihalelerinin %63.3'ünde ve SERVICES ihalelerinin %61.8'inde model piyasa fiyatının altında kalmıştır. Bu durum, modelin yüksek varyanslı ve büyük ölçekli ihalelerin üst limit dinamiklerini tam olarak öğrenemediğini göstermekte olup, 5. dilimdeki ihaleler için ayrı bir modelleme veya segment bazlı ağırlıklandırma ihtiyacına işaret etmektedir.

Yüksek bütçeli ihalelerdeki (Q_5 dilimi) sistematik altında tahmin (under-prediction) ve yüksek hatayı (MAE) gidermek için eğitim sürecinde örneklem ağırlıklandırma (sample weighting) stratejisi uygulanacaktır. XGBoost modelinin veri sayısı çok olan düşük bütçeli ihalelere odaklanmasını engellemek amacıyla, hedef değişkenin (y_train) en üst %20'lik dilimindeki (Q_5) gözlemlere fit adımı üzerinden 2.0-3.0 kat daha yüksek eğitim ağırlığı (sample_weight) verilecektir. Bu sayede karar ağaçları büyük ihalelerdeki sapmaları daha ağır cezalandırarak dallanacak ve test setinin yalın yapısı bozulmadan modelin üst limitlerdeki tahmin başarısı doğrudan artırılacaktır.

## COMPETITION_LEVEL (İhaledeki rekabet düzeyini (düşük, orta, yüksek vb.) ifade eden kategorik değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "competition_level"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
ROLLING_WINDOW_MAX = 2000

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır.")
if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")
if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {col.lower(): col for col in test_predictions.columns}

actual_candidates = [
    "actual_usd","actual","y_true","true","target",
    "actual_price","actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd","predicted","prediction","pred",
    "y_pred","predicted_price","predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd","residual","error","prediction_error"
]

actual_col = next(
    (columns_lower[c] for c in actual_candidates if c in columns_lower),
    None
)

predicted_col = next(
    (columns_lower[c] for c in predicted_candidates if c in columns_lower),
    None
)

residual_col = next(
    (columns_lower[c] for c in residual_candidates if c in columns_lower),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")
if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")
if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test: {len(df_test):,}\n"
        f"test_predictions: {len(test_predictions):,}"
    )

competition_df = pd.DataFrame({
    FEATURE: df_test[FEATURE].to_numpy(),
    "actual": pd.to_numeric(
        test_predictions[actual_col],
        errors="coerce"
    ).to_numpy(),
    "prediction": pd.to_numeric(
        test_predictions[predicted_col],
        errors="coerce"
    ).to_numpy(),
    "residual": pd.to_numeric(
        test_predictions[residual_col],
        errors="coerce"
    ).to_numpy()
})

competition_df[FEATURE] = (
    competition_df[FEATURE]
    .astype(str)
    .replace({
        "None": "MISSING",
        "nan": "MISSING",
        "null": "MISSING"
    })
)

competition_df["absolute_error"] = competition_df["residual"].abs()

valid_mask = (
    np.isfinite(competition_df["actual"].to_numpy())
    & np.isfinite(competition_df["prediction"].to_numpy())
    & np.isfinite(competition_df["residual"].to_numpy())
)

competition_df = competition_df.loc[valid_mask].reset_index(drop=True)

print(f"\nGeçerli gözlem sayısı: {len(competition_df):,}")

category_stats = (
    competition_df.groupby(FEATURE)
    .agg(
        count=("residual","size"),
        mean_residual=("residual","mean"),
        std_residual=("residual","std"),
        MAE=("absolute_error","mean"),
        MSE=("residual",lambda x: np.mean(x**2)),
        RMSE=("residual",lambda x: np.sqrt(np.mean(x**2)))
    )
    .reset_index()
)

effect_stats = category_stats[
    category_stats["count"] >= MIN_CATEGORY_COUNT
].copy()

global_std = competition_df["residual"].std()

if not np.isfinite(global_std) or global_std == 0:
    global_std = 1.0

effect_stats["standardized_effect"] = (
    effect_stats["mean_residual"] / global_std
)

effect_stats["absolute_standardized_effect"] = (
    effect_stats["standardized_effect"].abs()
)

preferred_order = [
    "Low Competition",
    "Medium Competition",
    "High Competition"
]

existing = effect_stats[FEATURE].astype(str).tolist()

final_order = [
    x for x in preferred_order
    if x in existing
]

final_order += [
    x for x in existing
    if x not in final_order
]

effect_stats[FEATURE] = pd.Categorical(
    effect_stats[FEATURE].astype(str),
    categories=final_order,
    ordered=True
)

effect_stats = (
    effect_stats
    .sort_values(FEATURE)
    .reset_index(drop=True)
)

visual_table = effect_stats[
    [
        FEATURE,
        "count",
        "mean_residual",
        "MAE",
        "MSE",
        "RMSE",
        "standardized_effect"
    ]
].copy()

visual_table.columns = [
    "Competition Level",
    "Observations",
    "Mean Residual",
    "MAE",
    "MSE",
    "RMSE",
    "Standardized Effect"
]

def highlight_max(series):
    mask = series == series.max()
    return [
        "background-color:#ffd6d6;color:#8b0000;font-weight:bold;"
        if value else ""
        for value in mask
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations":"{:,.0f}",
        "Mean Residual":"${:,.0f}",
        "MAE":"${:,.0f}",
        "MSE":"${:,.0f}",
        "RMSE":"${:,.0f}",
        "Standardized Effect":"{:+.3f}"
    })
    .apply(
        highlight_max,
        subset=[
            "MAE",
            "MSE",
            "RMSE",
            "Mean Residual",
            "Standardized Effect"
        ]
    )
    .set_properties(
        **{
            "text-align":"center",
            "font-size":"12pt",
            "padding":"8px"
        }
    )
    .set_table_styles([
        {
            "selector":"th",
            "props":[
                ("background-color","#404040"),
                ("color","white"),
                ("font-weight","bold"),
                ("text-align","center"),
                ("padding","8px")
            ]
        },
        {
            "selector":"td",
            "props":[
                ("text-align","center"),
                ("padding","8px")
            ]
        },
        {
            "selector":"caption",
            "props":[
                ("caption-side","top"),
                ("font-size","16pt"),
                ("font-weight","bold"),
                ("padding","8px")
            ]
        }
    ])
    .set_caption("TENDER COMPETITION LEVEL — RESIDUAL ANALYSIS")
)

display(styled_table)

residual_values = competition_df["residual"].to_numpy()
prediction_values = competition_df["prediction"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(19,14))
ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — NORMAL PROBABILITY — LINEAR SCALE
if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(42)
    residual_plot = rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
else:
    residual_plot = residual_values

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)

theoretical_quantiles = (
    stats.norm.ppf(
        (np.arange(1,n+1)-0.5)/n
    )
)

residual_mean = np.mean(sorted_residuals)
residual_std = np.std(
    sorted_residuals,
    ddof=1
)

if residual_std == 0:
    residual_std = 1

standardized_residuals = (
    sorted_residuals - residual_mean
) / residual_std

low = np.percentile(
    standardized_residuals,
    0.5
)

high = np.percentile(
    standardized_residuals,
    99.5
)

mask = (
    (standardized_residuals >= low)
    & (standardized_residuals <= high)
)

x_a1 = theoretical_quantiles[mask]
y_a1 = standardized_residuals[mask]

ax1.scatter(
    x_a1,
    y_a1,
    s=14,
    alpha=0.22,
    rasterized=True
)

x_line = np.array([
    x_a1.min(),
    x_a1.max()
])

ax1.plot(
    x_line,
    x_line,
    "--",
    linewidth=2.5,
    label="Normal reference"
)

# LINEER ÖLÇEK
ax1.set_yscale("linear")

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5,
    color="black",
    alpha=.8
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual",
    fontsize=13
)

ax1.set_title(
    "A1 — Normal Probability Plot\n"
    "Central 99% of Standardized Residuals",
    fontsize=16,
    fontweight="bold"
)

ax1.tick_params(
    axis="both",
    labelsize=11
)

ax1.grid(
    alpha=.25,
    linestyle="--"
)

ax1.legend(
    loc="upper left",
    fontsize=10
)

# A2 — RESIDUALS VS FITS
prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(
    prediction_m,
    [.5,99.5]
)

y_low, y_high = np.percentile(
    residual_m,
    [.5,99.5]
)

mask = (
    (prediction_m >= x_low)
    & (prediction_m <= x_high)
    & (residual_m >= y_low)
    & (residual_m <= y_high)
)

x_plot = prediction_m[mask]
y_plot = residual_m[mask]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=45,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual = np.median(y_plot)

ax2.axhline(
    median_residual,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=.02
)

cbar.set_label(
    "Observation Density\n(log scale)",
    fontsize=12
)

ax2.set_xlabel(
    "Predicted Price (Million USD)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Residuals Versus Fits\n"
    "Density-Based Visualization",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(
    alpha=.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)

# A3 — RESIDUAL HISTOGRAM
residual_low, residual_high = np.percentile(
    residual_values,
    [.5,99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

ax3.hist(
    central_residuals / 1e6,
    bins=35,
    alpha=.75,
    edgecolor="black",
    linewidth=.5
)

mean_residual = np.mean(
    residual_values
) / 1e6

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax3.axvline(
    mean_residual,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Mean = {mean_residual:.2f}M"
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Residual Distribution\n"
    "Central 99% of Observations",
    fontsize=16,
    fontweight="bold"
)

ax3.legend(fontsize=10)

ax3.grid(
    axis="y",
    alpha=.20
)

# A4 — OBSERVATION ORDER
order = np.arange(
    1,
    len(competition_df)+1
)

residual_order = (
    competition_df["residual"].to_numpy()
    / 1e6
)

q_low, q_high = np.percentile(
    residual_order,
    [.5,99.5]
)

mask = (
    (residual_order >= q_low)
    & (residual_order <= q_high)
)

order_plot = order[mask]
residual_plot = residual_order[mask]

ax4.scatter(
    order_plot,
    residual_plot,
    s=4,
    alpha=.055,
    rasterized=True
)

order_df = pd.DataFrame({
    "order":order_plot,
    "residual":residual_plot
})

WINDOW = max(
    500,
    min(
        ROLLING_WINDOW_MAX,
        len(order_df)//20
    )
)

min_periods = max(
    50,
    WINDOW//4
)

rolling_median = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .median()
)

rolling_q10 = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .quantile(.10)
)

rolling_q90 = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .quantile(.90)
)

ax4.fill_between(
    order_df["order"].to_numpy(),
    rolling_q10.to_numpy(),
    rolling_q90.to_numpy(),
    alpha=.20,
    label="Rolling 10–90% error range"
)

ax4.plot(
    order_df["order"].to_numpy(),
    rolling_median.to_numpy(),
    linewidth=3,
    label=f"Rolling Median ({WINDOW:,})"
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2.2,
    color="red",
    label="Residual = 0"
)

ax4.axhline(
    1,
    linestyle=":",
    linewidth=1.2,
    color="gray",
    alpha=.75,
    label="±$1M reference"
)

ax4.axhline(
    -1,
    linestyle=":",
    linewidth=1.2,
    color="gray",
    alpha=.75
)

ax4.axhline(
    5,
    linestyle=":",
    linewidth=1,
    color="gray",
    alpha=.45
)

ax4.axhline(
    -5,
    linestyle=":",
    linewidth=1,
    color="gray",
    alpha=.45
)

global_median = np.median(
    residual_order
)

ax4.axhline(
    global_median,
    linestyle="-.",
    linewidth=1.6,
    color="black",
    alpha=.75,
    label=f"Global Median = {global_median:+.2f}M"
)

ax4.set_yscale(
    "symlog",
    linthresh=1,
    linscale=1.2
)

ax4.set_xlabel(
    "Observation Order",
    fontsize=13
)

ax4.set_ylabel(
    "Residual (Million USD)\nSymlog Scale",
    fontsize=13
)

ax4.set_title(
    "A4 — Residuals Versus Observation Order\n"
    "Rolling Bias and Error Dispersion",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(
    which="major",
    alpha=.25,
    linestyle="--"
)

ax4.grid(
    which="minor",
    alpha=.1,
    linestyle=":"
)

ax4.legend(
    loc="upper right",
    fontsize=9
)

positive_ratio = (
    np.mean(residual_order > 0) * 100
)

negative_ratio = (
    np.mean(residual_order < 0) * 100
)

median_abs_error = np.median(
    np.abs(residual_order)
)

ax4.text(
    .02,
    .04,
    f"Central 99% shown\n"
    f"Rolling Window: {WINDOW:,}\n"
    f"Positive Residuals: {positive_ratio:.1f}%\n"
    f"Negative Residuals: {negative_ratio:.1f}%\n"
    f"Median |Residual|: ${median_abs_error:.2f}M",
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=.5",
        facecolor="white",
        edgecolor="gray",
        alpha=.9
    )
)

fig.suptitle(
    "TENDER COMPETITION LEVEL — RESIDUAL DIAGNOSTICS",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0,0,1,.95]
)

plt.show()
plt.close()

# B — STANDARDIZED EFFECTS
effects = effect_stats[
    "standardized_effect"
].to_numpy()

effects_sorted = np.sort(effects)
m = len(effects_sorted)

if m >= 2:
    theoretical_effect_quantiles = (
        stats.norm.ppf(
            (np.arange(1,m+1)-.5)/m
        )
    )

    fig, ax = plt.subplots(
        figsize=(12,8)
    )

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=130,
        alpha=.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope*x_ref,
        "--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():
        idx = np.argmin(
            np.abs(
                effects_sorted -
                row["standardized_effect"]
            )
        )

        ax.annotate(
            str(row[FEATURE]),
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(10,8),
            textcoords="offset points",
            fontsize=13,
            fontweight="bold"
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=1.8
    )

    ax.set_xlabel(
        "Theoretical Normal Quantiles",
        fontsize=13
    )

    ax.set_ylabel(
        "Standardized Competition-Level Effect",
        fontsize=13
    )

    ax.set_title(
        "B — Normal Plot of Standardized Competition-Level Effects",
        fontsize=17,
        fontweight="bold"
    )

    ax.grid(alpha=.20)

    plt.tight_layout()
    plt.show()
    plt.close()

# COMPONENT + RESIDUAL PLOTS
categories = effect_stats[FEATURE].tolist()

fig, axes = plt.subplots(
    1,
    len(categories),
    figsize=(22,8),
    squeeze=False
)

axes = axes.flatten()

for ax, category in zip(axes,categories):

    category_data = competition_df[
        competition_df[FEATURE] == category
    ].copy()

    x = (
        category_data["prediction"].to_numpy()
        / 1e6
    )

    y = (
        category_data["residual"].to_numpy()
        / 1e6
    )

    x_low,x_high = np.percentile(
        x,
        [1,99]
    )

    y_low,y_high = np.percentile(
        y,
        [1,99]
    )

    mask = (
        (x >= x_low)
        & (x <= x_high)
        & (y >= y_low)
        & (y <= y_high)
    )

    x_plot = x[mask]
    y_plot = y[mask]

    if len(x_plot) > 12000:
        rng = np.random.default_rng(42)

        idx = rng.choice(
            len(x_plot),
            12000,
            replace=False
        )

        x_scatter = x_plot[idx]
        y_scatter = y_plot[idx]

    else:
        x_scatter = x_plot
        y_scatter = y_plot

    ax.scatter(
        x_scatter,
        y_scatter,
        s=12,
        alpha=.15,
        rasterized=True
    )

    if len(x_plot) >= 3:

        slope,intercept = np.polyfit(
            x_plot,
            y_plot,
            1
        )

        x_line = np.linspace(
            x_plot.min(),
            x_plot.max(),
            300
        )

        ax.plot(
            x_line,
            intercept+slope*x_line,
            "--",
            linewidth=2.5,
            label="Linear fit"
        )

        sort_idx = np.argsort(x_plot)

        x_sorted = x_plot[sort_idx]
        y_sorted = y_plot[sort_idx]

        component_df = pd.DataFrame({
            "x":x_sorted,
            "y":y_sorted
        })

        smooth_window = max(
            50,
            min(
                500,
                len(component_df)//20
            )
        )

        rolling_component = (
            component_df["y"]
            .rolling(
                smooth_window,
                center=True,
                min_periods=max(
                    20,
                    smooth_window//4
                )
            )
            .median()
        )

        ax.plot(
            x_sorted,
            rolling_component,
            linewidth=3,
            label="Smoothed trend"
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=2.2,
        color="red",
        label="Residual = 0"
    )

    ax.set_xlabel(
        "Predicted Price (Million USD)",
        fontsize=14
    )

    ax.set_ylabel(
        "Residual (Million USD)",
        fontsize=14
    )

    ax.tick_params(
        axis="both",
        labelsize=12
    )

    ax.set_title(
        str(category),
        fontsize=19,
        fontweight="bold"
    )

    ax.grid(
        alpha=.22,
        linestyle="--"
    )

    row = effect_stats[
        effect_stats[FEATURE] == category
    ].iloc[0]

    mean_res = row["mean_residual"]/1e6
    mae = row["MAE"]/1e6
    rmse = row["RMSE"]/1e6
    effect = row["standardized_effect"]

    direction = (
        "UNDER-PREDICTION"
        if effect > 0
        else "OVER-PREDICTION"
    )

    ax.text(
        .03,
        .97,
        f"Observations : {int(row['count']):,}\n"
        f"Mean Residual: ${mean_res:+.2f}M\n"
        f"MAE          : ${mae:.2f}M\n"
        f"RMSE         : ${rmse:.2f}M\n"
        f"Std. Effect  : {effect:+.3f}\n"
        f"Direction    : {direction}",
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=11,
        linespacing=1.5,
        bbox=dict(
            boxstyle="round,pad=.6",
            facecolor="white",
            edgecolor="gray",
            alpha=.92
        )
    )

    ax.legend(
        loc="lower right",
        fontsize=10,
        frameon=True,
        framealpha=.90
    )

fig.suptitle(
    "COMPONENT + RESIDUAL PLOTS\nCompetition Level",
    fontsize=22,
    fontweight="bold",
    y=1.02
)

plt.tight_layout(
    rect=[0,0,1,.95]
)

plt.show()
plt.close()

# MAE KARŞILAŞTIRMASI
mae_plot = effect_stats.sort_values(
    "MAE",
    ascending=True
)

fig,ax = plt.subplots(
    figsize=(11,7)
)

bars = ax.bar(
    mae_plot[FEATURE].astype(str),
    mae_plot["MAE"]/1e6
)

ax.set_xlabel(
    "Competition Level",
    fontsize=13
)

ax.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax.set_title(
    "Competition Level — MAE Comparison",
    fontsize=17,
    fontweight="bold"
)

for bar,value in zip(
    bars,
    mae_plot["MAE"]/1e6
):
    ax.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+.03,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=.20
)

plt.tight_layout()
plt.show()
plt.close()

# MSE KARŞILAŞTIRMASI
mse_plot = effect_stats.sort_values(
    "MSE",
    ascending=True
)

fig,ax = plt.subplots(
    figsize=(11,7)
)

bars = ax.bar(
    mse_plot[FEATURE].astype(str),
    mse_plot["MSE"]/1e12
)

ax.set_xlabel(
    "Competition Level",
    fontsize=13
)

ax.set_ylabel(
    "MSE (Trillion USD²)",
    fontsize=13
)

ax.set_title(
    "Competition Level — MSE Comparison",
    fontsize=17,
    fontweight="bold"
)

for bar,value in zip(
    bars,
    mse_plot["MSE"]/1e12
):
    ax.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()
        + max(
            mse_plot["MSE"]/1e12
        )*.02,
        f"{value:.2f}T",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=.20
)

plt.tight_layout()
plt.show()
plt.close()

# RMSE KARŞILAŞTIRMASI
rmse_plot = effect_stats.sort_values(
    "RMSE",
    ascending=True
)

fig,ax = plt.subplots(
    figsize=(11,7)
)

bars = ax.bar(
    rmse_plot[FEATURE].astype(str),
    rmse_plot["RMSE"]/1e6
)

ax.set_xlabel(
    "Competition Level",
    fontsize=13
)

ax.set_ylabel(
    "RMSE (Million USD)",
    fontsize=13
)

ax.set_title(
    "Competition Level — RMSE Comparison",
    fontsize=17,
    fontweight="bold"
)

for bar,value in zip(
    bars,
    rmse_plot["RMSE"]/1e6
):
    ax.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+.03,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=.20
)

plt.tight_layout()
plt.show()
plt.close()

# EN PROBLEMLİ KATEGORİLER
if len(effect_stats) > 0:

    worst_mae = effect_stats.loc[
        effect_stats["MAE"].idxmax()
    ]

    worst_mse = effect_stats.loc[
        effect_stats["MSE"].idxmax()
    ]

    worst_rmse = effect_stats.loc[
        effect_stats["RMSE"].idxmax()
    ]

    worst_effect = effect_stats.loc[
        effect_stats[
            "absolute_standardized_effect"
        ].idxmax()
    ]

    print("\n" + "="*90)
    print("COMPETITION LEVEL — HATA ANALİZİ")
    print("="*90)

    print(
        f"\nEn yüksek MAE   : "
        f"{worst_mae[FEATURE]} → "
        f"${worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek MSE   : "
        f"{worst_mse[FEATURE]} → "
        f"${worst_mse['MSE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : "
        f"{worst_rmse[FEATURE]} → "
        f"${worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En güçlü etki   : "
        f"{worst_effect[FEATURE]} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    if worst_effect["standardized_effect"] > 0:

        print(
            "\nYorum: Pozitif residual → "
            "Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )

    else:

        print(
            "\nYorum: Negatif residual → "
            "Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )

# KATEGORİ DAĞILIMI
print("\n" + "="*90)
print("COMPETITION LEVEL DAĞILIMI")
print("="*90)

distribution = (
    competition_df[FEATURE]
    .value_counts()
    .rename_axis("Competition Level")
    .reset_index(name="Observations")
)

distribution["Percentage"] = (
    distribution["Observations"]
    / len(competition_df)
    * 100
)

print(
    distribution.to_string(
        index=False,
        formatters={
            "Observations":"{:,.0f}".format,
            "Percentage":"{:.2f}%".format
        }
    )
)

Görselleştirmelerde yer alan grafikler, ihalelerin rekabet düzeylerine (Tender Competition Level) ve tahmin büyüklüklerine göre modelin performansını detaylı biçimde ortaya koymaktadır.

**A1 — Normal Probability Plot (QQ-Plot)Grafik A1,** standartlaştırılmış artıkların teorik normal dağılım çizgisiyle uyumunu incelemektedir. Grafikte artıkların mavi referans doğrusu üzerinde düz bir hat izlemek yerine belirgin bir "S" eğrisi çizdiği ve özellikle sağ üst bölgede dik bir yükseliş gerçekleştirdiği görülmektedir. Bu yapı, model hatalarının normal dağılmadığını, ağır sağ kuyruk (heavy-tailed) sergilediğini ve yüksek bütçeli ihalelerde aşırı sapmalar yaşandığını doğrulamaktadır.

**A2 — Residuals Versus Fits (Density-Based)Grafik A2,** tahmin edilen ihale fiyatı arttıkça artıkların nasıl bir dağılım sergilediğini yoğunluk haritası üzerinden göstermektedir. Düşük tahmin değerlerinde (0–25M$ bandı) artıklar sıfır doğrusu etrafında son derece yoğun ve dar bir bantta toplanmışken, tahmin edilen fiyat büyüdükçe dikey yönlü saçılım dramatik şekilde genişlemektedir. Bu durum modelde ciddi bir heteroskedastisite (artan varyans) ve yüksek değerli ihalelerde tahmin kararsızlığı olduğunu açıkça ortaya koymaktadır.

**A3 — Residual DistributionGrafik A3,** artıkların frekans dağılımını ve ortalama sapma yönünü histogram şeklinde sunmaktadır. Verilerin ezici bir çoğunluğunun sıfır noktası etrafında kümelenmesine karşın, dağılımın sağa doğru oldukça uzun bir kuyruk oluşturduğu gözlenmektedir. Sıfır referans çizgisi ile ortalama hata çizgisi (Mean = +1.60M) arasındaki belirgin fark, modelin veriyi genel olarak eksik tahmin etme (under-prediction) eğiliminde olduğunu göstermektedir.

**A4 — Residuals Versus Observation OrderGrafik A4,** zaman veya gözlem sırasına bağlı olarak modelde zamansal bir kayma (bias) oluşup oluşmadığını test etmektedir. %10–90 hata aralığını temsil eden turuncu bant ve 2.000 gözlemlik hareketli ortanca çizgi (Rolling\ Median), gözlem sırası boyunca sıfır eksenine paralel ve sabit genişlikte devam etmektedir. Bu durum, modelin veri dizilim sırasından etkilenmediğini ve zamansal açıdan kararlı çalıştığını kanıtlamaktadır.

**B — Normal Plot of Standardized Competition-Level EffectsGrafik B,** rekabet seviyelerinin standartlaştırılmış etki büyüklüklerini teorik normallik bazında karşılaştırmaktadır. Tüm rekabet kategorileri (Low, High, Medium) sıfır çizgisinin üzerinde ve +0.098 ile +0.106 arasında pozitif değerler almıştır. Bu grafik, modelin eksik tahmin eğiliminin tek bir kategoriye özgü olmadığını, tüm rekabet seviyelerinde sistematik bir sorun olarak varlığını sürdürdüğünü göstermektedir.

**Component + Residual Plots (Low, Medium, High Competition)** Bu paneller, her bir rekabet grubunda tahmin edilen fiyat ile artıklar arasındaki ilişkiyi doğrusal fit ve düzleştirilmiş trend (Smoothed trend) çizgileriyle analiz etmektedir. Her üç grupta da Smoothed trend çizgisi düşük fiyatlarda sıfıra yakın seyrederken, fiyat büyüdükçe yukarı doğru bükülmektedir. Özellikle High Competition grubunda çok yüksek fiyatlı uç değerlerin sıklığı artarken, Low ve Medium gruplarında fiyat yükseldikçe hata payının katlanarak büyüdüğü izlenmektedir.

**MAE, MSE ve RMSE Karşılaştırma Grafikleri (Sütun Grafikleri)** Son üç bar grafiği, rekabet seviyelerine göre hata metriklerinin dağılımını özetlemektedir. MAE karşılaştırmasında en yüksek ortalama mutlak hata Medium Competition (2.11M) grubunda gerçekleşirken; karesel hataları temsil eden MSE (356.79T) ve RMSE (18.89M) grafiklerinde Low Competition açık farkla zirvededir. Bu zıtlık, Low Competition grubundaki ortalama hatanın makul seviyede olmasına rağmen, birkaç devasa aykırı değerin karesel metrikleri patlattığını gösteren net bir kanıttır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "competition_level"
N_BINS = 5

if "X_test" not in globals():
    raise NameError("X_test bulunamadı.")

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")

# X_test -> pandas
if hasattr(X_test, "select"):
    Xf = X_test.select(FEATURE).to_pandas().reset_index(drop=True)
else:
    Xf = X_test[[FEATURE]].copy().reset_index(drop=True)

# test_predictions -> pandas
if hasattr(test_predictions, "to_pandas"):
    pred_df = test_predictions.to_pandas().reset_index(drop=True)
elif isinstance(test_predictions, pd.DataFrame):
    pred_df = test_predictions.reset_index(drop=True).copy()
else:
    pred_df = None

# Actual / predicted
if pred_df is not None:

    pred_candidates = [
        "predicted_usd",
        "prediction_usd",
        "predicted",
        "prediction",
        "y_pred",
        "pred"
    ]

    actual_candidates = [
        "actual_usd",
        "actual",
        "y_true",
        "target",
        "true_price"
    ]

    pred_col = next(
        (c for c in pred_candidates if c in pred_df.columns),
        None
    )

    actual_col = next(
        (c for c in actual_candidates if c in pred_df.columns),
        None
    )

    if pred_col is None or actual_col is None:
        raise ValueError(
            "Actual/predicted kolonları bulunamadı.\n"
            f"Mevcut kolonlar: {list(pred_df.columns)}"
        )

    y_pred = pd.to_numeric(
        pred_df[pred_col],
        errors="coerce"
    ).to_numpy()

    y_true = pd.to_numeric(
        pred_df[actual_col],
        errors="coerce"
    ).to_numpy()

else:

    if "y_test" not in globals():
        raise NameError(
            "test_predictions array ise y_test gerekli."
        )

    y_pred = np.asarray(test_predictions).reshape(-1)
    y_true = np.asarray(y_test).reshape(-1)

# Uzunluk kontrolü
if not (len(Xf) == len(y_true) == len(y_pred)):
    raise ValueError(
        f"Uzunluklar eşleşmiyor: "
        f"X_test={len(Xf)}, "
        f"y_true={len(y_true)}, "
        f"y_pred={len(y_pred)}"
    )

# ------------------------------------------------------------
# ANALİZ DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame({
    FEATURE: Xf[FEATURE].astype(str),
    "actual": y_true,
    "predicted": y_pred
})

df["actual"] = pd.to_numeric(
    df["actual"],
    errors="coerce"
)

df["predicted"] = pd.to_numeric(
    df["predicted"],
    errors="coerce"
)

df = df.dropna(
    subset=["actual", "predicted"]
).copy()

# Negatif değerleri engelle
df["actual"] = df["actual"].clip(lower=0)
df["predicted"] = df["predicted"].clip(lower=0)

# ------------------------------------------------------------
# USD RESIDUAL
# ------------------------------------------------------------

df["residual_usd"] = (
    df["actual"] - df["predicted"]
)

# ------------------------------------------------------------
# LOG RESIDUAL
# ------------------------------------------------------------

df["actual_log"] = np.log1p(df["actual"])
df["predicted_log"] = np.log1p(df["predicted"])

# Pozitif = underprediction
df["log_residual"] = (
    df["actual_log"] - df["predicted_log"]
)

df["abs_log_error"] = (
    df["log_residual"].abs()
)

# ------------------------------------------------------------
# PREDICTED PRICE QUANTILE
# ------------------------------------------------------------

edges = df["predicted"].quantile(
    np.linspace(0, 1, N_BINS + 1)
).to_numpy()

edges = np.unique(edges)

if len(edges) < 3:
    raise ValueError(
        "Predicted price yeterli çeşitlilik içermiyor."
    )

df["price_bin"] = pd.cut(
    df["predicted"],
    bins=edges,
    labels=False,
    include_lowest=True,
    duplicates="drop"
)

n_bins = int(
    df["price_bin"].nunique()
)

# ------------------------------------------------------------
# COMPETITION LEVEL İSİMLERİ
# ------------------------------------------------------------

competition_names = {
    "0": "Low Competition",
    "1": "Medium Competition",
    "2": "High Competition",
    "Low": "Low Competition",
    "Medium": "Medium Competition",
    "High": "High Competition"
}

df["Competition"] = (
    df[FEATURE]
    .map(competition_names)
    .fillna(df[FEATURE])
)

# ------------------------------------------------------------
# LOG-SPACE METRİKLER
# ------------------------------------------------------------

rows = []

for (competition, price_bin), g in df.groupby(
    ["Competition", "price_bin"],
    observed=True
):

    r = g["log_residual"].to_numpy()

    rows.append({
        "Competition Level": competition,
        "Price Bin": int(price_bin) + 1,
        "N": len(g),
        "Predicted Mean ($M)":
            g["predicted"].mean() / 1e6,
        "Mean Log Residual":
            r.mean(),
        "Median Log Residual":
            np.median(r),
        "MAE Log":
            np.mean(np.abs(r)),
        "RMSE Log":
            np.sqrt(np.mean(r ** 2)),
        "Log Residual P90":
            np.percentile(r, 90),
        "Log Residual P99":
            np.percentile(r, 99),
        "Underprediction %":
            np.mean(r > 0) * 100,
        "Overprediction %":
            np.mean(r < 0) * 100
    })

summary_log = pd.DataFrame(rows)

summary_log = summary_log.sort_values(
    ["Competition Level", "Price Bin"]
).reset_index(drop=True)

# ------------------------------------------------------------
# ANA TABLO
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("COMPETITION LEVEL × PREDICTED PRICE — LOG RESIDUAL ANALYSIS")
print("=" * 120)

print(
    summary_log.to_string(
        index=False,
        formatters={
            "Predicted Mean ($M)": "{:.2f}".format,
            "Mean Log Residual": "{:+.4f}".format,
            "Median Log Residual": "{:+.4f}".format,
            "MAE Log": "{:.4f}".format,
            "RMSE Log": "{:.4f}".format,
            "Log Residual P90": "{:+.4f}".format,
            "Log Residual P99": "{:+.4f}".format,
            "Underprediction %": "{:.1f}".format,
            "Overprediction %": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# PIVOTLAR
# ------------------------------------------------------------

pivot_mean = summary_log.pivot(
    index="Competition Level",
    columns="Price Bin",
    values="Mean Log Residual"
)

pivot_mae = summary_log.pivot(
    index="Competition Level",
    columns="Price Bin",
    values="MAE Log"
)

pivot_rmse = summary_log.pivot(
    index="Competition Level",
    columns="Price Bin",
    values="RMSE Log"
)

pivot_under = summary_log.pivot(
    index="Competition Level",
    columns="Price Bin",
    values="Underprediction %"
)

print("\n" + "=" * 80)
print("MEAN LOG RESIDUAL")
print("=" * 80)
print(pivot_mean.round(4))

print("\n" + "=" * 80)
print("MAE LOG")
print("=" * 80)
print(pivot_mae.round(4))

print("\n" + "=" * 80)
print("RMSE LOG")
print("=" * 80)
print(pivot_rmse.round(4))

print("\n" + "=" * 80)
print("UNDERPREDICTION (%)")
print("=" * 80)
print(pivot_under.round(1))

# ------------------------------------------------------------
# GRAFİK 1 — MEAN LOG RESIDUAL
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for competition in pivot_mean.index:

    ax.plot(
        pivot_mean.columns,
        pivot_mean.loc[competition],
        marker="o",
        linewidth=2,
        label=str(competition)
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.set_title(
    "Competition Level × Predicted Price — Mean Log Residual",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Predicted Price Quantile")
ax.set_ylabel("Mean Log Residual")

ax.set_xticks(
    range(1, len(pivot_mean.columns) + 1)
)

ax.legend(title="Competition Level")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# GRAFİK 2 — MAE LOG
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for competition in pivot_mae.index:

    ax.plot(
        pivot_mae.columns,
        pivot_mae.loc[competition],
        marker="o",
        linewidth=2,
        label=str(competition)
    )

ax.set_title(
    "Competition Level × Predicted Price — Log MAE",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Predicted Price Quantile")
ax.set_ylabel("MAE in Log Space")

ax.set_xticks(
    range(1, len(pivot_mae.columns) + 1)
)

ax.legend(title="Competition Level")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# GRAFİK 3 — UNDERPREDICTION
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for competition in pivot_under.index:

    ax.plot(
        pivot_under.columns,
        pivot_under.loc[competition],
        marker="o",
        linewidth=2,
        label=str(competition)
    )

ax.axhline(
    50,
    linestyle="--",
    linewidth=1.5
)

ax.set_title(
    "Competition Level × Predicted Price — Underprediction Rate",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Predicted Price Quantile")
ax.set_ylabel("Underprediction (%)")

ax.set_ylim(0, 100)

ax.set_xticks(
    range(1, len(pivot_under.columns) + 1)
)

ax.legend(title="Competition Level")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# EN YÜKSEK LOG BIAS
# ------------------------------------------------------------

worst_bias = summary_log.loc[
    summary_log["Mean Log Residual"]
    .abs()
    .sort_values(ascending=False)
    .index
].head(10)

print("\n" + "=" * 105)
print("EN YÜKSEK LOG-SPACE BIAS GÖSTEREN SEGMENTLER")
print("=" * 105)

print(
    worst_bias[
        [
            "Competition Level",
            "Price Bin",
            "N",
            "Mean Log Residual",
            "MAE Log",
            "RMSE Log",
            "Underprediction %"
        ]
    ].to_string(
        index=False,
        formatters={
            "Mean Log Residual": "{:+.4f}".format,
            "MAE Log": "{:.4f}".format,
            "RMSE Log": "{:.4f}".format,
            "Underprediction %": "{:.1f}".format
        }
    )
)

Bu kod, competition_level değişkeninin model hataları üzerindeki etkisini log-space residual üzerinden inceler ve her rekabet seviyesini (Low, Medium, High) tahmin edilen fiyatın 5 farklı seviyesine ayırarak karşılaştırır. Her segment için ortalama ve medyan log residual, MAE, RMSE, uç hata değerleri ve underprediction oranını hesaplayarak modelin bias mı yoksa yüksek hata değişkenliği mi gösterdiğini belirlemeyi amaçlar. Sonuçlar, üç rekabet seviyesinde de ortalama log residual değerlerinin genel olarak 0'a yakın olduğunu ve dolayısıyla belirgin bir competition-level bias bulunmadığını göstermektedir. Buna karşılık Medium Competition'da MAE/RMSE'nin genel olarak daha yüksek olması, bu grubun tahmin edilmesinin biraz daha zor olduğunu göstermektedir. Ayrıca tüm rekabet seviyelerinde en yüksek fiyat grubunda pozitif residual ve artan underprediction görülmesi, problemin belirli bir rekabet seviyesinden ziyade yüksek fiyatlı ihalelerde ortaya çıkan genel tahmin zorluğu olduğunu göstermektedir. Bu nedenle competition_level modelden çıkarılmamalı veya değiştirilmemeli; ancak ileride yüksek fiyat × competition level etkileşimi incelenebilir.

**competition_supply_interaction Özelliği Neden Oluşturuldu?**

tender_competitionlevel (rekabet seviyesi) ve tender_supplytype (ihale türü) değişkenlerinin tek başlarına yakalayamadığı çapraz etkileşimleri modele doğrudan sunmak amacıyla oluşturulmuştur.Özellikle rekabetin düşük olduğu (Low Competition) segmentte gözlemlenen 356.79T^2 tutarındaki devasa karesel hata (MSE) ve uç değerler, rekabet seviyesinin tek başına bir sonucu değil; "Düşük Rekabetli Yapım İhalesi" (Low Competition_Works) gibi belirli ve büyük bütçeli kombinasyonların sonucudur. Bu iki kategorik verinin birleştirilmesiyle (String Concatenation) türetilen özellik, karar ağaçlarının uç bütçeli ihalelerin hangi spesifik alım türünde yoğunlaştığını tek bir karar düğümünde öğrenmesini sağlar ve Q_5 segmentindeki under-prediction sapmasını azaltmaya katkı sunar.

## TENDER_NATIONALPROCEDURETYPE (İhalenin ulusal mevzuata göre uygulanan prosedür türünü ifade eden kategorik değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "tender_nationalproceduretype"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
ROLLING_WINDOW_MAX = 2000
CATEGORIES_PER_PAGE = 8

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır.")
if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")
if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {col.lower(): col for col in test_predictions.columns}

actual_candidates = [
    "actual_usd", "actual", "y_true", "true", "target",
    "actual_price", "actual_tender_finalpriceusd"
]
predicted_candidates = [
    "predicted_usd", "predicted", "prediction", "pred",
    "y_pred", "predicted_price", "predicted_tender_finalpriceusd"
]
residual_candidates = ["residual_usd", "residual", "error", "prediction_error"]

actual_col = next((columns_lower[c] for c in actual_candidates if c in columns_lower), None)
predicted_col = next((columns_lower[c] for c in predicted_candidates if c in columns_lower), None)
residual_col = next((columns_lower[c] for c in residual_candidates if c in columns_lower), None)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")
if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")
if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test          : {len(df_test):,}\n"
        f"test_predictions : {len(test_predictions):,}"
    )

analysis_df = pd.DataFrame({
    FEATURE: df_test[FEATURE].to_numpy(),
    "actual": pd.to_numeric(test_predictions[actual_col], errors="coerce").to_numpy(),
    "prediction": pd.to_numeric(test_predictions[predicted_col], errors="coerce").to_numpy(),
    "residual": pd.to_numeric(test_predictions[residual_col], errors="coerce").to_numpy()
})

analysis_df[FEATURE] = (
    analysis_df[FEATURE].astype(str)
    .replace({"None": "MISSING", "nan": "MISSING", "null": "MISSING"})
)
analysis_df["absolute_error"] = analysis_df["residual"].abs()

valid_mask = (
    np.isfinite(analysis_df["actual"].to_numpy())
    & np.isfinite(analysis_df["prediction"].to_numpy())
    & np.isfinite(analysis_df["residual"].to_numpy())
)
analysis_df = analysis_df.loc[valid_mask].reset_index(drop=True)

print(f"\nGeçerli gözlem sayısı: {len(analysis_df):,}")

category_stats = (
    analysis_df.groupby(FEATURE)
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        std_residual=("residual", "std"),
        MAE=("absolute_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

effect_stats = category_stats[category_stats["count"] >= MIN_CATEGORY_COUNT].copy()
global_std = analysis_df["residual"].std()

effect_stats["standardized_effect"] = effect_stats["mean_residual"] / global_std
effect_stats["absolute_standardized_effect"] = effect_stats["standardized_effect"].abs()
effect_stats = effect_stats.sort_values("count", ascending=False).reset_index(drop=True)

print(f"Analiz edilen kategori sayısı: {len(effect_stats)}")

visual_table = effect_stats[
    [FEATURE, "count", "mean_residual", "MAE", "RMSE", "standardized_effect"]
].copy()

visual_table.columns = [
    "Procedure Type", "Observations", "Mean Residual",
    "MAE", "RMSE", "Standardized Effect"
]

def highlight_max(series):
    return [
        "background-color: #ffd6d6;color: #8b0000;font-weight: bold;"
        if value == series.max() else ""
        for value in series
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "${:,.0f}",
        "MAE": "${:,.0f}",
        "RMSE": "${:,.0f}",
        "Standardized Effect": "{:+.3f}"
    })
    .apply(highlight_max, subset=["MAE", "RMSE", "Mean Residual", "Standardized Effect"])
    .set_properties(**{"text-align": "center", "font-size": "11pt", "padding": "8px"})
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#404040"), ("color", "white"),
            ("font-weight", "bold"), ("text-align", "center"), ("padding", "9px")
        ]},
        {"selector": "td", "props": [
            ("text-align", "center"), ("padding", "8px")
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "16pt"),
            ("font-weight", "bold"), ("padding", "10px")
        ]}
    ])
    .set_caption(f"{FEATURE.upper()} — RESIDUAL ANALYSIS")
)

display(styled_table)

residual_values = analysis_df["residual"].to_numpy()
prediction_values = analysis_df["prediction"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(19, 14))
ax1, ax2, ax3, ax4 = axes.flatten()

if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(42)
    residual_plot = rng.choice(residual_values, size=PLOT_SAMPLE, replace=False)
else:
    residual_plot = residual_values

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)
theoretical_quantiles = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)

residual_mean = np.mean(sorted_residuals)
residual_std = np.std(sorted_residuals, ddof=1)
standardized_residuals = (sorted_residuals - residual_mean) / residual_std

lower_percentile = np.percentile(standardized_residuals, 0.5)
upper_percentile = np.percentile(standardized_residuals, 99.5)
a1_mask = (
    (standardized_residuals >= lower_percentile)
    & (standardized_residuals <= upper_percentile)
)

x_a1 = theoretical_quantiles[a1_mask]
y_a1 = standardized_residuals[a1_mask]

ax1.scatter(x_a1, y_a1, s=14, alpha=0.22, rasterized=True)
ax1.plot(
    [x_a1.min(), x_a1.max()],
    [x_a1.min(), x_a1.max()],
    linestyle="--", linewidth=2.5, label="Normal reference"
)
ax1.set_yscale("symlog", linthresh=1, linscale=1.2)
ax1.axhline(0, linestyle=":", linewidth=1.5, color="black")
ax1.set_xlabel("Theoretical Normal Quantiles", fontsize=13)
ax1.set_ylabel("Standardized Residual\n(Symlog Scale)", fontsize=13)
ax1.set_title("A1 — Normal Probability Plot\nCentral 99%", fontsize=16, fontweight="bold")
ax1.grid(which="major", alpha=0.25, linestyle="--")
ax1.grid(which="minor", alpha=0.10, linestyle=":")
ax1.legend(loc="upper left", fontsize=10)

prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(prediction_m, [0.5, 99.5])
y_low, y_high = np.percentile(residual_m, [0.5, 99.5])

plot_mask = (
    (prediction_m >= x_low) & (prediction_m <= x_high)
    & (residual_m >= y_low) & (residual_m <= y_high)
)

x_plot = prediction_m[plot_mask]
y_plot = residual_m[plot_mask]

hb = ax2.hexbin(
    x_plot, y_plot, gridsize=45, mincnt=1,
    bins="log", cmap="viridis", linewidths=0.15
)

ax2.axhline(0, linestyle="--", linewidth=2, color="red", label="Residual = 0")
median_residual = np.median(y_plot)
ax2.axhline(
    median_residual, linestyle=":", linewidth=2, color="black",
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(hb, ax=ax2, pad=0.02)
cbar.set_label("Observation Density\n(log scale)", fontsize=12)

ax2.set_xlabel("Predicted Price (Million USD)", fontsize=13)
ax2.set_ylabel("Residual (Million USD)", fontsize=13)
ax2.set_title(
    "A2 — Residuals Versus Fits\nDensity-Based Visualization",
    fontsize=16, fontweight="bold"
)
ax2.grid(alpha=0.20, linestyle="--")
ax2.legend(loc="upper right", fontsize=10)

residual_low, residual_high = np.percentile(residual_values, [0.5, 99.5])
central_residuals = residual_values[
    (residual_values >= residual_low) & (residual_values <= residual_high)
]

ax3.hist(
    central_residuals / 1e6,
    bins=35, alpha=0.75, edgecolor="black", linewidth=0.5
)

mean_residual = np.mean(residual_values) / 1e6
ax3.axvline(0, linestyle="--", linewidth=2, label="Residual = 0")
ax3.axvline(
    mean_residual, linestyle=":", linewidth=2,
    label=f"Mean = {mean_residual:.2f}M"
)
ax3.set_xlabel("Residual (Million USD)", fontsize=13)
ax3.set_ylabel("Frequency", fontsize=13)
ax3.set_title("A3 — Residual Distribution\nCentral 99%", fontsize=16, fontweight="bold")
ax3.legend(fontsize=10)
ax3.grid(axis="y", alpha=0.20)

order = np.arange(1, len(analysis_df) + 1)
residual_order = analysis_df["residual"].to_numpy() / 1e6

q_low, q_high = np.percentile(residual_order, [0.5, 99.5])
plot_mask = (residual_order >= q_low) & (residual_order <= q_high)

order_plot = order[plot_mask]
residual_plot = residual_order[plot_mask]

ax4.scatter(
    order_plot, residual_plot,
    s=4, alpha=0.055, rasterized=True
)

order_df = pd.DataFrame({"order": order_plot, "residual": residual_plot})

WINDOW = max(500, min(ROLLING_WINDOW_MAX, len(order_df) // 20))
min_periods = max(50, WINDOW // 4)

rolling_median = order_df["residual"].rolling(
    WINDOW, center=True, min_periods=min_periods
).median()

rolling_q10 = order_df["residual"].rolling(
    WINDOW, center=True, min_periods=min_periods
).quantile(0.10)

rolling_q90 = order_df["residual"].rolling(
    WINDOW, center=True, min_periods=min_periods
).quantile(0.90)

ax4.fill_between(
    order_df["order"].to_numpy(),
    rolling_q10.to_numpy(),
    rolling_q90.to_numpy(),
    alpha=0.20,
    label="Rolling 10–90% error range"
)

ax4.plot(
    order_df["order"].to_numpy(),
    rolling_median.to_numpy(),
    linewidth=3,
    label=f"Rolling Median ({WINDOW:,})"
)

ax4.axhline(0, linestyle="--", linewidth=2.2, color="red", label="Residual = 0")
ax4.axhline(1, linestyle=":", linewidth=1.2, color="gray", alpha=0.75, label="±$1M reference")
ax4.axhline(-1, linestyle=":", linewidth=1.2, color="gray", alpha=0.75)
ax4.axhline(5, linestyle=":", linewidth=1, color="gray", alpha=0.45)
ax4.axhline(-5, linestyle=":", linewidth=1, color="gray", alpha=0.45)

global_median = np.median(residual_order)
ax4.axhline(
    global_median, linestyle="-.", linewidth=1.6,
    color="black", alpha=0.75,
    label=f"Global Median = {global_median:+.2f}M"
)

ax4.set_yscale("symlog", linthresh=1, linscale=1.2)
ax4.set_xlabel("Observation Order", fontsize=13)
ax4.set_ylabel("Residual (Million USD)", fontsize=13)
ax4.set_title("A4 — Residuals Versus Observation Order\nRolling Bias", fontsize=16, fontweight="bold")
ax4.grid(which="major", alpha=0.25, linestyle="--")
ax4.grid(which="minor", alpha=0.10, linestyle=":")
ax4.legend(loc="upper right", fontsize=9)

positive_ratio = np.mean(residual_order > 0) * 100
negative_ratio = np.mean(residual_order < 0) * 100
median_abs_error = np.median(np.abs(residual_order))

info_text = (
    f"Central 99% shown\n"
    f"Rolling Window: {WINDOW:,}\n"
    f"Positive: {positive_ratio:.1f}%\n"
    f"Negative: {negative_ratio:.1f}%\n"
    f"Median |Residual|: ${median_abs_error:.2f}M"
)

ax4.text(
    0.02, 0.04, info_text,
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.90
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — RESIDUAL DIAGNOSTICS",
    fontsize=20, fontweight="bold"
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close()

effects = effect_stats["standardized_effect"].to_numpy()
effects_sorted = np.sort(effects)
m = len(effects_sorted)

if m >= 2:
    theoretical_effect_quantiles = stats.norm.ppf(
        (np.arange(1, m + 1) - 0.5) / m
    )

    fig, ax = plt.subplots(figsize=(14, 9))

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=100,
        alpha=0.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope * x_ref,
        linestyle="--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():
        idx = np.argmin(
            np.abs(effects_sorted - row["standardized_effect"])
        )

        label = str(row[FEATURE])
        if len(label) > 25:
            label = label[:22] + "..."

        ax.annotate(
            label,
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(8, 6),
            textcoords="offset points",
            fontsize=10
        )

    ax.axhline(0, linestyle=":", linewidth=1.8)
    ax.set_xlabel("Theoretical Normal Quantiles", fontsize=13)
    ax.set_ylabel("Standardized Effect", fontsize=13)
    ax.set_title(
        f"B — Standardized {FEATURE} Effects",
        fontsize=17, fontweight="bold"
    )
    ax.grid(alpha=0.20)

    plt.tight_layout()
    plt.show()
    plt.close()

categories = effect_stats[FEATURE].tolist()

for page_start in range(0, len(categories), CATEGORIES_PER_PAGE):
    page_categories = categories[page_start:page_start + CATEGORIES_PER_PAGE]
    n_categories = len(page_categories)
    ncols = 2
    nrows = int(np.ceil(n_categories / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(18, 7 * nrows),
        squeeze=False
    )
    axes = axes.flatten()

    for ax, category in zip(axes, page_categories):
        category_data = analysis_df[analysis_df[FEATURE] == category].copy()

        x = category_data["prediction"].to_numpy() / 1e6
        y = category_data["residual"].to_numpy() / 1e6

        if len(x) >= 10:
            x_low, x_high = np.percentile(x, [1, 99])
            y_low, y_high = np.percentile(y, [1, 99])

            mask = (
                (x >= x_low) & (x <= x_high)
                & (y >= y_low) & (y <= y_high)
            )

            x_plot = x[mask]
            y_plot = y[mask]
        else:
            x_plot = x
            y_plot = y

        if len(x_plot) > 12000:
            rng = np.random.default_rng(42)
            sample_idx = rng.choice(
                len(x_plot),
                size=12000,
                replace=False
            )
            x_scatter = x_plot[sample_idx]
            y_scatter = y_plot[sample_idx]
        else:
            x_scatter = x_plot
            y_scatter = y_plot

        ax.scatter(
            x_scatter, y_scatter,
            s=12, alpha=0.15, rasterized=True
        )

        if len(x_plot) >= 3:
            slope, intercept = np.polyfit(x_plot, y_plot, 1)

            x_line = np.linspace(
                x_plot.min(), x_plot.max(), 300
            )

            ax.plot(
                x_line,
                intercept + slope * x_line,
                linestyle="--",
                linewidth=2.5,
                label="Linear fit"
            )

            sort_idx = np.argsort(x_plot)
            x_sorted = x_plot[sort_idx]
            y_sorted = y_plot[sort_idx]

            component_df = pd.DataFrame({
                "x": x_sorted,
                "y": y_sorted
            })

            smooth_window = max(
                50,
                min(500, len(component_df) // 20)
            )

            rolling_component = component_df["y"].rolling(
                smooth_window,
                center=True,
                min_periods=max(20, smooth_window // 4)
            ).median()

            ax.plot(
                x_sorted,
                rolling_component,
                linewidth=3,
                label="Smoothed trend"
            )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=2.2,
            color="red",
            label="Residual = 0"
        )

        ax.set_xlabel("Predicted Price (Million USD)", fontsize=13)
        ax.set_ylabel("Residual (Million USD)", fontsize=13)
        ax.set_title(
            str(category),
            fontsize=15,
            fontweight="bold",
            pad=12
        )
        ax.grid(alpha=0.22, linestyle="--")

        row = effect_stats[effect_stats[FEATURE] == category].iloc[0]

        mean_res = row["mean_residual"] / 1e6
        mae = row["MAE"] / 1e6
        rmse = row["RMSE"] / 1e6
        effect = row["standardized_effect"]

        direction = (
            "UNDER-PREDICTION"
            if effect > 0
            else "OVER-PREDICTION"
        )

        metrics_text = (
            f"N            : {int(row['count']):,}\n"
            f"Mean Residual: ${mean_res:.2f}M\n"
            f"MAE          : ${mae:.2f}M\n"
            f"RMSE         : ${rmse:.2f}M\n"
            f"Std. Effect  : {effect:+.3f}\n"
            f"Direction    : {direction}"
        )

        ax.text(
            0.03, 0.97,
            metrics_text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            linespacing=1.4,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92
            )
        )

        ax.legend(
            loc="lower right",
            fontsize=9,
            frameon=True,
            framealpha=0.90
        )

    for ax in axes[n_categories:]:
        ax.axis("off")

    page_number = page_start // CATEGORIES_PER_PAGE + 1
    total_pages = int(np.ceil(len(categories) / CATEGORIES_PER_PAGE))

    fig.suptitle(
        f"COMPONENT + RESIDUAL PLOTS — {FEATURE}\n"
        f"Page {page_number}/{total_pages}",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
    plt.close()

# MAE
mae_plot = effect_stats.sort_values("MAE", ascending=True)

fig, ax = plt.subplots(figsize=(15, 8))

bars = ax.bar(
    mae_plot[FEATURE].astype(str),
    mae_plot["MAE"] / 1e6
)

ax.set_xlabel(FEATURE, fontsize=13)
ax.set_ylabel("MAE (Million USD)", fontsize=13)
ax.set_title(
    f"{FEATURE} — MAE Comparison",
    fontsize=17,
    fontweight="bold"
)
ax.tick_params(axis="x", labelsize=9, rotation=60)
ax.tick_params(axis="y", labelsize=11)

max_mae = (mae_plot["MAE"] / 1e6).max()

for bar, value in zip(bars, mae_plot["MAE"] / 1e6):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mae * 0.015,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=90
    )

ax.grid(axis="y", alpha=0.20)
plt.tight_layout()
plt.show()
plt.close()

# MSE
mse_plot = effect_stats.sort_values("RMSE", ascending=True).copy()
mse_plot["MSE"] = mse_plot["RMSE"] ** 2
mse_plot["MSE_T"] = mse_plot["MSE"] / 1e12

fig, ax = plt.subplots(figsize=(15, 8))

bars = ax.bar(
    mse_plot[FEATURE].astype(str),
    mse_plot["MSE_T"]
)

ax.set_xlabel(FEATURE, fontsize=13)
ax.set_ylabel("MSE (Trillion USD²)", fontsize=13)
ax.set_title(
    f"{FEATURE} — MSE Comparison",
    fontsize=17,
    fontweight="bold"
)
ax.tick_params(axis="x", labelsize=9, rotation=60)
ax.tick_params(axis="y", labelsize=11)

max_mse = mse_plot["MSE_T"].max()

for bar, value in zip(bars, mse_plot["MSE_T"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mse * 0.015,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=90
    )

ax.grid(axis="y", alpha=0.20)
plt.tight_layout()
plt.show()
plt.close()

# En problemli kategoriler
if len(effect_stats) > 0:
    worst_mae = effect_stats.loc[effect_stats["MAE"].idxmax()]
    worst_rmse = effect_stats.loc[effect_stats["RMSE"].idxmax()]
    worst_effect = effect_stats.loc[
        effect_stats["absolute_standardized_effect"].idxmax()
    ]

    print("\n" + "=" * 90)
    print(f"{FEATURE.upper()} — HATA ANALİZİ")
    print("=" * 90)

    print(
        f"\nEn yüksek MAE   : {worst_mae[FEATURE]} → "
        f"${worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : {worst_rmse[FEATURE]} → "
        f"${worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En yüksek MSE   : {worst_rmse[FEATURE]} → "
        f"${worst_rmse['RMSE'] ** 2:,.2f}"
    )

    print(
        f"En güçlü etki   : {worst_effect[FEATURE]} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    if worst_effect["standardized_effect"] > 0:
        print(
            "\nYorum: Pozitif residual → Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )
    else:
        print(
            "\nYorum: Negatif residual → Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )

44.575 gözlem ve 12 farklı ihale usulü kategorisi (tender_nationalproceduretype) üzerinden elde edilen artık analizi sonuçları, modelin kategorik bazda aşırı performans dengesizlikleri yaşadığını açıkça göstermektedir.
**Genel Model Teşhisi ve Ana Problem **
Model genelinde sistematik bir eksik tahmin (under-prediction) problemi mevcuttur; residual ortalamaları tüm kategorilerde pozitiftir. Model, ihale usullerinin şeffaflık, rekabet seviyesi ve bütçe büyüklüğü gibi karakteristik özelliklerini fiyatlama aşamasında yeterince ayrıştıramamakta, yüksek bütçeli özel Usul türlerinde ise çok ağır sapmalar üretmektedir.

**Grafiklerin Profesyonel Analizi ve Değerlendirmesi**
A1–A4 Residual Diagnostics (Temel Dağılım Grafikleri)A1 (QQ-Plot) ve A3 (Histogram) grafikleri, hataların normal dağılımdan dramatik şekilde uzaklaştığını ve ağır bir sağ kuyruk (right-skewed heavy tail) oluşturduğunu doğrular. A2 (Residuals vs Fits) grafiği, tahmin edilen fiyat yükseldikçe artıkların saçılımının dikeyleştiğini (heteroskedastisite) açıkça gösterir. A4 grafiği ise 2.000 gözlemlik hareketli ortancanın (Rolling Median) zero-line ekseninde sabit kaldığını, yani veri sırasına bağlı zamansal bir sapma olmadığını doğrular.

**B — Standardized tender_nationalproceduretype EffectsGrafik B,** usul türlerinin standartlaştırılmış etki büyüklüklerini karşılaştırır. Kategorilerin büyük çoğunluğu +0.051 ile +0.257 bandında toplanmışken, f18_pt_negotiated_without_publication_contract_notice kategorisi +0.803 etki değeriyle diğer tüm türlerden tamamen koparak sağ üst köşede aşırı bir aykırı değer (outlier) olarak konumlanmıştır.

**Component + Residual Plots (İhale Usulü Bazlı Saçılım Grafikleri)** pt_open (Açık İhale - 37.454 Gözlem): Verinin %84'ünü oluşturan bu ana grupta ortalama hata (MAE = 1.80M) makul seviyededir. Ancak 20M üzeri tahminlerde trend çizgisi bükülmekte ve model yüksek tutarlı açık ihaleleri eksik tahmin etmeye başlamaktadır.pt_award_contract_without_call (798 Gözlem): Tahmin edilen fiyat 0–1400M$ aralığına yayılan son derece geniş bir skalaya sahiptir. Smoothed trend çizgisindeki aniden dikleşen basamak yapısı, modelin devasa bütçeli ihaleleri yakalayamayıp ciddi tutarlarda eksik tahmin yaptığını (Mean Residual = 3.59M) gösterir.f18_pt_negotiated_without_publication_contract_notice (45 Gözlem): Sadece 45 gözlem barındırmasına rağmen modelin en başarısız olduğu alandır. İlan yapılmaksızın pazarlık usulüyle yürütülen bu ihalelerde model tamamen çökmüştür.

**MAE ve MSE Karşılaştırma Grafikleri (Sütun Grafikleri)**
Metrik karşılaştırmaları, modelin zayıf noktalarını netleştirmektedir:En Düşük Hata: Açık ilanlı ve rekabete açık f03_award_without_prior_publication (MAE = 1.15M) ve f03_pt_negotiated_without_competition (MAE = 1.39M) gibi kategorilerde model daha kararlı sonuçlar üretmiştir.Aşırı Sapma Bölgesi: f18_pt_negotiated_without_publication_contract_notice usulü 13.54M MAE ve 3,405.26T MSE değerleri ile bar grafiklerinde açık farkla tek başına zirvededir.Modelin Hata Yaptığı Temel AlanlarVeri Dengesizliği (Data Imbalance): Verinin %84'ü tek bir kategoriye (pt_open) aittir. Model, geriye kalan 11 azınlık kategorisini öğrenmek için yeterli gradyan güncellemesi alamamış ve genel veri eğilimine göre ezilmiştir.

**Pazarlık / İlansız Usul Dinamikleri:** İlan yayınlanmadan özel pazarlıkla yürütülen ihale tiplerinde (F18 ve doğrudan sözleşme usulleri) bütçeler doğası gereği yüksek ve değişkendir. Model bu Usul türlerindeki bütçe büyüklüğünü algılayamamaktadır.Kategorik Etkileşim Eksikliği: İhale usulü değişkeni, ihale bütçesi veya sektör kategorisi gibi değişkenlerle etkileşime sokulmadığı için model ihale usulünün bütçe üzerindeki çarpan etkisini öğrenememiştir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "tender_nationalproceduretype"
N_BINS = 5
MIN_CATEGORY_COUNT = 30

if "X_test" not in globals():
    raise NameError("X_test bulunamadı.")

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")

# X_test
if isinstance(X_test, pd.DataFrame):
    Xf = X_test[[FEATURE]].reset_index(drop=True)
else:
    Xf = X_test.select(FEATURE).to_pandas().reset_index(drop=True)

# test_predictions
if isinstance(test_predictions, pd.DataFrame):
    pred_df = test_predictions.reset_index(drop=True)
elif hasattr(test_predictions, "to_pandas"):
    pred_df = test_predictions.to_pandas().reset_index(drop=True)
else:
    pred_df = None

if pred_df is not None:
    pred_col = next(
        (c for c in [
            "predicted_usd", "prediction_usd",
            "predicted", "prediction", "y_pred", "pred"
        ] if c in pred_df.columns),
        None
    )

    actual_col = next(
        (c for c in [
            "actual_usd", "actual",
            "y_true", "target", "true_price"
        ] if c in pred_df.columns),
        None
    )

    if pred_col is None or actual_col is None:
        raise ValueError(
            f"Actual/predicted kolonları bulunamadı. "
            f"Mevcut kolonlar: {list(pred_df.columns)}"
        )

    y_pred = pd.to_numeric(
        pred_df[pred_col], errors="coerce"
    ).to_numpy()

    y_true = pd.to_numeric(
        pred_df[actual_col], errors="coerce"
    ).to_numpy()

else:
    if "y_test" not in globals():
        raise NameError("y_test bulunamadı.")

    y_pred = np.asarray(test_predictions).reshape(-1)
    y_true = np.asarray(y_test).reshape(-1)

# Uzunluk kontrolü
if not (len(Xf) == len(y_true) == len(y_pred)):
    raise ValueError(
        f"Uzunluk uyuşmazlığı: "
        f"X_test={len(Xf)}, "
        f"actual={len(y_true)}, "
        f"prediction={len(y_pred)}"
    )

# ------------------------------------------------------------
# DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame({
    FEATURE: Xf[FEATURE].astype(str),
    "actual": y_true,
    "predicted": y_pred
})

df["actual"] = pd.to_numeric(df["actual"], errors="coerce")
df["predicted"] = pd.to_numeric(df["predicted"], errors="coerce")

df = df.dropna(
    subset=["actual", "predicted"]
).copy()

df["actual"] = df["actual"].clip(lower=0)
df["predicted"] = df["predicted"].clip(lower=0)

# ------------------------------------------------------------
# LOG RESIDUAL
# ------------------------------------------------------------

df["actual_log"] = np.log1p(df["actual"])
df["predicted_log"] = np.log1p(df["predicted"])

# Pozitif = underprediction
df["log_residual"] = (
    df["actual_log"] - df["predicted_log"]
)

df["abs_log_error"] = (
    df["log_residual"].abs()
)

# ------------------------------------------------------------
# KATEGORİ FREKANSLARI
# ------------------------------------------------------------

category_counts = df[FEATURE].value_counts()

valid_categories = category_counts[
    category_counts >= MIN_CATEGORY_COUNT
].index

df = df[
    df[FEATURE].isin(valid_categories)
].copy()

print("\n" + "=" * 110)
print("TENDER_NATIONALPROCEDURETYPE — CATEGORY DISTRIBUTION")
print("=" * 110)

print(
    category_counts[
        category_counts.index.isin(valid_categories)
    ].to_string()
)

# ------------------------------------------------------------
# PREDICTED PRICE QUANTILE BIN
# ------------------------------------------------------------

edges = df["predicted"].quantile(
    np.linspace(0, 1, N_BINS + 1)
).to_numpy()

edges = np.unique(edges)

if len(edges) < 3:
    raise ValueError(
        "Predicted price yeterli çeşitlilik içermiyor."
    )

df["price_bin"] = pd.cut(
    df["predicted"],
    bins=edges,
    labels=False,
    include_lowest=True,
    duplicates="drop"
)

# ------------------------------------------------------------
# LOG-SPACE SEGMENT ANALYSIS
# ------------------------------------------------------------

rows = []

for (category, price_bin), g in df.groupby(
    [FEATURE, "price_bin"],
    observed=True
):

    r = g["log_residual"].to_numpy()

    rows.append({
        "Procedure Type": category,
        "Price Bin": int(price_bin) + 1,
        "N": len(g),
        "Predicted Mean ($M)": (
            g["predicted"].mean() / 1e6
        ),
        "Mean Log Residual": r.mean(),
        "Median Log Residual": np.median(r),
        "MAE Log": np.mean(np.abs(r)),
        "RMSE Log": np.sqrt(np.mean(r ** 2)),
        "Log Residual P90": np.percentile(r, 90),
        "Log Residual P99": np.percentile(r, 99),
        "Underprediction %": np.mean(r > 0) * 100,
        "Overprediction %": np.mean(r < 0) * 100
    })

summary_log = pd.DataFrame(rows)

summary_log = summary_log.sort_values(
    ["Procedure Type", "Price Bin"]
).reset_index(drop=True)

# ------------------------------------------------------------
# ANA TABLO
# ------------------------------------------------------------

print("\n" + "=" * 125)
print(
    "TENDER_NATIONALPROCEDURETYPE × PREDICTED PRICE "
    "— LOG RESIDUAL ANALYSIS"
)
print("=" * 125)

print(
    summary_log.to_string(
        index=False,
        formatters={
            "Predicted Mean ($M)": "{:.2f}".format,
            "Mean Log Residual": "{:+.4f}".format,
            "Median Log Residual": "{:+.4f}".format,
            "MAE Log": "{:.4f}".format,
            "RMSE Log": "{:.4f}".format,
            "Log Residual P90": "{:+.4f}".format,
            "Log Residual P99": "{:+.4f}".format,
            "Underprediction %": "{:.1f}".format,
            "Overprediction %": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# KATEGORİ BAZINDA GENEL LOG METRİKLER
# ------------------------------------------------------------

category_summary = (
    df.groupby(FEATURE)
    .agg(
        N=("log_residual", "size"),
        Mean_Log_Residual=("log_residual", "mean"),
        Median_Log_Residual=("log_residual", "median"),
        MAE_Log=("abs_log_error", "mean"),
        RMSE_Log=(
            "log_residual",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        Underprediction=(
            "log_residual",
            lambda x: np.mean(x > 0) * 100
        ),
        Overprediction=(
            "log_residual",
            lambda x: np.mean(x < 0) * 100
        )
    )
    .reset_index()
    .sort_values("Mean_Log_Residual", key=np.abs, ascending=False)
)

print("\n" + "=" * 115)
print("PROCEDURE TYPE — GENEL LOG-SPACE HATA ÖZETİ")
print("=" * 115)

print(
    category_summary.to_string(
        index=False,
        formatters={
            "Mean_Log_Residual": "{:+.4f}".format,
            "Median_Log_Residual": "{:+.4f}".format,
            "MAE_Log": "{:.4f}".format,
            "RMSE_Log": "{:.4f}".format,
            "Underprediction": "{:.1f}".format,
            "Overprediction": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# PIVOTLAR
# ------------------------------------------------------------

pivot_mean = summary_log.pivot(
    index="Procedure Type",
    columns="Price Bin",
    values="Mean Log Residual"
)

pivot_mae = summary_log.pivot(
    index="Procedure Type",
    columns="Price Bin",
    values="MAE Log"
)

pivot_rmse = summary_log.pivot(
    index="Procedure Type",
    columns="Price Bin",
    values="RMSE Log"
)

pivot_under = summary_log.pivot(
    index="Procedure Type",
    columns="Price Bin",
    values="Underprediction %"
)

print("\n" + "=" * 90)
print("MEAN LOG RESIDUAL")
print("=" * 90)
print(pivot_mean.round(4))

print("\n" + "=" * 90)
print("MAE LOG")
print("=" * 90)
print(pivot_mae.round(4))

print("\n" + "=" * 90)
print("RMSE LOG")
print("=" * 90)
print(pivot_rmse.round(4))

print("\n" + "=" * 90)
print("UNDERPREDICTION (%)")
print("=" * 90)
print(pivot_under.round(1))

# ------------------------------------------------------------
# EN YÜKSEK BIAS
# ------------------------------------------------------------

worst_bias = summary_log.loc[
    summary_log["Mean Log Residual"]
    .abs()
    .sort_values(ascending=False)
    .index
].head(10)

print("\n" + "=" * 115)
print("EN YÜKSEK LOG-SPACE BIAS GÖSTEREN SEGMENTLER")
print("=" * 115)

print(
    worst_bias[
        [
            "Procedure Type",
            "Price Bin",
            "N",
            "Mean Log Residual",
            "MAE Log",
            "RMSE Log",
            "Underprediction %"
        ]
    ].to_string(
        index=False,
        formatters={
            "Mean Log Residual": "{:+.4f}".format,
            "MAE Log": "{:.4f}".format,
            "RMSE Log": "{:.4f}".format,
            "Underprediction %": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# EN YÜKSEK HATA
# ------------------------------------------------------------

worst_error = category_summary.sort_values(
    "MAE_Log",
    ascending=False
).head(10)

print("\n" + "=" * 105)
print("EN YÜKSEK LOG-SPACE HATA ÜRETEN PROCEDURE TYPE'LAR")
print("=" * 105)

print(
    worst_error[
        [
            FEATURE,
            "N",
            "Mean_Log_Residual",
            "MAE_Log",
            "RMSE_Log",
            "Underprediction"
        ]
    ].to_string(
        index=False,
        formatters={
            "Mean_Log_Residual": "{:+.4f}".format,
            "MAE_Log": "{:.4f}".format,
            "RMSE_Log": "{:.4f}".format,
            "Underprediction": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# GRAFİK 1 — MEAN LOG RESIDUAL
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 7))

for category in pivot_mean.index:
    ax.plot(
        pivot_mean.columns,
        pivot_mean.loc[category],
        marker="o",
        linewidth=1.8,
        label=str(category)
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.3
)

ax.set_title(
    "National Procedure Type × Predicted Price — Mean Log Residual",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Predicted Price Bin")
ax.set_ylabel("Mean Log Residual")

ax.set_xticks(
    range(1, len(pivot_mean.columns) + 1)
)

ax.grid(alpha=0.25)
ax.legend(
    title="Procedure Type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# GRAFİK 2 — MAE LOG
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 7))

for category in pivot_mae.index:
    ax.plot(
        pivot_mae.columns,
        pivot_mae.loc[category],
        marker="o",
        linewidth=1.8,
        label=str(category)
    )

ax.set_title(
    "National Procedure Type × Predicted Price — Log MAE",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Predicted Price Bin")
ax.set_ylabel("MAE in Log Space")

ax.set_xticks(
    range(1, len(pivot_mae.columns) + 1)
)

ax.grid(alpha=0.25)
ax.legend(
    title="Procedure Type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# GRAFİK 3 — UNDERPREDICTION
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 7))

for category in pivot_under.index:
    ax.plot(
        pivot_under.columns,
        pivot_under.loc[category],
        marker="o",
        linewidth=1.8,
        label=str(category)
    )

ax.axhline(
    50,
    linestyle="--",
    linewidth=1.3
)

ax.set_title(
    "National Procedure Type × Predicted Price — Underprediction Rate",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Predicted Price Bin")
ax.set_ylabel("Underprediction (%)")

ax.set_ylim(0, 100)

ax.set_xticks(
    range(1, len(pivot_under.columns) + 1)
)

ax.grid(alpha=0.25)
ax.legend(
    title="Procedure Type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

Kod, tender_nationalproceduretype kategorilerini fiyat büyüklüğü (5 price bin) ile birlikte değerlendirerek her segment için log-space residual biasını, MAE/RMSE'yi ve under/overprediction oranlarını hesaplıyor. Sonuçlar, pt_open gibi yüksek örneklemli ana kategorilerde belirgin sistematik bias olmadığını, ancak hata seviyesinin fiyat arttıkça yükseldiğini gösteriyor. En yüksek hatalar pt_competitive_dialogue ve bazı düşük frekanslı özel prosedürlerde görülüyor; fakat bu kategorilerin gözlem sayıları çok düşük olduğu için sonuçları temkinli yorumlamak gerekiyor. Dolayısıyla temel problem kategorinin kendisinden çok, özellikle yüksek fiyatlı ihalelerde artan hata ve küçük kategorilerdeki model kararsızlığıdır.

**Neden Bu Yöntem Seçildi?**

Sadece 175 gözleme sahip f18... gibi kategoriler, karar ağaçlarının genel bir kural öğrenmesi için istatistiki olarak yetersizdir ve yüksek varyanslı gürültüye yol açar. Frekansı az kategorileri Target Encoding veya One-Hot Encoding işlemine doğrudan sokmak, modelin bu gruplardaki aşırı uç değerleri (outliers) ezberlemesine (overfitting) ve devasa tahmin hataları üretmesine neden olur. Rare Category Grouping yöntemi; bu az verili sınıfları tek bir temsil kütlesinde toplayarak gürültüyü sönümlemek, Q_5 bütçeli uç projelerin tekil kategorileri domine etmesini engellemek ve modelin genelleme kabiliyetini korumak için tercih edilmiştir.

**Çözüm Yöntemi**

tender_nationalproceduretype sütununda gözlemlenen aşırı under-prediction ve yüksek MSE sapmalarını gidermek amacıyla Eşik Tabanlı Nadir Sınıf Birleştirme (Threshold-Based Rare Category Grouping) yöntemi uygulanmıştır. Yöntem kapsamında, eğitim setindeki frekansı 200 gözlemin altında kalan 5 nadir ihale usulü (başta yüksek hata üreten f18... sınıfı olmak üzere) tespit edilerek "RARE" etiketi altında tek bir süper-kategoride toplanmıştır. Veri sızıntısını (data leakage) önlemek adına 200 eşik kuralı yalnızca X_train dağılımına göre belirlenmiş ve oluşturulan haritalama mimarisi validation ile test setlerine birebir aktarılmıştır. Bu sayede modelin yetersiz veriye sahip kategorilere özgü gürültüyü öğrenmesi engellenmiş, seyreklik (sparsity) problemi çözülmüş ve kategorik değişken daha kararlı bir yapıya kavuşturulmuştur.

## LOT_STATUS (İhaleye ait lotun durumunu/sonucunu (ör. kazandı, iptal edildi vb.) ifade eden kategorik değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "lot_status"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
ROLLING_WINDOW_MAX = 2000
CATEGORIES_PER_PAGE = 4
RANDOM_STATE = 42

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")
if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")
if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print("=" * 90)
print(f"{FEATURE.upper()} — RESIDUAL / COMPONENT ANALYSIS")
print("=" * 90)

columns_lower = {str(col).lower(): col for col in test_predictions.columns}

actual_candidates = [
    "actual_usd", "actual", "y_true", "true", "target",
    "actual_price", "actual_tender_finalpriceusd"
]
predicted_candidates = [
    "predicted_usd", "predicted", "prediction", "pred",
    "y_pred", "predicted_price", "predicted_tender_finalpriceusd"
]
residual_candidates = [
    "residual_usd", "residual", "error", "prediction_error"
]

actual_col = next(
    (columns_lower[c] for c in actual_candidates if c in columns_lower), None
)
predicted_col = next(
    (columns_lower[c] for c in predicted_candidates if c in columns_lower), None
)
residual_col = next(
    (columns_lower[c] for c in residual_candidates if c in columns_lower), None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")
if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")
if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test          : {len(df_test):,}\n"
        f"test_predictions : {len(test_predictions):,}"
    )

analysis_df = pd.DataFrame({
    FEATURE: np.asarray(df_test[FEATURE]),
    "actual": pd.to_numeric(
        test_predictions[actual_col], errors="coerce"
    ).to_numpy(),
    "prediction": pd.to_numeric(
        test_predictions[predicted_col], errors="coerce"
    ).to_numpy(),
    "residual": pd.to_numeric(
        test_predictions[residual_col], errors="coerce"
    ).to_numpy()
})

analysis_df[FEATURE] = (
    analysis_df[FEATURE]
    .astype("string")
    .fillna("MISSING")
    .replace({
        "None": "MISSING",
        "nan": "MISSING",
        "NaN": "MISSING",
        "null": "MISSING",
        "NULL": "MISSING",
        "<NA>": "MISSING"
    })
    .astype(str)
)

analysis_df["absolute_error"] = analysis_df["residual"].abs()

valid_mask = (
    np.isfinite(analysis_df["actual"].to_numpy())
    & np.isfinite(analysis_df["prediction"].to_numpy())
    & np.isfinite(analysis_df["residual"].to_numpy())
)

analysis_df = analysis_df.loc[valid_mask].reset_index(drop=True)

print(f"\nGeçerli gözlem sayısı: {len(analysis_df):,}")

category_stats = (
    analysis_df.groupby(FEATURE, dropna=False)
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        std_residual=("residual", "std"),
        MAE=("absolute_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(np.square(x)))),
        MSE=("residual", lambda x: np.mean(np.square(x)))
    )
    .reset_index()
)

effect_stats = category_stats[
    category_stats["count"] >= MIN_CATEGORY_COUNT
].copy()

if len(effect_stats) == 0:
    raise ValueError(
        f"{FEATURE} için {MIN_CATEGORY_COUNT} gözlemden fazla kategori bulunamadı."
    )

global_std = analysis_df["residual"].std()

if not np.isfinite(global_std) or global_std == 0:
    global_std = 1.0

effect_stats["standardized_effect"] = (
    effect_stats["mean_residual"] / global_std
)

effect_stats["absolute_standardized_effect"] = (
    effect_stats["standardized_effect"].abs()
)

effect_stats = (
    effect_stats
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(f"Analiz edilen kategori sayısı: {len(effect_stats)}")

required_effect_columns = [
    FEATURE,
    "count",
    "mean_residual",
    "MAE",
    "RMSE",
    "MSE",
    "standardized_effect",
    "absolute_standardized_effect"
]

visual_table = effect_stats[required_effect_columns].copy()

visual_table.columns = [
    "Lot Status",
    "Observations",
    "Mean Residual",
    "MAE",
    "RMSE",
    "MSE",
    "Standardized Effect",
    "Absolute Standardized Effect"
]

def highlight_max(series):
    mask = series == series.max()
    return [
        "background-color: #ffd6d6;color: #8b0000;font-weight: bold;"
        if value else ""
        for value in mask
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "{:,.0f}",
        "MAE": "{:,.0f}",
        "RMSE": "{:,.0f}",
        "MSE": "{:,.0f}",
        "Standardized Effect": "{:+.3f}",
        "Absolute Standardized Effect": "{:.3f}"
    })
    .apply(highlight_max, subset=["MAE"])
    .apply(highlight_max, subset=["RMSE"])
    .apply(highlight_max, subset=["MSE"])
    .apply(highlight_max, subset=["Absolute Standardized Effect"])
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "11pt",
            "padding": "8px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#404040"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "9px")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("padding", "8px")
            ]
        }
    ])
    .set_caption(f"{FEATURE.upper()} — RESIDUAL ANALYSIS")
)

display(styled_table)

residual_values = analysis_df["residual"].to_numpy()
prediction_values = analysis_df["prediction"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(20, 15))
ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — NORMAL PROBABILITY PLOT — LINEAR SCALE
if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(RANDOM_STATE)
    residual_plot = rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
else:
    residual_plot = residual_values

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)

if n >= 2:
    theoretical_quantiles = stats.norm.ppf(
        (np.arange(1, n + 1) - 0.5) / n
    )

    residual_mean = np.mean(sorted_residuals)
    residual_std = np.std(sorted_residuals, ddof=1)

    if residual_std > 0:
        standardized_residuals = (
            sorted_residuals - residual_mean
        ) / residual_std

        low = np.percentile(standardized_residuals, 0.5)
        high = np.percentile(standardized_residuals, 99.5)

        mask = (
            (standardized_residuals >= low)
            & (standardized_residuals <= high)
        )

        x_a1 = theoretical_quantiles[mask]
        y_a1 = standardized_residuals[mask]

        ax1.scatter(
            x_a1,
            y_a1,
            s=18,
            alpha=0.25,
            rasterized=True
        )

        ax1.plot(
            [x_a1.min(), x_a1.max()],
            [x_a1.min(), x_a1.max()],
            linestyle="--",
            linewidth=2.5,
            label="Normal reference"
        )

        ax1.axhline(
            0,
            linestyle=":",
            linewidth=1.5
        )

        # LINEER ÖLÇEK
        ax1.set_xscale("linear")
        ax1.set_yscale("linear")

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=12
)
ax1.set_ylabel(
    "Standardized Residual",
    fontsize=12
)
ax1.set_title(
    "A1 — Normal Probability Plot\nLinear Scale",
    fontsize=15,
    fontweight="bold"
)
ax1.grid(
    alpha=0.25,
    linestyle="--"
)
ax1.legend()

# A2 — RESIDUAL VS FITS — HEATMAP
prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(
    prediction_m,
    [0.5, 99.5]
)

y_low, y_high = np.percentile(
    residual_m,
    [0.5, 99.5]
)

plot_mask = (
    (prediction_m >= x_low)
    & (prediction_m <= x_high)
    & (residual_m >= y_low)
    & (residual_m <= y_high)
)

x_plot = prediction_m[plot_mask]
y_plot = residual_m[plot_mask]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=45,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=0.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual = np.median(y_plot)

ax2.axhline(
    median_residual,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=0.02
)

cbar.set_label(
    "Observation Density\n(log scale)"
)

ax2.set_xlabel(
    "Predicted Price (Million USD)",
    fontsize=12
)
ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=12
)
ax2.set_title(
    "A2 — Residuals Versus Fits\nDensity Heatmap",
    fontsize=15,
    fontweight="bold"
)
ax2.grid(
    alpha=0.20,
    linestyle="--"
)
ax2.legend()

# A3 — RESIDUAL DISTRIBUTION
residual_low, residual_high = np.percentile(
    residual_values,
    [0.5, 99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

ax3.hist(
    central_residuals / 1e6,
    bins=35,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.5
)

mean_residual = np.mean(residual_values) / 1e6
median_residual = np.median(residual_values) / 1e6

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Residual = 0"
)

ax3.axvline(
    mean_residual,
    linestyle=":",
    linewidth=2,
    label=f"Mean = {mean_residual:.2f}M"
)

ax3.axvline(
    median_residual,
    linestyle="-.",
    linewidth=2,
    label=f"Median = {median_residual:.2f}M"
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=12
)
ax3.set_ylabel(
    "Frequency",
    fontsize=12
)
ax3.set_title(
    "A3 — Residual Distribution\nCentral 99%",
    fontsize=15,
    fontweight="bold"
)
ax3.legend()
ax3.grid(
    axis="y",
    alpha=0.20
)

# A4 — RESIDUAL VS OBSERVATION ORDER
order = np.arange(
    1,
    len(analysis_df) + 1
)

residual_order = (
    analysis_df["residual"].to_numpy() / 1e6
)

q_low, q_high = np.percentile(
    residual_order,
    [0.5, 99.5]
)

plot_mask = (
    (residual_order >= q_low)
    & (residual_order <= q_high)
)

order_plot = order[plot_mask]
residual_plot = residual_order[plot_mask]

ax4.scatter(
    order_plot,
    residual_plot,
    s=4,
    alpha=0.055,
    rasterized=True
)

order_df = pd.DataFrame({
    "order": order_plot,
    "residual": residual_plot
})

if len(order_df) >= 100:
    WINDOW = max(
        500,
        min(
            ROLLING_WINDOW_MAX,
            len(order_df) // 20
        )
    )

    WINDOW = min(
        WINDOW,
        len(order_df)
    )

    min_periods = max(
        50,
        WINDOW // 4
    )

    rolling_median = (
        order_df["residual"]
        .rolling(
            WINDOW,
            center=True,
            min_periods=min_periods
        )
        .median()
    )

    rolling_q10 = (
        order_df["residual"]
        .rolling(
            WINDOW,
            center=True,
            min_periods=min_periods
        )
        .quantile(0.10)
    )

    rolling_q90 = (
        order_df["residual"]
        .rolling(
            WINDOW,
            center=True,
            min_periods=min_periods
        )
        .quantile(0.90)
    )

    ax4.fill_between(
        order_df["order"].to_numpy(),
        rolling_q10.to_numpy(),
        rolling_q90.to_numpy(),
        alpha=0.20,
        label="Rolling 10–90%"
    )

    ax4.plot(
        order_df["order"].to_numpy(),
        rolling_median.to_numpy(),
        linewidth=3,
        label=f"Rolling Median ({WINDOW:,})"
    )
else:
    WINDOW = len(order_df)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2.2,
    color="red",
    label="Residual = 0"
)

global_median = np.median(residual_order)

ax4.axhline(
    global_median,
    linestyle="-.",
    linewidth=1.6,
    color="black",
    label=f"Global Median = {global_median:+.2f}M"
)

ax4.set_yscale(
    "symlog",
    linthresh=1,
    linscale=1.2
)

ax4.set_xlabel(
    "Observation Order",
    fontsize=12
)
ax4.set_ylabel(
    "Residual (Million USD)",
    fontsize=12
)
ax4.set_title(
    "A4 — Residuals Versus Observation Order\nRolling Bias",
    fontsize=15,
    fontweight="bold"
)
ax4.grid(
    which="major",
    alpha=0.25,
    linestyle="--"
)
ax4.grid(
    which="minor",
    alpha=0.10,
    linestyle=":"
)
ax4.legend(fontsize=8)

positive_ratio = np.mean(residual_order > 0) * 100
negative_ratio = np.mean(residual_order < 0) * 100
median_abs_error = np.median(
    np.abs(residual_order)
)

info_text = (
    f"Central 99% shown\n"
    f"Rolling Window: {WINDOW:,}\n"
    f"Positive: {positive_ratio:.1f}%\n"
    f"Negative: {negative_ratio:.1f}%\n"
    f"Median |Residual|: {median_abs_error:.2f}M"
)

ax4.text(
    0.02,
    0.04,
    info_text,
    transform=ax4.transAxes,
    fontsize=9,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.90
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — RESIDUAL ANALYSIS DASHBOARD\nA1–A4",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)
plt.show()
plt.close()

# B — STANDARDIZED EFFECTS
effects_sorted = np.sort(
    effect_stats["standardized_effect"].to_numpy()
)

m = len(effects_sorted)

if m >= 2:
    theoretical_effect_quantiles = stats.norm.ppf(
        (np.arange(1, m + 1) - 0.5) / m
    )

    fig, ax = plt.subplots(
        figsize=(14, 9)
    )

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=100,
        alpha=0.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope * x_ref,
        linestyle="--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():
        idx = np.argmin(
            np.abs(
                effects_sorted -
                row["standardized_effect"]
            )
        )

        label = str(row[FEATURE])

        if len(label) > 25:
            label = label[:22] + "..."

        ax.annotate(
            label,
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(8, 6),
            textcoords="offset points",
            fontsize=10
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=1.8
    )

    ax.set_xlabel(
        "Theoretical Normal Quantiles",
        fontsize=13
    )
    ax.set_ylabel(
        "Standardized Effect",
        fontsize=13
    )
    ax.set_title(
        f"B — {FEATURE}\nStandardized Category Effects",
        fontsize=17,
        fontweight="bold"
    )
    ax.grid(alpha=0.20)

    plt.tight_layout()
    plt.show()
    plt.close()

# C — COMPONENT + RESIDUAL — 2x2
categories = effect_stats[FEATURE].tolist()

for page_start in range(
    0,
    len(categories),
    CATEGORIES_PER_PAGE
):
    page_categories = categories[
        page_start:
        page_start + CATEGORIES_PER_PAGE
    ]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(18, 14),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, category in zip(
        axes,
        page_categories
    ):
        category_data = analysis_df[
            analysis_df[FEATURE] == category
        ].copy()

        x = (
            category_data["prediction"].to_numpy()
            / 1e6
        )

        y = (
            category_data["residual"].to_numpy()
            / 1e6
        )

        if len(x) >= 10:
            x_low, x_high = np.percentile(
                x,
                [1, 99]
            )

            y_low, y_high = np.percentile(
                y,
                [1, 99]
            )

            mask = (
                (x >= x_low)
                & (x <= x_high)
                & (y >= y_low)
                & (y <= y_high)
            )

            x_plot = x[mask]
            y_plot = y[mask]
        else:
            x_plot = x
            y_plot = y

        if len(x_plot) > 12000:
            rng = np.random.default_rng(
                RANDOM_STATE
            )

            sample_idx = rng.choice(
                len(x_plot),
                size=12000,
                replace=False
            )

            x_scatter = x_plot[sample_idx]
            y_scatter = y_plot[sample_idx]
        else:
            x_scatter = x_plot
            y_scatter = y_plot

        ax.scatter(
            x_scatter,
            y_scatter,
            s=12,
            alpha=0.15,
            rasterized=True
        )

        if len(x_plot) >= 3:
            unique_x = np.unique(x_plot)

            if len(unique_x) >= 2:
                slope, intercept = np.polyfit(
                    x_plot,
                    y_plot,
                    1
                )

                x_line = np.linspace(
                    x_plot.min(),
                    x_plot.max(),
                    300
                )

                ax.plot(
                    x_line,
                    intercept + slope * x_line,
                    linestyle="--",
                    linewidth=2.5,
                    label="Linear fit"
                )

                sort_idx = np.argsort(x_plot)
                x_sorted = x_plot[sort_idx]
                y_sorted = y_plot[sort_idx]

                component_df = pd.DataFrame({
                    "x": x_sorted,
                    "y": y_sorted
                })

                smooth_window = max(
                    50,
                    min(
                        500,
                        len(component_df) // 20
                    )
                )

                rolling_component = (
                    component_df["y"]
                    .rolling(
                        smooth_window,
                        center=True,
                        min_periods=max(
                            20,
                            smooth_window // 4
                        )
                    )
                    .median()
                )

                ax.plot(
                    x_sorted,
                    rolling_component,
                    linewidth=3,
                    label="Smoothed trend"
                )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=2.2,
            color="red",
            label="Residual = 0"
        )

        ax.set_xlabel(
            "Predicted Price (Million USD)",
            fontsize=12
        )

        ax.set_ylabel(
            "Residual (Million USD)",
            fontsize=12
        )

        ax.set_title(
            str(category),
            fontsize=15,
            fontweight="bold",
            pad=12
        )

        ax.grid(
            alpha=0.22,
            linestyle="--"
        )

        row = effect_stats[
            effect_stats[FEATURE] == category
        ].iloc[0]

        mean_res = row["mean_residual"] / 1e6
        mae = row["MAE"] / 1e6
        rmse = row["RMSE"] / 1e6
        effect = row["standardized_effect"]

        direction = (
            "UNDER-PREDICTION"
            if effect > 0
            else "OVER-PREDICTION"
            if effect < 0
            else "NO CLEAR BIAS"
        )

        metrics_text = (
            f"N             : {int(row['count']):,}\n"
            f"Mean Residual : {mean_res:.2f}M\n"
            f"MAE           : {mae:.2f}M\n"
            f"RMSE          : {rmse:.2f}M\n"
            f"MSE           : {row['MSE'] / 1e12:.2f}T\n"
            f"Std. Effect   : {effect:+.3f}\n"
            f"Direction     : {direction}"
        )

        ax.text(
            0.03,
            0.97,
            metrics_text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=9,
            linespacing=1.4,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92
            )
        )

        ax.legend(
            loc="lower right",
            fontsize=8,
            frameon=True,
            framealpha=0.90
        )

    for ax in axes[len(page_categories):]:
        ax.axis("off")

    page_number = (
        page_start // CATEGORIES_PER_PAGE
    ) + 1

    total_pages = int(
        np.ceil(
            len(categories) /
            CATEGORIES_PER_PAGE
        )
    )

    fig.suptitle(
        f"COMPONENT + RESIDUAL PLOTS — {FEATURE.upper()}\n"
        f"Page {page_number}/{total_pages}",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.94]
    )

    plt.show()
    plt.close()

# D — MAE COMPARISON
mae_plot = effect_stats.sort_values(
    "MAE",
    ascending=True
)

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    mae_plot[FEATURE].astype(str),
    mae_plot["MAE"] / 1e6
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax.set_title(
    f"D — {FEATURE.upper()}\nMAE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    labelsize=9,
    rotation=60
)

max_mae = (
    mae_plot["MAE"] / 1e6
).max()

for bar, value in zip(
    bars,
    mae_plot["MAE"] / 1e6
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mae * 0.015,
        f"{value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=90
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# E — MSE COMPARISON
mse_plot = effect_stats.sort_values(
    "MSE",
    ascending=True
)

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    mse_plot[FEATURE].astype(str),
    mse_plot["MSE"] / 1e12
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "MSE (Trillion USD²)",
    fontsize=13
)

ax.set_title(
    f"E — {FEATURE.upper()}\nMSE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    labelsize=9,
    rotation=60
)

max_mse = (
    mse_plot["MSE"] / 1e12
).max()

for bar, value in zip(
    bars,
    mse_plot["MSE"] / 1e12
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mse * 0.015,
        f"{value:.2f}T",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=90
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# EN PROBLEMLİ KATEGORİLER
if len(effect_stats) > 0:
    worst_mae = effect_stats.loc[
        effect_stats["MAE"].idxmax()
    ]

    worst_rmse = effect_stats.loc[
        effect_stats["RMSE"].idxmax()
    ]

    worst_mse = effect_stats.loc[
        effect_stats["MSE"].idxmax()
    ]

    worst_effect = effect_stats.loc[
        effect_stats[
            "absolute_standardized_effect"
        ].idxmax()
    ]

    print("\n" + "=" * 90)
    print(f"{FEATURE.upper()} — HATA ANALİZİ")
    print("=" * 90)

    print(
        f"\nEn yüksek MAE   : {worst_mae[FEATURE]} → "
        f"{worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : {worst_rmse[FEATURE]} → "
        f"{worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En yüksek MSE   : {worst_mse[FEATURE]} → "
        f"{worst_mse['MSE']:,.2f}"
    )

    print(
        f"En güçlü mutlak standardized effect: "
        f"{worst_effect[FEATURE]} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    print(
        f"\nGözlem sayısı: "
        f"{int(worst_effect['count']):,}"
    )

    print(
        f"MAE: {worst_effect['MAE']:,.2f}"
    )

    print(
        f"RMSE: {worst_effect['RMSE']:,.2f}"
    )

    print(
        f"MSE: {worst_effect['MSE']:,.2f}"
    )

    if worst_effect["standardized_effect"] > 0:
        print(
            "\nYorum: Pozitif residual → Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )
    elif worst_effect["standardized_effect"] < 0:
        print(
            "\nYorum: Negatif residual → Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )
    else:
        print(
            "\nYorum: Bu kategoride belirgin "
            "sistematik residual biası yok."
        )

# EN GÜÇLÜ MUTLAK STANDARDIZED EFFECT
effect_stats["absolute_standardized_effect"] = (
    pd.to_numeric(
        effect_stats["standardized_effect"],
        errors="coerce"
    ).abs()
)

effect_ranking = (
    effect_stats[
        [
            FEATURE,
            "count",
            "mean_residual",
            "MAE",
            "RMSE",
            "MSE",
            "standardized_effect",
            "absolute_standardized_effect"
        ]
    ]
    .dropna(
        subset=["absolute_standardized_effect"]
    )
    .sort_values(
        "absolute_standardized_effect",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 90)
print("EN GÜÇLÜ MUTLAK STANDARDIZED EFFECT")
print("=" * 90)

display(
    effect_ranking.head(10).style.format({
        "count": "{:,.0f}",
        "mean_residual": "{:,.0f}",
        "MAE": "{:,.0f}",
        "RMSE": "{:,.0f}",
        "MSE": "{:,.0f}",
        "standardized_effect": "{:+.3f}",
        "absolute_standardized_effect": "{:.3f}"
    })
)

# ÖZET
print("\n" + "=" * 90)
print(f"{FEATURE.upper()} — ANALYSIS SUMMARY")
print("=" * 90)

print(
    f"Toplam geçerli gözlem : "
    f"{len(analysis_df):,}"
)

print(
    f"Kategori sayısı       : "
    f"{len(effect_stats):,}"
)

print(
    f"Minimum kategori N    : "
    f"{MIN_CATEGORY_COUNT}"
)

print(
    f"Genel Mean Residual   : "
    f"{analysis_df['residual'].mean():,.2f}"
)

print(
    f"Genel MAE             : "
    f"{analysis_df['absolute_error'].mean():,.2f}"
)

print(
    f"Genel RMSE            : "
    f"{np.sqrt(np.mean(analysis_df['residual'] ** 2)):,.2f}"
)

print(
    f"Genel MSE             : "
    f"{np.mean(analysis_df['residual'] ** 2):,.2f}"
)

print("=" * 90)
print("ANALİZ TAMAMLANDI")
print("=" * 90)

lot_status analizinde model AWARDED ihalelerde en stabil performansı gösterirken (MAE: 1.91M, standardized effect: +0.096), PREAWARDED grubunda hata belirgin şekilde artmaktadır (MAE: 2.53M, effect: +0.131). CANCELLED grubunda ise yalnızca 42 gözlem olmasına rağmen en yüksek hata (MAE: 4.98M, RMSE: 16.62M) ve en güçlü bias (+0.265) görülmektedir. Tüm gruplarda residual pozitif olduğu için model genel olarak under-prediction yapmaktadır. Ancak CANCELLED grubunun çok küçük örneklemli olması nedeniyle bu sonucu genellemek doğru değildir; burada öncelikle örneklem yetersizliği ve uç değerler kontrol edilmelidir. Genel olarak lot_status modelde anlamlı bir hata farklılaşması yaratıyor, fakat mevcut sonuçlar tek başına bu kolonda dönüşüm/değişiklik yapılmasını gerektirmiyor.

**Grafiklerin Detaylı AnaliziComponent + Residual Plots (AWARDED, PREAWARDED, CANCELLED):**

**AWARDED:** Verinin ezici çoğunluğunu (N=38,199) oluşturmaktadır. Tahmin edilen fiyat arttıkça turuncu trend çizgisi pozitif artık tarafında yukarı kaymakta, yani yüksek ihale tutarlarında model fiyatı eksik tahmin etmektedir (under-prediction).

**PREAWARDED:** N=6,334 gözlem barındırır. 0-50M USD bandındaki projelerde artıklar sıfır çizgisinde stabil seyrederken, 100 USD üzerindeki bütçelerde tekil noktalar saçılımı bozmaktadır.

**CANCELLED:** Yalnızca N=42 gözleme sahiptir. Aşırı düşük veri hacmine rağmen ortalama +4.16M artık ve 4.98 M MAE ile en yüksek hatayı üreten gruptur.

**Residual Diagnostics Paneli (A1 - A4):A1 (QQ Plot) & A3 (Distribution):** Artıklar 50M seviyesine kadar uzanan çok ağır bir sağ kuyruğa (right-skewed heavy tail) sahiptir; uç bütçelerde büyük under-prediction mevcuttur.

**A2 (Fits vs Density):** Gözlem yoğunluğu 0-10M USD bütçe aralığında sıfır artık etrafında mükemmel kümelenmiştir.

**A4 (Rolling Bias):** Tahmin sırası boyunca genel kütlede ortanca artık -0.01M civarında son derece dengelidir.

**Grafik B — Standardized Category Effects:** AWARDED (+0.096) ve PREAWARDED (+0.131) katsayılarıyla alt tarafta kalırken, CANCELLED kategorisi +0.265 standartlaştırılmış etki katsayısıyla doğrusal çizgiden saparak en belirgin pozitif bias'ı üretmiştir.

**Sütun Grafikleri (MAE ve MSE Comparison):**

MAE Comparison: AWARDED (1.91M}) ve PREAWARDED (2.53M}) makul seviyelerdeyken, CANCELLED kategorisi 4.98M ile ortalamanın iki katından fazla mutlak hataya sahiptir.

MSE Comparison: 276.24T USD^2 karesel hata ile CANCELLED sınıfı yine ilk sıradadır; ancak sınıflar arasındaki MSE farkı MAE kadar dramatik değildir, bu da karesel hatayı asıl domine eden unsurun genel Q_5 uç bütçeli ihaleler olduğunu göstermektedir.

**Analiz Paragrafı**

lot_status değişkenine ait residual analizi, modelin tamamlanmış (AWARDED) ve ön-kazanım aşamasındaki (PREAWARDED) ana ihalelerde kararlı çalıştığını, fakat iptal edilen ihalelerde (CANCELLED) ciddi bir performans kaybı yaşadığını göstermektedir. İptal edilen ihalelerin toplam veri setinde yalnızca 42 örneğe (N=42) sahip olması, modelin bu statüye özgü örüntüleri öğrenmesini imkansız kılmakta ve +4.16M ortalama artık ile 4.98textM MAE seviyesinde belirgin bir under-prediction yaratmaktadır. İptal süreçlerinin genellikle problemli, acil müdahale gerektiren veya bütçe/fiyat uyuşmazlığı yaşanan sıradışı ihalelerden oluşması, sabit ağaç yapılarının tek başına bu kategoriyi modellemesini zorlaştırmaktadır.

**ÇÖZÜM YÖNTEMİ**

lot_status sütununda yalnızca 42 gözlem içeren CANCELLED kategorisinin ürettiği aşırı yüksek hatayı (4.98M USD} MAE) ve pozitif bias sapmasını gidermek amacıyla Eşik Tabanlı Nadir Sınıf Birleştirme (Threshold-Based Rare Category Grouping) yöntemi uygulanmıştır. Eğitim setinde belirlenen frekans eşiğinin (N < 200) altında kalan bu yetersiz veri hacmine sahip sınıf, gürültü yaratmasını ve modelin uç değerleri ezberlemesini (overfitting) engellemek adına "RARE_STATUS" etiketi altında tek bir kategoride toplanmıştır. Veri sızıntısının (data leakage) önüne geçmek amacıyla eşikleşme kuralı sadece X_train verisi üzerinden hesaplanarak validation ve test setlerine birebir aktarılmış, böylece kategorinin seyreklik problemi çözülerek model daha istikrarlı bir yapıya kavuşturulmuştur.

## TENDER_SELECTIONMETHOD (İhalede kazanan teklifin hangi seçim yöntemiyle belirlendiğini ifade eden kategorik değişkendir.)

In [ ]:
# tender_selectionmethod — RESIDUAL / COMPONENT ANALYSIS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "tender_selectionmethod"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
ROLLING_WINDOW_MAX = 2000
CATEGORIES_PER_PAGE = 8

# KONTROLLER
if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

# ACTUAL / PREDICTION / RESIDUAL KOLONLARINI BUL
columns_lower = {col.lower(): col for col in test_predictions.columns}

actual_candidates = [
    "actual_usd", "actual", "y_true", "true",
    "target", "actual_price", "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd", "predicted", "prediction",
    "pred", "y_pred", "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd", "residual", "error", "prediction_error"
]

actual_col = next(
    (columns_lower[c] for c in actual_candidates if c in columns_lower),
    None
)

predicted_col = next(
    (columns_lower[c] for c in predicted_candidates if c in columns_lower),
    None
)

residual_col = next(
    (columns_lower[c] for c in residual_candidates if c in columns_lower),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

# SATIR KONTROLÜ
if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test          : {len(df_test):,}\n"
        f"test_predictions : {len(test_predictions):,}"
    )

# ANALİZ DATAFRAME
analysis_df = pd.DataFrame({
    FEATURE: df_test[FEATURE].to_numpy(),
    "actual": pd.to_numeric(
        test_predictions[actual_col], errors="coerce"
    ).to_numpy(),
    "prediction": pd.to_numeric(
        test_predictions[predicted_col], errors="coerce"
    ).to_numpy(),
    "residual": pd.to_numeric(
        test_predictions[residual_col], errors="coerce"
    ).to_numpy()
})

analysis_df[FEATURE] = (
    analysis_df[FEATURE]
    .astype(str)
    .replace({
        "None": "MISSING",
        "nan": "MISSING",
        "null": "MISSING"
    })
)

analysis_df["absolute_error"] = analysis_df["residual"].abs()

valid_mask = (
    np.isfinite(analysis_df["actual"].to_numpy())
    & np.isfinite(analysis_df["prediction"].to_numpy())
    & np.isfinite(analysis_df["residual"].to_numpy())
)

analysis_df = (
    analysis_df.loc[valid_mask]
    .reset_index(drop=True)
)

print(f"\nGeçerli gözlem sayısı: {len(analysis_df):,}")

# KATEGORİ İSTATİSTİKLERİ
category_stats = (
    analysis_df
    .groupby(FEATURE)
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        std_residual=("residual", "std"),
        MAE=("absolute_error", "mean"),
        MSE=("residual", lambda x: np.mean(x ** 2)),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

effect_stats = category_stats[
    category_stats["count"] >= MIN_CATEGORY_COUNT
].copy()

global_std = analysis_df["residual"].std()

effect_stats["standardized_effect"] = (
    effect_stats["mean_residual"] / global_std
)

effect_stats["absolute_standardized_effect"] = (
    effect_stats["standardized_effect"].abs()
)

effect_stats = (
    effect_stats
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(f"Analiz edilen kategori sayısı: {len(effect_stats)}")

# TABLO
visual_table = effect_stats[
    [
        FEATURE,
        "count",
        "mean_residual",
        "MAE",
        "MSE",
        "RMSE",
        "standardized_effect"
    ]
].copy()

visual_table.columns = [
    "Selection Method",
    "Observations",
    "Mean Residual",
    "MAE",
    "MSE",
    "RMSE",
    "Standardized Effect"
]

def highlight_max(series):
    mask = series == series.max()
    return [
        (
            "background-color: #ffd6d6;"
            "color: #8b0000;"
            "font-weight: bold;"
        )
        if value else ""
        for value in mask
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "{:,.0f}",
        "MAE": "{:,.0f}",
        "MSE": "{:,.0f}",
        "RMSE": "{:,.0f}",
        "Standardized Effect": "{:+.3f}"
    })
    .apply(highlight_max, subset=["MAE"])
    .apply(highlight_max, subset=["MSE"])
    .apply(highlight_max, subset=["RMSE"])
    .apply(highlight_max, subset=["Mean Residual"])
    .apply(highlight_max, subset=["Standardized Effect"])
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "11pt",
            "padding": "8px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#404040"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "9px")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("padding", "8px")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16pt"),
                ("font-weight", "bold"),
                ("padding", "10px")
            ]
        }
    ])
    .set_caption(f"{FEATURE.upper()} — RESIDUAL ANALYSIS")
)

display(styled_table)

# TEMEL VERİLER
residual_values = analysis_df["residual"].to_numpy()
prediction_values = analysis_df["prediction"].to_numpy()

# A1 — NORMAL PROBABILITY PLOT
fig, axes = plt.subplots(2, 2, figsize=(19, 14))
ax1, ax2, ax3, ax4 = axes.flatten()

if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(42)
    residual_plot = rng.choice(
        residual_values, size=PLOT_SAMPLE, replace=False
    )
else:
    residual_plot = residual_values

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - 0.5) / n
)

residual_mean = np.mean(sorted_residuals)
residual_std = np.std(sorted_residuals, ddof=1)

standardized_residuals = (
    sorted_residuals - residual_mean
) / residual_std

lower_percentile = np.percentile(
    standardized_residuals, 0.5
)

upper_percentile = np.percentile(
    standardized_residuals, 99.5
)

a1_mask = (
    (standardized_residuals >= lower_percentile)
    & (standardized_residuals <= upper_percentile)
)

x_a1 = theoretical_quantiles[a1_mask]
y_a1 = standardized_residuals[a1_mask]

ax1.scatter(
    x_a1, y_a1,
    s=14,
    alpha=0.22,
    rasterized=True
)

x_line = np.array([x_a1.min(), x_a1.max()])

ax1.plot(
    x_line,
    x_line,
    linestyle="--",
    linewidth=2.5,
    label="Normal reference"
)

ax1.set_yscale(
    "symlog",
    linthresh=1,
    linscale=1.2
)

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5,
    color="black"
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual\n(Symlog Scale)",
    fontsize=13
)

ax1.set_title(
    "A1 — Normal Probability Plot\nCentral 99%",
    fontsize=16,
    fontweight="bold"
)

ax1.grid(
    which="major",
    alpha=0.25,
    linestyle="--"
)

ax1.grid(
    which="minor",
    alpha=0.10,
    linestyle=":"
)

ax1.legend(
    loc="upper left",
    fontsize=10
)

# A2 — RESIDUALS VS FITS — HEXBIN
prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(
    prediction_m, [0.5, 99.5]
)

y_low, y_high = np.percentile(
    residual_m, [0.5, 99.5]
)

plot_mask = (
    (prediction_m >= x_low)
    & (prediction_m <= x_high)
    & (residual_m >= y_low)
    & (residual_m <= y_high)
)

x_plot = prediction_m[plot_mask]
y_plot = residual_m[plot_mask]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=45,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=0.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual = np.median(y_plot)

ax2.axhline(
    median_residual,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=0.02
)

cbar.set_label(
    "Observation Density\n(log scale)",
    fontsize=12
)

ax2.set_xlabel(
    "Predicted Price (Million USD)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Residuals Versus Fits\nDensity-Based Visualization",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(
    alpha=0.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)

# A3 — RESIDUAL DISTRIBUTION
residual_low, residual_high = np.percentile(
    residual_values, [0.5, 99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

ax3.hist(
    central_residuals / 1e6,
    bins=35,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.5
)

mean_residual = np.mean(residual_values) / 1e6

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Residual = 0"
)

ax3.axvline(
    mean_residual,
    linestyle=":",
    linewidth=2,
    label=f"Mean = {mean_residual:.2f}M"
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Residual Distribution\nCentral 99%",
    fontsize=16,
    fontweight="bold"
)

ax3.legend(fontsize=10)
ax3.grid(axis="y", alpha=0.20)

# A4 — RESIDUALS VS OBSERVATION ORDER
order = np.arange(1, len(analysis_df) + 1)

residual_order = (
    analysis_df["residual"].to_numpy() / 1e6
)

q_low, q_high = np.percentile(
    residual_order, [0.5, 99.5]
)

plot_mask = (
    (residual_order >= q_low)
    & (residual_order <= q_high)
)

order_plot = order[plot_mask]
residual_plot = residual_order[plot_mask]

ax4.scatter(
    order_plot,
    residual_plot,
    s=4,
    alpha=0.055,
    rasterized=True
)

order_df = pd.DataFrame({
    "order": order_plot,
    "residual": residual_plot
})

WINDOW = max(
    500,
    min(
        ROLLING_WINDOW_MAX,
        len(order_df) // 20
    )
)

min_periods = max(50, WINDOW // 4)

rolling_median = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .median()
)

rolling_q10 = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .quantile(0.10)
)

rolling_q90 = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .quantile(0.90)
)

ax4.fill_between(
    order_df["order"].to_numpy(),
    rolling_q10.to_numpy(),
    rolling_q90.to_numpy(),
    alpha=0.20,
    label="Rolling 10–90% error range"
)

ax4.plot(
    order_df["order"].to_numpy(),
    rolling_median.to_numpy(),
    linewidth=3,
    label=f"Rolling Median ({WINDOW:,})"
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2.2,
    color="red",
    label="Residual = 0"
)

ax4.axhline(
    1,
    linestyle=":",
    linewidth=1.2,
    color="gray",
    alpha=0.75,
    label="±$1M reference"
)

ax4.axhline(
    -1,
    linestyle=":",
    linewidth=1.2,
    color="gray",
    alpha=0.75
)

ax4.axhline(
    5,
    linestyle=":",
    linewidth=1,
    color="gray",
    alpha=0.45
)

ax4.axhline(
    -5,
    linestyle=":",
    linewidth=1,
    color="gray",
    alpha=0.45
)

global_median = np.median(residual_order)

ax4.axhline(
    global_median,
    linestyle="-.",
    linewidth=1.6,
    color="black",
    alpha=0.75,
    label=f"Global Median = {global_median:+.2f}M"
)

ax4.set_yscale(
    "symlog",
    linthresh=1,
    linscale=1.2
)

ax4.set_xlabel(
    "Observation Order",
    fontsize=13
)

ax4.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax4.set_title(
    "A4 — Residuals Versus Observation Order\nRolling Bias",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(
    which="major",
    alpha=0.25,
    linestyle="--"
)

ax4.grid(
    which="minor",
    alpha=0.10,
    linestyle=":"
)

ax4.legend(
    loc="upper right",
    fontsize=9
)

positive_ratio = np.mean(residual_order > 0) * 100
negative_ratio = np.mean(residual_order < 0) * 100
median_abs_error = np.median(np.abs(residual_order))

info_text = (
    f"Central 99% shown\n"
    f"Rolling Window: {WINDOW:,}\n"
    f"Positive: {positive_ratio:.1f}%\n"
    f"Negative: {negative_ratio:.1f}%\n"
    f"Median |Residual|: {median_abs_error:.2f}M"
)

ax4.text(
    0.02,
    0.04,
    info_text,
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.90
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — RESIDUAL DIAGNOSTICS",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close()

# B — STANDARDIZED EFFECTS
effects = effect_stats["standardized_effect"].to_numpy()
effects_sorted = np.sort(effects)
m = len(effects_sorted)

if m >= 2:
    theoretical_effect_quantiles = stats.norm.ppf(
        (np.arange(1, m + 1) - 0.5) / m
    )

    fig, ax = plt.subplots(figsize=(14, 9))

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=100,
        alpha=0.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope * x_ref,
        linestyle="--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():
        idx = np.argmin(
            np.abs(
                effects_sorted -
                row["standardized_effect"]
            )
        )

        label = str(row[FEATURE])

        if len(label) > 25:
            label = label[:22] + "..."

        ax.annotate(
            label,
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(8, 6),
            textcoords="offset points",
            fontsize=10
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=1.8
    )

    ax.set_xlabel(
        "Theoretical Normal Quantiles",
        fontsize=13
    )

    ax.set_ylabel(
        "Standardized Effect",
        fontsize=13
    )

    ax.set_title(
        f"B — Standardized {FEATURE} Effects",
        fontsize=17,
        fontweight="bold"
    )

    ax.grid(alpha=0.20)

    plt.tight_layout()
    plt.show()
    plt.close()

# C — COMPONENT + RESIDUAL
categories = effect_stats[FEATURE].tolist()

for page_start in range(
    0,
    len(categories),
    CATEGORIES_PER_PAGE
):
    page_categories = categories[
        page_start:
        page_start + CATEGORIES_PER_PAGE
    ]

    n_categories = len(page_categories)
    ncols = 2
    nrows = int(np.ceil(n_categories / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(18, 7 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, category in zip(
        axes,
        page_categories
    ):
        category_data = analysis_df[
            analysis_df[FEATURE] == category
        ].copy()

        x = (
            category_data["prediction"].to_numpy()
            / 1e6
        )

        y = (
            category_data["residual"].to_numpy()
            / 1e6
        )

        if len(x) >= 10:
            x_low, x_high = np.percentile(x, [1, 99])
            y_low, y_high = np.percentile(y, [1, 99])

            mask = (
                (x >= x_low)
                & (x <= x_high)
                & (y >= y_low)
                & (y <= y_high)
            )

            x_plot = x[mask]
            y_plot = y[mask]
        else:
            x_plot = x
            y_plot = y

        if len(x_plot) > 12000:
            rng = np.random.default_rng(42)

            sample_idx = rng.choice(
                len(x_plot),
                size=12000,
                replace=False
            )

            x_scatter = x_plot[sample_idx]
            y_scatter = y_plot[sample_idx]
        else:
            x_scatter = x_plot
            y_scatter = y_plot

        ax.scatter(
            x_scatter,
            y_scatter,
            s=12,
            alpha=0.15,
            rasterized=True
        )

        if len(x_plot) >= 3:
            slope, intercept = np.polyfit(
                x_plot,
                y_plot,
                1
            )

            x_line = np.linspace(
                x_plot.min(),
                x_plot.max(),
                300
            )

            ax.plot(
                x_line,
                intercept + slope * x_line,
                linestyle="--",
                linewidth=2.5,
                label="Linear fit"
            )

            sort_idx = np.argsort(x_plot)
            x_sorted = x_plot[sort_idx]
            y_sorted = y_plot[sort_idx]

            component_df = pd.DataFrame({
                "x": x_sorted,
                "y": y_sorted
            })

            smooth_window = max(
                50,
                min(
                    500,
                    len(component_df) // 20
                )
            )

            rolling_component = (
                component_df["y"]
                .rolling(
                    smooth_window,
                    center=True,
                    min_periods=max(
                        20,
                        smooth_window // 4
                    )
                )
                .median()
            )

            ax.plot(
                x_sorted,
                rolling_component,
                linewidth=3,
                label="Smoothed trend"
            )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=2.2,
            color="red",
            label="Residual = 0"
        )

        ax.set_xlabel(
            "Predicted Price (Million USD)",
            fontsize=13
        )

        ax.set_ylabel(
            "Residual (Million USD)",
            fontsize=13
        )

        ax.set_title(
            str(category),
            fontsize=15,
            fontweight="bold",
            pad=12
        )

        ax.grid(
            alpha=0.22,
            linestyle="--"
        )

        row = effect_stats[
            effect_stats[FEATURE] == category
        ].iloc[0]

        mean_res = row["mean_residual"] / 1e6
        mae = row["MAE"] / 1e6
        mse = row["MSE"] / 1e12
        rmse = row["RMSE"] / 1e6
        effect = row["standardized_effect"]

        direction = (
            "UNDER-PREDICTION"
            if effect > 0
            else "OVER-PREDICTION"
        )

        metrics_text = (
            f"N            : {int(row['count']):,}\n"
            f"Mean Residual: {mean_res:.2f}M\n"
            f"MAE          : {mae:.2f}M\n"
            f"MSE          : {mse:.2f}M²\n"
            f"RMSE         : {rmse:.2f}M\n"
            f"Std. Effect  : {effect:+.3f}\n"
            f"Direction    : {direction}"
        )

        ax.text(
            0.03,
            0.97,
            metrics_text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            linespacing=1.4,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92
            )
        )

        ax.legend(
            loc="lower right",
            fontsize=9,
            frameon=True,
            framealpha=0.90
        )

    for ax in axes[n_categories:]:
        ax.axis("off")

    page_number = (
        page_start // CATEGORIES_PER_PAGE
    ) + 1

    total_pages = int(
        np.ceil(
            len(categories) /
            CATEGORIES_PER_PAGE
        )
    )

    fig.suptitle(
        f"COMPONENT + RESIDUAL PLOTS — {FEATURE}\n"
        f"Page {page_number}/{total_pages}",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
    plt.close()

# D — MAE COMPARISON
mae_plot = effect_stats.sort_values(
    "MAE",
    ascending=True
)

fig, ax = plt.subplots(figsize=(15, 8))

bars = ax.bar(
    mae_plot[FEATURE].astype(str),
    mae_plot["MAE"] / 1e6
)

ax.set_xlabel(FEATURE, fontsize=13)
ax.set_ylabel("MAE (Million USD)", fontsize=13)

ax.set_title(
    f"{FEATURE} — MAE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    labelsize=9,
    rotation=60
)

ax.tick_params(
    axis="y",
    labelsize=11
)

max_mae = (mae_plot["MAE"] / 1e6).max()

for bar, value in zip(
    bars,
    mae_plot["MAE"] / 1e6
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mae * 0.015,
        f"{value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=90
    )

ax.grid(axis="y", alpha=0.20)
plt.tight_layout()
plt.show()
plt.close()

# E — MSE COMPARISON
mse_plot = effect_stats.sort_values(
    "MSE",
    ascending=True
)

fig, ax = plt.subplots(figsize=(15, 8))

bars = ax.bar(
    mse_plot[FEATURE].astype(str),
    mse_plot["MSE"] / 1e12
)

ax.set_xlabel(FEATURE, fontsize=13)
ax.set_ylabel("MSE (Million USD)²", fontsize=13)

ax.set_title(
    f"{FEATURE} — MSE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    labelsize=9,
    rotation=60
)

ax.tick_params(
    axis="y",
    labelsize=11
)

max_mse = (mse_plot["MSE"] / 1e12).max()

for bar, value in zip(
    bars,
    mse_plot["MSE"] / 1e12
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mse * 0.015,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=90
    )

ax.grid(axis="y", alpha=0.20)
plt.tight_layout()
plt.show()
plt.close()

# EN PROBLEMLİ KATEGORİLER
if len(effect_stats) > 0:
    worst_mae = effect_stats.loc[
        effect_stats["MAE"].idxmax()
    ]

    worst_mse = effect_stats.loc[
        effect_stats["MSE"].idxmax()
    ]

    worst_rmse = effect_stats.loc[
        effect_stats["RMSE"].idxmax()
    ]

    worst_effect = effect_stats.loc[
        effect_stats["absolute_standardized_effect"].idxmax()
    ]

    print("\n" + "=" * 80)
    print(f"{FEATURE.upper()} — HATA ANALİZİ")
    print("=" * 80)

    print(
        f"\nEn yüksek MAE   : "
        f"{worst_mae[FEATURE]} → "
        f"{worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek MSE   : "
        f"{worst_mse[FEATURE]} → "
        f"{worst_mse['MSE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : "
        f"{worst_rmse[FEATURE]} → "
        f"{worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En güçlü etki   : "
        f"{worst_effect[FEATURE]} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    if worst_effect["standardized_effect"] > 0:
        print(
            "\nYorum: Pozitif residual → "
            "Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )
    else:
        print(
            "\nYorum: Negatif residual → "
            "Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )

tender_selectionmethod (İhale Seçim Yöntemi) değişkenine göre gerçekleştirilen artık analizi, modelin performansının ihale kriterinin yapısına bağlı olarak ciddi düzeyde ayrıştığını ve genel olarak her iki kategoride de sistematik bir eksik tahmin (under-prediction) sergilediğini ortaya koymaktadır.

**Residual Diagnostik Analizi (A1 – A4)**

A1 (Normal Probability Plot) ve A3 (Residual Distribution): Residual dağılımları normal dağılım referans çizgisinden tamamen saparak belirgin bir sağ kuyruk (right-skewed heavy tail) yapmaktadır; ortalama artık değerinin +1.60M seviyesinde pozitif kalması, modelin ihale bedellerini genel olarak gerçek değerlerin altında tahmin ettiğini doğrular.
A2 (Residuals vs Fits): Tahmin edilen tutar arttıkça artıkların dikey saçılımının genişlediği heteroskedastik (değişen varyanslı) bir yapı görünmektedir.
A4 (Rolling Bias): 2.000 gözlemlik hareketli ortancanın (Rolling Median = -0.01M) sıfır çizgisi ekseninde yatay ilerlemesi, veri sıralamasına bağlı dönemsel bir kayma olmadığını, hatanın kategorik ve bütçesel ölçek sapmalarından beslendiğini gösterir.

**Kategorik Performans ve Hatanın Yoğunlaştığı Bölgeler**

LOWEST_PRICE (En Düşük Fiyat Kriteri - 25,201 Gözlem): Verinin %56.5'ini oluşturan bu grup, modelin en yüksek sapmayı ve en aşırı hatayı ürettiği alandır. MAE = 2.13M, RMSE = 18.58M, MSE = 345.36T ve Std. Effect = +0.112 değerleriyle tüm metriklerde ilk sıradadır. Component + Residual grafiği incelendiğinde, bu kategoride tahmin ekseninin 200M sınırına kadar uzandığı ve gerçek ihale bedellerinin tahminlerin 30M seviyesine kadar üzerine çıktığı aşırı uç değerler (outliers) göze çarpmaktadır. Model, sadece fiyat odaklı rekabete dayalı bu ihalelerdeki yüksek tutarlı sözleşme ölçeklerini yukarıya doğru esnetememekte ve ağır bir under-prediction yaratmaktadır.
MEAT (Most Economically Advantageous Tender / Ekonomik Açıdan En Avantajlı Teklif - 19,374 Gözlem): Niteliğin, teknik puanın ve fiyat dışı unsurların da işin içine girdiği bu kategoride model çok daha kararlı sonuçlar vermiştir. MAE = 1.83M, RMSE = 11.14M, MSE = 124.20T ve Std. Effect = +0.088 ile LOWEST_PRICE kategorisine kıyasla varyansını neredeyse üçte bir oranına indirmeyi başarmıştır. Bu grupta tahminler 25M ile daha dar bir ölçek bandında sıkıştığı için karesel ceza metrikleri (MSE/RMSE) düşük kalmıştır.

**Modelin Hata Kök Sebepleri**

Ölçek Esnekliği Eksikliği (LOWEST_PRICE Tarafında): En düşük fiyat usulüyle yapılan ihalelerin bir kısmı çok devasa bütçelere ulaşabilmektedir. Model, teklif seçim yöntemini tek başına aldığında bu bütçesel sıçramaları yakalamakta yetersiz kalmakta ve fiyatı ortalamaya doğru bastırarak büyük tutarlı ihalelerde yüksek artıklar (residual > 20M) üretmektedir.
Karesel Cezaların Sebebi (Outlier Etkisi): LOWEST_PRICE MSE değerinin (345.36T), MEAT MSE değerinden (124.20T) yaklaşık 2.8 kat yüksek olmasının temel sebebi, sadece ortalama hata yüksekliği değil, LOWEST_PRICE içinde yer alan ekstrem yüksek bütçeli münferit ihalelerin karesel hataları katlamasıdır.
Doğrusal Olmayan Eğilim (Smoothed Trend Flattening): Her iki kategoride de Smoothed Trend (turuncu çizgi) sıfır ekseninin üzerinde pozitif bir trend izledikten sonra high-value bölgesinde düzleşmektedir. Bu durum, modelin tutar büyüdükçe fiyatlama dinamiklerindeki doğrusal olmayan (non-linear) ilişkileri öğrenemediğini açıkça kanıtlar.

**Çözüm Yöntemi**

tender_selectionmethod değişkeninde N=25.201 gözlem içeren LOWEST_PRICE kategorisinin yüksek ihale bütçelerinde yarattığı under-prediction sapmasını ve karesel hatayı (345.36T MSE) gidermek amacıyla Örneklem Ağırlıklandırma (Sample Weighting) yöntemi uygulanmıştır. Bu yöntem, veri hacmi zaten yüksek olan kategorilerde yeni bir sütun türetmeden veya veri yapısını bozmadan, modelin kayıp fonksiyonunda (loss function) yüksek bütçeli ve yüksek hatalı LOWEST_PRICE örneklerine daha fazla ceza katsayısı verilmesi ilkesine dayanır. En yüksek bütçe segmentindeki LOWEST_PRICE ihalelerine daha yüksek ağırlık (sample_weight) atanması, karar ağaçlarının dallanma esnasında bu spesifik gruba daha fazla öncelik vermesini sağlar; böylece az verili kategoriler etkilenmeden, LOWEST_PRICE altındaki sistematik fiyat altı tahmin hatası doğrudan düzeltilerek genel MSE ve MAE değerleri optimize edilir.

## LOG_TENDER_RECORDEDBİDSCOUNT (İhaleye sunulan ve sisteme resmi olarak kaydedilen geçerli teklif sayısının doğal/standart logaritmik dönüşümüdür.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "log_tender_recordedbidscount"
PLOT_SAMPLE = 10000
N_BINS = 30
MIN_BIN_COUNT = 30

if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {
    str(col).lower(): col
    for col in test_predictions.columns
}

actual_candidates = [
    "actual_usd",
    "actual",
    "y_true",
    "true",
    "target",
    "actual_price",
    "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd",
    "predicted",
    "prediction",
    "pred",
    "y_pred",
    "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd",
    "residual",
    "error",
    "prediction_error"
]

actual_col = next(
    (
        columns_lower[c]
        for c in actual_candidates
        if c in columns_lower
    ),
    None
)

predicted_col = next(
    (
        columns_lower[c]
        for c in predicted_candidates
        if c in columns_lower
    ),
    None
)

residual_col = next(
    (
        columns_lower[c]
        for c in residual_candidates
        if c in columns_lower
    ),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test: {len(df_test):,}\n"
        f"test_predictions: {len(test_predictions):,}"
    )

def to_numpy_numeric(x):
    if isinstance(x, pd.Series):
        return pd.to_numeric(
            x,
            errors="coerce"
        ).to_numpy()

    return pd.to_numeric(
        np.asarray(x),
        errors="coerce"
    )

numeric_df = pd.DataFrame({
    FEATURE: to_numpy_numeric(df_test[FEATURE]),
    "actual": to_numpy_numeric(
        test_predictions[actual_col]
    ),
    "prediction": to_numpy_numeric(
        test_predictions[predicted_col]
    ),
    "residual": to_numpy_numeric(
        test_predictions[residual_col]
    )
})

numeric_df["absolute_error"] = numeric_df["residual"].abs()

valid_mask = (
    np.isfinite(numeric_df[FEATURE].to_numpy())
    & np.isfinite(numeric_df["actual"].to_numpy())
    & np.isfinite(numeric_df["prediction"].to_numpy())
    & np.isfinite(numeric_df["residual"].to_numpy())
)

numeric_df = numeric_df.loc[
    valid_mask
].reset_index(drop=True)

if len(numeric_df) < 10:
    raise ValueError(
        "Residual analizi için yeterli geçerli gözlem yok."
    )

print(
    f"\nGeçerli gözlem sayısı: "
    f"{len(numeric_df):,}"
)

feature_values = numeric_df[FEATURE].to_numpy()
residual_values = numeric_df["residual"].to_numpy()

residual_mean = np.mean(residual_values)
residual_median = np.median(residual_values)

mae = np.mean(
    np.abs(residual_values)
)

rmse = np.sqrt(
    np.mean(residual_values ** 2)
)

print(f"\n{FEATURE} İSTATİSTİKLERİ")
print(f"Minimum : {np.min(feature_values):.4f}")
print(f"Maximum : {np.max(feature_values):.4f}")
print(f"Mean    : {np.mean(feature_values):.4f}")
print(f"Median  : {np.median(feature_values):.4f}")

print("\nRESIDUAL İSTATİSTİKLERİ")
print(f"Mean Residual : ${residual_mean:,.2f}")
print(f"Median        : ${residual_median:,.2f}")
print(f"MAE           : ${mae:,.2f}")
print(f"RMSE          : ${rmse:,.2f}")

fig, axes = plt.subplots(
    2,
    2,
    figsize=(20, 15)
)

ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — NORMAL PROBABILITY / QQ PLOT

if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(42)
    residual_plot = rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
else:
    residual_plot = residual_values

sorted_residuals = np.sort(
    residual_plot
)

n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - 0.5) / n
)

plot_mean = np.mean(
    sorted_residuals
)

plot_std = np.std(
    sorted_residuals,
    ddof=1
)

if plot_std == 0 or not np.isfinite(plot_std):
    plot_std = 1.0

standardized_residuals = (
    sorted_residuals - plot_mean
) / plot_std

low, high = np.percentile(
    standardized_residuals,
    [0.5, 99.5]
)

mask_a1 = (
    (standardized_residuals >= low)
    & (standardized_residuals <= high)
)

x_a1 = theoretical_quantiles[mask_a1]
y_a1 = standardized_residuals[mask_a1]

ax1.scatter(
    x_a1,
    y_a1,
    s=14,
    alpha=.22,
    rasterized=True
)

x_line = np.array([
    x_a1.min(),
    x_a1.max()
])

ax1.plot(
    x_line,
    x_line,
    "--",
    linewidth=2.5,
    label="Normal reference"
)

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5,
    color="black",
    alpha=.8
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual",
    fontsize=13
)

ax1.set_title(
    "A1 — Normal Probability Plot\n"
    "Central 99% of Standardized Residuals",
    fontsize=16,
    fontweight="bold"
)

ax1.grid(
    alpha=.25,
    linestyle="--"
)

ax1.legend(
    loc="upper left",
    fontsize=10
)

# A2 — FEATURE VS RESIDUAL

residual_plot_m = residual_values / 1e6

x_low, x_high = np.percentile(
    feature_values,
    [.5, 99.5]
)

y_low, y_high = np.percentile(
    residual_plot_m,
    [.5, 99.5]
)

mask_a2 = (
    (feature_values >= x_low)
    & (feature_values <= x_high)
    & (residual_plot_m >= y_low)
    & (residual_plot_m <= y_high)
)

x_plot = feature_values[mask_a2]
y_plot = residual_plot_m[mask_a2]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=45,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual_m = np.median(
    y_plot
)

ax2.axhline(
    median_residual_m,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Median = {median_residual_m:+.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=.02
)

cbar.set_label(
    "Observation Density\n(log scale)",
    fontsize=12
)

ax2.set_xlabel(
    FEATURE,
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    f"A2 — {FEATURE} vs Residual\n"
    "Density-Based Visualization",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(
    alpha=.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)

# A3 — RESIDUAL DISTRIBUTION

residual_low, residual_high = np.percentile(
    residual_values,
    [.5, 99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

central_m = central_residuals / 1e6

ax3.hist(
    central_m,
    bins=35,
    alpha=.75,
    edgecolor="black",
    linewidth=.5
)

mean_residual_m = residual_mean / 1e6

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax3.axvline(
    mean_residual_m,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Mean = {mean_residual_m:+.2f}M"
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Residual Distribution\n"
    "Central 99% of Observations",
    fontsize=16,
    fontweight="bold"
)

ax3.legend(
    fontsize=10
)

ax3.grid(
    axis="y",
    alpha=.20
)

# A4 — FIXED-WIDTH BIN RESIDUAL TREND

feature_min = np.min(feature_values)
feature_max = np.max(feature_values)

bin_edges = np.linspace(
    feature_min,
    feature_max,
    N_BINS + 1
)

numeric_df["feature_bin"] = pd.cut(
    numeric_df[FEATURE],
    bins=bin_edges,
    include_lowest=True,
    duplicates="drop"
)

bin_stats = (
    numeric_df
    .groupby(
        "feature_bin",
        observed=True
    )
    .agg(
        observations=("residual", "size"),
        x_median=(FEATURE, "median"),
        median_residual=("residual", "median"),
        mean_residual=("residual", "mean"),
        q10=("residual", lambda x: np.quantile(x, .10)),
        q90=("residual", lambda x: np.quantile(x, .90))
    )
    .reset_index()
)

bin_stats = bin_stats[
    bin_stats["observations"] >= MIN_BIN_COUNT
].copy()

if len(bin_stats) >= 2:

    x_a4 = bin_stats["x_median"].to_numpy()
    median_a4 = (
        bin_stats["median_residual"].to_numpy()
        / 1e6
    )

    mean_a4 = (
        bin_stats["mean_residual"].to_numpy()
        / 1e6
    )

    q10_a4 = (
        bin_stats["q10"].to_numpy()
        / 1e6
    )

    q90_a4 = (
        bin_stats["q90"].to_numpy()
        / 1e6
    )

    counts_a4 = bin_stats[
        "observations"
    ].to_numpy()

    ax4.fill_between(
        x_a4,
        q10_a4,
        q90_a4,
        alpha=.20,
        label="Q10–Q90 residual range"
    )

    ax4.plot(
        x_a4,
        median_a4,
        marker="o",
        markersize=5,
        linewidth=3,
        label="Median residual"
    )

    ax4.plot(
        x_a4,
        mean_a4,
        marker=".",
        linewidth=1.8,
        linestyle="--",
        alpha=.8,
        label="Mean residual"
    )

    ax4.axhline(
        0,
        linestyle="--",
        linewidth=2.2,
        color="red",
        label="Residual = 0"
    )

    # Central 99% Y zoom
    a4_y_low, a4_y_high = np.percentile(
        residual_values / 1e6,
        [.5, 99.5]
    )

    padding = (
        a4_y_high - a4_y_low
    ) * .08

    if padding == 0:
        padding = 1

    ax4.set_ylim(
        a4_y_low - padding,
        a4_y_high + padding
    )

    ax4.set_xlabel(
        FEATURE,
        fontsize=13
    )

    ax4.set_ylabel(
        "Residual (Million USD)",
        fontsize=13
    )

    ax4.set_title(
        f"A4 — Residual Trend by {FEATURE}\n"
        f"Fixed-Width Bins | Minimum {MIN_BIN_COUNT} Observations",
        fontsize=16,
        fontweight="bold"
    )

    ax4.grid(
        alpha=.22,
        linestyle="--"
    )

    ax4.legend(
        loc="upper right",
        fontsize=9
    )

    ax4.text(
        .02,
        .04,
        f"Valid bins: {len(bin_stats)}\n"
        f"Min observations/bin: {MIN_BIN_COUNT}\n"
        f"Y-range: {a4_y_low:+.2f}M to {a4_y_high:+.2f}M",
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="bottom",
        bbox=dict(
            boxstyle="round,pad=.5",
            facecolor="white",
            edgecolor="gray",
            alpha=.9
        )
    )

else:
    ax4.text(
        .5,
        .5,
        "Yeterli gözleme sahip bin bulunamadı.",
        transform=ax4.transAxes,
        ha="center",
        va="center",
        fontsize=14
    )

    ax4.set_title(
        f"A4 — Residual Trend by {FEATURE}",
        fontsize=16,
        fontweight="bold"
    )

fig.suptitle(
    f"{FEATURE.upper()} — NUMERIC RESIDUAL DIAGNOSTICS",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, .95]
)

plt.show()
plt.close()

# HATA ÖZETİ

if len(bin_stats) > 0:

    worst_bias = bin_stats.loc[
        bin_stats["mean_residual"].abs().idxmax()
    ]

    worst_q90 = bin_stats.loc[
        bin_stats["q90"].idxmax()
    ]

    worst_q10 = bin_stats.loc[
        bin_stats["q10"].idxmin()
    ]

    print("\n" + "=" * 80)
    print(f"{FEATURE.upper()} — RESIDUAL HATA ANALİZİ")
    print("=" * 80)

    print(
        f"\nEn güçlü bias : "
        f"x={worst_bias['x_median']:.4f} → "
        f"${worst_bias['mean_residual']:,.2f}"
    )

    print(
        f"En yüksek Q90 : "
        f"x={worst_q90['x_median']:.4f} → "
        f"${worst_q90['q90']:,.2f}"
    )

    print(
        f"En düşük Q10  : "
        f"x={worst_q10['x_median']:.4f} → "
        f"${worst_q10['q10']:,.2f}"
    )

positive_ratio = (
    np.mean(residual_values > 0) * 100
)

negative_ratio = (
    np.mean(residual_values < 0) * 100
)

print(
    f"\nPozitif residual oranı : "
    f"{positive_ratio:.2f}%"
)

print(
    f"Negatif residual oranı : "
    f"{negative_ratio:.2f}%"
)

if residual_mean > 0:
    print(
        "\nGenel yorum: Pozitif residual → "
        "Actual > Prediction → "
        "model genel olarak UNDER-PREDICTION yapıyor."
    )
elif residual_mean < 0:
    print(
        "\nGenel yorum: Negatif residual → "
        "Actual < Prediction → "
        "model genel olarak OVER-PREDICTION yapıyor."
    )
else:
    print(
        "\nGenel yorum: Ortalama residual yaklaşık 0."
    )

**Model Hata Analizi**

**A1 — Normal Probability Plot**

Residual dağılımının normal dağılımdan belirgin şekilde saptığı görülüyor. Özellikle sağ kuyruktaki güçlü yükseliş, aşırı büyük pozitif residualların bulunduğunu gösteriyor. Bu durum modelin bazı yüksek tutarlı ihalelerde gerçek değeri yeterince yakalayamadığını ve under-prediction yaptığını düşündürüyor.

**A2 — Residuals vs log_tender_recordedbidscount**

Hataların en yoğun olduğu bölge log_tender_recordedbidscount ≈ 0.69 seviyesinde. Bu bölge düşük teklif sayısına sahip ihaleleri temsil ediyor. Buradaki geniş residual dağılımı, modelin az teklif alan ihalelerde daha yüksek ve değişken hata ürettiğini gösteriyor. Teklif sayısı arttıkça residual dağılımı genel olarak daralıyor.

**A3 — Residual Distribution**

Residual dağılımı sağa çarpık durumda. Gözlemlerin büyük bölümü sıfır çevresinde toplanırken, az sayıdaki çok büyük pozitif hata ortalamayı yukarı çekiyor. Bu nedenle ortalama residual, medyan residualdan belirgin şekilde yüksek.

**A4 — Residual Trend**

Teklif sayısı düşükken residual varyansı oldukça yüksek. log_tender_recordedbidscount arttıkça genel olarak hata bandı daralıyor ve model daha istikrarlı hale geliyor. Ancak veri sayısının az olduğu yüksek değerli bölgelerde bazı büyük hatalar görülüyor; bu nedenle bu bölgelerdeki sıçramalar genel bir trend olarak değil, seyrek gözlemlerin etkisi olarak değerlendirilmelidir.

**Genel Sonuç**

Modelin temel problemi düşük teklif sayısına sahip ve yüksek tutarlı ihalelerdeki büyük hatalar. Residualların sağ kuyruğundaki yapı, modelin özellikle uç değerlerde under-prediction eğiliminde olduğunu gösteriyor. Bu nedenle modelin genel performansı iyi olsa da, yüksek değerli ihalelerde hata kontrolü geliştirilmesi gereken temel noktadır.

## LOG_TENDER_LOTSCOUNT (İhaledeki lot sayısının logaritmik dönüşüm uygulanmış değerini ifade eden sayısal değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "log_tender_lotscount"
PLOT_SAMPLE = 10000
RANDOM_STATE = 42

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır.")
if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")
if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {str(c).lower(): c for c in test_predictions.columns}

actual_candidates = ["actual_usd","actual","y_true","true","target","actual_price","actual_tender_finalpriceusd"]
predicted_candidates = ["predicted_usd","predicted","prediction","pred","y_pred","predicted_price","predicted_tender_finalpriceusd"]
residual_candidates = ["residual_usd","residual","error","prediction_error"]

actual_col = next((columns_lower[c] for c in actual_candidates if c in columns_lower),None)
predicted_col = next((columns_lower[c] for c in predicted_candidates if c in columns_lower),None)
residual_col = next((columns_lower[c] for c in residual_candidates if c in columns_lower),None)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")
if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")
if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(f"Satır sayıları uyuşmuyor!\ndf_test: {len(df_test):,}\ntest_predictions: {len(test_predictions):,}")

def to_numpy_numeric(x):
    if isinstance(x,pd.Series):
        return pd.to_numeric(x,errors="coerce").to_numpy(dtype=float)
    return pd.to_numeric(np.asarray(x),errors="coerce").astype(float)

numeric_df = pd.DataFrame({
    FEATURE: to_numpy_numeric(df_test[FEATURE]),
    "actual": to_numpy_numeric(test_predictions[actual_col]),
    "prediction": to_numpy_numeric(test_predictions[predicted_col]),
    "residual": to_numpy_numeric(test_predictions[residual_col])
})

numeric_df["absolute_error"] = numeric_df["residual"].abs()

valid_mask = (
    np.isfinite(numeric_df[FEATURE].to_numpy())
    & np.isfinite(numeric_df["actual"].to_numpy())
    & np.isfinite(numeric_df["prediction"].to_numpy())
    & np.isfinite(numeric_df["residual"].to_numpy())
)

numeric_df = numeric_df.loc[valid_mask].reset_index(drop=True)

if len(numeric_df) < 10:
    raise ValueError("Residual analizi için yeterli geçerli gözlem yok.")

feature_values = numeric_df[FEATURE].to_numpy()
residual_values = numeric_df["residual"].to_numpy()

residual_mean = residual_values.mean()
residual_median = np.median(residual_values)
mae = np.mean(np.abs(residual_values))
rmse = np.sqrt(np.mean(residual_values**2))

print(f"\nGeçerli gözlem sayısı: {len(numeric_df):,}")
print(f"\n{FEATURE} İSTATİSTİKLERİ")
print(f"Minimum : {feature_values.min():.4f}")
print(f"Maximum : {feature_values.max():.4f}")
print(f"Mean    : {feature_values.mean():.4f}")
print(f"Median  : {np.median(feature_values):.4f}")

print("\nRESIDUAL İSTATİSTİKLERİ")
print(f"Mean Residual : ${residual_mean:,.2f}")
print(f"Median        : ${residual_median:,.2f}")
print(f"MAE           : ${mae:,.2f}")
print(f"RMSE          : ${rmse:,.2f}")

fig,axes = plt.subplots(3,2,figsize=(20,20))
ax1,ax2,ax3,ax4,ax5,ax6 = axes.flatten()

# A1 — NORMAL PROBABILITY PLOT

if len(residual_values)>PLOT_SAMPLE:
    rng=np.random.default_rng(RANDOM_STATE)
    residual_plot=rng.choice(residual_values,size=PLOT_SAMPLE,replace=False)
else:
    residual_plot=residual_values

sorted_residuals=np.sort(residual_plot)
n=len(sorted_residuals)
theoretical_quantiles=stats.norm.ppf((np.arange(1,n+1)-.5)/n)
plot_mean=sorted_residuals.mean()
plot_std=np.std(sorted_residuals,ddof=1)
plot_std=plot_std if plot_std>0 and np.isfinite(plot_std) else 1
standardized_residuals=(sorted_residuals-plot_mean)/plot_std

ax1.scatter(theoretical_quantiles,standardized_residuals,s=14,alpha=.22,rasterized=True,label="Standardized residuals")
x_line=np.array([theoretical_quantiles.min(),theoretical_quantiles.max()])
ax1.plot(x_line,x_line,"--",linewidth=2.5,label="Normal reference")
ax1.axhline(0,linestyle=":",linewidth=1.5,color="black",alpha=.8)
ax1.set_xlabel("Theoretical Normal Quantiles",fontsize=13)
ax1.set_ylabel("Standardized Residual",fontsize=13)
ax1.set_title("A1 — Residual Normal Probability Plot",fontsize=16,fontweight="bold")
ax1.grid(alpha=.25,linestyle="--")
ax1.legend(loc="upper left",fontsize=10)

# A2 — NUMBER OF LOTS VS RESIDUAL

number_of_lots=np.exp(feature_values)
residual_m=residual_values/1e6

hb=ax2.hexbin(number_of_lots,residual_m,gridsize=50,mincnt=1,bins="log",cmap="viridis",linewidths=.15)
ax2.axhline(0,linestyle="--",linewidth=2,color="red",label="Residual = 0")
median_residual_m=np.median(residual_m)
ax2.axhline(median_residual_m,linestyle=":",linewidth=2,color="black",label=f"Median = {median_residual_m:+.2f}M")
cbar=fig.colorbar(hb,ax=ax2,pad=.02)
cbar.set_label("Observation Density (log)",fontsize=12)
ax2.set_xlabel("Number of Lots",fontsize=13)
ax2.set_ylabel("Residual (Million USD)",fontsize=13)
ax2.set_title("A2 — Number of Lots vs Residual",fontsize=16,fontweight="bold")
ax2.grid(alpha=.20,linestyle="--")
ax2.legend(loc="upper right",fontsize=10)

# A3 — RESIDUAL DISTRIBUTION

scale=1e6
residual_m=residual_values/scale
symlog_scale=1
residual_transformed=np.sign(residual_m)*np.log1p(np.abs(residual_m)/symlog_scale)
hist_bins=np.linspace(residual_transformed.min(),residual_transformed.max(),70)

ax3.hist(residual_transformed,bins=hist_bins,alpha=.78,edgecolor="black",linewidth=.4)

def residual_to_symlog(value):
    return np.sign(value)*np.log1p(np.abs(value)/symlog_scale)

tick_values=np.array([-1000,-500,-100,-50,-10,-5,-1,0,1,5,10,50,100,500,1000],dtype=float)
tick_positions=np.array([residual_to_symlog(v) for v in tick_values])
valid_ticks=(tick_positions>=residual_transformed.min())&(tick_positions<=residual_transformed.max())

ax3.set_xticks(tick_positions[valid_ticks])
ax3.set_xticklabels([f"{v:g}" for v in tick_values[valid_ticks]],rotation=45,ha="right")

mean_position=residual_to_symlog(residual_mean/scale)
median_position=residual_to_symlog(residual_median/scale)

ax3.axvline(residual_to_symlog(0),linestyle="--",linewidth=2,color="red",label="Residual = 0")
ax3.axvline(mean_position,linestyle=":",linewidth=2,color="black",label=f"Mean = {residual_mean/scale:+.2f}M")
ax3.axvline(median_position,linestyle="-.",linewidth=1.8,color="gray",label=f"Median = {residual_median/scale:+.2f}M")

p01=np.percentile(residual_m,1)
p99=np.percentile(residual_m,99)

ax3.text(
    .98,.94,
    f"1st percentile: {p01:+.2f}M\n99th percentile: {p99:+.2f}M\n"
    f"Below P1: {np.sum(residual_m<p01):,}\nAbove P99: {np.sum(residual_m>p99):,}",
    transform=ax3.transAxes,ha="right",va="top",fontsize=10,
    bbox=dict(boxstyle="round,pad=.5",facecolor="white",edgecolor="gray",alpha=.9)
)

ax3.set_xlabel("Residual (Million USD) — Symmetric Log Scale",fontsize=13)
ax3.set_ylabel("Frequency",fontsize=13)
ax3.set_title("A3 — Full Residual Distribution",fontsize=16,fontweight="bold")
ax3.grid(axis="y",alpha=.20,linestyle="--")
ax3.legend(fontsize=9,loc="upper left")

# A4 — MODEL ERROR BY NUMBER OF LOTS

numeric_df["number_of_lots"]=np.exp(numeric_df[FEATURE])

lot_bins=[0,5,10,20,np.inf]
lot_labels=["1–5 lots","6–10 lots","11–20 lots","21+ lots"]

numeric_df["lot_group"]=pd.cut(
    numeric_df["number_of_lots"],
    bins=lot_bins,
    labels=lot_labels,
    include_lowest=True
)

lot_stats=(
    numeric_df.groupby("lot_group",observed=False)
    .agg(
        observations=("residual","size"),
        MAE=("absolute_error","mean"),
        RMSE=("residual",lambda x:np.sqrt(np.mean(x**2))),
        mean_residual=("residual","mean")
    )
    .reset_index()
)

x_pos=np.arange(len(lot_stats))
mae_m=lot_stats["MAE"].to_numpy()/1e6

bars=ax4.bar(x_pos,mae_m,width=.62,alpha=.82,edgecolor="black",linewidth=.7)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(lot_stats["lot_group"].astype(str))
ax4.set_xlabel("Number of Lots",fontsize=13)
ax4.set_ylabel("MAE (Million USD)",fontsize=13)
ax4.set_title("A4 — Model Error by Number of Lots",fontsize=16,fontweight="bold")
ax4.grid(axis="y",alpha=.20,linestyle="--")

for bar,mae_value,n_obs in zip(bars,mae_m,lot_stats["observations"]):
    ax4.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height(),
        f"MAE: {mae_value:.2f}M\nN = {n_obs:,}",
        ha="center",va="bottom",fontsize=10
    )

ax4.set_ylim(0,max(mae_m)*1.20)

# A5 — RESIDUAL DENSITY VS OBSERVATION ORDER

observation_order=np.arange(len(residual_values))
residual_order_m=residual_values/1e6

# Yoğunluğu görünür yapmak için observation order × residual heatmap
hb5=ax5.hexbin(
    observation_order,
    residual_order_m,
    gridsize=(70,45),
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.1
)

cbar5=fig.colorbar(hb5,ax=ax5,pad=.02)
cbar5.set_label("Observation Density (log)",fontsize=12)

ax5.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax5.axhline(
    residual_mean/1e6,
    linestyle=":",
    linewidth=2,
    color="black",
    label=f"Mean = {residual_mean/1e6:+.2f}M"
)

# Rolling median: aşırı residual değerlerinden daha az etkilenir
rolling_window=max(100,len(residual_order_m)//100)

rolling_median=(
    pd.Series(residual_order_m)
    .rolling(
        window=rolling_window,
        center=True,
        min_periods=max(10,rolling_window//4)
    )
    .median()
    .to_numpy()
)

ax5.plot(
    observation_order,
    rolling_median,
    color="white",
    linewidth=2.5,
    label=f"Rolling median ({rolling_window:,})"
)

# ±1M referans bölgesi
ax5.axhspan(
    -1,
    1,
    alpha=.08,
    color="white",
    label="±1M residual band"
)

ax5.set_xlabel(
    "Test Observation Order",
    fontsize=13
)

ax5.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax5.set_title(
    "A5 — Residual Density vs Observation Order",
    fontsize=16,
    fontweight="bold"
)

ax5.grid(
    alpha=.18,
    linestyle="--"
)

ax5.legend(
    fontsize=9,
    loc="upper right"
)

# A6 — ÖZET İSTATİSTİKLER

ax6.axis("off")

positive_ratio=np.mean(residual_values>0)*100
negative_ratio=np.mean(residual_values<0)*100

summary_text=(
    f"RESIDUAL SUMMARY\n\n"
    f"Observations : {len(residual_values):,}\n"
    f"MAE          : ${mae:,.0f}\n"
    f"RMSE         : ${rmse:,.0f}\n"
    f"Mean         : ${residual_mean:,.0f}\n"
    f"Median       : ${residual_median:,.0f}\n\n"
    f"Positive      : {positive_ratio:.2f}%\n"
    f"Negative      : {negative_ratio:.2f}%\n\n"
    f"P1            : ${p01*1e6:,.0f}\n"
    f"P99           : ${p99*1e6:,.0f}"
)

ax6.text(
    .05,.95,
    summary_text,
    transform=ax6.transAxes,
    va="top",
    fontsize=13,
    family="monospace",
    bbox=dict(
        boxstyle="round,pad=.8",
        facecolor="white",
        edgecolor="gray",
        alpha=.95
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — NUMERIC RESIDUAL DIAGNOSTICS",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(rect=[0,0,1,.96])
plt.show()
plt.close()

# HATA ÖZETİ

worst_lot_group=lot_stats.loc[lot_stats["MAE"].idxmax()]
worst_bias_group=lot_stats.loc[lot_stats["mean_residual"].abs().idxmax()]

print("\n"+"="*80)
print(f"{FEATURE.upper()} — RESIDUAL HATA ANALİZİ")
print("="*80)

print(
    f"\nEn yüksek MAE : {worst_lot_group['lot_group']} → "
    f"${worst_lot_group['MAE']:,.2f}"
)

print(
    f"Örnek sayısı  : {int(worst_lot_group['observations']):,}"
)

print(
    f"\nEn güçlü bias : {worst_bias_group['lot_group']} → "
    f"${worst_bias_group['mean_residual']:,.2f}"
)

print(f"\nPozitif residual oranı : {positive_ratio:.2f}%")
print(f"Negatif residual oranı : {negative_ratio:.2f}%")

if residual_mean>0:
    print(
        "\nGenel yorum: Pozitif residual → Actual > Prediction → "
        "model genel olarak UNDER-PREDICTION yapıyor."
    )
elif residual_mean<0:
    print(
        "\nGenel yorum: Negatif residual → Actual < Prediction → "
        "model genel olarak OVER-PREDICTION yapıyor."
    )
else:
    print("\nGenel yorum: Ortalama residual yaklaşık 0.")

print("\nLOT GRUBU BAZLI HATA")

for _,row in lot_stats.iterrows():
    direction="UNDER-PREDICTION" if row["mean_residual"]>0 else "OVER-PREDICTION"
    print(
        f"{str(row['lot_group']):>10} | "
        f"N={int(row['observations']):,} | "
        f"MAE=${row['MAE']/1e6:.2f}M | "
        f"RMSE=${row['RMSE']/1e6:.2f}M | "
        f"Bias=${row['mean_residual']/1e6:+.2f}M | "
        f"{direction}"
    )

**LOG_TENDER_LOTSCOUNT — Grafik Bazlı Artık (Residual) Teşhis Raporu**

Modelin LOG_TENDER_LOTSCOUNT değişkeni üzerindeki davranışını ve hata odaklarını gösteren grafik bazlı teknik analiz:

**A1 — Normal Probability Plot (Q-Q Plot)** Noktaların teorik normal referans doğrusundan pozitif yönde belirgin şekilde ayrılması, artıkların ağır kuyruklu (fat-tailed) bir yapıda olduğunu gösterir. Özellikle sağ kuyrukta standart artıkların 40 seviyelerine kadar fırlaması, ihale lot sayısı ne olursa olsun modelin tutar olarak aşırı yüksek ihale bedellerinde dramatik eksik tahminler (under-prediction) yaptığını ve uç değerlerde çöktüğünü açıkça koymaktadır.

**A2 — Number of Lots vs Residual** Teklif edilen lot sayısı arttıkça model hatasının saçılımı incelendiğinde, hataların ana gövdesinin 0 ekseninde paralel bir bant şeklinde ilerlediği görünmektedir. Ancak asıl sapmalar, lot sayısının az olduğu (0–50 aralığı) bölgelerde 1.200 Milyon USD seviyelerine varan devasa pozitif hatalar şeklinde kümelenmiştir. Bu durum, lot sayısı düşük olsa dahi yüksek bütçeli ihalelerde modelin varyans kontrolünü kaybettiğini kanıtlamaktadır.

**A3 — Full Residual Distribution (Symmetric Log Scale)** Dağılımın merkezindeki medyan hatanın -5.004 USD ile 0'a oldukça yakın durması, modelin genel ihale kütlesinde dengeli bir tahmin yürüttüğünü gösterir. Ancak histogramın ortalaması +1.60 Milyon USD seviyesine kaymıştır. Sağ taraftaki pozitif kuyruğun genliği, aşırı iyimser (düşük tutarlı) tahminlerin miktar olarak az fakat parasal ölçek olarak devasa bir eksik tahmin sapması (positive bias) yarattığını doğrulamaktadır.

**A4 — Model Error by Number of Lots (Binned MAE)**  Lot aralıklarına göre ortalama mutlak hata incelendiğinde, en yüksek MAE tutarına 2.71 Milyon USD ile 6–10 lots segmentinde ulaşıldığı görülmektedir (N=2.639). İhale ölçeği büyüyüp 21+ lots seviyesine çıktığında ise MAE 0.89 Milyon USD'ye kadar gerilemektedir. Model, rekabetin ve karmaşıklığın orta ölçekte olduğu (6–10 lot) projelerde birim değer projeksiyonunu yakalamakta zorlanmaktadır.

**A5 — Residual Density vs Observation Order** Test veri setindeki gözlem sırasına göre dağılım incelendiğinde, veri kümesi boyunca rastgele saçılan tekil uç değer patlamaları dışında zaman veya veri toplama sırasına bağlı sistematik bir otokorelasyon veya dalgalanma gözlenmemektedir. Bu da hatanın test sırasından bağımsız, tamamen öznitelik seviyesindeki ihale ölçeğinden kaynaklandığını doğrulamaktadır.

## LOG_SUBMISSION_PERIOD (Tekliflerin sunulması için tanınan sürenin logaritmik dönüşüm uygulanmış değerini ifade eden sayısal değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "log_submission_period"
PLOT_SAMPLE = 10000
RANDOM_STATE = 42

if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {
    str(c).lower(): c
    for c in test_predictions.columns
}

actual_candidates = [
    "actual_usd",
    "actual",
    "y_true",
    "true",
    "target",
    "actual_price",
    "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd",
    "predicted",
    "prediction",
    "pred",
    "y_pred",
    "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd",
    "residual",
    "error",
    "prediction_error"
]

actual_col = next(
    (
        columns_lower[c]
        for c in actual_candidates
        if c in columns_lower
    ),
    None
)

predicted_col = next(
    (
        columns_lower[c]
        for c in predicted_candidates
        if c in columns_lower
    ),
    None
)

residual_col = next(
    (
        columns_lower[c]
        for c in residual_candidates
        if c in columns_lower
    ),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test: {len(df_test):,}\n"
        f"test_predictions: {len(test_predictions):,}"
    )

def to_numpy_numeric(x):
    if isinstance(x, pd.Series):
        return pd.to_numeric(
            x,
            errors="coerce"
        ).to_numpy(dtype=float)

    return pd.to_numeric(
        np.asarray(x),
        errors="coerce"
    ).astype(float)

numeric_df = pd.DataFrame({
    FEATURE: to_numpy_numeric(df_test[FEATURE]),
    "actual": to_numpy_numeric(
        test_predictions[actual_col]
    ),
    "prediction": to_numpy_numeric(
        test_predictions[predicted_col]
    ),
    "residual": to_numpy_numeric(
        test_predictions[residual_col]
    )
})

numeric_df["absolute_error"] = (
    numeric_df["residual"].abs()
)

valid_mask = (
    np.isfinite(numeric_df[FEATURE].to_numpy())
    & np.isfinite(numeric_df["actual"].to_numpy())
    & np.isfinite(numeric_df["prediction"].to_numpy())
    & np.isfinite(numeric_df["residual"].to_numpy())
)

numeric_df = (
    numeric_df
    .loc[valid_mask]
    .reset_index(drop=True)
)

if len(numeric_df) < 10:
    raise ValueError(
        "Residual analizi için yeterli geçerli gözlem yok."
    )

feature_values = numeric_df[FEATURE].to_numpy()
residual_values = numeric_df["residual"].to_numpy()

residual_mean = residual_values.mean()
residual_median = np.median(residual_values)

mae = np.mean(
    np.abs(residual_values)
)

rmse = np.sqrt(
    np.mean(residual_values ** 2)
)

print(
    f"\nGeçerli gözlem sayısı: "
    f"{len(numeric_df):,}"
)

print(f"\n{FEATURE} İSTATİSTİKLERİ")
print(
    f"Minimum : {feature_values.min():.4f}"
)
print(
    f"Maximum : {feature_values.max():.4f}"
)
print(
    f"Mean    : {feature_values.mean():.4f}"
)
print(
    f"Median  : {np.median(feature_values):.4f}"
)

print("\nRESIDUAL İSTATİSTİKLERİ")
print(
    f"Mean Residual : ${residual_mean:,.2f}"
)
print(
    f"Median        : ${residual_median:,.2f}"
)
print(
    f"MAE           : ${mae:,.2f}"
)
print(
    f"RMSE          : ${rmse:,.2f}"
)

# log1p ile dönüştürülmüş submission period'u gerçek ölçeğe geri çevir
submission_period = np.expm1(
    feature_values
)

# Sayısal güvenlik
submission_period = np.maximum(
    submission_period,
    0
)

numeric_df["submission_period"] = (
    submission_period
)

fig, axes = plt.subplots(
    3,
    2,
    figsize=(20, 20)
)

ax1, ax2, ax3, ax4, ax5, ax6 = (
    axes.flatten()
)

# A1 — NORMAL PROBABILITY PLOT

if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(
        RANDOM_STATE
    )

    residual_plot = rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
else:
    residual_plot = residual_values

sorted_residuals = np.sort(
    residual_plot
)

n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - 0.5) / n
)

plot_mean = sorted_residuals.mean()

plot_std = np.std(
    sorted_residuals,
    ddof=1
)

plot_std = (
    plot_std
    if plot_std > 0
    and np.isfinite(plot_std)
    else 1
)

standardized_residuals = (
    sorted_residuals - plot_mean
) / plot_std

ax1.scatter(
    theoretical_quantiles,
    standardized_residuals,
    s=14,
    alpha=.22,
    rasterized=True,
    label="Standardized residuals"
)

x_line = np.array([
    theoretical_quantiles.min(),
    theoretical_quantiles.max()
])

ax1.plot(
    x_line,
    x_line,
    "--",
    linewidth=2.5,
    label="Normal reference"
)

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5,
    color="black",
    alpha=.8
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual",
    fontsize=13
)

ax1.set_title(
    "A1 — Residual Normal Probability Plot",
    fontsize=16,
    fontweight="bold"
)

ax1.grid(
    alpha=.25,
    linestyle="--"
)

ax1.legend(
    loc="upper left",
    fontsize=10
)

# A2 — SUBMISSION PERIOD VS RESIDUAL

period_values = numeric_df[
    "submission_period"
].to_numpy()

residual_m = (
    residual_values / 1e6
)

hb = ax2.hexbin(
    period_values,
    residual_m,
    gridsize=50,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.15,
    alpha=.90
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=.02
)

cbar.set_label(
    "Observation Density (log scale)",
    fontsize=12
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual_m = np.median(
    residual_m
)

ax2.axhline(
    median_residual_m,
    linestyle=":",
    linewidth=2,
    color="black",
    label=(
        f"Median = "
        f"{median_residual_m:+.2f}M"
    )
)

ax2.set_xlabel(
    "Submission Period (days)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Submission Period vs Residual",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(
    alpha=.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)

# A3 — RESIDUAL DISTRIBUTION

scale = 1e6

residual_m = (
    residual_values / scale
)

symlog_scale = 1

residual_transformed = (
    np.sign(residual_m)
    * np.log1p(
        np.abs(residual_m)
        / symlog_scale
    )
)

hist_bins = np.linspace(
    residual_transformed.min(),
    residual_transformed.max(),
    70
)

ax3.hist(
    residual_transformed,
    bins=hist_bins,
    alpha=.78,
    edgecolor="black",
    linewidth=.4
)

def residual_to_symlog(value):
    return (
        np.sign(value)
        * np.log1p(
            np.abs(value)
            / symlog_scale
        )
    )

tick_values = np.array([
    -1000,
    -500,
    -100,
    -50,
    -10,
    -5,
    -1,
    0,
    1,
    5,
    10,
    50,
    100,
    500,
    1000
], dtype=float)

tick_positions = np.array([
    residual_to_symlog(v)
    for v in tick_values
])

valid_ticks = (
    (tick_positions >= residual_transformed.min())
    & (tick_positions <= residual_transformed.max())
)

ax3.set_xticks(
    tick_positions[valid_ticks]
)

ax3.set_xticklabels(
    [
        f"{v:g}"
        for v in tick_values[valid_ticks]
    ],
    rotation=45,
    ha="right"
)

mean_position = residual_to_symlog(
    residual_mean / scale
)

median_position = residual_to_symlog(
    residual_median / scale
)

ax3.axvline(
    residual_to_symlog(0),
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax3.axvline(
    mean_position,
    linestyle=":",
    linewidth=2,
    color="black",
    label=(
        f"Mean = "
        f"{residual_mean / scale:+.2f}M"
    )
)

ax3.axvline(
    median_position,
    linestyle="-.",
    linewidth=1.8,
    color="gray",
    label=(
        f"Median = "
        f"{residual_median / scale:+.2f}M"
    )
)

p01 = np.percentile(
    residual_m,
    1
)

p99 = np.percentile(
    residual_m,
    99
)

ax3.text(
    .98,
    .94,
    f"1st percentile: {p01:+.2f}M\n"
    f"99th percentile: {p99:+.2f}M\n"
    f"Below P1: {np.sum(residual_m < p01):,}\n"
    f"Above P99: {np.sum(residual_m > p99):,}",
    transform=ax3.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=.5",
        facecolor="white",
        edgecolor="gray",
        alpha=.9
    )
)

ax3.set_xlabel(
    "Residual (Million USD) — Symmetric Log Scale",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Full Residual Distribution",
    fontsize=16,
    fontweight="bold"
)

ax3.grid(
    axis="y",
    alpha=.20,
    linestyle="--"
)

ax3.legend(
    fontsize=9,
    loc="upper left"
)

# A4 — MODEL ERROR BY SUBMISSION PERIOD

period_bins = [
    -np.inf,
    1,
    3,
    7,
    14,
    30,
    60,
    90,
    180,
    np.inf
]

period_labels = [
    "0–1 day",
    "2–3 days",
    "4–7 days",
    "8–14 days",
    "15–30 days",
    "31–60 days",
    "61–90 days",
    "91–180 days",
    "181+ days"
]

numeric_df["period_group"] = pd.cut(
    numeric_df["submission_period"],
    bins=period_bins,
    labels=period_labels,
    include_lowest=True
)

period_stats = (
    numeric_df
    .groupby(
        "period_group",
        observed=False
    )
    .agg(
        observations=("residual", "size"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "residual",
            lambda x: np.sqrt(
                np.mean(x ** 2)
            )
        ),
        mean_residual=(
            "residual",
            "mean"
        ),
        median_residual=(
            "residual",
            "median"
        )
    )
    .reset_index()
)

period_stats = period_stats[
    period_stats["observations"] > 0
].copy()

x_pos = np.arange(
    len(period_stats)
)

mae_m = (
    period_stats["MAE"].to_numpy()
    / 1e6
)

bars = ax4.bar(
    x_pos,
    mae_m,
    width=.62,
    alpha=.82,
    edgecolor="black",
    linewidth=.7
)

ax4.set_xticks(
    x_pos
)

ax4.set_xticklabels(
    period_stats["period_group"].astype(str),
    rotation=35,
    ha="right"
)

ax4.set_xlabel(
    "Submission Period",
    fontsize=13
)

ax4.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax4.set_title(
    "A4 — Model Error by Submission Period",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(
    axis="y",
    alpha=.20,
    linestyle="--"
)

for bar, mae_value, n_obs in zip(
    bars,
    mae_m,
    period_stats["observations"]
):
    ax4.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"MAE: {mae_value:.2f}M\n"
        f"N = {n_obs:,}",
        ha="center",
        va="bottom",
        fontsize=9
    )

if len(mae_m):
    ax4.set_ylim(
        0,
        max(mae_m) * 1.25
    )

# A5 — RESIDUAL DENSITY VS OBSERVATION ORDER

observation_order = np.arange(
    len(residual_values)
)

residual_order_m = (
    residual_values / 1e6
)

hb5 = ax5.hexbin(
    observation_order,
    residual_order_m,
    gridsize=(70, 45),
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.1,
    alpha=.90
)

cbar5 = fig.colorbar(
    hb5,
    ax=ax5,
    pad=.02
)

cbar5.set_label(
    "Observation Density (log scale)",
    fontsize=12
)

ax5.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax5.axhline(
    residual_mean / 1e6,
    linestyle=":",
    linewidth=2,
    color="black",
    label=(
        f"Mean = "
        f"{residual_mean / 1e6:+.2f}M"
    )
)

rolling_window = max(
    100,
    len(residual_order_m) // 100
)

rolling_median = (
    pd.Series(residual_order_m)
    .rolling(
        window=rolling_window,
        center=True,
        min_periods=max(
            10,
            rolling_window // 4
        )
    )
    .median()
    .to_numpy()
)

ax5.plot(
    observation_order,
    rolling_median,
    color="white",
    linewidth=2.5,
    label=(
        f"Rolling median "
        f"({rolling_window:,})"
    )
)

ax5.axhspan(
    -1,
    1,
    alpha=.08,
    color="white",
    label="±1M residual band"
)

ax5.set_xlabel(
    "Test Observation Order",
    fontsize=13
)

ax5.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax5.set_title(
    "A5 — Residual Density vs Observation Order",
    fontsize=16,
    fontweight="bold"
)

ax5.grid(
    alpha=.18,
    linestyle="--"
)

ax5.legend(
    fontsize=9,
    loc="upper right"
)

# A6 — ÖZET İSTATİSTİKLER

ax6.axis("off")

positive_ratio = (
    np.mean(residual_values > 0)
    * 100
)

negative_ratio = (
    np.mean(residual_values < 0)
    * 100
)

summary_text = (
    f"RESIDUAL SUMMARY\n\n"
    f"Observations : {len(residual_values):,}\n"
    f"MAE          : ${mae:,.0f}\n"
    f"RMSE         : ${rmse:,.0f}\n"
    f"Mean         : ${residual_mean:,.0f}\n"
    f"Median       : ${residual_median:,.0f}\n\n"
    f"Positive      : {positive_ratio:.2f}%\n"
    f"Negative      : {negative_ratio:.2f}%\n\n"
    f"P1            : ${p01 * 1e6:,.0f}\n"
    f"P99           : ${p99 * 1e6:,.0f}"
)

ax6.text(
    .05,
    .95,
    summary_text,
    transform=ax6.transAxes,
    va="top",
    fontsize=13,
    family="monospace",
    bbox=dict(
        boxstyle="round,pad=.8",
        facecolor="white",
        edgecolor="gray",
        alpha=.95
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — NUMERIC RESIDUAL DIAGNOSTICS",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, .96]
)

plt.savefig(
    "log_submission_period_numeric_residual_diagnostics.png",
    dpi=160,
    bbox_inches="tight"
)

plt.show()

plt.close()

# HATA ÖZETİ

worst_period_group = period_stats.loc[
    period_stats["MAE"].idxmax()
]

worst_bias_group = period_stats.loc[
    period_stats["mean_residual"].abs().idxmax()
]

print(
    "\n" + "=" * 80
)

print(
    f"{FEATURE.upper()} — RESIDUAL HATA ANALİZİ"
)

print(
    "=" * 80
)

print(
    f"\nEn yüksek MAE : "
    f"{worst_period_group['period_group']} → "
    f"${worst_period_group['MAE']:,.2f}"
)

print(
    f"Örnek sayısı  : "
    f"{int(worst_period_group['observations']):,}"
)

print(
    f"\nEn güçlü bias : "
    f"{worst_bias_group['period_group']} → "
    f"${worst_bias_group['mean_residual']:,.2f}"
)

print(
    f"\nPozitif residual oranı : "
    f"{positive_ratio:.2f}%"
)

print(
    f"Negatif residual oranı : "
    f"{negative_ratio:.2f}%"
)

if residual_mean > 0:
    print(
        "\nGenel yorum: Pozitif residual → "
        "Actual > Prediction → "
        "model genel olarak UNDER-PREDICTION yapıyor."
    )

elif residual_mean < 0:
    print(
        "\nGenel yorum: Negatif residual → "
        "Actual < Prediction → "
        "model genel olarak OVER-PREDICTION yapıyor."
    )

else:
    print(
        "\nGenel yorum: Ortalama residual yaklaşık 0."
    )

print(
    "\nSUBMISSION PERIOD GRUBU BAZLI HATA"
)

for _, row in period_stats.iterrows():

    if row["mean_residual"] > 0:
        direction = "UNDER-PREDICTION"
    elif row["mean_residual"] < 0:
        direction = "OVER-PREDICTION"
    else:
        direction = "BALANCED"

    print(
        f"{str(row['period_group']):>12} | "
        f"N={int(row['observations']):,} | "
        f"MAE=${row['MAE']/1e6:.2f}M | "
        f"RMSE=${row['RMSE']/1e6:.2f}M | "
        f"Bias=${row['mean_residual']/1e6:+.2f}M | "
        f"{direction}"
    )

print(
    "\nSUBMISSION PERIOD ÖZETİ"
)

print(
    f"Gerçek minimum : "
    f"{numeric_df['submission_period'].min():.2f} gün"
)

print(
    f"Gerçek maximum : "
    f"{numeric_df['submission_period'].max():.2f} gün"
)

print(
    f"Gerçek ortalama: "
    f"{numeric_df['submission_period'].mean():.2f} gün"
)

print(
    f"Gerçek medyan  : "
    f"{numeric_df['submission_period'].median():.2f} gün"
)

**LOG_SUBMISSION_PERIOD — Grafik Bazlı Artık (Residual) Teşhis Raporu**

Modelin teklif hazırlık/sunum süresi (LOG_SUBMISSION_PERIOD) özniteliği üzerindeki tahmin hatalarını ve kritik başarısızlık noktalarını inceleyen grafik bazlı teknik değerlendirme:

**A1 — Residual Normal Probability Plot (Q-Q Plot)** Artıkların teorik normallik çizgisinden sağ kuyrukta dramatik bir şekilde ayrılması, model hatalarının kalın kuyruklu (fat-tailed) bir yapıda olduğunu gösterir. Standartlaştırılmış artıkların +40 seviyelerine ulaşması, ihale başvuru süresinden bağımsız olarak yüksek bütçeli ihalelerde modelin çok ciddi seviyede eksik tahmin (under-prediction) yaptığını ve uç ihale tutarlarını öğrenmekte yetersiz kaldığını doğrulamaktadır.

**A2 — Submission Period vs Residual** Teklif başvuru süresinin gün cinsinden dağılımına bakıldığında, verilerin büyük çoğunluğunun 30–60 gün aralığında yoğunlaştığı ve hataların ana gövdesinin 0 çizgisi etrafında yatay bir hat izlediği görülmektedir. Ancak asıl devasa sapmalar, ihalelerin en yoğun gerçekleştiği bu 30–60 gün bandında ortaya çıkmaktadır. Model, 1.200 Milyon USD seviyesine kadar ulaşan aşırı hatayı (pozitif artık) tam da veri hacminin en yüksek olduğu bu standart başvuru sürelerinde yapmaktadır.

**A3 — Full Residual Distribution (Symmetric Log Scale)** Dağılımın merkezindeki medyan hata (-5.004 USD) neredeyse 0 hizasında kalarak genel veri kütlesinde simetrik bir tahmin izlenimi verse de, ortalama hatanın +1.60 Milyon USD seviyesinde kalması belirgin bir sağa çarpıklığa işaret eder. Modelin negatif taraftaki hataları daha dar bir aralıktayken, pozitif taraftaki hataların devasa tutarlara ulaşması, nadir fakat yüksek tutarlı ihalelerdeki sistematik eksik tahmin eğilimini (positive bias) ortaya koymaktadır.

**A4 — Model Error by Submission Period (Binned MAE)** Başvuru süreleri kategorik gruplara bölündüğünde, en yüksek MAE (2.32 Milyon USD) ve en güçlü sapmanın (+1.76 Milyon USD bias) 61–90 gün segmentinde gerçekleştiği görülmektedir (N=1.657). Süre 0–7 gün gibi çok kısa olduğunda MAE 0.20M–0.38M USD seviyesindeyken, sürenin 15–180 gün aralığına uzamasıyla hatanın 2 Milyon USD barajını aşması, modelin uzun hazırlık süresi gerektiren kompleks ve yüksek bütçeli projelerde fiyat projeksiyonu üretmekte zorlandığını gösterir.

**A5 — Residual Density vs Observation Order** Test gözlem sırasına göre artıkların dağılımı incelendiğinde, veri seti boyunca zamansal veya sistemsel bir otokorelasyon eğilimi görülmemektedir. Hatalar test sırasından bağımsız bir şekilde rastgele saçılmaktadır; ancak grafik üzerindeki tekil uç değer patlamaları (400M–1.200M USD residual noktaları), hatanın dönemsel bir dış etkenden değil, veri setindeki bütçesi son derece yüksek spesifik ihalelerden kaynaklandığını kanıtlar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

FEATURE = "log_submission_period"
TARGET = "log_tender_finalpriceUsd"
SAMPLE_SIZE = 12000
N_BINS = 35
SMOOTH_BINS = 5
OUTLIER_Q = 0.99
PLOT_LOW = 0.005
PLOT_HIGH = 0.995

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")

if "y_test" not in globals():
    raise NameError("y_test bulunamadı.")

time_values = None

for name in ["X_test", "test_df", "df_test"]:
    if name in globals():
        obj = globals()[name]

        if hasattr(obj, "columns") and FEATURE in obj.columns:
            time_values = np.asarray(
                obj[FEATURE],
                dtype=float
            ).reshape(-1)

            break

if time_values is None:
    raise KeyError(
        f"{FEATURE} bulunamadı."
    )

y_true = np.asarray(
    y_test,
    dtype=float
).reshape(-1)

y_pred = np.asarray(
    test_predictions,
    dtype=float
).reshape(-1)

n = min(
    len(time_values),
    len(y_true),
    len(y_pred)
)

time_values = time_values[:n]
y_true = y_true[:n]
y_pred = y_pred[:n]

mask = (
    np.isfinite(time_values)
    & np.isfinite(y_true)
    & np.isfinite(y_pred)
    & (time_values >= 0)
)

time_values = time_values[mask]
y_true = y_true[mask]
y_pred = y_pred[mask]

true_max = np.nanmax(y_true)
pred_max = np.nanmax(y_pred)

if true_max < 50 and pred_max > 100:
    print(
        "Prediction USD-space → log-space dönüşümü uygulanıyor."
    )

    y_pred = np.log1p(
        np.maximum(y_pred, 0)
    )

if (
    np.nanmax(np.abs(y_true)) > 100
    or np.nanmax(np.abs(y_pred)) > 100
):
    raise ValueError(
        "Actual ve prediction aynı log-space ölçekte değil."
    )

df = pd.DataFrame({
    "time": time_values,
    "actual": y_true,
    "predicted": y_pred
})

df["residual"] = (
    df["actual"]
    - df["predicted"]
)

df["abs_residual"] = (
    df["residual"].abs()
)

sample = (
    df.sample(
        SAMPLE_SIZE,
        random_state=42
    )
    if len(df) > SAMPLE_SIZE
    else df.copy()
)

def limits(
    x,
    low=PLOT_LOW,
    high=PLOT_HIGH,
    symmetric=False
):
    lo, hi = np.nanquantile(
        x,
        [low, high]
    )

    if symmetric:
        m = max(
            abs(lo),
            abs(hi),
            1e-6
        )

        return (
            -m * 1.08,
            m * 1.08
        )

    d = max(
        hi - lo,
        1e-6
    )

    return (
        lo - d * 0.05,
        hi + d * 0.05
    )

t_min = np.nanquantile(
    df["time"],
    PLOT_LOW
)

t_max = np.nanquantile(
    df["time"],
    PLOT_HIGH
)

y_lim = limits(
    np.r_[
        sample["actual"],
        sample["predicted"]
    ]
)

r_lim = limits(
    sample["residual"],
    symmetric=True
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(17, 11)
)

# G1 — Actual vs Predicted
ax = axes[0, 0]

g1 = sample[
    (sample["time"] >= t_min)
    & (sample["time"] <= t_max)
    & (sample["actual"] >= y_lim[0])
    & (sample["actual"] <= y_lim[1])
    & (sample["predicted"] >= y_lim[0])
    & (sample["predicted"] <= y_lim[1])
].copy()

x = g1["time"].to_numpy()
actual = g1["actual"].to_numpy()
pred = g1["predicted"].to_numpy()

ax.scatter(
    x,
    actual,
    s=8,
    alpha=0.10,
    label="Actual"
)

ax.scatter(
    x,
    pred,
    s=8,
    alpha=0.10,
    label="Predicted"
)

if len(x) > 2 and np.ptp(x) > 0:

    coef = np.polyfit(
        x,
        actual,
        1
    )

    x_grid = np.linspace(
        x.min(),
        x.max(),
        300
    )

    ax.plot(
        x_grid,
        np.polyval(
            coef,
            x_grid
        ),
        linewidth=2.3,
        label=(
            f"Actual Trend "
            f"(slope={coef[0]:.3f})"
        )
    )

idx = np.linspace(
    0,
    len(x) - 1,
    min(20, len(x)),
    dtype=int
)

for i in idx:
    ax.vlines(
        x[i],
        pred[i],
        actual[i],
        linewidth=0.5,
        alpha=0.20
    )

ax.set_xlim(
    t_min,
    t_max
)

ax.set_ylim(
    y_lim
)

ax.set_xlabel(
    FEATURE
)

ax.set_ylabel(
    TARGET + " (log-space)"
)

ax.set_title(
    "G1 — Actual vs Predicted"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.18
)

# G2 — Residual vs Submission Period
ax = axes[0, 1]

g2 = sample[
    (sample["time"] >= t_min)
    & (sample["time"] <= t_max)
].copy()

# Mevcut renk haritası korunuyor.
# Renk yoğunluğu = bin içerisindeki gözlem sayısı.
hb2 = ax.hexbin(
    g2["time"],
    g2["residual"],
    gridsize=45,
    mincnt=1,
    bins="log",
    alpha=0.20
)

# G2 colorbar
cbar2 = fig.colorbar(
    hb2,
    ax=ax,
    pad=0.02
)

cbar2.set_label(
    "Observation Density\n(log count)",
    rotation=270,
    labelpad=18
)

# Quantile binning
g2["bin"] = pd.qcut(
    g2["time"],
    q=min(
        N_BINS,
        g2["time"].nunique()
    ),
    duplicates="drop"
)

stats2 = (
    g2.groupby(
        "bin",
        observed=True
    )
    .agg(
        time=("time", "median"),
        median=("residual", "median"),
        mean=("residual", "mean"),
        count=("residual", "size")
    )
    .reset_index()
    .sort_values("time")
)

stats2["median_smooth"] = (
    stats2["median"]
    .rolling(
        SMOOTH_BINS,
        center=True,
        min_periods=1
    )
    .median()
)

stats2["mean_smooth"] = (
    stats2["mean"]
    .rolling(
        SMOOTH_BINS,
        center=True,
        min_periods=1
    )
    .median()
)

# Median — Kırmızı
ax.plot(
    stats2["time"],
    stats2["median_smooth"],
    color="crimson",
    marker="o",
    markersize=4,
    linewidth=3,
    label="Smoothed Median Residual",
    zorder=5
)

# Mean — Mavi
ax.plot(
    stats2["time"],
    stats2["mean_smooth"],
    color="royalblue",
    marker="s",
    markersize=3,
    linewidth=2.5,
    linestyle="--",
    label="Smoothed Mean Residual",
    zorder=5
)

ax.axhline(
    0,
    color="black",
    linestyle=":",
    linewidth=1.5,
    zorder=4
)

ax.set_xlim(
    t_min,
    t_max
)

ax.set_ylim(
    r_lim
)

ax.set_xlabel(
    FEATURE
)

ax.set_ylabel(
    "Residual (Actual - Predicted)"
)

ax.set_title(
    "G2 — Residuals vs Submission Period"
)

ax.legend(
    frameon=True,
    loc="best"
)

ax.grid(
    alpha=0.18
)

# G3 — Residual vs Fitted
ax = axes[1, 0]

p_low = sample[
    "predicted"
].quantile(PLOT_LOW)

p_high = sample[
    "predicted"
].quantile(PLOT_HIGH)

g3 = sample[
    (sample["predicted"] >= p_low)
    & (sample["predicted"] <= p_high)
].copy()

threshold = g3[
    "abs_residual"
].quantile(OUTLIER_Q)

g3["outlier"] = (
    g3["abs_residual"]
    >= threshold
)

# Mevcut renk haritası korunuyor.
# Renk yoğunluğu = bin içerisindeki gözlem sayısı.
hb3 = ax.hexbin(
    g3["predicted"],
    g3["residual"],
    gridsize=45,
    mincnt=1,
    bins="log",
    alpha=0.25
)

# G3 colorbar
cbar3 = fig.colorbar(
    hb3,
    ax=ax,
    pad=0.02
)

cbar3.set_label(
    "Observation Density\n(log count)",
    rotation=270,
    labelpad=18
)

# Quantile fitted bins
g3["bin"] = pd.qcut(
    g3["predicted"],
    q=min(
        N_BINS,
        g3["predicted"].nunique()
    ),
    duplicates="drop"
)

stats3 = (
    g3.groupby(
        "bin",
        observed=True
    )
    .agg(
        fitted=("predicted", "median"),
        residual=("residual", "median"),
        count=("residual", "size")
    )
    .reset_index()
    .sort_values("fitted")
)

stats3["smooth"] = (
    stats3["residual"]
    .rolling(
        SMOOTH_BINS,
        center=True,
        min_periods=1
    )
    .median()
)

ax.plot(
    stats3["fitted"],
    stats3["smooth"],
    color="darkorange",
    marker="o",
    markersize=3.5,
    linewidth=2.5,
    label="Smoothed Median Residual",
    zorder=5
)

out = g3[
    g3["outlier"]
]

if len(out):

    ax.scatter(
        out["predicted"],
        out["residual"],
        s=28,
        facecolors="none",
        edgecolors="red",
        linewidths=1.0,
        label="Top 1% |Residual|",
        zorder=6
    )

ax.axhline(
    0,
    color="black",
    linestyle=":",
    linewidth=1.3
)

ax.set_xlim(
    limits(
        g3["predicted"]
    )
)

ax.set_ylim(
    r_lim
)

ax.set_xlabel(
    "Fitted Values (log-space)"
)

ax.set_ylabel(
    "Residual (Actual - Predicted)"
)

ax.set_title(
    "G3 — Residuals vs Fitted Values"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.18
)

# G4 — Temporal Residual Pattern
ax = axes[1, 1]

g4 = df[
    (df["time"] >= t_min)
    & (df["time"] <= t_max)
].copy()

g4["bin"] = pd.qcut(
    g4["time"],
    q=min(
        N_BINS,
        g4["time"].nunique()
    ),
    duplicates="drop"
)

stats4 = (
    g4.groupby(
        "bin",
        observed=True
    )
    .agg(
        time=("time", "median"),
        median=("residual", "median"),
        mean=("residual", "mean"),
        q25=(
            "residual",
            lambda x: x.quantile(.25)
        ),
        q75=(
            "residual",
            lambda x: x.quantile(.75)
        ),
        count=("residual", "size")
    )
    .reset_index()
    .sort_values("time")
)

stats4["median_smooth"] = (
    stats4["median"]
    .rolling(
        SMOOTH_BINS,
        center=True,
        min_periods=1
    )
    .median()
)

stats4["mean_smooth"] = (
    stats4["mean"]
    .rolling(
        SMOOTH_BINS,
        center=True,
        min_periods=1
    )
    .median()
)

ax.fill_between(
    stats4["time"].to_numpy(),
    stats4["q25"].to_numpy(),
    stats4["q75"].to_numpy(),
    alpha=0.15,
    label="IQR"
)

ax.plot(
    stats4["time"],
    stats4["median_smooth"],
    color="crimson",
    marker="o",
    markersize=3.5,
    linewidth=2.5,
    label="Smoothed Median Residual"
)

ax.plot(
    stats4["time"],
    stats4["mean_smooth"],
    color="royalblue",
    linewidth=1.8,
    linestyle="--",
    alpha=0.9,
    label="Smoothed Mean Residual"
)

ax.axhline(
    0,
    color="black",
    linestyle=":",
    linewidth=1.3
)

ax.set_xlim(
    t_min,
    t_max
)

ax.set_ylim(
    r_lim
)

ax.set_xlabel(
    FEATURE
)

ax.set_ylabel(
    "Residual"
)

ax.set_title(
    "G4 — Temporal Residual Pattern"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.18
)

fig.suptitle(
    f"{FEATURE.upper()} — Residual Diagnostics",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.96]
)

fig.savefig(
    f"{FEATURE}_residual_diagnostics.png",
    dpi=160,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)

# Metrikler
mse = mean_squared_error(
    df["actual"],
    df["predicted"]
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    df["actual"],
    df["predicted"]
)

r2 = r2_score(
    df["actual"],
    df["predicted"]
)

metrics_table = pd.DataFrame({
    "Feature": [FEATURE],
    "Target": [TARGET],
    "MSE": [mse],
    "RMSE": [rmse],
    "MAE": [mae],
    "R²": [r2],
    "N": [len(df)]
})

print("\nMODEL METRİKLERİ — KOLON BAZLI")
print(
    metrics_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# Residual özeti
p99 = df[
    "abs_residual"
].quantile(.99)

print("\nRESIDUAL ANALİZİ")

print(
    f"Toplam gözlem     : {len(df):,}"
)

print(
    f"Grafik örneklemi  : {len(sample):,}"
)

print(
    f"Mean Residual     : "
    f"{df['residual'].mean():.6f}"
)

print(
    f"Median Residual   : "
    f"{df['residual'].median():.6f}"
)

print(
    f"MAE               : "
    f"{df['abs_residual'].mean():.6f}"
)

print(
    f"Residual P99      : "
    f"{p99:.6f}"
)

print(
    f"Top 1% Outlier    : "
    f"{(df['abs_residual'] >= p99).sum():,}"
)

print("\nLOG_SUBMISSION_PERIOD")

print(
    f"Minimum           : "
    f"{df['time'].min():.6f}"
)

print(
    f"Maximum           : "
    f"{df['time'].max():.6f}"
)

print("\nTREND BİNLERİ")

print(
    f"G2 dolu bin       : "
    f"{len(stats2):,}"
)

print(
    f"G4 dolu bin       : "
    f"{len(stats4):,}"
)

print("\nFITTED VALUES")

print(
    f"Prediction P99    : "
    f"{df['predicted'].quantile(.99):.6f}"
)

print(
    f"Prediction P99.5  : "
    f"{df['predicted'].quantile(.995):.6f}"
)

print(
    f"Prediction max    : "
    f"{df['predicted'].max():.6f}"
)

**LOG_SUBMISSION_PERIOD — Log-Uzay İstatistiksel Artık Teşhis Raporu**

Metrik değerlerindeki korkunç R^2 = -26.35 seviyesi ve log-uzay ölçeğindeki grafik çıktıları, modelin log_submission_period değişkeni üzerinden log_tender_finalpriceUsd hedef değişkenini öğrenirken geçirdiği matematiksel çöküşü net bir şekilde ortaya koymaktadır. Grafikler bazında modelin hata yaptığı kritik noktalar şu şekildedir:

**G1 — Actual vs Predicted** Gerçek değerler (mavi) 13-14 log-uzay bandında düz bir hatta seyrederken, modelin tahminleri (turuncu) 0 ile 30+ log-uzay değerleri arasında aşırı geniş bir aralığa fırlamaktadır. Model, teklif süresinden bağımsız olarak ihalelere son derece kontrolsüz tahminler atamakta; özellikle x 3.6 hizasında dikey bir hat boyunca yoğunlaşan sabit tahmin patlamaları üreterek varyans kontrolünü tamamen kaybetmektedir.

**G2 — Residuals vs Submission Period** Smoothed Median ve Mean artık çizgilerinin y = 0 referans doğrusu yerine y 4.0 - 4.5 bandında yatay bir hat olarak takılı kalması, modelin sistemik bir sabit sapma (constant bias) yaşadığını doğrular. Başvuru süresinin büyüklüğü ne olursa olsun model ortalamada gerçek değerlerin yaklaşık 4 log birim altında kalmakta (under-prediction), verideki yoğunlaşma bölgelerinde dahi bu yanlılıktan kurtulamamaktadır.

**G3 — Residuals vs Fitted Values** Modelin en dramatik başarısızlık noktası bu grafikte açığa çıkmaktadır. Artıkların 0 etrafında rastgele dağılması gerekirken soldan sağa kesintisiz ve dik bir negatif eğim izlemesi, modelin fitted (tahmin) değerleri büyüdükçe hatalarını lineer olarak artırdığını gösterir. Model düşük tahminlerde aşırı eksik tahmin (y -y +18), yüksek tahminlerde ise devasa fazla tahmin (y -y approx -18) yapmaktadır. Tahminler 30+ log-uzay değerlerine fırladıkça Top 1% outlier kırmızı halkaları sağ alt köşede yığılmakta ve regresyon mantığı tamamen çökmektedir.

**G4 — Temporal Residual Pattern (IQR & Trend)** G4 panelindeki mavi IQR gölgesinin 0 ile +10 log-uzay bandı gibi son derece geniş bir aralığı kapsaması, model hatalarının varyansının çok yüksek ve belirsiz olduğunu gösterir. Başvuru süresi boyunca median ve mean çizgilerinin yatay seyretmesi, hatanın zamansal/süreçsel değişimden ziyade modelin hedef değişken ölçeğini tutturamamasından ve regülasyon eksikliğinden kaynaklandığını kanıtlar.

## LOG_DECISION_PERIOD ( İhale kararının verilmesi için geçen sürenin logaritmik dönüşüm uygulanmış değerini ifade eden sayısal değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "log_decision_period"
PLOT_SAMPLE = 10000
N_SEGMENTS = 5
RANDOM_STATE = 42
PERCENTILE_LOW = 0.5
PERCENTILE_HIGH = 99.5

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")
if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")
if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

columns_lower = {
    str(c).lower(): c
    for c in test_predictions.columns
}

actual_candidates = [
    "actual_usd",
    "actual",
    "y_true",
    "true",
    "target",
    "actual_price",
    "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd",
    "predicted",
    "prediction",
    "pred",
    "y_pred",
    "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd",
    "residual",
    "error",
    "prediction_error"
]

actual_col = next(
    (columns_lower[c] for c in actual_candidates if c in columns_lower),
    None
)

predicted_col = next(
    (columns_lower[c] for c in predicted_candidates if c in columns_lower),
    None
)

residual_col = next(
    (columns_lower[c] for c in residual_candidates if c in columns_lower),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")
if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")
if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor: "
        f"df_test={len(df_test):,}, "
        f"test_predictions={len(test_predictions):,}"
    )

def to_numeric(x):
    if hasattr(x, "to_numpy"):
        x = x.to_numpy()

    x = np.asarray(x)

    if np.issubdtype(x.dtype, np.number):
        return x.astype(float)

    return pd.to_numeric(
        pd.Series(x),
        errors="coerce"
    ).to_numpy(dtype=float)

numeric_df = pd.DataFrame({
    FEATURE: to_numeric(df_test[FEATURE]),
    "actual": to_numeric(test_predictions[actual_col]),
    "prediction": to_numeric(test_predictions[predicted_col]),
    "residual": to_numeric(test_predictions[residual_col])
})

numeric_df["absolute_error"] = numeric_df["residual"].abs()
numeric_df["squared_error"] = numeric_df["residual"] ** 2

valid_mask = (
    np.isfinite(numeric_df[FEATURE].to_numpy())
    & np.isfinite(numeric_df["actual"].to_numpy())
    & np.isfinite(numeric_df["prediction"].to_numpy())
    & np.isfinite(numeric_df["residual"].to_numpy())
)

numeric_df = numeric_df.loc[valid_mask].reset_index(drop=True)

if len(numeric_df) < 30:
    raise ValueError("Residual analizi için yeterli geçerli gözlem yok.")

x = numeric_df[FEATURE].to_numpy()
residual = numeric_df["residual"].to_numpy()
actual = numeric_df["actual"].to_numpy()
prediction = numeric_df["prediction"].to_numpy()

mae = np.mean(np.abs(residual))
mse = np.mean(residual ** 2)
rmse = np.sqrt(mse)
bias = np.mean(residual)

print(f"Geçerli gözlem sayısı: {len(numeric_df):,}")
print(f"MAE : ${mae / 1e6:.2f}M")
print(f"MSE : {mse / 1e12:.2f} T USD²")
print(f"RMSE: ${rmse / 1e6:.2f}M")
print(f"Bias: ${bias / 1e6:+.2f}M")

unique_count = numeric_df[FEATURE].nunique()
segment_count = min(N_SEGMENTS, unique_count)

if segment_count < 2:
    raise ValueError(
        f"{FEATURE} en az 2 farklı değer içermelidir."
    )

numeric_df["segment"] = pd.qcut(
    numeric_df[FEATURE],
    q=segment_count,
    duplicates="drop"
)

segment_stats = (
    numeric_df
    .groupby("segment", observed=True)
    .agg(
        N=("residual", "size"),
        MAE=("absolute_error", "mean"),
        MSE=("squared_error", "mean"),
        RMSE=("residual", lambda z: np.sqrt(np.mean(z ** 2))),
        Bias=("residual", "mean")
    )
    .reset_index()
)

segment_stats["label"] = [
    f"{max(0, v.left):.3f}–{v.right:.3f}"
    for v in segment_stats["segment"]
]

y_low, y_high = np.percentile(
    residual,
    [PERCENTILE_LOW, PERCENTILE_HIGH]
)

plot_mask = (
    (residual >= y_low)
    & (residual <= y_high)
)

x_plot = x[plot_mask]
residual_plot = residual[plot_mask] / 1e6

if len(x_plot) > PLOT_SAMPLE:
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(
        len(x_plot),
        PLOT_SAMPLE,
        replace=False
    )
    x_plot = x_plot[idx]
    residual_plot = residual_plot[idx]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(20, 14)
)

ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — RESIDUAL VS DECISION PERIOD

hb = ax1.hexbin(
    x_plot,
    residual_plot,
    gridsize=(60, 45),
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.08
)

cbar = fig.colorbar(
    hb,
    ax=ax1,
    pad=.02
)

cbar.set_label(
    "Observation Density (log)",
    fontsize=10
)

ax1.axhline(
    0,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Residual = 0"
)

segment_mid = [
    interval.mid
    for interval in segment_stats["segment"]
]

ax1.plot(
    segment_mid,
    segment_stats["Bias"] / 1e6,
    color="white",
    marker="o",
    markersize=6,
    linewidth=2.5,
    label="Segment mean residual"
)

ax1.set_xlim(left=0)
ax1.set_ylim(
    y_low / 1e6,
    y_high / 1e6
)

ax1.set_xlabel(
    "log_decision_period",
    fontsize=12
)

ax1.set_ylabel(
    "Residual (Million USD)",
    fontsize=12
)

ax1.set_title(
    "A1 — Residuals vs Decision Period",
    fontsize=15,
    fontweight="bold"
)

ax1.grid(
    alpha=.18,
    linestyle="--"
)

ax1.legend(fontsize=9)

# A2 — RESIDUAL DISTRIBUTION

residual_central = residual[
    (residual >= y_low)
    & (residual <= y_high)
] / 1e6

ax2.hist(
    residual_central,
    bins=55,
    edgecolor="black",
    linewidth=.4,
    alpha=.8
)

ax2.axvline(
    0,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Residual = 0"
)

ax2.axvline(
    bias / 1e6,
    color="black",
    linestyle=":",
    linewidth=2,
    label=f"Mean = {bias / 1e6:+.2f}M"
)

ax2.set_xlabel(
    "Residual (Million USD)",
    fontsize=12
)

ax2.set_ylabel(
    "Frequency",
    fontsize=12
)

ax2.set_title(
    "A2 — Residual Distribution",
    fontsize=15,
    fontweight="bold"
)

ax2.grid(
    axis="y",
    alpha=.2,
    linestyle="--"
)

ax2.legend(fontsize=9)

ax2.text(
    .97,
    .94,
    f"Central range:\n"
    f"{y_low / 1e6:+.2f}M → {y_high / 1e6:+.2f}M\n\n"
    f"MAE = {mae / 1e6:.2f}M\n"
    f"MSE = {mse / 1e12:.2f} T USD²\n"
    f"RMSE = {rmse / 1e6:.2f}M",
    transform=ax2.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=.5",
        facecolor="white",
        edgecolor="gray",
        alpha=.9
    )
)

# A3 — SEGMENT ERROR

x_pos = np.arange(len(segment_stats))
width = .25

mae_values = segment_stats["MAE"].to_numpy() / 1e6
mse_values = segment_stats["MSE"].to_numpy() / 1e12
rmse_values = segment_stats["RMSE"].to_numpy() / 1e6

bars1 = ax3.bar(
    x_pos - width,
    mae_values,
    width,
    label="MAE ($M)",
    alpha=.85,
    edgecolor="black",
    linewidth=.6
)

bars2 = ax3.bar(
    x_pos,
    mse_values,
    width,
    label="MSE (T USD²)",
    alpha=.65,
    edgecolor="black",
    linewidth=.6
)

bars3 = ax3.bar(
    x_pos + width,
    rmse_values,
    width,
    label="RMSE ($M)",
    alpha=.5,
    edgecolor="black",
    linewidth=.6
)

ax3.set_xticks(x_pos)

ax3.set_xticklabels(
    segment_stats["label"],
    fontsize=9
)

ax3.set_xlabel(
    "log_decision_period Segment",
    fontsize=12
)

ax3.set_ylabel(
    "Error Magnitude",
    fontsize=12
)

ax3.set_title(
    "A3 — Segment-Based Model Error",
    fontsize=15,
    fontweight="bold"
)

ax3.grid(
    axis="y",
    alpha=.2,
    linestyle="--"
)

ax3.legend(fontsize=9)

for bar, value in zip(bars1, mae_values):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

for bar, value in zip(bars2, mse_values):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

for bar, value in zip(bars3, rmse_values):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

# A4 — ACTUAL VS PREDICTION

grouped = (
    numeric_df
    .groupby("segment", observed=True)
    .agg(
        Actual=("actual", "mean"),
        Prediction=("prediction", "mean")
    )
    .reset_index()
)

grouped["label"] = [
    f"{max(0, v.left):.3f}–{v.right:.3f}"
    for v in grouped["segment"]
]

x_alt = np.arange(len(grouped))

ax4.plot(
    x_alt,
    grouped["Actual"] / 1e6,
    marker="o",
    linewidth=2.5,
    label="Actual mean"
)

ax4.plot(
    x_alt,
    grouped["Prediction"] / 1e6,
    marker="o",
    linewidth=2.5,
    label="Prediction mean"
)

ax4.set_xticks(x_alt)

ax4.set_xticklabels(
    grouped["label"],
    rotation=20,
    ha="right"
)

ax4.set_xlabel(
    "log_decision_period Segment",
    fontsize=12
)

ax4.set_ylabel(
    "Mean Price (Million USD)",
    fontsize=12
)

ax4.set_title(
    "A4 — Actual vs Prediction",
    fontsize=15,
    fontweight="bold"
)

ax4.grid(
    alpha=.2,
    linestyle="--"
)

ax4.legend(fontsize=9)

fig.suptitle(
    f"{FEATURE.upper()} — PROFESSIONAL RESIDUAL DASHBOARD",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, .95]
)

plt.show()
plt.close()

# GÖRSEL TABLO

table_data = []

for _, row in segment_stats.iterrows():

    direction = (
        "UNDER"
        if row["Bias"] > 0
        else "OVER"
    )

    table_data.append([
        row["label"],
        f"{int(row['N']):,}",
        f"${row['MAE'] / 1e6:.2f}M",
        f"{row['MSE'] / 1e12:.2f} T",
        f"${row['RMSE'] / 1e6:.2f}M",
        f"${row['Bias'] / 1e6:+.2f}M",
        direction
    ])

fig_table, ax_table = plt.subplots(
    figsize=(18, 4.2)
)

ax_table.axis("off")

table = ax_table.table(
    cellText=table_data,
    colLabels=[
        "Decision Period",
        "Observations",
        "MAE ($M)",
        "MSE (T USD²)",
        "RMSE ($M)",
        "Bias ($M)",
        "Direction"
    ],
    cellLoc="center",
    colLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.2)

cells = table.get_celld()

for col in range(len(table_data[0])):
    if (0, col) in cells:
        cells[(0, col)].set_text_props(
            fontweight="bold"
        )

for key, cell in cells.items():
    cell.set_linewidth(.8)

worst_mae_idx = segment_stats["MAE"].idxmax()
worst_mse_idx = segment_stats["MSE"].idxmax()
worst_rmse_idx = segment_stats["RMSE"].idxmax()
worst_bias_idx = segment_stats["Bias"].abs().idxmax()

for row_idx in range(
    1,
    len(segment_stats) + 1
):

    source_idx = row_idx - 1

    if source_idx == worst_mae_idx:
        cells[(row_idx, 2)].set_text_props(
            fontweight="bold"
        )

    if source_idx == worst_mse_idx:
        cells[(row_idx, 3)].set_text_props(
            fontweight="bold"
        )

    if source_idx == worst_rmse_idx:
        cells[(row_idx, 4)].set_text_props(
            fontweight="bold"
        )

    if source_idx == worst_bias_idx:
        cells[(row_idx, 5)].set_text_props(
            fontweight="bold"
        )

ax_table.set_title(
    "DECISION PERIOD BAZLI MODEL HATA TABLOSU",
    fontsize=16,
    fontweight="bold",
    pad=18
)

plt.tight_layout()
plt.show()
plt.close()

# ÖZET

worst_mae = segment_stats.loc[
    segment_stats["MAE"].idxmax()
]

worst_mse = segment_stats.loc[
    segment_stats["MSE"].idxmax()
]

worst_rmse = segment_stats.loc[
    segment_stats["RMSE"].idxmax()
]

worst_bias = segment_stats.loc[
    segment_stats["Bias"].abs().idxmax()
]

print("\n" + "=" * 70)
print(f"{FEATURE.upper()} — HATA ÖZETİ")
print("=" * 70)

print(
    f"Genel MAE  : ${mae / 1e6:.2f}M"
)

print(
    f"Genel MSE  : {mse / 1e12:.2f} T USD²"
)

print(
    f"Genel RMSE : ${rmse / 1e6:.2f}M"
)

print(
    f"Genel Bias : ${bias / 1e6:+.2f}M"
)

print(
    f"\nEn yüksek MAE : "
    f"{worst_mae['label']} → "
    f"${worst_mae['MAE'] / 1e6:.2f}M"
)

print(
    f"En yüksek MSE : "
    f"{worst_mse['label']} → "
    f"{worst_mse['MSE'] / 1e12:.2f} T USD²"
)

print(
    f"En yüksek RMSE: "
    f"{worst_rmse['label']} → "
    f"${worst_rmse['RMSE'] / 1e6:.2f}M"
)

print(
    f"En güçlü Bias : "
    f"{worst_bias['label']} → "
    f"${worst_bias['Bias'] / 1e6:+.2f}M"
)

print(
    f"\nPozitif residual : "
    f"{np.mean(residual > 0) * 100:.2f}%"
)

print(
    f"Negatif residual : "
    f"{np.mean(residual < 0) * 100:.2f}%"
)

**LOG_DECISION_PERIOD — Professional Residual Dashboard Teşhis Raporu**

Modelin karar süresi (LOG_DECISION_PERIOD) özniteliği üzerindeki tahmin hatalarını ve segment bazlı davranışını inceleyen grafik bazlı teknik değerlendirme:

**A1 — Residuals vs Decision Period** Veri yoğunluğunun 4.0 - 5.0 log-uzay aralığında (sarı/yeşil hexbin odağı) kümelendiği görülmektedir. Karar süresi büyüdükçe hataların saçılımı 0 çizgisinin üzerinde dikey olarak patlamakta ve 50+ Milyon USD seviyelerine kadar tırmanmaktadır. Beyaz ortalama hattının (segment mean residual) 0 çizgisinin üzerinde pozitif bir seyir izlemesi, modelin karar süresinin neredeyse tüm aralıklarında sistematik bir eksik tahmin (under-prediction) yaptığını göstermektedir.

**A2 — Residual Distribution** Artıkların dağılımı 0 etrafında dar bir gövdeye sahip görünse de, 0'ın sağ tarafında 10M - 50M+ USD aralığına uzanan uzun bir kuyruk içermektedir. Siyah noktalı çizgi ile gösterilen ortalama hata tutarının (+1.60M USD) kırmızı kesikli çizgi olan 0 referansından belirgin şekilde sağa kayması, negatif taraftaki hataların küçük, pozitif taraftaki sapmaların ise devasa boyutlarda olduğunu ve güçlü bir pozitif bias yarattığını doğrular.

**A3 — Segment-Based Model Error** Segment bazlı hata metrikleri incelendiğinde iki farklı hata odağı öne çıkmaktadır: Uç değerlerin (outlier) yarattığı varyans hatası en yüksek 0.000–4.143 segmentindedir (395.94 T USD² MSE ve 19.90M RMSE). Bu durum, kısa/orta karar süreli ihaleler arasında çok yüksek bütçeli extrem ihalelerin bulunduğunu gösterir. Ortalama mutlak hata (MAE) açısından ise hata en yüksek 4.700–5.903 segmentinde (2.32M MAE) gerçekleşmektedir.

**A4 — Actual vs Prediction** Modelin en net basitleştirilmiş hatası bu segment trend grafiğinde açığa çıkmaktadır. Tüm karar süresi segmentleri boyunca mavi çizgi (Actual mean) turuncu çizginin (Prediction mean) paralel olarak üstünde yer almaktadır. Karar süresi büyüdükçe gerçek ortalama fiyat 2.5M'dan 12M'a fırlarken, model tahmini 1M'dan 10M'a çıkmakta ancak aradaki sapma kapanmamaktadır. Model, karar süresi uzayan büyük ölçekli projelerin değerini istikrarlı bir şekilde gerçek değerinin altında tahmin etmektedir (+1.79M USD bias).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "log_decision_period"
PLOT_SAMPLE = 10000
RANDOM_STATE = 42

if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. "
        "Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {
    str(c).lower(): c
    for c in test_predictions.columns
}

actual_candidates = [
    "actual_usd",
    "actual",
    "y_true",
    "true",
    "target",
    "actual_price",
    "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd",
    "predicted",
    "prediction",
    "pred",
    "y_pred",
    "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd",
    "residual",
    "error",
    "prediction_error"
]

actual_col = next(
    (
        columns_lower[c]
        for c in actual_candidates
        if c in columns_lower
    ),
    None
)

predicted_col = next(
    (
        columns_lower[c]
        for c in predicted_candidates
        if c in columns_lower
    ),
    None
)

residual_col = next(
    (
        columns_lower[c]
        for c in residual_candidates
        if c in columns_lower
    ),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test: {len(df_test):,}\n"
        f"test_predictions: {len(test_predictions):,}"
    )


def to_numpy_numeric(x):
    if isinstance(x, pd.Series):
        return pd.to_numeric(
            x,
            errors="coerce"
        ).to_numpy(dtype=float)

    return pd.to_numeric(
        np.asarray(x),
        errors="coerce"
    ).astype(float)


numeric_df = pd.DataFrame({
    FEATURE: to_numpy_numeric(df_test[FEATURE]),
    "actual": to_numpy_numeric(
        test_predictions[actual_col]
    ),
    "prediction": to_numpy_numeric(
        test_predictions[predicted_col]
    ),
    "residual": to_numpy_numeric(
        test_predictions[residual_col]
    )
})

numeric_df["absolute_error"] = (
    numeric_df["residual"].abs()
)

valid_mask = (
    np.isfinite(
        numeric_df[FEATURE].to_numpy()
    )
    &
    np.isfinite(
        numeric_df["actual"].to_numpy()
    )
    &
    np.isfinite(
        numeric_df["prediction"].to_numpy()
    )
    &
    np.isfinite(
        numeric_df["residual"].to_numpy()
    )
)

numeric_df = (
    numeric_df
    .loc[valid_mask]
    .reset_index(drop=True)
)

if len(numeric_df) < 10:
    raise ValueError(
        "Residual analizi için yeterli geçerli gözlem yok."
    )

feature_values = numeric_df[FEATURE].to_numpy()
residual_values = numeric_df["residual"].to_numpy()

residual_mean = residual_values.mean()
residual_median = np.median(residual_values)
mae = np.mean(np.abs(residual_values))
rmse = np.sqrt(np.mean(residual_values ** 2))

print(
    f"\nGeçerli gözlem sayısı: "
    f"{len(numeric_df):,}"
)

print(f"\n{FEATURE} İSTATİSTİKLERİ")
print(
    f"Minimum : {feature_values.min():.4f}"
)
print(
    f"Maximum : {feature_values.max():.4f}"
)
print(
    f"Mean    : {feature_values.mean():.4f}"
)
print(
    f"Median  : {np.median(feature_values):.4f}"
)

print("\nRESIDUAL İSTATİSTİKLERİ")
print(
    f"Mean Residual : ${residual_mean:,.2f}"
)
print(
    f"Median        : ${residual_median:,.2f}"
)
print(
    f"MAE           : ${mae:,.2f}"
)
print(
    f"RMSE          : ${rmse:,.2f}"
)


fig, axes = plt.subplots(
    3,
    2,
    figsize=(20, 20)
)

ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()


# A1 — NORMAL PROBABILITY PLOT

if len(residual_values) > PLOT_SAMPLE:
    rng = np.random.default_rng(
        RANDOM_STATE
    )

    residual_plot = rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
else:
    residual_plot = residual_values


sorted_residuals = np.sort(
    residual_plot
)

n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - .5) / n
)

plot_mean = sorted_residuals.mean()

plot_std = np.std(
    sorted_residuals,
    ddof=1
)

plot_std = (
    plot_std
    if plot_std > 0 and np.isfinite(plot_std)
    else 1
)

standardized_residuals = (
    sorted_residuals - plot_mean
) / plot_std

ax1.scatter(
    theoretical_quantiles,
    standardized_residuals,
    s=14,
    alpha=.22,
    rasterized=True,
    label="Standardized residuals"
)

x_line = np.array([
    theoretical_quantiles.min(),
    theoretical_quantiles.max()
])

ax1.plot(
    x_line,
    x_line,
    "--",
    linewidth=2.5,
    label="Normal reference"
)

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5,
    color="black",
    alpha=.8
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual",
    fontsize=13
)

ax1.set_title(
    "A1 — Residual Normal Probability Plot",
    fontsize=16,
    fontweight="bold"
)

ax1.grid(
    alpha=.25,
    linestyle="--"
)

ax1.legend(
    loc="upper left",
    fontsize=10
)


# A2 — DECISION PERIOD VS RESIDUAL

decision_period = np.exp(
    feature_values
)

residual_m = (
    residual_values / 1e6
)

hb = ax2.hexbin(
    decision_period,
    residual_m,
    gridsize=50,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual_m = np.median(
    residual_m
)

ax2.axhline(
    median_residual_m,
    linestyle=":",
    linewidth=2,
    color="black",
    label=(
        f"Median = "
        f"{median_residual_m:+.2f}M"
    )
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=.02
)

cbar.set_label(
    "Observation Density (log)",
    fontsize=12
)

ax2.set_xlabel(
    "Decision Period (days)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Decision Period vs Residual",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(
    alpha=.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)


# A3 — RESIDUAL DISTRIBUTION

scale = 1e6

residual_m = (
    residual_values / scale
)

symlog_scale = 1

residual_transformed = (
    np.sign(residual_m)
    *
    np.log1p(
        np.abs(residual_m)
        / symlog_scale
    )
)

hist_bins = np.linspace(
    residual_transformed.min(),
    residual_transformed.max(),
    70
)

ax3.hist(
    residual_transformed,
    bins=hist_bins,
    alpha=.78,
    edgecolor="black",
    linewidth=.4
)


def residual_to_symlog(value):
    return (
        np.sign(value)
        *
        np.log1p(
            np.abs(value)
            / symlog_scale
        )
    )


tick_values = np.array(
    [
        -1000,
        -500,
        -100,
        -50,
        -10,
        -5,
        -1,
        0,
        1,
        5,
        10,
        50,
        100,
        500,
        1000
    ],
    dtype=float
)

tick_positions = np.array([
    residual_to_symlog(v)
    for v in tick_values
])

valid_ticks = (
    (tick_positions >= residual_transformed.min())
    &
    (tick_positions <= residual_transformed.max())
)

ax3.set_xticks(
    tick_positions[valid_ticks]
)

ax3.set_xticklabels(
    [
        f"{v:g}"
        for v in tick_values[valid_ticks]
    ],
    rotation=45,
    ha="right"
)

mean_position = residual_to_symlog(
    residual_mean / scale
)

median_position = residual_to_symlog(
    residual_median / scale
)

ax3.axvline(
    residual_to_symlog(0),
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax3.axvline(
    mean_position,
    linestyle=":",
    linewidth=2,
    color="black",
    label=(
        f"Mean = "
        f"{residual_mean / scale:+.2f}M"
    )
)

ax3.axvline(
    median_position,
    linestyle="-.",
    linewidth=1.8,
    color="gray",
    label=(
        f"Median = "
        f"{residual_median / scale:+.2f}M"
    )
)

p01 = np.percentile(
    residual_m,
    1
)

p99 = np.percentile(
    residual_m,
    99
)

ax3.text(
    .98,
    .94,
    (
        f"1st percentile: {p01:+.2f}M\n"
        f"99th percentile: {p99:+.2f}M\n"
        f"Below P1: "
        f"{np.sum(residual_m < p01):,}\n"
        f"Above P99: "
        f"{np.sum(residual_m > p99):,}"
    ),
    transform=ax3.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=.5",
        facecolor="white",
        edgecolor="gray",
        alpha=.9
    )
)

ax3.set_xlabel(
    "Residual (Million USD) — Symmetric Log Scale",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Full Residual Distribution",
    fontsize=16,
    fontweight="bold"
)

ax3.grid(
    axis="y",
    alpha=.20,
    linestyle="--"
)

ax3.legend(
    fontsize=9,
    loc="upper left"
)


# A4 — MODEL ERROR BY DECISION PERIOD

numeric_df["decision_period"] = (
    np.exp(numeric_df[FEATURE])
)

period_bins = [
    0,
    1,
    3,
    7,
    14,
    30,
    60,
    90,
    np.inf
]

period_labels = [
    "0–1 day",
    "2–3 days",
    "4–7 days",
    "8–14 days",
    "15–30 days",
    "31–60 days",
    "61–90 days",
    "91+ days"
]

numeric_df["period_group"] = pd.cut(
    numeric_df["decision_period"],
    bins=period_bins,
    labels=period_labels,
    include_lowest=True
)

period_stats = (
    numeric_df
    .groupby(
        "period_group",
        observed=False
    )
    .agg(
        observations=("residual", "size"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "residual",
            lambda x: np.sqrt(
                np.mean(x ** 2)
            )
        ),
        mean_residual=("residual", "mean")
    )
    .reset_index()
)

x_pos = np.arange(
    len(period_stats)
)

mae_m = (
    period_stats["MAE"].to_numpy()
    / 1e6
)

bars = ax4.bar(
    x_pos,
    mae_m,
    width=.62,
    alpha=.82,
    edgecolor="black",
    linewidth=.7
)

ax4.set_xticks(
    x_pos
)

ax4.set_xticklabels(
    period_stats["period_group"].astype(str),
    rotation=35,
    ha="right"
)

ax4.set_xlabel(
    "Decision Period",
    fontsize=13
)

ax4.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax4.set_title(
    "A4 — Model Error by Decision Period",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(
    axis="y",
    alpha=.20,
    linestyle="--"
)

for bar, mae_value, n_obs in zip(
    bars,
    mae_m,
    period_stats["observations"]
):
    ax4.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        (
            f"MAE: {mae_value:.2f}M\n"
            f"N = {n_obs:,}"
        ),
        ha="center",
        va="bottom",
        fontsize=9
    )

if len(mae_m) > 0 and np.nanmax(mae_m) > 0:
    ax4.set_ylim(
        0,
        np.nanmax(mae_m) * 1.20
    )


# A5 — RESIDUAL DENSITY VS OBSERVATION ORDER

observation_order = np.arange(
    len(residual_values)
)

residual_order_m = (
    residual_values / 1e6
)

hb5 = ax5.hexbin(
    observation_order,
    residual_order_m,
    gridsize=(70, 45),
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=.1
)

cbar5 = fig.colorbar(
    hb5,
    ax=ax5,
    pad=.02
)

cbar5.set_label(
    "Observation Density (log)",
    fontsize=12
)

ax5.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

ax5.axhline(
    residual_mean / 1e6,
    linestyle=":",
    linewidth=2,
    color="black",
    label=(
        f"Mean = "
        f"{residual_mean / 1e6:+.2f}M"
    )
)

rolling_window = max(
    100,
    len(residual_order_m) // 100
)

rolling_median = (
    pd.Series(residual_order_m)
    .rolling(
        window=rolling_window,
        center=True,
        min_periods=max(
            10,
            rolling_window // 4
        )
    )
    .median()
    .to_numpy()
)

ax5.plot(
    observation_order,
    rolling_median,
    color="white",
    linewidth=2.5,
    label=(
        f"Rolling median "
        f"({rolling_window:,})"
    )
)

ax5.axhspan(
    -1,
    1,
    alpha=.08,
    color="white",
    label="±1M residual band"
)

ax5.set_xlabel(
    "Test Observation Order",
    fontsize=13
)

ax5.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax5.set_title(
    "A5 — Residual Density vs Observation Order",
    fontsize=16,
    fontweight="bold"
)

ax5.grid(
    alpha=.18,
    linestyle="--"
)

ax5.legend(
    fontsize=9,
    loc="upper right"
)


# A6 — SUMMARY

ax6.axis("off")

positive_ratio = (
    np.mean(residual_values > 0)
    * 100
)

negative_ratio = (
    np.mean(residual_values < 0)
    * 100
)

summary_text = (
    "RESIDUAL SUMMARY\n\n"
    f"Feature      : {FEATURE}\n"
    f"Observations : {len(residual_values):,}\n\n"
    f"MAE          : ${mae:,.0f}\n"
    f"RMSE         : ${rmse:,.0f}\n"
    f"Mean         : ${residual_mean:,.0f}\n"
    f"Median       : ${residual_median:,.0f}\n\n"
    f"Positive     : {positive_ratio:.2f}%\n"
    f"Negative     : {negative_ratio:.2f}%\n\n"
    f"P1           : ${p01 * 1e6:,.0f}\n"
    f"P99          : ${p99 * 1e6:,.0f}"
)

ax6.text(
    .05,
    .95,
    summary_text,
    transform=ax6.transAxes,
    va="top",
    fontsize=13,
    family="monospace",
    bbox=dict(
        boxstyle="round,pad=.8",
        facecolor="white",
        edgecolor="gray",
        alpha=.95
    )
)


fig.suptitle(
    f"{FEATURE.upper()} — NUMERIC RESIDUAL DIAGNOSTICS",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, .96]
)

plt.show()

plt.close()


# HATA ÖZETİ

worst_period_group = period_stats.loc[
    period_stats["MAE"].idxmax()
]

worst_bias_group = period_stats.loc[
    period_stats["mean_residual"].abs().idxmax()
]

print(
    "\n" + "=" * 80
)

print(
    f"{FEATURE.upper()} — "
    f"RESIDUAL HATA ANALİZİ"
)

print(
    "=" * 80
)

print(
    f"\nEn yüksek MAE : "
    f"{worst_period_group['period_group']} → "
    f"${worst_period_group['MAE']:,.2f}"
)

print(
    f"Örnek sayısı  : "
    f"{int(worst_period_group['observations']):,}"
)

print(
    f"\nEn güçlü bias : "
    f"{worst_bias_group['period_group']} → "
    f"${worst_bias_group['mean_residual']:,.2f}"
)

print(
    f"\nPozitif residual oranı : "
    f"{positive_ratio:.2f}%"
)

print(
    f"Negatif residual oranı : "
    f"{negative_ratio:.2f}%"
)

if residual_mean > 0:
    print(
        "\nGenel yorum: Pozitif residual → "
        "Actual > Prediction → "
        "model genel olarak UNDER-PREDICTION yapıyor."
    )

elif residual_mean < 0:
    print(
        "\nGenel yorum: Negatif residual → "
        "Actual < Prediction → "
        "model genel olarak OVER-PREDICTION yapıyor."
    )

else:
    print(
        "\nGenel yorum: Ortalama residual yaklaşık 0."
    )

print(
    "\nDECISION PERIOD GRUBU BAZLI HATA"
)

for _, row in period_stats.iterrows():

    if pd.isna(row["observations"]):
        continue

    direction = (
        "UNDER-PREDICTION"
        if row["mean_residual"] > 0
        else "OVER-PREDICTION"
    )

    print(
        f"{str(row['period_group']):>12} | "
        f"N={int(row['observations']):,} | "
        f"MAE=${row['MAE']/1e6:.2f}M | "
        f"RMSE=${row['RMSE']/1e6:.2f}M | "
        f"Bias=${row['mean_residual']/1e6:+.2f}M | "
        f"{direction}"
    )

**LOG_DECISION_PERIOD — Grafik Bazlı Artık (Residual) Teşhis Raporu**

Modelin karar/değerlendirme süresi (LOG_DECISION_PERIOD) özniteliği üzerindeki hata dağılımlarını ve kritik sapma noktalarını inceleyen grafik bazlı detaylı teknik analiz:

**A1 — Residual Normal Probability Plot (Q-Q Plot)** Noktaların teorik normal referans doğrusundan pozitif tarafta üstel bir biçimde kırılarak dikleşmesi, hataların ağır kuyruklu (fat-tailed) bir dağılıma sahip olduğunu doğrular. Standartlaştırılmış artıkların +40 seviyesine kadar tırmanması, karar sürecinin uzunluğundan bağımsız olarak, yüksek bütçeli uç ihalelerde modelin hedef fiyatı yakalayamadığını ve ciddi tutarlarda eksik tahmin (under-prediction) yaptığını açıkça ortaya koymaktadır.

**A2 — Decision Period vs Residual** Karar süresinin gün bazlı saçılımı incelendiğinde, verilerin ağırlıklı olarak 60–90 gün bandında (sarı/yeşil yoğunluk odağı) kümelendiği görülmektedir. Ancak en kritik hata patlamaları tam da verinin en yoğun olduğu bu 60–100 gün aralığında gerçekleşmektedir. Model bu aralıkta 1.200 Milyon USD seviyesine ulaşan devasa pozitif hatalar (under-prediction) üreterek, karar süresi standart olan büyük ölçekli ihalelerde varyans kontrolünü tamamen kaybetmektedir.

**A3 — Full Residual Distribution (Symmetric Log Scale)** Dağılımın merkezindeki medyan hata (-5.004 USD) 0 referans hattına tam oturarak modelin genel ihale kütlesinde ortanca seviyede dengeli çalıştığını gösterir. Ancak histogramın ortalaması +1.60 Milyon USD seviyesine kaymıştır. Bu durum, pozitif taraftaki hataların sayıca simetrik görünse de parasal büyüklük olarak negatif taraftaki hatalardan katbekat fazla olduğunu ve modelde güçlü bir pozitif bias (eksik tahmin sapması) yarattığını kanıtlar.

**A4 — Model Error by Decision Period (Binned MAE)** Karar süreleri segmentlere ayrıldığında, hata miktarının karar süresi uzadıkça belirgin bir şekilde arttığı görülmektedir. En yüksek ortalama mutlak hata (2.19 Milyon USD MAE) ve en güçlü sapma (+1.69 Milyon USD bias) 91+ gün sürüncette kalan ihalelerde gerçekleşmiştir (N=12.579). Hataların 0–7 gün arasında 0.05M–0.43M USD gibi düşük seviyelerde başlayıp süre uzadıkça kademeli artması, modelin karar süresi uzayan karmaşık ve büyük ihale projelerinde birim fiyat projeksiyonu üretmekte zorlandığını doğrular.

**A5 — Residual Density vs Observation Order** Test gözlem sırasındaki hata dağılımı incelediğinde, veri seti boyunca zamansal veya sistemsel bir otokorelasyon izlenmemektedir. Hataların ana kütlesi test sırasından bağımsız olarak 0 hattında yatay bir şerit oluştursa da, eksen boyunca tekil olarak yükselen 400M–1.200M USD değerindeki sapma noktaları, problemin veri sırasından değil, veri kümesindeki aşırı yüksek tutarlı uç ihalelerden kaynaklandığını gösterir.

## CORR_SINGLEB (İhale sürecinde tek bir teklif verilip verilmediğini ifade eden ikili (binary) değişkendir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "corr_singleb"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
ROLLING_WINDOW_MAX = 2000
CATEGORIES_PER_PAGE = 8
RANDOM_STATE = 42

# KONTROLLER

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır.")

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

# KOLONLARI BUL

columns_lower = {c.lower(): c for c in test_predictions.columns}

actual_candidates = [
    "actual_usd", "actual", "y_true", "true", "target",
    "actual_price", "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd", "predicted", "prediction", "pred",
    "y_pred", "predicted_price", "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd", "residual", "error", "prediction_error"
]

actual_col = next(
    (columns_lower[c] for c in actual_candidates if c in columns_lower), None
)

predicted_col = next(
    (columns_lower[c] for c in predicted_candidates if c in columns_lower), None
)

residual_col = next(
    (columns_lower[c] for c in residual_candidates if c in columns_lower), None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test          : {len(df_test):,}\n"
        f"test_predictions : {len(test_predictions):,}"
    )

# ANALİZ DATAFRAME

analysis_df = pd.DataFrame({
    FEATURE: df_test[FEATURE].to_numpy(),
    "actual": pd.to_numeric(
        test_predictions[actual_col], errors="coerce"
    ).to_numpy(),
    "prediction": pd.to_numeric(
        test_predictions[predicted_col], errors="coerce"
    ).to_numpy(),
    "residual": pd.to_numeric(
        test_predictions[residual_col], errors="coerce"
    ).to_numpy()
})

analysis_df[FEATURE] = (
    analysis_df[FEATURE]
    .astype(str)
    .replace({
        "None": "MISSING",
        "nan": "MISSING",
        "null": "MISSING"
    })
)

analysis_df["absolute_error"] = analysis_df["residual"].abs()

valid_mask = (
    np.isfinite(analysis_df["actual"].to_numpy())
    & np.isfinite(analysis_df["prediction"].to_numpy())
    & np.isfinite(analysis_df["residual"].to_numpy())
)

analysis_df = analysis_df.loc[valid_mask].reset_index(drop=True)

print(f"\nGeçerli gözlem sayısı: {len(analysis_df):,}")

# KATEGORİ İSTATİSTİKLERİ

category_stats = (
    analysis_df
    .groupby(FEATURE)
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        std_residual=("residual", "std"),
        MAE=("absolute_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

effect_stats = category_stats[
    category_stats["count"] >= MIN_CATEGORY_COUNT
].copy()

global_std = analysis_df["residual"].std()

effect_stats["standardized_effect"] = (
    effect_stats["mean_residual"] / global_std
)

effect_stats["absolute_standardized_effect"] = (
    effect_stats["standardized_effect"].abs()
)

effect_stats = (
    effect_stats
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(f"Analiz edilen kategori sayısı: {len(effect_stats)}")

# TABLO

visual_table = effect_stats[
    [
        FEATURE,
        "count",
        "mean_residual",
        "MAE",
        "RMSE",
        "standardized_effect"
    ]
].copy()

visual_table.columns = [
    "Category",
    "Observations",
    "Mean Residual",
    "MAE",
    "RMSE",
    "Standardized Effect"
]

def highlight_max(series):
    mask = series == series.max()
    return [
        "background-color: #ffd6d6;color: #8b0000;font-weight: bold;"
        if value else ""
        for value in mask
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "${:,.0f}",
        "MAE": "${:,.0f}",
        "RMSE": "${:,.0f}",
        "Standardized Effect": "{:+.3f}"
    })
    .apply(highlight_max, subset=["MAE"])
    .apply(highlight_max, subset=["RMSE"])
    .apply(highlight_max, subset=["Mean Residual"])
    .apply(highlight_max, subset=["Standardized Effect"])
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "11pt",
            "padding": "8px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#404040"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "9px")
            ]
        }
    ])
    .set_caption(f"{FEATURE.upper()} — RESIDUAL ANALYSIS")
)

display(styled_table)

# A — ANA RESIDUAL DİAGNOSTICS

residual_values = analysis_df["residual"].to_numpy()
prediction_values = analysis_df["prediction"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(19, 14))
ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — NORMAL PROBABILITY PLOT

rng = np.random.default_rng(RANDOM_STATE)

residual_plot = (
    rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
    if len(residual_values) > PLOT_SAMPLE
    else residual_values
)

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - 0.5) / n
)

residual_mean = np.mean(sorted_residuals)
residual_std = np.std(sorted_residuals, ddof=1)

standardized_residuals = (
    (sorted_residuals - residual_mean) / residual_std
)

low, high = np.percentile(
    standardized_residuals,
    [0.5, 99.5]
)

mask = (
    (standardized_residuals >= low)
    & (standardized_residuals <= high)
)

x_a1 = theoretical_quantiles[mask]
y_a1 = standardized_residuals[mask]

ax1.scatter(
    x_a1,
    y_a1,
    s=14,
    alpha=0.22,
    rasterized=True
)

ax1.plot(
    [x_a1.min(), x_a1.max()],
    [x_a1.min(), x_a1.max()],
    linestyle="--",
    linewidth=2.5,
    label="Normal reference"
)

ax1.set_yscale("symlog", linthresh=1, linscale=1.2)
ax1.axhline(0, linestyle=":", linewidth=1.5)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual\n(Symlog Scale)",
    fontsize=13
)

ax1.set_title(
    "A1 — Normal Probability Plot\nCentral 99%",
    fontsize=16,
    fontweight="bold"
)

ax1.grid(which="major", alpha=0.25, linestyle="--")
ax1.grid(which="minor", alpha=0.10, linestyle=":")
ax1.legend(loc="upper left", fontsize=10)

# A2 — RESIDUAL VS FITS / HEATMAP

prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(
    prediction_m,
    [0.5, 99.5]
)

y_low, y_high = np.percentile(
    residual_m,
    [0.5, 99.5]
)

mask = (
    (prediction_m >= x_low)
    & (prediction_m <= x_high)
    & (residual_m >= y_low)
    & (residual_m <= y_high)
)

x_plot = prediction_m[mask]
y_plot = residual_m[mask]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=55,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=0.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual = np.median(y_plot)

ax2.axhline(
    median_residual,
    linestyle=":",
    linewidth=2,
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=0.02
)

cbar.set_label(
    "Observation Density\n(log scale)",
    fontsize=12
)

ax2.set_xlabel(
    "Predicted Price (Million USD)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Residuals Versus Fits\nDensity Heatmap",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(alpha=0.20, linestyle="--")
ax2.legend(loc="upper right", fontsize=10)

# A3 — RESIDUAL DISTRIBUTION

residual_low, residual_high = np.percentile(
    residual_values,
    [0.5, 99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

ax3.hist(
    central_residuals / 1e6,
    bins=35,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.5
)

mean_residual = np.mean(residual_values) / 1e6

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Residual = 0"
)

ax3.axvline(
    mean_residual,
    linestyle=":",
    linewidth=2,
    label=f"Mean = {mean_residual:.2f}M"
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Residual Distribution\nCentral 99%",
    fontsize=16,
    fontweight="bold"
)

ax3.legend(fontsize=10)
ax3.grid(axis="y", alpha=0.20)

# A4 — RESIDUAL VS OBSERVATION ORDER

order = np.arange(1, len(analysis_df) + 1)
residual_order = analysis_df["residual"].to_numpy() / 1e6

q_low, q_high = np.percentile(
    residual_order,
    [0.5, 99.5]
)

mask = (
    (residual_order >= q_low)
    & (residual_order <= q_high)
)

order_plot = order[mask]
residual_plot = residual_order[mask]

ax4.scatter(
    order_plot,
    residual_plot,
    s=4,
    alpha=0.055,
    rasterized=True
)

order_df = pd.DataFrame({
    "order": order_plot,
    "residual": residual_plot
})

WINDOW = max(
    500,
    min(
        ROLLING_WINDOW_MAX,
        len(order_df) // 20
    )
)

min_periods = max(
    50,
    WINDOW // 4
)

rolling_median = order_df["residual"].rolling(
    WINDOW,
    center=True,
    min_periods=min_periods
).median()

rolling_q10 = order_df["residual"].rolling(
    WINDOW,
    center=True,
    min_periods=min_periods
).quantile(0.10)

rolling_q90 = order_df["residual"].rolling(
    WINDOW,
    center=True,
    min_periods=min_periods
).quantile(0.90)

ax4.fill_between(
    order_df["order"].to_numpy(),
    rolling_q10.to_numpy(),
    rolling_q90.to_numpy(),
    alpha=0.20,
    label="Rolling 10–90% error range"
)

ax4.plot(
    order_df["order"].to_numpy(),
    rolling_median.to_numpy(),
    linewidth=3,
    label=f"Rolling Median ({WINDOW:,})"
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2.2,
    color="red",
    label="Residual = 0"
)

global_median = np.median(residual_order)

ax4.axhline(
    global_median,
    linestyle="-.",
    linewidth=1.6,
    alpha=0.75,
    label=f"Global Median = {global_median:+.2f}M"
)

ax4.set_yscale(
    "symlog",
    linthresh=1,
    linscale=1.2
)

ax4.set_xlabel(
    "Observation Order",
    fontsize=13
)

ax4.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax4.set_title(
    "A4 — Residuals Versus Observation Order\nRolling Bias",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(which="major", alpha=0.25, linestyle="--")
ax4.grid(which="minor", alpha=0.10, linestyle=":")
ax4.legend(loc="upper right", fontsize=9)

positive_ratio = np.mean(residual_order > 0) * 100
negative_ratio = np.mean(residual_order < 0) * 100
median_abs_error = np.median(np.abs(residual_order))

ax4.text(
    0.02,
    0.04,
    f"Central 99% shown\n"
    f"Rolling Window: {WINDOW:,}\n"
    f"Positive: {positive_ratio:.1f}%\n"
    f"Negative: {negative_ratio:.1f}%\n"
    f"Median |Residual|: ${median_abs_error:.2f}M",
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.90
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — RESIDUAL DIAGNOSTICS",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close()

# B — STANDARDIZED EFFECTS

effects = effect_stats["standardized_effect"].to_numpy()
effects_sorted = np.sort(effects)
m = len(effects_sorted)

if m >= 2:

    theoretical_effect_quantiles = stats.norm.ppf(
        (np.arange(1, m + 1) - 0.5) / m
    )

    fig, ax = plt.subplots(figsize=(14, 9))

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=100,
        alpha=0.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope * x_ref,
        linestyle="--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():

        idx = np.argmin(
            np.abs(
                effects_sorted
                - row["standardized_effect"]
            )
        )

        label = str(row[FEATURE])

        if len(label) > 25:
            label = label[:22] + "..."

        ax.annotate(
            label,
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(8, 6),
            textcoords="offset points",
            fontsize=10
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=1.8
    )

    ax.set_xlabel(
        "Theoretical Normal Quantiles",
        fontsize=13
    )

    ax.set_ylabel(
        "Standardized Effect",
        fontsize=13
    )

    ax.set_title(
        f"B — Standardized {FEATURE} Effects",
        fontsize=17,
        fontweight="bold"
    )

    ax.grid(alpha=0.20)

    plt.tight_layout()
    plt.show()
    plt.close()

# C — CATEGORY COMPONENT + RESIDUAL

categories = effect_stats[FEATURE].tolist()

for page_start in range(
    0,
    len(categories),
    CATEGORIES_PER_PAGE
):

    page_categories = categories[
        page_start:page_start + CATEGORIES_PER_PAGE
    ]

    n_categories = len(page_categories)
    ncols = 2
    nrows = int(np.ceil(n_categories / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(18, 7 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, category in zip(
        axes,
        page_categories
    ):

        category_data = analysis_df[
            analysis_df[FEATURE] == category
        ]

        x = (
            category_data["prediction"].to_numpy()
            / 1e6
        )

        y = (
            category_data["residual"].to_numpy()
            / 1e6
        )

        if len(x) >= 10:

            x_low, x_high = np.percentile(
                x,
                [1, 99]
            )

            y_low, y_high = np.percentile(
                y,
                [1, 99]
            )

            mask = (
                (x >= x_low)
                & (x <= x_high)
                & (y >= y_low)
                & (y <= y_high)
            )

            x_plot = x[mask]
            y_plot = y[mask]

        else:
            x_plot = x
            y_plot = y

        if len(x_plot) > 12000:

            rng = np.random.default_rng(RANDOM_STATE)

            idx = rng.choice(
                len(x_plot),
                12000,
                replace=False
            )

            x_scatter = x_plot[idx]
            y_scatter = y_plot[idx]

        else:

            x_scatter = x_plot
            y_scatter = y_plot

        ax.scatter(
            x_scatter,
            y_scatter,
            s=12,
            alpha=0.15,
            rasterized=True
        )

        if len(x_plot) >= 3:

            slope, intercept = np.polyfit(
                x_plot,
                y_plot,
                1
            )

            x_line = np.linspace(
                x_plot.min(),
                x_plot.max(),
                300
            )

            ax.plot(
                x_line,
                intercept + slope * x_line,
                linestyle="--",
                linewidth=2.5,
                label="Linear fit"
            )

            sort_idx = np.argsort(x_plot)

            x_sorted = x_plot[sort_idx]
            y_sorted = y_plot[sort_idx]

            smooth_window = max(
                50,
                min(
                    500,
                    len(x_sorted) // 20
                )
            )

            rolling_component = pd.Series(
                y_sorted
            ).rolling(
                smooth_window,
                center=True,
                min_periods=max(
                    20,
                    smooth_window // 4
                )
            ).median()

            ax.plot(
                x_sorted,
                rolling_component,
                linewidth=3,
                label="Smoothed trend"
            )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=2.2,
            color="red",
            label="Residual = 0"
        )

        ax.set_xlabel(
            "Predicted Price (Million USD)",
            fontsize=13
        )

        ax.set_ylabel(
            "Residual (Million USD)",
            fontsize=13
        )

        ax.set_title(
            str(category),
            fontsize=15,
            fontweight="bold",
            pad=12
        )

        ax.grid(
            alpha=0.22,
            linestyle="--"
        )

        row = effect_stats[
            effect_stats[FEATURE] == category
        ].iloc[0]

        mean_res = row["mean_residual"] / 1e6
        mae = row["MAE"] / 1e6
        rmse = row["RMSE"] / 1e6
        effect = row["standardized_effect"]

        direction = (
            "UNDER-PREDICTION"
            if effect > 0
            else "OVER-PREDICTION"
        )

        ax.text(
            0.03,
            0.97,
            f"N            : {int(row['count']):,}\n"
            f"Mean Residual: ${mean_res:.2f}M\n"
            f"MAE          : ${mae:.2f}M\n"
            f"RMSE         : ${rmse:.2f}M\n"
            f"Std. Effect  : {effect:+.3f}\n"
            f"Direction    : {direction}",
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            linespacing=1.4,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92
            )
        )

        ax.legend(
            loc="lower right",
            fontsize=9,
            frameon=True,
            framealpha=0.90
        )

    for ax in axes[n_categories:]:
        ax.axis("off")

    page_number = (
        page_start // CATEGORIES_PER_PAGE
    ) + 1

    total_pages = int(
        np.ceil(
            len(categories) / CATEGORIES_PER_PAGE
        )
    )

    fig.suptitle(
        f"COMPONENT + RESIDUAL PLOTS — {FEATURE}\n"
        f"Page {page_number}/{total_pages}",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.94]
    )

    plt.show()
    plt.close()

# D — MAE COMPARISON

mae_plot = effect_stats.sort_values(
    "MAE",
    ascending=True
)

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    mae_plot[FEATURE].astype(str),
    mae_plot["MAE"] / 1e6
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax.set_title(
    f"{FEATURE} — MAE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    labelsize=10,
    rotation=0
)

ax.tick_params(
    axis="y",
    labelsize=11
)

max_mae = (
    mae_plot["MAE"] / 1e6
).max()

for bar, value in zip(
    bars,
    mae_plot["MAE"] / 1e6
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mae * 0.015,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=0
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# E — MSE COMPARISON

mse_plot = effect_stats.sort_values(
    "RMSE",
    ascending=True
).copy()

mse_plot["MSE"] = mse_plot["RMSE"] ** 2
mse_plot["MSE_T"] = mse_plot["MSE"] / 1e12

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    mse_plot[FEATURE].astype(str),
    mse_plot["MSE_T"]
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "MSE (Trillion USD²)",
    fontsize=13
)

ax.set_title(
    f"{FEATURE} — MSE Comparison",
    fontsize=17,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    labelsize=10,
    rotation=0
)

ax.tick_params(
    axis="y",
    labelsize=11
)

max_mse = mse_plot["MSE_T"].max()

for bar, value in zip(
    bars,
    mse_plot["MSE_T"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mse * 0.015,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        rotation=0
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# EN PROBLEMLİ KATEGORİLER

if len(effect_stats) > 0:

    worst_mae = effect_stats.loc[
        effect_stats["MAE"].idxmax()
    ]

    worst_rmse = effect_stats.loc[
        effect_stats["RMSE"].idxmax()
    ]

    worst_effect = effect_stats.loc[
        effect_stats["absolute_standardized_effect"].idxmax()
    ]

    print("\n" + "=" * 90)
    print(f"{FEATURE.upper()} — HATA ANALİZİ")
    print("=" * 90)

    print(
        f"\nEn yüksek MAE   : "
        f"{worst_mae[FEATURE]} → "
        f"${worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : "
        f"{worst_rmse[FEATURE]} → "
        f"${worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En güçlü etki   : "
        f"{worst_effect[FEATURE]} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    if worst_effect["standardized_effect"] > 0:
        print(
            "\nYorum: Pozitif residual → "
            "Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )
    else:
        print(
            "\nYorum: Negatif residual → "
            "Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )

# ÖZET

print("\n" + "=" * 90)
print(f"{FEATURE.upper()} ANALİZİ TAMAMLANDI")
print("=" * 90)

print(
    f"Toplam gözlem          : "
    f"{len(analysis_df):,}"
)

print(
    f"Analiz edilen kategori : "
    f"{len(effect_stats):,}"
)

print(
    f"Global Mean Residual   : "
    f"${analysis_df['residual'].mean():,.2f}"
)

print(
    f"Global MAE             : "
    f"${analysis_df['absolute_error'].mean():,.2f}"
)

print(
    f"Global RMSE            : "
    f"${np.sqrt(np.mean(analysis_df['residual'] ** 2)):,.2f}"
)

print(
    f"Global Median Residual : "
    f"${analysis_df['residual'].median():,.2f}"
)

corr_singleb değişkenine ilişkin residual (artık) ve hata analizleri, modelin veri gruplarındaki bütçe ölçeklerine ve kategorik risk profillerine göre belirgin bir biçimde farklılaştığını ortaya koymaktadır. Tüm gruplarda residual ortalamalarının pozitif kalması, modelin genel olarak sistematik bir eksik tahmin (under-prediction) eğiliminde olduğunu ve gerçek ihale bedellerinin (Actual) model tahminlerinin (Prediction) üzerinde kaldığını doğrulamaktadır.

**1. Residual Diagnostik Grafikleri (A1 – A4)**

**A1 (QQ-Plot) & A3 (Histogram):** Hata dağılımı teorik normal dağılımdan dramatik şekilde sapmaktadır. Histogramda 0-2M$ bandında yoğunlaşan artıklar, sağ tarafa doğru çok uzun ve kalın bir kuyruk (right-skewed heavy tail) oluşturmaktadır.
**A2 (Residuals vs Fits Heatmap):** Tahmin edilen değerler büyüdükçe artıkların saçılım genişliği artmakta, tipik bir heteroskedastisite (değişen varyans) sergilemektedir. Tahmin fiyatı yükseldikçe modelin belirsizlik payı büyümektedir.
**A4 (Rolling Bias):** 2.000 gözlemlik hareketli ortanca (Rolling Median) sıfır eksenine tam paralel seyretmektedir. Bu durum, veri sıralamasına bağlı zamansal veya sistemsel bir kayma (bias) olmadığını, hatanın tamamen veri gruplarının bütçe büyüklüklerinden kaynaklandığını gösterir.

**2. Kategorik Hata Dağılımları ve Çözümlemesi**

**Kategori 1 (Düşük Risk / Kararlı Grup):**
Metrikler: MAE = $1.54M, RMSE = $11.14M, MSE = 123.99T, Std. Effect = +0.074
Değerlendirme: Modelin en başarılı ve kararlı tahmin ürettiği kategoridir. Bütçe ölçeğinin görece dar bir aralıkta kalması ve uç değerlerin azlığı nedeniyle varyans son derece düşüktür.

**Kategori 0 (Yüksek Hacimli / Yüksek Uç Değer Riski):**
Metrikler: MAE = $1.96M, RMSE = $16.43M, MSE = 269.95T, Std. Effect = +0.100
Değerlendirme: 31.005 gözlem ile toplam verinin %69.5'ini oluşturmaktadır. MAE seviyesi makul olsa da, en yüksek RMSE ve MSE değerleri bu grupta ortaya çıkmıştır. Bunun sebebi, verinin ezici çoğunluğunun küçük bütçeli olması ancak grupta birkaç adet devasa bütçeli uç değerin yer almasıdır. Karesel hata metrikleri (RMSE/MSE) bu uç değerleri ağır cezalandırdığı için varyans yüksek çıkmaktadır.

**Kategori -1 (Aşırı Bütçeli Uç Değer / En Yüksek MAE):**
Metrikler: MAE = $2.57M, RMSE = $16.33M, MSE = 266.59T, Std. Effect = +0.133
Değerlendirme: Modelin ortalama bazda en çok sapma yaptığı ve en yüksek sistematik etkiyi (+0.133) gösterdiği kategoridir. Component + Residual grafiği incelendiğinde, bu kategorideki tahmin ekseninin $350M bandına kadar uzandığı ve gerçek değerlerin tahminlerin $40M üzerinde kaldığı noktalar görülmektedir. Model, bu kategorideki devasa ihale tutarlarını sönümlemekte ve fiyatı yukarıya doğru ölçekleyemeyerek ağır eksik tahmin yapmaktadır.

**Modelin Hata Yapma Nedenleri ve Kök Sebepler**

Ölçek Sapması (Scale Volatility): Model, -1 kategorisinde yer alan çok yüksek bütçeli ihalelerin büyüklüğünü açıklayacak yeterli öznitelik (feature) derinliğine sahip değildir. Bu nedenle yüksek tutarları alt sınırlara doğru bastırmaktadır.
Uç Değer Hassasiyeti: $0$ kategorisinde gözlem sayısı çok fazla olduğu için genel performans iyi görünse de, karesel hata metriklerinin (MSE) yüksekliği, bu grup içinde az sayıda ama çok büyük tutarlı fiyat kaçaklarının olduğunu gösterir.
Doğrusal Olmayan Dinamikler: Component + Residual grafiklerindeki Smoothed Trend çizgilerinin yüksek tahminlerde düzleşmesi, modelin yüksek fiyat aralıklarında doğrusal olmayan ilişkileri öğrenmekte yetersiz kaldığını kanıtlamaktadır.

## CORR_PROC (İhalenin satın alma / ihale usulü ve süreç adımlarından kaynaklanan usulsüzlük ve risk düzeyini ölçen bir göstergedir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "corr_proc"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
ROLLING_WINDOW_MAX = 2000
CATEGORIES_PER_PAGE = 8
RANDOM_STATE = 42

if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

columns_lower = {c.lower(): c for c in test_predictions.columns}

actual_candidates = [
    "actual_usd", "actual", "y_true", "true",
    "target", "actual_price", "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd", "predicted", "prediction",
    "pred", "y_pred", "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd", "residual", "error", "prediction_error"
]

actual_col = next(
    (columns_lower[c] for c in actual_candidates if c in columns_lower),
    None
)

predicted_col = next(
    (columns_lower[c] for c in predicted_candidates if c in columns_lower),
    None
)

residual_col = next(
    (columns_lower[c] for c in residual_candidates if c in columns_lower),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test          : {len(df_test):,}\n"
        f"test_predictions : {len(test_predictions):,}"
    )

analysis_df = pd.DataFrame({
    FEATURE: df_test[FEATURE].to_numpy(),
    "actual": pd.to_numeric(
        test_predictions[actual_col],
        errors="coerce"
    ).to_numpy(),
    "prediction": pd.to_numeric(
        test_predictions[predicted_col],
        errors="coerce"
    ).to_numpy(),
    "residual": pd.to_numeric(
        test_predictions[residual_col],
        errors="coerce"
    ).to_numpy()
})

analysis_df[FEATURE] = (
    analysis_df[FEATURE]
    .astype(str)
    .replace({
        "None": "MISSING",
        "nan": "MISSING",
        "null": "MISSING"
    })
)

analysis_df["absolute_error"] = analysis_df["residual"].abs()

valid_mask = (
    np.isfinite(analysis_df["actual"].to_numpy())
    & np.isfinite(analysis_df["prediction"].to_numpy())
    & np.isfinite(analysis_df["residual"].to_numpy())
)

analysis_df = (
    analysis_df
    .loc[valid_mask]
    .reset_index(drop=True)
)

print(f"\nGeçerli gözlem sayısı: {len(analysis_df):,}")

category_stats = (
    analysis_df
    .groupby(FEATURE)
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        std_residual=("residual", "std"),
        MAE=("absolute_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

effect_stats = category_stats[
    category_stats["count"] >= MIN_CATEGORY_COUNT
].copy()

global_std = analysis_df["residual"].std()

effect_stats["standardized_effect"] = (
    effect_stats["mean_residual"] / global_std
)

effect_stats["absolute_standardized_effect"] = (
    effect_stats["standardized_effect"].abs()
)

# Kategorileri doğal / sayısal sırada tut
def natural_sort_key(value):
    value = str(value)
    try:
        return (0, float(value))
    except ValueError:
        return (1, value.lower())

effect_stats = effect_stats.sort_values(
    FEATURE,
    key=lambda s: s.map(natural_sort_key)
).reset_index(drop=True)

print(f"Analiz edilen kategori sayısı: {len(effect_stats)}")

visual_table = effect_stats[
    [
        FEATURE,
        "count",
        "mean_residual",
        "MAE",
        "RMSE",
        "standardized_effect"
    ]
].copy()

visual_table.columns = [
    "Category",
    "Observations",
    "Mean Residual",
    "MAE",
    "RMSE",
    "Standardized Effect"
]

def highlight_max(series):
    mask = series == series.max()
    return [
        "background-color: #ffd6d6;color: #8b0000;font-weight: bold;"
        if value else ""
        for value in mask
    ]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "${:,.0f}",
        "MAE": "${:,.0f}",
        "RMSE": "${:,.0f}",
        "Standardized Effect": "{:+.3f}"
    })
    .apply(highlight_max, subset=["MAE"])
    .apply(highlight_max, subset=["RMSE"])
    .apply(highlight_max, subset=["Mean Residual"])
    .apply(highlight_max, subset=["Standardized Effect"])
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "11pt",
            "padding": "8px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#404040"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "9px")
            ]
        }
    ])
    .set_caption(f"{FEATURE.upper()} — RESIDUAL ANALYSIS")
)

display(styled_table)

residual_values = analysis_df["residual"].to_numpy()
prediction_values = analysis_df["prediction"].to_numpy()

fig, axes = plt.subplots(
    2,
    2,
    figsize=(19, 14)
)

ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — NORMAL PROBABILITY PLOT — LINEAR SCALE

rng = np.random.default_rng(RANDOM_STATE)

residual_plot = (
    rng.choice(
        residual_values,
        size=PLOT_SAMPLE,
        replace=False
    )
    if len(residual_values) > PLOT_SAMPLE
    else residual_values
)

sorted_residuals = np.sort(residual_plot)
n = len(sorted_residuals)

theoretical_quantiles = stats.norm.ppf(
    (np.arange(1, n + 1) - 0.5) / n
)

residual_mean = np.mean(sorted_residuals)
residual_std = np.std(
    sorted_residuals,
    ddof=1
)

standardized_residuals = (
    (sorted_residuals - residual_mean)
    / residual_std
)

low, high = np.percentile(
    standardized_residuals,
    [0.5, 99.5]
)

mask = (
    (standardized_residuals >= low)
    & (standardized_residuals <= high)
)

x_a1 = theoretical_quantiles[mask]
y_a1 = standardized_residuals[mask]

ax1.scatter(
    x_a1,
    y_a1,
    s=14,
    alpha=0.22,
    rasterized=True
)

ax1.plot(
    [x_a1.min(), x_a1.max()],
    [x_a1.min(), x_a1.max()],
    linestyle="--",
    linewidth=2.5,
    label="Normal reference"
)

ax1.axhline(
    0,
    linestyle=":",
    linewidth=1.5
)

ax1.set_xlabel(
    "Theoretical Normal Quantiles",
    fontsize=13
)

ax1.set_ylabel(
    "Standardized Residual",
    fontsize=13
)

ax1.set_title(
    "A1 — Normal Probability Plot\nCentral 99% — Linear Scale",
    fontsize=16,
    fontweight="bold"
)

ax1.grid(
    alpha=0.25,
    linestyle="--"
)

ax1.legend(
    loc="upper left",
    fontsize=10
)

# A2 — RESIDUAL VS FITS — DENSITY HEATMAP

prediction_m = prediction_values / 1e6
residual_m = residual_values / 1e6

x_low, x_high = np.percentile(
    prediction_m,
    [0.5, 99.5]
)

y_low, y_high = np.percentile(
    residual_m,
    [0.5, 99.5]
)

mask = (
    (prediction_m >= x_low)
    & (prediction_m <= x_high)
    & (residual_m >= y_low)
    & (residual_m <= y_high)
)

x_plot = prediction_m[mask]
y_plot = residual_m[mask]

hb = ax2.hexbin(
    x_plot,
    y_plot,
    gridsize=55,
    mincnt=1,
    bins="log",
    cmap="viridis",
    linewidths=0.15
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

median_residual = np.median(y_plot)

ax2.axhline(
    median_residual,
    linestyle=":",
    linewidth=2,
    label=f"Median = {median_residual:.2f}M"
)

cbar = fig.colorbar(
    hb,
    ax=ax2,
    pad=0.02
)

cbar.set_label(
    "Observation Density\n(log scale)",
    fontsize=12
)

ax2.set_xlabel(
    "Predicted Price (Million USD)",
    fontsize=13
)

ax2.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax2.set_title(
    "A2 — Residuals Versus Fits\nDensity Heatmap",
    fontsize=16,
    fontweight="bold"
)

ax2.grid(
    alpha=0.20,
    linestyle="--"
)

ax2.legend(
    loc="upper right",
    fontsize=10
)

# A3 — RESIDUAL DISTRIBUTION

residual_low, residual_high = np.percentile(
    residual_values,
    [0.5, 99.5]
)

central_residuals = residual_values[
    (residual_values >= residual_low)
    & (residual_values <= residual_high)
]

ax3.hist(
    central_residuals / 1e6,
    bins=35,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.5
)

mean_residual = (
    np.mean(residual_values) / 1e6
)

ax3.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Residual = 0"
)

ax3.axvline(
    mean_residual,
    linestyle=":",
    linewidth=2,
    label=f"Mean = {mean_residual:.2f}M"
)

ax3.set_xlabel(
    "Residual (Million USD)",
    fontsize=13
)

ax3.set_ylabel(
    "Frequency",
    fontsize=13
)

ax3.set_title(
    "A3 — Residual Distribution\nCentral 99%",
    fontsize=16,
    fontweight="bold"
)

ax3.legend(fontsize=10)

ax3.grid(
    axis="y",
    alpha=0.20
)

# A4 — RESIDUAL VS OBSERVATION ORDER

order = np.arange(
    1,
    len(analysis_df) + 1
)

residual_order = (
    analysis_df["residual"].to_numpy()
    / 1e6
)

q_low, q_high = np.percentile(
    residual_order,
    [0.5, 99.5]
)

mask = (
    (residual_order >= q_low)
    & (residual_order <= q_high)
)

order_plot = order[mask]
residual_plot = residual_order[mask]

ax4.scatter(
    order_plot,
    residual_plot,
    s=4,
    alpha=0.055,
    rasterized=True
)

order_df = pd.DataFrame({
    "order": order_plot,
    "residual": residual_plot
})

WINDOW = max(
    500,
    min(
        ROLLING_WINDOW_MAX,
        len(order_df) // 20
    )
)

min_periods = max(
    50,
    WINDOW // 4
)

rolling_median = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .median()
)

rolling_q10 = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .quantile(0.10)
)

rolling_q90 = (
    order_df["residual"]
    .rolling(
        WINDOW,
        center=True,
        min_periods=min_periods
    )
    .quantile(0.90)
)

ax4.fill_between(
    order_df["order"].to_numpy(),
    rolling_q10.to_numpy(),
    rolling_q90.to_numpy(),
    alpha=0.20,
    label="Rolling 10–90% error range"
)

ax4.plot(
    order_df["order"].to_numpy(),
    rolling_median.to_numpy(),
    linewidth=3,
    label=f"Rolling Median ({WINDOW:,})"
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2.2,
    color="red",
    label="Residual = 0"
)

global_median = np.median(residual_order)

ax4.axhline(
    global_median,
    linestyle="-.",
    linewidth=1.6,
    alpha=0.75,
    label=f"Global Median = {global_median:+.2f}M"
)

ax4.set_yscale(
    "symlog",
    linthresh=1,
    linscale=1.2
)

ax4.set_xlabel(
    "Observation Order",
    fontsize=13
)

ax4.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax4.set_title(
    "A4 — Residuals Versus Observation Order\nRolling Bias",
    fontsize=16,
    fontweight="bold"
)

ax4.grid(
    which="major",
    alpha=0.25,
    linestyle="--"
)

ax4.grid(
    which="minor",
    alpha=0.10,
    linestyle=":"
)

ax4.legend(
    loc="upper right",
    fontsize=9
)

positive_ratio = (
    np.mean(residual_order > 0) * 100
)

negative_ratio = (
    np.mean(residual_order < 0) * 100
)

median_abs_error = (
    np.median(np.abs(residual_order))
)

ax4.text(
    0.02,
    0.04,
    f"Central 99% shown\n"
    f"Rolling Window: {WINDOW:,}\n"
    f"Positive: {positive_ratio:.1f}%\n"
    f"Negative: {negative_ratio:.1f}%\n"
    f"Median |Residual|: ${median_abs_error:.2f}M",
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.90
    )
)

fig.suptitle(
    f"{FEATURE.upper()} — RESIDUAL DIAGNOSTICS",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

plt.show()
plt.close()

# B — STANDARDIZED EFFECTS

effects = effect_stats[
    "standardized_effect"
].to_numpy()

effects_sorted = np.sort(effects)
m = len(effects_sorted)

if m >= 2:

    theoretical_effect_quantiles = stats.norm.ppf(
        (np.arange(1, m + 1) - 0.5) / m
    )

    fig, ax = plt.subplots(
        figsize=(14, 9)
    )

    ax.scatter(
        theoretical_effect_quantiles,
        effects_sorted,
        s=100,
        alpha=0.85
    )

    slope, intercept = np.polyfit(
        theoretical_effect_quantiles,
        effects_sorted,
        1
    )

    x_ref = np.array([
        theoretical_effect_quantiles.min(),
        theoretical_effect_quantiles.max()
    ])

    ax.plot(
        x_ref,
        intercept + slope * x_ref,
        linestyle="--",
        linewidth=2
    )

    for _, row in effect_stats.iterrows():

        idx = np.argmin(
            np.abs(
                effects_sorted
                - row["standardized_effect"]
            )
        )

        label = str(row[FEATURE])

        ax.annotate(
            label,
            (
                theoretical_effect_quantiles[idx],
                effects_sorted[idx]
            ),
            xytext=(8, 6),
            textcoords="offset points",
            fontsize=10
        )

    ax.axhline(
        0,
        linestyle=":",
        linewidth=1.8
    )

    ax.set_xlabel(
        "Theoretical Normal Quantiles",
        fontsize=13
    )

    ax.set_ylabel(
        "Standardized Effect",
        fontsize=13
    )

    ax.set_title(
        f"B — Standardized {FEATURE} Effects",
        fontsize=17,
        fontweight="bold"
    )

    ax.grid(alpha=0.20)

    plt.tight_layout()
    plt.show()
    plt.close()

# C — CATEGORY COMPONENT + RESIDUAL

categories = effect_stats[FEATURE].tolist()

for page_start in range(
    0,
    len(categories),
    CATEGORIES_PER_PAGE
):

    page_categories = categories[
        page_start:
        page_start + CATEGORIES_PER_PAGE
    ]

    n_categories = len(page_categories)
    ncols = 2
    nrows = int(
        np.ceil(n_categories / ncols)
    )

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(18, 7 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, category in zip(
        axes,
        page_categories
    ):

        category_data = analysis_df[
            analysis_df[FEATURE] == category
        ]

        x = (
            category_data["prediction"].to_numpy()
            / 1e6
        )

        y = (
            category_data["residual"].to_numpy()
            / 1e6
        )

        if len(x) >= 10:

            x_low, x_high = np.percentile(
                x,
                [1, 99]
            )

            y_low, y_high = np.percentile(
                y,
                [1, 99]
            )

            mask = (
                (x >= x_low)
                & (x <= x_high)
                & (y >= y_low)
                & (y <= y_high)
            )

            x_plot = x[mask]
            y_plot = y[mask]

        else:

            x_plot = x
            y_plot = y

        if len(x_plot) > 12000:

            rng = np.random.default_rng(
                RANDOM_STATE
            )

            idx = rng.choice(
                len(x_plot),
                12000,
                replace=False
            )

            x_scatter = x_plot[idx]
            y_scatter = y_plot[idx]

        else:

            x_scatter = x_plot
            y_scatter = y_plot

        ax.scatter(
            x_scatter,
            y_scatter,
            s=12,
            alpha=0.15,
            rasterized=True
        )

        if len(x_plot) >= 3:

            slope, intercept = np.polyfit(
                x_plot,
                y_plot,
                1
            )

            x_line = np.linspace(
                x_plot.min(),
                x_plot.max(),
                300
            )

            ax.plot(
                x_line,
                intercept + slope * x_line,
                linestyle="--",
                linewidth=2.5,
                label="Linear fit"
            )

            sort_idx = np.argsort(x_plot)

            x_sorted = x_plot[sort_idx]
            y_sorted = y_plot[sort_idx]

            smooth_window = max(
                50,
                min(
                    500,
                    len(x_sorted) // 20
                )
            )

            rolling_component = (
                pd.Series(y_sorted)
                .rolling(
                    smooth_window,
                    center=True,
                    min_periods=max(
                        20,
                        smooth_window // 4
                    )
                )
                .median()
            )

            ax.plot(
                x_sorted,
                rolling_component,
                linewidth=3,
                label="Smoothed trend"
            )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=2.2,
            color="red",
            label="Residual = 0"
        )

        ax.set_xlabel(
            "Predicted Price (Million USD)",
            fontsize=13
        )

        ax.set_ylabel(
            "Residual (Million USD)",
            fontsize=13
        )

        ax.set_title(
            str(category),
            fontsize=15,
            fontweight="bold",
            pad=12
        )

        ax.grid(
            alpha=0.22,
            linestyle="--"
        )

        row = effect_stats[
            effect_stats[FEATURE] == category
        ].iloc[0]

        mean_res = (
            row["mean_residual"] / 1e6
        )

        mae = row["MAE"] / 1e6
        rmse = row["RMSE"] / 1e6
        effect = row["standardized_effect"]

        direction = (
            "UNDER-PREDICTION"
            if effect > 0
            else "OVER-PREDICTION"
        )

        ax.text(
            0.03,
            0.97,
            f"N            : {int(row['count']):,}\n"
            f"Mean Residual: ${mean_res:.2f}M\n"
            f"MAE          : ${mae:.2f}M\n"
            f"RMSE         : ${rmse:.2f}M\n"
            f"Std. Effect  : {effect:+.3f}\n"
            f"Direction    : {direction}",
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            linespacing=1.4,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92
            )
        )

        ax.legend(
            loc="lower right",
            fontsize=9,
            frameon=True,
            framealpha=0.90
        )

    for ax in axes[n_categories:]:
        ax.axis("off")

    page_number = (
        page_start // CATEGORIES_PER_PAGE
    ) + 1

    total_pages = int(
        np.ceil(
            len(categories)
            / CATEGORIES_PER_PAGE
        )
    )

    fig.suptitle(
        f"COMPONENT + RESIDUAL PLOTS — {FEATURE}\n"
        f"Page {page_number}/{total_pages}",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.94]
    )

    plt.show()
    plt.close()

# D — MAE COMPARISON — SIRALI X EKSENİ

mae_plot = effect_stats.copy()

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    np.arange(len(mae_plot)),
    mae_plot["MAE"] / 1e6
)

ax.set_xticks(
    np.arange(len(mae_plot))
)

ax.set_xticklabels(
    mae_plot[FEATURE].astype(str),
    rotation=0
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax.set_title(
    f"{FEATURE} — MAE Comparison",
    fontsize=17,
    fontweight="bold"
)

max_mae = (
    mae_plot["MAE"] / 1e6
).max()

for bar, value in zip(
    bars,
    mae_plot["MAE"] / 1e6
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mae * 0.015,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# E — RMSE COMPARISON — SIRALI X EKSENİ

rmse_plot = effect_stats.copy()

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    np.arange(len(rmse_plot)),
    rmse_plot["RMSE"] / 1e6
)

ax.set_xticks(
    np.arange(len(rmse_plot))
)

ax.set_xticklabels(
    rmse_plot[FEATURE].astype(str),
    rotation=0
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "RMSE (Million USD)",
    fontsize=13
)

ax.set_title(
    f"{FEATURE} — RMSE Comparison",
    fontsize=17,
    fontweight="bold"
)

max_rmse = (
    rmse_plot["RMSE"] / 1e6
).max()

for bar, value in zip(
    bars,
    rmse_plot["RMSE"] / 1e6
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_rmse * 0.015,
        f"${value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# F — MSE COMPARISON — SIRALI X EKSENİ

mse_plot = effect_stats.copy()

mse_plot["MSE"] = (
    mse_plot["RMSE"] ** 2
)

mse_plot["MSE_T"] = (
    mse_plot["MSE"] / 1e12
)

fig, ax = plt.subplots(
    figsize=(15, 8)
)

bars = ax.bar(
    np.arange(len(mse_plot)),
    mse_plot["MSE_T"]
)

ax.set_xticks(
    np.arange(len(mse_plot))
)

ax.set_xticklabels(
    mse_plot[FEATURE].astype(str),
    rotation=0
)

ax.set_xlabel(
    FEATURE,
    fontsize=13
)

ax.set_ylabel(
    "MSE (Trillion USD²)",
    fontsize=13
)

ax.set_title(
    f"{FEATURE} — MSE Comparison",
    fontsize=17,
    fontweight="bold"
)

max_mse = mse_plot["MSE_T"].max()

for bar, value in zip(
    bars,
    mse_plot["MSE_T"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_mse * 0.015,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()
plt.show()
plt.close()

# EN PROBLEMLİ KATEGORİLER

if len(effect_stats) > 0:

    worst_mae = effect_stats.loc[
        effect_stats["MAE"].idxmax()
    ]

    worst_rmse = effect_stats.loc[
        effect_stats["RMSE"].idxmax()
    ]

    worst_effect = effect_stats.loc[
        effect_stats["absolute_standardized_effect"].idxmax()
    ]

    print("\n" + "=" * 90)
    print(f"{FEATURE.upper()} — HATA ANALİZİ")
    print("=" * 90)

    print(
        f"\nEn yüksek MAE   : "
        f"{worst_mae[FEATURE]} → "
        f"${worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek RMSE  : "
        f"{worst_rmse[FEATURE]} → "
        f"${worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En güçlü etki   : "
        f"{worst_effect[FEATURE]} → "
        f"{worst_effect['standardized_effect']:+.3f}"
    )

    if worst_effect["standardized_effect"] > 0:

        print(
            "\nYorum: Pozitif residual → "
            "Actual > Prediction → "
            "model UNDER-PREDICTION yapıyor."
        )

    else:

        print(
            "\nYorum: Negatif residual → "
            "Actual < Prediction → "
            "model OVER-PREDICTION yapıyor."
        )

print("\n" + "=" * 90)
print(f"{FEATURE.upper()} ANALİZİ TAMAMLANDI")
print("=" * 90)

print(
    f"Toplam gözlem          : "
    f"{len(analysis_df):,}"
)

print(
    f"Analiz edilen kategori : "
    f"{len(effect_stats):,}"
)

print(
    f"Global Mean Residual   : "
    f"${analysis_df['residual'].mean():,.2f}"
)

print(
    f"Global MAE             : "
    f"${analysis_df['absolute_error'].mean():,.2f}"
)

print(
    f"Global RMSE            : "
    f"${np.sqrt(np.mean(analysis_df['residual'] ** 2)):,.2f}"
)

print(
    f"Global Median Residual : "
    f"${analysis_df['residual'].median():,.2f}"
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "corr_proc"
MIN_CATEGORY_COUNT = 30
PLOT_SAMPLE = 10000
RANDOM_STATE = 42
N_RISK_BINS = 5

# KONTROLLER

if "test_predictions" not in globals():
    raise NameError(
        "test_predictions bulunamadı. "
        "Önce test_predictions.csv yükleme hücresini çalıştır."
    )

if "df_test" not in globals():
    raise NameError("df_test bulunamadı.")

if FEATURE not in df_test.columns:
    raise KeyError(f"{FEATURE} df_test içinde bulunamadı.")

print(f"Test predictions shape: {test_predictions.shape}")

# KOLONLARI BUL

columns_lower = {
    c.lower(): c
    for c in test_predictions.columns
}

actual_candidates = [
    "actual_usd",
    "actual",
    "y_true",
    "true",
    "target",
    "actual_price",
    "actual_tender_finalpriceusd"
]

predicted_candidates = [
    "predicted_usd",
    "predicted",
    "prediction",
    "pred",
    "y_pred",
    "predicted_price",
    "predicted_tender_finalpriceusd"
]

residual_candidates = [
    "residual_usd",
    "residual",
    "error",
    "prediction_error"
]

actual_col = next(
    (
        columns_lower[c]
        for c in actual_candidates
        if c in columns_lower
    ),
    None
)

predicted_col = next(
    (
        columns_lower[c]
        for c in predicted_candidates
        if c in columns_lower
    ),
    None
)

residual_col = next(
    (
        columns_lower[c]
        for c in residual_candidates
        if c in columns_lower
    ),
    None
)

if actual_col is None:
    raise ValueError("Actual kolonu bulunamadı.")

if predicted_col is None:
    raise ValueError("Prediction kolonu bulunamadı.")

if residual_col is None:
    raise ValueError("Residual kolonu bulunamadı.")

print(f"Actual kolon    : {actual_col}")
print(f"Prediction kolon: {predicted_col}")
print(f"Residual kolon  : {residual_col}")

if len(df_test) != len(test_predictions):
    raise ValueError(
        f"Satır sayıları uyuşmuyor!\n"
        f"df_test          : {len(df_test):,}\n"
        f"test_predictions : {len(test_predictions):,}"
    )

# ANALİZ DATAFRAME

analysis_df = pd.DataFrame({
    FEATURE: df_test[FEATURE].to_numpy(),

    "actual": pd.to_numeric(
        test_predictions[actual_col],
        errors="coerce"
    ).to_numpy(),

    "prediction": pd.to_numeric(
        test_predictions[predicted_col],
        errors="coerce"
    ).to_numpy(),

    "residual": pd.to_numeric(
        test_predictions[residual_col],
        errors="coerce"
    ).to_numpy()
})

analysis_df[FEATURE] = pd.to_numeric(
    analysis_df[FEATURE],
    errors="coerce"
)

analysis_df["absolute_error"] = (
    analysis_df["residual"].abs()
)

valid_mask = (
    np.isfinite(analysis_df[FEATURE].to_numpy())
    & np.isfinite(analysis_df["actual"].to_numpy())
    & np.isfinite(analysis_df["prediction"].to_numpy())
    & np.isfinite(analysis_df["residual"].to_numpy())
)

analysis_df = (
    analysis_df.loc[valid_mask]
    .reset_index(drop=True)
)

print(
    f"\nGeçerli gözlem sayısı: "
    f"{len(analysis_df):,}"
)

print(
    f"{FEATURE} minimum: "
    f"{analysis_df[FEATURE].min():.4f}"
)

print(
    f"{FEATURE} maksimum: "
    f"{analysis_df[FEATURE].max():.4f}"
)

print(
    f"{FEATURE} farklı değer sayısı: "
    f"{analysis_df[FEATURE].nunique():,}"
)

# RİSK KOLONU TÜRÜNÜ BELİRLE

unique_risk = np.sort(
    analysis_df[FEATURE].dropna().unique()
)

if len(unique_risk) <= 10:
    risk_type = "categorical"

elif (
    analysis_df[FEATURE].min() >= 0
    and analysis_df[FEATURE].max() <= 1
):
    risk_type = "continuous_01"

else:
    risk_type = "continuous"

print(f"Risk kolon tipi: {risk_type}")

# RİSK SEGMENTLERİ

if risk_type == "categorical":

    analysis_df["risk_bin"] = (
        analysis_df[FEATURE]
        .astype(str)
    )

elif risk_type == "continuous_01":

    bins = np.linspace(
        0,
        1,
        N_RISK_BINS + 1
    )

    analysis_df["risk_bin"] = pd.cut(
        analysis_df[FEATURE],
        bins=bins,
        include_lowest=True,
        duplicates="drop"
    )

else:

    quantile_edges = np.unique(
        np.nanquantile(
            analysis_df[FEATURE],
            np.linspace(
                0,
                1,
                N_RISK_BINS + 1
            )
        )
    )

    if len(quantile_edges) > 2:

        analysis_df["risk_bin"] = pd.cut(
            analysis_df[FEATURE],
            bins=quantile_edges,
            include_lowest=True,
            duplicates="drop"
        )

    else:

        analysis_df["risk_bin"] = (
            analysis_df[FEATURE]
            .astype(str)
        )

# RİSK SEGMENT İSTATİSTİKLERİ

risk_stats = (
    analysis_df
    .groupby(
        "risk_bin",
        observed=True
    )
    .agg(
        count=("residual", "size"),
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "residual",
            lambda x: np.sqrt(
                np.mean(x ** 2)
            )
        ),
        actual_mean=("actual", "mean"),
        prediction_mean=("prediction", "mean")
    )
    .reset_index()
)

risk_stats = risk_stats[
    risk_stats["count"] >= MIN_CATEGORY_COUNT
].copy()

risk_stats["bias_m"] = (
    risk_stats["mean_residual"] / 1e6
)

risk_stats["MAE_m"] = (
    risk_stats["MAE"] / 1e6
)

risk_stats["RMSE_m"] = (
    risk_stats["RMSE"] / 1e6
)

risk_stats["actual_mean_m"] = (
    risk_stats["actual_mean"] / 1e6
)

risk_stats["prediction_mean_m"] = (
    risk_stats["prediction_mean"] / 1e6
)

print(
    f"\nAnaliz edilen risk segmenti: "
    f"{len(risk_stats)}"
)

# RİSK İSTATİSTİK TABLOSU

visual_table = risk_stats[
    [
        "risk_bin",
        "count",
        "mean_residual",
        "MAE",
        "RMSE",
        "actual_mean",
        "prediction_mean"
    ]
].copy()

visual_table.columns = [
    "Risk Segment",
    "Observations",
    "Mean Residual",
    "MAE",
    "RMSE",
    "Actual Mean",
    "Predicted Mean"
]

styled_table = (
    visual_table.style
    .format({
        "Observations": "{:,.0f}",
        "Mean Residual": "${:,.0f}",
        "MAE": "${:,.0f}",
        "RMSE": "${:,.0f}",
        "Actual Mean": "${:,.0f}",
        "Predicted Mean": "${:,.0f}"
    })
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "11pt",
            "padding": "8px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#404040"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "9px")
            ]
        }
    ])
    .set_caption(
        f"{FEATURE.upper()} — RISK SEGMENT RESIDUAL ANALYSIS"
    )
)

display(styled_table)

# GRAFİKLER İÇİN ÖRNEKLEM

rng = np.random.default_rng(
    RANDOM_STATE
)

if len(analysis_df) > PLOT_SAMPLE:

    sample_idx = rng.choice(
        len(analysis_df),
        PLOT_SAMPLE,
        replace=False
    )

    plot_df = (
        analysis_df
        .iloc[sample_idx]
        .copy()
    )

else:

    plot_df = analysis_df.copy()

# 2x2 RESIDUAL DIAGNOSTIC PANELİ

fig, axes = plt.subplots(
    2,
    2,
    figsize=(20, 15)
)

ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — ABSOLUTE ERROR DISTRIBUTION BY RISK SEGMENT

risk_order = (
    risk_stats["risk_bin"]
    .tolist()
)

absolute_error_box_data = []

for risk_value in risk_order:

    values = analysis_df.loc[
        analysis_df["risk_bin"] == risk_value,
        "absolute_error"
    ].to_numpy() / 1e6

    absolute_error_box_data.append(
        values
    )

box = ax1.boxplot(
    absolute_error_box_data,
    positions=np.arange(
        1,
        len(absolute_error_box_data) + 1
    ),
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={
        "color": "black",
        "linewidth": 2.5
    },
    boxprops={
        "alpha": 0.65,
        "linewidth": 1.5
    },
    whiskerprops={
        "linewidth": 1.5
    },
    capprops={
        "linewidth": 1.5
    }
)

# MEDIAN ERROR TREND

absolute_medians = [
    np.median(values)
    for values in absolute_error_box_data
]

ax1.plot(
    np.arange(
        1,
        len(absolute_medians) + 1
    ),
    absolute_medians,
    marker="o",
    markersize=7,
    linewidth=2.5,
    label="Median absolute error"
)

ax1.set_xticks(
    np.arange(
        1,
        len(risk_order) + 1
    )
)

ax1.set_xticklabels(
    [
        str(value)
        for value in risk_order
    ],
    fontsize=10
)

ax1.set_xlabel(
    "Risk Segment",
    fontsize=13
)

ax1.set_ylabel(
    "Absolute Error (Million USD)",
    fontsize=13
)

ax1.set_title(
    "A1 — Absolute Error Distribution by Risk Segment",
    fontsize=17,
    fontweight="bold"
)

ax1.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax1.legend(
    fontsize=9,
    loc="best"
)

# MEDYAN DEĞERLERİNİ GÖSTER

for i, median_value in enumerate(
    absolute_medians,
    start=1
):

    ax1.text(
        i,
        median_value,
        f"{median_value:.2f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

# A2 — RISK SEGMENTS VS MAE + BIAS

x_pos = np.arange(
    len(risk_stats)
)

width = 0.38

bars_mae = ax2.bar(
    x_pos - width / 2,
    risk_stats["MAE_m"],
    width,
    label="MAE"
)

ax2.set_ylabel(
    "MAE (Million USD)",
    fontsize=13
)

ax2.set_xlabel(
    "Risk Segment",
    fontsize=13
)

ax2.set_xticks(
    x_pos
)

ax2.set_xticklabels(
    risk_stats["risk_bin"].astype(str),
    rotation=0
)

ax2.set_title(
    "A2 — Risk Segments vs MAE and Bias",
    fontsize=17,
    fontweight="bold"
)

ax2.grid(
    axis="y",
    alpha=0.20
)

ax2_twin = ax2.twinx()

bars_bias = ax2_twin.bar(
    x_pos + width / 2,
    risk_stats["bias_m"],
    width,
    alpha=0.55,
    label="Mean Residual / Bias"
)

ax2_twin.axhline(
    0,
    linestyle="--",
    linewidth=1.8,
    color="red"
)

ax2_twin.set_ylabel(
    "Mean Residual / Bias (Million USD)",
    fontsize=13
)

for bar, value in zip(
    bars_mae,
    risk_stats["MAE_m"]
):

    ax2.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

for bar, value in zip(
    bars_bias,
    risk_stats["bias_m"]
):

    offset = (
        0.02
        * max(
            1,
            np.max(
                np.abs(
                    risk_stats["bias_m"]
                )
            )
        )
    )

    ax2_twin.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + (
            offset if value >= 0
            else -offset
        ),
        f"{value:+.2f}",
        ha="center",
        va="bottom"
        if value >= 0
        else "top",
        fontsize=9,
        fontweight="bold"
    )

ax2.legend(
    [bars_mae, bars_bias],
    [
        "MAE",
        "Mean Residual / Bias"
    ],
    loc="upper left",
    fontsize=9
)

# A3 — ACTUAL VS PREDICTED BY RISK

x_pos = np.arange(
    len(risk_stats)
)

width = 0.36

actual_bars = ax3.bar(
    x_pos - width / 2,
    risk_stats["actual_mean_m"],
    width,
    label="Actual Mean"
)

pred_bars = ax3.bar(
    x_pos + width / 2,
    risk_stats["prediction_mean_m"],
    width,
    label="Predicted Mean"
)

ax3.set_xlabel(
    "Risk Segment",
    fontsize=13
)

ax3.set_ylabel(
    "Mean Price (Million USD)",
    fontsize=13
)

ax3.set_xticks(
    x_pos
)

ax3.set_xticklabels(
    risk_stats["risk_bin"].astype(str),
    rotation=0
)

ax3.set_title(
    "A3 — Actual vs Predicted by Risk Level",
    fontsize=17,
    fontweight="bold"
)

ax3.grid(
    axis="y",
    alpha=0.20
)

ax3.legend(
    fontsize=10
)

for bar, value in zip(
    actual_bars,
    risk_stats["actual_mean_m"]
):

    ax3.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

for bar, value in zip(
    pred_bars,
    risk_stats["prediction_mean_m"]
):

    ax3.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

# A4 — FITTED VALUES VS RESIDUALS + OUTLIER

fitted = (
    plot_df["prediction"].to_numpy()
    / 1e6
)

residual = (
    plot_df["residual"].to_numpy()
    / 1e6
)

x_low, x_high = np.percentile(
    fitted,
    [0.5, 99.5]
)

y_low, y_high = np.percentile(
    residual,
    [0.5, 99.5]
)

mask = (
    (fitted >= x_low)
    & (fitted <= x_high)
    & (residual >= y_low)
    & (residual <= y_high)
)

x_plot = fitted[mask]
y_plot = residual[mask]

ax4.scatter(
    x_plot,
    y_plot,
    s=14,
    alpha=0.16,
    rasterized=True
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Residual = 0"
)

# SMOOTH RESIDUAL TREND

if len(x_plot) >= 50:

    sort_idx = np.argsort(
        x_plot
    )

    x_sorted = x_plot[sort_idx]
    y_sorted = y_plot[sort_idx]

    smooth_window = max(
        50,
        min(
            500,
            len(x_sorted) // 20
        )
    )

    rolling_residual = (
        pd.Series(y_sorted)
        .rolling(
            smooth_window,
            center=True,
            min_periods=max(
                20,
                smooth_window // 4
            )
        )
        .median()
    )

    ax4.plot(
        x_sorted,
        rolling_residual,
        linewidth=3,
        label="Smoothed residual trend"
    )

# OUTLIER TESPİTİ

residual_median = np.median(
    residual
)

mad = np.median(
    np.abs(
        residual - residual_median
    )
)

if mad > 0:

    robust_z = (
        0.6745
        * (
            residual
            - residual_median
        )
        / mad
    )

    outlier_mask = (
        np.abs(robust_z) > 3.5
    )

else:

    outlier_mask = np.zeros(
        len(residual),
        dtype=bool
    )

outlier_x = fitted[outlier_mask]
outlier_y = residual[outlier_mask]

if len(outlier_x) > 0:

    ax4.scatter(
        outlier_x,
        outlier_y,
        s=65,
        facecolors="none",
        edgecolors="red",
        linewidths=1.8,
        label=f"Outliers: {len(outlier_x):,}"
    )

ax4.set_xlabel(
    "Predicted Price / Fitted Value (Million USD)",
    fontsize=13
)

ax4.set_ylabel(
    "Residual (Million USD)",
    fontsize=13
)

ax4.set_title(
    "A4 — Fitted Values vs Residuals + Outliers",
    fontsize=17,
    fontweight="bold"
)

ax4.grid(
    alpha=0.20,
    linestyle="--"
)

ax4.legend(
    fontsize=9,
    loc="best"
)

fig.suptitle(
    f"{FEATURE.upper()} — RISK-SPECIFIC MODEL ERROR DIAGNOSTICS",
    fontsize=21,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

plt.show()
plt.close()

# NON-RANDOM PATTERN İÇİN İSTATİSTİKSEL KONTROL

print("\n" + "=" * 90)
print(
    f"{FEATURE.upper()} — NON-RANDOM PATTERN ANALİZİ"
)
print("=" * 90)

if risk_type != "categorical":

    risk_values = analysis_df[
        FEATURE
    ].to_numpy()

    residual_values = analysis_df[
        "residual"
    ].to_numpy()

    spearman_rho, spearman_p = (
        stats.spearmanr(
            risk_values,
            residual_values
        )
    )

    print(
        f"\nSpearman korelasyonu : "
        f"{spearman_rho:+.4f}"
    )

    print(
        f"p-value              : "
        f"{spearman_p:.6g}"
    )

    if spearman_p < 0.05:

        if spearman_rho > 0:

            print(
                "Yorum: Risk arttıkça residual "
                "pozitif yönde değişiyor → "
                "modelin under-prediction eğilimi "
                "artıyor olabilir."
            )

        else:

            print(
                "Yorum: Risk arttıkça residual "
                "negatif yönde değişiyor → "
                "modelin over-prediction eğilimi "
                "artıyor olabilir."
            )

    else:

        print(
            "Yorum: Risk ile residual arasında "
            "istatistiksel olarak anlamlı "
            "monotonik ilişki bulunamadı."
        )

# RİSK SEGMENTLERİNİN KARŞILAŞTIRILMASI

print("\n" + "=" * 90)
print(
    f"{FEATURE.upper()} — RİSK SEGMENT SONUÇLARI"
)
print("=" * 90)

for _, row in risk_stats.iterrows():

    direction = (
        "UNDER-PREDICTION"
        if row["mean_residual"] > 0
        else "OVER-PREDICTION"
        if row["mean_residual"] < 0
        else "BALANCED"
    )

    print(
        f"\nRisk: {row['risk_bin']}"
    )

    print(
        f"  N              : "
        f"{int(row['count']):,}"
    )

    print(
        f"  Mean Residual  : "
        f"${row['mean_residual']:,.2f}"
    )

    print(
        f"  MAE            : "
        f"${row['MAE']:,.2f}"
    )

    print(
        f"  RMSE           : "
        f"${row['RMSE']:,.2f}"
    )

    print(
        f"  Actual Mean    : "
        f"${row['actual_mean']:,.2f}"
    )

    print(
        f"  Predicted Mean : "
        f"${row['prediction_mean']:,.2f}"
    )

    print(
        f"  Direction      : "
        f"{direction}"
    )

# EN PROBLEMLİ RİSK SEGMENTLERİ

if len(risk_stats) > 0:

    worst_mae = risk_stats.loc[
        risk_stats["MAE"].idxmax()
    ]

    worst_rmse = risk_stats.loc[
        risk_stats["RMSE"].idxmax()
    ]

    strongest_bias = risk_stats.loc[
        risk_stats["mean_residual"]
        .abs()
        .idxmax()
    ]

    print("\n" + "=" * 90)
    print(
        f"{FEATURE.upper()} — EN PROBLEMLİ RİSK SEGMENTLERİ"
    )
    print("=" * 90)

    print(
        f"\nEn yüksek MAE:"
        f" {worst_mae['risk_bin']} → "
        f"${worst_mae['MAE']:,.2f}"
    )

    print(
        f"En yüksek RMSE:"
        f" {worst_rmse['risk_bin']} → "
        f"${worst_rmse['RMSE']:,.2f}"
    )

    print(
        f"En güçlü bias:"
        f" {strongest_bias['risk_bin']} → "
        f"${strongest_bias['mean_residual']:,.2f}"
    )

    if strongest_bias["mean_residual"] > 0:

        print(
            "\nYorum: Bu risk segmentinde "
            "Actual > Prediction olduğundan "
            "model sistematik olarak "
            "UNDER-PREDICTION yapıyor."
        )

    elif strongest_bias["mean_residual"] < 0:

        print(
            "\nYorum: Bu risk segmentinde "
            "Actual < Prediction olduğundan "
            "model sistematik olarak "
            "OVER-PREDICTION yapıyor."
        )

# GENEL ÖZET

global_mean_residual = (
    analysis_df["residual"].mean()
)

global_mae = (
    analysis_df["absolute_error"].mean()
)

global_rmse = np.sqrt(
    np.mean(
        analysis_df["residual"] ** 2
    )
)

print("\n" + "=" * 90)
print(
    f"{FEATURE.upper()} — ANALİZ TAMAMLANDI"
)
print("=" * 90)

print(
    f"\nToplam gözlem       : "
    f"{len(analysis_df):,}"
)

print(
    f"Risk segmenti       : "
    f"{len(risk_stats):,}"
)

print(
    f"Global Mean Residual: "
    f"${global_mean_residual:,.2f}"
)

print(
    f"Global MAE          : "
    f"${global_mae:,.2f}"
)

print(
    f"Global RMSE         : "
    f"${global_rmse:,.2f}"
)

print(
    f"Outlier sayısı      : "
    f"{int(outlier_mask.sum()):,}"
)

## CORR_SUBM (İlan tarihi ile teklif teslim tarihi arasındaki sürenin kısıtlılığından kaynaklanan rekabet engeli ve ihale usulsüzlük riskini ölçen bir göstergedir.)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "corr_subm"
FEATURE_NAME = "Offer Duration Risk"

OUTPUT_DIR = "xgboost_final_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if hasattr(X_test, "to_list"):
    risk_values = np.asarray(
        X_test[FEATURE].to_list(),
        dtype=np.float64
    )
else:
    risk_values = np.asarray(
        X_test[FEATURE],
        dtype=np.float64
    )

analysis_df = pd.DataFrame({
    "risk": risk_values,
    "actual_usd": np.asarray(
        test_predictions["actual_usd"],
        dtype=np.float64
    ),
    "predicted_usd": np.asarray(
        test_predictions["predicted_usd"],
        dtype=np.float64
    ),
    "residual_usd": np.asarray(
        test_predictions["residual_usd"],
        dtype=np.float64
    ),
    "absolute_error_usd": np.asarray(
        test_predictions["absolute_error_usd"],
        dtype=np.float64
    )
})

analysis_df = (
    analysis_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

risk_levels = sorted(
    analysis_df["risk"].unique()
)

risk_labels = [
    f"{x:g}" for x in risk_levels
]

risk_summary = (
    analysis_df
    .groupby("risk")
    .agg(
        count=("risk", "size"),
        mean_residual_usd=("residual_usd", "mean"),
        median_residual_usd=("residual_usd", "median"),
        mean_absolute_error_usd=("absolute_error_usd", "mean"),
        median_absolute_error_usd=("absolute_error_usd", "median"),
        rmse_usd=(
            "residual_usd",
            lambda x: np.sqrt(np.mean(np.square(x)))
        ),
        mean_actual_usd=("actual_usd", "mean"),
        mean_predicted_usd=("predicted_usd", "mean")
    )
    .reindex(risk_levels)
)

risk_summary["bias_percent"] = (
    (
        risk_summary["mean_predicted_usd"]
        - risk_summary["mean_actual_usd"]
    )
    / risk_summary["mean_actual_usd"].replace(0, np.nan)
) * 100

risk_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "corr_subm_risk_analysis.csv"
    ),
    index=True
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(17, 12),
    dpi=300
)

ax1, ax2, ax3, ax4 = axes.flatten()

# A1 — RESIDUAL BOXPLOT
residual_groups = [
    analysis_df.loc[
        analysis_df["risk"] == level,
        "residual_usd"
    ].values / 1e6
    for level in risk_levels
]

bp1 = ax1.boxplot(
    residual_groups,
    labels=risk_labels,
    patch_artist=True,
    showfliers=False,
    widths=0.60,
    medianprops=dict(
        color="black",
        linewidth=2
    ),
    whiskerprops=dict(
        linewidth=1.2
    ),
    capprops=dict(
        linewidth=1.2
    ),
    boxprops=dict(
        linewidth=1.2
    )
)

for box in bp1["boxes"]:
    box.set_facecolor("#6baed6")
    box.set_alpha(0.75)

ax1.axhline(
    0,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Zero Residual"
)

# A1 için sadece merkezi residual bölgesini göster
ax1.set_ylim(-3, 3)

ax1.set_title(
    "A1 — Residual Distribution by Offer Duration Risk",
    fontsize=13,
    fontweight="bold"
)

ax1.set_xlabel(
    "Offer Duration Risk (corr_subm)",
    fontsize=11
)

ax1.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax1.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax1.legend(
    fontsize=9,
    loc="upper right"
)

# A1 medyan residual değerleri
for i, level in enumerate(risk_levels, 1):

    median_residual = (
        risk_summary.loc[
            level,
            "median_residual_usd"
        ] / 1e6
    )

    ax1.annotate(
        f"{median_residual:+.2f}M",
        xy=(i, median_residual),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=8.5,
        fontweight="bold"
    )

# A2 — ABSOLUTE ERROR BOXPLOT
absolute_error_groups = [
    analysis_df.loc[
        analysis_df["risk"] == level,
        "absolute_error_usd"
    ].values
    for level in risk_levels
]

bp2 = ax2.boxplot(
    absolute_error_groups,
    labels=risk_labels,
    patch_artist=True,
    showfliers=False,
    widths=0.60,
    medianprops=dict(
        color="black",
        linewidth=2
    ),
    whiskerprops=dict(
        linewidth=1.2
    ),
    capprops=dict(
        linewidth=1.2
    )
)

ax2.set_title(
    "A2 — Absolute Prediction Error by Risk",
    fontsize=13,
    fontweight="bold"
)

ax2.set_xlabel(
    "Offer Duration Risk (corr_subm)",
    fontsize=11
)

ax2.set_ylabel(
    "Absolute Error (USD)",
    fontsize=11
)

ax2.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

for i, level in enumerate(risk_levels, 1):

    median_value = risk_summary.loc[
        level,
        "median_absolute_error_usd"
    ]

    ax2.annotate(
        f"${median_value / 1e6:.2f}M",
        xy=(i, median_value),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        fontweight="bold"
    )

# A3 — MAE vs RMSE
x = np.arange(len(risk_levels))
width = 0.36

mae_values = (
    risk_summary["mean_absolute_error_usd"]
    .values
)

rmse_values = (
    risk_summary["rmse_usd"]
    .values
)

bars_mae = ax3.bar(
    x - width / 2,
    mae_values / 1e6,
    width,
    label="MAE"
)

bars_rmse = ax3.bar(
    x + width / 2,
    rmse_values / 1e6,
    width,
    label="RMSE"
)

ax3.set_title(
    "A3 — Model Error by Offer Duration Risk",
    fontsize=13,
    fontweight="bold"
)

ax3.set_xlabel(
    "Offer Duration Risk (corr_subm)",
    fontsize=11
)

ax3.set_ylabel(
    "Error (Million USD)",
    fontsize=11
)

ax3.set_xticks(x)
ax3.set_xticklabels(risk_labels)

ax3.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax3.legend(
    fontsize=9
)

for bars in [bars_mae, bars_rmse]:

    for bar in bars:

        height = bar.get_height()

        ax3.annotate(
            f"${height:.2f}M",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                height
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold"
        )

# A4 — ACTUAL vs PREDICTED MEAN
actual_mean = (
    risk_summary["mean_actual_usd"]
    .values / 1e6
)

predicted_mean = (
    risk_summary["mean_predicted_usd"]
    .values / 1e6
)

bars_actual = ax4.bar(
    x - width / 2,
    actual_mean,
    width,
    label="Actual Mean"
)

bars_predicted = ax4.bar(
    x + width / 2,
    predicted_mean,
    width,
    label="Predicted Mean"
)

ax4.set_title(
    "A4 — Actual vs Predicted Mean by Risk",
    fontsize=13,
    fontweight="bold"
)

ax4.set_xlabel(
    "Offer Duration Risk (corr_subm)",
    fontsize=11
)

ax4.set_ylabel(
    "Mean Tender Price (Million USD)",
    fontsize=11
)

ax4.set_xticks(x)
ax4.set_xticklabels(risk_labels)

ax4.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax4.legend(
    fontsize=9
)

for i, level in enumerate(risk_levels):

    bias = risk_summary.loc[
        level,
        "bias_percent"
    ]

    if np.isfinite(bias):

        y_position = max(
            actual_mean[i],
            predicted_mean[i]
        )

        ax4.annotate(
            f"Bias: {bias:+.1f}%",
            xy=(i, y_position),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            fontweight="bold",
            color="red"
        )

fig.suptitle(
    "XGBoost Final Model — Offer Duration Risk Error Analysis",
    fontsize=16,
    fontweight="bold",
    y=0.995
)

fig.text(
    0.5,
    0.965,
    "corr_subm: Teklif süresi kaynaklı risk göstergesi",
    ha="center",
    fontsize=10
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

risk_plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_subm_risk_error_analysis_2x2.png"
)

plt.savefig(
    risk_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n" + "=" * 75)
print("corr_subm — OFFER DURATION RISK ERROR ANALYSIS")
print("=" * 75)

print(
    f"{'Risk':<10}"
    f"{'N':>10}"
    f"{'MAE':>18}"
    f"{'RMSE':>18}"
    f"{'Mean Residual':>20}"
    f"{'Bias':>12}"
)

print("-" * 88)

for level in risk_levels:

    row = risk_summary.loc[level]

    print(
        f"{level:<10.1f}"
        f"{int(row['count']):>10,}"
        f"${row['mean_absolute_error_usd']:>16,.0f}"
        f"${row['rmse_usd']:>16,.0f}"
        f"${row['mean_residual_usd']:>18,.0f}"
        f"{row['bias_percent']:>10.2f}%"
    )

print("\nGrafik kaydedildi:")
print(risk_plot_path)

print("\nRisk analiz tablosu:")
print(
    os.path.join(
        OUTPUT_DIR,
        "corr_subm_risk_analysis.csv"
    )
)

## CORR_NOCFT (İhalenin önceden ilan edilmemesi veya doğrudan davet/çağrı usulüyle yapılması nedeniyle şeffaflık ve rekabetin kısıtlanmasından kaynaklanan risk düzeyini ölçen bir göstergedir.)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "corr_nocft"
FEATURE_NAME = "No-CFT Risk"
OUTPUT_DIR = "xgboost_final_test"

RANDOM_STATE = 42
PLOT_SAMPLE = 15000
OBS_SAMPLE = 12000
N_BINS = 40

os.makedirs(OUTPUT_DIR, exist_ok=True)

if hasattr(X_test[FEATURE], "to_list"):
    risk_values = np.asarray(
        X_test[FEATURE].to_list(),
        dtype=np.float64
    )
else:
    risk_values = np.asarray(
        X_test[FEATURE],
        dtype=np.float64
    )

analysis_df = pd.DataFrame({
    "risk": risk_values,
    "actual_usd": np.asarray(
        test_predictions["actual_usd"],
        dtype=np.float64
    ),
    "predicted_usd": np.asarray(
        test_predictions["predicted_usd"],
        dtype=np.float64
    ),
    "residual_usd": np.asarray(
        test_predictions["residual_usd"],
        dtype=np.float64
    ),
    "absolute_error_usd": np.asarray(
        test_predictions["absolute_error_usd"],
        dtype=np.float64
    )
})

analysis_df = (
    analysis_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

risk_levels = sorted(
    analysis_df["risk"].unique()
)

risk_labels = [
    f"{x:g}" for x in risk_levels
]

analysis_df["fitted_m"] = (
    analysis_df["predicted_usd"] / 1e6
)

analysis_df["residual_m"] = (
    analysis_df["residual_usd"] / 1e6
)

analysis_df["absolute_error_m"] = (
    analysis_df["absolute_error_usd"] / 1e6
)

residual_std = analysis_df[
    "residual_usd"
].std()

if residual_std > 0:
    analysis_df["standardized_residual"] = (
        analysis_df["residual_usd"] /
        residual_std
    )
else:
    analysis_df["standardized_residual"] = 0.0

analysis_df["scale_location"] = np.sqrt(
    np.abs(
        analysis_df["standardized_residual"]
    )
)

analysis_df["relative_error_percent"] = np.nan

valid_actual = (
    analysis_df["actual_usd"] > 0
)

analysis_df.loc[
    valid_actual,
    "relative_error_percent"
] = (
    analysis_df.loc[
        valid_actual,
        "residual_usd"
    ] /
    analysis_df.loc[
        valid_actual,
        "actual_usd"
    ]
) * 100

if len(analysis_df) > PLOT_SAMPLE:
    plot_df = analysis_df.sample(
        PLOT_SAMPLE,
        random_state=RANDOM_STATE
    )
else:
    plot_df = analysis_df.copy()


def get_bin_statistics(
    x,
    y,
    bins=N_BINS,
    min_count=20
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = (
        np.isfinite(x) &
        np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < min_count:
        return None

    x_min = np.min(x)
    x_max = np.max(x)

    if x_min == x_max:
        return None

    edges = np.linspace(
        x_min,
        x_max,
        bins + 1
    )

    centers = []
    medians = []
    q1_values = []
    q3_values = []
    counts = []

    for i in range(bins):

        if i == bins - 1:
            mask = (
                (x >= edges[i]) &
                (x <= edges[i + 1])
            )
        else:
            mask = (
                (x >= edges[i]) &
                (x < edges[i + 1])
            )

        if mask.sum() >= min_count:

            values = y[mask]

            centers.append(
                np.median(x[mask])
            )

            medians.append(
                np.median(values)
            )

            q1_values.append(
                np.percentile(values, 25)
            )

            q3_values.append(
                np.percentile(values, 75)
            )

            counts.append(
                mask.sum()
            )

    if len(centers) < 2:
        return None

    return (
        np.asarray(centers),
        np.asarray(medians),
        np.asarray(q1_values),
        np.asarray(q3_values),
        np.asarray(counts)
    )


def symmetric_ylim(
    values,
    low=1,
    high=99,
    minimum=1
):
    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return -minimum, minimum

    q_low, q_high = np.percentile(
        values,
        [low, high]
    )

    limit = max(
        abs(q_low),
        abs(q_high),
        minimum
    )

    return (
        -limit * 1.15,
        limit * 1.15
    )


fig, axes = plt.subplots(
    2,
    3,
    figsize=(20, 12),
    dpi=300
)

ax1, ax2, ax3, ax4, ax5, ax6 = (
    axes.flatten()
)

# A1 — RESIDUALS VS FITTED

a1_stats = get_bin_statistics(
    plot_df["fitted_m"],
    plot_df["residual_m"]
)

scatter_a1 = plot_df.sample(
    min(4000, len(plot_df)),
    random_state=RANDOM_STATE
)

ax1.scatter(
    scatter_a1["fitted_m"],
    scatter_a1["residual_m"],
    s=7,
    alpha=0.10,
    rasterized=True,
    label="Observations"
)

if a1_stats is not None:

    x, median, q1, q3, counts = a1_stats

    ax1.fill_between(
        x,
        q1,
        q3,
        alpha=0.18,
        label="25–75% residual range"
    )

    ax1.plot(
        x,
        median,
        linewidth=2.5,
        marker="o",
        markersize=3,
        label="Median residual"
    )

ax1.axhline(
    0,
    linestyle="--",
    linewidth=1.6,
    color="black",
    label="Zero residual"
)

ax1.set_ylim(
    *symmetric_ylim(
        plot_df["residual_m"],
        minimum=1
    )
)

ax1.set_title(
    "A1 — Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

ax1.set_xlabel(
    "Predicted Tender Price (Million USD)",
    fontsize=11
)

ax1.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax1.grid(
    alpha=0.20,
    linestyle="--"
)

ax1.legend(
    fontsize=8,
    loc="best"
)

# A2 — NORMAL Q-Q

qq_values = plot_df[
    "standardized_residual"
].values

qq_values = qq_values[
    np.isfinite(qq_values)
]

stats.probplot(
    qq_values,
    dist="norm",
    plot=ax2
)

for line in ax2.get_lines():
    line.set_markersize(2.5)

ax2.set_title(
    "A2 — Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

ax2.set_xlabel(
    "Theoretical Quantiles",
    fontsize=11
)

ax2.set_ylabel(
    "Standardized Residuals",
    fontsize=11
)

ax2.grid(
    alpha=0.20,
    linestyle="--"
)

# A3 — SCALE LOCATION

a3_stats = get_bin_statistics(
    plot_df["fitted_m"],
    plot_df["scale_location"]
)

scatter_a3 = plot_df.sample(
    min(4000, len(plot_df)),
    random_state=RANDOM_STATE
)

ax3.scatter(
    scatter_a3["fitted_m"],
    scatter_a3["scale_location"],
    s=7,
    alpha=0.10,
    rasterized=True,
    label="Observations"
)

if a3_stats is not None:

    x, median, q1, q3, counts = a3_stats

    ax3.fill_between(
        x,
        q1,
        q3,
        alpha=0.18,
        label="25–75% spread"
    )

    ax3.plot(
        x,
        median,
        linewidth=2.5,
        marker="o",
        markersize=3,
        label="Median spread"
    )

ax3.set_title(
    "A3 — Residual Spread vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

ax3.set_xlabel(
    "Predicted Tender Price (Million USD)",
    fontsize=11
)

ax3.set_ylabel(
    "√|Standardized Residual|",
    fontsize=11
)

ax3.grid(
    alpha=0.20,
    linestyle="--"
)

ax3.legend(
    fontsize=8,
    loc="best"
)

# A4 — RESIDUALS VS RISK

risk_map = {
    level: i
    for i, level in enumerate(risk_levels)
}

risk_colors = {
    0: "tab:blue",
    1: "tab:red"
}

rng = np.random.default_rng(
    RANDOM_STATE
)

risk_numeric = (
    plot_df["risk"]
    .map(risk_map)
    .astype(float)
)

x_jitter = (
    risk_numeric.values +
    rng.normal(
        0,
        0.055,
        len(plot_df)
    )
)

for level in risk_levels:

    mask = (
        plot_df["risk"] == level
    )

    ax4.scatter(
        x_jitter[mask.values],
        plot_df.loc[
            mask,
            "residual_m"
        ],
        s=8,
        alpha=0.18,
        color=risk_colors.get(
            level,
            "gray"
        ),
        rasterized=True
    )

for i, level in enumerate(risk_levels):

    group = analysis_df.loc[
        analysis_df["risk"] == level,
        "residual_m"
    ].dropna()

    if len(group) == 0:
        continue

    q1, median, q3 = np.percentile(
        group,
        [25, 50, 75]
    )

    mae = np.mean(
        np.abs(group)
    )

    ax4.vlines(
        i,
        q1,
        q3,
        linewidth=8,
        alpha=0.55,
        color=risk_colors.get(
            level,
            "gray"
        )
    )

    ax4.scatter(
        i,
        median,
        s=70,
        edgecolors="black",
        linewidths=1.3,
        color=risk_colors.get(
            level,
            "gray"
        ),
        zorder=5
    )

    ax4.annotate(
        f"Bias: {median:+.2f}M\nMAE: {mae:.2f}M",
        xy=(i, median),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        fontweight="bold"
    )

ax4.axhline(
    0,
    linestyle="--",
    linewidth=1.6,
    color="black",
    label="Zero residual"
)

ax4.set_ylim(
    *symmetric_ylim(
        plot_df["residual_m"],
        minimum=1
    )
)

ax4.set_title(
    f"A4 — Residual Error by {FEATURE_NAME}",
    fontsize=14,
    fontweight="bold"
)

ax4.set_xlabel(
    f"{FEATURE_NAME} ({FEATURE})",
    fontsize=11
)

ax4.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax4.set_xticks(
    np.arange(len(risk_levels))
)

ax4.set_xticklabels(
    risk_labels
)

ax4.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax4.legend(
    fontsize=8
)

# A5 — ABSOLUTE ERROR DISTRIBUTION

absolute_values = analysis_df[
    "absolute_error_m"
].dropna().values

if len(absolute_values) > 0:

    p95 = np.percentile(
        absolute_values,
        95
    )

    p99 = np.percentile(
        absolute_values,
        99
    )

    upper_limit = max(
        p99,
        1
    )

    clipped = absolute_values[
        absolute_values <= upper_limit
    ]

    counts, edges = np.histogram(
        clipped,
        bins=60,
        range=(0, upper_limit)
    )

    centers = (
        edges[:-1] +
        np.diff(edges) / 2
    )

    widths = np.diff(edges)

    ax5.bar(
        centers,
        counts,
        width=widths * 0.90,
        alpha=0.72,
        align="center",
        edgecolor="none",
        label="Absolute error"
    )

    median_error = np.median(
        absolute_values
    )

    mean_error = np.mean(
        absolute_values
    )

    if median_error <= upper_limit:

        ax5.axvline(
            median_error,
            linewidth=2.2,
            linestyle="--",
            color="black",
            label=(
                f"Median: "
                f"{median_error:.2f}M"
            )
        )

    if mean_error <= upper_limit:

        ax5.axvline(
            mean_error,
            linewidth=2.2,
            linestyle=":",
            color="black",
            label=(
                f"Mean: "
                f"{mean_error:.2f}M"
            )
        )

    ax5.axvline(
        p99,
        linewidth=1.8,
        linestyle="-.",
        color="gray",
        label=(
            f"P99: "
            f"{p99:.2f}M"
        )
    )

    n_outliers = np.sum(
        absolute_values > upper_limit
    )

    outlier_pct = (
        n_outliers /
        len(absolute_values)
    ) * 100

    # Bilgi kutusu kaldırıldı.
    # P99 ve outlier bilgisi başlığın altına taşındı.
    ax5.text(
        0.5,
        1.01,
        f"View: 0–{upper_limit:.2f}M | "
        f"P99+: {n_outliers:,} observations "
        f"({outlier_pct:.2f}%)",
        transform=ax5.transAxes,
        ha="center",
        va="bottom",
        fontsize=8,
        color="dimgray"
    )

    ax5.axvline(
        p95,
        linewidth=1.2,
        linestyle="--",
        color="gray",
        alpha=0.65,
        label=(
            f"P95: "
            f"{p95:.2f}M"
        )
    )

    ax5.set_xlim(
        0,
        upper_limit
    )

    ymax = counts.max()

    if ymax > 0:
        ax5.set_ylim(
            0,
            ymax * 1.12
        )

ax5.set_title(
    "A5 — Absolute Prediction Error Distribution",
    fontsize=14,
    fontweight="bold",
    pad=18
)

ax5.set_xlabel(
    "Absolute Error (Million USD)",
    fontsize=11
)

ax5.set_ylabel(
    "Number of Tenders",
    fontsize=11
)

ax5.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax5.legend(
    fontsize=7.5,
    loc="upper right"
)

# A6 — RELATIVE RESIDUALS VS RISK

relative_df = plot_df[
    plot_df["relative_error_percent"].notna()
].copy()

if len(relative_df) > 0:

    relative_numeric = (
        relative_df["risk"]
        .map(risk_map)
        .astype(float)
    )

    relative_jitter = (
        relative_numeric.values +
        rng.normal(
            0,
            0.055,
            len(relative_df)
        )
    )

    for level in risk_levels:

        mask = (
            relative_df["risk"] == level
        )

        ax6.scatter(
            relative_jitter[mask.values],
            relative_df.loc[
                mask,
                "relative_error_percent"
            ],
            s=8,
            alpha=0.18,
            color=risk_colors.get(
                level,
                "gray"
            ),
            rasterized=True
        )

    for i, level in enumerate(risk_levels):

        values = analysis_df.loc[
            (
                analysis_df["risk"] == level
            ) &
            (
                analysis_df[
                    "relative_error_percent"
                ].notna()
            ),
            "relative_error_percent"
        ].values

        if len(values) == 0:
            continue

        q1, median, q3 = np.percentile(
            values,
            [25, 50, 75]
        )

        mae_relative = np.mean(
            np.abs(values)
        )

        ax6.vlines(
            i,
            q1,
            q3,
            linewidth=8,
            alpha=0.55,
            color=risk_colors.get(
                level,
                "gray"
            )
        )

        ax6.scatter(
            i,
            median,
            s=70,
            color=risk_colors.get(
                level,
                "gray"
            ),
            edgecolors="black",
            linewidths=1.3,
            zorder=5
        )

        ax6.annotate(
            f"Bias: {median:+.1f}%\n"
            f"MAE: {mae_relative:.1f}%",
            xy=(i, median),
            xytext=(0, 12),
            textcoords="offset points",
            ha="center",
            fontsize=8,
            fontweight="bold"
        )

    ax6.axhline(
        0,
        linestyle="--",
        linewidth=1.6,
        color="black",
        label="Zero relative error"
    )

    ax6.axhline(
        30,
        linestyle=":",
        linewidth=1.2,
        color="gray",
        label="+30%"
    )

    ax6.axhline(
        -30,
        linestyle=":",
        linewidth=1.2,
        color="gray",
        label="-30%"
    )

    rel_values = relative_df[
        "relative_error_percent"
    ].values

    rel_low, rel_high = np.percentile(
        rel_values,
        [1, 99]
    )

    rel_low = max(
        rel_low,
        -150
    )

    rel_high = min(
        rel_high,
        150
    )

    if rel_low >= rel_high:
        rel_low = -100
        rel_high = 100

    ax6.set_ylim(
        rel_low * 1.10,
        rel_high * 1.10
    )

ax6.set_title(
    f"A6 — Relative Residual Error by {FEATURE_NAME}",
    fontsize=14,
    fontweight="bold"
)

ax6.set_xlabel(
    f"{FEATURE_NAME} ({FEATURE})",
    fontsize=11
)

ax6.set_ylabel(
    "Relative Residual (%)\n"
    "(Actual − Predicted) / Actual",
    fontsize=11
)

ax6.set_xticks(
    np.arange(len(risk_levels))
)

ax6.set_xticklabels(
    risk_labels
)

ax6.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax6.legend(
    fontsize=8
)

fig.suptitle(
    "XGBoost Final Model — Residual Diagnostics for No-CFT Risk",
    fontsize=18,
    fontweight="bold",
    y=0.995
)

fig.text(
    0.5,
    0.965,
    "Residual = Actual − Predicted | "
    "Positive = underprediction | "
    "Negative = overprediction",
    ha="center",
    fontsize=10
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95],
    h_pad=3.0,
    w_pad=2.5
)

plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_residual_diagnostics.png"
)

plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# OBSERVATION ORDER — CLEAR VERSION

order_df = analysis_df.copy().reset_index(
    drop=True
)

order_df["observation_order"] = np.arange(
    len(order_df)
)

if len(order_df) > OBS_SAMPLE:

    order_plot = order_df.sample(
        OBS_SAMPLE,
        random_state=RANDOM_STATE
    ).sort_values(
        "observation_order"
    )

else:

    order_plot = order_df.copy()

order_fig, order_ax = plt.subplots(
    figsize=(18, 8),
    dpi=300
)

for level in sorted(
    order_plot["risk"].unique()
):

    subset = order_plot[
        order_plot["risk"] == level
    ]

    order_ax.scatter(
        subset["observation_order"],
        subset["residual_m"],
        s=11,
        alpha=0.30,
        color=risk_colors.get(
            level,
            "gray"
        ),
        label=f"Risk {level:g}",
        rasterized=True
    )

order_ax.axhline(
    0,
    linestyle="--",
    linewidth=2,
    color="black",
    label="Zero residual"
)

risk_means = {}

for level in sorted(
    order_df["risk"].unique()
):

    values = order_df.loc[
        order_df["risk"] == level,
        "residual_m"
    ].dropna()

    if len(values) == 0:
        continue

    mean_value = values.mean()

    risk_means[level] = mean_value

    order_ax.axhline(
        mean_value,
        linestyle=":",
        linewidth=1.8,
        color=risk_colors.get(
            level,
            "gray"
        ),
        alpha=0.85,
        label=(
            f"Risk {level:g} mean: "
            f"{mean_value:+.2f}M"
        )
    )

residual_values = order_df[
    "residual_m"
].dropna().values

y_low, y_high = np.percentile(
    residual_values,
    [1, 99]
)

y_limit = max(
    abs(y_low),
    abs(y_high),
    5
) * 1.15

positive_df = order_df[
    order_df["residual_m"] > 0
]

negative_df = order_df[
    order_df["residual_m"] < 0
]

if len(positive_df) > 0:

    target = positive_df.loc[
        positive_df["residual_m"].idxmax()
    ]

    target_x = target[
        "observation_order"
    ]

    target_y = target[
        "residual_m"
    ]

    order_ax.annotate(
        "UNDERPREDICTION\n"
        "Actual > Predicted\n"
        "Positive residual",
        xy=(target_x, target_y),
        xytext=(
            len(order_df) * 0.10,
            y_limit * 0.60
        ),
        fontsize=10,
        fontweight="bold",
        ha="left",
        va="center",
        arrowprops=dict(
            arrowstyle="->",
            linewidth=2.5,
            shrinkA=5,
            shrinkB=5
        ),
        bbox=dict(
            boxstyle="round,pad=0.45",
            alpha=0.88
        )
    )

if len(negative_df) > 0:

    target = negative_df.loc[
        negative_df["residual_m"].idxmin()
    ]

    target_x = target[
        "observation_order"
    ]

    target_y = target[
        "residual_m"
    ]

    order_ax.annotate(
        "OVERPREDICTION\n"
        "Actual < Predicted\n"
        "Negative residual",
        xy=(target_x, target_y),
        xytext=(
            len(order_df) * 0.67,
            -y_limit * 0.60
        ),
        fontsize=10,
        fontweight="bold",
        ha="left",
        va="center",
        arrowprops=dict(
            arrowstyle="->",
            linewidth=2.5,
            shrinkA=5,
            shrinkB=5
        ),
        bbox=dict(
            boxstyle="round,pad=0.45",
            alpha=0.88
        )
    )

order_ax.set_ylim(
    -y_limit,
    y_limit
)

order_ax.set_title(
    "Residuals Versus Observation Order — No-CFT Risk",
    fontsize=16,
    fontweight="bold"
)

order_ax.set_xlabel(
    "Observation Order",
    fontsize=11
)

order_ax.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

order_ax.grid(
    alpha=0.20,
    linestyle="--"
)

order_ax.legend(
    fontsize=9,
    loc="upper right"
)

order_ax.text(
    0.5,
    0.02,
    "Positive residual = underprediction   |   "
    "Negative residual = overprediction",
    transform=order_ax.transAxes,
    ha="center",
    fontsize=9,
    bbox=dict(
        boxstyle="round,pad=0.3",
        alpha=0.75
    )
)

order_fig.suptitle(
    "XGBoost Final Model — Observation Order Residual Analysis",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.94]
)

order_plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_residuals_vs_observation_order.png"
)

plt.savefig(
    order_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# RİSK BAZLI ÖZET

summary_rows = []

for level in risk_levels:

    group = analysis_df[
        analysis_df["risk"] == level
    ]

    residuals = group[
        "residual_usd"
    ].dropna()

    relative = group[
        "relative_error_percent"
    ].dropna()

    summary_rows.append({
        "risk": level,
        "n": len(group),
        "mean_residual_usd": residuals.mean(),
        "median_residual_usd": residuals.median(),
        "std_residual_usd": residuals.std(),
        "mae_usd": group[
            "absolute_error_usd"
        ].mean(),
        "rmse_usd": np.sqrt(
            np.mean(
                residuals.values ** 2
            )
        ),
        "mean_relative_error_pct": (
            relative.mean()
            if len(relative) > 0
            else np.nan
        ),
        "median_relative_error_pct": (
            relative.median()
            if len(relative) > 0
            else np.nan
        ),
        "mae_relative_error_pct": (
            relative.abs().mean()
            if len(relative) > 0
            else np.nan
        ),
        "error_over_30pct": (
            (relative.abs() > 30).mean() * 100
            if len(relative) > 0
            else np.nan
        ),
        "error_over_50pct": (
            (relative.abs() > 50).mean() * 100
            if len(relative) > 0
            else np.nan
        )
    })

risk_summary = pd.DataFrame(
    summary_rows
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_residual_summary.csv"
)

risk_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 90)
print(
    "corr_nocft — RESIDUAL DIAGNOSTIC ANALYSIS"
)
print("=" * 90)

for _, row in risk_summary.iterrows():

    print(
        f"\nRisk {row['risk']:g}"
        f"\n  Gözlem sayısı            : {row['n']:,}"
        f"\n  Mean residual            : ${row['mean_residual_usd']:,.0f}"
        f"\n  Median residual          : ${row['median_residual_usd']:,.0f}"
        f"\n  Residual std             : ${row['std_residual_usd']:,.0f}"
        f"\n  MAE                      : ${row['mae_usd']:,.0f}"
        f"\n  RMSE                     : ${row['rmse_usd']:,.0f}"
        f"\n  Mean relative residual   : {row['mean_relative_error_pct']:+.2f}%"
        f"\n  Median relative residual : {row['median_relative_error_pct']:+.2f}%"
        f"\n  MAE relative error       : {row['mae_relative_error_pct']:.2f}%"
        f"\n  Error > ±30%             : {row['error_over_30pct']:.2f}%"
        f"\n  Error > ±50%             : {row['error_over_50pct']:.2f}%"
    )

print("\n6'lı residual paneli:")
print(plot_path)

print("\nObservation Order grafiği:")
print(order_plot_path)

print("\nRisk bazlı residual özeti:")
print(summary_path)

print("\n" + "=" * 90)
print(
    "corr_nocft RESIDUAL DIAGNOSTIC ANALİZİ TAMAMLANDI"
)
print("=" * 90)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "corr_nocft"
FEATURE_NAME = "No-CFT Risk"

OUTPUT_DIR = "xgboost_final_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test sonuçlarını hazırla
if hasattr(X_test, "to_list"):
    risk_values = np.asarray(
        X_test[FEATURE].to_list(),
        dtype=np.float64
    )
else:
    risk_values = np.asarray(
        X_test[FEATURE],
        dtype=np.float64
    )

analysis_df = pd.DataFrame({
    "risk": risk_values,
    "actual_usd": np.asarray(
        test_predictions["actual_usd"],
        dtype=np.float64
    ),
    "predicted_usd": np.asarray(
        test_predictions["predicted_usd"],
        dtype=np.float64
    ),
    "residual_usd": np.asarray(
        test_predictions["residual_usd"],
        dtype=np.float64
    ),
    "absolute_error_usd": np.asarray(
        test_predictions["absolute_error_usd"],
        dtype=np.float64
    )
})

analysis_df = (
    analysis_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# Risk seviyelerini belirle
risk_levels = sorted(
    analysis_df["risk"].unique()
)

risk_labels = [
    f"{x:g}" for x in risk_levels
]

# Gerçek değer 0 olmayan gözlemler için yüzde hata
analysis_df["absolute_percentage_error"] = np.nan

valid_actual = analysis_df["actual_usd"] > 0

analysis_df.loc[
    valid_actual,
    "absolute_percentage_error"
] = (
    analysis_df.loc[valid_actual, "absolute_error_usd"]
    / analysis_df.loc[valid_actual, "actual_usd"]
) * 100

# Büyük hata göstergeleri
analysis_df["error_over_30pct"] = (
    analysis_df["absolute_percentage_error"] > 30
)

analysis_df["error_over_50pct"] = (
    analysis_df["absolute_percentage_error"] > 50
)

# Risk bazlı özet
risk_summary = (
    analysis_df
    .groupby("risk")
    .agg(
        count=("risk", "size"),

        mean_actual_usd=(
            "actual_usd",
            "mean"
        ),

        mean_predicted_usd=(
            "predicted_usd",
            "mean"
        ),

        mean_residual_usd=(
            "residual_usd",
            "mean"
        ),

        mae_usd=(
            "absolute_error_usd",
            "mean"
        ),

        rmse_usd=(
            "residual_usd",
            lambda x: np.sqrt(
                np.mean(np.square(x))
            )
        ),

        median_absolute_error_usd=(
            "absolute_error_usd",
            "median"
        )
    )
    .reindex(risk_levels)
)

# Bias USD
risk_summary["bias_usd"] = (
    risk_summary["mean_predicted_usd"]
    - risk_summary["mean_actual_usd"]
)

# Bias %
risk_summary["bias_percent"] = (
    risk_summary["bias_usd"]
    / risk_summary["mean_actual_usd"].replace(
        0,
        np.nan
    )
) * 100

# MdAPE
mdape_values = []

# >30% hata oranı
large_error_30_values = []

# >50% hata oranı
large_error_50_values = []

for level in risk_levels:

    group = analysis_df[
        analysis_df["risk"] == level
    ]

    ape = group[
        "absolute_percentage_error"
    ].dropna()

    if len(ape) > 0:

        mdape = np.median(ape)

        error_30 = (
            group["error_over_30pct"]
            .mean()
            * 100
        )

        error_50 = (
            group["error_over_50pct"]
            .mean()
            * 100
        )

    else:

        mdape = np.nan
        error_30 = np.nan
        error_50 = np.nan

    mdape_values.append(mdape)
    large_error_30_values.append(error_30)
    large_error_50_values.append(error_50)

risk_summary["mdape_percent"] = mdape_values
risk_summary["error_over_30pct"] = large_error_30_values
risk_summary["error_over_50pct"] = large_error_50_values

# MAE / Actual
risk_summary["mae_percent_of_actual"] = (
    risk_summary["mae_usd"]
    / risk_summary["mean_actual_usd"].replace(
        0,
        np.nan
    )
) * 100

# CSV
summary_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_risk_analysis.csv"
)

risk_summary.to_csv(
    summary_path,
    encoding="utf-8-sig"
)

# 3 PANELLİ GRAFİK
fig, axes = plt.subplots(
    1,
    3,
    figsize=(21, 7),
    dpi=300
)

ax1, ax2, ax3 = axes

x = np.arange(
    len(risk_levels)
)

width = 0.34

# ============================================================
# A1 — ACTUAL VS PREDICTED + BIAS
# ============================================================

actual_mean = (
    risk_summary[
        "mean_actual_usd"
    ].values / 1e6
)

predicted_mean = (
    risk_summary[
        "mean_predicted_usd"
    ].values / 1e6
)

bias_usd = (
    risk_summary[
        "bias_usd"
    ].values / 1e6
)

bias_percent = (
    risk_summary[
        "bias_percent"
    ].values
)

bars_actual = ax1.bar(
    x - width / 2,
    actual_mean,
    width,
    label="Actual Mean"
)

bars_predicted = ax1.bar(
    x + width / 2,
    predicted_mean,
    width,
    label="Predicted Mean"
)

ax1.set_title(
    "A1 — Prediction Bias by No-CFT Risk",
    fontsize=14,
    fontweight="bold"
)

ax1.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax1.set_ylabel(
    "Mean Tender Price (Million USD)",
    fontsize=11
)

ax1.set_xticks(x)
ax1.set_xticklabels(risk_labels)

ax1.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax1.legend(
    fontsize=9,
    loc="upper right"
)

# Actual / predicted değerleri
for bars in [
    bars_actual,
    bars_predicted
]:

    for bar in bars:

        height = bar.get_height()

        ax1.annotate(
            f"${height:.2f}M",
            xy=(
                bar.get_x()
                + bar.get_width() / 2,
                height
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

# Bias bilgisi
for i in range(len(risk_levels)):

    top_value = max(
        actual_mean[i],
        predicted_mean[i]
    )

    ax1.annotate(
        f"Bias: {bias_usd[i]:+.2f}M USD\n"
        f"({bias_percent[i]:+.1f}%)",
        xy=(
            x[i],
            top_value
        ),
        xytext=(0, 28),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

# ============================================================
# A2 — MAE vs RMSE
# ============================================================

mae_values = (
    risk_summary[
        "mae_usd"
    ].values / 1e6
)

rmse_values = (
    risk_summary[
        "rmse_usd"
    ].values / 1e6
)

bars_mae = ax2.bar(
    x - width / 2,
    mae_values,
    width,
    label="MAE"
)

bars_rmse = ax2.bar(
    x + width / 2,
    rmse_values,
    width,
    label="RMSE"
)

ax2.set_title(
    "A2 — Error Magnitude & Outlier Effect",
    fontsize=14,
    fontweight="bold"
)

ax2.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax2.set_ylabel(
    "Error (Million USD)",
    fontsize=11
)

ax2.set_xticks(x)
ax2.set_xticklabels(risk_labels)

ax2.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax2.legend(
    fontsize=9,
    loc="upper left"
)

for bars in [
    bars_mae,
    bars_rmse
]:

    for bar in bars:

        height = bar.get_height()

        ax2.annotate(
            f"${height:.2f}M",
            xy=(
                bar.get_x()
                + bar.get_width() / 2,
                height
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

# ============================================================
# A3 — LARGE ERROR RATE + MdAPE
# ============================================================

error_30 = (
    risk_summary[
        "error_over_30pct"
    ].values
)

error_50 = (
    risk_summary[
        "error_over_50pct"
    ].values
)

mdape = (
    risk_summary[
        "mdape_percent"
    ].values
)

bar_30 = ax3.bar(
    x - width / 2,
    error_30,
    width,
    label="Error > ±30%"
)

bar_50 = ax3.bar(
    x + width / 2,
    error_50,
    width,
    label="Error > ±50%"
)

ax3.set_title(
    "A3 — Large Prediction Error Rate",
    fontsize=14,
    fontweight="bold"
)

ax3.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax3.set_ylabel(
    "Share of Tenders (%)",
    fontsize=11
)

ax3.set_xticks(x)
ax3.set_xticklabels(risk_labels)

ax3.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax3.legend(
    fontsize=9,
    loc="upper right"
)

# Y eksenini yüzde olarak göster
max_error_rate = np.nanmax(
    np.concatenate([
        error_30,
        error_50
    ])
)

ax3.set_ylim(
    0,
    max(
        10,
        max_error_rate * 1.25
    )
)

# Büyük hata oranı değerleri
for bars in [
    bar_30,
    bar_50
]:

    for bar in bars:

        height = bar.get_height()

        ax3.annotate(
            f"{height:.1f}%",
            xy=(
                bar.get_x()
                + bar.get_width() / 2,
                height
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

# MdAPE bilgisini panelin alt kısmında göster
for i in range(len(risk_levels)):

    if np.isfinite(mdape[i]):

        ax3.annotate(
            f"MdAPE: {mdape[i]:.1f}%",
            xy=(x[i], 0),
            xytext=(0, -35),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold"
        )

# ============================================================
# GENEL BAŞLIK
# ============================================================

fig.suptitle(
    "XGBoost Final Model — No-CFT Risk Error Analysis",
    fontsize=17,
    fontweight="bold",
    y=1.03
)

fig.text(
    0.5,
    0.985,
    "corr_nocft: İlan yapılmaması / çağrı süreci risk göstergesi",
    ha="center",
    fontsize=10
)

plt.tight_layout(
    rect=[0, 0, 1, 0.94]
)

# Grafik kaydet
risk_plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_risk_error_analysis_3panel.png"
)

plt.savefig(
    risk_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# RİSK BAZLI SONUÇLAR
# ============================================================

print("\n" + "=" * 115)
print(
    "corr_nocft — NO-CFT RISK ERROR ANALYSIS"
)
print("=" * 115)

print(
    f"{'Risk':<8}"
    f"{'N':>10}"
    f"{'Actual Mean':>18}"
    f"{'Predicted Mean':>20}"
    f"{'Bias USD':>18}"
    f"{'Bias %':>12}"
    f"{'MAE':>18}"
    f"{'RMSE':>18}"
    f"{'>30%':>10}"
    f"{'>50%':>10}"
    f"{'MdAPE':>10}"
)

print("-" * 135)

for level in risk_levels:

    row = risk_summary.loc[level]

    print(
        f"{level:<8.1f}"
        f"{int(row['count']):>10,}"
        f"${row['mean_actual_usd']:>16,.0f}"
        f"${row['mean_predicted_usd']:>18,.0f}"
        f"${row['bias_usd']:>16,.0f}"
        f"{row['bias_percent']:>10.2f}%"
        f"${row['mae_usd']:>16,.0f}"
        f"${row['rmse_usd']:>16,.0f}"
        f"{row['error_over_30pct']:>9.2f}%"
        f"{row['error_over_50pct']:>9.2f}%"
        f"{row['mdape_percent']:>9.2f}%"
    )

print("\nRisk bazlı detaylı yorum değerleri:")

for level in risk_levels:

    row = risk_summary.loc[level]

    print(
        f"\nRisk {level:g}:"
        f"\n  Gözlem sayısı       : {int(row['count']):,}"
        f"\n  Gerçek ortalama     : ${row['mean_actual_usd']:,.0f}"
        f"\n  Tahmin ortalaması   : ${row['mean_predicted_usd']:,.0f}"
        f"\n  Bias                : ${row['bias_usd']:,.0f}"
        f"\n  Bias %              : {row['bias_percent']:+.2f}%"
        f"\n  MAE                 : ${row['mae_usd']:,.0f}"
        f"\n  RMSE                : ${row['rmse_usd']:,.0f}"
        f"\n  MAE / Actual        : {row['mae_percent_of_actual']:.2f}%"
        f"\n  MdAPE               : {row['mdape_percent']:.2f}%"
        f"\n  Hata > ±30%         : {row['error_over_30pct']:.2f}%"
        f"\n  Hata > ±50%         : {row['error_over_50pct']:.2f}%"
    )

print("\nKaydedilen grafik:")
print(risk_plot_path)

print("\nKaydedilen risk analiz tablosu:")
print(summary_path)

print("\n" + "=" * 115)
print("corr_nocft RİSK ANALİZİ TAMAMLANDI")
print("=" * 115)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "corr_nocft"
FEATURE_NAME = "No-CFT Risk"
OUTPUT_DIR = "xgboost_final_test"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test verisindeki risk değerlerini al
if hasattr(X_test, "to_list"):
    risk_values = np.asarray(
        X_test[FEATURE].to_list(),
        dtype=np.float64
    )
else:
    risk_values = np.asarray(
        X_test[FEATURE],
        dtype=np.float64
    )

# Residual sonuçlarını hazırla
analysis_df = pd.DataFrame({
    "risk": risk_values,
    "actual_usd": np.asarray(
        test_predictions["actual_usd"],
        dtype=np.float64
    ),
    "predicted_usd": np.asarray(
        test_predictions["predicted_usd"],
        dtype=np.float64
    ),
    "residual_usd": np.asarray(
        test_predictions["residual_usd"],
        dtype=np.float64
    ),
    "absolute_error_usd": np.asarray(
        test_predictions["absolute_error_usd"],
        dtype=np.float64
    )
})

analysis_df = (
    analysis_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# Relative error
analysis_df["relative_error_percent"] = np.nan

valid_actual = analysis_df["actual_usd"] > 0

analysis_df.loc[
    valid_actual,
    "relative_error_percent"
] = (
    analysis_df.loc[
        valid_actual,
        "residual_usd"
    ]
    / analysis_df.loc[
        valid_actual,
        "actual_usd"
    ]
) * 100

analysis_df["absolute_relative_error_percent"] = (
    analysis_df["relative_error_percent"].abs()
)

# Büyük hata göstergeleri
analysis_df["error_over_30pct"] = (
    analysis_df["absolute_relative_error_percent"] > 30
)

analysis_df["error_over_50pct"] = (
    analysis_df["absolute_relative_error_percent"] > 50
)

risk_levels = sorted(
    analysis_df["risk"].unique()
)

risk_labels = [
    f"{x:g}"
    for x in risk_levels
]

# Risk bazlı istatistikler
risk_summary = (
    analysis_df
    .groupby("risk")
    .agg(
        count=("risk", "size"),
        mean_residual_usd=(
            "residual_usd",
            "mean"
        ),
        median_residual_usd=(
            "residual_usd",
            "median"
        ),
        std_residual_usd=(
            "residual_usd",
            "std"
        ),
        mean_absolute_error_usd=(
            "absolute_error_usd",
            "mean"
        ),
        median_absolute_error_usd=(
            "absolute_error_usd",
            "median"
        ),
        rmse_usd=(
            "residual_usd",
            lambda x: np.sqrt(
                np.mean(np.square(x))
            )
        )
    )
    .reindex(risk_levels)
)

# Relative error istatistikleri
mdape_values = []
error_30_values = []
error_50_values = []

for level in risk_levels:

    group = analysis_df[
        analysis_df["risk"] == level
    ]

    relative_errors = (
        group["absolute_relative_error_percent"]
        .dropna()
    )

    if len(relative_errors) > 0:

        mdape_values.append(
            np.median(relative_errors)
        )

        error_30_values.append(
            (
                relative_errors > 30
            ).mean() * 100
        )

        error_50_values.append(
            (
                relative_errors > 50
            ).mean() * 100
        )

    else:

        mdape_values.append(np.nan)
        error_30_values.append(np.nan)
        error_50_values.append(np.nan)

risk_summary["MdAPE_percent"] = mdape_values
risk_summary["error_over_30pct"] = error_30_values
risk_summary["error_over_50pct"] = error_50_values

# CSV olarak kaydet
summary_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_residual_analysis.csv"
)

risk_summary.to_csv(
    summary_path,
    encoding="utf-8-sig"
)

# ============================================================
# RESIDUAL DIAGNOSTIC — 2x2
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(18, 13),
    dpi=300
)

ax1, ax2, ax3, ax4 = axes.flatten()

x = np.arange(
    len(risk_levels)
)

# ============================================================
# A1 — RESIDUAL vs RISK
# ============================================================

# Jitter ile noktaların üst üste binmesini azalt
rng = np.random.default_rng(42)

jitter_strength = 0.055

risk_numeric = (
    analysis_df["risk"].map(
        {
            level: i
            for i, level in enumerate(risk_levels)
        }
    ).astype(float)
)

x_jitter = (
    risk_numeric.values
    + rng.normal(
        0,
        jitter_strength,
        len(analysis_df)
    )
)

residual_m = (
    analysis_df["residual_usd"].values
    / 1e6
)

# Yoğunluk scatter
ax1.scatter(
    x_jitter,
    residual_m,
    s=5,
    alpha=0.12
)

# Risk bazlı median residual
median_residual_m = (
    risk_summary[
        "median_residual_usd"
    ].values
    / 1e6
)

mean_residual_m = (
    risk_summary[
        "mean_residual_usd"
    ].values
    / 1e6
)

ax1.plot(
    x,
    median_residual_m,
    marker="o",
    linewidth=2.2,
    markersize=6,
    label="Median Residual"
)

ax1.plot(
    x,
    mean_residual_m,
    marker="s",
    linewidth=2.2,
    markersize=6,
    label="Mean Residual"
)

ax1.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero Error"
)

ax1.set_title(
    "A1 — Residual vs No-CFT Risk",
    fontsize=14,
    fontweight="bold"
)

ax1.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax1.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax1.set_xticks(x)
ax1.set_xticklabels(risk_labels)

# Residual dağılımının büyük kısmını görünür tut
residual_q = np.percentile(
    residual_m,
    [1, 99]
)

limit = max(
    abs(residual_q[0]),
    abs(residual_q[1]),
    1
)

ax1.set_ylim(
    -limit * 1.15,
    limit * 1.15
)

ax1.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax1.legend(
    fontsize=9,
    loc="best"
)

# Median değerlerini açıkça yaz
for i, value in enumerate(
    median_residual_m
):

    if np.isfinite(value):

        ax1.annotate(
            f"{value:+.2f}M",
            (
                x[i],
                value
            ),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            fontweight="bold"
        )

# ============================================================
# A2 — ABSOLUTE RESIDUAL vs RISK
# ============================================================

absolute_error_m = (
    analysis_df[
        "absolute_error_usd"
    ].values
    / 1e6
)

ax2.scatter(
    x_jitter,
    absolute_error_m,
    s=5,
    alpha=0.12
)

median_abs_m = (
    risk_summary[
        "median_absolute_error_usd"
    ].values
    / 1e6
)

mean_abs_m = (
    risk_summary[
        "mean_absolute_error_usd"
    ].values
    / 1e6
)

ax2.plot(
    x,
    median_abs_m,
    marker="o",
    linewidth=2.2,
    markersize=6,
    label="Median Absolute Error"
)

ax2.plot(
    x,
    mean_abs_m,
    marker="s",
    linewidth=2.2,
    markersize=6,
    label="MAE"
)

ax2.set_title(
    "A2 — Absolute Error vs No-CFT Risk",
    fontsize=14,
    fontweight="bold"
)

ax2.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax2.set_ylabel(
    "Absolute Error (Million USD)",
    fontsize=11
)

ax2.set_xticks(x)
ax2.set_xticklabels(risk_labels)

# Outlierların grafiği ezmesini engelle
abs_q = np.percentile(
    absolute_error_m,
    [1, 99]
)

ax2.set_ylim(
    0,
    max(
        abs_q[1] * 1.15,
        1
    )
)

ax2.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax2.legend(
    fontsize=9,
    loc="best"
)

# Median + MAE değerlerini göster
for i in range(
    len(risk_levels)
):

    if np.isfinite(
        median_abs_m[i]
    ):

        ax2.annotate(
            f"Median: {median_abs_m[i]:.2f}M",
            (
                x[i],
                median_abs_m[i]
            ),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            fontweight="bold"
        )

# ============================================================
# A3 — RESIDUAL DISTRIBUTION BY RISK
# ============================================================

residual_groups = [
    analysis_df.loc[
        analysis_df["risk"] == level,
        "residual_usd"
    ].values / 1e6
    for level in risk_levels
]

bp = ax3.boxplot(
    residual_groups,
    labels=risk_labels,
    patch_artist=True,
    showfliers=False,
    widths=0.55,
    medianprops=dict(
        color="black",
        linewidth=2
    ),
    whiskerprops=dict(
        linewidth=1.2
    ),
    capprops=dict(
        linewidth=1.2
    )
)

ax3.axhline(
    0,
    linestyle="--",
    linewidth=1.3,
    label="Zero Error"
)

ax3.set_title(
    "A3 — Residual Distribution by No-CFT Risk",
    fontsize=14,
    fontweight="bold"
)

ax3.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax3.set_ylabel(
    "Residual (Million USD)",
    fontsize=11
)

# Boxplot için de merkezi dağılımı gösterecek uygun limit
box_low = np.percentile(
    residual_m,
    2
)

box_high = np.percentile(
    residual_m,
    98
)

box_limit = max(
    abs(box_low),
    abs(box_high),
    1
)

ax3.set_ylim(
    -box_limit * 1.10,
    box_limit * 1.10
)

ax3.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax3.legend(
    fontsize=9,
    loc="best"
)

# Median residual değerleri
for i, value in enumerate(
    median_residual_m,
    1
):

    if np.isfinite(value):

        ax3.annotate(
            f"Median\n{value:+.2f}M",
            (
                i,
                value
            ),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            fontweight="bold"
        )

# ============================================================
# A4 — RELATIVE ERROR vs RISK
# ============================================================

relative_error = (
    analysis_df[
        "relative_error_percent"
    ].values
)

valid_relative = np.isfinite(
    relative_error
)

relative_risk = (
    x_jitter[valid_relative]
)

relative_values = (
    relative_error[valid_relative]
)

ax4.scatter(
    relative_risk,
    relative_values,
    s=5,
    alpha=0.12
)

# Median relative error
median_relative = []

for level in risk_levels:

    values = analysis_df.loc[
        (
            analysis_df["risk"]
            == level
        )
        & (
            analysis_df[
                "relative_error_percent"
            ].notna()
        ),
        "relative_error_percent"
    ]

    if len(values) > 0:
        median_relative.append(
            np.median(values)
        )
    else:
        median_relative.append(
            np.nan
        )

median_relative = np.asarray(
    median_relative
)

ax4.plot(
    x,
    median_relative,
    marker="o",
    linewidth=2.2,
    markersize=6,
    label="Median Relative Error"
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero Bias"
)

ax4.axhline(
    30,
    linestyle=":",
    linewidth=1.2,
    label="+30% Error"
)

ax4.axhline(
    -30,
    linestyle=":",
    linewidth=1.2,
    label="-30% Error"
)

# Extreme percentage errors grafiği bozmasın
relative_q = np.percentile(
    relative_values,
    [1, 99]
)

rel_low = max(
    relative_q[0],
    -150
)

rel_high = min(
    relative_q[1],
    150
)

if rel_low >= rel_high:
    rel_low = -100
    rel_high = 100

ax4.set_ylim(
    rel_low * 1.10,
    rel_high * 1.10
)

ax4.set_title(
    "A4 — Relative Prediction Error vs No-CFT Risk",
    fontsize=14,
    fontweight="bold"
)

ax4.set_xlabel(
    "No-CFT Risk (corr_nocft)",
    fontsize=11
)

ax4.set_ylabel(
    "Relative Error (%)\n(Actual − Predicted) / Actual",
    fontsize=11
)

ax4.set_xticks(x)
ax4.set_xticklabels(risk_labels)

ax4.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax4.legend(
    fontsize=8.5,
    loc="best"
)

# Median relative error etiketleri
for i, value in enumerate(
    median_relative
):

    if np.isfinite(value):

        ax4.annotate(
            f"{value:+.1f}%",
            (
                x[i],
                value
            ),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            fontweight="bold"
        )

# ============================================================
# GENEL BAŞLIK
# ============================================================

fig.suptitle(
    "XGBoost Final Model — Residual Diagnostics by No-CFT Risk",
    fontsize=17,
    fontweight="bold",
    y=0.995
)

fig.text(
    0.5,
    0.965,
    "corr_nocft: İlan yapılmaması / çağrı süreci risk göstergesi",
    ha="center",
    fontsize=10
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

risk_plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_residual_diagnostics_2x2.png"
)

plt.savefig(
    risk_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# SONUÇLARI YAZDIR
# ============================================================

print("\n" + "=" * 120)
print(
    "corr_nocft — RESIDUAL DIAGNOSTIC ANALYSIS"
)
print("=" * 120)

print(
    f"{'Risk':<8}"
    f"{'N':>10}"
    f"{'Mean Residual':>20}"
    f"{'Median Residual':>20}"
    f"{'MAE':>18}"
    f"{'RMSE':>18}"
    f"{'MdAPE':>12}"
    f"{'>30%':>10}"
    f"{'>50%':>10}"
)

print("-" * 130)

for level in risk_levels:

    row = risk_summary.loc[level]

    print(
        f"{level:<8.1f}"
        f"{int(row['count']):>10,}"
        f"${row['mean_residual_usd']:>18,.0f}"
        f"${row['median_residual_usd']:>18,.0f}"
        f"${row['mean_absolute_error_usd']:>16,.0f}"
        f"${row['rmse_usd']:>16,.0f}"
        f"{row['MdAPE_percent']:>10.2f}%"
        f"{row['error_over_30pct']:>9.2f}%"
        f"{row['error_over_50pct']:>9.2f}%"
    )

print("\nGrafik:")
print(risk_plot_path)

print("\nRisk analiz tablosu:")
print(summary_path)

print("\n" + "=" * 120)
print(
    "corr_nocft RESIDUAL ANALİZİ TAMAMLANDI"
)
print("=" * 120)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FEATURE = "corr_nocft"
FEATURE_NAME = "No-CFT Risk"
OUTPUT_DIR = "xgboost_final_test"

RANDOM_STATE = 42
PLOT_SAMPLE = 15000
N_BINS = 40

os.makedirs(OUTPUT_DIR, exist_ok=True)

if hasattr(X_test, "to_list"):
    risk_values = np.asarray(X_test[FEATURE].to_list(), dtype=np.float64)
else:
    risk_values = np.asarray(X_test[FEATURE], dtype=np.float64)

analysis_df = pd.DataFrame({
    "risk": risk_values,
    "actual_usd": np.asarray(test_predictions["actual_usd"], dtype=np.float64),
    "predicted_usd": np.asarray(test_predictions["predicted_usd"], dtype=np.float64),
    "residual_usd": np.asarray(test_predictions["residual_usd"], dtype=np.float64),
    "absolute_error_usd": np.asarray(test_predictions["absolute_error_usd"], dtype=np.float64)
})

analysis_df = (
    analysis_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

risk_levels = sorted(analysis_df["risk"].unique())
risk_labels = [f"{x:g}" for x in risk_levels]

analysis_df["fitted_m"] = analysis_df["predicted_usd"] / 1e6
analysis_df["residual_m"] = analysis_df["residual_usd"] / 1e6
analysis_df["absolute_error_m"] = analysis_df["absolute_error_usd"] / 1e6

residual_std = analysis_df["residual_usd"].std()

if residual_std > 0:
    analysis_df["standardized_residual"] = (
        analysis_df["residual_usd"] / residual_std
    )
else:
    analysis_df["standardized_residual"] = 0.0

analysis_df["scale_location"] = np.sqrt(
    np.abs(analysis_df["standardized_residual"])
)

analysis_df["relative_error_percent"] = np.nan

valid_actual = analysis_df["actual_usd"] > 0

analysis_df.loc[valid_actual, "relative_error_percent"] = (
    analysis_df.loc[valid_actual, "residual_usd"] /
    analysis_df.loc[valid_actual, "actual_usd"]
) * 100

if len(analysis_df) > PLOT_SAMPLE:
    plot_df = analysis_df.sample(
        PLOT_SAMPLE,
        random_state=RANDOM_STATE
    )
else:
    plot_df = analysis_df.copy()


def get_bin_statistics(x, y, bins=N_BINS, min_count=20):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < min_count:
        return None

    edges = np.linspace(
        np.min(x),
        np.max(x),
        bins + 1
    )

    centers = []
    medians = []
    q1_values = []
    q3_values = []
    counts = []

    for i in range(bins):
        if i == bins - 1:
            mask = (
                (x >= edges[i]) &
                (x <= edges[i + 1])
            )
        else:
            mask = (
                (x >= edges[i]) &
                (x < edges[i + 1])
            )

        if mask.sum() >= min_count:
            values = y[mask]

            centers.append(np.median(x[mask]))
            medians.append(np.median(values))
            q1_values.append(np.percentile(values, 25))
            q3_values.append(np.percentile(values, 75))
            counts.append(mask.sum())

    if len(centers) < 2:
        return None

    return (
        np.asarray(centers),
        np.asarray(medians),
        np.asarray(q1_values),
        np.asarray(q3_values),
        np.asarray(counts)
    )


def symmetric_ylim(values, low=1, high=99):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return -1, 1

    q_low, q_high = np.percentile(
        values,
        [low, high]
    )

    limit = max(
        abs(q_low),
        abs(q_high),
        1
    )

    return (
        -limit * 1.15,
        limit * 1.15
    )


fig, axes = plt.subplots(
    2,
    3,
    figsize=(20, 12),
    dpi=300
)

ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()


# A1 — RESIDUALS VS FITTED

a1_stats = get_bin_statistics(
    plot_df["fitted_m"],
    plot_df["residual_m"]
)

if a1_stats is not None:
    x, median, q1, q3, counts = a1_stats

    ax1.fill_between(
        x,
        q1,
        q3,
        alpha=0.20,
        label="25–75% residual range"
    )

    ax1.plot(
        x,
        median,
        linewidth=2.8,
        marker="o",
        markersize=3.5,
        label="Median residual"
    )

    scatter_sample = plot_df.sample(
        min(4000, len(plot_df)),
        random_state=RANDOM_STATE
    )

    ax1.scatter(
        scatter_sample["fitted_m"],
        scatter_sample["residual_m"],
        s=6,
        alpha=0.08,
        rasterized=True
    )

ax1.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero residual"
)

ax1.set_ylim(
    *symmetric_ylim(plot_df["residual_m"])
)

ax1.set_title(
    "A1 — Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

ax1.set_xlabel(
    "Predicted Tender Price (Million USD)",
    fontsize=11
)

ax1.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax1.grid(
    axis="both",
    alpha=0.20,
    linestyle="--"
)

ax1.legend(fontsize=8.5)


# A2 — NORMAL Q-Q

qq_values = plot_df["standardized_residual"].values
qq_values = qq_values[np.isfinite(qq_values)]

stats.probplot(
    qq_values,
    dist="norm",
    plot=ax2
)

for line in ax2.get_lines():
    line.set_markersize(2.5)

ax2.set_title(
    "A2 — Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

ax2.set_xlabel(
    "Theoretical Quantiles",
    fontsize=11
)

ax2.set_ylabel(
    "Standardized Residuals",
    fontsize=11
)

ax2.grid(
    alpha=0.20,
    linestyle="--"
)


# A3 — SCALE LOCATION

a3_stats = get_bin_statistics(
    plot_df["fitted_m"],
    plot_df["scale_location"]
)

if a3_stats is not None:
    x, median, q1, q3, counts = a3_stats

    ax3.fill_between(
        x,
        q1,
        q3,
        alpha=0.20,
        label="25–75% spread"
    )

    ax3.plot(
        x,
        median,
        linewidth=2.8,
        marker="o",
        markersize=3.5,
        label="Median spread"
    )

ax3.set_title(
    "A3 — Residual Spread vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

ax3.set_xlabel(
    "Predicted Tender Price (Million USD)",
    fontsize=11
)

ax3.set_ylabel(
    "√|Standardized Residual|",
    fontsize=11
)

ax3.grid(
    axis="both",
    alpha=0.20,
    linestyle="--"
)

ax3.legend(fontsize=8.5)


# A4 — RESIDUALS VS RISK

risk_map = {
    level: i
    for i, level in enumerate(risk_levels)
}

risk_numeric = (
    plot_df["risk"]
    .map(risk_map)
    .astype(float)
)

rng = np.random.default_rng(RANDOM_STATE)

x_jitter = (
    risk_numeric.values +
    rng.normal(
        0,
        0.055,
        len(plot_df)
    )
)

ax4.scatter(
    x_jitter,
    plot_df["residual_m"],
    s=7,
    alpha=0.10,
    rasterized=True
)

for i, level in enumerate(risk_levels):
    group = analysis_df.loc[
        analysis_df["risk"] == level,
        "residual_m"
    ].dropna()

    if len(group) == 0:
        continue

    q1, median, q3 = np.percentile(
        group,
        [25, 50, 75]
    )

    mae = np.mean(np.abs(group))

    ax4.vlines(
        i,
        q1,
        q3,
        linewidth=9,
        alpha=0.55
    )

    ax4.scatter(
        i,
        median,
        s=70,
        marker="o",
        edgecolors="black",
        linewidths=1.3,
        zorder=5
    )

    ax4.annotate(
        f"Bias: {median:+.2f}M\nMAE: {mae:.2f}M",
        xy=(i, median),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        fontweight="bold"
    )

ax4.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero residual"
)

ax4.set_ylim(
    *symmetric_ylim(plot_df["residual_m"])
)

ax4.set_title(
    f"A4 — Residual Error by {FEATURE_NAME}",
    fontsize=14,
    fontweight="bold"
)

ax4.set_xlabel(
    f"{FEATURE_NAME} ({FEATURE})",
    fontsize=11
)

ax4.set_ylabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax4.set_xticks(
    np.arange(len(risk_levels))
)

ax4.set_xticklabels(
    risk_labels
)

ax4.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax4.legend(fontsize=8.5)


# A5 — RESIDUAL DISTRIBUTION

residual_values = (
    analysis_df["residual_m"]
    .dropna()
    .values
)

if len(residual_values) > 0:
    clipped = residual_values[
        (residual_values >= -100) &
        (residual_values <= 100)
    ]

    if len(clipped) > 0:
        counts, edges = np.histogram(
            clipped,
            bins=60,
            range=(-100, 100)
        )

        residual_centers = (
            edges[:-1] +
            np.diff(edges) / 2
        )

        widths = np.diff(edges)

        ax5.bar(
            residual_centers,
            counts,
            width=widths * 0.90,
            alpha=0.65,
            align="center"
        )

    median_residual = np.median(
        residual_values
    )

    mean_residual = np.mean(
        residual_values
    )

    ax5.axvline(
        0,
        linestyle="--",
        linewidth=1.5,
        label="Zero residual"
    )

    if -100 <= median_residual <= 100:
        ax5.axvline(
            median_residual,
            linewidth=2,
            label=f"Median: {median_residual:+.2f}M"
        )

    if -100 <= mean_residual <= 100:
        ax5.axvline(
            mean_residual,
            linestyle=":",
            linewidth=2,
            label=f"Mean: {mean_residual:+.2f}M"
        )

ax5.set_xlim(-100, 100)

ax5.set_title(
    "A5 — Residual Error Distribution",
    fontsize=14,
    fontweight="bold"
)

ax5.set_xlabel(
    "Residual (Million USD)\nActual − Predicted",
    fontsize=11
)

ax5.set_ylabel(
    "Number of Tenders",
    fontsize=11
)

ax5.set_xticks(
    np.arange(-100, 101, 20)
)

ax5.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax5.legend(
    fontsize=8.5
)


# A6 — RELATIVE RESIDUALS VS RISK

relative_df = plot_df[
    plot_df["relative_error_percent"].notna()
].copy()

if len(relative_df) > 0:
    relative_numeric = (
        relative_df["risk"]
        .map(risk_map)
        .astype(float)
    )

    relative_jitter = (
        relative_numeric.values +
        rng.normal(
            0,
            0.055,
            len(relative_df)
        )
    )

    ax6.scatter(
        relative_jitter,
        relative_df["relative_error_percent"],
        s=7,
        alpha=0.10,
        rasterized=True
    )

    for i, level in enumerate(risk_levels):
        values = analysis_df.loc[
            (
                analysis_df["risk"] == level
            ) &
            (
                analysis_df[
                    "relative_error_percent"
                ].notna()
            ),
            "relative_error_percent"
        ].values

        if len(values) == 0:
            continue

        q1, median, q3 = np.percentile(
            values,
            [25, 50, 75]
        )

        mae_relative = np.mean(
            np.abs(values)
        )

        ax6.vlines(
            i,
            q1,
            q3,
            linewidth=9,
            alpha=0.55
        )

        ax6.scatter(
            i,
            median,
            s=70,
            marker="o",
            edgecolors="black",
            linewidths=1.3,
            zorder=5
        )

        ax6.annotate(
            f"Bias: {median:+.1f}%\nMAE: {mae_relative:.1f}%",
            xy=(i, median),
            xytext=(0, 12),
            textcoords="offset points",
            ha="center",
            fontsize=8,
            fontweight="bold"
        )

    ax6.axhline(
        0,
        linestyle="--",
        linewidth=1.5,
        label="Zero relative error"
    )

    ax6.axhline(
        30,
        linestyle=":",
        linewidth=1.2,
        label="+30%"
    )

    ax6.axhline(
        -30,
        linestyle=":",
        linewidth=1.2,
        label="-30%"
    )

    rel_values = relative_df[
        "relative_error_percent"
    ].values

    rel_low, rel_high = np.percentile(
        rel_values,
        [1, 99]
    )

    rel_low = max(rel_low, -150)
    rel_high = min(rel_high, 150)

    if rel_low >= rel_high:
        rel_low = -100
        rel_high = 100

    ax6.set_ylim(
        rel_low * 1.10,
        rel_high * 1.10
    )

ax6.set_title(
    f"A6 — Relative Residual Error by {FEATURE_NAME}",
    fontsize=14,
    fontweight="bold"
)

ax6.set_xlabel(
    f"{FEATURE_NAME} ({FEATURE})",
    fontsize=11
)

ax6.set_ylabel(
    "Relative Residual (%)\n"
    "(Actual − Predicted) / Actual",
    fontsize=11
)

ax6.set_xticks(
    np.arange(len(risk_levels))
)

ax6.set_xticklabels(
    risk_labels
)

ax6.grid(
    axis="y",
    alpha=0.20,
    linestyle="--"
)

ax6.legend(
    fontsize=8
)


fig.suptitle(
    "XGBoost Final Model — Residual Analysis for No-CFT Risk",
    fontsize=18,
    fontweight="bold",
    y=0.995
)

fig.text(
    0.5,
    0.965,
    "Residual = Actual − Predicted | Positive = underprediction | Negative = overprediction",
    ha="center",
    fontsize=10
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95],
    h_pad=3.0,
    w_pad=2.5
)

plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_true_residual_diagnostics.png"
)

plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# RİSK BAZLI ÖZET

summary_rows = []

for level in risk_levels:
    group = analysis_df[
        analysis_df["risk"] == level
    ]

    residuals = group[
        "residual_usd"
    ].dropna()

    relative = group[
        "relative_error_percent"
    ].dropna()

    summary_rows.append({
        "risk": level,
        "n": len(group),
        "mean_residual_usd": residuals.mean(),
        "median_residual_usd": residuals.median(),
        "std_residual_usd": residuals.std(),
        "mae_usd": group[
            "absolute_error_usd"
        ].mean(),
        "rmse_usd": np.sqrt(
            np.mean(
                residuals.values ** 2
            )
        ),
        "mean_relative_error_pct": (
            relative.mean()
            if len(relative) > 0
            else np.nan
        ),
        "median_relative_error_pct": (
            relative.median()
            if len(relative) > 0
            else np.nan
        ),
        "mae_relative_error_pct": (
            relative.abs().mean()
            if len(relative) > 0
            else np.nan
        ),
        "error_over_30pct": (
            (relative.abs() > 30).mean() * 100
            if len(relative) > 0
            else np.nan
        ),
        "error_over_50pct": (
            (relative.abs() > 50).mean() * 100
            if len(relative) > 0
            else np.nan
        )
    })

risk_summary = pd.DataFrame(
    summary_rows
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "corr_nocft_true_residual_summary.csv"
)

risk_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print("corr_nocft — TRUE RESIDUAL DIAGNOSTIC ANALYSIS")
print("=" * 100)

for _, row in risk_summary.iterrows():
    print(
        f"\nRisk {row['risk']:g}"
        f"\n  Gözlem sayısı             : {row['n']:,}"
        f"\n  Mean residual             : ${row['mean_residual_usd']:,.0f}"
        f"\n  Median residual           : ${row['median_residual_usd']:,.0f}"
        f"\n  Residual std              : ${row['std_residual_usd']:,.0f}"
        f"\n  MAE                       : ${row['mae_usd']:,.0f}"
        f"\n  RMSE                      : ${row['rmse_usd']:,.0f}"
        f"\n  Mean relative residual    : {row['mean_relative_error_pct']:+.2f}%"
        f"\n  Median relative residual  : {row['median_relative_error_pct']:+.2f}%"
        f"\n  MAE relative error        : {row['mae_relative_error_pct']:.2f}%"
        f"\n  Error > ±30%              : {row['error_over_30pct']:.2f}%"
        f"\n  Error > ±50%              : {row['error_over_50pct']:.2f}%"
    )

print("\nGrafik:")
print(plot_path)

print("\nRisk bazlı residual özeti:")
print(summary_path)

print("\n" + "=" * 100)
print("corr_nocft RESIDUAL DIAGNOSTIC ANALİZİ TAMAMLANDI")
print("=" * 100)

## CORR_DECP (İhale kararlarının alınması ve sonuçlandırılması süreçlerindeki şeffaflık eksikliğinden veya olası usulsüzlüklerden kaynaklanan risk düzeyini ölçen bir göstergedir.)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURE = "corr_decp"
FEATURE_NAME = "Decision Duration Risk"
OUTPUT_DIR = "xgboost_final_test"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test verisinden risk değerlerini al
if hasattr(X_test, "to_list"):
    risk_values = np.asarray(
        X_test[FEATURE].to_list(),
        dtype=np.float64
    )
else:
    risk_values = np.asarray(
        X_test[FEATURE],
        dtype=np.float64
    )

# Test sonuçlarını hazırla
analysis_df = pd.DataFrame({
    "risk": risk_values,
    "actual_usd": np.asarray(
        test_predictions["actual_usd"],
        dtype=np.float64
    ),
    "predicted_usd": np.asarray(
        test_predictions["predicted_usd"],
        dtype=np.float64
    ),
    "residual_usd": np.asarray(
        test_predictions["residual_usd"],
        dtype=np.float64
    ),
    "absolute_error_usd": np.asarray(
        test_predictions["absolute_error_usd"],
        dtype=np.float64
    )
})

analysis_df = analysis_df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

# Risk seviyelerini otomatik belirle
risk_levels = sorted(
    analysis_df["risk"].unique()
)

risk_labels = [
    f"{x:g}" for x in risk_levels
]

# Relative Error
analysis_df["relative_error_percent"] = np.nan

positive_actual = analysis_df["actual_usd"] > 0

analysis_df.loc[positive_actual, "relative_error_percent"] = (
    (
        analysis_df.loc[positive_actual, "predicted_usd"]
        - analysis_df.loc[positive_actual, "actual_usd"]
    )
    / analysis_df.loc[positive_actual, "actual_usd"]
) * 100

# Risk bazlı özet
risk_summary = (
    analysis_df
    .groupby("risk")
    .agg(
        count=("risk", "size"),

        mean_actual_usd=(
            "actual_usd",
            "mean"
        ),

        mean_predicted_usd=(
            "predicted_usd",
            "mean"
        ),

        mean_residual_usd=(
            "residual_usd",
            "mean"
        ),

        median_residual_usd=(
            "residual_usd",
            "median"
        ),

        mean_absolute_error_usd=(
            "absolute_error_usd",
            "mean"
        ),

        median_absolute_error_usd=(
            "absolute_error_usd",
            "median"
        ),

        rmse_usd=(
            "residual_usd",
            lambda x: np.sqrt(
                np.mean(np.square(x))
            )
        )
    )
    .reindex(risk_levels)
)

# Bias %
risk_summary["bias_percent"] = np.where(
    risk_summary["mean_actual_usd"] != 0,
    (
        (
            risk_summary["mean_predicted_usd"]
            - risk_summary["mean_actual_usd"]
        )
        / risk_summary["mean_actual_usd"]
    ) * 100,
    np.nan
)

# Büyük hata oranı
# |Relative Error| > 30%
large_error_rate = []

for level in risk_levels:

    group = analysis_df[
        analysis_df["risk"] == level
    ]

    valid_relative = group[
        "relative_error_percent"
    ].dropna()

    if len(valid_relative) > 0:
        rate = (
            np.abs(valid_relative) > 30
        ).mean() * 100
    else:
        rate = np.nan

    large_error_rate.append(rate)

risk_summary["large_error_rate_percent"] = (
    large_error_rate
)

# Medyan mutlak yüzde hata
median_relative_error = []

for level in risk_levels:

    group = analysis_df[
        analysis_df["risk"] == level
    ]

    valid_relative = group[
        "relative_error_percent"
    ].dropna()

    if len(valid_relative) > 0:
        value = np.median(
            np.abs(valid_relative)
        )
    else:
        value = np.nan

    median_relative_error.append(value)

risk_summary[
    "median_absolute_relative_error_percent"
] = median_relative_error

# CSV kaydet
risk_summary_path = os.path.join(
    OUTPUT_DIR,
    "corr_decp_risk_analysis.csv"
)

risk_summary.to_csv(
    risk_summary_path
)

# Grafik
x = np.arange(len(risk_levels))
width = 0.34

fig, axes = plt.subplots(
    1,
    3,
    figsize=(21, 7.5),
    dpi=300
)

ax1, ax2, ax3 = axes

# ------------------------------------------------------------
# A1 — ACTUAL VS PREDICTED MEAN
# ------------------------------------------------------------

actual_mean = (
    risk_summary[
        "mean_actual_usd"
    ].values / 1e6
)

predicted_mean = (
    risk_summary[
        "mean_predicted_usd"
    ].values / 1e6
)

bias_percent = (
    risk_summary[
        "bias_percent"
    ].values
)

bias_usd = (
    risk_summary[
        "mean_predicted_usd"
    ].values
    -
    risk_summary[
        "mean_actual_usd"
    ].values
)

bars_actual = ax1.bar(
    x - width / 2,
    actual_mean,
    width,
    label="Actual Mean"
)

bars_predicted = ax1.bar(
    x + width / 2,
    predicted_mean,
    width,
    label="Predicted Mean"
)

ax1.set_title(
    "A1 — Prediction Bias by Decision Duration Risk",
    fontsize=13,
    fontweight="bold",
    pad=18
)

ax1.set_xlabel(
    "Decision Duration Risk (corr_decp)",
    fontsize=11,
    labelpad=10
)

ax1.set_ylabel(
    "Mean Tender Price (Million USD)",
    fontsize=11,
    labelpad=10
)

ax1.set_xticks(x)
ax1.set_xticklabels(risk_labels)

ax1.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax1.legend(
    fontsize=9,
    loc="upper left"
)

# Y ekseninin üstünde yeterli boşluk oluştur
max_bar_value = max(
    np.max(actual_mean),
    np.max(predicted_mean)
)

ax1.set_ylim(
    0,
    max_bar_value * 1.28
)

# Actual / Predicted değerleri
# Her barın hemen üzerinde ayrı ayrı göster
for bar in bars_actual:

    height = bar.get_height()

    ax1.annotate(
        f"${height:.2f}M",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            height
        ),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

for bar in bars_predicted:

    height = bar.get_height()

    ax1.annotate(
        f"${height:.2f}M",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            height
        ),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

# Bias bilgisini barların üstüne koymak yerine
# x eksenine yakın ayrı bir satırda göster
for i in range(len(risk_levels)):

    if np.isfinite(bias_percent[i]):

        bias_text = (
            f"Bias {bias_percent[i]:+.1f}%\n"
            f"Δ ${bias_usd[i] / 1e6:+.2f}M"
        )

        ax1.annotate(
            bias_text,
            xy=(i, 0),
            xytext=(0, -42),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8.5,
            fontweight="bold",
            clip_on=False
        )

# ------------------------------------------------------------
# A2 — MAE VS RMSE
# ------------------------------------------------------------

mae_values = (
    risk_summary[
        "mean_absolute_error_usd"
    ].values / 1e6
)

rmse_values = (
    risk_summary[
        "rmse_usd"
    ].values / 1e6
)

bars_mae = ax2.bar(
    x - width / 2,
    mae_values,
    width,
    label="MAE"
)

bars_rmse = ax2.bar(
    x + width / 2,
    rmse_values,
    width,
    label="RMSE"
)

ax2.set_title(
    "A2 — Error Magnitude & Outlier Effect",
    fontsize=13,
    fontweight="bold",
    pad=18
)

ax2.set_xlabel(
    "Decision Duration Risk (corr_decp)",
    fontsize=11,
    labelpad=10
)

ax2.set_ylabel(
    "Error (Million USD)",
    fontsize=11,
    labelpad=10
)

ax2.set_xticks(x)
ax2.set_xticklabels(risk_labels)

ax2.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

ax2.legend(
    fontsize=9,
    loc="upper left"
)

for bars in [bars_mae, bars_rmse]:

    for bar in bars:

        height = bar.get_height()

        ax2.annotate(
            f"${height:.2f}M",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                height
            ),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8.5
        )

# ------------------------------------------------------------
# A3 — LARGE RELATIVE ERROR RATE
# ------------------------------------------------------------

large_error_values = (
    risk_summary[
        "large_error_rate_percent"
    ].values
)

median_ape_values = (
    risk_summary[
        "median_absolute_relative_error_percent"
    ].values
)

bars_large = ax3.bar(
    x,
    large_error_values,
    width=0.50,
    label="|Relative Error| > 30%"
)

ax3.set_title(
    "A3 — Large Prediction Error Rate",
    fontsize=13,
    fontweight="bold",
    pad=18
)

ax3.set_xlabel(
    "Decision Duration Risk (corr_decp)",
    fontsize=11,
    labelpad=10
)

ax3.set_ylabel(
    "Ihales with Large Error (%)",
    fontsize=11,
    labelpad=10
)

ax3.set_xticks(x)
ax3.set_xticklabels(risk_labels)

ax3.grid(
    axis="y",
    alpha=0.25,
    linestyle="--"
)

for bar in bars_large:

    height = bar.get_height()

    if np.isfinite(height):

        ax3.annotate(
            f"{height:.1f}%",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                height
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

for i, value in enumerate(median_ape_values):

    if np.isfinite(value):

        ax3.annotate(
            f"Median |RE|: {value:.1f}%",
            xy=(
                i,
                large_error_values[i]
            ),
            xytext=(0, 25),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8
        )

# Genel başlık
fig.suptitle(
    "XGBoost Final Model — Decision Duration Risk Error Analysis",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

fig.text(
    0.5,
    0.985,
    "corr_decp: Decision duration related risk indicator",
    ha="center",
    fontsize=10
)

# Paneller arası boşluk
plt.subplots_adjust(
    left=0.055,
    right=0.985,
    bottom=0.20,
    top=0.84,
    wspace=0.32
)

risk_plot_path = os.path.join(
    OUTPUT_DIR,
    "corr_decp_risk_error_analysis_3panel.png"
)

plt.savefig(
    risk_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Sonuçları yazdır
print("\n" + "=" * 80)
print("corr_decp — DECISION DURATION RISK ERROR ANALYSIS")
print("=" * 80)

print(
    f"{'Risk':<10}"
    f"{'N':>10}"
    f"{'MAE':>18}"
    f"{'RMSE':>18}"
    f"{'Bias':>14}"
    f"{'Large Error':>18}"
)

print("-" * 88)

for level in risk_levels:

    row = risk_summary.loc[level]

    print(
        f"{level:<10.1f}"
        f"{int(row['count']):>10,}"
        f"${row['mean_absolute_error_usd']:>16,.0f}"
        f"${row['rmse_usd']:>16,.0f}"
        f"{row['bias_percent']:>12.2f}%"
        f"{row['large_error_rate_percent']:>16.2f}%"
    )

print("\nGrafik:")
print(risk_plot_path)

print("\nRisk analiz tablosu:")
print(risk_summary_path)

**A1 — Tahmin Sapması ve Sistematik Yanlılık Analizi (Prediction Bias)**

A1 panelindeki tahmin sapması (Prediction Bias) analiz edildiğinde, XGBoost modelinin genel olarak ihale değerlerini eksik tahmin etme (under-prediction) eğiliminde olduğu, ancak karar süresi riski (corr_decp) yükseldikçe bu sapmanın katlanarak arttığı görülmektedir. Karar süresi riski bulunmayan ihalelerde (Risk = 0) ortalama 9.94M USD tutarındaki gerçek ihale bedeli model tarafından 8.23M USD olarak tahmin edilerek -%17.2'lik (-1.71M USD) kabul edilebilir bir yanılma sergilenmektedir. Buna karşın karar süresi riski taşıyan ihalelerde (Risk = 1) ortalama 3.21M USD olan gerçek bedel, model tarafından yarı yarıya kırılarak 1.68M USD seviyesinde tahmin edilmekte ve sistemli bias tutarı -%47.6'ya (-1.53M USD) fırlamaktadır. Bu durum, modelin karar süresi uzayan veya şeffaflık riski barındıran ihalelerde ihale değerini olduğundan çok daha düşük öngördüğünü açıkça kanıtlamaktadır.

**A2 — Hata Büyüklüğü ve Uç Değer Etkisi Analizi (Error Magnitude & Outlier Effect)**

A2 panelindeki hata büyüklüğü ve uç değer etkisi (Error Magnitude & Outlier Effect) incelendiğinde, tipik ihalelerdeki mutlak hata (MAE) ile yüksek bütçeli ihalelerdeki karesel hata (RMSE) arasındaki devasa uçurum dikkat çekmektedir. Mutlak ortalama hata riski olmayan ihalelerde 2.18M USD, riski olan ihalelerde ise 1.90M USD seviyesinde neredeyse sabit kalırken; RMSE değerleri Risk 0 grubunda 13.16M USD, Risk 1 grubunda ise 17.15M USD seviyesine fırlamaktadır. MAE ile RMSE arasındaki bu yaklaşık 9 katlık sapma, model hatasının portföyün geneline yayılmış bir fiyasko olmadığını, özellikle karar süresi riski içeren ve aşırı yüksek bütçeli birkaç dev ihalenin (outliers) tahmininde gerçekleşen devasa sapmalardan kaynaklandığını göstermektedir.

**A3 — Yüksek Hata Oranı ve Frekans Dağılımı Analizi (Large Prediction Error Rate)**

A3 panelindeki yüksek hata oranı (Large Prediction Error Rate) değerlendirildiğinde ise, hatanın sadece uç değerlerde kalmadığı, ihalelerin genel frekansına da yüksek oranda sirayet ettiği gözlemlenmektedir. Karar süresi riski bulunmayan ihalelerin %60.2'sinde yüksek hata payı gerçekleşirken (Medyan Göreli Hata %43.1), riski olan ihalelerde bu oran %66.0'a yükselmekte ve Medyan Göreli Hata (MedRE) %49.8 seviyesine ulaşmaktadır. Özetle model; karar süresi riski (corr_decp = 1) olan her 3 ihaleden 2'sinde ciddi oranlarda hatalı tahmin üretmekte, hem ortalama değer bazında %47.6'lık ağır bir eksik tahmin (under-prediction) yapmakta hem de portföydeki dev bütçeli outlier ihalelerde hatası katlanarak büyümektedir.

## CORR_BUYER_CONCENTRATION (İhalelerin belirli alıcı kurum veya idareler üzerinde yoğunlaşmasından kaynaklanan pazar hakimiyeti ve rekabet kısıtlaması riskini ölçen bir göstergedir.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy import stats

FEATURE = "corr_buyer_concentration"
N_BINS = 10
PLOT_SAMPLE = 30000
RANDOM_STATE = 42

if "test_predictions" not in globals():
    raise NameError("test_predictions bulunamadı.")

if "X_test" not in globals():
    raise NameError("X_test bulunamadı.")

if FEATURE not in X_test.columns:
    raise ValueError(f"{FEATURE} X_test içinde bulunamadı.")

required = [
    "index",
    "actual_usd",
    "predicted_usd",
    "residual_usd",
    "absolute_error_usd"
]

missing = [
    c for c in required
    if c not in test_predictions.columns
]

if missing:
    raise ValueError(
        f"test_predictions içinde eksik kolonlar: {missing}"
    )

# X_test'i pandas'a çevir
if isinstance(X_test, pd.DataFrame):
    x_test_df = X_test.copy()
elif hasattr(X_test, "to_pandas"):
    x_test_df = X_test.to_pandas()
else:
    x_test_df = pd.DataFrame(X_test)

# Risk kolonunu ve gerçek index'i hazırla
x_test_df = x_test_df.copy()

x_test_df["_row_index"] = x_test_df.index

risk_df = x_test_df[
    ["_row_index", FEATURE]
].copy()

risk_df[FEATURE] = pd.to_numeric(
    risk_df[FEATURE],
    errors="coerce"
)

# Prediction sonuçlarını hazırla
pred_df = test_predictions.copy()

pred_df["index"] = pd.to_numeric(
    pred_df["index"],
    errors="coerce"
)

for col in [
    "actual_usd",
    "predicted_usd",
    "residual_usd",
    "absolute_error_usd"
]:
    pred_df[col] = pd.to_numeric(
        pred_df[col],
        errors="coerce"
    )

# Kritik düzeltme:
# Risk ve residual INDEX üzerinden eşleştiriliyor.
df = pred_df.merge(
    risk_df,
    left_on="index",
    right_on="_row_index",
    how="inner"
)

df = df.rename(
    columns={
        FEATURE: "risk",
        "residual_usd": "residual",
        "absolute_error_usd": "abs_error"
    }
)

df = df[
    [
        "index",
        "risk",
        "actual_usd",
        "predicted_usd",
        "residual",
        "abs_error"
    ]
]

df = (
    df.replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
    .reset_index(drop=True)
)

if len(df) == 0:
    raise ValueError(
        "Risk ve prediction sonuçları index üzerinden eşleştirilemedi."
    )

print(
    f"Index eşleşmesi başarılı: {len(df):,} gözlem"
)

# Risk quantile
df["risk_bin"] = pd.qcut(
    df["risk"],
    q=N_BINS,
    duplicates="drop"
)

# Genel istatistikler
spearman_r, spearman_p = stats.spearmanr(
    df["risk"],
    df["residual"]
)

rmse = np.sqrt(
    np.mean(df["residual"] ** 2)
)

mae = df["abs_error"].mean()

mean_bias = df["residual"].mean()

median_residual = df["residual"].median()

# Risk grubu istatistikleri
grouped = (
    df.groupby(
        "risk_bin",
        observed=True
    )
    .agg(
        n=("residual", "size"),
        mean_risk=("risk", "mean"),
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        MAE=("abs_error", "mean"),
        RMSE=(
            "residual",
            lambda x: np.sqrt(
                np.mean(x ** 2)
            )
        )
    )
    .reset_index()
)

# Grafik örneklemi
if len(df) > PLOT_SAMPLE:
    plot_df = df.sample(
        PLOT_SAMPLE,
        random_state=RANDOM_STATE
    )
else:
    plot_df = df.copy()

fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 11)
)

ax1, ax2 = axes[0]
ax3, ax4 = axes[1]

# A1 — Residual Density Heatmap

x = plot_df["risk"].to_numpy()
y = plot_df["residual"].to_numpy()

valid = (
    np.isfinite(x) &
    np.isfinite(y)
)

x = x[valid]
y = y[valid]

if len(x) == 0:
    raise ValueError(
        "A1 için geçerli veri bulunamadı."
    )

# Residual'ı signed-log uzayına taşı.
# Böylece küçük residual'lar ezilmez,
# milyonluk ekstrem hatalar da grafiği bozmaz.
residual_scale = 1_000_000.0

y_plot = (
    np.sign(y)
    * np.log10(
        1 + np.abs(y) / residual_scale
    )
)

# X aralığını doğrudan gerçek veriden al
x_min = np.min(x)
x_max = np.max(x)

if x_min == x_max:
    x_min -= 0.05
    x_max += 0.05

# Y ekseni için gerçek dönüşmüş aralık
y_min = np.min(y_plot)
y_max = np.max(y_plot)

if y_min == y_max:
    y_min -= 0.1
    y_max += 0.1

# 2D histogram
H, x_edges, y_edges = np.histogram2d(
    x,
    y_plot,
    bins=[
        50,
        60
    ]
)

# Sadece dolu hücreleri kullan
positive_counts = H[H > 0]

if len(positive_counts) == 0:
    raise ValueError(
        "A1 heatmap için dolu hücre bulunamadı."
    )

# Log renk skalası
vmin = max(
    1,
    positive_counts.min()
)

vmax = positive_counts.max()

mesh = ax1.pcolormesh(
    x_edges,
    y_edges,
    H.T,
    shading="auto",
    cmap="viridis",
    norm=LogNorm(
        vmin=vmin,
        vmax=vmax
    )
)

# Zero residual'ın signed-log karşılığı
ax1.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero Residual"
)

# Median residual trendini aynı signed-log uzayında göster
trend_y = (
    np.sign(
        grouped["median_residual"].to_numpy()
    )
    * np.log10(
        1
        + np.abs(
            grouped["median_residual"].to_numpy()
        ) / residual_scale
    )
)

ax1.plot(
    grouped["mean_risk"],
    trend_y,
    marker="o",
    linewidth=2.5,
    label="Median Residual"
)

ax1.set_xlim(
    x_min,
    x_max
)

ax1.set_ylim(
    y_min,
    y_max
)

# Signed-log Y ticklerini USD olarak göster
tick_values = np.array([
    -1e8,
    -1e7,
    -1e6,
    -1e5,
    -1e4,
    -1e3,
    0,
    1e3,
    1e4,
    1e5,
    1e6,
    1e7,
    1e8
])

tick_positions = (
    np.sign(tick_values)
    * np.log10(
        1
        + np.abs(tick_values)
        / residual_scale
    )
)

valid_ticks = (
    (tick_positions >= y_min) &
    (tick_positions <= y_max)
)

ax1.set_yticks(
    tick_positions[valid_ticks]
)

def format_usd(value):
    value = float(value)

    if value == 0:
        return "$0"

    abs_value = abs(value)

    if abs_value >= 1e9:
        text = f"${abs_value / 1e9:.0f}B"
    elif abs_value >= 1e6:
        text = f"${abs_value / 1e6:.0f}M"
    elif abs_value >= 1e3:
        text = f"${abs_value / 1e3:.0f}K"
    else:
        text = f"${abs_value:.0f}"

    return (
        "-" + text
        if value < 0
        else text
    )

ax1.set_yticklabels(
    [
        format_usd(v)
        for v in tick_values[valid_ticks]
    ]
)

ax1.set_xlabel(
    "CORR_BUYER_CONCENTRATION"
)

ax1.set_ylabel(
    "Residual (USD)"
)

ax1.set_title(
    "A1 — Residual Density vs Buyer Concentration Risk\n"
    f"Spearman ρ = {spearman_r:.3f}, "
    f"p = {spearman_p:.3g}"
)

ax1.grid(
    alpha=0.15
)

ax1.legend(
    loc="upper right"
)

cbar = fig.colorbar(
    mesh,
    ax=ax1
)

cbar.set_label(
    "Observation Density (log scale)"
)

# A2 — Residual Distribution by Risk

box_data = [
    group["residual"].values
    for _, group in df.groupby(
        "risk_bin",
        observed=True
    )
]

ax2.boxplot(
    box_data,
    showfliers=False
)

ax2.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero Residual"
)

ax2.set_xticks(
    range(
        1,
        len(box_data) + 1
    )
)

ax2.set_xticklabels(
    [
        f"Q{i}"
        for i in range(
            1,
            len(box_data) + 1
        )
    ]
)

ax2.set_xlabel(
    "Risk Quantile"
)

ax2.set_ylabel(
    "Residual (USD)"
)

ax2.set_title(
    "A2 — Residual Distribution by Risk Level"
)

a2_low, a2_high = np.percentile(
    df["residual"],
    [1, 99]
)

a2_limit = max(
    abs(a2_low),
    abs(a2_high)
)

if a2_limit > 0:
    ax2.set_ylim(
        -a2_limit,
        a2_limit
    )

ax2.ticklabel_format(
    axis="y",
    style="sci",
    scilimits=(6, 6)
)

ax2.grid(
    axis="y",
    alpha=0.2
)

ax2.legend(
    loc="upper right"
)

# A3 — MAE vs RMSE

x_pos = np.arange(
    len(grouped)
)

width = 0.36

ax3.bar(
    x_pos - width / 2,
    grouped["MAE"],
    width,
    label="MAE"
)

ax3.bar(
    x_pos + width / 2,
    grouped["RMSE"],
    width,
    label="RMSE"
)

ax3.set_xticks(
    x_pos
)

ax3.set_xticklabels(
    [
        f"Q{i}"
        for i in range(
            1,
            len(grouped) + 1
        )
    ]
)

ax3.set_xlabel(
    "Risk Quantile"
)

ax3.set_ylabel(
    "Error (USD)"
)

ax3.set_title(
    "A3 — Model Error Magnitude by Risk Level"
)

ax3.ticklabel_format(
    axis="y",
    style="sci",
    scilimits=(6, 6)
)

ax3.grid(
    axis="y",
    alpha=0.2
)

ax3.legend()

# A4 — Prediction Bias

bias = grouped["mean_residual"]

ax4.bar(
    x_pos,
    bias
)

ax4.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero Bias"
)

ax4.set_xticks(
    x_pos
)

ax4.set_xticklabels(
    [
        f"Q{i}"
        for i in range(
            1,
            len(grouped) + 1
        )
    ]
)

ax4.set_xlabel(
    "Risk Quantile"
)

ax4.set_ylabel(
    "Mean Residual (USD)"
)

ax4.set_title(
    "A4 — Prediction Bias by Risk Level"
)

ax4.ticklabel_format(
    axis="y",
    style="sci",
    scilimits=(6, 6)
)

max_bias = max(
    abs(bias.min()),
    abs(bias.max())
)

if max_bias > 0:

    for i, value in enumerate(bias):

        offset = max_bias * 0.025

        ax4.text(
            i,
            value + (
                offset
                if value >= 0
                else -offset
            ),
            f"${value / 1e6:.2f}M",
            ha="center",
            va="bottom"
            if value >= 0
            else "top",
            fontsize=9
        )

ax4.grid(
    axis="y",
    alpha=0.2
)

ax4.legend()

fig.suptitle(
    "CORR_BUYER_CONCENTRATION — Residual Diagnostics",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.96
    ]
)

plt.show()

# Özet
print("\n" + "=" * 72)
print("CORR_BUYER_CONCENTRATION — RESIDUAL SUMMARY")
print("=" * 72)

print(
    f"Gözlem sayısı       : {len(df):,}"
)

print(
    f"Risk minimum        : "
    f"{df['risk'].min():.6f}"
)

print(
    f"Risk maksimum       : "
    f"{df['risk'].max():.6f}"
)

print(
    f"Risk medyan         : "
    f"{df['risk'].median():.6f}"
)

print(
    f"Mean residual       : "
    f"${mean_bias:,.2f}"
)

print(
    f"Median residual     : "
    f"${median_residual:,.2f}"
)

print(
    f"MAE                 : "
    f"${mae:,.2f}"
)

print(
    f"RMSE                : "
    f"${rmse:,.2f}"
)

print(
    f"Spearman rho        : "
    f"{spearman_r:.4f}"
)

print(
    f"Spearman p-value    : "
    f"{spearman_p:.6g}"
)

print("\nRisk Quantile Sonuçları")

print(
    grouped[
        [
            "risk_bin",
            "n",
            "mean_risk",
            "mean_residual",
            "median_residual",
            "MAE",
            "RMSE"
        ]
    ].to_string(
        index=False
    )
)

print(
    "\nRisk Seviyesine Göre Model Davranışı"
)

for i, row in grouped.iterrows():

    bias_value = row["mean_residual"]

    if bias_value > 0:
        direction = "DÜŞÜK TAHMİN"
    elif bias_value < 0:
        direction = "YÜKSEK TAHMİN"
    else:
        direction = "NÖTR"

    print(
        f"Q{i + 1}: "
        f"Risk={row['mean_risk']:.4f} | "
        f"Bias=${bias_value:,.0f} | "
        f"MAE=${row['MAE']:,.0f} | "
        f"RMSE=${row['RMSE']:,.0f} | "
        f"{direction}"
    )

## KATEGORİK KOLONLAR

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

OUTPUT_DIR = "xgboost_weighted_high_price"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "test_predictions.csv")
PLOT_DIR = os.path.join(OUTPUT_DIR, "categorical_residual_plots")

MAX_CATEGORIES = 15
MAX_POINTS = 12000
BOOTSTRAP = 500
SEED = 42

os.makedirs(PLOT_DIR, exist_ok=True)

# TEST SONUÇLARINI OKU
if not os.path.exists(PREDICTION_PATH):
    raise FileNotFoundError(
        f"Test prediction dosyası bulunamadı: {PREDICTION_PATH}"
    )

pred_df = pd.read_csv(PREDICTION_PATH)

if "df_test" not in globals():
    raise NameError(
        "df_test bulunamadı. Kategorik kolonlar için df_test gerekli."
    )

# df_test -> pandas
if isinstance(df_test, pd.DataFrame):
    df_res = df_test.copy()
elif hasattr(df_test, "to_pandas"):
    df_res = df_test.to_pandas()
else:
    df_res = pd.DataFrame(df_test)

if len(pred_df) != len(df_res):
    raise ValueError(
        f"Satır sayısı uyuşmuyor: "
        f"predictions={len(pred_df):,}, "
        f"df_test={len(df_res):,}"
    )

print("\n" + "=" * 90)
print("CATEGORICAL RESIDUAL ANALYSIS — SAVED TEST RESULTS")
print("=" * 90)
print(f"Test Samples : {len(pred_df):,}")
print(f"Prediction   : {PREDICTION_PATH}")
print("Model testi  : YAPILMADI")
print("Residual     : test_predictions.csv üzerinden")

# TEST RESIDUAL SONUÇLARINI EKLE
df_res["actual_usd"] = pred_df["actual_usd"].to_numpy()
df_res["predicted_usd"] = pred_df["predicted_usd"].to_numpy()
df_res["residual_usd"] = pred_df["residual_usd"].to_numpy()
df_res["absolute_error_usd"] = pred_df["absolute_error_usd"].to_numpy()

# SADECE GERÇEK KATEGORİK KOLONLAR
categorical_cols = df_res.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Prediction kolonlarını çıkar
excluded_cols = {
    "actual_usd",
    "predicted_usd",
    "residual_usd",
    "absolute_error_usd"
}

categorical_cols = [
    col for col in categorical_cols
    if col not in excluded_cols
]

print(f"Kategorik kolon sayısı : {len(categorical_cols)}")
print("Kategorik kolonlar:")
for col in categorical_cols:
    print(f"  - {col}")

# KATEGORİK ÖZET
summary_rows = []

for col in categorical_cols:

    temp = df_res[
        [col, "residual_usd", "absolute_error_usd"]
    ].copy()

    temp[col] = temp[col].astype("object").where(
        temp[col].notna(),
        "MISSING"
    )

    for category, group in temp.groupby(
        col,
        dropna=False
    ):

        residual = group["residual_usd"].to_numpy()
        absolute_error = group["absolute_error_usd"].to_numpy()

        if len(residual) == 0:
            continue

        summary_rows.append({
            "feature": col,
            "category": str(category),
            "count": len(residual),
            "mean_residual_usd": np.mean(residual),
            "median_residual_usd": np.median(residual),
            "mae_usd": np.mean(absolute_error),
            "rmse_usd": np.sqrt(np.mean(residual ** 2)),
            "underprediction_pct": np.mean(residual > 0) * 100,
            "overprediction_pct": np.mean(residual < 0) * 100,
            "zero_residual_pct": np.mean(residual == 0) * 100
        })

summary_df = pd.DataFrame(summary_rows)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "categorical_residual_analysis.csv"
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

# SADECE KATEGORİK KOLONLAR
plot_features = [
    col for col in categorical_cols
    if 2 <= df_res[col].nunique(dropna=True) <= 20
]

print(
    f"\nPlot oluşturulacak kategorik kolon sayısı : "
    f"{len(plot_features)}"
)

# 2'Lİ GRUPLAR HALİNDE PLOT
for page_start in range(0, len(plot_features), 2):

    page_features = plot_features[
        page_start:page_start + 2
    ]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(18, 7)
    )

    # Tek grafik varsa axes'i liste haline getir
    if len(page_features) == 1:
        axes = [axes]

    for ax, col in zip(axes, page_features):

        temp = df_res[
            [col, "residual_usd"]
        ].copy()

        temp[col] = temp[col].astype("object").where(
            temp[col].notna(),
            "MISSING"
        )

        # En fazla 12 kategori
        category_counts = (
            temp[col]
            .value_counts()
            .head(MAX_CATEGORIES)
        )

        categories = category_counts.index.tolist()

        temp = temp[
            temp[col].isin(categories)
        ].copy()

        # Plot için örnekleme
        if len(temp) > MAX_POINTS:
            temp = temp.sample(
                MAX_POINTS,
                random_state=SEED
            )

        x_map = {
            category: i
            for i, category in enumerate(categories)
        }

        x = temp[col].map(x_map).to_numpy(
            dtype=float
        )

        y = temp["residual_usd"].to_numpy(
            dtype=float
        )

        rng = np.random.default_rng(SEED)

        jitter = rng.uniform(
            -0.20,
            0.20,
            len(x)
        )

        x_jitter = x + jitter

        # YOĞUNLUK HESABI
        points = np.vstack([
            x_jitter,
            y
        ])

        try:

            if len(y) >= 20 and np.std(y) > 0:

                density = gaussian_kde(
                    points,
                    bw_method="scott"
                )(points)

            else:

                density = np.ones(
                    len(y)
                )

        except Exception:

            density = np.ones(
                len(y)
            )

        # Yoğunluk sıralaması
        order = np.argsort(density)

        scatter = ax.scatter(
            x_jitter[order],
            y[order],
            c=density[order],
            cmap="viridis",
            s=14,
            alpha=0.65,
            edgecolors="none"
        )

        # Kategori ortalaması + %95 CI
        for i, category in enumerate(categories):

            group = temp.loc[
                temp[col] == category,
                "residual_usd"
            ].to_numpy()

            if len(group) == 0:
                continue

            mean_residual = np.mean(group)

            if len(group) >= 10:

                boot_means = np.array([
                    np.mean(
                        rng.choice(
                            group,
                            size=len(group),
                            replace=True
                        )
                    )
                    for _ in range(BOOTSTRAP)
                ])

                ci_low, ci_high = np.percentile(
                    boot_means,
                    [2.5, 97.5]
                )

            else:

                ci_low = mean_residual
                ci_high = mean_residual

            ax.errorbar(
                i,
                mean_residual,
                yerr=[
                    [mean_residual - ci_low],
                    [ci_high - mean_residual]
                ],
                fmt="o",
                markersize=8,
                capsize=5,
                linewidth=2
            )

        # Sıfır residual
        ax.axhline(
            0,
            linestyle="--",
            linewidth=1.5
        )

        ax.set_xticks(
            range(len(categories))
        )

        ax.set_xticklabels(
            [str(c) for c in categories],
            rotation=35,
            ha="right"
        )

        ax.set_xlabel(
            col
        )

        ax.set_ylabel(
            "Residual (USD)\nActual − Predicted"
        )

        ax.set_title(
            f"{col}\n"
            f"Categorical Residual | N={len(temp):,}"
        )

        ax.grid(
            axis="y",
            alpha=0.25
        )

        # Yoğunluk renk skalası
        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=0.02
        )

        cbar.set_label(
            "Point Density"
        )

    # Kullanılmayan subplot
    if len(page_features) == 1:
        axes[1].axis("off")

    fig.suptitle(
        "Categorical Residual Analysis — Saved Test Results",
        fontsize=15
    )

    plt.tight_layout()

    page_number = (
        page_start // 2
    ) + 1

    plot_path = os.path.join(
        PLOT_DIR,
        f"categorical_residual_page_{page_number}.png"
    )

    plt.savefig(
        plot_path,
        dpi=200,
        bbox_inches="tight"
    )

    print(
        f"\n✓ Sayfa {page_number}: "
        f"{', '.join(page_features)}"
    )
    print(
        f"  Plot: {plot_path}"
    )

    plt.show()
    plt.close()

# FEATURE ÖZETİ
feature_summary = (
    summary_df
    .groupby("feature")
    .agg(
        categories=("category", "count"),
        total_samples=("count", "sum"),
        mean_residual_usd=("mean_residual_usd", "mean"),
        mean_mae_usd=("mae_usd", "mean"),
        mean_rmse_usd=("rmse_usd", "mean")
    )
    .reset_index()
)

FEATURE_SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "categorical_feature_residual_summary.csv"
)

feature_summary.to_csv(
    FEATURE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 90)
print("KAYDEDİLEN DOSYALAR")
print("=" * 90)
print(f"Residual Summary : {SUMMARY_PATH}")
print(f"Feature Summary  : {FEATURE_SUMMARY_PATH}")
print(f"Plots            : {PLOT_DIR}")

print("\n" + "=" * 90)
print("KATEGORİK RESIDUAL ANALİZİ TAMAMLANDI")
print("=" * 90)

## NUMERİC KOLONLAR

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

OUTPUT_DIR = "xgboost_weighted_high_price"
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "test_predictions.csv"
)
PLOT_DIR = os.path.join(
    OUTPUT_DIR,
    "numeric_residual_plots"
)

MAX_POINTS = 12000
SEED = 42

os.makedirs(PLOT_DIR, exist_ok=True)

# TEST SONUÇLARINI OKU
if not os.path.exists(PREDICTION_PATH):
    raise FileNotFoundError(
        f"Test prediction dosyası bulunamadı: {PREDICTION_PATH}"
    )

pred_df = pd.read_csv(PREDICTION_PATH)

if "df_test" not in globals():
    raise NameError(
        "df_test bulunamadı. Numerik kolonlar için df_test gerekli."
    )

# df_test -> pandas
if isinstance(df_test, pd.DataFrame):
    df_res = df_test.copy()
elif hasattr(df_test, "to_pandas"):
    df_res = df_test.to_pandas()
else:
    df_res = pd.DataFrame(df_test)

if len(pred_df) != len(df_res):
    raise ValueError(
        f"Satır sayısı uyuşmuyor: "
        f"predictions={len(pred_df):,}, "
        f"df_test={len(df_res):,}"
    )

print("\n" + "=" * 90)
print("NUMERIC RESIDUAL ANALYSIS — SAVED TEST RESULTS")
print("=" * 90)
print(f"Test Samples : {len(pred_df):,}")
print(f"Prediction   : {PREDICTION_PATH}")
print("Model testi  : YAPILMADI")
print("X            : Numeric Feature")
print("Y            : Residual USD (Actual - Predicted)")

# TEST SONUÇLARINI EKLE
df_res["actual_usd"] = pred_df["actual_usd"].to_numpy()
df_res["predicted_usd"] = pred_df["predicted_usd"].to_numpy()
df_res["residual_usd"] = pred_df["residual_usd"].to_numpy()
df_res["absolute_error_usd"] = pred_df["absolute_error_usd"].to_numpy()

# SADECE GERÇEK NUMERİK KOLONLAR
numeric_cols = df_res.select_dtypes(
    include=[np.number]
).columns.tolist()

excluded_cols = {
    "actual_usd",
    "predicted_usd",
    "residual_usd",
    "absolute_error_usd"
}

numeric_cols = [
    col for col in numeric_cols
    if col not in excluded_cols
]

print(f"Numerik kolon sayısı : {len(numeric_cols)}")
print("Numerik kolonlar:")

for col in numeric_cols:
    print(f"  - {col}")

# NUMERİK ÖZET
summary_rows = []

for col in numeric_cols:

    temp = df_res[
        [col, "residual_usd", "absolute_error_usd"]
    ].copy()

    temp[col] = pd.to_numeric(
        temp[col],
        errors="coerce"
    )

    temp = temp.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna(
        subset=[
            col,
            "residual_usd"
        ]
    )

    if len(temp) == 0:
        continue

    x = temp[col].to_numpy(
        dtype=float
    )

    residual = temp[
        "residual_usd"
    ].to_numpy(
        dtype=float
    )

    absolute_error = temp[
        "absolute_error_usd"
    ].to_numpy(
        dtype=float
    )

    summary_rows.append({
        "feature": col,
        "count": len(x),
        "mean_value": np.mean(x),
        "median_value": np.median(x),
        "min_value": np.min(x),
        "max_value": np.max(x),
        "mean_residual_usd": np.mean(residual),
        "median_residual_usd": np.median(residual),
        "mae_usd": np.mean(absolute_error),
        "rmse_usd": np.sqrt(
            np.mean(residual ** 2)
        ),
        "underprediction_pct": np.mean(
            residual > 0
        ) * 100,
        "overprediction_pct": np.mean(
            residual < 0
        ) * 100,
        "zero_residual_pct": np.mean(
            residual == 0
        ) * 100
    })

summary_df = pd.DataFrame(
    summary_rows
)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "numeric_residual_analysis.csv"
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nPlot oluşturulacak numerik kolon sayısı : "
    f"{len(numeric_cols)}"
)

# 2'Lİ GRUPLAR HALİNDE PLOT
for page_start in range(
    0,
    len(numeric_cols),
    2
):

    page_features = numeric_cols[
        page_start:page_start + 2
    ]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(18, 7)
    )

    if len(page_features) == 1:
        axes = list(axes)

    for ax, col in zip(
        axes,
        page_features
    ):

        temp = df_res[
            [
                col,
                "residual_usd"
            ]
        ].copy()

        temp[col] = pd.to_numeric(
            temp[col],
            errors="coerce"
        )

        temp = temp.replace(
            [np.inf, -np.inf],
            np.nan
        ).dropna(
            subset=[
                col,
                "residual_usd"
            ]
        )

        if len(temp) == 0:
            ax.set_title(
                f"{col}\nNo valid data"
            )
            continue

        # Plot için örnekleme
        if len(temp) > MAX_POINTS:
            temp = temp.sample(
                MAX_POINTS,
                random_state=SEED
            )

        # X = NUMERİK KOLON
        x = temp[
            col
        ].to_numpy(
            dtype=float
        )

        # Y = RESIDUAL
        y = temp[
            "residual_usd"
        ].to_numpy(
            dtype=float
        )

        # 2D YOĞUNLUK
        points = np.vstack([
            x,
            y
        ])

        try:

            if (
                len(y) >= 20
                and np.std(x) > 0
                and np.std(y) > 0
            ):

                density = gaussian_kde(
                    points,
                    bw_method="scott"
                )(points)

            else:

                density = np.ones(
                    len(y)
                )

        except Exception:

            density = np.ones(
                len(y)
            )

        # Düşük yoğunluk önce,
        # yüksek yoğunluk sonra çizilir
        order = np.argsort(
            density
        )

        scatter = ax.scatter(
            x[order],
            y[order],
            c=density[order],
            cmap="viridis",
            s=14,
            alpha=0.65,
            edgecolors="none"
        )

        # RESIDUAL = 0
        ax.axhline(
            0,
            linestyle="--",
            linewidth=1.5
        )

        # X ekseni
        ax.set_xlabel(
            col
        )

        # Y ekseni
        ax.set_ylabel(
            "Residual (USD)\nActual − Predicted"
        )

        ax.set_title(
            f"{col}\n"
            f"Residual vs {col} | "
            f"N={len(temp):,}"
        )

        ax.grid(
            alpha=0.25
        )

        # Yoğunluk renk skalası
        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=0.02
        )

        cbar.set_label(
            "Point Density"
        )

        # Aşırı outlier'ların grafiği ezmesini önle
        x_low, x_high = np.percentile(
            x,
            [1, 99]
        )

        y_low, y_high = np.percentile(
            y,
            [1, 99]
        )

        if x_low < x_high:
            ax.set_xlim(
                x_low,
                x_high
            )

        if y_low < y_high:
            ax.set_ylim(
                y_low,
                y_high
            )

        # Büyük değerleri bilimsel göster
        if np.max(np.abs(x)) >= 1e6:
            ax.ticklabel_format(
                style="sci",
                axis="x",
                scilimits=(6, 6)
            )

        if np.max(np.abs(y)) >= 1e6:
            ax.ticklabel_format(
                style="sci",
                axis="y",
                scilimits=(6, 6)
            )

    # Tek grafik kaldıysa ikinci alanı kapat
    if len(page_features) == 1:
        axes[1].axis("off")

    fig.suptitle(
        "Numeric Residual Analysis — "
        "Feature vs Residual",
        fontsize=15
    )

    plt.tight_layout()

    page_number = (
        page_start // 2
    ) + 1

    plot_path = os.path.join(
        PLOT_DIR,
        f"numeric_residual_page_{page_number}.png"
    )

    plt.savefig(
        plot_path,
        dpi=200,
        bbox_inches="tight"
    )

    print(
        f"\n✓ Sayfa {page_number}: "
        f"{', '.join(page_features)}"
    )

    print(
        f"  Plot: {plot_path}"
    )

    plt.show()
    plt.close()

print("\n" + "=" * 90)
print("KAYDEDİLEN DOSYALAR")
print("=" * 90)
print(
    f"Residual Summary : {SUMMARY_PATH}"
)
print(
    f"Plots            : {PLOT_DIR}"
)

print("\n" + "=" * 90)
print("NUMERİK RESIDUAL ANALİZİ TAMAMLANDI")
print("=" * 90)

**Kolon Kolon Analiz, Sorunlar ve Çözüm Yöntemleri**

**log_tender_lotscount & log_tender_recordedbidscount**

**Sorun:** Lot sayısı ve teklif sayısı arttıkça dağılım sağa kaymakta, LOWESS trendi hafif yukarı eğilmektedir. Yüksek teklif ve çoklu lot içeren büyük kompleks ihalelerde model fiyatı eksik tahmin etmektedir.

**Çözüm:** Teklif sayısı ile lot sayısı arasında etkileşim sütunu türetilmelidir (bids_per_lot = recordedbidscount / lotscount). Ağaç tabanlı modeller için bu ölçek doğrusal olmayan bir karmaşıklık sunduğundan, logaritma öncesi ham değerlerin oranları modelin öğrenmesini kolaylaştırır.

**tender_selectionmethod**

**Sorun:** tender_selectionmethod değişkeninde özellikle LOWEST_PRICE kategorisinin yüksek fiyatlı ihalelerde sistematik olarak under-prediction ürettiği görülmektedir. Model, bu gruptaki yüksek değerli ihalelerin gerçek fiyatını yeterince öğrenememekte ve tahminleri gerçek değerlerin altında kalmaktadır. Bu durum, LOWEST_PRICE kategorisinin yüksek fiyat segmentindeki gözlemlerin model eğitimindeki etkisinin yetersiz kalmasından kaynaklanabilecek bir örnek ağırlığı problemine işaret etmektedir.

**Çözüm:** LOWEST_PRICE kategorisindeki yüksek fiyatlı gözlemlerin model eğitimindeki etkisini artırmak amacıyla Sample Weighting yöntemi uygulanmalıdır. Bunun için yalnızca eğitim verisi kullanılarak hedef değişkenin %75 quantile değeri eşik olarak belirlenmeli ve tender_selectionmethod == LOWEST_PRICE olan ve hedef değeri bu eşikten büyük veya eşit olan gözlemlere 2.0 ağırlık, diğer tüm gözlemlere ise 1.0 ağırlık atanmalıdır. Böylece model, yüksek fiyatlı LOWEST_PRICE ihalelerindeki tahmin hatalarını eğitim sırasında daha fazla dikkate alarak bu gruptaki sistematik under-prediction biasını azaltmayı öğrenebilecektir.

**tender_year_since_start**

**Sorun:** tender_year yalnızca takvim yılını göstermektedir ve yıllar arasındaki zaman ilerlemesinin doğrudan etkisini temsil etmekte sınırlı kalabilmektedir.

**Çözüm:** İhale yılının veri setindeki başlangıç yılına uzaklığını gösteren tender_year_since_start değişkeni oluşturulmuştur. Böylece modelin zaman içerisindeki genel trendi öğrenmesi desteklenmiştir.

**log_submission_period & log_decision_period & decision_duration_days**

**Sorun:** Teklif verme ve karar sürelerinde belirli gün değerlerinde yoğunlaşmalar ve süre arttıkça belirginleşen artık (residual) sapmaları gözlemlenmiştir. Özellikle uzun süren ihalelerde modelin fiyatı doğru şekilde ayırt etmekte zorlandığı görülmüştür. Bu nedenle yalnızca sürelerin ham değerlerini kullanmak, süre ile ihale fiyatı arasındaki doğrusal olmayan ilişkiyi yeterince temsil etmemektedir.

**Çözüm:** Teklif verme ve karar sürelerinin sağa çarpık dağılımını ve aşırı yüksek değerlerin etkisini azaltmak amacıyla bu değişkenlerin logaritmik dönüşümleri kullanılarak log_submission_period ve log_decision_period oluşturulmuştur. Ayrıca olağandışı uzun teklif verme ve karar süreçlerini doğrudan gösterebilmek için sırasıyla is_long_submission ve is_long_decision bayrak değişkenleri oluşturulmuştur. decision_duration_days değişkenindeki kronolojik olarak hatalı negatif değerler önce NaN haline getirilmiş, ardından oluşan eksiklikler medyan yöntemiyle doldurulmuştur. Bunun yanında karar süresinin teklif verme süresine göre göreli uzunluğunu modele aktarabilmek amacıyla decision_to_submission_ratio değişkeni oluşturulmuştur. Böylece model, sürelerin yalnızca mutlak değerlerini değil, sürelerin uzunluk durumunu ve karar süresinin teklif süresine göre ilişkisini de öğrenebilecek şekilde zenginleştirilmiştir.

**Risk Kolonları**

**Sorun:** Risk göstergelerinde eksik değerlerin -1 ile doldurulması özellikle corr_buyer_concentration değişkeninde gerçek değer aralığının dışına çıkarak yapay bir negatif uç değer oluşturmuş, corr_singleb ve corr_subm değişkenlerinde ise -1 değerinin gerçek bir risk/korelasyon değeri ile eksik veriyi birbirinden ayırmayı zorlaştırmıştır. Bu durum, eksikliklerin gerçek risk değerleriyle karışmasına ve modelin risk değişkenlerini hatalı yorumlamasına neden olabilecek bir dağılım bozulması oluşturmuştur.

**Çözüm:** Risk kolonlarındaki yapay olarak atanmış -1 değerleri NaN değerlerine geri dönüştürülmüş ve eksik gözlemlerin bilgi değerini kaybetmemesi amacıyla üç ayrı eksiklik bayrağı oluşturulmuştur: corr_singleb_is_missing, corr_subm_is_missing ve corr_buyer_concentration_is_missing. Bu bayraklarda 1 eksik, 0 mevcut değeri göstermektedir. Böylece corr_singleb, corr_subm ve corr_buyer_concentration değişkenlerinin gerçek dağılımları korunurken, eksikliğin kendisi de model tarafından ayrı bir bilgi olarak öğrenilebilir hale getirilmiştir. corr_proc, corr_nocft ve corr_decp değişkenlerinde ise -1 bulunmadığı ve değerler kendi geçerli aralıklarında olduğu için herhangi bir dönüşüm uygulanmamıştır. Sonuç olarak risk değişkenleri için gerçek değer ile eksik değer birbirinden ayrılmış, yapay -1 değerlerinin oluşturduğu dağılım bozulması ortadan kaldırılmış ve mevcut risk göstergeleri ile eksiklik bayrakları birlikte modele dahil edilmiştir.



Evet. Bu sonuçlar önceki residual analizleriyle birlikte değerlendirildiğinde modelin hatasının bütün continuous feature'lara homojen dağılmadığını, belirli koşullarda belirgin şekilde büyüdüğünü gösteriyor. En kritik problem corr_buyer_concentration, log_decision_period ve tender_year değişkenlerinde ortaya çıkıyor: alıcı yoğunlaşması yükseldiğinde, karar süresi uzadığında ve özellikle 2020 sonrası yıllarda MAE'nin belirgin biçimde artması, modelin bu bölgelerde ihale bedelinin ölçeğini yeterince yakalayamadığını düşündürüyor. log_tender_lotscount ve tender_awardcriteria_count ise özellikle az lotlu ve az kriterli ihalelerde hata varyansının yükseldiğini gösteriyor. Buna karşılık log_submission_period ve corr_subm gibi değişkenlerde MAE'nin binler arasında oldukça sabit kalması, bu değişkenlerin mevcut haliyle model hatasının ana kaynağı olmadığını gösteriyor. corr_proc, corr_decp, corr_nocft ve corr_singleb tarafında ise 0 çevresinde yoğunlaşan gözlemler nedeniyle klasik LOWESS eğimi tek başına güvenilir bir teşhis olarak görülmemeli; burada 0 değerinin gerçekten "risk yok" mu yoksa "eksik/bilinmiyor" mu olduğunu ayırmak daha önemli. Ayrıca MAE ≈ 2.0M iken RMSE ≈ 15.8M olması özellikle dikkat çekici: bu fark, modelin tipik gözlemlerde yaklaşık birkaç milyon dolar seviyesinde hata yaparken az sayıdaki çok büyük ihalede olağanüstü yüksek residual'lar ürettiğini gösteriyor. Dolayısıyla şu aşamada bütün feature'ları değiştirmek yerine, 1) büyük hedef değerlerin oluşturduğu sağ kuyruğu ayrı analiz etmek, 2) corr_buyer_concentration için yüksek değerli bölgeyi incelemek, 3) uzun karar sürelerini (>236 gün) ayrı analiz etmek, 4) tender_year için dönemsel bias'ı incelemek ve 5) corr_* = 0 değerlerinin missing/nötr ayrımını kontrol etmek en doğru sonraki adımlar olur. Özellikle feature engineering yapmadan önce bu beş bölgedeki actual–predicted–residual dağılımını görmek gerekir; çünkü aksi halde modelin gerçekten feature'ı mı yanlış öğrendiğini, yoksa birkaç büyük outlier'ın bütün metriği mi bozduğunu birbirinden ayıramayız.

**tender_year:** Zaman ilerledikçe modelin hata büyüklüğü (MAE) belirgin bir şekilde artıyor. 2006-2011 arası dönemde ortalama hata ~1.2M seviyesindeyken, 2018-2021 aralığındaki güncel ihalelerde bu hata 2.64M ile 2.91M bandına ulaşarak zirve yapıyor. Model yakın tarihli tahminlerde zorlanıyor.

**decision_duration_days & log_decision_period:** İhale karar süresi uzadıkça modelin tahmin yeteneği dramatik şekilde düşüyor. Özellikle 236 günden uzun süren ekstrem vakalarda hata büyüklüğü en yüksek seviyelere (2.72M - 2.99M MAE) ulaşıyor. Öte yandan, miktar olarak hataların en çok yığıldığı nokta 97-102 gün aralığıdır (~16 bin gözlem).

**tender_awardcriteria_count:** Ödül kriteri sayısının çok düşük olduğu (0-1 arası) durumlarda model hem frekans olarak çok sık yanılıyor (31 binden fazla gözlem) hem de en büyük ortalama hatayı (2.16M MAE) yapıyor. Kriter sayısı arttıkça hata 1.56M seviyelerine inerek nispeten düzeliyor.

**log_tender_lotscount:** Hataların ezici çoğunluğu (33 binden fazla) en düşük lot sayısı aralığında (0.69-1.10) gerçekleşiyor ve burada MAE oldukça yüksek (2.11M). En yüksek oransal sapma (2.29M MAE) ise 1.39-2.30 orta-yüksek değer aralığında görülüyor.

**log_submission_period:** Hata sayısında spesifik bir yığılma noktası var; model özellikle 3.56-3.61 aralığında (~13.8 bin gözlem) yoğunlukla hata yapıyor. Ortalama hata büyüklüğü ise daha yüksek değer aralığı olan 3.91-4.71 bandında (2.16M MAE) en üst seviyeye çıkıyor.

**corr_singleb & corr_subm:** corr_singleb değeri negatif veya düşük olduğunda (-1.0 ile 0.0 arası) model bariz şekilde daha çok yanılıyor ve hata payı (2.08M MAE) daha yüksek çıkıyor. Değer 0'ın üstüne çıktığında MAE 1.54M'e kadar iniyor. corr_subm değişkeninde de düşük değer aralığında hata frekansı yüksek olsa da, hata büyüklüğü iki grupta da (~2.0M) benzer kalıyor.

**log_tender_recordedbidscount & corr_proc:** Bu iki özelliğin grafikleri tek bir değer aralığında toplanmış (tek çubuk). Veri dağılımı tek bir grupta yığıldığı için aralıksal zayıf nokta tespiti yapılamıyor; her ikisi için de genel ortalama hata ~2.00M seviyesinde seyrediyor.

**corr_buyer_concentration:** Model bu değişkende hem yüksek frekansta hem de aşırı yüksek sapmayla hata yapıyor.

Hata Büyüklüğü (MAE): Konsantrasyon oranı yükseldikçe hata boyutu uç noktalara ulaşıyor. Özellikle 0.25 – 1.00 aralığında ortalama hata 4.48M MAE ile tüm grafikler içindeki en yüksek seviyeye çıkıyor. 0.12 – 0.25 aralığında da 2.40M MAE ile oldukça yüksek bir sapma var.

Hata Sayısı: -1.00 – 0.00 aralığında (muhtemelen eksik veya negatif olarak kodlanmış veriler) 13,373 gözlemle en yüksek hata frekansı görülüyor.

**corr_nocft:** Veriler tek bir grupta (-0.00 – 1.00) toplanmış durumda. Bu grupta toplam 44,575 hata gözlemi yer alıyor ve ortalama hata büyüklüğü 2.00M MAE seviyesinde sabit duruyor. Alt aralık kırılımı olmadığı için spesifik bir zayıf bölge ayrımı yapılamıyor.

**corr_decp:** Tıpkı corr_nocft gibi tek bir çubuk halinde gruplanmış (-0.00 – 1.00). Toplam 44,575 gözlemde ortalama 2.00M MAE hata mevcut. Değişkenin dağılımı ayrıştırılmadığı için kırılım bazlı zayıf nokta tespiti verilemiyor.

## MODEL EĞİTİMİ

Bir regresyon modelinin ne kadar iyi tahmin yaptığını ölçmek için MSE, RMSE, MAE ve R² en sık kullanılan metriklerdendir. Hepsi gerçek değer (y) ile modelin tahmin ettiği değer (ŷ) arasındaki farkı, yani hatayı farklı şekillerde ölçer.

**MSE (Mean Squared Error – Ortalama Kare Hata),** her gözlem için gerçek değer ile tahmin arasındaki farkın karesini alır ve bunların ortalamasını hesaplar. Hataların karesi alındığı için büyük hataları çok güçlü şekilde cezalandırır. Örneğin model bir ihaleyi 1 milyon dolar yerine 10 milyon dolar tahmin ederse, oluşan 9 milyon dolarlık hata karesi alınarak MSE'yi ciddi şekilde yükseltir. Bu nedenle özellikle büyük tahmin hatalarına duyarlıdır. Birimi de hedef değişkenin biriminin karesidir; sizin modelinizde USD² olduğu için doğrudan yorumlanması zordur.

**RMSE (Root Mean Squared Error – Kök Ortalama Kare Hata),** MSE'nin kareköküdür. Böylece hata tekrar hedef değişkenle aynı birime döner. Sizin modelinizde RMSE'nin birimi USD'dir. Örneğin RMSE = 16.000.000 ise bunu kabaca modelin tahmin hatasının tipik büyüklüğünün 16 milyon dolar civarında olduğu şeklinde yorumlayabiliriz; ancak RMSE kareleme nedeniyle büyük hatalardan özellikle etkilenir. Bu yüzden veri setinizdeki çok yüksek fiyatlı ihalelerin tahmin hataları RMSE'yi ciddi biçimde etkileyebilir.

**MAE (Mean Absolute Error – Ortalama Mutlak Hata),** gerçek değer ile tahmin arasındaki farkın mutlak değerini alıp ortalamasını hesaplar: \(MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|\). Burada hata karesi alınmadığı için büyük hataları RMSE/MSE kadar agresif biçimde cezalandırmaz. Örneğin MAE = $2.000.000 ise modelin tahminlerinin gerçek değerlerden ortalama mutlak olarak yaklaşık 2 milyon dolar uzaklıkta olduğunu söyleyebiliriz. Bu nedenle MAE, modelin genel tahmin hatasını anlamak açısından daha doğrudan ve yorumlanabilir bir metriktir.

**R² (R-squared – Belirleme Katsayısı)** ise diğer üçünden biraz farklıdır; doğrudan "ortalama kaç dolar hata yaptım?" sorusunu değil, modelin hedef değişkendeki varyasyonun ne kadarını açıkladığını ölçer.  (R²=1) mükemmel tahmin anlamına gelirken, (R²=0) modelin basitçe tüm gözlemler için ortalama hedef değeri tahmin etmekten daha iyi olmadığını ifade eder. Negatif R² ise modelin bu basit ortalama tahmininden bile daha kötü olduğunu gösterir. Örneğin R² = 0.81, modelin hedef değişkendeki varyasyonun yaklaşık %81'ini açıkladığı anlamına gelir; fakat bu, "tahminlerin %81'i doğru" anlamına gelmez.

In [ ]:
# XGBOOST NON-OVERFIT FINAL + SAMPLE WEIGHTING
# STRATIFIED 9-FOLD + TEK FINAL MODEL
#
# SAMPLE WEIGHTING:
# tender_selectionmethod == LOWEST_PRICE
# ve target >= TRAIN %75 quantile
# => weight = 2.0
# diğerleri => weight = 1.0

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# AYARLAR
SEED = 42
K = 9
N_ESTIMATORS = 5000
EARLY_STOPPING_ROUNDS = 9
CURRENT_GAP_THRESHOLD = 0.20
POST_BEST_THRESHOLD = 0.05
TARGET_IS_LOG = True

# SAMPLE WEIGHT
WEIGHT_BASE = 1.0
WEIGHT_LOWEST_PRICE_HIGH = 2.0
PRICE_QUANTILE = 0.75

MODEL_NAME = "XGBoost_NonOverfit_SampleWeighted"
OUTPUT_DIR = "xgboost_non_over_fit_sample_weighted"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# BEST PARAMETERS
BEST_PARAMS = {
    "max_depth": 9,
    "learning_rate": 0.06780801631911669,
    "min_child_weight": 15,
    "subsample": 0.7072478977786331,
    "colsample_bytree": 0.8784958115626409,
    "gamma": 1.0206765935941309,
    "reg_alpha": 4.940127643286809,
    "reg_lambda": 0.31923634586063737
}

# XGBOOST MODEL PARAMETERS
MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    **BEST_PARAMS,
    "seed": SEED,
    "nthread": -1
}

# EĞİTİM KONFİGÜRASYONU
training_configuration = {
    "model_name": MODEL_NAME,
    "framework": "XGBoost",
    "seed": SEED,
    "K": K,
    "n_estimators": N_ESTIMATORS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "current_gap_threshold": CURRENT_GAP_THRESHOLD,
    "post_best_threshold": POST_BEST_THRESHOLD,
    "target_is_log": TARGET_IS_LOG,
    "sample_weighting": {
        "enabled": True,
        "target_feature": "tender_selectionmethod",
        "target_category": "LOWEST_PRICE",
        "price_quantile": PRICE_QUANTILE,
        "low_weight": WEIGHT_BASE,
        "high_weight": WEIGHT_LOWEST_PRICE_HIGH
    },
    "best_params": BEST_PARAMS,
    "model_params": MODEL_PARAMS
}

with open(
    os.path.join(OUTPUT_DIR, "training_parameters.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        training_configuration,
        f,
        indent=4,
        ensure_ascii=False
    )

# ENCODE EDİLMİŞ VERİLER
if not isinstance(X_train_encoded, pd.DataFrame):
    X_train_encoded = pd.DataFrame(X_train_encoded)

if not isinstance(X_val_encoded, pd.DataFrame):
    X_val_encoded = pd.DataFrame(X_val_encoded)

# TRAIN + VALIDATION
X_cv = pd.concat(
    [X_train_encoded, X_val_encoded],
    ignore_index=True
)

y_cv = pd.concat(
    [pd.Series(y_train), pd.Series(y_val)],
    ignore_index=True
)

y_cv_np = np.asarray(
    y_cv,
    dtype=np.float64
)

# TARGET
if TARGET_IS_LOG:
    y_log = y_cv_np
    y_usd = np.maximum(
        np.expm1(y_log),
        0
    )
else:
    y_usd = np.maximum(
        y_cv_np,
        0
    )
    y_log = np.log1p(y_usd)

# ------------------------------------------------------------
# SAMPLE WEIGHTING
# SADECE TRAIN TARGET'I KULLANILARAK THRESHOLD HESAPLANIR
# ------------------------------------------------------------

y_train_np = np.asarray(
    y_train,
    dtype=np.float64
)

price_threshold = np.quantile(
    y_train_np,
    PRICE_QUANTILE
)

selection_train = (
    X_train[
        "tender_selectionmethod"
    ]
    .cast(str)
    .to_numpy()
)

sample_weights_train = np.ones(
    len(X_train),
    dtype=np.float64
)

high_lowest_price_mask = (
    (selection_train == "LOWEST_PRICE") &
    (y_train_np >= price_threshold)
)

sample_weights_train[
    high_lowest_price_mask
] = WEIGHT_LOWEST_PRICE_HIGH

print("\n" + "=" * 80)
print("SAMPLE WEIGHTING")
print("=" * 80)

print(
    f"Price Quantile      : {PRICE_QUANTILE:.2f}"
)

print(
    f"Price Threshold Log : "
    f"{price_threshold:.6f}"
)

print(
    f"LOWEST_PRICE + High Price : "
    f"{high_lowest_price_mask.sum():,}"
)

print(
    f"Weight 1.0          : "
    f"{(sample_weights_train == WEIGHT_BASE).sum():,}"
)

print(
    f"Weight 2.0          : "
    f"{(sample_weights_train == WEIGHT_LOWEST_PRICE_HIGH).sum():,}"
)

print(
    f"Mean Train Weight   : "
    f"{sample_weights_train.mean():.6f}"
)

# CV İÇİN AĞIRLIKLARI TRAIN/VAL SIRASINDA KULLANMAK İÇİN
# TRAIN VE VALIDATION ORİJİNAL İNDEKSLERİ
train_size_original = len(X_train)
val_size_original = len(X_val)

sample_weights_cv = np.concatenate([
    sample_weights_train,
    np.ones(
        val_size_original,
        dtype=np.float64
    )
])

# STRATIFICATION
supply_cv = np.concatenate([
    np.asarray(
        X_train["tender_supplytype"].to_list(),
        dtype=object
    ),
    np.asarray(
        X_val["tender_supplytype"].to_list(),
        dtype=object
    )
])

competition_cv = np.concatenate([
    np.asarray(
        X_train["competition_level"].to_list(),
        dtype=object
    ),
    np.asarray(
        X_val["competition_level"].to_list(),
        dtype=object
    )
])

# PRICE BIN
price_bin = pd.qcut(
    y_cv_np,
    q=10,
    labels=False,
    duplicates="drop"
)

stratify_df = pd.DataFrame({
    "price_bin": price_bin,
    "tender_supplytype": supply_cv,
    "competition_level": competition_cv
})

stratify_col = (
    stratify_df.astype(str)
    .agg("_".join, axis=1)
)

counts = stratify_col.value_counts()

# KONTROLLER
if len(X_cv) != len(y_cv):
    raise ValueError(
        "X ve y uzunlukları eşleşmiyor."
    )

if len(X_cv) != len(stratify_col):
    raise ValueError(
        "X ve stratification uzunlukları eşleşmiyor."
    )

if len(X_cv) != len(sample_weights_cv):
    raise ValueError(
        "X ve sample weight uzunlukları eşleşmiyor."
    )

if counts.min() < K:
    raise ValueError(
        f"En küçük stratification kombinasyonu "
        f"{counts.min()} örnek. K={K} için yeterli değil."
    )

# OVERFITTING CALLBACK
class OverfittingCallback(
    xgb.callback.TrainingCallback
):

    def __init__(self):
        self.history = []
        self.best_iteration = -1
        self.best_val = np.inf
        self.best_train = np.inf
        self.no_improvement = 0
        self.stopping_reason = "MAX_ESTIMATORS"

    def after_iteration(
        self,
        model,
        epoch,
        evals_log
    ):

        train = float(
            evals_log["train"]["rmse"][-1]
        )

        val = float(
            evals_log["validation"]["rmse"][-1]
        )

        gap = (
            (val - train) /
            max(train, 1e-12)
        )

        if val < self.best_val:
            self.best_val = val
            self.best_train = train
            self.best_iteration = epoch
            self.no_improvement = 0
        else:
            self.no_improvement += 1

        degradation = (
            (
                (val - self.best_val) /
                max(self.best_val, 1e-12)
            )
            if epoch > self.best_iteration
            else 0
        )

        gap_overfit = (
            gap > CURRENT_GAP_THRESHOLD
        )

        degradation_overfit = (
            degradation > POST_BEST_THRESHOLD
        )

        overfit = (
            gap_overfit or
            degradation_overfit
        )

        self.history.append({
            "iteration": epoch + 1,
            "train_rmse": train,
            "validation_rmse": val,
            "relative_gap": gap,
            "best_iteration": self.best_iteration,
            "best_validation_rmse": self.best_val,
            "best_train_rmse": self.best_train,
            "post_best_degradation_ratio": degradation,
            "no_improvement_rounds": self.no_improvement,
            "current_gap_overfit": gap_overfit,
            "degradation_overfit": degradation_overfit,
            "overfit": overfit
        })

        if overfit:
            self.stopping_reason = "OVERFITTING"

            print(
                f"  OVERFIT STOP {epoch + 1} | "
                f"Best={self.best_iteration + 1} | "
                f"Gap={gap * 100:.2f}% | "
                f"Degradation={degradation * 100:.2f}%"
            )

            return True

        if (
            self.no_improvement >=
            EARLY_STOPPING_ROUNDS
        ):
            self.stopping_reason = (
                "EARLY_STOPPING"
            )

            print(
                f"  EARLY STOP {epoch + 1} | "
                f"Best={self.best_iteration + 1}"
            )

            return True

        return False

# METRİK
def calculate_metrics(
    y_true_log,
    pred_log
):

    y_true_usd = np.maximum(
        np.expm1(y_true_log),
        0
    )

    pred_usd = np.maximum(
        np.expm1(pred_log),
        0
    )

    return {
        "rmse_log": np.sqrt(
            mean_squared_error(
                y_true_log,
                pred_log
            )
        ),
        "mae_log": mean_absolute_error(
            y_true_log,
            pred_log
        ),
        "r2_log": r2_score(
            y_true_log,
            pred_log
        ),
        "rmse_usd": np.sqrt(
            mean_squared_error(
                y_true_usd,
                pred_usd
            )
        ),
        "mae_usd": mean_absolute_error(
            y_true_usd,
            pred_usd
        ),
        "r2_usd": r2_score(
            y_true_usd,
            pred_usd
        )
    }

# STRATIFIED K-FOLD
skf = StratifiedKFold(
    n_splits=K,
    shuffle=True,
    random_state=SEED
)

fold_models = []
fold_results = []
history_all = []

oof = np.empty(
    len(X_cv),
    dtype=np.float64
)

print("\n" + "=" * 80)
print(
    f"XGBOOST NON-OVERFIT + SAMPLE WEIGHTING "
    f"— STRATIFIED {K}-FOLD"
)
print("=" * 80)

print(
    f"Toplam CV veri  : {len(X_cv):,}"
)

print(
    f"Encoded feature : {X_cv.shape[1]}"
)

print(
    f"K               : {K}"
)

for fold, (
    train_idx,
    val_idx
) in enumerate(
    skf.split(
        X_cv,
        stratify_col
    ),
    1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{K}")
    print("=" * 70)

    print(
        f"Fold Train      : "
        f"{len(train_idx):,}"
    )

    print(
        f"Fold Validation : "
        f"{len(val_idx):,}"
    )

    X_train_fold = X_cv.iloc[
        train_idx
    ]

    X_val_fold = X_cv.iloc[
        val_idx
    ]

    y_train_fold = y_log[
        train_idx
    ]

    y_val_fold = y_log[
        val_idx
    ]

    # FOLD SAMPLE WEIGHTS
    fold_weights = sample_weights_cv[
        train_idx
    ]

    # VALIDATION AĞIRLIĞI 1.0
    validation_weights = np.ones(
        len(val_idx),
        dtype=np.float64
    )

    # D-MATRIX
    dtrain = xgb.DMatrix(
        X_train_fold,
        label=y_train_fold,
        weight=fold_weights
    )

    dval = xgb.DMatrix(
        X_val_fold,
        label=y_val_fold,
        weight=validation_weights
    )

    callback = OverfittingCallback()

    model = xgb.train(
        params=MODEL_PARAMS,
        dtrain=dtrain,
        num_boost_round=N_ESTIMATORS,
        evals=[
            (dtrain, "train"),
            (dval, "validation")
        ],
        callbacks=[callback],
        verbose_eval=False
    )

    history = pd.DataFrame(
        callback.history
    )

    history["fold"] = fold

    history["is_best_iteration"] = (
        history["iteration"] ==
        callback.best_iteration + 1
    )

    history_all.append(history)

    best_tree = (
        callback.best_iteration + 1
    )

    # TRAIN PREDICTION
    pred_train = model.predict(
        dtrain,
        iteration_range=(
            0,
            best_tree
        )
    )

    # VALIDATION PREDICTION
    pred_val = model.predict(
        dval,
        iteration_range=(
            0,
            best_tree
        )
    )

    # OOF
    oof[val_idx] = pred_val

    # METRICS
    train_metrics = calculate_metrics(
        y_train_fold,
        pred_train
    )

    val_metrics = calculate_metrics(
        y_val_fold,
        pred_val
    )

    # TRAIN / VALIDATION YÜZDELİKLER
    validation_rmse_vs_train_pct = (
        (
            val_metrics["rmse_usd"] -
            train_metrics["rmse_usd"]
        ) /
        max(
            train_metrics["rmse_usd"],
            1e-12
        )
    ) * 100

    validation_mae_vs_train_pct = (
        (
            val_metrics["mae_usd"] -
            train_metrics["mae_usd"]
        ) /
        max(
            train_metrics["mae_usd"],
            1e-12
        )
    ) * 100

    total_rmse = (
        train_metrics["rmse_usd"] +
        val_metrics["rmse_usd"]
    )

    train_rmse_pct = (
        train_metrics["rmse_usd"] /
        max(total_rmse, 1e-12)
    ) * 100

    validation_rmse_pct = (
        val_metrics["rmse_usd"] /
        max(total_rmse, 1e-12)
    ) * 100

    total_mae = (
        train_metrics["mae_usd"] +
        val_metrics["mae_usd"]
    )

    train_mae_pct = (
        train_metrics["mae_usd"] /
        max(total_mae, 1e-12)
    ) * 100

    validation_mae_pct = (
        val_metrics["mae_usd"] /
        max(total_mae, 1e-12)
    ) * 100

    train_r2_pct = (
        train_metrics["r2_usd"] *
        100
    )

    validation_r2_pct = (
        val_metrics["r2_usd"] *
        100
    )

    best_row = history.iloc[
        callback.best_iteration
    ]

    best_train = float(
        best_row["train_rmse"]
    )

    best_val = float(
        best_row["validation_rmse"]
    )

    best_gap = (
        (best_val - best_train) /
        max(best_train, 1e-12)
    )

    fold_results.append({

        "fold": fold,
        "train_size": len(train_idx),
        "validation_size": len(val_idx),
        "actual_training_rounds": len(history),
        "stopping_reason":
            callback.stopping_reason,
        "best_iteration":
            callback.best_iteration,
        "best_tree": best_tree,

        "weighted_train_samples":
            int(
                (
                    fold_weights >
                    WEIGHT_BASE
                ).sum()
            ),

        "train_rmse_log":
            train_metrics["rmse_log"],
        "train_mae_log":
            train_metrics["mae_log"],
        "train_r2_log":
            train_metrics["r2_log"],

        "validation_rmse_log":
            val_metrics["rmse_log"],
        "validation_mae_log":
            val_metrics["mae_log"],
        "validation_r2_log":
            val_metrics["r2_log"],

        "train_rmse_usd":
            train_metrics["rmse_usd"],
        "train_mae_usd":
            train_metrics["mae_usd"],
        "train_r2_usd":
            train_metrics["r2_usd"],

        "validation_rmse_usd":
            val_metrics["rmse_usd"],
        "validation_mae_usd":
            val_metrics["mae_usd"],
        "validation_r2_usd":
            val_metrics["r2_usd"],

        "validation_rmse_vs_train_pct":
            validation_rmse_vs_train_pct,

        "validation_mae_vs_train_pct":
            validation_mae_vs_train_pct,

        "train_r2_pct":
            train_r2_pct,

        "validation_r2_pct":
            validation_r2_pct,

        "train_rmse_pct":
            train_rmse_pct,

        "validation_rmse_pct":
            validation_rmse_pct,

        "train_mae_pct":
            train_mae_pct,

        "validation_mae_pct":
            validation_mae_pct,

        "best_train_rmse":
            best_train,

        "best_validation_rmse":
            best_val,

        "best_relative_gap":
            best_gap,

        "max_relative_gap":
            history["relative_gap"].max(),

        "max_post_best_degradation":
            history[
                "post_best_degradation_ratio"
            ].max(),

        "current_gap_overfit":
            bool(
                history[
                    "current_gap_overfit"
                ].any()
            ),

        "degradation_overfit":
            bool(
                history[
                    "degradation_overfit"
                ].any()
            ),

        "overfit":
            bool(
                history["overfit"].any()
            )
    })

    fold_models.append(model)

    print("\n" + "-" * 60)
    print("TRAIN PERFORMANCE")
    print("-" * 60)

    print(
        f"Train RMSE Log : "
        f"{train_metrics['rmse_log']:.6f}"
    )

    print(
        f"Train RMSE USD : "
        f"${train_metrics['rmse_usd']:,.2f}"
    )

    print(
        f"Train MAE USD  : "
        f"${train_metrics['mae_usd']:,.2f}"
    )

    print(
        f"Train R² USD   : "
        f"{train_metrics['r2_usd']:.6f}"
    )

    print(
        f"Train R²       : "
        f"{train_r2_pct:.2f}%"
    )

    print("\n" + "-" * 60)
    print("VALIDATION PERFORMANCE")
    print("-" * 60)

    print(
        f"Val RMSE Log   : "
        f"{val_metrics['rmse_log']:.6f}"
    )

    print(
        f"Val RMSE USD   : "
        f"${val_metrics['rmse_usd']:,.2f}"
    )

    print(
        f"Val MAE USD    : "
        f"${val_metrics['mae_usd']:,.2f}"
    )

    print(
        f"Val R² USD     : "
        f"{val_metrics['r2_usd']:.6f}"
    )

    print(
        f"Val R²         : "
        f"{validation_r2_pct:.2f}%"
    )

    print("\n" + "-" * 60)
    print("TRAIN / VALIDATION PERCENTAGE")
    print("-" * 60)

    print(
        f"Train RMSE %             : "
        f"{train_rmse_pct:.2f}%"
    )

    print(
        f"Validation RMSE %        : "
        f"{validation_rmse_pct:.2f}%"
    )

    print(
        f"Train MAE %              : "
        f"{train_mae_pct:.2f}%"
    )

    print(
        f"Validation MAE %         : "
        f"{validation_mae_pct:.2f}%"
    )

    print(
        f"Validation RMSE vs Train : "
        f"{validation_rmse_vs_train_pct:.2f}%"
    )

    print(
        f"Validation MAE vs Train  : "
        f"{validation_mae_vs_train_pct:.2f}%"
    )

    print("\n" + "-" * 60)

    print(
        f"Weighted samples : "
        f"{(fold_weights > WEIGHT_BASE).sum():,}"
    )

    print(
        f"Best Tree        : "
        f"{best_tree}"
    )

    print(
        f"Best Gap         : "
        f"{best_gap * 100:.2f}%"
    )

    print(
        f"Stop Reason      : "
        f"{callback.stopping_reason}"
    )

# DATAFRAME'LER
fold_df = pd.DataFrame(
    fold_results
)

history_df = pd.concat(
    history_all,
    ignore_index=True
)

# OOF METRICS
oof_metrics = calculate_metrics(
    y_log,
    oof
)

# EN İYİ FOLD
best_fold_idx = fold_df[
    "validation_rmse_log"
].idxmin()

best_fold = int(
    fold_df.loc[
        best_fold_idx,
        "fold"
    ]
)

best_fold_tree = int(
    fold_df.loc[
        best_fold_idx,
        "best_tree"
    ]
)

# MEDIAN BEST TREE
FINAL_BEST_TREE = int(
    round(
        fold_df["best_tree"].median()
    )
)

MEAN_BEST_TREE = float(
    fold_df["best_tree"].mean()
)

MEDIAN_BEST_TREE = float(
    fold_df["best_tree"].median()
)

print("\n" + "=" * 80)
print("FOLD TRAINING TAMAMLANDI")
print("=" * 80)

print(
    f"Mean Best Tree   : "
    f"{MEAN_BEST_TREE:.2f}"
)

print(
    f"Median Best Tree : "
    f"{MEDIAN_BEST_TREE:.2f}"
)

print(
    f"Final Best Tree  : "
    f"{FINAL_BEST_TREE}"
)

# FINAL MODEL SAMPLE WEIGHTS
# FINAL MODEL TRAIN + VALIDATION ÜZERİNDE
# TRAIN SAMPLE WEIGHTING KORUNUR
final_weights = sample_weights_cv

# TEK FINAL MODEL
print("\n" + "=" * 80)
print("TEK FINAL MODEL EĞİTİMİ")
print("=" * 80)

print(
    "Final training data : "
    "Train + Validation"
)

print(
    f"Final samples       : "
    f"{len(X_cv):,}"
)

print(
    f"Final features      : "
    f"{X_cv.shape[1]}"
)

print(
    f"Final tree count    : "
    f"{FINAL_BEST_TREE}"
)

print(
    f"Final weighted rows : "
    f"{(final_weights > WEIGHT_BASE).sum():,}"
)

dtrain_final = xgb.DMatrix(
    X_cv,
    label=y_log,
    weight=final_weights
)

final_model = xgb.train(
    params=MODEL_PARAMS,
    dtrain=dtrain_final,
    num_boost_round=FINAL_BEST_TREE,
    evals=[
        (dtrain_final, "train")
    ],
    verbose_eval=False
)

final_train_pred_log = final_model.predict(
    dtrain_final
)

final_train_metrics = calculate_metrics(
    y_log,
    final_train_pred_log
)

# FINAL MODEL KAYDET
FINAL_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_model.json"
)

final_model.save_model(
    FINAL_MODEL_PATH
)

print("\nFinal model kaydedildi:")
print(FINAL_MODEL_PATH)

# BEST FOLD
fold_models[
    best_fold - 1
].save_model(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_best_fold_model.json"
    )
)

# SUMMARY
summary = {

    "model_name": MODEL_NAME,
    "framework": "XGBoost",
    "fold_strategy": "StratifiedKFold",
    "n_splits": K,
    "total_cv_samples": len(X_cv),
    "encoded_features": X_cv.shape[1],

    "sample_weighting": {
        "enabled": True,
        "target_feature":
            "tender_selectionmethod",
        "target_category":
            "LOWEST_PRICE",
        "price_quantile":
            PRICE_QUANTILE,
        "price_threshold":
            float(price_threshold),
        "base_weight":
            WEIGHT_BASE,
        "high_price_weight":
            WEIGHT_LOWEST_PRICE_HIGH,
        "weighted_samples":
            int(
                high_lowest_price_mask.sum()
            )
    },

    "mean_best_tree":
        MEAN_BEST_TREE,

    "median_best_tree":
        MEDIAN_BEST_TREE,

    "final_best_tree":
        FINAL_BEST_TREE,

    "mean_validation_rmse_log":
        fold_df[
            "validation_rmse_log"
        ].mean(),

    "mean_validation_mae_log":
        fold_df[
            "validation_mae_log"
        ].mean(),

    "mean_validation_r2_log":
        fold_df[
            "validation_r2_log"
        ].mean(),

    "mean_validation_rmse_usd":
        fold_df[
            "validation_rmse_usd"
        ].mean(),

    "mean_validation_mae_usd":
        fold_df[
            "validation_mae_usd"
        ].mean(),

    "mean_validation_r2_usd":
        fold_df[
            "validation_r2_usd"
        ].mean(),

    "mean_validation_rmse_vs_train_pct":
        fold_df[
            "validation_rmse_vs_train_pct"
        ].mean(),

    "mean_validation_mae_vs_train_pct":
        fold_df[
            "validation_mae_vs_train_pct"
        ].mean(),

    "mean_train_r2_pct":
        fold_df[
            "train_r2_pct"
        ].mean(),

    "mean_validation_r2_pct":
        fold_df[
            "validation_r2_pct"
        ].mean(),

    "mean_train_rmse_pct":
        fold_df[
            "train_rmse_pct"
        ].mean(),

    "mean_validation_rmse_pct":
        fold_df[
            "validation_rmse_pct"
        ].mean(),

    "mean_train_mae_pct":
        fold_df[
            "train_mae_pct"
        ].mean(),

    "mean_validation_mae_pct":
        fold_df[
            "validation_mae_pct"
        ].mean(),

    "oof_metrics":
        oof_metrics,

    "mean_best_gap":
        fold_df[
            "best_relative_gap"
        ].mean(),

    "mean_max_gap":
        fold_df[
            "max_relative_gap"
        ].mean(),

    "max_gap":
        fold_df[
            "max_relative_gap"
        ].max(),

    "overfit_folds":
        int(
            fold_df["overfit"].sum()
        ),

    "overfit_ratio":
        fold_df["overfit"].mean(),

    "final_train_metrics":
        final_train_metrics,

    "best_fold":
        best_fold,

    "best_fold_tree":
        best_fold_tree,

    "final_model": {
        "file":
            "xgboost_final_model.json",
        "training_data":
            "Train + Validation",
        "samples":
            len(X_cv),
        "features":
            X_cv.shape[1],
        "num_boost_round":
            FINAL_BEST_TREE,
        "tree_selection":
            "median_best_tree",
        "sample_weighting":
            True
    },

    "stopping_configuration": {
        "current_gap_threshold":
            CURRENT_GAP_THRESHOLD,
        "post_best_threshold":
            POST_BEST_THRESHOLD,
        "early_stopping_rounds":
            EARLY_STOPPING_ROUNDS,
        "max_estimators":
            N_ESTIMATORS
    },

    "best_params":
        BEST_PARAMS,

    "model_params":
        MODEL_PARAMS
}

# OOF DATAFRAME
oof_df = pd.DataFrame({

    "index":
        np.arange(len(X_cv)),

    "actual_log":
        y_log,

    "predicted_log":
        oof,

    "actual_usd":
        y_usd,

    "predicted_usd":
        np.maximum(
            np.expm1(oof),
            0
        )
})

oof_df["residual_usd"] = (
    oof_df["actual_usd"] -
    oof_df["predicted_usd"]
)

oof_df["absolute_error_usd"] = (
    oof_df["residual_usd"].abs()
)

# FOLD CSV
fold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_metrics.csv"
    ),
    index=False
)

# HISTORY CSV
history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_training_history.csv"
    ),
    index=False
)

# OOF CSV
oof_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_oof_predictions.csv"
    ),
    index=False
)

# SUMMARY JSON
with open(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_9fold_complete_results.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4,
        ensure_ascii=False
    )

# TÜM FOLD MODELLERİ
for i, model in enumerate(
    fold_models,
    1
):

    model.save_model(
        os.path.join(
            OUTPUT_DIR,
            f"xgboost_fold_{i}.json"
        )
    )

# FOLD RESULTS JSON
fold_records = (
    fold_df
    .replace({np.nan: None})
    .to_dict(
        orient="records"
    )
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "fold_results_for_wandb.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        fold_records,
        f,
        indent=4,
        ensure_ascii=False
    )

# TRAINING HISTORY JSON
history_records = (
    history_df
    .replace({np.nan: None})
    .to_dict(
        orient="records"
    )
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "training_history_for_wandb.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history_records,
        f,
        indent=4,
        ensure_ascii=False
    )

# W&B PARAMETERS
wandb_parameters = {

    "model_name":
        MODEL_NAME,

    "framework":
        "XGBoost",

    "seed":
        SEED,

    "k":
        K,

    "n_estimators":
        N_ESTIMATORS,

    "early_stopping_rounds":
        EARLY_STOPPING_ROUNDS,

    "current_gap_threshold":
        CURRENT_GAP_THRESHOLD,

    "post_best_threshold":
        POST_BEST_THRESHOLD,

    "target_is_log":
        TARGET_IS_LOG,

    "sample_weighting": {
        "enabled":
            True,
        "feature":
            "tender_selectionmethod",
        "category":
            "LOWEST_PRICE",
        "price_quantile":
            PRICE_QUANTILE,
        "price_threshold":
            float(price_threshold),
        "base_weight":
            WEIGHT_BASE,
        "high_price_weight":
            WEIGHT_LOWEST_PRICE_HIGH
    },

    "best_params":
        BEST_PARAMS,

    "model_params":
        MODEL_PARAMS,

    "final_model": {
        "filename":
            "xgboost_final_model.json",
        "train_data":
            "Train + Validation",
        "n_samples":
            len(X_cv),
        "n_features":
            X_cv.shape[1],
        "final_best_tree":
            FINAL_BEST_TREE,
        "tree_selection":
            "median_best_tree"
    }
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "wandb_parameters.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        wandb_parameters,
        f,
        indent=4,
        ensure_ascii=False
    )

# TRAIN LOSS VS VALIDATION LOSS
plt.figure(
    figsize=(14, 7)
)

for fold in range(
    1,
    K + 1
):

    h = history_df[
        history_df["fold"] == fold
    ]

    plt.plot(
        h["iteration"],
        h["train_rmse"],
        alpha=0.30
    )

    plt.plot(
        h["iteration"],
        h["validation_rmse"],
        alpha=0.65,
        linestyle="--",
        label=f"Fold {fold}"
    )

plt.xlabel(
    "Boosting Round"
)

plt.ylabel(
    "RMSE Loss (Log)"
)

plt.title(
    f"{K}-Fold Train Loss vs Validation Loss "
    "(Sample Weighted)"
)

plt.grid(
    alpha=0.3
)

plt.legend(
    ncol=3
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "train_validation_loss.png"
    ),
    dpi=300
)

plt.show()

# 9-FOLD ORTALAMA
mean_history = (
    history_df
    .groupby("iteration")[
        [
            "train_rmse",
            "validation_rmse"
        ]
    ]
    .mean()
    .reset_index()
)

plt.figure(
    figsize=(14, 7)
)

plt.plot(
    mean_history["iteration"],
    mean_history["train_rmse"],
    linewidth=3,
    label="Mean Train Loss"
)

plt.plot(
    mean_history["iteration"],
    mean_history["validation_rmse"],
    linewidth=3,
    linestyle="--",
    label="Mean Validation Loss"
)

plt.xlabel(
    "Boosting Round"
)

plt.ylabel(
    "RMSE Loss (Log)"
)

plt.title(
    f"{K}-Fold Mean Train Loss vs Validation Loss "
    "(Sample Weighted)"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "mean_train_validation_loss.png"
    ),
    dpi=300
)

plt.show()

# RELATIVE GAP
plt.figure(
    figsize=(14, 7)
)

for fold in range(
    1,
    K + 1
):

    h = history_df[
        history_df["fold"] == fold
    ]

    plt.plot(
        h["iteration"],
        h["relative_gap"] * 100,
        alpha=0.5
    )

plt.axhline(
    CURRENT_GAP_THRESHOLD * 100,
    linestyle="--",
    label="20% Threshold"
)

plt.xlabel(
    "Boosting Round"
)

plt.ylabel(
    "Train-Validation Gap (%)"
)

plt.title(
    "Current Relative Gap "
    "(Sample Weighted)"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "relative_gap.png"
    ),
    dpi=300
)

plt.show()

# FINAL SUMMARY
print("\n" + "=" * 80)
print(
    "XGBOOST NON-OVERFIT + SAMPLE WEIGHTING SUMMARY"
)
print("=" * 80)

print(
    f"K                : {K}"
)

print(
    f"Total CV Samples : {len(X_cv):,}"
)

print(
    f"Encoded Features : {X_cv.shape[1]}"
)

print(
    f"Max Estimators   : {N_ESTIMATORS}"
)

print(
    f"Early Stopping   : {EARLY_STOPPING_ROUNDS}"
)

print(
    f"Gap Threshold    : "
    f"{CURRENT_GAP_THRESHOLD * 100:.1f}%"
)

print("\n" + "-" * 80)
print("SAMPLE WEIGHTING")
print("-" * 80)

print(
    "Feature          : "
    "tender_selectionmethod"
)

print(
    "Category         : "
    "LOWEST_PRICE"
)

print(
    f"Price Quantile   : "
    f"{PRICE_QUANTILE:.2f}"
)

print(
    f"Price Threshold  : "
    f"{price_threshold:.6f}"
)

print(
    f"Normal Weight    : "
    f"{WEIGHT_BASE:.1f}"
)

print(
    f"High Price Weight: "
    f"{WEIGHT_LOWEST_PRICE_HIGH:.1f}"
)

print(
    f"Weighted Samples : "
    f"{high_lowest_price_mask.sum():,}"
)

# TREE
print("\n" + "-" * 80)
print("TREE INFORMATION")
print("-" * 80)

print(
    f"Mean Best Tree   : "
    f"{summary['mean_best_tree']:.2f}"
)

print(
    f"Median Best Tree : "
    f"{summary['median_best_tree']:.2f}"
)

print(
    f"FINAL Best Tree  : "
    f"{summary['final_best_tree']}"
)

# VALIDATION
print("\n" + "-" * 80)
print("VALIDATION PERFORMANCE")
print("-" * 80)

print(
    f"Mean Validation RMSE Log : "
    f"{summary['mean_validation_rmse_log']:.6f}"
)

print(
    f"Mean Validation MAE Log  : "
    f"{summary['mean_validation_mae_log']:.6f}"
)

print(
    f"Mean Validation R² Log   : "
    f"{summary['mean_validation_r2_log']:.6f}"
)

print(
    f"Mean Validation RMSE USD : "
    f"${summary['mean_validation_rmse_usd']:,.2f}"
)

print(
    f"Mean Validation MAE USD  : "
    f"${summary['mean_validation_mae_usd']:,.2f}"
)

print(
    f"Mean Validation R² USD   : "
    f"{summary['mean_validation_r2_usd']:.6f}"
)

# TRAIN / VALIDATION
print("\n" + "-" * 80)
print("TRAIN / VALIDATION PERCENTAGE ANALYSIS")
print("-" * 80)

print(
    f"Mean Train RMSE %        : "
    f"{summary['mean_train_rmse_pct']:.2f}%"
)

print(
    f"Mean Validation RMSE %   : "
    f"{summary['mean_validation_rmse_pct']:.2f}%"
)

print(
    f"Mean Train MAE %         : "
    f"{summary['mean_train_mae_pct']:.2f}%"
)

print(
    f"Mean Validation MAE %    : "
    f"{summary['mean_validation_mae_pct']:.2f}%"
)

print(
    f"Mean Train R² %          : "
    f"{summary['mean_train_r2_pct']:.2f}%"
)

print(
    f"Mean Validation R² %     : "
    f"{summary['mean_validation_r2_pct']:.2f}%"
)

# GENERALIZATION
print("\n" + "-" * 80)
print("TRAIN → VALIDATION GENERALIZATION GAP")
print("-" * 80)

print(
    f"Validation RMSE vs Train : "
    f"{summary['mean_validation_rmse_vs_train_pct']:.2f}%"
)

print(
    f"Validation MAE vs Train  : "
    f"{summary['mean_validation_mae_vs_train_pct']:.2f}%"
)

# OOF
print("\n" + "-" * 80)
print("OOF PERFORMANCE")
print("-" * 80)

print(
    f"OOF RMSE Log     : "
    f"{oof_metrics['rmse_log']:.6f}"
)

print(
    f"OOF MAE Log      : "
    f"{oof_metrics['mae_log']:.6f}"
)

print(
    f"OOF R² Log       : "
    f"{oof_metrics['r2_log']:.6f}"
)

print(
    f"OOF RMSE USD     : "
    f"${oof_metrics['rmse_usd']:,.2f}"
)

print(
    f"OOF MAE USD      : "
    f"${oof_metrics['mae_usd']:,.2f}"
)

print(
    f"OOF R² USD       : "
    f"{oof_metrics['r2_usd']:.6f}"
)

# FINAL TRAIN
print("\n" + "-" * 80)
print("FINAL MODEL TRAIN PERFORMANCE")
print("-" * 80)

print(
    f"Final Train RMSE : "
    f"${final_train_metrics['rmse_usd']:,.2f}"
)

print(
    f"Final Train MAE  : "
    f"${final_train_metrics['mae_usd']:,.2f}"
)

print(
    f"Final Train R²   : "
    f"{final_train_metrics['r2_usd']:.6f}"
)

# OVERFITTING
print("\n" + "-" * 80)
print("OVERFITTING ANALYSIS")
print("-" * 80)

print(
    f"Mean Best Gap    : "
    f"{summary['mean_best_gap'] * 100:.2f}%"
)

print(
    f"Mean Max Gap     : "
    f"{summary['mean_max_gap'] * 100:.2f}%"
)

print(
    f"Global Max Gap   : "
    f"{summary['max_gap'] * 100:.2f}%"
)

print(
    f"Overfit Folds    : "
    f"{summary['overfit_folds']}/{K}"
)

print(
    f"Overfit Ratio    : "
    f"{summary['overfit_ratio'] * 100:.2f}%"
)

# BEST FOLD
print("\n" + "-" * 80)
print("BEST FOLD")
print("-" * 80)

print(
    f"Best Fold        : "
    f"{best_fold}"
)

print(
    f"Best Fold Tree   : "
    f"{best_fold_tree}"
)

# DOSYALAR
print("\n" + "=" * 80)
print("KAYDEDİLEN MODEL DOSYALARI")
print("=" * 80)

for i in range(
    1,
    K + 1
):

    print(
        os.path.join(
            OUTPUT_DIR,
            f"xgboost_fold_{i}.json"
        )
    )

print(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_best_fold_model.json"
    )
)

print(
    os.path.join(
        OUTPUT_DIR,
        "xgboost_final_model.json"
    )
)

# W&B
print("\n" + "=" * 80)
print("W&B AKTARIM DEĞİŞKENLERİ HAZIR")
print("=" * 80)

print(
    "fold_df          -> "
    "Fold train/validation sonuçları"
)

print(
    "history_df       -> "
    "Her boosting round train/validation loss"
)

print(
    "oof_df           -> "
    "OOF validation tahminleri"
)

print(
    "fold_models      -> "
    "Eğitilmiş 9 fold modeli"
)

print(
    "summary          -> "
    "Genel sonuçlar"
)

print(
    "BEST_PARAMS      -> "
    "Best hyperparameters"
)

print(
    "MODEL_PARAMS     -> "
    "XGBoost model parameters"
)

print(
    "final_model      -> "
    "Train + Validation üzerinde "
    "eğitilmiş tek final model"
)

print(
    "FINAL_BEST_TREE  -> "
    "Median best tree"
)

print(
    "Sample Weighting -> "
    "LOWEST_PRICE + yüksek fiyat = 2.0"
)

print("\n" + "=" * 80)

print(
    f"Output Directory: "
    f"{OUTPUT_DIR}"
)

print("=" * 80)

In [ ]:
# XGBOOST SAMPLE WEIGHTED — TEST
# EĞİTİLMİŞ FINAL MODEL İLE TEST DEĞERLENDİRMESİ

import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

MODEL_DIR = "xgboost_non_over_fit_sample_weighted"
MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_final_model.json"
)

# ------------------------------------------------------------
# KONTROL
# ------------------------------------------------------------

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Final model bulunamadı: {MODEL_PATH}"
    )

if not isinstance(X_test_encoded, pd.DataFrame):
    X_test_encoded = pd.DataFrame(X_test_encoded)

y_test_np = np.asarray(
    y_test,
    dtype=np.float64
)

# ------------------------------------------------------------
# FINAL MODEL
# ------------------------------------------------------------

model = xgb.Booster()

model.load_model(
    MODEL_PATH
)

print("\n" + "=" * 80)
print("XGBOOST SAMPLE WEIGHTED — TEST")
print("=" * 80)

print(
    f"Test Samples   : {len(X_test_encoded):,}"
)

print(
    f"Test Features  : {X_test_encoded.shape[1]}"
)

print(
    f"Model          : {MODEL_PATH}"
)

# ------------------------------------------------------------
# TEST D-MATRIX
# TEST'TE SAMPLE WEIGHT KULLANILMAZ
# ------------------------------------------------------------

dtest = xgb.DMatrix(
    X_test_encoded
)

# ------------------------------------------------------------
# TEST PREDICTION
# ------------------------------------------------------------

pred_test_log = model.predict(
    dtest
)

# ------------------------------------------------------------
# LOG → USD
# ------------------------------------------------------------

actual_test_usd = np.maximum(
    np.expm1(y_test_np),
    0
)

pred_test_usd = np.maximum(
    np.expm1(pred_test_log),
    0
)

# ------------------------------------------------------------
# LOG METRICS
# ------------------------------------------------------------

test_rmse_log = np.sqrt(
    mean_squared_error(
        y_test_np,
        pred_test_log
    )
)

test_mae_log = mean_absolute_error(
    y_test_np,
    pred_test_log
)

test_r2_log = r2_score(
    y_test_np,
    pred_test_log
)

# ------------------------------------------------------------
# USD METRICS
# ------------------------------------------------------------

test_rmse_usd = np.sqrt(
    mean_squared_error(
        actual_test_usd,
        pred_test_usd
    )
)

test_mae_usd = mean_absolute_error(
    actual_test_usd,
    pred_test_usd
)

test_r2_usd = r2_score(
    actual_test_usd,
    pred_test_usd
)

# ------------------------------------------------------------
# RESIDUALS
# ------------------------------------------------------------

residual_usd = (
    actual_test_usd -
    pred_test_usd
)

absolute_error_usd = (
    np.abs(residual_usd)
)

# ------------------------------------------------------------
# SONUÇLAR
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("TEST PERFORMANCE — LOG SPACE")
print("-" * 80)

print(
    f"Test RMSE Log : "
    f"{test_rmse_log:.6f}"
)

print(
    f"Test MAE Log  : "
    f"{test_mae_log:.6f}"
)

print(
    f"Test R² Log   : "
    f"{test_r2_log:.6f}"
)

print("\n" + "-" * 80)
print("TEST PERFORMANCE — USD SPACE")
print("-" * 80)

print(
    f"Test RMSE USD : "
    f"${test_rmse_usd:,.2f}"
)

print(
    f"Test MAE USD  : "
    f"${test_mae_usd:,.2f}"
)

print(
    f"Test R² USD   : "
    f"{test_r2_usd:.6f}"
)

# ------------------------------------------------------------
# RESIDUAL SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("TEST RESIDUAL SUMMARY")
print("-" * 80)

print(
    f"Mean Residual : "
    f"${residual_usd.mean():,.2f}"
)

print(
    f"Median Residual : "
    f"${np.median(residual_usd):,.2f}"
)

print(
    f"MAE           : "
    f"${absolute_error_usd.mean():,.2f}"
)

print(
    f"Residual P99  : "
    f"${np.percentile(absolute_error_usd, 99):,.2f}"
)

# ------------------------------------------------------------
# TEST OOF-LIKE DATAFRAME
# ------------------------------------------------------------

test_df = pd.DataFrame({

    "index":
        np.arange(len(X_test_encoded)),

    "actual_log":
        y_test_np,

    "predicted_log":
        pred_test_log,

    "actual_usd":
        actual_test_usd,

    "predicted_usd":
        pred_test_usd,

    "residual_usd":
        residual_usd,

    "absolute_error_usd":
        absolute_error_usd
})

# ------------------------------------------------------------
# METRICS JSON
# ------------------------------------------------------------

test_metrics = {

    "model":
        "XGBoost_NonOverfit_SampleWeighted",

    "model_file":
        "xgboost_final_model.json",

    "test_samples":
        len(X_test_encoded),

    "test_features":
        X_test_encoded.shape[1],

    "sample_weighting":
        "Training only",

    "log_space": {

        "rmse":
            float(test_rmse_log),

        "mae":
            float(test_mae_log),

        "r2":
            float(test_r2_log)
    },

    "usd_space": {

        "rmse":
            float(test_rmse_usd),

        "mae":
            float(test_mae_usd),

        "r2":
            float(test_r2_usd)
    },

    "residual": {

        "mean":
            float(residual_usd.mean()),

        "median":
            float(np.median(residual_usd)),

        "mae":
            float(absolute_error_usd.mean()),

        "p99":
            float(np.percentile(
                absolute_error_usd,
                99
            ))
    }
}

# ------------------------------------------------------------
# KAYDET
# ------------------------------------------------------------

test_df.to_csv(
    os.path.join(
        MODEL_DIR,
        "xgboost_test_predictions.csv"
    ),
    index=False
)

with open(
    os.path.join(
        MODEL_DIR,
        "xgboost_test_metrics.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_metrics,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\n" + "=" * 80)
print("TEST DOSYALARI KAYDEDİLDİ")
print("=" * 80)

print(
    os.path.join(
        MODEL_DIR,
        "xgboost_test_predictions.csv"
    )
)

print(
    os.path.join(
        MODEL_DIR,
        "xgboost_test_metrics.json"
    )
)

print("\n" + "=" * 80)
print("TEST TAMAMLANDI")
print("=" * 80)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASELINE_DIR = "xgboost_non_over_fit_final"
WEIGHTED_DIR = "xgboost_non_over_fit_sample_weighted"

BASELINE_OOF = os.path.join(
    BASELINE_DIR,
    "xgboost_9fold_oof_predictions.csv"
)

WEIGHTED_OOF = os.path.join(
    WEIGHTED_DIR,
    "xgboost_9fold_oof_predictions.csv"
)

# OOF
base = pd.read_csv(BASELINE_OOF)
weighted = pd.read_csv(WEIGHTED_OOF)

if len(base) != len(weighted):
    raise ValueError(
        f"OOF uzunlukları farklı: "
        f"{len(base):,} vs {len(weighted):,}"
    )

if not np.array_equal(
    base["index"].to_numpy(),
    weighted["index"].to_numpy()
):
    raise ValueError(
        "İki OOF dosyasının index sırası aynı değil."
    )

# GERÇEK DEĞER
actual = base["actual_usd"].to_numpy(dtype=float)

pred_base = base["predicted_usd"].to_numpy(dtype=float)
pred_weighted = weighted["predicted_usd"].to_numpy(dtype=float)

# RESIDUAL
res_base = actual - pred_base
res_weighted = actual - pred_weighted

# ------------------------------------------------------------
# METRİK FONKSİYONU
# ------------------------------------------------------------

def metrics(y_true, pred):
    residual = y_true - pred

    return {
        "RMSE": np.sqrt(
            mean_squared_error(y_true, pred)
        ),
        "MAE": mean_absolute_error(
            y_true, pred
        ),
        "R2": r2_score(
            y_true, pred
        ),
        "Mean Residual": residual.mean(),
        "Median Residual": np.median(residual),
        "Underprediction %": (
            residual > 0
        ).mean() * 100,
        "Overprediction %": (
            residual < 0
        ).mean() * 100,
        "P90 Absolute Error": np.percentile(
            np.abs(residual), 90
        ),
        "P99 Absolute Error": np.percentile(
            np.abs(residual), 99
        )
    }

# ------------------------------------------------------------
# 1. GENEL KARŞILAŞTIRMA
# ------------------------------------------------------------

m_base = metrics(
    actual,
    pred_base
)

m_weighted = metrics(
    actual,
    pred_weighted
)

comparison = pd.DataFrame(
    {
        "Baseline": m_base,
        "Sample Weighted": m_weighted
    }
)

comparison["Change"] = (
    comparison["Sample Weighted"]
    - comparison["Baseline"]
)

print("\n" + "=" * 90)
print("SAMPLE WEIGHTING — GENEL MODEL KARŞILAŞTIRMASI")
print("=" * 90)

print(comparison.to_string(float_format=lambda x: f"{x:,.4f}"))

# ------------------------------------------------------------
# 2. VERİYİ ORİJİNAL TRAIN'DEN AL
# ------------------------------------------------------------

selection = (
    X_train["tender_selectionmethod"]
    .cast(str)
    .to_numpy()
)

if len(selection) != len(X_train):
    raise ValueError(
        "X_train uzunluğu beklenenden farklı."
    )

# X_cv = Train + Validation
selection_cv = np.concatenate(
    [
        selection,
        np.asarray(
            X_val["tender_selectionmethod"]
            .cast(str)
            .to_numpy()
        )
    ]
)

# TRAIN TARGET'TAN THRESHOLD
y_train_np = np.asarray(
    y_train,
    dtype=np.float64
)

price_threshold = np.quantile(
    y_train_np,
    0.75
)

# CV target
y_cv_np = np.asarray(
    pd.concat(
        [
            pd.Series(y_train),
            pd.Series(y_val)
        ],
        ignore_index=True
    ),
    dtype=np.float64
)

# ------------------------------------------------------------
# 3. LOWEST_PRICE + HIGH PRICE
# ------------------------------------------------------------

lowest_price = (
    selection_cv == "LOWEST_PRICE"
)

high_price = (
    y_cv_np >= price_threshold
)

target_group = (
    lowest_price &
    high_price
)

lowest_group = lowest_price

print("\n" + "=" * 90)
print("SAMPLE WEIGHTING HEDEF GRUBU")
print("=" * 90)

print(
    f"Price threshold : "
    f"{price_threshold:,.6f}"
)

print(
    f"LOWEST_PRICE toplam : "
    f"{lowest_group.sum():,}"
)

print(
    f"LOWEST_PRICE + High Price : "
    f"{target_group.sum():,}"
)

print(
    f"Hedef grup oranı : "
    f"{target_group.mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 4. ALT GRUP ANALİZİ
# ------------------------------------------------------------

def subgroup_result(
    name,
    mask
):
    y = actual[mask]
    pb = pred_base[mask]
    pw = pred_weighted[mask]

    mb = metrics(y, pb)
    mw = metrics(y, pw)

    return {
        "Group": name,
        "N": len(y),

        "Baseline RMSE":
            mb["RMSE"],

        "Weighted RMSE":
            mw["RMSE"],

        "RMSE Change %":
            (
                (mw["RMSE"] - mb["RMSE"])
                / max(mb["RMSE"], 1e-12)
            ) * 100,

        "Baseline MAE":
            mb["MAE"],

        "Weighted MAE":
            mw["MAE"],

        "MAE Change %":
            (
                (mw["MAE"] - mb["MAE"])
                / max(mb["MAE"], 1e-12)
            ) * 100,

        "Baseline R2":
            mb["R2"],

        "Weighted R2":
            mw["R2"],

        "R2 Change":
            mw["R2"] - mb["R2"],

        "Baseline Mean Residual":
            mb["Mean Residual"],

        "Weighted Mean Residual":
            mw["Mean Residual"],

        "Bias Change":
            mw["Mean Residual"]
            - mb["Mean Residual"],

        "Baseline Underprediction %":
            mb["Underprediction %"],

        "Weighted Underprediction %":
            mw["Underprediction %"],

        "Underprediction Change":
            mw["Underprediction %"]
            - mb["Underprediction %"]
    }

groups = [

    (
        "ALL",
        np.ones(
            len(actual),
            dtype=bool
        )
    ),

    (
        "LOWEST_PRICE",
        lowest_group
    ),

    (
        "LOWEST_PRICE + HIGH PRICE",
        target_group
    ),

    (
        "OTHER",
        ~target_group
    )
]

results = []

for name, mask in groups:
    results.append(
        subgroup_result(
            name,
            mask
        )
    )

group_df = pd.DataFrame(
    results
)

print("\n" + "=" * 90)
print("ALT GRUP KARŞILAŞTIRMASI")
print("=" * 90)

print(
    group_df.to_string(
        index=False,
        float_format=lambda x: f"{x:,.4f}"
    )
)

# ------------------------------------------------------------
# 5. BIAS ANALİZİ
# ------------------------------------------------------------

target_base_bias = (
    res_base[target_group].mean()
)

target_weighted_bias = (
    res_weighted[target_group].mean()
)

bias_reduction = (
    abs(target_base_bias)
    - abs(target_weighted_bias)
)

bias_reduction_pct = (
    bias_reduction
    / max(abs(target_base_bias), 1e-12)
) * 100

print("\n" + "=" * 90)
print("HEDEF GRUP — BIAS ANALİZİ")
print("=" * 90)

print(
    f"Baseline Mean Residual : "
    f"${target_base_bias:,.2f}"
)

print(
    f"Weighted Mean Residual : "
    f"${target_weighted_bias:,.2f}"
)

print(
    f"Bias Reduction         : "
    f"${bias_reduction:,.2f}"
)

print(
    f"Bias Reduction %       : "
    f"{bias_reduction_pct:.2f}%"
)

# ------------------------------------------------------------
# 6. UNDER-PREDICTION ANALİZİ
# ------------------------------------------------------------

base_under = (
    res_base[target_group] > 0
).mean() * 100

weighted_under = (
    res_weighted[target_group] > 0
).mean() * 100

print("\n" + "=" * 90)
print("HEDEF GRUP — UNDER-PREDICTION")
print("=" * 90)

print(
    f"Baseline Underprediction : "
    f"{base_under:.2f}%"
)

print(
    f"Weighted Underprediction : "
    f"{weighted_under:.2f}%"
)

print(
    f"Change                   : "
    f"{weighted_under - base_under:+.2f} percentage points"
)

# ------------------------------------------------------------
# 7. KAÇ GÖZLEM İYİLEŞTİ?
# ------------------------------------------------------------

abs_error_base = np.abs(
    res_base[target_group]
)

abs_error_weighted = np.abs(
    res_weighted[target_group]
)

improved = (
    abs_error_weighted
    < abs_error_base
)

worsened = (
    abs_error_weighted
    > abs_error_base
)

same = (
    abs_error_weighted
    == abs_error_base
)

print("\n" + "=" * 90)
print("HEDEF GRUP — GÖZLEM BAZLI DEĞİŞİM")
print("=" * 90)

print(
    f"İyileşen gözlem : "
    f"{improved.sum():,} "
    f"({improved.mean() * 100:.2f}%)"
)

print(
    f"Kötüleşen gözlem: "
    f"{worsened.sum():,} "
    f"({worsened.mean() * 100:.2f}%)"
)

print(
    f"Aynı kalan      : "
    f"{same.sum():,} "
    f"({same.mean() * 100:.2f}%)"
)

# ------------------------------------------------------------
# 8. FİNAL KARAR
# ------------------------------------------------------------

target_rmse_base = m_base["RMSE"]
target_rmse_weighted = m_weighted["RMSE"]

sub_rmse_base = (
    group_df.loc[
        group_df["Group"]
        == "LOWEST_PRICE + HIGH PRICE",
        "Baseline RMSE"
    ].iloc[0]
)

sub_rmse_weighted = (
    group_df.loc[
        group_df["Group"]
        == "LOWEST_PRICE + HIGH PRICE",
        "Weighted RMSE"
    ].iloc[0]
)

print("\n" + "=" * 90)
print("SAMPLE WEIGHTING SONUÇ")
print("=" * 90)

if (
    sub_rmse_weighted < sub_rmse_base
    and abs(target_weighted_bias)
    < abs(target_base_bias)
):
    print(
        "SONUÇ: SAMPLE WEIGHTING BAŞARILI"
    )
    print(
        "Hedef gruptaki hata ve bias azalmış."
    )

elif (
    sub_rmse_weighted < sub_rmse_base
):
    print(
        "SONUÇ: KISMEN BAŞARILI"
    )
    print(
        "Hedef grubun RMSE'si azalmış fakat "
        "bias tamamen düzelmemiş."
    )

else:
    print(
        "SONUÇ: SAMPLE WEIGHTING KATKI SAĞLAMAMIŞ"
    )
    print(
        "Hedef gruptaki RMSE iyileşmemiş."
    )

# ------------------------------------------------------------
# 9. SONUÇLARI KAYDET
# ------------------------------------------------------------

comparison.to_csv(
    os.path.join(
        WEIGHTED_DIR,
        "sample_weighting_overall_comparison.csv"
    )
)

group_df.to_csv(
    os.path.join(
        WEIGHTED_DIR,
        "sample_weighting_subgroup_comparison.csv"
    ),
    index=False
)

print("\nSonuçlar kaydedildi:")
print(
    os.path.join(
        WEIGHTED_DIR,
        "sample_weighting_overall_comparison.csv"
    )
)

print(
    os.path.join(
        WEIGHTED_DIR,
        "sample_weighting_subgroup_comparison.csv"
    )
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASELINE_DIR = "xgboost_non_over_fit_final"
WEIGHTED_DIR = "xgboost_non_over_fit_sample_weighted"

BASELINE_OOF = os.path.join(
    BASELINE_DIR,
    "xgboost_9fold_oof_predictions.csv"
)

WEIGHTED_OOF = os.path.join(
    WEIGHTED_DIR,
    "xgboost_9fold_oof_predictions.csv"
)

OUTPUT_DIR = os.path.join(
    WEIGHTED_DIR,
    "sample_weighting_pages"
)

OUTPUT_COLUMN_CSV = os.path.join(
    WEIGHTED_DIR,
    "sample_weighting_column_analysis.csv"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

N_BINS = 10
MIN_GROUP_SIZE = 100

COLUMN_ANALYSIS = [
    "tender_supplytype",
    "tender_nationalproceduretype",
    "lot_status",
    "tender_isframeworkagreement",
    "tender_selectionmethod",
    "buyer_mainactivities",
    "competition_level",
    "log_tender_lotscount",
    "log_tender_recordedbidscount",
    "tender_awardcriteria_count",
    "log_submission_period",
    "log_decision_period",
    "corr_proc",
    "corr_subm",
    "corr_nocft",
    "corr_decp",
    "corr_singleb",
    "corr_buyer_concentration"
]

BASE_COLOR = "#4285F4"
WEIGHTED_COLOR = "#FF7F0E"
GOOD_COLOR = "#2CA02C"
BAD_COLOR = "#E74C3C"

if not os.path.exists(BASELINE_OOF):
    raise FileNotFoundError(BASELINE_OOF)

if not os.path.exists(WEIGHTED_OOF):
    raise FileNotFoundError(WEIGHTED_OOF)

base = pd.read_csv(BASELINE_OOF)
weighted = pd.read_csv(WEIGHTED_OOF)

if len(base) != len(weighted):
    raise ValueError("Baseline ve Weighted OOF uzunlukları farklı.")

if not np.array_equal(
    base["index"].to_numpy(),
    weighted["index"].to_numpy()
):
    raise ValueError(
        "Baseline ve Weighted OOF index sırası aynı değil."
    )

actual = base["actual_usd"].to_numpy(float)
pred_base = base["predicted_usd"].to_numpy(float)
pred_weighted = weighted["predicted_usd"].to_numpy(float)

res_base = actual - pred_base
res_weighted = actual - pred_weighted

abs_error_base = np.abs(res_base)
abs_error_weighted = np.abs(res_weighted)


def metrics(y, p):
    r = y - p

    return {
        "RMSE": float(
            np.sqrt(mean_squared_error(y, p))
        ),
        "MAE": float(
            mean_absolute_error(y, p)
        ),
        "R2": float(
            r2_score(y, p)
        ),
        "Mean Residual": float(
            r.mean()
        ),
        "Median Residual": float(
            np.median(r)
        ),
        "Underprediction %": float(
            (r > 0).mean() * 100
        ),
        "Overprediction %": float(
            (r < 0).mean() * 100
        ),
        "P90": float(
            np.percentile(np.abs(r), 90)
        ),
        "P99": float(
            np.percentile(np.abs(r), 99)
        )
    }


m_base = metrics(actual, pred_base)
m_weighted = metrics(actual, pred_weighted)


def to_pd(df):
    if isinstance(df, pd.DataFrame):
        return df.copy()

    if hasattr(df, "to_pandas"):
        return df.to_pandas()

    return pd.DataFrame(df)


feature_cv = pd.concat(
    [
        to_pd(X_train),
        to_pd(X_val)
    ],
    ignore_index=True
)

if "idx_train" in globals() and "idx_val" in globals():
    cv_index = np.concatenate([
        np.asarray(idx_train),
        np.asarray(idx_val)
    ])
else:
    cv_index = None
    print("UYARI: idx_train/idx_val bulunamadı.")

oof_index = base["index"].to_numpy()

if (
    cv_index is not None
    and len(feature_cv) == len(cv_index)
):
    if set(oof_index) == set(cv_index):
        pos = {
            idx: i
            for i, idx in enumerate(cv_index)
        }

        order = [
            pos[idx]
            for idx in oof_index
        ]

        feature_cv = (
            feature_cv
            .iloc[order]
            .reset_index(drop=True)
        )

        print(
            "Feature ↔ OOF index eşleşmesi: BAŞARILI"
        )

    else:
        print(
            "UYARI: Feature ↔ OOF index eşleşmedi."
        )
        print("Mevcut sıra kullanılacak.")

else:
    print(
        "UYARI: Feature ↔ OOF index doğrulanamadı."
    )
    print("Mevcut sıra kullanılacak.")

if len(feature_cv) != len(actual):
    raise ValueError(
        f"Feature OOF uzunluğu uyuşmuyor: "
        f"{len(feature_cv):,} / {len(actual):,}"
    )


selection_cv = (
    feature_cv["tender_selectionmethod"]
    .astype(str)
    .to_numpy()
)

y_train_np = np.asarray(
    y_train,
    dtype=float
)

y_val_np = np.asarray(
    y_val,
    dtype=float
)

y_cv = np.concatenate([
    y_train_np,
    y_val_np
])

if len(y_cv) != len(actual):
    raise ValueError(
        f"y_train+y_val uzunluğu OOF ile uyuşmuyor: "
        f"{len(y_cv):,} / {len(actual):,}"
    )


price_threshold = np.quantile(
    y_train_np,
    0.75
)

lowest_price = (
    selection_cv == "LOWEST_PRICE"
)

high_price = (
    y_cv >= price_threshold
)

target_group = (
    lowest_price & high_price
)

groups = {
    "ALL": np.ones(
        len(actual),
        dtype=bool
    ),
    "LOWEST_PRICE": lowest_price,
    "LOWEST_PRICE + HIGH PRICE": target_group,
    "OTHER": ~target_group
}

group_metrics = {}

for name, mask in groups.items():

    group_metrics[name] = {
        "N": int(mask.sum()),
        "Baseline": metrics(
            actual[mask],
            pred_base[mask]
        ),
        "Weighted": metrics(
            actual[mask],
            pred_weighted[mask]
        )
    }


def column_type(s):

    if (
        pd.api.types.is_object_dtype(s)
        or pd.api.types.is_string_dtype(s)
        or pd.api.types.is_bool_dtype(s)
    ):
        return "categorical"

    return (
        "categorical"
        if s.nunique(dropna=True) <= 20
        else "continuous"
    )


def make_groups(s):

    if column_type(s) == "categorical":

        return (
            s.astype(str)
            .replace({
                "nan": "MISSING",
                "None": "MISSING"
            })
        )

    x = pd.to_numeric(
        s,
        errors="coerce"
    )

    if x.notna().sum() == 0:

        return pd.Series(
            ["MISSING"] * len(s),
            index=s.index
        )

    try:

        g = pd.qcut(
            x,
            q=N_BINS,
            duplicates="drop"
        ).astype(str)

    except Exception:

        try:

            g = pd.cut(
                x,
                bins=N_BINS
            ).astype(str)

        except Exception:

            g = pd.Series(
                ["ALL"] * len(s),
                index=s.index
            )

    return g.replace({
        "nan": "MISSING"
    })


column_results = []

for col in COLUMN_ANALYSIS:

    if col not in feature_cv.columns:
        print(f"Atlandı: {col}")
        continue

    grouped = make_groups(
        feature_cv[col]
    )

    temp = pd.DataFrame({
        "Group": grouped.to_numpy(),
        "Actual": actual,
        "Baseline": pred_base,
        "Weighted": pred_weighted
    })

    for group_name, data in temp.groupby(
        "Group",
        dropna=False
    ):

        if len(data) < MIN_GROUP_SIZE:
            continue

        y = data["Actual"].to_numpy()
        pb = data["Baseline"].to_numpy()
        pw = data["Weighted"].to_numpy()

        mb = metrics(y, pb)
        mw = metrics(y, pw)

        base_bias = abs(
            mb["Mean Residual"]
        )

        weighted_bias = abs(
            mw["Mean Residual"]
        )

        column_results.append({

            "Feature": col,
            "Group": str(group_name),
            "N": len(data),

            "Baseline RMSE": mb["RMSE"],
            "Weighted RMSE": mw["RMSE"],

            "RMSE Change %": (
                (mw["RMSE"] - mb["RMSE"])
                / max(mb["RMSE"], 1e-12)
                * 100
            ),

            "Baseline MAE": mb["MAE"],
            "Weighted MAE": mw["MAE"],

            "MAE Change %": (
                (mw["MAE"] - mb["MAE"])
                / max(mb["MAE"], 1e-12)
                * 100
            ),

            "Baseline R2": mb["R2"],
            "Weighted R2": mw["R2"],

            "R2 Change": (
                mw["R2"] - mb["R2"]
            ),

            "Baseline Mean Residual":
                mb["Mean Residual"],

            "Weighted Mean Residual":
                mw["Mean Residual"],

            "Bias Improvement %": (
                (base_bias - weighted_bias)
                / max(base_bias, 1e-12)
                * 100
            ),

            "Baseline Underprediction %":
                mb["Underprediction %"],

            "Weighted Underprediction %":
                mw["Underprediction %"],

            "Underprediction Change": (
                mw["Underprediction %"]
                - mb["Underprediction %"]
            )
        })


column_df = pd.DataFrame(
    column_results
)

if len(column_df):

    column_df["Label"] = (
        column_df["Feature"]
        + " | "
        + column_df["Group"]
    )

column_df.to_csv(
    OUTPUT_COLUMN_CSV,
    index=False
)


target_name = "LOWEST_PRICE + HIGH PRICE"
target_mask = target_group

target_base_error = (
    abs_error_base[target_mask]
)

target_weighted_error = (
    abs_error_weighted[target_mask]
)

improved = (
    target_weighted_error
    < target_base_error
)

worsened = (
    target_weighted_error
    > target_base_error
)

same = (
    target_weighted_error
    == target_base_error
)

target_base_bias = (
    res_base[target_mask].mean()
)

target_weighted_bias = (
    res_weighted[target_mask].mean()
)

target_bias_improvement = (
    (
        abs(target_base_bias)
        - abs(target_weighted_bias)
    )
    / max(abs(target_base_bias), 1e-12)
) * 100


overall_rmse_change = (
    (m_weighted["RMSE"] - m_base["RMSE"])
    / m_base["RMSE"]
) * 100

overall_mae_change = (
    (m_weighted["MAE"] - m_base["MAE"])
    / m_base["MAE"]
) * 100

overall_r2_change = (
    m_weighted["R2"]
    - m_base["R2"]
)


target_rmse_base = (
    group_metrics[target_name]
    ["Baseline"]["RMSE"]
)

target_rmse_weighted = (
    group_metrics[target_name]
    ["Weighted"]["RMSE"]
)

target_rmse_change = (
    (target_rmse_weighted - target_rmse_base)
    / target_rmse_base
) * 100


def money(x):

    x = float(x)

    if abs(x) >= 1e9:
        return f"${x / 1e9:.2f}B"

    if abs(x) >= 1e6:
        return f"${x / 1e6:.2f}M"

    if abs(x) >= 1e3:
        return f"${x / 1e3:.2f}K"

    return f"${x:.2f}"


def pct(x):
    return f"{float(x):+.2f}%"


def add_value_labels(
    ax,
    bars,
    values,
    formatter,
    fontsize=10
):

    ymin, ymax = ax.get_ylim()
    offset = (ymax - ymin) * 0.025

    for bar, value in zip(
        bars,
        values
    ):

        height = bar.get_height()

        if height >= 0:

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height + offset,
                formatter(value),
                ha="center",
                va="bottom",
                fontsize=fontsize,
                fontweight="bold"
            )

        else:

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height - offset,
                formatter(value),
                ha="center",
                va="top",
                fontsize=fontsize,
                fontweight="bold"
            )


def save_page(fig, filename):

    fig.subplots_adjust(
        left=0.07,
        right=0.98,
        top=0.88,
        bottom=0.15,
        hspace=0.38,
        wspace=0.28
    )

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    fig.savefig(
        path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# SAYFA 1 — GENEL MODEL PERFORMANSI
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18, 11)
)

fig.suptitle(
    "SAMPLE WEIGHTING — GENEL MODEL PERFORMANSI",
    fontsize=20,
    fontweight="bold",
    y=0.96
)

metric_data = [

    (
        "RMSE",
        m_base["RMSE"],
        m_weighted["RMSE"],
        money
    ),

    (
        "MAE",
        m_base["MAE"],
        m_weighted["MAE"],
        money
    ),

    (
        "R²",
        m_base["R2"],
        m_weighted["R2"],
        lambda x: f"{x:.4f}"
    ),

    (
        "Mean Residual",
        m_base["Mean Residual"],
        m_weighted["Mean Residual"],
        money
    ),

    (
        "Underprediction",
        m_base["Underprediction %"],
        m_weighted["Underprediction %"],
        lambda x: f"{x:.2f}%"
    ),

    (
        "P90 Absolute Error",
        m_base["P90"],
        m_weighted["P90"],
        money
    )
]

for ax, (
    title,
    b,
    w,
    formatter
) in zip(
    axes.flat,
    metric_data
):

    bars = ax.bar(
        ["Baseline", "Weighted"],
        [b, w],
        width=0.55,
        color=[
            BASE_COLOR,
            WEIGHTED_COLOR
        ]
    )

    ax.set_title(
        title,
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    ax.set_axisbelow(True)

    ax.margins(
        y=0.18
    )

    add_value_labels(
        ax,
        bars,
        [b, w],
        formatter,
        fontsize=10
    )


fig.text(
    0.5,
    0.055,
    f"Overall RMSE Change: {pct(overall_rmse_change)}"
    f"    |    "
    f"Overall MAE Change: {pct(overall_mae_change)}"
    f"    |    "
    f"Overall R² Change: {overall_r2_change:+.4f}",
    ha="center",
    fontsize=12,
    fontweight="bold"
)

fig.text(
    0.5,
    0.025,
    "Baseline = Mavi    |    Weighted = Turuncu",
    ha="center",
    fontsize=10
)

save_page(
    fig,
    "01_general_performance.png"
)


# ============================================================
# SAYFA 2 — HEDEF GRUP VE RESIDUAL ANALİZİ
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18, 11)
)

fig.suptitle(
    "SAMPLE WEIGHTING — HEDEF GRUP & RESIDUAL ANALİZİ",
    fontsize=20,
    fontweight="bold",
    y=0.96
)

group_names = list(
    groups.keys()
)

x = np.arange(
    len(group_names)
)

width = 0.34


def grouped_bar(
    ax,
    baseline,
    weighted_values,
    title,
    ylabel,
    formatter=None
):

    bars_b = ax.bar(
        x - width / 2,
        baseline,
        width,
        label="Baseline",
        color=BASE_COLOR
    )

    bars_w = ax.bar(
        x + width / 2,
        weighted_values,
        width,
        label="Weighted",
        color=WEIGHTED_COLOR
    )

    ax.set_title(
        title,
        fontweight="bold",
        fontsize=14,
        pad=12
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        group_names,
        rotation=18,
        ha="right",
        fontsize=9
    )

    ax.set_ylabel(
        ylabel
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    ax.set_axisbelow(True)

    ax.legend(
        frameon=False
    )

    ax.margins(
        y=0.18
    )

    if formatter is not None:

        add_value_labels(
            ax,
            bars_b,
            baseline,
            formatter,
            fontsize=8
        )

        add_value_labels(
            ax,
            bars_w,
            weighted_values,
            formatter,
            fontsize=8
        )


# Grup RMSE

b = [
    group_metrics[g]["Baseline"]["RMSE"]
    for g in group_names
]

w = [
    group_metrics[g]["Weighted"]["RMSE"]
    for g in group_names
]

grouped_bar(
    axes[0, 0],
    b,
    w,
    "Grup RMSE",
    "RMSE (USD)",
    money
)


# Grup MAE

b = [
    group_metrics[g]["Baseline"]["MAE"]
    for g in group_names
]

w = [
    group_metrics[g]["Weighted"]["MAE"]
    for g in group_names
]

grouped_bar(
    axes[0, 1],
    b,
    w,
    "Grup MAE",
    "MAE (USD)",
    money
)


# Underprediction

b = [
    group_metrics[g]["Baseline"]
    ["Underprediction %"]
    for g in group_names
]

w = [
    group_metrics[g]["Weighted"]
    ["Underprediction %"]
    for g in group_names
]

grouped_bar(
    axes[0, 2],
    b,
    w,
    "Grup Underprediction",
    "%",
    lambda x: f"{x:.1f}%"
)


# Actual vs Predicted

ax = axes[1, 0]

plot_limit = np.percentile(
    actual,
    99.5
)

mask = (
    actual <= plot_limit
)

ax.scatter(
    actual[mask],
    pred_base[mask],
    s=4,
    alpha=0.15,
    color=BASE_COLOR,
    label="Baseline"
)

ax.scatter(
    actual[mask],
    pred_weighted[mask],
    s=4,
    alpha=0.15,
    color=WEIGHTED_COLOR,
    label="Weighted"
)

ax.plot(
    [0, plot_limit],
    [0, plot_limit],
    "--",
    linewidth=1.5,
    color="black",
    label="Perfect Prediction"
)

ax.set_title(
    "Actual vs Predicted",
    fontweight="bold",
    fontsize=14,
    pad=12
)

ax.set_xlabel(
    "Actual USD"
)

ax.set_ylabel(
    "Predicted USD"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.2
)


# Residual dağılımı

ax = axes[1, 1]

res_limit = np.percentile(
    np.abs(
        np.concatenate([
            res_base,
            res_weighted
        ])
    ),
    99
)

ax.hist(
    res_base[
        np.abs(res_base) <= res_limit
    ],
    bins=60,
    alpha=0.55,
    color=BASE_COLOR,
    label="Baseline"
)

ax.hist(
    res_weighted[
        np.abs(res_weighted) <= res_limit
    ],
    bins=60,
    alpha=0.55,
    color=WEIGHTED_COLOR,
    label="Weighted"
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.5,
    color="black"
)

ax.set_title(
    "Residual Dağılımı",
    fontweight="bold",
    fontsize=14,
    pad=12
)

ax.set_xlabel(
    "Residual USD"
)

ax.set_ylabel(
    "Frequency"
)

ax.legend(
    frameon=False
)

ax.grid(
    axis="y",
    alpha=0.2
)


# Hedef grup bias

ax = axes[1, 2]

bias_values = [
    target_base_bias,
    target_weighted_bias
]

bars = ax.bar(
    ["Baseline", "Weighted"],
    bias_values,
    width=0.55,
    color=[
        BASE_COLOR,
        WEIGHTED_COLOR
    ]
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
    color="black"
)

ax.set_title(
    "LOWEST_PRICE + HIGH PRICE — Bias",
    fontweight="bold",
    fontsize=14,
    pad=12
)

ax.set_ylabel(
    "Mean Residual USD"
)

ax.grid(
    axis="y",
    alpha=0.25
)

ax.set_axisbelow(True)

ax.margins(
    y=0.20
)

add_value_labels(
    ax,
    bars,
    bias_values,
    money,
    fontsize=10
)


fig.text(
    0.5,
    0.055,
    f"Target RMSE Change: {pct(target_rmse_change)}"
    f"    |    "
    f"Target Bias Improvement: {pct(target_bias_improvement)}"
    f"    |    "
    f"Improved Observations: {improved.mean()*100:.2f}%",
    ha="center",
    fontsize=12,
    fontweight="bold"
)

fig.text(
    0.5,
    0.025,
    "Baseline = Mavi    |    Weighted = Turuncu",
    ha="center",
    fontsize=10
)

save_page(
    fig,
    "02_target_and_residuals.png"
)


# ============================================================
# SAYFA 3 — KOLON / KATEGORİ ANALİZİ
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(19, 13)
)

fig.suptitle(
    "SAMPLE WEIGHTING — KOLON / KATEGORİ ETKİ ANALİZİ",
    fontsize=20,
    fontweight="bold",
    y=0.96
)


def plot_column_metric(
    ax,
    metric,
    title,
    n=12,
    good_when_negative=True
):

    if not len(column_df):

        ax.text(
            0.5,
            0.5,
            "Kolon analizi bulunamadı",
            ha="center",
            va="center"
        )

        ax.set_title(
            title
        )

        return

    d = column_df.sort_values(
        metric,
        ascending=True
    )

    if len(d) > n:

        d = pd.concat([
            d.head(n // 2),
            d.tail(n // 2)
        ]).drop_duplicates()

    d = d.sort_values(
        metric
    )

    values = (
        d[metric]
        .astype(float)
        .to_numpy()
    )

    labels = []

    for label in d["Label"].astype(str):

        label = label.replace(
            "LOWEST_PRICE + HIGH PRICE",
            "LOWEST_PRICE + HIGH"
        )

        if len(label) > 38:
            label = label[:35] + "..."

        labels.append(label)

    colors = [
        GOOD_COLOR
        if (
            value < 0
            if good_when_negative
            else value > 0
        )
        else BAD_COLOR
        for value in values
    ]

    bars = ax.barh(
        labels,
        values,
        color=colors,
        height=0.68
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
        color="black"
    )

    ax.set_title(
        title,
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    ax.grid(
        axis="x",
        alpha=0.25
    )

    ax.set_axisbelow(True)

    ax.tick_params(
        axis="y",
        labelsize=8
    )

    max_abs = max(
        np.max(np.abs(values)),
        1e-9
    )

    ax.set_xlim(
        -max_abs * 1.18,
        max_abs * 1.18
    )

    for bar, value in zip(
        bars,
        values
    ):

        if value >= 0:

            ax.annotate(
                f"{value:+.2f}",
                xy=(
                    value,
                    bar.get_y()
                    + bar.get_height() / 2
                ),
                xytext=(5, 0),
                textcoords="offset points",
                ha="left",
                va="center",
                fontsize=8,
                fontweight="bold"
            )

        else:

            ax.annotate(
                f"{value:+.2f}",
                xy=(
                    value,
                    bar.get_y()
                    + bar.get_height() / 2
                ),
                xytext=(-5, 0),
                textcoords="offset points",
                ha="right",
                va="center",
                fontsize=8,
                fontweight="bold"
            )


plot_column_metric(
    axes[0, 0],
    "RMSE Change %",
    "RMSE Değişimi (%)",
    good_when_negative=True
)

plot_column_metric(
    axes[0, 1],
    "Bias Improvement %",
    "Bias İyileşmesi (%)",
    good_when_negative=False
)

plot_column_metric(
    axes[1, 0],
    "Underprediction Change",
    "Underprediction Değişimi (pp)",
    good_when_negative=True
)

plot_column_metric(
    axes[1, 1],
    "R2 Change",
    "R² Değişimi",
    good_when_negative=False
)


fig.text(
    0.5,
    0.055,
    "Yeşil = Weighted model lehine değişim"
    "    |    "
    "Kırmızı = Weighted model aleyhine değişim",
    ha="center",
    fontsize=12,
    fontweight="bold"
)

fig.text(
    0.5,
    0.025,
    "Not: RMSE/MAE için negatif değişim iyileşme; "
    "R² için pozitif değişim iyileşme anlamına gelir.",
    ha="center",
    fontsize=9
)

save_page(
    fig,
    "03_column_analysis.png"
)


# ============================================================
# SONUÇLAR
# ============================================================

print("\n" + "=" * 80)
print("SAMPLE WEIGHTING ANALİZİ TAMAMLANDI")
print("=" * 80)

print("\n3 SAYFA:")

print(
    os.path.join(
        OUTPUT_DIR,
        "01_general_performance.png"
    )
)

print(
    os.path.join(
        OUTPUT_DIR,
        "02_target_and_residuals.png"
    )
)

print(
    os.path.join(
        OUTPUT_DIR,
        "03_column_analysis.png"
    )
)

print("\nKOLON ANALİZİ:")
print(
    OUTPUT_COLUMN_CSV
)

print("\nGENEL SONUÇ:")

print(
    f"RMSE Change : "
    f"{overall_rmse_change:+.2f}%"
)

print(
    f"MAE Change  : "
    f"{overall_mae_change:+.2f}%"
)

print(
    f"R² Change   : "
    f"{overall_r2_change:+.4f}"
)

print("\nHEDEF GRUP:")

print(
    f"RMSE Change : "
    f"{target_rmse_change:+.2f}%"
)

print(
    f"Bias Improve: "
    f"{target_bias_improvement:+.2f}%"
)

print(
    f"Improved    : "
    f"{improved.sum():,} "
    f"({improved.mean()*100:.2f}%)"
)

print(
    f"Worsened    : "
    f"{worsened.sum():,} "
    f"({worsened.mean()*100:.2f}%)"
)

print(
    f"Same        : "
    f"{same.sum():,} "
    f"({same.mean()*100:.2f}%)"
)

print("=" * 80)

Genel model performans grafikleri incelendiğinde, yapılan örnek ağırlıklandırma (sample weighting) çalışmasının modelin genel tahmin gücünü net bir şekilde iyileştirdiği görülmektedir. Genel performans metriklerinde RMSE değerinin %2.42 düşerek 17.06M USD seviyesinden 16.65M USD'ye, MAE değerinin %2.00 azalarak 2.06M USD'den 2.02M USD'ye gerilemesi ve $R^2$ skorunun 0.8018'den 0.8113'e yükselmesi modelin tutarlılığının arttığını göstermektedir. Ayrıca, modelin eksik tahmin yapma eğilimini gösteren Underprediction oranının %48.05'ten %45.84'e düşmesi ve ortalama kalıntı hatanın (Mean Residual) 1.63M USD'den 1.51M USD'ye inmesi, modelin yanlılığının (bias) azaldığına işaret etmektedir.Özel hedef grup ve kalıntı (residual) analizleri, iyileşmenin arkasındaki temel dinamikleri daha açık bir şekilde ortaya koymaktadır. Özellikle hedef segment olan "LOWEST_PRICE + HIGH PRICE" grubunda hatanın belirgin şekilde baskılandığı görülmektedir; bu gruptaki RMSE %5.89 oranında düşerek 34.99M USD'den 32.93M USD'ye, ortalama sapma (bias) ise %10.12'lik bir iyileşmeyle 7.44M USD'den 6.69M USD seviyesine gerilemiştir. Gözlemlerin %73.00'ünde daha başarılı sonuçlar elde edilmiş olması, uygulanan ağırlıklandırma stratejisinin doğrudan problemli alana odaklanarak olumlu sonuç verdiğini kanıtlamaktadır.Kolon ve kategori bazlı etki analizleri incelendiğinde ise stratejinin her segmentte aynı etkiyi yaratmadığı, bazı alt gruplarda performans kaybı yaşandığı anlaşılmaktadır. Örneğin, log_decision_period ve belli corr_buyer_concentration aralıklarında RMSE %16 ila %23 bandında gerileyerek model lehine ciddi bir katkı sağlarken, buyer_mainactivities | GAS_AND_HEAT... ve SOCIAL_PROTECTION gibi kategorilerde RMSE değerlerinin %18'in üzerinde arttığı görülmektedir. Benzer şekilde log_tender_lotscount tarafında $R^2$ skorunda +0.15'lik belirgin bir artış elde edilmişken, bazı alıcı ana faaliyet alanlarında bias değerlerinin olumsuz etkilendiği göze çarpmaktadır. Özetle, yapılan müdahale hedef grupta ve genel hatlarıyla modeli kesin olarak iyileştirmiş olsa da, performans kaybı yaşanan belirli alt kategoriler için feature engineering veya ikincil bir ağırlıklandırma kuralı ile ek düzenlemeler yapılması faydalı olacaktır.

## KERAS

In [ ]:
import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import r2_score

SEED = 42
N_TRIALS = 20
MAX_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 20
LR_PATIENCE = 7
MAX_OVERFIT_GAP = 0.20

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 100)
print("KERAS — 20 RANDOM TRIAL NEURAL NETWORK TUNING")
print("TRAIN + VALIDATION + RMSE + MAE + R² + OVERFIT CONTROL")
print("=" * 100)

# DATA
required = [
    "X_train_encoded",
    "X_val_encoded",
    "y_train",
    "y_val"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise NameError(f"Eksik değişkenler: {missing}")

Xtr = np.asarray(X_train_encoded, dtype=np.float32)
Xv = np.asarray(X_val_encoded, dtype=np.float32)

ytr = np.asarray(y_train, dtype=np.float32).reshape(-1)
yv = np.asarray(y_val, dtype=np.float32).reshape(-1)

if len(Xtr) != len(ytr):
    raise ValueError(
        f"Train boyutları uyuşmuyor: "
        f"X_train={len(Xtr)}, y_train={len(ytr)}"
    )

if len(Xv) != len(yv):
    raise ValueError(
        f"Validation boyutları uyuşmuyor: "
        f"X_val={len(Xv)}, y_val={len(yv)}"
    )

INPUT_DIM = Xtr.shape[1]

print(f"\nX_train_encoded : {Xtr.shape}")
print(f"X_val_encoded   : {Xv.shape}")
print(f"Features        : {INPUT_DIM}")
print(f"Max Overfit Gap : {MAX_OVERFIT_GAP * 100:.0f}%")

# RANDOM PARAMETER SPACE
rng = np.random.default_rng(SEED)

def random_params():

    n_layers = int(rng.integers(2, 6))

    units_options = [
        64,
        128,
        256,
        384,
        512,
        768,
        1024
    ]

    return {
        "units": [
            int(rng.choice(units_options))
            for _ in range(n_layers)
        ],
        "activation": str(
            rng.choice(["relu", "swish"])
        ),
        "dropout": float(
            rng.uniform(0.05, 0.35)
        ),
        "l2": float(
            10 ** rng.uniform(-6, -3)
        ),
        "learning_rate": float(
            10 ** rng.uniform(-4, -2.5)
        ),
        "batch_size": int(
            rng.choice([256, 512, 1024, 2048])
        )
    }

# MODEL BUILDER
def build_model(params):

    model = keras.Sequential([
        layers.Input(shape=(INPUT_DIM,))
    ])

    for units in params["units"]:

        model.add(
            layers.Dense(
                units,
                activation=params["activation"],
                kernel_regularizer=regularizers.l2(
                    params["l2"]
                )
            )
        )

        model.add(
            layers.BatchNormalization()
        )

        model.add(
            layers.Dropout(params["dropout"])
        )

    model.add(
        layers.Dense(1)
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=params["learning_rate"]
        ),
        loss="mse"
    )

    return model

# RANDOM TRIALS
results = []
trial_models = {}
trial_histories = {}

for trial in range(1, N_TRIALS + 1):

    print("\n" + "=" * 100)
    print(f"TRIAL {trial}/{N_TRIALS}")
    print("=" * 100)

    tf.keras.backend.clear_session()

    params = random_params()

    print("\nPARAMETERS")

    for key, value in params.items():
        print(f"{key:<18}: {value}")

    model = build_model(params)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            min_delta=1e-5,
            mode="min",
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=LR_PATIENCE,
            min_lr=1e-7,
            mode="min",
            verbose=1
        )
    ]

    history = model.fit(
        Xtr,
        ytr,
        validation_data=(Xv, yv),
        epochs=MAX_EPOCHS,
        batch_size=params["batch_size"],
        callbacks=callbacks,
        shuffle=True,
        verbose=1
    )

    best_epoch = (
        int(
            np.argmin(
                history.history["val_loss"]
            )
        ) + 1
    )

    # PREDICTIONS
    train_pred = model.predict(
        Xtr,
        batch_size=params["batch_size"],
        verbose=0
    ).reshape(-1)

    val_pred = model.predict(
        Xv,
        batch_size=params["batch_size"],
        verbose=0
    ).reshape(-1)

    # TRAIN METRICS
    train_error = ytr - train_pred

    train_rmse = float(
        np.sqrt(
            np.mean(
                train_error ** 2
            )
        )
    )

    train_mae = float(
        np.mean(
            np.abs(train_error)
        )
    )

    train_r2 = float(
        r2_score(
            ytr,
            train_pred
        )
    )

    # VALIDATION METRICS
    val_error = yv - val_pred

    val_rmse = float(
        np.sqrt(
            np.mean(
                val_error ** 2
            )
        )
    )

    val_mae = float(
        np.mean(
            np.abs(val_error)
        )
    )

    val_r2 = float(
        r2_score(
            yv,
            val_pred
        )
    )

    # OVERFIT GAP
    overfit_gap = (
        (val_rmse - train_rmse)
        / max(train_rmse, 1e-8)
    )

    overfit_status = (
        "ACCEPTED"
        if overfit_gap <= MAX_OVERFIT_GAP
        else "OVERFIT"
    )

    results.append({
        "trial": trial,
        "train_rmse": train_rmse,
        "val_rmse": val_rmse,
        "train_mae": train_mae,
        "val_mae": val_mae,
        "train_r2": train_r2,
        "val_r2": val_r2,
        "overfit_gap": overfit_gap,
        "overfit_status": overfit_status,
        "best_epoch": best_epoch,
        "epochs": len(history.history["loss"]),
        **params
    })

    trial_models[trial] = model
    trial_histories[trial] = history

    print("\nRESULT")
    print(f"Train RMSE : {train_rmse:,.4f}")
    print(f"Val RMSE   : {val_rmse:,.4f}")
    print(f"Train MAE  : {train_mae:,.4f}")
    print(f"Val MAE    : {val_mae:,.4f}")
    print(f"Train R²   : {train_r2:.4f}")
    print(f"Val R²     : {val_r2:.4f}")
    print(f"Overfit Gap: {overfit_gap * 100:.2f}%")
    print(f"Status     : {overfit_status}")
    print(f"Best Epoch : {best_epoch}")

# RESULTS
results_df = (
    pd.DataFrame(results)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("20 TRIAL SONUÇLARI")
print("=" * 100)

display_columns = [
    "trial",
    "train_rmse",
    "val_rmse",
    "train_mae",
    "val_mae",
    "train_r2",
    "val_r2",
    "overfit_gap",
    "overfit_status",
    "best_epoch",
    "units",
    "activation",
    "dropout",
    "l2",
    "learning_rate",
    "batch_size"
]

print(
    results_df[
        display_columns
    ].to_string(index=False)
)

# OVERFIT FILTER
accepted_results = (
    results_df[
        results_df["overfit_gap"] <= MAX_OVERFIT_GAP
    ]
    .copy()
)

print("\n" + "=" * 100)
print("OVERFIT KONTROLÜ")
print("=" * 100)

print(f"Toplam Trial       : {len(results_df)}")
print(f"Kabul Edilen       : {len(accepted_results)}")
print(
    f"Overfit Olan       : "
    f"{len(results_df) - len(accepted_results)}"
)
print(
    f"Max Overfit Gap    : "
    f"{MAX_OVERFIT_GAP * 100:.0f}%"
)

# BEST MODEL SELECTION
if len(accepted_results) > 0:

    accepted_results = (
        accepted_results
        .sort_values(
            ["val_rmse", "overfit_gap"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

    best_row = accepted_results.iloc[0]

else:

    print(
        "\nUYARI: Overfit gap sınırı altında "
        "hiçbir model bulunamadı."
    )

    print(
        "En düşük overfit gap'e sahip "
        "model seçilecek."
    )

    best_row = (
        results_df
        .sort_values(
            ["overfit_gap", "val_rmse"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
        .iloc[0]
    )

BEST_TRIAL = int(best_row["trial"])

BEST_MODEL = trial_models[
    BEST_TRIAL
]

# BEST PARAMETERS
BEST_PARAMS = {
    "units": best_row["units"],
    "activation": best_row["activation"],
    "dropout": float(best_row["dropout"]),
    "l2": float(best_row["l2"]),
    "learning_rate": float(best_row["learning_rate"]),
    "batch_size": int(best_row["batch_size"])
}

BEST_EPOCH = int(
    best_row["best_epoch"]
)

print("\n" + "=" * 100)
print("BEST KERAS MODEL")
print("=" * 100)

print(f"Best Trial      : {BEST_TRIAL}")
print(
    f"Validation RMSE : "
    f"{best_row['val_rmse']:,.4f}"
)
print(
    f"Validation MAE  : "
    f"{best_row['val_mae']:,.4f}"
)
print(
    f"Validation R²   : "
    f"{best_row['val_r2']:.4f}"
)
print(
    f"Train RMSE      : "
    f"{best_row['train_rmse']:,.4f}"
)
print(
    f"Train MAE       : "
    f"{best_row['train_mae']:,.4f}"
)
print(
    f"Train R²        : "
    f"{best_row['train_r2']:.4f}"
)
print(
    f"Overfit Gap     : "
    f"{best_row['overfit_gap'] * 100:.2f}%"
)
print(
    f"Best Epoch      : "
    f"{BEST_EPOCH}"
)

print("\nBEST PARAMETERS")

for key, value in BEST_PARAMS.items():
    print(f"{key:<18}: {value}")

print("\n" + "=" * 100)
print("BEST MODEL ARCHITECTURE")
print("=" * 100)

BEST_MODEL.summary()

# BEST MODEL LOSS GRAPH
best_history = trial_histories[
    BEST_TRIAL
]

plt.figure(figsize=(10, 6))

plt.plot(
    best_history.history["loss"],
    label="Train Loss"
)

plt.plot(
    best_history.history["val_loss"],
    label="Validation Loss"
)

plt.axvline(
    BEST_EPOCH - 1,
    linestyle="--",
    label=f"Best Epoch = {BEST_EPOCH}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title(
    f"Best Keras Model — Train vs Validation Loss "
    f"(Trial {BEST_TRIAL})"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# SAVE
BEST_KERAS_MODEL = BEST_MODEL
BEST_KERAS_PARAMS = BEST_PARAMS.copy()
NN_RESULTS = results_df.copy()

BEST_KERAS_MODEL.save(
    "best_keras_neural_network.keras"
)

with open(
    "best_keras_parameters.json",
    "w"
) as f:

    json.dump(
        {
            **BEST_KERAS_PARAMS,
            "best_trial": BEST_TRIAL,
            "best_epoch": BEST_EPOCH,
            "train_rmse": float(
                best_row["train_rmse"]
            ),
            "val_rmse": float(
                best_row["val_rmse"]
            ),
            "train_mae": float(
                best_row["train_mae"]
            ),
            "val_mae": float(
                best_row["val_mae"]
            ),
            "train_r2": float(
                best_row["train_r2"]
            ),
            "val_r2": float(
                best_row["val_r2"]
            ),
            "overfit_gap": float(
                best_row["overfit_gap"]
            )
        },
        f,
        indent=4
    )

NN_RESULTS.to_csv(
    "keras_20_random_trials.csv",
    index=False
)

print("\n" + "=" * 100)
print("KAYDEDİLDİ")
print("=" * 100)

print("best_keras_neural_network.keras")
print("best_keras_parameters.json")
print("keras_20_random_trials.csv")

print("\nBEST MODEL VE PARAMETRELER HAZIR.")
print(f"Best Trial : {BEST_TRIAL}")
print(f"Best Epoch : {BEST_EPOCH}")

print("=" * 100)

## NEURALNET

**Neural Network Parametreleri ve Hiperparametreleri**

**Units (Nöron Sayıları):** Neural Network modelinde her Dense katmanında bulunan nöron sayısını ifade eder. Kullanılan modelde [256, 384, 512, 64, 384] şeklinde beş gizli katman bulunmaktadır. Buna göre birinci gizli katmanda 256, ikinci gizli katmanda 384, üçüncü gizli katmanda 512, dördüncü gizli katmanda 64 ve beşinci gizli katmanda 384 nöron bulunmaktadır. Nöron sayısı modelin öğrenme kapasitesini doğrudan etkiler. Nöron sayısının artırılması modelin daha karmaşık ilişkileri öğrenmesini sağlayabilir ancak gereğinden fazla nöron kullanılması aşırı öğrenme (overfitting) riskini artırabilir.

**Activation Function (Aktivasyon Fonksiyonu):** Modelde gizli katmanların aktivasyon fonksiyonu olarak **Swish** kullanılmıştır. Swish, girdiyi doğrusal olmayan bir dönüşümden geçirerek sinir ağının karmaşık ve doğrusal olmayan ilişkileri öğrenmesini sağlar. Swish fonksiyonu özellikle derin sinir ağlarında ReLU'ya alternatif olarak kullanılabilir ve küçük negatif değerlerin de ağ içerisinde işlenmesine izin verir. Modelin son katmanında ise aktivasyon fonksiyonu kullanılmamıştır çünkü model sürekli bir hedef değişkenin tahminini gerçekleştiren regresyon problemidir.

**Dropout:** Dropout oranı modelde **0.2492940621**, yani yaklaşık %24,93 olarak belirlenmiştir. Dropout, eğitim sırasında belirli orandaki nöronların rastgele devre dışı bırakılması yoluyla modelin belirli nöronlara aşırı bağımlı hale gelmesini engelleyen bir regularizasyon yöntemidir. Bu çalışmada yaklaşık her eğitim adımında nöronların %24,93'ü geçici olarak devre dışı bırakılmaktadır. Böylece modelin genelleme yeteneğinin artırılması ve overfitting riskinin azaltılması amaçlanmaktadır.

**L2 Regularization:** L2 regularization katsayısı **1.6563823126 × 10⁻⁵** olarak belirlenmiştir. L2 regularization, modelin ağırlıklarının gereğinden fazla büyümesini cezalandırarak daha basit ve genellenebilir bir model elde edilmesini amaçlar. Kayıp fonksiyonuna ağırlıkların karelerinin toplamına bağlı bir ceza terimi eklenir. Katsayının küçük olması regularization etkisinin kontrollü ve düşük seviyede uygulandığını gösterir.

**Learning Rate (Öğrenme Oranı):** Learning rate değeri **0.0016635298** olarak belirlenmiştir. Learning rate, eğitim sırasında model ağırlıklarının her güncellemede ne kadar değiştirileceğini belirleyen temel hiperparametredir. Çok yüksek bir learning rate modelin optimum noktayı aşmasına ve eğitimin kararsız hale gelmesine neden olabilirken, çok düşük bir learning rate modelin öğrenmesinin çok yavaş gerçekleşmesine neden olabilir. Bu nedenle uygun learning rate seçimi modelin hem eğitim hızını hem de yakınsadığı çözümün kalitesini etkiler.

**Batch Size:** Batch size değeri **2048** olarak belirlenmiştir. Batch size, model ağırlıkları güncellenmeden önce aynı anda işlenen eğitim örneği sayısını ifade eder. Bu modelde her ağırlık güncellemesinde 2048 gözlem kullanılmaktadır. Büyük batch size değerleri GPU belleğinin daha etkin kullanılmasını ve daha kararlı gradyan tahminleri elde edilmesini sağlayabilir; ancak çok büyük batch size değerleri modelin genelleme davranışını olumsuz etkileyebilir.

**Epoch:** Model için maksimum epoch sayısı **400** olarak belirlenmiştir. Bir epoch, eğitim veri setinin model tarafından bir kez tamamen işlenmesini ifade eder. 400 değeri modelin en fazla 400 kez eğitim verisi üzerinden öğrenmesine izin verildiğini gösterir. Ancak bu değer modelin mutlaka 400 epoch boyunca eğitileceği anlamına gelmez. Early Stopping mekanizması devreye girerse eğitim daha erken sonlandırılabilir.

**Early Stopping Patience:** Early stopping patience değeri **25** olarak belirlenmiştir. Bu mekanizma validation loss değerinin iyileşmediği ardışık epoch sayısını takip eder. Validation loss 25 epoch boyunca daha iyi bir değer üretmezse eğitim durdurulur ve validation loss açısından en iyi ağırlıklar geri yüklenir. Böylece modelin gereksiz yere eğitilmeye devam etmesi ve overfitting oluşması engellenmeye çalışılır.

**Learning Rate Patience:** ReduceLROnPlateau mekanizmasında patience değeri **8** olarak belirlenmiştir. Validation loss 8 epoch boyunca iyileşmezse learning rate azaltılır. Bu yaklaşım, modelin eğitim sırasında bir plato bölgesine ulaşması durumunda daha küçük adımlarla öğrenmeye devam etmesini sağlar.

**Learning Rate Factor:** Learning rate factor değeri **0.5** olarak belirlenmiştir. Validation loss belirlenen patience süresi boyunca iyileşmediğinde mevcut learning rate 0.5 ile çarpılır. Örneğin learning rate 0.0016635 ise azaltma sonrasında yaklaşık 0.0008318 seviyesine düşer. Bu işlem modelin optimum noktaya yaklaşırken daha küçük ağırlık güncellemeleri yapmasını sağlar.

**Minimum Learning Rate:** Minimum learning rate değeri **1 × 10⁻⁶** olarak belirlenmiştir. ReduceLROnPlateau mekanizması learning rate değerini azaltırken bu değerin altına inemez. Böylece learning rate'in aşırı derecede küçülerek model eğitimini etkisiz hale getirmesi önlenir.

**Batch Normalization:** Modelde her Dense katmanından sonra Batch Normalization katmanı kullanılmıştır. Batch Normalization, ara katmanların çıktılarının eğitim sırasında daha dengeli bir dağılıma sahip olmasına yardımcı olur. Bu yöntem eğitim sürecinin daha kararlı ilerlemesini ve farklı ölçeklerdeki aktivasyonların model tarafından daha kolay işlenmesini sağlayabilir. Ayrıca bazı durumlarda regularization etkisi sağlayarak modelin genelleme performansına katkıda bulunabilir.

**Optimizer:** Modelde optimizer olarak **Adam** kullanılmıştır. Adam, gradient descent yaklaşımının geliştirilmiş bir optimizasyon yöntemidir ve her parametre için geçmiş gradyan bilgilerini kullanarak ağırlık güncellemelerini adaptif şekilde gerçekleştirir. Bu nedenle özellikle çok katmanlı sinir ağlarının eğitiminde yaygın olarak kullanılmaktadır.

**Loss Function:** Modelin loss function'ı **Mean Squared Error (MSE)** olarak belirlenmiştir. MSE, gerçek değer ile modelin tahmin ettiği değer arasındaki farkın karesinin ortalamasını hesaplar. Büyük hataların karesi alındığı için yüksek miktardaki tahmin hatalarını daha güçlü şekilde cezalandırır. Model eğitim sırasında ağırlıklarını MSE değerini minimize edecek şekilde günceller.

**RMSE:** Root Mean Squared Error, MSE değerinin karekökünün alınmasıyla elde edilen hata ölçüsüdür. Modelde eğitim sırasında temel loss olarak MSE kullanılmasına rağmen RMSE ayrıca bir performans metriği olarak takip edilmektedir. RMSE'nin avantajı, hedef değişken ile aynı birimde ifade edilmesidir. Dolayısıyla tahmin hatasının büyüklüğünü yorumlamak MSE'ye göre daha kolaydır.

**MAE:** Mean Absolute Error, gerçek değer ile tahmin arasındaki mutlak farkların ortalamasını ifade eder. RMSE'den farklı olarak hataların karesini almadığı için aşırı büyük hatalara karşı daha az duyarlıdır. Bu nedenle modelin tipik tahmin hatasının değerlendirilmesinde kullanılabilir.

**R² (Determination Coefficient):** R², modelin hedef değişkendeki varyansın ne kadarını açıklayabildiğini gösteren bir performans ölçüsüdür. R² değerinin 1'e yaklaşması modelin hedef değişkeni açıklama gücünün yüksek olduğunu gösterirken, 0'a yakın değerler modelin açıklayıcılığının düşük olduğunu gösterir. Bu çalışmada R² eğitim sonrasında hem train hem de validation veri seti üzerinde hesaplanmaktadır.

**Input Dimension (Girdi Boyutu):** Input dimension, Neural Network modeline verilen özellik sayısını ifade eder. Kodda `input_dim = X_train_encoded.shape[1]` şeklinde belirlenmektedir. Dolayısıyla modelin girdi boyutu doğrudan `X_train_encoded` veri setindeki sütun sayısına bağlıdır. Modelin ilk Dense katmanının giriş boyutu bu değere göre otomatik olarak oluşturulmaktadır.

**Model Parametreleri:** Parametreler, model eğitimi sırasında veriden otomatik olarak öğrenilen değerlerdir. Neural Network açısından temel parametreler Dense katmanlarındaki **ağırlıklar (weights)** ve **bias** değerleridir. Örneğin bir Dense katmanında her nöronun girişlerden aldığı ağırlıklar eğitim sırasında gradient descent ve Adam optimizer kullanılarak güncellenir. Bu değerler modelin veriden öğrendiği gerçek matematiksel katsayılardır ve kullanıcı tarafından doğrudan belirlenmez.

**Hiperparametreler:** Hiperparametreler ise model eğitilmeye başlamadan önce kullanıcı tarafından belirlenen ve eğitim sürecinin nasıl gerçekleşeceğini kontrol eden değerlerdir. Bu çalışmadaki **Units, Activation, Dropout, L2, Learning Rate, Batch Size, Maximum Epochs, Early Stopping Patience, Learning Rate Patience, Learning Rate Factor ve Minimum Learning Rate** hiperparametrelerdir. Bu değerler doğrudan veriden öğrenilmez; modelin öğrenme kapasitesini, regularization seviyesini ve eğitim sürecini kontrol eder.

**Genel olarak:** Bu Neural Network modelinde öğrenilebilir parametreler ağırlıklar ve bias değerleri iken, modelin mimarisi ve eğitim sürecini belirleyen Units, Swish aktivasyon fonksiyonu, Dropout, L2 regularization, Learning Rate, Batch Size ve epoch ile ilgili değerler hiperparametrelerdir. Hiperparametrelerin **Best Trial 13** kapsamında belirlenmiş olması, bu değerlerin tuning süreci sonucunda seçildiğini ve ardından final model eğitiminde kullanıldığını ifade eder.


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

UNITS = [256, 384, 512, 384, 256]
ACTIVATION = "leaky_relu"
LEAKY_RELU_ALPHA = 0.01
DROPOUT = 0.15
L2 = 1.656382312621769e-05
LEARNING_RATE = 0.0016635297689606636
BATCH_SIZE = 1024

MAX_EPOCHS = 400
EARLY_STOPPING_PATIENCE = 25

LR_PATIENCE = 8
LR_FACTOR = 0.5
MIN_LR = 1e-6

TARGET_IS_LOG = True
POWER = 0.35

P50_WEIGHT = 1.0
P75_WEIGHT = 1.2
P90_WEIGHT = 1.5
P95_WEIGHT = 2.0
P99_WEIGHT = 10.0
P995_WEIGHT = 25.0

OUTPUT_DIR = "keras_final_model_weighted"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. DATA
# ============================================================

X_train_encoded = np.asarray(
    X_train_encoded,
    dtype=np.float32
)

X_val_encoded = np.asarray(
    X_val_encoded,
    dtype=np.float32
)

y_train = np.asarray(
    y_train,
    dtype=np.float32
).reshape(-1)

y_val = np.asarray(
    y_val,
    dtype=np.float32
).reshape(-1)

print("=" * 100)
print("KERAS — POWER 0.25 + HIGH PRICE WEIGHTED FINAL")
print("=" * 100)

print(f"Train shape      : {X_train_encoded.shape}")
print(f"Validation shape : {X_val_encoded.shape}")
print(f"Train dtype      : {X_train_encoded.dtype}")
print(f"Validation dtype : {X_val_encoded.dtype}")

if not np.all(np.isfinite(X_train_encoded)):
    raise ValueError(
        "X_train_encoded içinde NaN veya Inf bulundu."
    )

if not np.all(np.isfinite(X_val_encoded)):
    raise ValueError(
        "X_val_encoded içinde NaN veya Inf bulundu."
    )

if not np.all(np.isfinite(y_train)):
    raise ValueError(
        "y_train içinde NaN veya Inf bulundu."
    )

if not np.all(np.isfinite(y_val)):
    raise ValueError(
        "y_val içinde NaN veya Inf bulundu."
    )

if len(X_train_encoded) != len(y_train):
    raise ValueError(
        "X_train_encoded ve y_train satır sayıları eşleşmiyor."
    )

if len(X_val_encoded) != len(y_val):
    raise ValueError(
        "X_val_encoded ve y_val satır sayıları eşleşmiyor."
    )

# ============================================================
# 2. LOG TARGET → USD → POWER TARGET
# ============================================================

if TARGET_IS_LOG:

    y_train_usd = np.maximum(
        np.expm1(y_train),
        0
    )

    y_val_usd = np.maximum(
        np.expm1(y_val),
        0
    )

else:

    y_train_usd = np.maximum(
        y_train,
        0
    )

    y_val_usd = np.maximum(
        y_val,
        0
    )

y_train_power = np.power(
    y_train_usd,
    POWER
)

y_val_power = np.power(
    y_val_usd,
    POWER
)

if not np.all(np.isfinite(y_train_power)):
    raise ValueError(
        "y_train_power içinde NaN veya Inf bulundu."
    )

if not np.all(np.isfinite(y_val_power)):
    raise ValueError(
        "y_val_power içinde NaN veya Inf bulundu."
    )

# ============================================================
# 3. HIGH PRICE WEIGHTS
# ============================================================

combined_usd = np.concatenate([
    y_train_usd,
    y_val_usd
])

p50 = np.percentile(
    combined_usd,
    50
)

p75 = np.percentile(
    combined_usd,
    75
)

p90 = np.percentile(
    combined_usd,
    90
)

p95 = np.percentile(
    combined_usd,
    95
)

p99 = np.percentile(
    combined_usd,
    99
)

p995 = np.percentile(
    combined_usd,
    99.5
)

def create_weights(y):

    weights = np.ones(
        len(y),
        dtype=np.float32
    )

    weights[
        (y >= p75)
        & (y < p90)
    ] = P75_WEIGHT

    weights[
        (y >= p90)
        & (y < p95)
    ] = P90_WEIGHT

    weights[
        (y >= p95)
        & (y < p99)
    ] = P95_WEIGHT

    weights[
        (y >= p99)
        & (y < p995)
    ] = P99_WEIGHT

    weights[
        y >= p995
    ] = P995_WEIGHT

    return weights

train_weights = create_weights(
    y_train_usd
)

val_weights = create_weights(
    y_val_usd
)

print()
print("=" * 100)
print("TARGET TRANSFORMATION")
print("=" * 100)

print("Original target     : log1p(USD)")
print("Recovered target    : USD")
print(f"Model target        : USD^{POWER}")
print(f"Inverse transform   : prediction^(1/{POWER})")

print()
print("PRICE PERCENTILES")
print("-" * 100)

print(f"P50   : ${p50:,.2f}")
print(f"P75   : ${p75:,.2f}")
print(f"P90   : ${p90:,.2f}")
print(f"P95   : ${p95:,.2f}")
print(f"P99   : ${p99:,.2f}")
print(f"P99.5 : ${p995:,.2f}")

print()
print("WEIGHTS")
print("-" * 100)

print(f"P50–P75     : {P50_WEIGHT}")
print(f"P75–P90     : {P75_WEIGHT}")
print(f"P90–P95     : {P90_WEIGHT}")
print(f"P95–P99     : {P95_WEIGHT}")
print(f"P99–P99.5   : {P99_WEIGHT}")
print(f"P99.5+      : {P995_WEIGHT}")

print()
print("WEIGHT DISTRIBUTION")
print("-" * 100)

for weight in [
    P50_WEIGHT,
    P75_WEIGHT,
    P90_WEIGHT,
    P95_WEIGHT,
    P99_WEIGHT,
    P995_WEIGHT
]:

    print(
        f"Weight {weight:>5.1f} : "
        f"Train {(train_weights == weight).sum():,} | "
        f"Validation {(val_weights == weight).sum():,}"
    )

print()
print("TARGET SUMMARY")
print("-" * 100)

print(
    f"Train USD mean       : ${np.mean(y_train_usd):,.2f}"
)

print(
    f"Validation USD mean  : ${np.mean(y_val_usd):,.2f}"
)

print(
    f"Train Power mean     : {np.mean(y_train_power):.6f}"
)

print(
    f"Validation Power mean: {np.mean(y_val_power):.6f}"
)

# ============================================================
# 4. MODEL
# ============================================================

input_dim = X_train_encoded.shape[1]

model = keras.Sequential([

    layers.Input(
        shape=(input_dim,)
    ),

    layers.Dense(
        UNITS[0],
        activation=ACTIVATION,
        kernel_regularizer=regularizers.l2(L2)
    ),

    layers.BatchNormalization(),

    layers.Dropout(
        DROPOUT
    ),

    layers.Dense(
        UNITS[1],
        activation=ACTIVATION,
        kernel_regularizer=regularizers.l2(L2)
    ),

    layers.BatchNormalization(),

    layers.Dropout(
        DROPOUT
    ),

    layers.Dense(
        UNITS[2],
        activation=ACTIVATION,
        kernel_regularizer=regularizers.l2(L2)
    ),

    layers.BatchNormalization(),

    layers.Dropout(
        DROPOUT
    ),

    layers.Dense(
        UNITS[3],
        activation=ACTIVATION,
        kernel_regularizer=regularizers.l2(L2)
    ),

    layers.BatchNormalization(),

    layers.Dropout(
        DROPOUT
    ),

    layers.Dense(
        UNITS[4],
        activation=ACTIVATION,
        kernel_regularizer=regularizers.l2(L2)
    ),

    layers.BatchNormalization(),

    layers.Dropout(
        DROPOUT
    ),

    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="mse",
    metrics=[
        keras.metrics.RootMeanSquaredError(
            name="rmse"
        ),
        keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

# ============================================================
# 5. CALLBACKS
# ============================================================

checkpoint_path = os.path.join(
    OUTPUT_DIR,
    "best_keras_model.keras"
)

checkpoint = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=MIN_LR,
    mode="min",
    verbose=1
)

# ============================================================
# 6. TRAIN / VALIDATION EVALUATION
# ============================================================

class TrainEvaluationCallback(
    keras.callbacks.Callback
):

    def __init__(
        self,
        x_train,
        y_train,
        train_weights,
        x_val,
        y_val,
        val_weights,
        batch_size
    ):

        super().__init__()

        self.x_train = x_train
        self.y_train = y_train
        self.train_weights = train_weights

        self.x_val = x_val
        self.y_val = y_val
        self.val_weights = val_weights

        self.batch_size = batch_size

        self.train_eval_loss = []
        self.val_eval_loss = []

        self.train_eval_rmse = []
        self.val_eval_rmse = []

        self.train_eval_mae = []
        self.val_eval_mae = []

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        train_result = self.model.evaluate(
            self.x_train,
            self.y_train,
            sample_weight=self.train_weights,
            batch_size=self.batch_size,
            verbose=0,
            return_dict=True
        )

        val_result = self.model.evaluate(
            self.x_val,
            self.y_val,
            sample_weight=self.val_weights,
            batch_size=self.batch_size,
            verbose=0,
            return_dict=True
        )

        self.train_eval_loss.append(
            float(train_result["loss"])
        )

        self.val_eval_loss.append(
            float(val_result["loss"])
        )

        self.train_eval_rmse.append(
            float(train_result["rmse"])
        )

        self.val_eval_rmse.append(
            float(val_result["rmse"])
        )

        self.train_eval_mae.append(
            float(train_result["mae"])
        )

        self.val_eval_mae.append(
            float(val_result["mae"])
        )

        print(
            f" — Weighted Train MSE: "
            f"{train_result['loss']:.6f}"
            f" | Weighted Val MSE: "
            f"{val_result['loss']:.6f}"
        )

train_eval_callback = TrainEvaluationCallback(
    X_train_encoded,
    y_train_power,
    train_weights,
    X_val_encoded,
    y_val_power,
    val_weights,
    BATCH_SIZE
)

# ============================================================
# 7. TRAINING
# ============================================================

print()
print("=" * 100)
print("MODEL TRAINING")
print("=" * 100)

history = model.fit(
    X_train_encoded,
    y_train_power,
    validation_data=(
        X_val_encoded,
        y_val_power,
        val_weights
    ),
    sample_weight=train_weights,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        train_eval_callback,
        checkpoint,
        early_stopping,
        reduce_lr
    ],
    verbose=1
)

# ============================================================
# 8. HISTORY
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(history_df) + 1
    )
)

history_df["train_eval_loss"] = (
    train_eval_callback.train_eval_loss
)

history_df["val_eval_loss"] = (
    train_eval_callback.val_eval_loss
)

history_df["train_eval_rmse"] = (
    train_eval_callback.train_eval_rmse
)

history_df["val_eval_rmse"] = (
    train_eval_callback.val_eval_rmse
)

history_df["train_eval_mae"] = (
    train_eval_callback.train_eval_mae
)

history_df["val_eval_mae"] = (
    train_eval_callback.val_eval_mae
)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "training_history.csv"
    ),
    index=False
)

best_epoch = (
    history_df[
        "val_eval_loss"
    ].idxmin()
    + 1
)

best_val_loss = (
    history_df[
        "val_eval_loss"
    ].min()
)

print()
print("=" * 100)
print("BEST EPOCH")
print("=" * 100)

print(
    f"Best Epoch       : {best_epoch}"
)

print(
    f"Best Validation  : {best_val_loss:.6f}"
)

# ============================================================
# 9. BEST MODEL
# ============================================================

model = keras.models.load_model(
    checkpoint_path
)

train_pred_power = model.predict(
    X_train_encoded,
    batch_size=BATCH_SIZE,
    verbose=0
).ravel()

val_pred_power = model.predict(
    X_val_encoded,
    batch_size=BATCH_SIZE,
    verbose=0
).ravel()

train_pred_power = np.maximum(
    train_pred_power,
    0
)

val_pred_power = np.maximum(
    val_pred_power,
    0
)

# POWER → USD

train_pred_usd = np.power(
    train_pred_power,
    1.0 / POWER
)

val_pred_usd = np.power(
    val_pred_power,
    1.0 / POWER
)

train_pred_usd = np.maximum(
    train_pred_usd,
    0
)

val_pred_usd = np.maximum(
    val_pred_usd,
    0
)

# ============================================================
# 10. USD METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(
        y_train_usd,
        train_pred_usd
    )
)

train_mae = mean_absolute_error(
    y_train_usd,
    train_pred_usd
)

train_r2 = r2_score(
    y_train_usd,
    train_pred_usd
)

val_rmse = np.sqrt(
    mean_squared_error(
        y_val_usd,
        val_pred_usd
    )
)

val_mae = mean_absolute_error(
    y_val_usd,
    val_pred_usd
)

val_r2 = r2_score(
    y_val_usd,
    val_pred_usd
)

overfit_gap = (
    (val_rmse - train_rmse)
    / val_rmse
) * 100

# ============================================================
# 11. POWER METRICS
# ============================================================

train_power_rmse = np.sqrt(
    mean_squared_error(
        y_train_power,
        train_pred_power
    )
)

train_power_mae = mean_absolute_error(
    y_train_power,
    train_pred_power
)

train_power_r2 = r2_score(
    y_train_power,
    train_pred_power
)

val_power_rmse = np.sqrt(
    mean_squared_error(
        y_val_power,
        val_pred_power
    )
)

val_power_mae = mean_absolute_error(
    y_val_power,
    val_pred_power
)

val_power_r2 = r2_score(
    y_val_power,
    val_pred_power
)

# ============================================================
# 12. FINAL RESULTS
# ============================================================

print()
print("=" * 100)
print("FINAL TRAIN + VALIDATION RESULTS — USD")
print("=" * 100)

print(
    f"Best Epoch       : {best_epoch}"
)

print()
print(
    f"Train RMSE       : ${train_rmse:,.2f}"
)

print(
    f"Train MAE        : ${train_mae:,.2f}"
)

print(
    f"Train R²         : {train_r2:.6f}"
)

print()
print(
    f"Validation RMSE  : ${val_rmse:,.2f}"
)

print(
    f"Validation MAE   : ${val_mae:,.2f}"
)

print(
    f"Validation R²    : {val_r2:.6f}"
)

print()
print(
    f"Overfit Gap      : {overfit_gap:.2f}%"
)

print()
print("=" * 100)
print("POWER TARGET RESULTS")
print("=" * 100)

print(
    f"Train Power RMSE      : {train_power_rmse:.6f}"
)

print(
    f"Train Power MAE       : {train_power_mae:.6f}"
)

print(
    f"Train Power R²        : {train_power_r2:.6f}"
)

print()
print(
    f"Validation Power RMSE : {val_power_rmse:.6f}"
)

print(
    f"Validation Power MAE  : {val_power_mae:.6f}"
)

print(
    f"Validation Power R²   : {val_power_r2:.6f}"
)

# ============================================================
# 13. BEST PARAMETERS
# ============================================================

print()
print("=" * 100)
print("BEST PARAMETERS")
print("=" * 100)

print(
    f"Units            : {UNITS}"
)

print(
    f"Activation       : {ACTIVATION}"
)

print(
    f"Dropout          : {DROPOUT:.6f}"
)

print(
    f"L2               : {L2:.10f}"
)

print(
    f"Learning Rate    : {LEARNING_RATE:.8f}"
)

print(
    f"Batch Size       : {BATCH_SIZE}"
)

print(
    f"Power            : {POWER}"
)

print(
    f"Input Features   : {input_dim}"
)

print(
    f"P99 Weight       : {P99_WEIGHT}"
)

print(
    f"P99.5 Weight     : {P995_WEIGHT}"
)

print()
model.summary()

# ============================================================
# 14. MODEL SUMMARY
# ============================================================

summary_lines = []

model.summary(
    print_fn=lambda x: summary_lines.append(x)
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "model_summary.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(summary_lines)
    )

# ============================================================
# 15. PARAMETERS JSON
# ============================================================

parameters = {

    "seed": SEED,

    "best_trial": 13,

    "units": UNITS,

    "activation": ACTIVATION,

    "dropout": DROPOUT,

    "l2": L2,

    "learning_rate": LEARNING_RATE,

    "batch_size": BATCH_SIZE,

    "max_epochs": MAX_EPOCHS,

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "lr_patience":
        LR_PATIENCE,

    "lr_factor":
        LR_FACTOR,

    "min_lr":
        MIN_LR,

    "input_dim":
        input_dim,

    "target_is_log":
        TARGET_IS_LOG,

    "power":
        POWER,

    "weighting": {

        "P50_P75":
            P50_WEIGHT,

        "P75_P90":
            P75_WEIGHT,

        "P90_P95":
            P90_WEIGHT,

        "P95_P99":
            P95_WEIGHT,

        "P99_P99_5":
            P99_WEIGHT,

        "P99_5_plus":
            P995_WEIGHT
    },

    "percentiles": {

        "P50":
            float(p50),

        "P75":
            float(p75),

        "P90":
            float(p90),

        "P95":
            float(p95),

        "P99":
            float(p99),

        "P99_5":
            float(p995)
    },

    "train_loss_for_plot":
        "weighted_power_target_mse",

    "validation_loss_for_plot":
        "weighted_power_target_mse"
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "best_parameters.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        parameters,
        f,
        indent=4
    )

# ============================================================
# 16. RESULTS JSON
# ============================================================

results = {

    "best_trial":
        13,

    "best_epoch":
        int(best_epoch),

    "power":
        POWER,

    "train_rmse":
        float(train_rmse),

    "train_mae":
        float(train_mae),

    "train_r2":
        float(train_r2),

    "validation_rmse":
        float(val_rmse),

    "validation_mae":
        float(val_mae),

    "validation_r2":
        float(val_r2),

    "overfit_gap_percent":
        float(overfit_gap),

    "train_power_rmse":
        float(train_power_rmse),

    "train_power_mae":
        float(train_power_mae),

    "train_power_r2":
        float(train_power_r2),

    "validation_power_rmse":
        float(val_power_rmse),

    "validation_power_mae":
        float(val_power_mae),

    "validation_power_r2":
        float(val_power_r2),

    "train_eval_loss_at_best_epoch":
        float(
            history_df.loc[
                best_epoch - 1,
                "train_eval_loss"
            ]
        ),

    "validation_eval_loss_at_best_epoch":
        float(
            history_df.loc[
                best_epoch - 1,
                "val_eval_loss"
            ]
        )
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "model_results.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

# ============================================================
# 17. LOSS GRAPH
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 7)
)

ax.plot(
    history_df["epoch"],
    history_df["train_eval_loss"],
    linewidth=2.2,
    label="Train Weighted Loss"
)

ax.plot(
    history_df["epoch"],
    history_df["val_eval_loss"],
    linewidth=2.2,
    label="Validation Weighted Loss"
)

ax.axvline(
    best_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Best Epoch = {best_epoch}"
)

ax.set_xlabel(
    "Epoch",
    fontsize=12
)

ax.set_ylabel(
    "Weighted MSE — USD^0.30",
    fontsize=12
)

ax.set_title(
    "Keras POWER 0.30 — Weighted Training vs Validation Loss",
    fontsize=16,
    fontweight="bold"
)

ax.grid(
    alpha=0.25,
    linestyle="--"
)

ax.legend(
    loc="upper right",
    frameon=True
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "loss_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 18. RMSE GRAPH
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 7)
)

ax.plot(
    history_df["epoch"],
    history_df["train_eval_rmse"],
    linewidth=2.2,
    label="Train Weighted RMSE"
)

ax.plot(
    history_df["epoch"],
    history_df["val_eval_rmse"],
    linewidth=2.2,
    label="Validation Weighted RMSE"
)

ax.axvline(
    best_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Best Epoch = {best_epoch}"
)

ax.set_xlabel(
    "Epoch",
    fontsize=12
)

ax.set_ylabel(
    "Weighted RMSE — USD^0.30",
    fontsize=12
)

ax.set_title(
    "Keras POWER 0.30 — Weighted Training vs Validation RMSE",
    fontsize=16,
    fontweight="bold"
)

ax.grid(
    alpha=0.25,
    linestyle="--"
)

ax.legend(
    loc="upper right",
    frameon=True
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "rmse_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 19. FINAL OUTPUT
# ============================================================

print()
print("=" * 100)
print("SAVED FILES")
print("=" * 100)

print(
    f"✓ {OUTPUT_DIR}/best_keras_model.keras"
)

print(
    f"✓ {OUTPUT_DIR}/model_results.json"
)

print(
    f"✓ {OUTPUT_DIR}/best_parameters.json"
)

print(
    f"✓ {OUTPUT_DIR}/training_history.csv"
)

print(
    f"✓ {OUTPUT_DIR}/model_summary.txt"
)

print(
    f"✓ {OUTPUT_DIR}/loss_curve.png"
)

print(
    f"✓ {OUTPUT_DIR}/rmse_curve.png"
)

print()
print("=" * 100)
print("TRAIN + VALIDATION TAMAMLANDI")
print("TEST SETİ KULLANILMADI")
print("=" * 100)

## TEST

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
POWER = 0.35
OUTPUT_DIR = "keras_final_model_weighted"
MODEL_PATH = os.path.join(OUTPUT_DIR, "best_keras_model.keras")
PARAMS_PATH = os.path.join(OUTPUT_DIR, "best_parameters.json")

np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 100)
print("KERAS TEST — POWER 0.25 + HIGH-PRICE WEIGHTING")
print("=" * 100)

# ------------------------------------------------------------
# Model ve kayıtlı parametreleri yükle
# ------------------------------------------------------------
model = keras.models.load_model(MODEL_PATH)

with open(PARAMS_PATH, "r", encoding="utf-8") as f:
    saved_params = json.load(f)

print(f"Model                 : {MODEL_PATH}")
print(f"POWER                 : {POWER}")
print("Weights               : P50=1.0 | P75=1.2 | P90=1.5 | P95=2.0 | P99=10.0 | P99.5+=25.0")

# ------------------------------------------------------------
# Test verisini hazırla
# y_test log1p(USD) formatında
# ------------------------------------------------------------
X_test = np.asarray(X_test_encoded, dtype=np.float32)
y_test_log = np.asarray(y_test, dtype=np.float32).reshape(-1)

# log1p -> USD
y_test_usd = np.maximum(np.expm1(y_test_log), 0).astype(np.float32)

# USD -> USD^0.25
y_test_power = np.power(y_test_usd, POWER).astype(np.float32)

# ------------------------------------------------------------
# Test ağırlıkları
# ------------------------------------------------------------
# Önce kayıtlı percentile değerlerini bul
def find_param(d, keys):
    if not isinstance(d, dict):
        return None

    for key in keys:
        if key in d:
            return d[key]

    for value in d.values():
        if isinstance(value, dict):
            result = find_param(value, keys)
            if result is not None:
                return result

    return None

p75 = find_param(saved_params, ["p75", "P75", "percentile_75", "P75_THRESHOLD"])
p90 = find_param(saved_params, ["p90", "P90", "percentile_90", "P90_THRESHOLD"])
p95 = find_param(saved_params, ["p95", "P95", "percentile_95", "P95_THRESHOLD"])
p99 = find_param(saved_params, ["p99", "P99", "percentile_99", "P99_THRESHOLD"])
p995 = find_param(saved_params, ["p995", "P995", "percentile_99_5", "P99.5", "P995_THRESHOLD"])

# Eğer kayıtlı eşikler bulunamazsa train verisinden üret
if None in [p75, p90, p95, p99, p995]:
    print("\nKayıtlı percentile eşikleri bulunamadı.")
    print("Train verisinden yeniden hesaplanıyor...")

    y_train_log_np = np.asarray(y_train, dtype=np.float32).reshape(-1)
    y_train_usd = np.maximum(np.expm1(y_train_log_np), 0)

    p75, p90, p95, p99, p995 = np.percentile(
        y_train_usd,
        [75, 90, 95, 99, 99.5]
    )

p75 = float(p75)
p90 = float(p90)
p95 = float(p95)
p99 = float(p99)
p995 = float(p995)

test_weights = np.ones(len(y_test_usd), dtype=np.float32)

test_weights[
    (y_test_usd >= p75) &
    (y_test_usd < p90)
] = 1.2

test_weights[
    (y_test_usd >= p90) &
    (y_test_usd < p95)
] = 1.5

test_weights[
    (y_test_usd >= p95) &
    (y_test_usd < p99)
] = 2.0

test_weights[
    (y_test_usd >= p99) &
    (y_test_usd < p995)
] = 10.0

test_weights[
    y_test_usd >= p995
] = 25.0

# ------------------------------------------------------------
# Model tahmini
# ------------------------------------------------------------
print("\nTEST PREDICTION")

prediction_power = model.predict(
    X_test,
    batch_size=1024,
    verbose=1
).reshape(-1)

prediction_power = np.maximum(prediction_power, 0)

# USD^0.25 -> USD
prediction_usd = np.power(
    prediction_power,
    1.0 / POWER
).astype(np.float64)

prediction_usd = np.maximum(prediction_usd, 0)

# ------------------------------------------------------------
# Geçerli değerleri filtrele
# ------------------------------------------------------------
valid_mask = (
    np.isfinite(y_test_usd) &
    np.isfinite(prediction_usd) &
    np.isfinite(y_test_power) &
    np.isfinite(prediction_power) &
    np.isfinite(test_weights)
)

actual_usd = y_test_usd[valid_mask].astype(np.float64)
pred_usd = prediction_usd[valid_mask]

actual_power = y_test_power[valid_mask].astype(np.float64)
pred_power = prediction_power[valid_mask].astype(np.float64)

weights = test_weights[valid_mask].astype(np.float64)

# ------------------------------------------------------------
# USD metrikleri — NORMAL
# ------------------------------------------------------------
mse_usd = mean_squared_error(actual_usd, pred_usd)
rmse_usd = np.sqrt(mse_usd)
mae_usd = mean_absolute_error(actual_usd, pred_usd)
r2_usd = r2_score(actual_usd, pred_usd)

# ------------------------------------------------------------
# USD metrikleri — AĞIRLIKLI
# ------------------------------------------------------------
mse_usd_weighted = mean_squared_error(
    actual_usd,
    pred_usd,
    sample_weight=weights
)

rmse_usd_weighted = np.sqrt(mse_usd_weighted)

mae_usd_weighted = mean_absolute_error(
    actual_usd,
    pred_usd,
    sample_weight=weights
)

r2_usd_weighted = r2_score(
    actual_usd,
    pred_usd,
    sample_weight=weights
)

# ------------------------------------------------------------
# POWER SPACE metrikleri
# ------------------------------------------------------------
mse_power = mean_squared_error(actual_power, pred_power)
rmse_power = np.sqrt(mse_power)
mae_power = mean_absolute_error(actual_power, pred_power)
r2_power = r2_score(actual_power, pred_power)

# ------------------------------------------------------------
# POWER SPACE — AĞIRLIKLI
# ------------------------------------------------------------
mse_power_weighted = mean_squared_error(
    actual_power,
    pred_power,
    sample_weight=weights
)

rmse_power_weighted = np.sqrt(mse_power_weighted)

mae_power_weighted = mean_absolute_error(
    actual_power,
    pred_power,
    sample_weight=weights
)

r2_power_weighted = r2_score(
    actual_power,
    pred_power,
    sample_weight=weights
)

# ------------------------------------------------------------
# Ağırlık dağılımı
# ------------------------------------------------------------
weight_counts = {
    "1.0": int(np.sum(weights == 1.0)),
    "1.2": int(np.sum(weights == 1.2)),
    "1.5": int(np.sum(weights == 1.5)),
    "2.0": int(np.sum(weights == 2.0)),
    "10.0": int(np.sum(weights == 10.0)),
    "25.0": int(np.sum(weights == 25.0))
}

# ------------------------------------------------------------
# Sonuçları yazdır
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("TEST SONUÇLARI — USD SPACE")
print("=" * 100)

print(f"Test Samples       : {len(actual_usd):,}")
print(f"Test Features      : {X_test.shape[1]:,}")

print("\nNORMAL:")
print(f"MSE                : ${mse_usd:,.2f}")
print(f"RMSE               : ${rmse_usd:,.2f}")
print(f"MAE                : ${mae_usd:,.2f}")
print(f"R²                 : {r2_usd:.6f}")

print("\nWEIGHTED:")
print(f"MSE                : ${mse_usd_weighted:,.2f}")
print(f"RMSE               : ${rmse_usd_weighted:,.2f}")
print(f"MAE                : ${mae_usd_weighted:,.2f}")
print(f"R²                 : {r2_usd_weighted:.6f}")

print("\n" + "=" * 100)
print("POWER TARGET — USD^0.25")
print("=" * 100)

print("\nNORMAL:")
print(f"MSE                : {mse_power:.6f}")
print(f"RMSE               : {rmse_power:.6f}")
print(f"MAE                : {mae_power:.6f}")
print(f"R²                 : {r2_power:.6f}")

print("\nWEIGHTED:")
print(f"MSE                : {mse_power_weighted:.6f}")
print(f"RMSE               : {rmse_power_weighted:.6f}")
print(f"MAE                : {mae_power_weighted:.6f}")
print(f"R²                 : {r2_power_weighted:.6f}")

print("\n" + "=" * 100)
print("WEIGHT DISTRIBUTION")
print("=" * 100)

for weight, count in weight_counts.items():
    print(f"Weight {weight:>4} : {count:>10,}")

print("\nPercentile thresholds:")
print(f"P75                : ${p75:,.2f}")
print(f"P90                : ${p90:,.2f}")
print(f"P95                : ${p95:,.2f}")
print(f"P99                : ${p99:,.2f}")
print(f"P99.5              : ${p995:,.2f}")

# ------------------------------------------------------------
# Tahmin dosyası
# ------------------------------------------------------------
predictions_df = pd.DataFrame({
    "actual_usd": actual_usd,
    "predicted_usd": pred_usd,
    "residual_usd": actual_usd - pred_usd,
    "absolute_error_usd": np.abs(actual_usd - pred_usd),
    "actual_power_0.25": actual_power,
    "predicted_power_0.25": pred_power,
    "weight": weights
})

predictions_path = os.path.join(
    OUTPUT_DIR,
    "test_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# Sonuç JSON
# ------------------------------------------------------------
results = {
    "model": MODEL_PATH,
    "power": POWER,
    "target_transformation": "log1p(USD) -> USD -> USD^0.25",
    "inverse_transformation": "USD^0.25 -> USD^(1/0.25)",
    "test_samples": int(len(actual_usd)),
    "test_features": int(X_test.shape[1]),

    "weighting": {
        "P50-P75": 1.0,
        "P75-P90": 1.2,
        "P90-P95": 1.5,
        "P95-P99": 2.0,
        "P99-P99.5": 10.0,
        "P99.5+": 25.0
    },

    "percentile_thresholds_usd": {
        "P75": p75,
        "P90": p90,
        "P95": p95,
        "P99": p99,
        "P99.5": p995
    },

    "usd_metrics": {
        "normal": {
            "mse": float(mse_usd),
            "rmse": float(rmse_usd),
            "mae": float(mae_usd),
            "r2": float(r2_usd)
        },
        "weighted": {
            "mse": float(mse_usd_weighted),
            "rmse": float(rmse_usd_weighted),
            "mae": float(mae_usd_weighted),
            "r2": float(r2_usd_weighted)
        }
    },

    "power_0.25_metrics": {
        "normal": {
            "mse": float(mse_power),
            "rmse": float(rmse_power),
            "mae": float(mae_power),
            "r2": float(r2_power)
        },
        "weighted": {
            "mse": float(mse_power_weighted),
            "rmse": float(rmse_power_weighted),
            "mae": float(mae_power_weighted),
            "r2": float(r2_power_weighted)
        }
    },

    "weight_distribution": weight_counts,
    "predictions_file": predictions_path
}

results_path = os.path.join(
    OUTPUT_DIR,
    "test_results.json"
)

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\n" + "=" * 100)
print("SAVED FILES")
print("=" * 100)
print(f"✓ {MODEL_PATH}")
print(f"✓ {predictions_path}")
print(f"✓ {results_path}")
print("=" * 100)

Bu Keras modelinin bağımsız test performansı normal metriklerde iyi görünüyor: RMSE 16.71M, MAE 2.59M ve R² 0.8301. Özellikle MAE’nin validation sonucundaki 2.68M’den daha düşük olması, test setinde ortalama mutlak hatanın beklenenden daha iyi olduğunu gösteriyor. Ağırlıklı sonuçlarda ise R² 0.8874 seviyesine yükselirken RMSE 54.79M ve MAE 9.28M seviyesine çıkıyor; bunun nedeni ağırlıklandırmanın özellikle çok yüksek fiyatlı örneklerin hatalarını çok daha fazla öne çıkarmasıdır. Power-space normal R² 0.5390 iken weighted R² 0.8472 olması, modelin ağırlık verilen yüksek fiyat segmentlerinde hedefi çok daha iyi açıklayabildiğini gösteriyor. Test setinde P99.5 üzerindeki 242 örneğe 25 kat, P99–P99.5 arasındaki 201 örneğe 10 kat ağırlık verilmesi oldukça agresif bir tail odaklanması oluşturmuş. Genel olarak model normal test performansında R²=0.83 ile başarılı, yüksek fiyatlı örneklerde ise ağırlıklandırma sayesinde belirgin şekilde daha güçlü; ancak weighted RMSE ve MAE'nin ciddi yükselmesi, uç fiyatlardaki hataların hâlâ modelin temel zayıf noktası olduğunu gösteriyor.

## KATEGORİK KOLONLAR

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

KERAS_DIR="keras_final_model_weighted"
XGB_DIR="xgboost_weighted_high_price"
OUT="categorical_residual_plots"
os.makedirs(OUT,exist_ok=True)

SEED=42
MIN_COUNT=30
TOP_N=12

keras=pd.read_csv(f"{KERAS_DIR}/test_predictions.csv")
xgb=pd.read_csv(f"{XGB_DIR}/test_predictions.csv")

if len(keras)!=len(xgb):
    raise ValueError("Keras ve XGBoost test prediction sayıları farklı.")

df=df_test.to_pandas() if hasattr(df_test,"to_pandas") else df_test.copy()

if len(df)!=len(keras):
    raise ValueError("df_test ve prediction satır sayıları farklı.")

exclude={
    "tender_finalpriceusd","actual_target","predicted_target",
    "actual_usd","predicted_usd","residual_usd",
    "absolute_error_usd","index"
}

cat_cols=[]

for c in df.columns:
    if c.lower() in exclude:
        continue

    s=df[c]

    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
        continue

    if pd.to_numeric(s,errors="coerce").notna().mean()<=.95:
        cat_cols.append(c)

rng=np.random.default_rng(SEED)

def residual_plot(ax,residuals,categories,cats,title):

    y=np.asarray(residuals)/1e6
    x=np.array([np.where(cats==c)[0][0] for c in categories])
    x=x+rng.normal(0,.10,len(x))

    try:
        density=gaussian_kde(y)(y)
        order=np.argsort(density)

        sc=ax.scatter(
            x[order],y[order],
            c=density[order],
            cmap="turbo",
            s=8,
            alpha=.65,
            edgecolors="none",
            rasterized=True
        )

        plt.colorbar(sc,ax=ax,label="Point Density")

    except Exception:
        ax.scatter(
            x,y,
            s=8,
            alpha=.65,
            edgecolors="none"
        )

    ax.axhline(0,linestyle="--",linewidth=1)

    ax.set_xticks(range(len(cats)))
    ax.set_xticklabels(cats,rotation=35,ha="right")

    ax.set_ylabel("Residual (USD)")
    ax.set_title(title)
    ax.grid(alpha=.15)

for feature in cat_cols:

    cat=df[feature].fillna("MISSING").astype(str)

    counts=cat.value_counts()
    cats=counts[counts>=MIN_COUNT].head(TOP_N).index.to_numpy()

    if len(cats)==0:
        continue

    mask=cat.isin(cats).to_numpy()
    categories=cat[mask].to_numpy()

    xgb_res=xgb.loc[mask,"residual_usd"].to_numpy()
    keras_res=keras.loc[mask,"residual_usd"].to_numpy()

    fig,axes=plt.subplots(
        1,2,
        figsize=(16,6),
        sharey=True
    )

    residual_plot(
        axes[0],
        xgb_res,
        categories,
        cats,
        "XGBoost Weighted"
    )

    residual_plot(
        axes[1],
        keras_res,
        categories,
        cats,
        "Keras Weighted"
    )

    fig.suptitle(
        f"{feature} — Categorical Residual Analysis",
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout()

    safe_name=str(feature).replace("/","_").replace("\\","_")

    plt.savefig(
        f"{OUT}/{safe_name}_residual.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

print(f"\nResidual plotları tamamlandı: {OUT}")

**tender_nationalproceduretype (Ulusal Prosedür Tipi)**

En yüksek dikey artık saçılımı pt_open ve pt_competitive_negotiation kategorilerinde gözlenmektedir. XGBoost modeli sapmaları ağırlıklı olarak pozitif eksende tutup varyansı kısıtlarken, Keras modeli pt_open kategorisinde -750 USD'ye ulaşan derin negatif uç hatalar sergilemiştir.

**tender_supplytype (İhale Tedarik Tipi)**

WORKS, SERVICES ve SUPPLIES kategorilerinin tamamında ana veri kümesi sıfır çizgisinde yoğunlaşmıştır. XGBoost dikey varyansı pozitif tarafta kontrol altında tutmayı başarırken; Keras modeli özellikle SERVICES ve WORKS sınıflarında simetrik olarak negatif eksene doğru çok geniş ve derin hatalar üretmiştir.

**lot_status (Lot Durumu)**

Artık saçılımının büyük kısmı AWARDED ve PREAWARDED sınıflarında toplanmaktadır; CANCELLED tarafında sapmalar yok denecek kadar azdır. XGBoost pozitif aykırı değerleri sınırlı bir bantta tutarken, Keras modeli AWARDED kategorisinde negatif yönde ciddi bir hata varyansı göstererek ayrışmıştır.

**tender_isframeworkagreement (Çerçeve Anlaşma)**

Anlaşmanın olmadığı (f) grupta veri hacmine bağlı olarak artık varyansı t grubuna göre belirgin şekilde daha yüksektir. XGBoost her iki grupta da sapmaları pozitif eksende derli toplu yönetirken, Keras f kategorisinde alt tarafa doğru fırlayan aşırı negatif uç hatalara izin vermiştir.

**tender_selectionmethod (Seçim Yöntemi)**

LOWEST_PRICE kategorisi, MEAT (en ekonomik avantajlı teklif) kategorisine kıyasla daha geniş bir hata saçılımına sahiptir. XGBoost tarafında sapmalar 0 ile +500 USD aralığında kalırken, Keras modeli LOWEST_PRICE altında -750 USD seviyelerine kadar inen tutarsız tahminler sergilemiştir.

**buyer_mainactivities (Alıcı Ana Faaliyet Alanı)**

En yüksek sapmalar ve veri yoğunluğu GENERAL_PUBLIC_SERVICES ve OTHER sektörlerinde kümelenmiştir. XGBoost modeli bu ana sektörlerde pozitif uç hataları dar bir varyansta tutarken, Keras modeli aynı sektörlerde alt tarafa doğru sapan derin negatif artıklar vermiştir.

**competition_level (Rekabet Seviyesi)**

Low Competition, High Competition ve Medium Competition sınıflarının tümünde sıfır merkezli yoğunlaşma mevcuttur. XGBoost varyansı pozitif tarafta dengeli biçimde yönetirken, Keras modeli özellikle Low Competition kategorisinde her iki yönde de derinleşen yüksek aykırı değer frekansı sergilemiştir.

## NUMERİK KOLONLAR

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

KERAS_DIR="keras_final_model_weighted"
XGB_DIR="xgboost_weighted_high_price"
OUT="numeric_residual_plots"
os.makedirs(OUT,exist_ok=True)

SEED=42
rng=np.random.default_rng(SEED)

keras=pd.read_csv(f"{KERAS_DIR}/test_predictions.csv")
xgb=pd.read_csv(f"{XGB_DIR}/test_predictions.csv")

if len(keras)!=len(xgb):
    raise ValueError("Keras ve XGBoost test prediction sayıları farklı.")

df=df_test.to_pandas() if hasattr(df_test,"to_pandas") else df_test.copy()

if len(df)!=len(keras):
    raise ValueError("df_test ve prediction satır sayıları farklı.")

exclude={
    "tender_finalpriceusd","actual_target","predicted_target",
    "actual_usd","predicted_usd","residual_usd",
    "absolute_error_usd","index"
}

num_cols=[
    c for c in df.columns
    if c.lower() not in exclude
    and pd.api.types.is_numeric_dtype(df[c])
    and not pd.api.types.is_bool_dtype(df[c])
]

def residual_plot(ax,x,residuals,title):

    x=np.asarray(x,dtype=float)
    y=np.asarray(residuals,dtype=float)/1e6

    valid=np.isfinite(x)&np.isfinite(y)
    x=x[valid]
    y=y[valid]

    if len(x)<2:
        return

    try:
        xy=np.vstack([x,y])
        density=gaussian_kde(xy)(xy)
        order=np.argsort(density)

        sc=ax.scatter(
            x[order],y[order],
            c=density[order],
            cmap="turbo",
            s=8,
            alpha=.65,
            edgecolors="none",
            rasterized=True
        )

        plt.colorbar(sc,ax=ax,label="Point Density")

    except Exception:
        ax.scatter(
            x,y,
            s=8,
            alpha=.65,
            edgecolors="none",
            rasterized=True
        )

    ax.axhline(0,linestyle="--",linewidth=1)
    ax.set_xlabel(title)
    ax.set_ylabel("Residual (USD)")
    ax.grid(alpha=.15)

for feature in num_cols:

    values=pd.to_numeric(df[feature],errors="coerce").to_numpy()

    valid=np.isfinite(values)

    if valid.sum()<2:
        continue

    x=values[valid]
    xgb_res=xgb.loc[valid,"residual_usd"].to_numpy()
    keras_res=keras.loc[valid,"residual_usd"].to_numpy()

    fig,axes=plt.subplots(
        1,2,
        figsize=(16,6),
        sharey=True
    )

    residual_plot(
        axes[0],
        x,
        xgb_res,
        feature
    )

    residual_plot(
        axes[1],
        x,
        keras_res,
        feature
    )

    axes[0].set_title("XGBoost Weighted")
    axes[1].set_title("Keras Weighted")

    fig.suptitle(
        f"{feature} — Numeric Residual Analysis",
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout()

    safe_name=str(feature).replace("/","_").replace("\\","_")

    plt.savefig(
        f"{OUT}/{safe_name}_residual.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

print(f"\nNumerik residual plotları tamamlandı: {OUT}")
print(f"Analiz edilen numerik kolon: {len(num_cols)}")

**log_tender_lotscount (Lot Sayısı)**

Düşük lot sayılarında dikey sapmalar ve aykırı değer frekansı oldukça yüksekken, logaritmik lot sayısı arttıkça hata varyansı daralarak sıfır eksenine tam oturmaktadır. XGBoost modeli sapmaları yalnızca pozitif tarafta kısıtlayarak daha kararlı bir profil çizerken, Keras tarafında düşük lot sayılarında eksi yönlü derin ve kontrolsüz uç hatalar gözlenmektedir.

**log_tender_recordedbidscount (Teklif Sayısı)**

Düşük teklif sayısına sahip ihalelerde belirsizlik yüksek olduğundan her iki modelde de artık varyansı geniştir; ancak teklif sayısı arttıkça tahminler oldukça kararlı hale gelmektedir. XGBoost modeli yüksek sapmaları tek yönlü (pozitif) tutarak genel varyansı dar bir bantta yönetmeyi başarırken, Keras modeli özellikle düşük teklif sayılarında eksi yönlü aşırı uç değerler üreterek daha yüksek dalgalanma göstermektedir.

**tender_awardcriteria_count (Değerlendirme Kriteri Sayısı)**

Kriter sayısının az olduğu (özellikle 1 ve civarı) ihalelerde veri yoğunluğu yüksek olup model sapmaları bu bölgede tepe yapmaktadır. Kriter sayısı arttıkça hata varyansı hızla sönümlenmekte ve sıfır çizgisine paralel stabil bir hat izlemektedir. XGBoost dikey varyansı daha derli toplu tutarken, Keras az kriterli gruplarda negatif tarafa doğru saçılan daha geniş bir hata aralığı sergilemektedir.

**tender_year (İhale Yılı)**

Tüm yıllar boyunca ana veri kütlesi sıfır çizgisinde son derece kararlı ve yoğun bir kümelenme göstermektedir. Son yıllara doğru (özellikle 2018-2021) veri hacmindeki veya tutarlardaki değişimlere bağlı olarak üst ve alt aykırı değerlerin sıklığı artmıştır. XGBoost yılı fark etmeksizin hataları üst bantta daha sınırlı tutarken, Keras modeli özellikle yakın yıllarda alt tarafa doğru fırlayan negatif uç hatalara izin vermiştir.

**log_submission_period (Teklif Sunma Süresi)**

Yoğunluğun merkezlendiği orta uzunluktaki başvuru sürelerinde (log değeri 3.5 civarı) sapma miktarı ve aykırı değerler pik yapmaktadır. Çok kısa veya çok uzun başvuru sürelerinde ise modeller oldukça düşük hata varyansı ile çalışmaktadır. XGBoost bu orta bantta sadece pozitif yönlü sapmalarla varyansı kontrol altında tutarken, Keras modeli hem pozitif hem negatif yönde çok geniş bir saçılım sergilemektedir.

**log_decision_period (Karar Süresi Logaritmik)**

Orta ve yüksek karar sürelerinde (log değeri 4 ve üzeri) veri yoğunluğu artmakta ve bu bölgede pozitif yönlü artık saçılımları tepe yapmaktadır. Kısa karar sürelerinde tahminler sıfır çizgisine yakın, stabil bir hat izlemektedir. XGBoost dikey sapmaları tek taraflı tutarak varyansı sınırlı bir bantta kontrol ederken, Keras modeli özellikle orta-uzun karar sürelerinde negatif yönde aşırı derin ve uç hatalar üretmiştir.

**decision_duration_days (Karar Süresi Gün)**

Verinin büyük bir kısmı kısa gün sürelerinde (0-250 gün aralığında) kümelenmiş olup dikey varyans ve aykırı değer frekansı bu bölgede en yüksek seviyededir. Süre uzadıkça (500 gün ve üzeri) veri seyrekleşmekte ve modeller sıfır eksenine paralel oldukça kararlı tahminler yapmaktadır. XGBoost düşük gün sürelerindeki sapmaları pozitif tarafta tutarken, Keras aynı bölgede negatif yönde kontrolsüz fırlamalar sergilemiştir.

**corr_singleb (Tek Teklif Veren Korelasyonu)**

Belirli değerlerde (-1, 0, 1) ayrıklaşan bu özellikte ana veri kütlesi sıfır noktasında yoğunlaşmıştır. Sıfır değerinde dikey sapma miktarı her iki modelde de tepe noktasına ulaşmaktadır. XGBoost dikey varyansı daha dar bir aralıkta tutmayı başarırken, Keras özellikle sıfır kategorisinde alt tarafa doğru çok daha geniş bir hata saçılımı göstermiştir.

**corr_proc (Süreç Korelasyonu)**

Korelasyonun 0, 0.5 ve 1 olduğu belirgin noktalarda veri kümelenmektedir; en yüksek hata dikey saçılımı 0 noktasında gözlenmektedir. Korelasyon değeri 1'e yaklaştıkça hata varyansı gözle görülür şekilde azalmaktadır. XGBoost aykırı değerleri pozitif eksende sınırlandırıp varyansı kontrol altında tutarken, Keras modeli 0 noktasında negatif yönde çok yüksek uç hatalara izin vermiştir.

**corr_subm (Başvuru Korelasyonu)**

Ayrık korelasyon değerlerinde (-1, 0, 0.5, 1) kümelenen veride ana kütle sıfır ekseninde stabil oturmaktadır. XGBoost tüm kategorilerde üst banttaki birkaç aykırı değer dışında dikey saçılımı dar bir aralıkta yönetmiştir. Keras tarafında ise -1 ve 1 uç değerlerinde negatif sapmalar derinleşerek modelin hata varyansını belirgin şekilde artırmıştır.

**corr_nocft (Ihale Cağrısı Yayınlanmama Korelasyonu)**

İkili (binary) kümelenme gösteren bu özellikte veriler 0 ve 1 noktalarında toplanmıştır. XGBoost dikey sapmaları ağırlıklı olarak pozitif tarafta tutarak varyansı oldukça dar bir aralıkta kısıtlamıştır. Keras modeli ise özellikle 0 noktasında alt tarafa doğru fırlayan derin negatif hatalar üreterek belirgin şekilde daha geniş bir hata dağılımı sergilemiştir.

**corr_decp (Karar Süresi İhlal Korelasyonu)**

Verilerin 0 ve 1 değerlerinde kümelendiği bu değişkende, XGBoost tarafında artıklar sıfır eksenine son derece yakın ve derli toplu bir görünüm sunmaktadır. Keras modeli ise 1 noktasında hem pozitif hem negatif yönde aşırı geniş sapmalar göstererek yüksek aykırı değer frekansına ulaşmıştır.

**corr_buyer_concentration (Alıcı Yoğunlaşması Korelasyonu)**

Veriler -1 ile 0-1 aralığında dağılmış olup ana yoğunluk 0-0.25 bandındadır. XGBoost modeli bu yoğun bölgedeki sapmaları pozitif tarafta sınırlandırıp varyansı kontrol altında tutarken, Keras modeli 0 yakınlarında negatif eksene doğru oldukça derin ve kontrolsüz uç hatalar vermiştir.

**log_tender_finalpriceUsd (İhale Final Fiyatı Logaritmik)**

Düşük ve orta bütçeli ihalelerde (log değeri 5-16 arası) her iki model de son derece düşük hatayla çalışırken, 16 ve üzeri yüksek bütçeli ihalelerde eksponansiyel bir artık artışı (underfitting) gözlenmektedir. XGBoost bu yüksek bütçelerde yalnızca pozitif sapmalar gösterirken, Keras modeli orta bütçelerde eksi yönlü uç hatalar, yüksek bütçelerde ise pozitif eksonda fırlamalar sergilemiştir.

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

POWER = 0.35

XGB_FILE = "xgboost_weighted_high_price/final_test_predictions.csv"
NN_FILE = "keras_final_model_weighted/test_predictions.csv"

OUT_DIR = "plotly_residuals_log_target"
os.makedirs(OUT_DIR, exist_ok=True)


def target_to_usd(x):
    return np.power(np.maximum(np.asarray(x, dtype=np.float64), 0), 1 / POWER)


def get_log_target(df):
    if "log_tender_finalpriceUsd" in df.columns:
        return df["log_tender_finalpriceUsd"].to_numpy()

    for name in ["df_test", "test_df", "test_data"]:
        if name in globals():
            test_df = globals()[name]

            if "log_tender_finalpriceUsd" not in test_df.columns:
                continue

            if "index" in df.columns and "index" in test_df.columns:
                lookup = test_df.set_index("index")["log_tender_finalpriceUsd"]
                x = df["index"].map(lookup).to_numpy()
            else:
                x = test_df["log_tender_finalpriceUsd"].to_numpy()

            if len(x) == len(df):
                return x

    raise ValueError("log_tender_finalpriceUsd bulunamadı.")


def make_plot(file_path, model_name, output_file):
    df = pd.read_csv(file_path)

    if {"actual_usd", "predicted_usd"}.issubset(df.columns):
        actual_usd = df["actual_usd"].to_numpy(dtype=np.float64)
        predicted_usd = df["predicted_usd"].to_numpy(dtype=np.float64)
    else:
        actual_usd = target_to_usd(df["actual_target"])
        predicted_usd = target_to_usd(df["predicted_target"])

    log_target = get_log_target(df)

    residual_usd = actual_usd - predicted_usd
    absolute_error_usd = np.abs(residual_usd)

    valid = (
        np.isfinite(log_target)
        & np.isfinite(actual_usd)
        & np.isfinite(predicted_usd)
        & np.isfinite(residual_usd)
    )

    log_target = log_target[valid]
    actual_usd = actual_usd[valid]
    predicted_usd = predicted_usd[valid]
    residual_usd = residual_usd[valid]
    absolute_error_usd = absolute_error_usd[valid]

    rmse = np.sqrt(mean_squared_error(actual_usd, predicted_usd))
    mae = mean_absolute_error(actual_usd, predicted_usd)
    r2 = r2_score(actual_usd, predicted_usd)

    customdata = np.column_stack([
        actual_usd,
        predicted_usd,
        absolute_error_usd
    ])

    fig = go.Figure()

    fig.add_trace(go.Scattergl(
        x=log_target,
        y=residual_usd,
        mode="markers",
        marker=dict(size=4, opacity=0.45),
        customdata=customdata,
        hovertemplate=(
            "<b>Log Target:</b> %{x:.4f}<br>"
            "<b>Actual USD:</b> $%{customdata[0]:,.2f}<br>"
            "<b>Predicted USD:</b> $%{customdata[1]:,.2f}<br>"
            "<b>Residual USD:</b> $%{y:,.2f}<br>"
            "<b>Absolute Error:</b> $%{customdata[2]:,.2f}"
            "<extra></extra>"
        )
    ))

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_width=2
    )

    fig.update_layout(
        title=(
            f"{model_name} — Residual vs Log Target<br>"
            f"<sup>"
            f"RMSE: ${rmse:,.2f} | "
            f"MAE: ${mae:,.2f} | "
            f"R²: {r2:.4f}"
            f"</sup>"
        ),
        xaxis_title="log_tender_finalpriceUsd",
        yaxis_title="Residual (Actual USD − Predicted USD)",
        template="plotly_white",
        height=750
    )

    # Grafiği doğrudan göster
    fig.show()

    # Aynı zamanda HTML olarak kaydet
    fig.write_html(output_file)

    print(f"\n{model_name}")
    print(f"Samples : {len(residual_usd):,}")
    print(f"RMSE    : ${rmse:,.2f}")
    print(f"MAE     : ${mae:,.2f}")
    print(f"R²      : {r2:.6f}")
    print(f"Saved   : {output_file}")


make_plot(
    XGB_FILE,
    "XGBoost",
    f"{OUT_DIR}/xgboost_residual_vs_log_target.html"
)

make_plot(
    NN_FILE,
    "Neural Network",
    f"{OUT_DIR}/neural_network_residual_vs_log_target.html"
)

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

XGB_FILE = "xgboost_weighted_high_price/final_test_predictions.csv"
NN_FILE = "keras_final_model_weighted/test_predictions.csv"

OUT_DIR = "plotly_residuals_log_target"
os.makedirs(OUT_DIR, exist_ok=True)


def get_log_target(df):
    if "log_tender_finalpriceUsd" in df.columns:
        return np.asarray(
            df["log_tender_finalpriceUsd"],
            dtype=np.float64
        )

    for name in ["df_test", "test_df", "test_data"]:
        if name in globals():
            test_df = globals()[name]

            if "log_tender_finalpriceUsd" not in test_df.columns:
                continue

            if "index" in df.columns and "index" in test_df.columns:
                lookup = test_df.set_index("index")[
                    "log_tender_finalpriceUsd"
                ]

                x = np.asarray(
                    df["index"].map(lookup),
                    dtype=np.float64
                )
            else:
                x = np.asarray(
                    test_df["log_tender_finalpriceUsd"],
                    dtype=np.float64
                )

            if len(x) == len(df):
                return x

    raise ValueError(
        "log_tender_finalpriceUsd bulunamadı."
    )


def make_plot(file_path, model_name, output_file):
    df = pd.read_csv(file_path)

    required = [
        "actual_usd",
        "predicted_usd",
        "residual_usd"
    ]

    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(
            f"{model_name} dosyasında eksik kolonlar: {missing}"
        )

    actual_usd = np.asarray(
        df["actual_usd"],
        dtype=np.float64
    )

    predicted_usd = np.asarray(
        df["predicted_usd"],
        dtype=np.float64
    )

    residual_usd = np.asarray(
        df["residual_usd"],
        dtype=np.float64
    )

    absolute_error_usd = np.abs(residual_usd)

    # X ekseni: log_tender_finalpriceUsd
    log_target = get_log_target(df)

    valid = (
        np.isfinite(log_target)
        & np.isfinite(actual_usd)
        & np.isfinite(predicted_usd)
        & np.isfinite(residual_usd)
    )

    log_target = log_target[valid]
    actual_usd = actual_usd[valid]
    predicted_usd = predicted_usd[valid]
    residual_usd = residual_usd[valid]
    absolute_error_usd = absolute_error_usd[valid]

    rmse = np.sqrt(
        mean_squared_error(actual_usd, predicted_usd)
    )

    mae = mean_absolute_error(
        actual_usd,
        predicted_usd
    )

    r2 = r2_score(
        actual_usd,
        predicted_usd
    )

    customdata = np.column_stack([
        actual_usd,
        predicted_usd,
        absolute_error_usd
    ])

    fig = go.Figure()

    fig.add_trace(
        go.Scattergl(
            x=log_target,
            y=residual_usd,
            mode="markers",
            marker=dict(
                size=4,
                opacity=0.45
            ),
            customdata=customdata,
            hovertemplate=(
                "<b>Log Target:</b> %{x:.4f}<br>"
                "<b>Actual USD:</b> $%{customdata[0]:,.2f}<br>"
                "<b>Predicted USD:</b> $%{customdata[1]:,.2f}<br>"
                "<b>Residual USD:</b> $%{y:,.2f}<br>"
                "<b>Absolute Error:</b> $%{customdata[2]:,.2f}"
                "<extra></extra>"
            ),
            name=model_name
        )
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_width=2
    )

    fig.update_layout(
        title=(
            f"{model_name} — Residual vs Log Target<br>"
            f"<sup>"
            f"RMSE: ${rmse:,.2f} | "
            f"MAE: ${mae:,.2f} | "
            f"R²: {r2:.4f}"
            f"</sup>"
        ),
        xaxis_title="log_tender_finalpriceUsd",
        yaxis_title="Residual (Actual USD − Predicted USD)",
        template="plotly_white",
        height=750
    )

    # Grafiği notebook çıktısında göster
    fig.show()

    # HTML olarak da kaydet
    fig.write_html(output_file)

    print(f"\n{'=' * 70}")
    print(model_name)
    print(f"{'=' * 70}")
    print(f"Samples : {len(residual_usd):,}")
    print(f"RMSE    : ${rmse:,.2f}")
    print(f"MAE     : ${mae:,.2f}")
    print(f"R²      : {r2:.6f}")
    print(f"Saved   : {output_file}")


make_plot(
    XGB_FILE,
    "XGBoost",
    f"{OUT_DIR}/xgboost_residual_vs_log_target.html"
)

make_plot(
    NN_FILE,
    "Neural Network",
    f"{OUT_DIR}/neural_network_residual_vs_log_target.html"
)